In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from mpl_toolkits.mplot3d import Axes3D
from scipy.interpolate import interp1d

torch.manual_seed(42)
np.random.seed(42)

# ==========================================
# Part 1: Color Math (Oklab) - Unchanged
# ==========================================
M1 = torch.tensor([[0.8189330101, 0.3618667424, -0.1288597137],
                   [0.0329845436, 0.9293118715, 0.0361456387],
                   [0.0482003018, 0.2643662703, 0.6338517070]], dtype=torch.float32)
M2 = torch.tensor([[0.2104542553, 0.7936177850, -0.0040720468],
                   [1.9779984951, -2.4285922050, 0.4505937099],
                   [0.0259040371, 0.7827717662, -0.8086757660]], dtype=torch.float32)
M1_inv = torch.linalg.inv(M1)
M2_inv = torch.linalg.inv(M2)

def xyz_to_linear_srgb(xyz):
    M_xyz_rgb = torch.tensor([[3.2404542, -1.5371385, -0.4985314],
                              [-0.9692660, 1.8760108, 0.0415560],
                              [0.0556434, -0.2040259, 1.0572252]], 
                              dtype=torch.float32, device=xyz.device).T
    return xyz @ M_xyz_rgb

def oklab_to_linear_srgb(lab):
    lms_prime = lab @ M2_inv.T.to(lab.device)
    lms = torch.pow(lms_prime, 3)
    xyz = lms @ M1_inv.T.to(lab.device)
    return xyz_to_linear_srgb(xyz)

def linear_srgb_to_oklab(rgb):
    M_rgb_xyz = torch.tensor([[0.4124564, 0.3575761, 0.1804375],
                              [0.2126729, 0.7151522, 0.0721750],
                              [0.0193339, 0.1191920, 0.9503041]], 
                              dtype=torch.float32, device=rgb.device).T
    xyz = rgb @ M_rgb_xyz
    lms = xyz @ M1.T.to(rgb.device)
    lms_prime = torch.pow(torch.clamp(lms, min=1e-9), 1/3)
    return lms_prime @ M2.T.to(rgb.device)

def linear_to_gamma_srgb(rgb_lin):
    mask = rgb_lin > 0.0031308
    rgb_gamma = torch.zeros_like(rgb_lin)
    rgb_gamma[mask] = 1.055 * torch.pow(rgb_lin[mask], 1/2.4) - 0.055
    rgb_gamma[~mask] = 12.92 * rgb_lin[~mask]
    return torch.clamp(rgb_gamma, 0.0, 1.0)

# ==========================================
# Part 2: Parametric Model
# ==========================================
class RepulsionLoop(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Lightness: Needs to be flexible to ride the roller coaster
        self.order_L = 4
        self.L_dc = nn.Parameter(torch.tensor(0.65)) 
        self.L_coeffs = nn.Parameter(torch.zeros(self.order_L, 2) * 0.02)
        
        # Chroma: We keep this reasonably flexible (Order 3) 
        # because we will fix the shape using the LOSS FUNCTION, not the parameter limit.
        self.order_C = 3
        self.C_dc = nn.Parameter(torch.tensor(0.15)) 
        self.C_coeffs = nn.Parameter(torch.zeros(self.order_C, 2) * 0.02)

        with torch.no_grad():
             self.L_coeffs[0, 0] = 0.20 

    def forward(self, t):
        L = self.L_dc.expand_as(t)
        C = self.C_dc.expand_as(t)
        
        for k in range(1, self.order_L + 1):
            theta = k * t
            L = L + self.L_coeffs[k-1, 0]*torch.sin(theta) + self.L_coeffs[k-1, 1]*torch.cos(theta)
            
        for k in range(1, self.order_C + 1):
            theta = k * t
            C = C + self.C_coeffs[k-1, 0]*torch.sin(theta) + self.C_coeffs[k-1, 1]*torch.cos(theta)
        
        C = torch.abs(C)
        a = C * torch.cos(t)
        b = C * torch.sin(t)
        
        return torch.stack([L, a, b], dim=1)

# ==========================================
# Part 3: Optimization with HARD REPULSION
# ==========================================

def optimize_repulsion_loop(iterations=3000, device='cpu'):
    model = RepulsionLoop().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.015)
    
    t_grid = torch.linspace(0, 2*np.pi, 360, device=device)
    
    # Tuned Weights
    W_GAMUT = 500.0
    W_CHROMA = 8.0 
    W_DARKNESS = 15.0
    
    # THE NEW FORCE:
    # This weight must be high enough to fight the Gamut constraint
    W_REPULSION = 50.0 
    
    # TARGET MINIMUM CHROMA
    # We want C to never drop below 0.14 (approx half the max saturation)
    TARGET_MIN_C = 0.14
    
    print("Optimizing with Center Repulsion...")
    
    for i in range(iterations):
        optimizer.zero_grad()
        lab_out = model(t_grid)
        
        # 1. Gamut
        rgb_lin = oklab_to_linear_srgb(lab_out)
        lower = F.relu(0.0001 - rgb_lin)
        upper = F.relu(rgb_lin - 0.9999)
        gamut_loss = torch.sum((lower + upper)**2)
        
        # 2. Maximize Average Chroma
        C_vals = torch.sqrt(lab_out[:,1]**2 + lab_out[:,2]**2)
        chroma_loss = -torch.mean(C_vals)
        
        # 3. Anti-Mud Floor (Lightness)
        L_vals = lab_out[:, 0]
        darkness_penalty = torch.sum(F.relu(0.35 - L_vals)**2)
        
        # 4. HARD REPULSION (The "Antigravity" Wall)
        # Instead of just penalizing C < 0.14 linearly, we use an exponential barrier
        # If C drops near 0.14, the loss explodes.
        # This forces the optimizer to "inflate" the balloon.
        
        # Soft barrier implementation:
        # Loss = ReLU(Target - C)^2 * Scaling
        repulsion_loss = torch.sum(F.relu(TARGET_MIN_C - C_vals)**2)
        
        # 5. Convexity / Smoothness Regularization
        reg_loss = 0.0
        for k in range(1, model.order_C + 1):
            # Heavily penalize high-frequency wiggles in Chroma
            reg_loss += (k**3) * torch.sum(model.C_coeffs[k-1]**2)
            
        total_loss = (W_GAMUT*gamut_loss + 
                      W_CHROMA*chroma_loss + 
                      W_DARKNESS*darkness_penalty + 
                      W_REPULSION*repulsion_loss +
                      W_REG*reg_loss if 'W_REG' in locals() else 1.0*reg_loss) # Default reg weight
        
        total_loss.backward()
        optimizer.step()
        
        if i % 500 == 0:
            print(f"Iter {i} | G: {gamut_loss.item():.2f} | Min C: {C_vals.min().item():.3f}")
            
    return model

# ==========================================
# Part 4: Visualization
# ==========================================

def reparameterize_by_arclength(lab_points_np):
    dists = np.linalg.norm(lab_points_np - np.roll(lab_points_np, 1, axis=0), axis=1)
    dists[0] = np.linalg.norm(lab_points_np[0] - lab_points_np[-1])
    cumulative = np.cumsum(dists)
    total = cumulative[-1]
    t_curr = np.concatenate(([0], cumulative / total))
    pts = np.vstack([lab_points_np[-1:], lab_points_np])
    
    iL = interp1d(t_curr, pts[:,0], kind='cubic')
    ia = interp1d(t_curr, pts[:,1], kind='cubic')
    ib = interp1d(t_curr, pts[:,2], kind='cubic')
    t_targ = np.linspace(0, 1, len(lab_points_np) + 1)[:-1]
    return np.stack([iL(t_targ), ia(t_targ), ib(t_targ)], axis=1)

def get_gamut_edges(res=20):
    t = torch.linspace(0, 1, res)
    edges = []
    for i in [0, 1]:
        for j in [0, 1]:
             edges.append(torch.stack([t, torch.full_like(t,i), torch.full_like(t,j)],1))
             edges.append(torch.stack([torch.full_like(t,i), t, torch.full_like(t,j)],1))
             edges.append(torch.stack([torch.full_like(t,i), torch.full_like(t,j), t],1))
    return linear_srgb_to_oklab(torch.cat(edges,0)).cpu().numpy()

def plot_3d(path, edges):
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(edges[:,1], edges[:,2], edges[:,0], c='k', s=0.5, alpha=0.1)
    
    rgb = linear_to_gamma_srgb(oklab_to_linear_srgb(torch.from_numpy(path).float())).numpy()
    rgb = np.clip(rgb, 0, 1)
    
    for i in range(len(path)-1):
        ax.plot(path[i:i+2, 1], path[i:i+2, 2], path[i:i+2, 0], color=rgb[i], linewidth=5)
    ax.plot([path[-1,1], path[0,1]], [path[-1,2], path[0,2]], [path[-1,0], path[0,0]], color=rgb[-1], linewidth=5)
    
    ax.set_xlabel('a'); ax.set_ylabel('b'); ax.set_zlabel('L')
    ax.set_zlim(0, 1)
    ax.view_init(25, -60)
    plt.title("Final Convex Loop with Hard Repulsion")

def plot_disk(rgb_data):
    N = 500
    theta = np.linspace(0, 2*np.pi, N)
    r = np.linspace(0, 1, N)
    Theta, R = np.meshgrid(theta, r)
    rgb_rolled = np.roll(rgb_data, shift=0, axis=0) 
    cmap = ListedColormap(np.clip(rgb_rolled, 0, 1))
    
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'projection': 'polar'})
    ax.pcolormesh(Theta, R, Theta, cmap=cmap, shading='gouraud')
    ax.set_yticks([]); ax.set_xticks([])
    plt.title("Repulsed Hue Disk")

if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # 1. Optimization
    model = optimize_repulsion_loop(device=device)
    
    # 2. Sampling
    with torch.no_grad():
        t = torch.linspace(0, 2*np.pi, 512, device=device)
        lab_raw = model(t).cpu().numpy()
        
    # 3. Balance
    lab_final = reparameterize_by_arclength(lab_raw)
    
    # 4. View
    gamut = get_gamut_edges()
    plot_3d(lab_final, gamut)
    
    rgb_final = linear_to_gamma_srgb(oklab_to_linear_srgb(torch.from_numpy(lab_final).float())).numpy()
    plot_disk(rgb_final)
    plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from scipy.interpolate import interp1d

torch.manual_seed(42)
np.random.seed(42)

# ==========================================
# Part 1: Color Math
# ==========================================
M1 = torch.tensor([[0.8189330101, 0.3618667424, -0.1288597137],
                   [0.0329845436, 0.9293118715, 0.0361456387],
                   [0.0482003018, 0.2643662703, 0.6338517070]], dtype=torch.float32)
M2 = torch.tensor([[0.2104542553, 0.7936177850, -0.0040720468],
                   [1.9779984951, -2.4285922050, 0.4505937099],
                   [0.0259040371, 0.7827717662, -0.8086757660]], dtype=torch.float32)
M1_inv = torch.linalg.inv(M1)
M2_inv = torch.linalg.inv(M2)

def oklab_to_linear_srgb(lab):
    lms_prime = lab @ M2_inv.T.to(lab.device)
    lms = torch.pow(lms_prime, 3)
    xyz = lms @ M1_inv.T.to(lab.device)
    M_xyz_rgb = torch.tensor([[3.2404542, -1.5371385, -0.4985314],
                              [-0.9692660, 1.8760108, 0.0415560],
                              [0.0556434, -0.2040259, 1.0572252]], 
                              dtype=torch.float32, device=lab.device).T
    return xyz @ M_xyz_rgb

def linear_to_gamma_srgb(rgb_lin):
    mask = rgb_lin > 0.0031308
    rgb_gamma = torch.zeros_like(rgb_lin)
    rgb_gamma[mask] = 1.055 * torch.pow(rgb_lin[mask], 1/2.4) - 0.055
    rgb_gamma[~mask] = 12.92 * rgb_lin[~mask]
    return torch.clamp(rgb_gamma, 0.0, 1.0)

def linear_srgb_to_oklab(rgb):
    M_rgb_xyz = torch.tensor([[0.4124564, 0.3575761, 0.1804375],
                              [0.2126729, 0.7151522, 0.0721750],
                              [0.0193339, 0.1191920, 0.9503041]], 
                              dtype=torch.float32, device=rgb.device).T
    xyz = rgb @ M_rgb_xyz
    lms = xyz @ M1.T.to(rgb.device)
    lms_prime = torch.pow(torch.clamp(lms, min=1e-9), 1/3)
    return lms_prime @ M2.T.to(rgb.device)

# ==========================================
# Part 2: The Anchored Radial Model
# ==========================================
class RadialSplineLoop(nn.Module):
    def __init__(self, num_spokes=24): # Increased spoke count for precision
        super().__init__()
        self.num_spokes = num_spokes
        
        # Center Pivot
        # Raised from 0.60 to 0.65 to lift the purple section off the floor
        self.L_center = 0.65
        
        # Spoke Lengths (Radius)
        self.spoke_lengths = nn.Parameter(torch.ones(num_spokes) * 0.05)
        
        # Lightness Tilt
        self.slope_a = nn.Parameter(torch.tensor(0.0))
        self.slope_b = nn.Parameter(torch.tensor(0.15)) 

    def get_curve_points(self, num_points=360):
        spoke_indices = torch.arange(self.num_spokes, dtype=torch.float32, device=self.spoke_lengths.device)
        spoke_angles = (spoke_indices / self.num_spokes) * 2 * np.pi
        
        lengths = F.softplus(self.spoke_lengths)
        target_angles = torch.linspace(0, 2*np.pi, num_points, device=self.spoke_lengths.device)
        
        # Interpolation (Inverse Distance Weighting for smooth derivatives)
        radius_at_t = torch.zeros_like(target_angles)
        total_weight = torch.zeros_like(target_angles)
        
        # Sharper kernel (factor 20) to allow tighter fits to corners
        for i in range(self.num_spokes):
            diff = torch.abs(target_angles - spoke_angles[i])
            diff = torch.min(diff, 2*np.pi - diff)
            weight = torch.exp(-20 * diff**2) 
            radius_at_t += lengths[i] * weight
            total_weight += weight
            
        radius_at_t = radius_at_t / (total_weight + 1e-8)
        
        a = radius_at_t * torch.cos(target_angles)
        b = radius_at_t * torch.sin(target_angles)
        
        L = self.L_center + (self.slope_a * a) + (self.slope_b * b)
        
        return torch.stack([L, a, b], dim=1)

# ==========================================
# Part 3: Optimization (The Barrier Method)
# ==========================================
def optimize_radial_barrier(iterations=4000, device='cpu'):
    model = RadialSplineLoop(num_spokes=24).to(device)
    # Reduced LR slightly to prevent large jumps
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
    
    print("Optimizing with Numerically Stable Barrier...")
    
    for i in range(iterations):
        optimizer.zero_grad()
        lab_out = model.get_curve_points(num_points=256)
        rgb_lin = oklab_to_linear_srgb(lab_out)
        
        # --- NUMERICALLY STABLE BARRIER ---
        # Instead of exp(x), we use a high-slope Softplus.
        # This mimics the "wall" effect but doesn't overflow to infinity.
        # We treat < 0.0 and > 1.0 as "violations".
        
        # Strength of the wall
        k = 100.0 
        
        # Penalize < 0
        # softplus(-k * x) / k approaches 0 for x > 0, and -x for x < 0
        # This provides a smooth gradient even deep inside the forbidden zone.
        barrier_0 = F.softplus(-k * rgb_lin) / k
        
        # Penalize > 1
        barrier_1 = F.softplus(k * (rgb_lin - 1.0)) / k
        
        # Total Barrier Loss (L2 style to make it stricter)
        gamut_loss = torch.sum((barrier_0 + barrier_1)**2) * 1000.0
        
        # Expansion (Push outward)
        radii = torch.sqrt(lab_out[:,1]**2 + lab_out[:,2]**2)
        expansion_loss = -torch.mean(radii) * 2.0
        
        # Smoothness (Spoke consistency)
        spokes = F.softplus(model.spoke_lengths)
        diffs = spokes - torch.roll(spokes, 1)
        smooth_loss = torch.sum(diffs**2) * 10.0
        
        total_loss = gamut_loss + expansion_loss + smooth_loss
        
        # Add gradient clipping to prevent explosion if it hits a wall too hard
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        if i % 500 == 0:
            # Check safely
            r_min = rgb_lin.min().item()
            r_max = rgb_lin.max().item()
            print(f"Iter {i:04d} | RGB Range: [{r_min:.4f}, {r_max:.4f}]")
            
    return model

# ==========================================
# Part 4: Visualization
# ==========================================
def reparameterize_by_arclength(lab_points_np):
    dists = np.linalg.norm(lab_points_np - np.roll(lab_points_np, 1, axis=0), axis=1)
    dists[0] = np.linalg.norm(lab_points_np[0] - lab_points_np[-1])
    cumulative = np.cumsum(dists)
    total = cumulative[-1]
    t_curr = np.concatenate(([0], cumulative / total))
    pts = np.vstack([lab_points_np[-1:], lab_points_np])
    iL = interp1d(t_curr, pts[:,0], kind='cubic')
    ia = interp1d(t_curr, pts[:,1], kind='cubic')
    ib = interp1d(t_curr, pts[:,2], kind='cubic')
    t_targ = np.linspace(0, 1, len(lab_points_np) + 1)[:-1]
    return np.stack([iL(t_targ), ia(t_targ), ib(t_targ)], axis=1)

def get_srgb_wireframe_oklab(res=15):
    edges = []
    t = torch.linspace(0, 1, res)
    def add_edge(start, end_dim):
        line = start.repeat(res, 1)
        line[:, end_dim] = t
        return line
    # 12 cube edges
    edges.extend([
        add_edge(torch.tensor([0.,0.,0.]), 0), add_edge(torch.tensor([0.,0.,0.]), 1), add_edge(torch.tensor([0.,0.,0.]), 2),
        add_edge(torch.tensor([1.,1.,1.]), 0), add_edge(torch.tensor([1.,1.,1.]), 1), add_edge(torch.tensor([1.,1.,1.]), 2),
        add_edge(torch.tensor([1.,0.,0.]), 1), add_edge(torch.tensor([1.,0.,0.]), 2),
        add_edge(torch.tensor([0.,1.,0.]), 0), add_edge(torch.tensor([0.,1.,0.]), 2),
        add_edge(torch.tensor([0.,0.,1.]), 0), add_edge(torch.tensor([0.,0.,1.]), 1)
    ])
    all_edges = []
    for edge in edges:
        lab_edge = linear_srgb_to_oklab(edge).cpu().numpy()
        all_edges.append(lab_edge)
        all_edges.append(np.array([[None, None, None]]))
    return np.concatenate(all_edges, axis=0)

if __name__ == "__main__":
    import plotly.io as pio
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Optimize
    model = optimize_radial_barrier(device=device)
    
    # Sample
    with torch.no_grad():
        lab_out = model.get_curve_points(num_points=512)
        lab_raw = lab_out.cpu().numpy()
    
    # Balance
    lab_final = reparameterize_by_arclength(lab_raw)
    
    # Render
    lab_tensor = torch.from_numpy(lab_final).float()
    rgb_lin = oklab_to_linear_srgb(lab_tensor)
    rgb_gamma = linear_to_gamma_srgb(rgb_lin).numpy()
    rgb_gamma = np.clip(rgb_gamma, 0, 1) * 255
    colors = [f'rgb({r:.0f},{g:.0f},{b:.0f})' for r,g,b in rgb_gamma]
    
    gamut_lines = get_srgb_wireframe_oklab(res=20)
    
    trace_gamut = go.Scatter3d(
        x=gamut_lines[:, 1], y=gamut_lines[:, 2], z=gamut_lines[:, 0],
        mode='lines',
        line=dict(color='rgba(100, 100, 100, 0.2)', width=2),
        name='sRGB Gamut', hoverinfo='none'
    )
    
    trace_loop = go.Scatter3d(
        x=lab_final[:, 1], y=lab_final[:, 2], z=lab_final[:, 0],
        mode='markers',
        marker=dict(size=6, color=colors, line=dict(width=0)),
        name='Optimized Path',
        hovertemplate='L: %{z:.2f}<br>a: %{x:.2f}<br>b: %{y:.2f}<extra></extra>'
    )
    
    trace_line = go.Scatter3d(
        x=lab_final[:, 1], y=lab_final[:, 2], z=lab_final[:, 0],
        mode='lines',
        line=dict(color='white', width=4),
        opacity=0.5, showlegend=False, hoverinfo='none'
    )

    layout = go.Layout(
        template='plotly_dark',
        title='Strict Barrier Colormap (In-Gamut Guaranteed)',
        scene=dict(
            xaxis_title='a (Green-Red)',
            yaxis_title='b (Blue-Yellow)',
            zaxis_title='L (Lightness)',
            aspectmode='data',
            xaxis=dict(showgrid=True, gridcolor='rgba(255,255,255,0.1)', zeroline=False),
            yaxis=dict(showgrid=True, gridcolor='rgba(255,255,255,0.1)', zeroline=False),
            zaxis=dict(showgrid=True, gridcolor='rgba(255,255,255,0.1)', zeroline=False),
        ),
        margin=dict(l=0, r=0, b=0, t=40),
        height=800
    )
    
    fig = go.Figure(data=[trace_gamut, trace_line, trace_loop], layout=layout)
    
    # Display Logic
    try:
        import google.colab
        pio.renderers.default = "colab"
    except ImportError:
        pio.renderers.default = "notebook_connected"
    fig.show()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from scipy.interpolate import interp1d
import os
from pathlib import Path

# Optional imports for Flow Vis
try:
    import omnirefactor
    from omnirefactor import io
    import fastremap
    OMNIPOSE_AVAILABLE = True
except ImportError:
    OMNIPOSE_AVAILABLE = False
    print("Warning: Omnipose/Cellpose not installed. Flow visualization will be skipped.")

torch.manual_seed(42)
np.random.seed(42)
dtype = torch.float32
device = torch.device("cpu")

# ============================================================
# Part 1: Color Math (User Provided Matrices)
# ============================================================
M1 = torch.tensor([[0.8189330101, 0.3618667424, -0.1288597137],
                   [0.0329845436, 0.9293118715, 0.0361456387],
                   [0.0482003018, 0.2643662703, 0.6338517070]], dtype=dtype, device=device)
M2 = torch.tensor([[0.2104542553, 0.7936177850, -0.0040720468],
                   [1.9779984951, -2.4285922050, 0.4505937099],
                   [0.0259040371, 0.7827717662, -0.8086757660]], dtype=dtype, device=device)
M1_inv = torch.linalg.inv(M1)
M2_inv = torch.linalg.inv(M2)

def oklab_to_linear_srgb(lab):
    lms_prime = lab @ M2_inv.T.to(lab.device)
    lms = torch.pow(lms_prime, 3)
    xyz = lms @ M1_inv.T.to(lab.device)
    # Standard sRGB matrix
    M_xyz_rgb = torch.tensor([[3.2404542, -1.5371385, -0.4985314],
                              [-0.9692660, 1.8760108, 0.0415560],
                              [0.0556434, -0.2040259, 1.0572252]], 
                              dtype=dtype, device=lab.device).T
    return xyz @ M_xyz_rgb

def linear_srgb_to_oklab(rgb):
    M_rgb_xyz = torch.tensor([[0.4124564, 0.3575761, 0.1804375],
                              [0.2126729, 0.7151522, 0.0721750],
                              [0.0193339, 0.1191920, 0.9503041]], 
                              dtype=dtype, device=rgb.device).T
    xyz = rgb @ M_rgb_xyz
    lms = xyz @ M1.T.to(rgb.device)
    lms_prime = torch.pow(torch.clamp(lms, min=1e-9), 1/3)
    return lms_prime @ M2.T.to(rgb.device)

def linear_to_gamma_srgb(rgb_lin):
    mask = rgb_lin > 0.0031308
    rgb_gamma = torch.zeros_like(rgb_lin)
    rgb_gamma[mask] = 1.055 * torch.pow(rgb_lin[mask], 1/2.4) - 0.055
    rgb_gamma[~mask] = 12.92 * rgb_lin[~mask]
    return torch.clamp(rgb_gamma, 0.0, 1.0)

def linear_to_srgb_np(rgb):
    rgb = np.asarray(rgb, dtype=float)
    a = 0.055
    return np.where(rgb <= 0.0031308, 12.92 * rgb, (1 + a) * np.power(np.clip(rgb, 0, None), 1/2.4) - a)

# ============================================================
# Part 2: Vectorized Radial Spline (Speed Fix)
# ============================================================
class RadialSplineLoop(nn.Module):
    def __init__(self, num_spokes=24): 
        super().__init__()
        self.num_spokes = num_spokes
        self.L_center = 0.65
        self.spoke_lengths = nn.Parameter(torch.ones(num_spokes) * 0.05)
        self.slope_a = nn.Parameter(torch.tensor(0.0))
        self.slope_b = nn.Parameter(torch.tensor(0.15)) 

    def get_curve_points(self, num_points=360):
        # Vectorized Spoke Calculation (Much Faster)
        spoke_indices = torch.arange(self.num_spokes, dtype=dtype, device=self.spoke_lengths.device)
        spoke_angles = (spoke_indices / self.num_spokes) * 2 * np.pi # [S]
        lengths = F.softplus(self.spoke_lengths) # [S]
        
        target_angles = torch.linspace(0, 2*np.pi, num_points, device=self.spoke_lengths.device) # [N]
        
        # Broadcast for difference matrix [N, S]
        # diff[i, j] = angle_i - spoke_j
        diff = torch.abs(target_angles.unsqueeze(1) - spoke_angles.unsqueeze(0))
        diff = torch.min(diff, 2*np.pi - diff)
        
        # Gaussian Kernel weights [N, S]
        weights = torch.exp(-20 * diff**2)
        
        # Weighted Average
        # Sum(weights * lengths) / Sum(weights)
        radius_at_t = torch.sum(weights * lengths.unsqueeze(0), dim=1) / (torch.sum(weights, dim=1) + 1e-8)
        
        a = radius_at_t * torch.cos(target_angles)
        b = radius_at_t * torch.sin(target_angles)
        L = self.L_center + (self.slope_a * a) + (self.slope_b * b)
        
        return torch.stack([L, a, b], dim=1)

# ============================================================
# Part 3: Optimization
# ============================================================
def optimize_radial_barrier(iterations=3000, device='cpu'):
    model = RadialSplineLoop(num_spokes=24).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    
    print("Optimizing (Stable Barrier + Vectorized)...")
    
    for i in range(iterations):
        optimizer.zero_grad()
        lab_out = model.get_curve_points(num_points=256)
        rgb_lin = oklab_to_linear_srgb(lab_out)
        
        # Stable Barrier
        k = 100.0 
        barrier_0 = F.softplus(-k * rgb_lin) / k
        barrier_1 = F.softplus(k * (rgb_lin - 1.0)) / k
        gamut_loss = torch.sum((barrier_0 + barrier_1)**2) * 1000.0
        
        radii = torch.sqrt(lab_out[:,1]**2 + lab_out[:,2]**2)
        expansion_loss = -torch.mean(radii) * 2.0
        
        spokes = F.softplus(model.spoke_lengths)
        diffs = spokes - torch.roll(spokes, 1)
        smooth_loss = torch.sum(diffs**2) * 10.0
        
        total_loss = gamut_loss + expansion_loss + smooth_loss
        
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        if i % 1000 == 0:
            r_min = rgb_lin.min().item()
            r_max = rgb_lin.max().item()
            print(f"Iter {i:04d} | RGB: [{r_min:.4f}, {r_max:.4f}]")
            
    return model

# ============================================================
# Part 4: Flow & Disk Visualization Logic
# ============================================================

def reparameterize_by_arclength(lab_points_np):
    dists = np.linalg.norm(lab_points_np - np.roll(lab_points_np, 1, axis=0), axis=1)
    dists[0] = np.linalg.norm(lab_points_np[0] - lab_points_np[-1])
    cumulative = np.cumsum(dists)
    total = cumulative[-1]
    t_curr = np.concatenate(([0], cumulative / total))
    pts = np.vstack([lab_points_np[-1:], lab_points_np])
    iL = interp1d(t_curr, pts[:,0], kind='cubic')
    ia = interp1d(t_curr, pts[:,1], kind='cubic')
    ib = interp1d(t_curr, pts[:,2], kind='cubic')
    t_targ = np.linspace(0, 1, len(lab_points_np) + 1)[:-1]
    return np.stack([iL(t_targ), ia(t_targ), ib(t_targ)], axis=1)

def build_hue_disk_from_ring(rgb_ring_lin, center_lin, size=512):
    n_hues = rgb_ring_lin.shape[0]
    N = size
    y, x = np.mgrid[0:N, 0:N]
    x = (x + 0.5) / N * 2 - 1
    y = (y + 0.5) / N * 2 - 1
    r = np.sqrt(x * x + y * y)
    theta = np.mod(np.arctan2(y, x), 2 * np.pi)

    hue_f = theta / (2 * np.pi) * n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)

    ring0 = rgb_ring_lin[idx0]
    ring1 = rgb_ring_lin[idx1]
    ring_interp = (1 - t[..., None]) * ring0 + t[..., None] * ring1

    r_exp = r[..., None]
    rgb_lin = (1 - r_exp) * center_lin + r_exp * ring_interp
    rgb_lin[r > 1] = 0.0 
    return np.clip(linear_to_srgb_np(rgb_lin), 0, 1)

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    n_hues = rgb_ring_lin.shape[0]
    disk = build_hue_disk_from_ring(rgb_ring_lin, center_lin, size=512)

    u = flows[0].cpu().numpy()
    v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2 * np.pi)
    mag = np.sqrt(u * u + v * v)
    mag /= (mag.max() + 1e-8)

    hue_f = angle / (2 * np.pi) * n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)

    ring0 = rgb_ring_lin[idx0]
    ring1 = rgb_ring_lin[idx1]
    ring_interp = (1 - t[..., None]) * ring0 + t[..., None] * ring1

    r = mag[..., None]
    rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow), 0, 1)

    alpha = mag[..., None]
    white = np.ones_like(flow_rgb)
    black = np.zeros_like(flow_rgb)
    flow_white = alpha * flow_rgb + (1 - alpha) * white
    flow_black = alpha * flow_rgb + (1 - alpha) * black

    return disk, flow_rgb, flow_white, flow_black

# ============================================================
# Part 5: Plotly 3D Helper (Uses YOUR specific Matrices)
# ============================================================
def get_srgb_wireframe_oklab(res=15):
    edges = []
    t = torch.linspace(0, 1, res)
    def add_edge(start, end_dim):
        line = start.repeat(res, 1)
        line[:, end_dim] = t
        return line
    edges.extend([
        add_edge(torch.tensor([0.,0.,0.]), 0), add_edge(torch.tensor([0.,0.,0.]), 1), add_edge(torch.tensor([0.,0.,0.]), 2),
        add_edge(torch.tensor([1.,1.,1.]), 0), add_edge(torch.tensor([1.,1.,1.]), 1), add_edge(torch.tensor([1.,1.,1.]), 2),
        add_edge(torch.tensor([1.,0.,0.]), 1), add_edge(torch.tensor([1.,0.,0.]), 2),
        add_edge(torch.tensor([0.,1.,0.]), 0), add_edge(torch.tensor([0.,1.,0.]), 2),
        add_edge(torch.tensor([0.,0.,1.]), 0), add_edge(torch.tensor([0.,0.,1.]), 1)
    ])
    all_edges = []
    for edge in edges:
        # IMPORTANT: Uses the global linear_srgb_to_oklab (your matrices)
        lab_edge = linear_srgb_to_oklab(edge).cpu().numpy()
        all_edges.append(lab_edge)
        all_edges.append(np.array([[None, None, None]]))
    return np.concatenate(all_edges, axis=0)

def plot_3d(lab_path):
    lab_tensor = torch.from_numpy(lab_path).float()
    rgb_lin = oklab_to_linear_srgb(lab_tensor)
    rgb_gamma = linear_to_gamma_srgb(rgb_lin).numpy()
    rgb_gamma = np.clip(rgb_gamma, 0, 1) * 255
    colors = [f'rgb({r:.0f},{g:.0f},{b:.0f})' for r,g,b in rgb_gamma]
    
    gamut_lines = get_srgb_wireframe_oklab(res=20)
    
    trace_gamut = go.Scatter3d(
        x=gamut_lines[:, 1], y=gamut_lines[:, 2], z=gamut_lines[:, 0],
        mode='lines',
        line=dict(color='rgba(100, 100, 100, 0.2)', width=2),
        name='sRGB Gamut', hoverinfo='none'
    )
    
    trace_loop = go.Scatter3d(
        x=lab_path[:, 1], y=lab_path[:, 2], z=lab_path[:, 0],
        mode='markers',
        marker=dict(size=6, color=colors, line=dict(width=0)),
        name='Optimized Path',
        hovertemplate='L: %{z:.2f}<br>a: %{x:.2f}<br>b: %{y:.2f}<extra></extra>'
    )
    
    layout = go.Layout(
        template='plotly_dark',
        title='Radial Spline Colormap',
        scene=dict(
            xaxis_title='a (Green-Red)',
            yaxis_title='b (Blue-Yellow)',
            zaxis_title='L (Lightness)',
            aspectmode='data'
        ),
        height=800
    )
    
    fig = go.Figure(data=[trace_gamut, trace_loop], layout=layout)
    try:
        import google.colab
        pio.renderers.default = "colab"
    except ImportError:
        pio.renderers.default = "notebook_connected"
    fig.show()

# ============================================================
# Main Execution
# ============================================================
if __name__ == "__main__":
    # 1. Optimize
    model = optimize_radial_barrier(device=device)
    
    # 2. Sample & Balance
    with torch.no_grad():
        lab_out = model.get_curve_points(num_points=512)
        lab_raw = lab_out.cpu().numpy()
    lab_final = reparameterize_by_arclength(lab_raw)
    
    # 3. Show 3D
    plot_3d(lab_final)
    
    # 4. Flow Visualization (If Omnipose available)
    if OMNIPOSE_AVAILABLE:
        print("Generating Flow Visualizations...")
        try:
            # Load Data
            omnidir = Path(omnirefactor.__file__).parent.parent.parent
            basedir = os.path.join(omnidir, "docs", "_static")
            name = "ecoli"
            ext = ".tif"
            image_path = os.path.join(basedir, name + ext)
            mask_path = os.path.join(basedir, name + "_labels" + ext)
            
            if os.path.exists(image_path):
                image = io.imread(image_path)
                masks = io.imread(mask_path)
                slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
                masks = fastremap.renumber(masks[slc])[0]
                flows = omnirefactor.core.masks_to_flows(masks, device="cpu")[4].to("cpu")
                
                # Prepare Ring
                lab_tensor = torch.from_numpy(lab_final).float()
                rgb_lin = oklab_to_linear_srgb(lab_tensor)
                rgb_lin_np = rgb_lin.clamp(0,1).numpy()
                
                center_srgb = np.array([0.5, 0.5, 0.5], dtype=float)
                center_lin = linear_to_srgb_np(center_srgb)
                
                # Generate Plots
                d, f, fw, fb = make_flow_images_for_ring(rgb_lin_np, center_lin, flows)
                
                # Display Grid
                fig, axes = plt.subplots(1, 4, figsize=(16, 4))
                axes[0].imshow(d); axes[0].set_title("Hue Disk"); axes[0].axis("off")
                axes[1].imshow(f); axes[1].set_title("Flow"); axes[1].axis("off")
                axes[2].imshow(fw); axes[2].set_title("Flow (White)"); axes[2].axis("off")
                axes[3].imshow(fb); axes[3].set_title("Flow (Black)"); axes[3].axis("off")
                plt.tight_layout()
                plt.show()
            else:
                print("Omnipose sample data not found.")
        except Exception as e:
            print(f"Error in Flow Vis: {e}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from scipy.interpolate import interp1d
import os
from pathlib import Path

# Check for Omnipose
try:
    import omnirefactor
    from omnirefactor import io
    import fastremap
    OMNIPOSE_AVAILABLE = True
except ImportError:
    OMNIPOSE_AVAILABLE = False

torch.manual_seed(42)
np.random.seed(42)
dtype = torch.float32
device = torch.device("cpu")

# ============================================================
# Part 1: Color Math
# ============================================================
M1 = torch.tensor([[0.8189330101, 0.3618667424, -0.1288597137],
                   [0.0329845436, 0.9293118715, 0.0361456387],
                   [0.0482003018, 0.2643662703, 0.6338517070]], dtype=dtype, device=device)
M2 = torch.tensor([[0.2104542553, 0.7936177850, -0.0040720468],
                   [1.9779984951, -2.4285922050, 0.4505937099],
                   [0.0259040371, 0.7827717662, -0.8086757660]], dtype=dtype, device=device)
M1_inv = torch.linalg.inv(M1)
M2_inv = torch.linalg.inv(M2)

def oklab_to_linear_srgb(lab):
    lms_prime = lab @ M2_inv.T
    lms = torch.pow(lms_prime, 3)
    xyz = lms @ M1_inv.T
    M_xyz_rgb = torch.tensor([[3.2404542, -1.5371385, -0.4985314],
                              [-0.9692660, 1.8760108, 0.0415560],
                              [0.0556434, -0.2040259, 1.0572252]], 
                              dtype=dtype, device=device).T
    return xyz @ M_xyz_rgb

def linear_to_gamma_srgb(rgb_lin):
    mask = rgb_lin > 0.0031308
    rgb_gamma = torch.zeros_like(rgb_lin)
    rgb_gamma[mask] = 1.055 * torch.pow(rgb_lin[mask], 1/2.4) - 0.055
    rgb_gamma[~mask] = 12.92 * rgb_lin[~mask]
    return torch.clamp(rgb_gamma, 0.0, 1.0)

def linear_to_srgb_np(rgb):
    rgb = np.asarray(rgb, dtype=float)
    a = 0.055
    return np.where(rgb <= 0.0031308, 12.92 * rgb, (1 + a) * np.power(np.clip(rgb, 0, None), 1/2.4) - a)

def linear_srgb_to_oklab(rgb):
    M_rgb_xyz = torch.tensor([[0.4124564, 0.3575761, 0.1804375],
                              [0.2126729, 0.7151522, 0.0721750],
                              [0.0193339, 0.1191920, 0.9503041]], 
                              dtype=dtype, device=rgb.device).T
    xyz = rgb @ M_rgb_xyz
    lms = xyz @ M1.T.to(rgb.device)
    lms_prime = torch.pow(torch.clamp(lms, min=1e-9), 1/3)
    return lms_prime @ M2.T.to(rgb.device)

# ============================================================
# Part 2: The Stiff Wire Model (Fourier = Fast)
# ============================================================
class StiffWireLoop(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Lightness: Order 3 (Enough to ride the ridge, not enough to wiggle)
        self.order_L = 3
        self.L_dc = nn.Parameter(torch.tensor(0.65)) 
        self.L_coeffs = nn.Parameter(torch.zeros(self.order_L, 2) * 0.01)
        
        # Chroma: Order 2 (Strict Ellipse topology - prevents cusps)
        self.order_C = 2
        self.C_dc = nn.Parameter(torch.tensor(0.15)) 
        self.C_coeffs = nn.Parameter(torch.zeros(self.order_C, 2) * 0.01)

        # Initial Tilt (Yellow High, Blue Low)
        with torch.no_grad():
             self.L_coeffs[0, 0] = 0.20 

    def forward(self, t):
        L = self.L_dc.expand_as(t)
        C = self.C_dc.expand_as(t)
        
        # Vectorized Fourier Sum (Instant)
        for k in range(1, self.order_L + 1):
            theta = k * t
            L = L + self.L_coeffs[k-1, 0]*torch.sin(theta) + self.L_coeffs[k-1, 1]*torch.cos(theta)
            
        for k in range(1, self.order_C + 1):
            theta = k * t
            C = C + self.C_coeffs[k-1, 0]*torch.sin(theta) + self.C_coeffs[k-1, 1]*torch.cos(theta)
        
        C = torch.abs(C)
        a = C * torch.cos(t)
        b = C * torch.sin(t)
        return torch.stack([L, a, b], dim=1)

# ============================================================
# Part 3: Optimization
# ============================================================
def optimize_final(iterations=4000, device='cpu'):
    model = StiffWireLoop().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    t_grid = torch.linspace(0, 2*np.pi, 360, device=device)
    
    print("Optimizing: Fourier Model + Stable Barrier + Repulsion...")
    
    for i in range(iterations):
        optimizer.zero_grad()
        lab_out = model(t_grid)
        rgb_lin = oklab_to_linear_srgb(lab_out)
        
        # 1. Stable Softplus Barrier (Fixes Out-of-Gamut)
        k = 100.0 
        barrier_0 = F.softplus(-k * rgb_lin) / k
        barrier_1 = F.softplus(k * (rgb_lin - 1.0)) / k
        gamut_loss = torch.sum((barrier_0 + barrier_1)**2) * 2000.0
        
        # 2. Expansion (Maximize Radius)
        C_vals = torch.sqrt(lab_out[:,1]**2 + lab_out[:,2]**2)
        chroma_loss = -torch.mean(C_vals)
        
        # 3. Repulsion Pillar (Fixes Gray Lobes)
        # Force radius > 0.16. Physically prevents collapsing to gray.
        repulsion_loss = torch.sum(F.relu(0.16 - C_vals)**2) * 100.0
        
        # 4. Anti-Mud Floor (Fixes Dark Blue)
        L_vals = lab_out[:, 0]
        floor_loss = torch.sum(F.relu(0.40 - L_vals)**2) * 50.0
        
        # 5. Regularization (Stiffness)
        reg_loss = 0.0
        for k in range(1, model.order_L + 1):
            reg_loss += (k**2) * torch.sum(model.L_coeffs[k-1]**2)
        for k in range(1, model.order_C + 1):
            reg_loss += (k**2) * torch.sum(model.C_coeffs[k-1]**2)
            
        total_loss = gamut_loss + chroma_loss + repulsion_loss + floor_loss + reg_loss * 2.0
        
        total_loss.backward()
        # Clip gradients to allow safe barrier navigation
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        if i % 1000 == 0:
            r_min = rgb_lin.min().item()
            r_max = rgb_lin.max().item()
            print(f"Iter {i:04d} | RGB: [{r_min:.4f}, {r_max:.4f}] | Min C: {C_vals.min().item():.3f}")
            
    return model

# ============================================================
# Part 4: Visualization Helpers
# ============================================================

def reparameterize_by_arclength(lab_points_np):
    dists = np.linalg.norm(lab_points_np - np.roll(lab_points_np, 1, axis=0), axis=1)
    dists[0] = np.linalg.norm(lab_points_np[0] - lab_points_np[-1])
    cumulative = np.cumsum(dists)
    total = cumulative[-1]
    t_curr = np.concatenate(([0], cumulative / total))
    pts = np.vstack([lab_points_np[-1:], lab_points_np])
    iL = interp1d(t_curr, pts[:,0], kind='cubic')
    ia = interp1d(t_curr, pts[:,1], kind='cubic')
    ib = interp1d(t_curr, pts[:,2], kind='cubic')
    t_targ = np.linspace(0, 1, len(lab_points_np) + 1)[:-1]
    return np.stack([iL(t_targ), ia(t_targ), ib(t_targ)], axis=1)

def build_hue_disk_from_ring(rgb_ring_lin, center_lin, size=512):
    n_hues = rgb_ring_lin.shape[0]
    N = size
    y, x = np.mgrid[0:N, 0:N]
    x = (x + 0.5) / N * 2 - 1
    y = (y + 0.5) / N * 2 - 1
    r = np.sqrt(x * x + y * y)
    theta = np.mod(np.arctan2(y, x), 2 * np.pi)
    hue_f = theta / (2 * np.pi) * n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)
    ring0 = rgb_ring_lin[idx0]; ring1 = rgb_ring_lin[idx1]
    ring_interp = (1 - t[..., None]) * ring0 + t[..., None] * ring1
    r_exp = r[..., None]
    rgb_lin = (1 - r_exp) * center_lin + r_exp * ring_interp
    rgb_lin[r > 1] = 0.0 
    return np.clip(linear_to_srgb_np(rgb_lin), 0, 1)

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    n_hues = rgb_ring_lin.shape[0]
    disk = build_hue_disk_from_ring(rgb_ring_lin, center_lin, size=512)
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2 * np.pi)
    mag = np.sqrt(u * u + v * v); mag /= (mag.max() + 1e-8)
    hue_f = angle / (2 * np.pi) * n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)
    ring0 = rgb_ring_lin[idx0]; ring1 = rgb_ring_lin[idx1]
    ring_interp = (1 - t[..., None]) * ring0 + t[..., None] * ring1
    r = mag[..., None]
    rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow), 0, 1)
    alpha = mag[..., None]
    white = np.ones_like(flow_rgb); black = np.zeros_like(flow_rgb)
    flow_white = alpha * flow_rgb + (1 - alpha) * white
    flow_black = alpha * flow_rgb + (1 - alpha) * black
    return disk, flow_rgb, flow_white, flow_black

def get_srgb_wireframe_oklab(res=15):
    t = torch.linspace(0, 1, res, device=device)
    edges = []
    def add(s, d):
        l = s.repeat(res, 1); l[:, d] = t; return l
    edges.extend([add(torch.tensor([0.,0.,0.]), i) for i in range(3)])
    edges.extend([add(torch.tensor([1.,1.,1.]), i) for i in range(3)])
    edges.extend([
        add(torch.tensor([1.,0.,0.]), 1), add(torch.tensor([1.,0.,0.]), 2),
        add(torch.tensor([0.,1.,0.]), 0), add(torch.tensor([0.,1.,0.]), 2),
        add(torch.tensor([0.,0.,1.]), 0), add(torch.tensor([0.,0.,1.]), 1)
    ])
    all_edges = []
    for e in edges:
        all_edges.append(linear_srgb_to_oklab(e).cpu().numpy())
        all_edges.append(np.array([[None, None, None]]))
    return np.concatenate(all_edges, axis=0)

def plot_3d(lab_path):
    lab_t = torch.from_numpy(lab_path).float().to(device)
    rgb_lin = oklab_to_linear_srgb(lab_t)
    rgb_gamma = linear_to_gamma_srgb(rgb_lin).cpu().numpy()
    rgb_gamma = np.clip(rgb_gamma, 0, 1) * 255
    colors = [f'rgb({r:.0f},{g:.0f},{b:.0f})' for r,g,b in rgb_gamma]
    
    gamut = get_srgb_wireframe_oklab(res=20)
    
    fig = go.Figure(data=[
        go.Scatter3d(x=gamut[:,1], y=gamut[:,2], z=gamut[:,0], mode='lines', 
                     line=dict(color='rgba(150,150,150,0.3)', width=3), name='sRGB', hoverinfo='none'),
        go.Scatter3d(x=lab_path[:,1], y=lab_path[:,2], z=lab_path[:,0], mode='markers', 
                     marker=dict(size=6, color=colors), name='Optimal',
                     hovertemplate='L: %{z:.2f}<br>a: %{x:.2f}<br>b: %{y:.2f}<extra></extra>')
    ])
    fig.update_layout(template='plotly_dark', title='Fast & Smooth Colormap', 
                      scene=dict(xaxis_title='a', yaxis_title='b', zaxis_title='L', aspectmode='data'),
                      margin=dict(l=0,r=0,b=0,t=40), height=700)
    
    try:
        import google.colab; pio.renderers.default = "colab"
    except:
        pio.renderers.default = "notebook_connected"
    fig.show()

if __name__ == "__main__":
    # 1. Fast Optimize
    model = optimize_final(device=device)
    
    # 2. High Res Sample
    with torch.no_grad():
        t = torch.linspace(0, 2*np.pi, 512, device=device)
        lab_raw = model(t).cpu().numpy()
    lab_final = reparameterize_by_arclength(lab_raw)
    
    # 3. 3D Plot
    plot_3d(lab_final)
    
    # 4. Flow Vis
    if OMNIPOSE_AVAILABLE:
        try:
            omnidir = Path(omnirefactor.__file__).parent.parent.parent
            basedir = os.path.join(omnidir, "docs", "_static")
            img_p = os.path.join(basedir, "ecoli.tif")
            msk_p = os.path.join(basedir, "ecoli_labels.tif")
            
            if os.path.exists(img_p):
                masks = io.imread(msk_p)
                slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
                masks = fastremap.renumber(masks[slc])[0]
                flows = omnirefactor.core.masks_to_flows(masks, device="cpu")[4].to("cpu")
                
                lab_t = torch.from_numpy(lab_final).float().to(device)
                rgb_lin = oklab_to_linear_srgb(lab_t).cpu().numpy().clip(0,1)
                center = linear_to_srgb_np(np.array([0.5,0.5,0.5]))
                
                d, f, fw, fb = make_flow_images_for_ring(rgb_lin, center, flows)
                
                fig, ax = plt.subplots(1, 4, figsize=(16, 4))
                ax[0].imshow(d); ax[0].axis('off'); ax[0].set_title("Hue Disk")
                ax[1].imshow(f); ax[1].axis('off'); ax[1].set_title("Flow")
                ax[2].imshow(fw); ax[2].axis('off'); ax[2].set_title("White BG")
                ax[3].imshow(fb); ax[3].axis('off'); ax[3].set_title("Black BG")
                plt.tight_layout()
                plt.show()
        except Exception as e:
            print(f"Flow vis error: {e}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from scipy.interpolate import interp1d
import os
from pathlib import Path

# Optional imports for Flow Vis
try:
    import omnirefactor
    from omnirefactor import io
    import fastremap
    OMNIPOSE_AVAILABLE = True
except ImportError:
    OMNIPOSE_AVAILABLE = False

torch.manual_seed(42)
np.random.seed(42)
dtype = torch.float32

# ============================================================
# 0. DEVICE CONFIGURATION
# ============================================================
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"🚀 Running on GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🚀 Running on Apple MPS (Metal)")
else:
    device = torch.device("cpu")
    print("⚠️ Running on CPU (GPU not found)")

# ============================================================
# 1. Color Math (Vectorized Module)
# ============================================================

# Define matrices
M1 = torch.tensor([[0.8189330101, 0.3618667424, -0.1288597137],
                   [0.0329845436, 0.9293118715, 0.0361456387],
                   [0.0482003018, 0.2643662703, 0.6338517070]], dtype=dtype)
M2 = torch.tensor([[0.2104542553, 0.7936177850, -0.0040720468],
                   [1.9779984951, -2.4285922050, 0.4505937099],
                   [0.0259040371, 0.7827717662, -0.8086757660]], dtype=dtype)
M1_inv = torch.linalg.inv(M1)
M2_inv = torch.linalg.inv(M2)

class ColorConverter(nn.Module):
    def __init__(self):
        super().__init__()
        # Register as buffers so they move to GPU automatically with the model
        self.register_buffer('m1_inv', M1_inv)
        self.register_buffer('m2_inv', M2_inv)
        self.register_buffer('m1', M1)
        self.register_buffer('m2', M2)

    def oklab_to_linear_srgb(self, lab):
        # Vectorized conversion
        lms_prime = lab @ self.m2_inv.T
        lms = torch.pow(lms_prime, 3)
        xyz = lms @ self.m1_inv.T
        
        # XYZ to Linear RGB (Standard D65)
        r = 3.2404542*xyz[:,0] - 1.5371385*xyz[:,1] - 0.4985314*xyz[:,2]
        g = -0.9692660*xyz[:,0] + 1.8760108*xyz[:,1] + 0.0415560*xyz[:,2]
        b = 0.0556434*xyz[:,0] - 0.2040259*xyz[:,1] + 1.0572252*xyz[:,2]
        
        return torch.stack([r, g, b], dim=1)

    def linear_srgb_to_oklab(self, rgb):
        r = rgb[:,0]; g = rgb[:,1]; b = rgb[:,2]
        x = 0.4124564*r + 0.3575761*g + 0.1804375*b
        y = 0.2126729*r + 0.7151522*g + 0.0721750*b
        z = 0.0193339*r + 0.1191920*g + 0.9503041*b
        xyz = torch.stack([x,y,z], dim=1)
        
        lms = xyz @ self.m1.T
        lms = torch.clamp(lms, min=1e-8) 
        lms_prime = torch.pow(lms, 1.0/3.0)
        return lms_prime @ self.m2.T

# Initialize converter
converter = ColorConverter().to(device)

def linear_to_srgb_np(rgb):
    rgb = np.asarray(rgb, dtype=float)
    a = 0.055
    return np.where(rgb <= 0.0031308, 12.92 * rgb, (1 + a) * np.power(np.clip(rgb, 0, None), 1/2.4) - a)

# ============================================================
# 2. The Optimized Model (Stiff Wire)
# ============================================================
class StiffWireLoop(nn.Module):
    def __init__(self):
        super().__init__()
        # Lightness Order 3 (Flexible but controlled)
        self.order_L = 3
        self.L_dc = nn.Parameter(torch.tensor(0.65, device=device)) 
        self.L_coeffs = nn.Parameter(torch.zeros((self.order_L, 2), device=device) * 0.01)
        
        # Chroma Order 2 (Elliptical topology)
        self.order_C = 2
        self.C_dc = nn.Parameter(torch.tensor(0.15, device=device)) 
        self.C_coeffs = nn.Parameter(torch.zeros((self.order_C, 2), device=device) * 0.01)

        # Initialize Tilt
        with torch.no_grad():
             self.L_coeffs[0, 0] = 0.20 

    def forward(self, t):
        L = self.L_dc.expand_as(t)
        C = self.C_dc.expand_as(t)
        
        sin_t = torch.sin(t)
        cos_t = torch.cos(t)
        
        # Unrolled Fourier Sum for speed
        # L
        L = L + self.L_coeffs[0,0]*sin_t + self.L_coeffs[0,1]*cos_t
        L = L + self.L_coeffs[1,0]*torch.sin(2*t) + self.L_coeffs[1,1]*torch.cos(2*t)
        L = L + self.L_coeffs[2,0]*torch.sin(3*t) + self.L_coeffs[2,1]*torch.cos(3*t)
        
        # C
        C = C + self.C_coeffs[0,0]*sin_t + self.C_coeffs[0,1]*cos_t
        C = C + self.C_coeffs[1,0]*torch.sin(2*t) + self.C_coeffs[1,1]*torch.cos(2*t)
        
        C = torch.abs(C)
        a = C * cos_t
        b = C * sin_t
        return torch.stack([L, a, b], dim=1)

# ============================================================
# 3. Optimization Loop
# ============================================================
def optimize_final_fast():
    model = StiffWireLoop().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
    
    # 256 points is enough for optimization
    t_grid = torch.linspace(0, 2*np.pi, 256, device=device)
    
    print("⚡ Starting Optimization (1500 iters)...")
    
    for i in range(1500):
        optimizer.zero_grad(set_to_none=True)
        
        lab_out = model(t_grid)
        rgb_lin = converter.oklab_to_linear_srgb(lab_out)
        
        # 1. Stable Barrier (Prevents Out-of-Gamut)
        k = 100.0 
        barrier_0 = F.softplus(-k * rgb_lin) / k
        barrier_1 = F.softplus(k * (rgb_lin - 1.0)) / k
        gamut_loss = torch.sum((barrier_0 + barrier_1)**2) * 2000.0
        
        # 2. Expansion
        radii = torch.norm(lab_out[:, 1:], dim=1)
        chroma_loss = -torch.mean(radii)
        
        # 3. Repulsion (Pillar) - Prevents Gray Lobes
        repulsion_loss = torch.sum(F.relu(0.16 - radii)**2) * 100.0
        
        # 4. Floor - Prevents Dark Blue Dive
        L_vals = lab_out[:, 0]
        floor_loss = torch.sum(F.relu(0.40 - L_vals)**2) * 50.0
        
        # 5. Stiffness
        reg_loss = torch.sum(model.L_coeffs**2) + torch.sum(model.C_coeffs**2)
        
        total_loss = gamut_loss + chroma_loss + repulsion_loss + floor_loss + reg_loss * 2.0
        
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        if i % 500 == 0:
            print(f"Iter {i} | Loss: {total_loss.item():.4f}")
            
    return model

# ============================================================
# 4. Visualization
# ============================================================

def reparameterize_by_arclength(lab_points_np):
    dists = np.linalg.norm(lab_points_np - np.roll(lab_points_np, 1, axis=0), axis=1)
    dists[0] = np.linalg.norm(lab_points_np[0] - lab_points_np[-1])
    cumulative = np.cumsum(dists)
    total = cumulative[-1]
    t_curr = np.concatenate(([0], cumulative / total))
    pts = np.vstack([lab_points_np[-1:], lab_points_np])
    iL = interp1d(t_curr, pts[:,0], kind='cubic')
    ia = interp1d(t_curr, pts[:,1], kind='cubic')
    ib = interp1d(t_curr, pts[:,2], kind='cubic')
    t_targ = np.linspace(0, 1, len(lab_points_np) + 1)[:-1]
    return np.stack([iL(t_targ), ia(t_targ), ib(t_targ)], axis=1)

def get_srgb_wireframe():
    res = 20
    t = torch.linspace(0, 1, res, device=device)
    edges = []
    def add(s, d):
        l = s.repeat(res, 1); l[:, d] = t; return l
    
    c = torch.tensor([[0,0,0], [1,0,0], [0,1,0], [0,0,1], [1,1,0], [1,0,1], [0,1,1], [1,1,1]], dtype=dtype, device=device)
    edges += [add(c[0], 0), add(c[0], 1), add(c[0], 2)]
    edges += [add(c[7], 0), add(c[7], 1), add(c[7], 2)]
    edges += [add(c[1], 1), add(c[1], 2), add(c[2], 0), add(c[2], 2), add(c[3], 0), add(c[3], 1)]
    
    all_edges_rgb = torch.cat(edges, dim=0)
    all_edges_lab = converter.linear_srgb_to_oklab(all_edges_rgb)
    
    lab_np = all_edges_lab.cpu().numpy()
    out = []
    for i in range(0, len(lab_np), res):
        out.append(lab_np[i:i+res])
        out.append(np.array([[None, None, None]]))
    return np.concatenate(out, axis=0)

def plot_3d(lab_path):
    lab_t = torch.from_numpy(lab_path).float().to(device)
    rgb_lin = converter.oklab_to_linear_srgb(lab_t)
    rgb_gamma = torch.where(rgb_lin <= 0.0031308, 12.92 * rgb_lin, 1.055 * torch.pow(rgb_lin.clamp(min=1e-8), 1/2.4) - 0.055)
    rgb_np = np.clip(rgb_gamma.cpu().numpy(), 0, 1) * 255
    colors = [f'rgb({r:.0f},{g:.0f},{b:.0f})' for r,g,b in rgb_np]
    
    gamut = get_srgb_wireframe()
    
    fig = go.Figure(data=[
        go.Scatter3d(x=gamut[:,1], y=gamut[:,2], z=gamut[:,0], mode='lines', 
                     line=dict(color='rgba(150,150,150,0.3)', width=2), name='sRGB', hoverinfo='none'),
        go.Scatter3d(x=lab_path[:,1], y=lab_path[:,2], z=lab_path[:,0], mode='markers', 
                     marker=dict(size=6, color=colors), name='Optimal',
                     hovertemplate='L: %{z:.2f}<br>a: %{x:.2f}<br>b: %{y:.2f}<extra></extra>')
    ])
    fig.update_layout(template='plotly_dark', title='Fast & Smooth Colormap', 
                      scene=dict(xaxis_title='a', yaxis_title='b', zaxis_title='L', aspectmode='data'),
                      margin=dict(l=0,r=0,b=0,t=40), height=700)
    
    try:
        import google.colab; pio.renderers.default = "colab"
    except:
        pio.renderers.default = "notebook_connected"
    fig.show()

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    n_hues = rgb_ring_lin.shape[0]
    N=512
    y, x = np.mgrid[0:N, 0:N]
    x = (x + 0.5) / N * 2 - 1; y = (y + 0.5) / N * 2 - 1
    r_grid = np.sqrt(x*x + y*y); theta = np.mod(np.arctan2(y, x), 2*np.pi)
    
    def apply_cmap(angle_norm, mag_norm):
        hue_f = angle_norm * n_hues
        idx0 = np.floor(hue_f).astype(int) % n_hues
        idx1 = (idx0 + 1) % n_hues
        t = hue_f - np.floor(hue_f)
        ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
        r = mag_norm[..., None]
        rgb = (1 - r) * center_lin + r * ring_interp
        return np.clip(linear_to_srgb_np(rgb), 0, 1)

    disk = apply_cmap(theta / (2*np.pi), np.clip(r_grid, 0, 1))
    
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi) / (2*np.pi)
    mag = np.sqrt(u*u + v*v); mag /= (mag.max() + 1e-8)
    
    flow_rgb = apply_cmap(angle, mag)
    alpha = mag[..., None]
    flow_white = alpha * flow_rgb + (1 - alpha)
    flow_black = alpha * flow_rgb
    
    return disk, flow_rgb, flow_white, flow_black

if __name__ == "__main__":
    # 1. Fast Optimize
    model = optimize_final_fast()
    
    # 2. Sample
    with torch.no_grad():
        t = torch.linspace(0, 2*np.pi, 512, device=device)
        lab_out = model(t)
        lab_raw = lab_out.cpu().numpy()
        
    lab_final = reparameterize_by_arclength(lab_raw)
    plot_3d(lab_final)
    
    # 3. Omnipose Vis
    if OMNIPOSE_AVAILABLE:
        try:
            omnidir = Path(omnirefactor.__file__).parent.parent.parent
            basedir = os.path.join(omnidir, "docs", "_static")
            img_p = os.path.join(basedir, "ecoli.tif")
            msk_p = os.path.join(basedir, "ecoli_labels.tif")
            
            if os.path.exists(img_p):
                masks = io.imread(msk_p)
                slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
                masks = fastremap.renumber(masks[slc])[0]
                flows = omnirefactor.core.masks_to_flows(masks, device="cpu")[4].to("cpu")
                
                lab_t = torch.from_numpy(lab_final).float().to(device)
                rgb_lin = converter.oklab_to_linear_srgb(lab_t).cpu().numpy().clip(0,1)
                center = linear_to_srgb_np(np.array([0.5,0.5,0.5]))
                
                d, f, fw, fb = make_flow_images_for_ring(rgb_lin, center, flows)
                
                fig, ax = plt.subplots(1, 4, figsize=(16, 4))
                ax[0].imshow(d); ax[0].axis('off'); ax[0].set_title("Hue Disk")
                ax[1].imshow(f); ax[1].axis('off'); ax[1].set_title("Flow")
                ax[2].imshow(fw); ax[2].axis('off'); ax[2].set_title("White BG")
                ax[3].imshow(fb); ax[3].axis('off'); ax[3].set_title("Black BG")
                plt.tight_layout()
                plt.show()
        except Exception as e:
            print(f"Vis error: {e}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from scipy.interpolate import interp1d
import os
from pathlib import Path

# Optional imports for Flow Vis
try:
    import omnirefactor
    from omnirefactor import io
    import fastremap
    OMNIPOSE_AVAILABLE = True
except ImportError:
    OMNIPOSE_AVAILABLE = False

torch.manual_seed(42)
np.random.seed(42)
dtype = torch.float32

# ============================================================
# 0. DEVICE CONFIGURATION
# ============================================================
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"🚀 Running on GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🚀 Running on Apple MPS (Metal)")
else:
    device = torch.device("cpu")
    print("⚠️ Running on CPU (GPU not found)")

# ============================================================
# 1. Color Math (Vectorized)
# ============================================================
M1 = torch.tensor([[0.8189330101, 0.3618667424, -0.1288597137],
                   [0.0329845436, 0.9293118715, 0.0361456387],
                   [0.0482003018, 0.2643662703, 0.6338517070]], dtype=dtype)
M2 = torch.tensor([[0.2104542553, 0.7936177850, -0.0040720468],
                   [1.9779984951, -2.4285922050, 0.4505937099],
                   [0.0259040371, 0.7827717662, -0.8086757660]], dtype=dtype)
M1_inv = torch.linalg.inv(M1)
M2_inv = torch.linalg.inv(M2)

class ColorConverter(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer('m1_inv', M1_inv)
        self.register_buffer('m2_inv', M2_inv)
        self.register_buffer('m1', M1)
        self.register_buffer('m2', M2)

    def oklab_to_linear_srgb(self, lab):
        lms_prime = lab @ self.m2_inv.T
        lms = torch.pow(lms_prime, 3)
        xyz = lms @ self.m1_inv.T
        r = 3.2404542*xyz[:,0] - 1.5371385*xyz[:,1] - 0.4985314*xyz[:,2]
        g = -0.9692660*xyz[:,0] + 1.8760108*xyz[:,1] + 0.0415560*xyz[:,2]
        b = 0.0556434*xyz[:,0] - 0.2040259*xyz[:,1] + 1.0572252*xyz[:,2]
        return torch.stack([r, g, b], dim=1)

    def linear_srgb_to_oklab(self, rgb):
        r = rgb[:,0]; g = rgb[:,1]; b = rgb[:,2]
        x = 0.4124564*r + 0.3575761*g + 0.1804375*b
        y = 0.2126729*r + 0.7151522*g + 0.0721750*b
        z = 0.0193339*r + 0.1191920*g + 0.9503041*b
        xyz = torch.stack([x,y,z], dim=1)
        lms = xyz @ self.m1.T
        lms = torch.clamp(lms, min=1e-8) 
        lms_prime = torch.pow(lms, 1.0/3.0)
        return lms_prime @ self.m2.T

converter = ColorConverter().to(device)

def linear_to_srgb_np(rgb):
    rgb = np.asarray(rgb, dtype=float)
    a = 0.055
    return np.where(rgb <= 0.0031308, 12.92 * rgb, (1 + a) * np.power(np.clip(rgb, 0, None), 1/2.4) - a)

def srgb_to_linear_np(rgb):
    rgb = np.asarray(rgb, dtype=float)
    a = 0.055
    return np.where(rgb <= 0.04045, rgb / 12.92, ((rgb + a) / (1 + a))**2.4)

def linear_to_gamma_srgb(rgb_lin):
    mask = rgb_lin > 0.0031308
    rgb_gamma = torch.zeros_like(rgb_lin)
    rgb_gamma[mask] = 1.055 * torch.pow(rgb_lin[mask], 1/2.4) - 0.055
    rgb_gamma[~mask] = 12.92 * rgb_lin[~mask]
    return torch.clamp(rgb_gamma, 0.0, 1.0)

# ============================================================
# 2. The Planar Tilt Model
# ============================================================
class PlanarTiltLoop(nn.Module):
    def __init__(self):
        super().__init__()
        self.order_C = 2
        self.C_dc = nn.Parameter(torch.tensor(0.15, device=device)) 
        self.C_coeffs = nn.Parameter(torch.zeros((self.order_C, 2), device=device) * 0.01)
        # L = Bias + Slope_A * a + Slope_B * b
        self.L_bias = nn.Parameter(torch.tensor(0.65, device=device))
        self.L_slope_a = nn.Parameter(torch.tensor(0.0, device=device))
        self.L_slope_b = nn.Parameter(torch.tensor(0.80, device=device)) 

    def forward(self, t):
        C = self.C_dc.expand_as(t)
        sin_t = torch.sin(t); cos_t = torch.cos(t)
        C = C + self.C_coeffs[0,0]*sin_t + self.C_coeffs[0,1]*cos_t
        C = C + self.C_coeffs[1,0]*torch.sin(2*t) + self.C_coeffs[1,1]*torch.cos(2*t)
        C = torch.abs(C)
        a = C * cos_t
        b = C * sin_t
        L = self.L_bias + (self.L_slope_a * a) + (self.L_slope_b * b)
        return torch.stack([L, a, b], dim=1)

# ============================================================
# 3. Optimization Loop
# ============================================================
def optimize_planar_tilt():
    model = PlanarTiltLoop().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    t_grid = torch.linspace(0, 2*np.pi, 256, device=device)
    
    print("⚡ Optimizing Planar Tilt Loop...")
    for i in range(1500):
        optimizer.zero_grad(set_to_none=True)
        lab_out = model(t_grid)
        rgb_lin = converter.oklab_to_linear_srgb(lab_out)
        
        k = 100.0 
        barrier_0 = F.softplus(-k * rgb_lin) / k
        barrier_1 = F.softplus(k * (rgb_lin - 1.0)) / k
        gamut_loss = torch.sum((barrier_0 + barrier_1)**2) * 2000.0
        
        radii = torch.norm(lab_out[:, 1:], dim=1)
        chroma_loss = -torch.mean(radii)
        
        repulsion_loss = torch.sum(F.relu(0.16 - radii)**2) * 100.0
        center_drift = torch.sum(torch.mean(lab_out[:, 1:], dim=0)**2) * 50.0
        reg_loss = torch.sum(model.C_coeffs**2)
        
        total_loss = gamut_loss + chroma_loss + repulsion_loss + center_drift + reg_loss * 2.0
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
    return model

# ============================================================
# 4. Visualization Helpers
# ============================================================

def reparameterize_by_arclength(lab_points_np):
    dists = np.linalg.norm(lab_points_np - np.roll(lab_points_np, 1, axis=0), axis=1)
    dists[0] = np.linalg.norm(lab_points_np[0] - lab_points_np[-1])
    cumulative = np.cumsum(dists)
    total = cumulative[-1]
    t_curr = np.concatenate(([0], cumulative / total))
    pts = np.vstack([lab_points_np[-1:], lab_points_np])
    iL = interp1d(t_curr, pts[:,0], kind='cubic')
    ia = interp1d(t_curr, pts[:,1], kind='cubic')
    ib = interp1d(t_curr, pts[:,2], kind='cubic')
    t_targ = np.linspace(0, 1, len(lab_points_np) + 1)[:-1]
    return np.stack([iL(t_targ), ia(t_targ), ib(t_targ)], axis=1)

def align_colormap_to_red(lab_path):
    lab_tensor = torch.from_numpy(lab_path).float().to(device)
    rgb_lin = converter.oklab_to_linear_srgb(lab_tensor)
    rgb_srgb = linear_to_gamma_srgb(rgb_lin).cpu().numpy()
    target = np.array([1.0, 0.0, 0.0])
    dists = np.linalg.norm(rgb_srgb - target, axis=1)
    best_idx = np.argmin(dists)
    return np.roll(lab_path, -best_idx, axis=0)

def get_srgb_wireframe():
    res = 20
    t = torch.linspace(0, 1, res, device=device)
    edges = []
    def add(s, d): l = s.repeat(res, 1); l[:, d] = t; return l
    c = torch.tensor([[0,0,0], [1,0,0], [0,1,0], [0,0,1], [1,1,0], [1,0,1], [0,1,1], [1,1,1]], dtype=dtype, device=device)
    edges += [add(c[0], 0), add(c[0], 1), add(c[0], 2), add(c[7], 0), add(c[7], 1), add(c[7], 2)]
    edges += [add(c[1], 1), add(c[1], 2), add(c[2], 0), add(c[2], 2), add(c[3], 0), add(c[3], 1)]
    all_edges_rgb = torch.cat(edges, dim=0)
    all_edges_lab = converter.linear_srgb_to_oklab(all_edges_rgb)
    lab_np = all_edges_lab.cpu().numpy()
    out = []
    for i in range(0, len(lab_np), res):
        out.append(lab_np[i:i+res])
        out.append(np.array([[None, None, None]]))
    return np.concatenate(out, axis=0)

def plot_3d(lab_path):
    lab_t = torch.from_numpy(lab_path).float().to(device)
    rgb_lin = converter.oklab_to_linear_srgb(lab_t)
    rgb_gamma = linear_to_gamma_srgb(rgb_lin).cpu().numpy()
    rgb_gamma = np.clip(rgb_gamma, 0, 1) * 255
    colors = [f'rgb({r:.0f},{g:.0f},{b:.0f})' for r,g,b in rgb_gamma]
    
    gamut = get_srgb_wireframe()
    
    fig = go.Figure(data=[
        go.Scatter3d(x=gamut[:,1], y=gamut[:,2], z=gamut[:,0], mode='lines', 
                     line=dict(color='rgba(150,150,150,0.3)', width=2), name='sRGB', hoverinfo='none'),
        go.Scatter3d(x=lab_path[:,1], y=lab_path[:,2], z=lab_path[:,0], mode='markers', 
                     marker=dict(size=6, color=colors), name='Optimal',
                     hovertemplate='L: %{z:.2f}<br>a: %{x:.2f}<br>b: %{y:.2f}<extra></extra>')
    ])
    fig.update_layout(template='plotly_dark', title='Planar-Tilt Colormap', 
                      scene=dict(xaxis_title='a', yaxis_title='b', zaxis_title='L', aspectmode='data'),
                      margin=dict(l=0,r=0,b=0,t=40), height=700)
    try:
        import google.colab; pio.renderers.default = "colab"
    except:
        pio.renderers.default = "notebook_connected"
    fig.show()

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    n_hues = rgb_ring_lin.shape[0]
    N=512
    y, x = np.mgrid[0:N, 0:N]
    x = (x + 0.5) / N * 2 - 1; y = (y + 0.5) / N * 2 - 1
    r_grid = np.sqrt(x*x + y*y); theta = np.mod(np.arctan2(y, x), 2*np.pi)
    
    def apply_cmap(angle_norm, mag_norm):
        hue_f = angle_norm * n_hues
        idx0 = np.floor(hue_f).astype(int) % n_hues
        idx1 = (idx0 + 1) % n_hues
        t = hue_f - np.floor(hue_f)
        ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
        r = mag_norm[..., None]
        rgb = (1 - r) * center_lin + r * ring_interp
        return np.clip(linear_to_srgb_np(rgb), 0, 1)

    disk = apply_cmap(theta / (2*np.pi), np.clip(r_grid, 0, 1))
    
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi) / (2*np.pi)
    mag = np.sqrt(u*u + v*v); mag /= (mag.max() + 1e-8)
    
    flow_rgb = apply_cmap(angle, mag)
    alpha = mag[..., None]
    flow_white = alpha * flow_rgb + (1 - alpha)
    flow_black = alpha * flow_rgb
    
    return disk, flow_rgb, flow_white, flow_black

if __name__ == "__main__":
    # 1. Optimize
    model = optimize_planar_tilt()
    
    # 2. Sample & Align
    with torch.no_grad():
        t = torch.linspace(0, 2*np.pi, 512, device=device)
        lab_out = model(t)
        lab_raw = lab_out.cpu().numpy()
        
    lab_balanced = reparameterize_by_arclength(lab_raw)
    lab_final = align_colormap_to_red(lab_balanced)
    
    plot_3d(lab_final)
    
    # 3. Omnipose Vis with Rotations
    if OMNIPOSE_AVAILABLE:
        try:
            omnidir = Path(omnirefactor.__file__).parent.parent.parent
            basedir = os.path.join(omnidir, "docs", "_static")
            img_p = os.path.join(basedir, "ecoli.tif")
            msk_p = os.path.join(basedir, "ecoli_labels.tif")
            
            if os.path.exists(img_p):
                masks = io.imread(msk_p)
                slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
                masks = fastremap.renumber(masks[slc])[0]
                flows = omnirefactor.core.masks_to_flows(masks, device="cpu")[4].to("cpu")
                
                # Base Map (Linear RGB)
                lab_t = torch.from_numpy(lab_final).float().to(device)
                rgb_lin_base = converter.oklab_to_linear_srgb(lab_t).cpu().numpy().clip(0,1)
                
                # Create Rotated Maps
                N_HUES = len(rgb_lin_base)
                rgb_lin_rot13 = np.roll(rgb_lin_base, int(N_HUES/3), axis=0)
                rgb_lin_rot14 = np.roll(rgb_lin_base, int(N_HUES/4), axis=0)
                
                center_lin = srgb_to_linear_np(np.array([0.5, 0.5, 0.5]))
                
                maps_to_plot = [
                    (rgb_lin_base, "Base (Red=0°)"),
                    (rgb_lin_rot13, "Rotated 1/3 Turn"),
                    (rgb_lin_rot14, "Rotated 1/4 Turn")
                ]
                
                # Plot 3 rows
                fig, axes = plt.subplots(3, 4, figsize=(16, 12))
                
                for i, (cmap_lin, title) in enumerate(maps_to_plot):
                    d, f, fw, fb = make_flow_images_for_ring(cmap_lin, center_lin, flows)
                    
                    axes[i,0].imshow(d); axes[i,0].axis('off')
                    axes[i,1].imshow(f); axes[i,1].axis('off')
                    axes[i,2].imshow(fw); axes[i,2].axis('off')
                    axes[i,3].imshow(fb); axes[i,3].axis('off')
                    
                    # Labels
                    axes[i,0].set_ylabel(title, fontsize=12, rotation=90, labelpad=20)
                    if i == 0:
                        axes[i,0].set_title("Hue Disk")
                        axes[i,1].set_title("Flow")
                        axes[i,2].set_title("White BG")
                        axes[i,3].set_title("Black BG")

                plt.tight_layout()
                plt.show()
        except Exception as e:
            print(f"Vis error: {e}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from scipy.interpolate import interp1d
import os
from pathlib import Path

# Optional imports for Flow Vis
try:
    import omnirefactor
    from omnirefactor import io
    import fastremap
    OMNIPOSE_AVAILABLE = True
except ImportError:
    OMNIPOSE_AVAILABLE = False

torch.manual_seed(42)
np.random.seed(42)
dtype = torch.float32

# ============================================================
# 0. DEVICE CONFIGURATION
# ============================================================
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"🚀 Running on GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🚀 Running on Apple MPS (Metal)")
else:
    device = torch.device("cpu")
    print("⚠️ Running on CPU (GPU not found)")

# ============================================================
# 1. Color Math (Vectorized)
# ============================================================
M1 = torch.tensor([[0.8189330101, 0.3618667424, -0.1288597137],
                   [0.0329845436, 0.9293118715, 0.0361456387],
                   [0.0482003018, 0.2643662703, 0.6338517070]], dtype=dtype, device=device)
M2 = torch.tensor([[0.2104542553, 0.7936177850, -0.0040720468],
                   [1.9779984951, -2.4285922050, 0.4505937099],
                   [0.0259040371, 0.7827717662, -0.8086757660]], dtype=dtype, device=device)
M1_inv = torch.linalg.inv(M1)
M2_inv = torch.linalg.inv(M2)

class ColorConverter(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer('m1_inv', M1_inv)
        self.register_buffer('m2_inv', M2_inv)
        self.register_buffer('m1', M1)
        self.register_buffer('m2', M2)

    def oklab_to_linear_srgb(self, lab):
        lms_prime = lab @ self.m2_inv.T
        lms = torch.pow(lms_prime, 3)
        xyz = lms @ self.m1_inv.T
        r = 3.2404542*xyz[:,0] - 1.5371385*xyz[:,1] - 0.4985314*xyz[:,2]
        g = -0.9692660*xyz[:,0] + 1.8760108*xyz[:,1] + 0.0415560*xyz[:,2]
        b = 0.0556434*xyz[:,0] - 0.2040259*xyz[:,1] + 1.0572252*xyz[:,2]
        return torch.stack([r, g, b], dim=1)

    def linear_srgb_to_oklab(self, rgb):
        r = rgb[:,0]; g = rgb[:,1]; b = rgb[:,2]
        x = 0.4124564*r + 0.3575761*g + 0.1804375*b
        y = 0.2126729*r + 0.7151522*g + 0.0721750*b
        z = 0.0193339*r + 0.1191920*g + 0.9503041*b
        xyz = torch.stack([x,y,z], dim=1)
        lms = xyz @ self.m1.T
        lms = torch.clamp(lms, min=1e-8) 
        lms_prime = torch.pow(lms, 1.0/3.0)
        return lms_prime @ self.m2.T

converter = ColorConverter().to(device)

def linear_to_srgb_np(rgb):
    rgb = np.asarray(rgb, dtype=float)
    a = 0.055
    return np.where(rgb <= 0.0031308, 12.92 * rgb, (1 + a) * np.power(np.clip(rgb, 0, None), 1/2.4) - a)

def srgb_to_linear_np(rgb):
    rgb = np.asarray(rgb, dtype=float)
    a = 0.055
    return np.where(rgb <= 0.04045, rgb / 12.92, ((rgb + a) / (1 + a))**2.4)

def linear_to_gamma_srgb(rgb_lin):
    mask = rgb_lin > 0.0031308
    rgb_gamma = torch.zeros_like(rgb_lin)
    rgb_gamma[mask] = 1.055 * torch.pow(rgb_lin[mask], 1/2.4) - 0.055
    rgb_gamma[~mask] = 12.92 * rgb_lin[~mask]
    return torch.clamp(rgb_gamma, 0.0, 1.0)

# ============================================================
# 2. The Symmetric Planar Model (Corrected)
# ============================================================
class SymmetricPlanarLoop(nn.Module):
    def __init__(self):
        super().__init__()
        
        # CHROMA: Symmetric Fourier (Order 2 - Even harmonics only)
        # C(t) = C(t + pi). This forces antipodal symmetry.
        self.C_base = nn.Parameter(torch.tensor(0.16, device=device)) # Start Wide
        self.C_cos2 = nn.Parameter(torch.tensor(0.0, device=device))
        self.C_sin2 = nn.Parameter(torch.tensor(0.0, device=device))
        
        # LIGHTNESS: Planar
        # L = Bias + Slope_a*a + Slope_b*b
        self.L_bias = nn.Parameter(torch.tensor(0.65, device=device))
        self.L_slope_a = nn.Parameter(torch.tensor(0.0, device=device))
        self.L_slope_b = nn.Parameter(torch.tensor(0.60, device=device)) # Gentle Tilt

    def forward(self, t):
        # Chroma: Base + 2nd Harmonic (Oval shape)
        cos2 = torch.cos(2*t); sin2 = torch.sin(2*t)
        C = self.C_base + self.C_cos2 * cos2 + self.C_sin2 * sin2
        C = F.softplus(C) # Ensure positive radius
        
        a = C * torch.cos(t)
        b = C * torch.sin(t)
        
        # Lightness: Plane
        L = self.L_bias + (self.L_slope_a * a) + (self.L_slope_b * b)
        
        return torch.stack([L, a, b], dim=1)

# ============================================================
# 3. Optimization Loop
# ============================================================
def optimize_symmetric_planar():
    model = SymmetricPlanarLoop().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    t_grid = torch.linspace(0, 2*np.pi, 256, device=device)
    
    print("⚡ Optimizing Symmetric Planar Loop...")
    
    for i in range(2000):
        optimizer.zero_grad(set_to_none=True)
        lab_out = model(t_grid)
        rgb_lin = converter.oklab_to_linear_srgb(lab_out)
        
        # 1. Stable Gamut Barrier
        k = 100.0 
        barrier_0 = F.softplus(-k * rgb_lin) / k
        barrier_1 = F.softplus(k * (rgb_lin - 1.0)) / k
        gamut_loss = torch.sum((barrier_0 + barrier_1)**2) * 5000.0
        
        # 2. Maximize Chroma
        radii = torch.norm(lab_out[:, 1:], dim=1)
        chroma_loss = -torch.mean(radii) * 10.0
        
        # 3. CONCRETE PILLAR (Repulsion)
        # Critical: Prevents the collapse seen in your image.
        # Must be > 0.14 radius at all times.
        repulsion_loss = torch.sum(F.relu(0.14 - radii)**2) * 500.0
        
        # 4. SAFETY FLOOR (Lightness)
        # Critical: Prevents diving into black.
        L_vals = lab_out[:, 0]
        floor_loss = torch.sum(F.relu(0.40 - L_vals)**2) * 100.0
        
        # 5. Centering & Slope Regularization
        center_drift = (model.L_bias - 0.65)**2 * 50.0
        slope_reg = (model.L_slope_a**2 + model.L_slope_b**2) * 0.1 # Don't tilt TOO extreme
        
        total_loss = gamut_loss + chroma_loss + repulsion_loss + floor_loss + center_drift + slope_reg
        
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        if i % 500 == 0:
            r_min = rgb_lin.min().item()
            r_max = rgb_lin.max().item()
            print(f"Iter {i:04d} | RGB: [{r_min:.3f}, {r_max:.3f}] | Min C: {radii.min().item():.3f}")
            
    return model

# ============================================================
# 4. Visualization Helpers
# ============================================================

def reparameterize_by_arclength(lab_points_np):
    dists = np.linalg.norm(lab_points_np - np.roll(lab_points_np, 1, axis=0), axis=1)
    dists[0] = np.linalg.norm(lab_points_np[0] - lab_points_np[-1])
    cumulative = np.cumsum(dists)
    total = cumulative[-1]
    t_curr = np.concatenate(([0], cumulative / total))
    pts = np.vstack([lab_points_np[-1:], lab_points_np])
    iL = interp1d(t_curr, pts[:,0], kind='cubic')
    ia = interp1d(t_curr, pts[:,1], kind='cubic')
    ib = interp1d(t_curr, pts[:,2], kind='cubic')
    t_targ = np.linspace(0, 1, len(lab_points_np) + 1)[:-1]
    return np.stack([iL(t_targ), ia(t_targ), ib(t_targ)], axis=1)

def align_colormap_to_red(lab_path):
    lab_tensor = torch.from_numpy(lab_path).float().to(device)
    rgb_lin = converter.oklab_to_linear_srgb(lab_tensor)
    rgb_srgb = linear_to_gamma_srgb(rgb_lin).cpu().numpy()
    target = np.array([1.0, 0.0, 0.0])
    dists = np.linalg.norm(rgb_srgb - target, axis=1)
    best_idx = np.argmin(dists)
    return np.roll(lab_path, -best_idx, axis=0)

def get_srgb_wireframe():
    res = 20
    t = torch.linspace(0, 1, res, device=device)
    edges = []
    def add(s, d): l = s.repeat(res, 1); l[:, d] = t; return l
    c = torch.tensor([[0,0,0], [1,0,0], [0,1,0], [0,0,1], [1,1,0], [1,0,1], [0,1,1], [1,1,1]], dtype=dtype, device=device)
    edges += [add(c[0], 0), add(c[0], 1), add(c[0], 2), add(c[7], 0), add(c[7], 1), add(c[7], 2)]
    edges += [add(c[1], 1), add(c[1], 2), add(c[2], 0), add(c[2], 2), add(c[3], 0), add(c[3], 1)]
    all_edges_rgb = torch.cat(edges, dim=0)
    all_edges_lab = converter.linear_srgb_to_oklab(all_edges_rgb)
    lab_np = all_edges_lab.cpu().numpy()
    out = []
    for i in range(0, len(lab_np), res):
        out.append(lab_np[i:i+res])
        out.append(np.array([[None, None, None]]))
    return np.concatenate(out, axis=0)

def plot_3d(lab_path):
    lab_t = torch.from_numpy(lab_path).float().to(device)
    rgb_lin = converter.oklab_to_linear_srgb(lab_t)
    rgb_gamma = linear_to_gamma_srgb(rgb_lin).cpu().numpy()
    rgb_gamma = np.clip(rgb_gamma, 0, 1) * 255
    colors = [f'rgb({r:.0f},{g:.0f},{b:.0f})' for r,g,b in rgb_gamma]
    
    gamut = get_srgb_wireframe()
    
    fig = go.Figure(data=[
        go.Scatter3d(x=gamut[:,1], y=gamut[:,2], z=gamut[:,0], mode='lines', 
                     line=dict(color='rgba(150,150,150,0.3)', width=2), name='sRGB', hoverinfo='none'),
        go.Scatter3d(x=lab_path[:,1], y=lab_path[:,2], z=lab_path[:,0], mode='markers', 
                     marker=dict(size=6, color=colors), name='Optimal',
                     hovertemplate='L: %{z:.2f}<br>a: %{x:.2f}<br>b: %{y:.2f}<extra></extra>')
    ])
    fig.update_layout(template='plotly_dark', title='Symmetric Planar Loop', 
                      scene=dict(xaxis_title='a', yaxis_title='b', zaxis_title='L', 
                                 aspectmode='manual', aspectratio=dict(x=1, y=1, z=1)),
                      margin=dict(l=0,r=0,b=0,t=40), height=700)
    try:
        import google.colab; pio.renderers.default = "colab"
    except:
        pio.renderers.default = "notebook_connected"
    fig.show()

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    n_hues = rgb_ring_lin.shape[0]
    N=512
    y, x = np.mgrid[0:N, 0:N]
    x = (x + 0.5) / N * 2 - 1; y = (y + 0.5) / N * 2 - 1
    r_grid = np.sqrt(x*x + y*y); theta = np.mod(np.arctan2(y, x), 2*np.pi)
    
    def apply_cmap(angle_norm, mag_norm):
        hue_f = angle_norm * n_hues
        idx0 = np.floor(hue_f).astype(int) % n_hues
        idx1 = (idx0 + 1) % n_hues
        t = hue_f - np.floor(hue_f)
        ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
        r = mag_norm[..., None]
        rgb = (1 - r) * center_lin + r * ring_interp
        return np.clip(linear_to_srgb_np(rgb), 0, 1)

    disk = apply_cmap(theta / (2*np.pi), np.clip(r_grid, 0, 1))
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi) / (2*np.pi)
    mag = np.sqrt(u*u + v*v); mag /= (mag.max() + 1e-8)
    
    flow_rgb = apply_cmap(angle, mag)
    alpha = mag[..., None]
    flow_white = alpha * flow_rgb + (1 - alpha)
    flow_black = alpha * flow_rgb
    return disk, flow_rgb, flow_white, flow_black

if __name__ == "__main__":
    # 1. Optimize
    model = optimize_symmetric_planar()
    
    # 2. Sample & Align
    with torch.no_grad():
        t = torch.linspace(0, 2*np.pi, 512, device=device)
        lab_out = model(t)
        lab_raw = lab_out.cpu().numpy()
        
    lab_balanced = reparameterize_by_arclength(lab_raw)
    lab_final = align_colormap_to_red(lab_balanced)
    
    plot_3d(lab_final)
    
    # 3. Omnipose Vis
    if OMNIPOSE_AVAILABLE:
        try:
            omnidir = Path(omnirefactor.__file__).parent.parent.parent
            basedir = os.path.join(omnidir, "docs", "_static")
            img_p = os.path.join(basedir, "ecoli.tif")
            msk_p = os.path.join(basedir, "ecoli_labels.tif")
            
            if os.path.exists(img_p):
                masks = io.imread(msk_p)
                slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
                masks = fastremap.renumber(masks[slc])[0]
                flows = omnirefactor.core.masks_to_flows(masks, device="cpu")[4].to("cpu")
                
                lab_t = torch.from_numpy(lab_final).float().to(device)
                rgb_lin_base = converter.oklab_to_linear_srgb(lab_t).cpu().numpy().clip(0,1)
                
                N_HUES = len(rgb_lin_base)
                rgb_lin_rot13 = np.roll(rgb_lin_base, int(N_HUES/3), axis=0)
                rgb_lin_rot14 = np.roll(rgb_lin_base, int(N_HUES/4), axis=0)
                
                center_lin = srgb_to_linear_np(np.array([0.5, 0.5, 0.5]))
                
                maps_to_plot = [
                    (rgb_lin_base, "Symmetric Planar"),
                    (rgb_lin_rot13, "Rotated 1/3"),
                    (rgb_lin_rot14, "Rotated 1/4")
                ]
                
                fig, axes = plt.subplots(3, 4, figsize=(16, 12))
                for i, (cmap_lin, title) in enumerate(maps_to_plot):
                    d, f, fw, fb = make_flow_images_for_ring(cmap_lin, center_lin, flows)
                    axes[i,0].imshow(d); axes[i,0].axis('off')
                    axes[i,1].imshow(f); axes[i,1].axis('off')
                    axes[i,2].imshow(fw); axes[i,2].axis('off')
                    axes[i,3].imshow(fb); axes[i,3].axis('off')
                    axes[i,0].set_ylabel(title, fontsize=12, rotation=90, labelpad=20)
                    if i == 0:
                        axes[i,0].set_title("Hue Disk")
                        axes[i,1].set_title("Flow")
                        axes[i,2].set_title("White BG")
                        axes[i,3].set_title("Black BG")

                plt.tight_layout()
                plt.show()
        except Exception as e:
            print(f"Vis error: {e}")

In [ ]:
seems like to stand out from gray, we want to either be dark or light, close to saturated

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from scipy.interpolate import interp1d
import os
from pathlib import Path

# Optional imports for Flow Vis
try:
    import omnirefactor
    from omnirefactor import io
    import fastremap
    OMNIPOSE_AVAILABLE = True
except ImportError:
    OMNIPOSE_AVAILABLE = False

torch.manual_seed(42)
np.random.seed(42)
dtype = torch.float32

# ============================================================
# 0. DEVICE CONFIGURATION
# ============================================================
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"🚀 Running on GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🚀 Running on Apple MPS (Metal)")
else:
    device = torch.device("cpu")
    print("⚠️ Running on CPU (GPU not found)")

# ============================================================
# 1. Color Math (Vectorized)
# ============================================================
M1 = torch.tensor([[0.8189330101, 0.3618667424, -0.1288597137],
                   [0.0329845436, 0.9293118715, 0.0361456387],
                   [0.0482003018, 0.2643662703, 0.6338517070]], dtype=dtype, device=device)
M2 = torch.tensor([[0.2104542553, 0.7936177850, -0.0040720468],
                   [1.9779984951, -2.4285922050, 0.4505937099],
                   [0.0259040371, 0.7827717662, -0.8086757660]], dtype=dtype, device=device)
M1_inv = torch.linalg.inv(M1)
M2_inv = torch.linalg.inv(M2)

class ColorConverter(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer('m1_inv', M1_inv)
        self.register_buffer('m2_inv', M2_inv)
        self.register_buffer('m1', M1)
        self.register_buffer('m2', M2)

    def oklab_to_linear_srgb(self, lab):
        lms_prime = lab @ self.m2_inv.T
        lms = torch.pow(lms_prime, 3)
        xyz = lms @ self.m1_inv.T
        r = 3.2404542*xyz[:,0] - 1.5371385*xyz[:,1] - 0.4985314*xyz[:,2]
        g = -0.9692660*xyz[:,0] + 1.8760108*xyz[:,1] + 0.0415560*xyz[:,2]
        b = 0.0556434*xyz[:,0] - 0.2040259*xyz[:,1] + 1.0572252*xyz[:,2]
        return torch.stack([r, g, b], dim=1)

    def linear_srgb_to_oklab(self, rgb):
        r = rgb[:,0]; g = rgb[:,1]; b = rgb[:,2]
        x = 0.4124564*r + 0.3575761*g + 0.1804375*b
        y = 0.2126729*r + 0.7151522*g + 0.0721750*b
        z = 0.0193339*r + 0.1191920*g + 0.9503041*b
        xyz = torch.stack([x,y,z], dim=1)
        lms = xyz @ self.m1.T
        lms = torch.clamp(lms, min=1e-8) 
        lms_prime = torch.pow(lms, 1.0/3.0)
        return lms_prime @ self.m2.T

converter = ColorConverter().to(device)

def linear_to_srgb_np(rgb):
    rgb = np.asarray(rgb, dtype=float)
    a = 0.055
    return np.where(rgb <= 0.0031308, 12.92 * rgb, (1 + a) * np.power(np.clip(rgb, 0, None), 1/2.4) - a)

def srgb_to_linear_np(rgb):
    rgb = np.asarray(rgb, dtype=float)
    a = 0.055
    return np.where(rgb <= 0.04045, rgb / 12.92, ((rgb + a) / (1 + a))**2.4)

def linear_to_gamma_srgb(rgb_lin):
    mask = rgb_lin > 0.0031308
    rgb_gamma = torch.zeros_like(rgb_lin)
    rgb_gamma[mask] = 1.055 * torch.pow(rgb_lin[mask], 1/2.4) - 0.055
    rgb_gamma[~mask] = 12.92 * rgb_lin[~mask]
    return torch.clamp(rgb_gamma, 0.0, 1.0)

# ============================================================
# 2. The Relaxed Saddle Model (Corrected)
# ============================================================
class RelaxedSaddleLoop(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Lightness: Order 2 (Saddle Shape)
        # Allows L to curve: L(t) = Bias + Tilt*sin(t) + Warp*sin(2t)
        self.order_L = 2
        self.L_dc = nn.Parameter(torch.tensor(0.65, device=device)) 
        self.L_coeffs = nn.Parameter(torch.zeros((self.order_L, 2), device=device) * 0.01)
        
        # Chroma: Order 2 (Oval Shape)
        self.order_C = 2
        self.C_dc = nn.Parameter(torch.tensor(0.15, device=device)) 
        self.C_coeffs = nn.Parameter(torch.zeros((self.order_C, 2), device=device) * 0.01)

        # Initialize Tilt (Yellow High, Blue Low)
        with torch.no_grad():
             self.L_coeffs[0, 0] = 0.20 

    def forward(self, t):
        L = self.L_dc.expand_as(t)
        C = self.C_dc.expand_as(t)
        
        sin_t = torch.sin(t); cos_t = torch.cos(t)
        sin_2t = torch.sin(2*t); cos_2t = torch.cos(2*t)
        
        # Vectorized Sum
        # L (Order 2)
        L = L + self.L_coeffs[0,0]*sin_t + self.L_coeffs[0,1]*cos_t
        L = L + self.L_coeffs[1,0]*sin_2t + self.L_coeffs[1,1]*cos_2t
        
        # C (Order 2)
        C = C + self.C_coeffs[0,0]*sin_t + self.C_coeffs[0,1]*cos_t
        C = C + self.C_coeffs[1,0]*sin_2t + self.C_coeffs[1,1]*cos_2t
        
        C = torch.abs(C)
        a = C * cos_t
        b = C * sin_t
        return torch.stack([L, a, b], dim=1)

# ============================================================
# 3. Optimization Loop
# ============================================================
def optimize_relaxed_saddle(iterations=4000, device=device):
    model = RelaxedSaddleLoop().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.015)
    t_grid = torch.linspace(0, 2*np.pi, 256, device=device)
    
    print(f"⚡ Optimizing Relaxed Saddle on {device}...")
    
    for i in range(iterations):
        optimizer.zero_grad(set_to_none=True)
        lab_out = model(t_grid)
        rgb_lin = converter.oklab_to_linear_srgb(lab_out)
        
        # 1. Stable Barrier
        k = 100.0 
        barrier_0 = F.softplus(-k * rgb_lin) / k
        barrier_1 = F.softplus(k * (rgb_lin - 1.0)) / k
        gamut_loss = torch.sum((barrier_0 + barrier_1)**2) * 2000.0
        
        # 2. Expansion
        radii = torch.norm(lab_out[:, 1:], dim=1)
        chroma_loss = -torch.mean(radii)
        
        # 3. SAFETY RAILS
        # A. The Concrete Pillar (Prevents Gray Lobe collapse)
        repulsion_loss = torch.sum(F.relu(0.15 - radii)**2) * 500.0
        
        # B. The Safety Floor (Prevents Dark Pit collapse)
        L_vals = lab_out[:, 0]
        floor_loss = torch.sum(F.relu(0.40 - L_vals)**2) * 100.0
        
        # 4. Regularization (Stiffness)
        reg_loss = (torch.sum(model.L_coeffs**2) + torch.sum(model.C_coeffs**2)) * 2.0
        
        total_loss = gamut_loss + chroma_loss + repulsion_loss + floor_loss + reg_loss
        
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        if i % 1000 == 0:
            r_min = rgb_lin.min().item()
            r_max = rgb_lin.max().item()
            print(f"Iter {i:04d} | RGB: [{r_min:.3f}, {r_max:.3f}] | Min C: {radii.min().item():.3f}")
            
    return model

# ============================================================
# 4. Visualization Helpers
# ============================================================

def reparameterize_by_arclength(lab_points_np):
    dists = np.linalg.norm(lab_points_np - np.roll(lab_points_np, 1, axis=0), axis=1)
    dists[0] = np.linalg.norm(lab_points_np[0] - lab_points_np[-1])
    cumulative = np.cumsum(dists)
    total = cumulative[-1]
    t_curr = np.concatenate(([0], cumulative / total))
    pts = np.vstack([lab_points_np[-1:], lab_points_np])
    iL = interp1d(t_curr, pts[:,0], kind='cubic')
    ia = interp1d(t_curr, pts[:,1], kind='cubic')
    ib = interp1d(t_curr, pts[:,2], kind='cubic')
    t_targ = np.linspace(0, 1, len(lab_points_np) + 1)[:-1]
    return np.stack([iL(t_targ), ia(t_targ), ib(t_targ)], axis=1)

def align_colormap_to_red(lab_path):
    lab_tensor = torch.from_numpy(lab_path).float().to(device)
    rgb_lin = converter.oklab_to_linear_srgb(lab_tensor)
    rgb_srgb = linear_to_gamma_srgb(rgb_lin).cpu().numpy()
    target = np.array([1.0, 0.0, 0.0])
    dists = np.linalg.norm(rgb_srgb - target, axis=1)
    best_idx = np.argmin(dists)
    return np.roll(lab_path, -best_idx, axis=0)

def get_srgb_wireframe():
    res = 20
    t = torch.linspace(0, 1, res, device=device)
    edges = []
    def add(s, d): l = s.repeat(res, 1); l[:, d] = t; return l
    c = torch.tensor([[0,0,0], [1,0,0], [0,1,0], [0,0,1], [1,1,0], [1,0,1], [0,1,1], [1,1,1]], dtype=dtype, device=device)
    edges += [add(c[0], 0), add(c[0], 1), add(c[0], 2), add(c[7], 0), add(c[7], 1), add(c[7], 2)]
    edges += [add(c[1], 1), add(c[1], 2), add(c[2], 0), add(c[2], 2), add(c[3], 0), add(c[3], 1)]
    all_edges_rgb = torch.cat(edges, dim=0)
    all_edges_lab = converter.linear_srgb_to_oklab(all_edges_rgb)
    lab_np = all_edges_lab.cpu().numpy()
    out = []
    for i in range(0, len(lab_np), res):
        out.append(lab_np[i:i+res])
        out.append(np.array([[None, None, None]]))
    return np.concatenate(out, axis=0)

def plot_3d(lab_path):
    lab_t = torch.from_numpy(lab_path).float().to(device)
    rgb_lin = converter.oklab_to_linear_srgb(lab_t)
    rgb_gamma = linear_to_gamma_srgb(rgb_lin).cpu().numpy()
    rgb_gamma = np.clip(rgb_gamma, 0, 1) * 255
    colors = [f'rgb({r:.0f},{g:.0f},{b:.0f})' for r,g,b in rgb_gamma]
    
    gamut = get_srgb_wireframe()
    
    fig = go.Figure(data=[
        go.Scatter3d(x=gamut[:,1], y=gamut[:,2], z=gamut[:,0], mode='lines', 
                     line=dict(color='rgba(150,150,150,0.3)', width=2), name='sRGB', hoverinfo='none'),
        go.Scatter3d(x=lab_path[:,1], y=lab_path[:,2], z=lab_path[:,0], mode='markers', 
                     marker=dict(size=6, color=colors), name='Optimal',
                     hovertemplate='L: %{z:.2f}<br>a: %{x:.2f}<br>b: %{y:.2f}<extra></extra>')
    ])
    fig.update_layout(template='plotly_dark', title='Relaxed Saddle Loop (Safety Rails)', 
                      scene=dict(xaxis_title='a', yaxis_title='b', zaxis_title='L', 
                                 aspectmode='manual', aspectratio=dict(x=1, y=1, z=1)),
                      margin=dict(l=0,r=0,b=0,t=40), height=700)
    try:
        import google.colab; pio.renderers.default = "colab"
    except:
        pio.renderers.default = "notebook_connected"
    fig.show()

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    n_hues = rgb_ring_lin.shape[0]
    N=512
    y, x = np.mgrid[0:N, 0:N]
    x = (x + 0.5) / N * 2 - 1; y = (y + 0.5) / N * 2 - 1
    r_grid = np.sqrt(x*x + y*y); theta = np.mod(np.arctan2(y, x), 2*np.pi)
    
    def apply_cmap(angle_norm, mag_norm):
        hue_f = angle_norm * n_hues
        idx0 = np.floor(hue_f).astype(int) % n_hues
        idx1 = (idx0 + 1) % n_hues
        t = hue_f - np.floor(hue_f)
        ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
        r = mag_norm[..., None]
        rgb = (1 - r) * center_lin + r * ring_interp
        return np.clip(linear_to_srgb_np(rgb), 0, 1)

    disk = apply_cmap(theta / (2*np.pi), np.clip(r_grid, 0, 1))
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi) / (2*np.pi)
    mag = np.sqrt(u*u + v*v); mag /= (mag.max() + 1e-8)
    flow_rgb = apply_cmap(angle, mag)
    alpha = mag[..., None]
    flow_white = alpha * flow_rgb + (1 - alpha)
    flow_black = alpha * flow_rgb
    return disk, flow_rgb, flow_white, flow_black

if __name__ == "__main__":
    # 1. Optimize (Explicit device passed)
    model = optimize_relaxed_saddle(device=device)
    
    # 2. Sample & Align
    with torch.no_grad():
        t = torch.linspace(0, 2*np.pi, 512, device=device)
        lab_out = model(t)
        lab_raw = lab_out.cpu().numpy()
        
    lab_balanced = reparameterize_by_arclength(lab_raw)
    lab_final = align_colormap_to_red(lab_balanced)
    
    plot_3d(lab_final)
    
    # 3. Omnipose Vis
    if OMNIPOSE_AVAILABLE:
        try:
            omnidir = Path(omnirefactor.__file__).parent.parent.parent
            basedir = os.path.join(omnidir, "docs", "_static")
            img_p = os.path.join(basedir, "ecoli.tif")
            msk_p = os.path.join(basedir, "ecoli_labels.tif")
            
            if os.path.exists(img_p):
                masks = io.imread(msk_p)
                slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
                masks = fastremap.renumber(masks[slc])[0]
                flows = omnirefactor.core.masks_to_flows(masks, device="cpu")[4].to("cpu")
                
                lab_t = torch.from_numpy(lab_final).float().to(device)
                rgb_lin_base = converter.oklab_to_linear_srgb(lab_t).cpu().numpy().clip(0,1)
                
                N_HUES = len(rgb_lin_base)
                rgb_lin_rot13 = np.roll(rgb_lin_base, int(N_HUES/3), axis=0)
                rgb_lin_rot14 = np.roll(rgb_lin_base, int(N_HUES/4), axis=0)
                
                center_lin = srgb_to_linear_np(np.array([0.5, 0.5, 0.5]))
                
                maps_to_plot = [
                    (rgb_lin_base, "Relaxed Saddle"),
                    (rgb_lin_rot13, "Rotated 1/3"),
                    (rgb_lin_rot14, "Rotated 1/4")
                ]
                
                fig, axes = plt.subplots(3, 4, figsize=(16, 12))
                for i, (cmap_lin, title) in enumerate(maps_to_plot):
                    d, f, fw, fb = make_flow_images_for_ring(cmap_lin, center_lin, flows)
                    axes[i,0].imshow(d); axes[i,0].axis('off')
                    axes[i,1].imshow(f); axes[i,1].axis('off')
                    axes[i,2].imshow(fw); axes[i,2].axis('off')
                    axes[i,3].imshow(fb); axes[i,3].axis('off')
                    axes[i,0].set_ylabel(title, fontsize=12, rotation=90, labelpad=20)
                    if i == 0:
                        axes[i,0].set_title("Hue Disk")
                        axes[i,1].set_title("Flow")
                        axes[i,2].set_title("White BG")
                        axes[i,3].set_title("Black BG")

                plt.tight_layout()
                plt.show()
        except Exception as e:
            print(f"Vis error: {e}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from scipy.interpolate import interp1d
import os
from pathlib import Path

# Optional imports
try:
    import omnirefactor
    from omnirefactor import io
    import fastremap
    OMNIPOSE_AVAILABLE = True
except ImportError:
    OMNIPOSE_AVAILABLE = False

torch.manual_seed(42)
np.random.seed(42)
dtype = torch.float32

# ============================================================
# 0. DEVICE
# ============================================================
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"🚀 Running on GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🚀 Running on Apple MPS (Metal)")
else:
    device = torch.device("cpu")
    print("⚠️ Running on CPU (GPU not found)")

# ============================================================
# 1. Oklab Color Math (Standard)
# ============================================================
M1 = torch.tensor([[0.8189330101, 0.3618667424, -0.1288597137],
                   [0.0329845436, 0.9293118715, 0.0361456387],
                   [0.0482003018, 0.2643662703, 0.6338517070]], dtype=dtype, device=device)
M2 = torch.tensor([[0.2104542553, 0.7936177850, -0.0040720468],
                   [1.9779984951, -2.4285922050, 0.4505937099],
                   [0.0259040371, 0.7827717662, -0.8086757660]], dtype=dtype, device=device)
M1_inv = torch.linalg.inv(M1)
M2_inv = torch.linalg.inv(M2)

class ColorConverter(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer('m1_inv', M1_inv)
        self.register_buffer('m2_inv', M2_inv)
        self.register_buffer('m1', M1)
        self.register_buffer('m2', M2)

    def oklab_to_linear_srgb(self, lab):
        lms_prime = lab @ self.m2_inv.T
        lms = torch.pow(lms_prime, 3)
        xyz = lms @ self.m1_inv.T
        r = 3.2404542*xyz[:,0] - 1.5371385*xyz[:,1] - 0.4985314*xyz[:,2]
        g = -0.9692660*xyz[:,0] + 1.8760108*xyz[:,1] + 0.0415560*xyz[:,2]
        b = 0.0556434*xyz[:,0] - 0.2040259*xyz[:,1] + 1.0572252*xyz[:,2]
        return torch.stack([r, g, b], dim=1)

    def linear_srgb_to_oklab(self, rgb):
        r = rgb[:,0]; g = rgb[:,1]; b = rgb[:,2]
        x = 0.4124564*r + 0.3575761*g + 0.1804375*b
        y = 0.2126729*r + 0.7151522*g + 0.0721750*b
        z = 0.0193339*r + 0.1191920*g + 0.9503041*b
        xyz = torch.stack([x,y,z], dim=1)
        lms = xyz @ self.m1.T
        lms = torch.clamp(lms, min=1e-8) 
        lms_prime = torch.pow(lms, 1.0/3.0)
        return lms_prime @ self.m2.T

converter = ColorConverter().to(device)

# Anchors for Salience (Black & Gray)
P_BLACK = torch.tensor([[0.0, 0.0, 0.0]], dtype=dtype, device=device)
P_GRAY  = torch.tensor([[0.5, 0.0, 0.0]], dtype=dtype, device=device) # Middle Gray

# Helpers
def linear_to_srgb_np(rgb):
    rgb = np.asarray(rgb, dtype=float); a = 0.055
    return np.where(rgb <= 0.0031308, 12.92 * rgb, (1 + a) * np.power(np.clip(rgb, 0, None), 1/2.4) - a)

def srgb_to_linear_np(rgb):
    rgb = np.asarray(rgb, dtype=float); a = 0.055
    return np.where(rgb <= 0.04045, rgb / 12.92, ((rgb + a) / (1 + a))**2.4)

# ============================================================
# 2. The High-Key Model
# ============================================================
class HighKeyLoop(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Lightness: Order 3 (Saddle)
        # Start HIGH (0.70) to bias towards brightness
        self.order_L = 3
        self.L_dc = nn.Parameter(torch.tensor(0.70, device=device)) 
        self.L_coeffs = nn.Parameter(torch.zeros((self.order_L, 2), device=device) * 0.01)
        
        # Chroma: Order 2 (Circle)
        self.order_C = 2
        self.C_dc = nn.Parameter(torch.tensor(0.15, device=device)) 
        self.C_coeffs = nn.Parameter(torch.zeros((self.order_C, 2), device=device) * 0.01)

        # Tilt
        with torch.no_grad():
             self.L_coeffs[0, 0] = 0.15 

    def forward(self, t):
        L = self.L_dc.expand_as(t); C = self.C_dc.expand_as(t)
        sin_t = torch.sin(t); cos_t = torch.cos(t)
        
        for k in range(1, self.order_L + 1):
            L = L + self.L_coeffs[k-1,0]*torch.sin(k*t) + self.L_coeffs[k-1,1]*torch.cos(k*t)
        for k in range(1, self.order_C + 1):
            C = C + self.C_coeffs[k-1,0]*torch.sin(k*t) + self.C_coeffs[k-1,1]*torch.cos(k*t)
        
        C = torch.abs(C)
        a = C * cos_t; b = C * sin_t
        return torch.stack([L, a, b], dim=1)

# ============================================================
# 3. Optimization (Bright Salience)
# ============================================================
def optimize_bright_salience(iterations=4000, device=device):
    model = HighKeyLoop().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    t_grid = torch.linspace(0, 2*np.pi, 256, device=device)
    
    print("⚡ Optimizing High-Key Salience (Biased against Black)...")
    
    for i in range(iterations):
        optimizer.zero_grad(set_to_none=True)
        lab_out = model(t_grid)
        rgb_lin = converter.oklab_to_linear_srgb(lab_out)
        
        # 1. Stable Barrier
        k = 100.0 
        barrier = F.softplus(-k * rgb_lin) + F.softplus(k * (rgb_lin - 1.0))
        gamut_loss = torch.sum(barrier**2) * 2000.0
        
        # 2. ASYMMETRIC SALIENCE
        # We care more about being far from Black than far from White.
        d_black = torch.norm(lab_out - P_BLACK, dim=1)
        d_gray  = torch.norm(lab_out - P_GRAY, dim=1)
        
        # Metric: Weighted geometric mean
        # Combines "Brightness" (d_black) and "Colorfulness" (d_gray)
        # We exclude White to allow high-key pastels.
        salience = (d_black ** 0.7) * (d_gray ** 0.3)
        
        # A. Maximize
        expansion_loss = -torch.mean(salience) * 500.0
        
        # B. Iso-Salience (Uniformity)
        iso_loss = torch.var(salience) * 5000.0
        
        # 3. High Floor
        # Prevent diving below 0.50 (Solidly bright)
        L_vals = lab_out[:, 0]
        floor_loss = torch.sum(F.relu(0.50 - L_vals)**2) * 50.0
        
        # 4. Repulsion (Pillar)
        radii = torch.norm(lab_out[:, 1:], dim=1)
        repulsion_loss = torch.sum(F.relu(0.14 - radii)**2) * 100.0
        
        # 5. Regularization
        reg_loss = (torch.sum(model.L_coeffs**2) + torch.sum(model.C_coeffs**2)) * 10.0
        
        total_loss = gamut_loss + expansion_loss + iso_loss + floor_loss + repulsion_loss + reg_loss
        
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        if i % 1000 == 0:
            print(f"Iter {i:04d} | Salience: {salience.mean().item():.4f} | Min L: {L_vals.min().item():.3f}")
            
    return model

# ============================================================
# 4. Vis & Run
# ============================================================
def reparameterize_by_arclength(lab_points_np):
    dists = np.linalg.norm(lab_points_np - np.roll(lab_points_np, 1, axis=0), axis=1)
    cumulative = np.cumsum(dists); total = cumulative[-1]
    t_curr = np.concatenate(([0], cumulative / total))
    pts = np.vstack([lab_points_np[-1:], lab_points_np])
    iL = interp1d(t_curr, pts[:,0], kind='cubic')
    ia = interp1d(t_curr, pts[:,1], kind='cubic')
    ib = interp1d(t_curr, pts[:,2], kind='cubic')
    t_targ = np.linspace(0, 1, len(lab_points_np) + 1)[:-1]
    return np.stack([iL(t_targ), ia(t_targ), ib(t_targ)], axis=1)

def align_red(lab_path):
    lab_t = torch.from_numpy(lab_path).float().to(device)
    rgb_lin = converter.oklab_to_linear_srgb(lab_t)
    rgb_srgb = linear_to_srgb_np(rgb_lin.cpu().numpy())
    target = np.array([1.0, 0.0, 0.0])
    idx = np.argmin(np.linalg.norm(rgb_srgb - target, axis=1))
    return np.roll(lab_path, -idx, axis=0)

def get_srgb_wireframe():
    res = 20; t = torch.linspace(0, 1, res, device=device)
    def add(s, d): l = s.repeat(res, 1); l[:, d] = t; return l
    c = torch.tensor([[0,0,0],[1,0,0],[0,1,0],[0,0,1],[1,1,0],[1,0,1],[0,1,1],[1,1,1]], dtype=dtype, device=device)
    edges = []
    edges += [add(c[0],0), add(c[0],1), add(c[0],2), add(c[7],0), add(c[7],1), add(c[7],2)]
    edges += [add(c[1],1), add(c[1],2), add(c[2],0), add(c[2],2), add(c[3],0), add(c[3],1)]
    rgb = torch.cat(edges, dim=0)
    lab = converter.linear_srgb_to_oklab(rgb).cpu().numpy()
    out = []
    for i in range(0, len(lab), res):
        out.append(lab[i:i+res]); out.append(np.array([[None, None, None]]))
    return np.concatenate(out, axis=0)

def plot_3d(lab_path):
    lab_t = torch.from_numpy(lab_path).float().to(device)
    rgb_lin = converter.oklab_to_linear_srgb(lab_t)
    rgb_srgb = linear_to_srgb_np(rgb_lin.cpu().numpy())
    rgb_np = np.clip(rgb_srgb, 0, 1) * 255
    colors = [f'rgb({r:.0f},{g:.0f},{b:.0f})' for r,g,b in rgb_np]
    gamut = get_srgb_wireframe()
    
    fig = go.Figure(data=[
        go.Scatter3d(x=gamut[:,1], y=gamut[:,2], z=gamut[:,0], mode='lines', 
                     line=dict(color='rgba(150,150,150,0.2)', width=2), name='sRGB', hoverinfo='none'),
        go.Scatter3d(x=lab_path[:,1], y=lab_path[:,2], z=lab_path[:,0], mode='markers', 
                     marker=dict(size=6, color=colors), name='Optimal',
                     hovertemplate='L: %{z:.4f}<br>a: %{x:.4f}<br>b: %{y:.4f}<extra></extra>')
    ])
    fig.update_layout(template='plotly_dark', title='High-Key Salience Ring', 
                      scene=dict(xaxis_title='a', yaxis_title='b', zaxis_title='L', aspectmode='data'),
                      margin=dict(l=0,r=0,b=0,t=40), height=700)
    try:
        import google.colab; pio.renderers.default = "colab"
    except:
        pio.renderers.default = "notebook_connected"
    fig.show()

def make_flow_images(rgb_path, center, flows):
    n = len(rgb_path); N=512
    y,x = np.mgrid[0:N, 0:N]
    x = (x+0.5)/N*2-1; y = (y+0.5)/N*2-1
    r = np.sqrt(x*x+y*y); th = np.mod(np.arctan2(y,x), 2*np.pi)
    
    def map(ang, mag):
        idx0 = (ang * n / (2*np.pi)).astype(int) % n
        idx1 = (idx0 + 1) % n
        t = (ang * n / (2*np.pi)) % 1
        col = (1-t[...,None])*rgb_path[idx0] + t[...,None]*rgb_path[idx1]
        out = (1-mag[...,None])*center + mag[...,None]*col
        return np.clip(linear_to_srgb_np(out), 0, 1)

    disk = map(th, np.clip(r,0,1))
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    ang = np.mod(np.arctan2(v,u), 2*np.pi)
    mag = np.sqrt(u*u+v*v); mag /= (mag.max()+1e-8)
    flow = map(ang, mag)
    alpha = mag[...,None]
    fw = alpha*flow + (1-alpha)
    fb = alpha*flow
    return disk, flow, fw, fb

if __name__ == "__main__":
    model = optimize_bright_salience(device=device)
    
    with torch.no_grad():
        t = torch.linspace(0, 2*np.pi, 512, device=device)
        lab = model(t).cpu().numpy()
        
    lab_balanced = reparameterize_by_arclength(lab)
    lab_final = align_red(lab_balanced)
    plot_3d(lab_final)
    
    if OMNIPOSE_AVAILABLE:
        try:
            omnidir = Path(omnirefactor.__file__).parent.parent.parent
            basedir = os.path.join(omnidir, "docs", "_static")
            img = io.imread(os.path.join(basedir, "ecoli.tif"))
            msk = io.imread(os.path.join(basedir, "ecoli_labels.tif"))
            slc = omnirefactor.measure.crop_bbox(msk>0, pad=0)[0]
            flows = omnirefactor.core.masks_to_flows(fastremap.renumber(msk[slc])[0], device="cpu")[4].to("cpu")
            
            lab_t = torch.from_numpy(lab_final).float().to(device)
            rgb_lin = converter.oklab_to_linear_srgb(lab_t).cpu().numpy().clip(0,1)
            # Bright center (Linear)
            center = srgb_to_linear_np(np.array([0.7,0.7,0.7]))
            
            maps = [
                (rgb_lin, "High-Key Salience"),
                (np.roll(rgb_lin, 512//3, 0), "Rot 1/3"),
                (np.roll(rgb_lin, 512//4, 0), "Rot 1/4")
            ]
            
            fig, ax = plt.subplots(3,4, figsize=(16,12))
            for i, (m, t) in enumerate(maps):
                d, f, fw, fb = make_flow_images(m, center, flows)
                ax[i,0].imshow(d); ax[i,1].imshow(f); ax[i,2].imshow(fw); ax[i,3].imshow(fb)
                for a in ax[i]: a.axis('off')
                ax[i,0].set_ylabel(t, fontsize=12)
            plt.tight_layout(); plt.show()
            
        except Exception as e: print(e)

In [ ]:
we might also need to specify that sudden changes in angle anso coeespond to the same delta E,
so waht shapes have the properly that the distance bwteen most distanct poits is a cosntant? 

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from scipy.interpolate import interp1d
import os
from pathlib import Path

# Optional imports
try:
    import omnirefactor
    from omnirefactor import io
    import fastremap
    OMNIPOSE_AVAILABLE = True
except ImportError:
    OMNIPOSE_AVAILABLE = False

torch.manual_seed(42)
np.random.seed(42)
dtype = torch.float32

# ============================================================
# 0. DEVICE CONFIGURATION
# ============================================================
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"🚀 Running on GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🚀 Running on Apple MPS (Metal)")
else:
    device = torch.device("cpu")
    print("⚠️ Running on CPU (GPU not found)")

# ============================================================
# 1. Oklab Color Math (Standard)
# ============================================================
M1 = torch.tensor([[0.8189330101, 0.3618667424, -0.1288597137],
                   [0.0329845436, 0.9293118715, 0.0361456387],
                   [0.0482003018, 0.2643662703, 0.6338517070]], dtype=dtype, device=device)
M2 = torch.tensor([[0.2104542553, 0.7936177850, -0.0040720468],
                   [1.9779984951, -2.4285922050, 0.4505937099],
                   [0.0259040371, 0.7827717662, -0.8086757660]], dtype=dtype, device=device)
M1_inv = torch.linalg.inv(M1)
M2_inv = torch.linalg.inv(M2)

class ColorConverter(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer('m1_inv', M1_inv)
        self.register_buffer('m2_inv', M2_inv)
        self.register_buffer('m1', M1)
        self.register_buffer('m2', M2)

    def oklab_to_linear_srgb(self, lab):
        lms_prime = lab @ self.m2_inv.T
        lms = torch.pow(lms_prime, 3)
        xyz = lms @ self.m1_inv.T
        r = 3.2404542*xyz[:,0] - 1.5371385*xyz[:,1] - 0.4985314*xyz[:,2]
        g = -0.9692660*xyz[:,0] + 1.8760108*xyz[:,1] + 0.0415560*xyz[:,2]
        b = 0.0556434*xyz[:,0] - 0.2040259*xyz[:,1] + 1.0572252*xyz[:,2]
        return torch.stack([r, g, b], dim=1)

    def linear_srgb_to_oklab(self, rgb):
        r = rgb[:,0]; g = rgb[:,1]; b = rgb[:,2]
        x = 0.4124564*r + 0.3575761*g + 0.1804375*b
        y = 0.2126729*r + 0.7151522*g + 0.0721750*b
        z = 0.0193339*r + 0.1191920*g + 0.9503041*b
        xyz = torch.stack([x,y,z], dim=1)
        lms = xyz @ self.m1.T
        lms = torch.clamp(lms, min=1e-8) 
        lms_prime = torch.pow(lms, 1.0/3.0)
        return lms_prime @ self.m2.T

converter = ColorConverter().to(device)

# Anchors in Oklab
P_WHITE = torch.tensor([[1.0, 0.0, 0.0]], dtype=dtype, device=device)
P_BLACK = torch.tensor([[0.0, 0.0, 0.0]], dtype=dtype, device=device)

# Helpers
def linear_to_srgb_np(rgb):
    rgb = np.asarray(rgb, dtype=float); a = 0.055
    return np.where(rgb <= 0.0031308, 12.92 * rgb, (1 + a) * np.power(np.clip(rgb, 0, None), 1/2.4) - a)

def srgb_to_linear_np(rgb):
    rgb = np.asarray(rgb, dtype=float); a = 0.055
    return np.where(rgb <= 0.04045, rgb / 12.92, ((rgb + a) / (1 + a))**2.4)

def linear_to_gamma_srgb(rgb_lin):
    mask = rgb_lin > 0.0031308
    rgb_gamma = torch.zeros_like(rgb_lin)
    rgb_gamma[mask] = 1.055 * torch.pow(rgb_lin[mask], 1/2.4) - 0.055
    rgb_gamma[~mask] = 12.92 * rgb_lin[~mask]
    return torch.clamp(rgb_gamma, 0.0, 1.0)

# ============================================================
# 2. The Stiff Wire Model
# ============================================================
class StiffWireLoop(nn.Module):
    def __init__(self, start_L=0.6):
        super().__init__()
        
        # Order 3 for Lightness (Saddle shape)
        self.order_L = 3
        self.L_dc = nn.Parameter(torch.tensor(start_L, device=device)) 
        self.L_coeffs = nn.Parameter(torch.zeros((self.order_L, 2), device=device) * 0.01)
        
        # Order 2 for Chroma (Ellipse shape)
        self.order_C = 2
        self.C_dc = nn.Parameter(torch.tensor(0.15, device=device)) 
        self.C_coeffs = nn.Parameter(torch.zeros((self.order_C, 2), device=device) * 0.01)

        with torch.no_grad():
             self.L_coeffs[0, 0] = 0.15 

    def forward(self, t):
        L = self.L_dc.expand_as(t); C = self.C_dc.expand_as(t)
        sin_t = torch.sin(t); cos_t = torch.cos(t)
        
        # Vectorized Fourier Sum
        for k in range(1, self.order_L + 1):
            L = L + self.L_coeffs[k-1,0]*torch.sin(k*t) + self.L_coeffs[k-1,1]*torch.cos(k*t)
        for k in range(1, self.order_C + 1):
            C = C + self.C_coeffs[k-1,0]*torch.sin(k*t) + self.C_coeffs[k-1,1]*torch.cos(k*t)
        
        C = torch.abs(C)
        a = C * cos_t; b = C * sin_t
        return torch.stack([L, a, b], dim=1)

# ============================================================
# 3. Optimization (Power-Law Salience)
# ============================================================
def optimize_power_salience(alpha_black=0.6, iterations=3000):
    """
    alpha_black: exponent for distance to black. 
    alpha_white = 1.0 - alpha_black.
    """
    alpha_white = 1.0 - alpha_black
    
    # Init model near target lightness
    model = StiffWireLoop(start_L=alpha_black).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.015)
    t_grid = torch.linspace(0, 2*np.pi, 256, device=device)
    
    print(f"⚡ Optimizing Alpha={alpha_black} (Black^{alpha_black} * White^{alpha_white:.1f})...")
    
    for i in range(iterations):
        optimizer.zero_grad(set_to_none=True)
        lab_out = model(t_grid)
        rgb_lin = converter.oklab_to_linear_srgb(lab_out)
        
        # 1. Stable Barrier
        k = 100.0 
        barrier_0 = F.softplus(-k * rgb_lin) / k
        barrier_1 = F.softplus(k * (rgb_lin - 1.0)) / k
        gamut_loss = torch.sum((barrier_0 + barrier_1)**2) * 2000.0
        
        # 2. POWER-LAW SALIENCE METRIC
        d_black = torch.norm(lab_out - P_BLACK, dim=1)
        d_white = torch.norm(lab_out - P_WHITE, dim=1)
        
        # Metric: Weighted Geometric Mean
        salience = (d_black ** alpha_black) * (d_white ** alpha_white)
        
        # A. Maximize Salience (Expansion)
        expansion_loss = -torch.mean(salience) * 500.0
        
        # B. Iso-Salience (Uniformity)
        iso_loss = torch.var(salience) * 5000.0
        
        # 3. Repulsion (Pillar)
        radii = torch.norm(lab_out[:, 1:], dim=1)
        repulsion_loss = torch.sum(F.relu(0.14 - radii)**2) * 100.0
        
        # 4. Stiffness
        reg_loss = (torch.sum(model.L_coeffs**2) + torch.sum(model.C_coeffs**2)) * 5.0
        
        total_loss = gamut_loss + expansion_loss + iso_loss + repulsion_loss + reg_loss
        
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
    return model

# ============================================================
# 4. Visualization Helpers
# ============================================================
def reparameterize_by_arclength(lab_points_np):
    dists = np.linalg.norm(lab_points_np - np.roll(lab_points_np, 1, axis=0), axis=1)
    cumulative = np.cumsum(dists); total = cumulative[-1]
    t_curr = np.concatenate(([0], cumulative / total))
    pts = np.vstack([lab_points_np[-1:], lab_points_np])
    iL = interp1d(t_curr, pts[:,0], kind='cubic')
    ia = interp1d(t_curr, pts[:,1], kind='cubic')
    ib = interp1d(t_curr, pts[:,2], kind='cubic')
    t_targ = np.linspace(0, 1, len(lab_points_np) + 1)[:-1]
    return np.stack([iL(t_targ), ia(t_targ), ib(t_targ)], axis=1)

def align_red(lab_path):
    lab_tensor = torch.from_numpy(lab_path).float().to(device)
    rgb_lin = converter.oklab_to_linear_srgb(lab_tensor)
    rgb_srgb = linear_to_srgb_np(rgb_lin.cpu().numpy())
    target = np.array([1.0, 0.0, 0.0])
    idx = np.argmin(np.linalg.norm(rgb_srgb - target, axis=1))
    return np.roll(lab_path, -idx, axis=0)

def get_srgb_wireframe():
    res = 20; t = torch.linspace(0, 1, res, device=device)
    def add(s, d): l = s.repeat(res, 1); l[:, d] = t; return l
    c = torch.tensor([[0,0,0],[1,0,0],[0,1,0],[0,0,1],[1,1,0],[1,0,1],[0,1,1],[1,1,1]], dtype=dtype, device=device)
    edges = []
    edges += [add(c[0],0), add(c[0],1), add(c[0],2), add(c[7],0), add(c[7],1), add(c[7],2)]
    edges += [add(c[1],1), add(c[1],2), add(c[2],0), add(c[2],2), add(c[3],0), add(c[3],1)]
    rgb = torch.cat(edges, dim=0)
    lab = converter.linear_srgb_to_oklab(rgb).cpu().numpy()
    out = []
    for i in range(0, len(lab), res):
        out.append(lab[i:i+res]); out.append(np.array([[None, None, None]]))
    return np.concatenate(out, axis=0)

def make_flow_images(rgb_path, center, flows):
    n = len(rgb_path); N=512
    y,x = np.mgrid[0:N, 0:N]
    x = (x+0.5)/N*2-1; y = (y+0.5)/N*2-1
    r = np.sqrt(x*x+y*y); th = np.mod(np.arctan2(y,x), 2*np.pi)
    
    def map(ang, mag):
        idx0 = (ang * n / (2*np.pi)).astype(int) % n
        idx1 = (idx0 + 1) % n
        t = (ang * n / (2*np.pi)) % 1
        col = (1-t[...,None])*rgb_path[idx0] + t[...,None]*rgb_path[idx1]
        out = (1-mag[...,None])*center + mag[...,None]*col
        return np.clip(linear_to_srgb_np(out), 0, 1)

    disk = map(th, np.clip(r,0,1))
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    ang = np.mod(np.arctan2(v,u), 2*np.pi)
    mag = np.sqrt(u*u+v*v); mag /= (mag.max()+1e-8)
    flow = map(ang, mag)
    alpha = mag[...,None]
    fw = alpha*flow + (1-alpha)
    fb = alpha*flow
    return disk, flow, fw, fb

# ============================================================
# Main
# ============================================================
if __name__ == "__main__":
    alphas = [0.5, 0.6, 0.8]
    
    # 1. Optimize All
    results = []
    for a in alphas:
        model = optimize_power_salience(alpha_black=a)
        with torch.no_grad():
            t = torch.linspace(0, 2*np.pi, 512, device=device)
            lab = model(t).cpu().numpy()
        
        lab_bal = reparameterize_by_arclength(lab)
        lab_final = align_red(lab_bal)
        
        # Convert to RGB for plot
        lab_t = torch.from_numpy(lab_final).float().to(device)
        rgb_lin = converter.oklab_to_linear_srgb(lab_t).cpu().numpy().clip(0,1)
        rgb_srgb = linear_to_srgb_np(rgb_lin)
        
        results.append({
            'alpha': a,
            'lab': lab_final,
            'rgb_lin': rgb_lin,
            'rgb_srgb': rgb_srgb
        })
        
    # 2. 3D Comparison Plot
    gamut = get_srgb_wireframe()
    data = [
        go.Scatter3d(x=gamut[:,1], y=gamut[:,2], z=gamut[:,0], mode='lines', 
                     line=dict(color='rgba(100,100,100,0.2)', width=2), name='sRGB', hoverinfo='none')
    ]
    
    trace_colors = ['cyan', 'magenta', 'yellow']
    for i, res in enumerate(results):
        lab = res['lab']
        a = res['alpha']
        data.append(go.Scatter3d(
            x=lab[:,1], y=lab[:,2], z=lab[:,0], mode='lines', 
            line=dict(width=5, color=trace_colors[i]), 
            name=f'Power Law (a={a})'
        ))
        
    fig = go.Figure(data=data)
    fig.update_layout(template='plotly_dark', title='Power-Law Salience Comparison', 
                      scene=dict(xaxis_title='a', yaxis_title='b', zaxis_title='L', aspectmode='data'),
                      margin=dict(l=0,r=0,b=0,t=40), height=700)
    
    try:
        import google.colab; pio.renderers.default = "colab"
    except:
        pio.renderers.default = "notebook_connected"
    fig.show()

    # 3. 2D Flow Vis
    if OMNIPOSE_AVAILABLE:
        try:
            omnidir = Path(omnirefactor.__file__).parent.parent.parent
            basedir = os.path.join(omnidir, "docs", "_static")
            img = io.imread(os.path.join(basedir, "ecoli.tif"))
            msk = io.imread(os.path.join(basedir, "ecoli_labels.tif"))
            slc = omnirefactor.measure.crop_bbox(msk>0, pad=0)[0]
            flows = omnirefactor.core.masks_to_flows(fastremap.renumber(msk[slc])[0], device="cpu")[4].to("cpu")
            
            center = srgb_to_linear_np(np.array([0.5,0.5,0.5]))
            
            fig, axes = plt.subplots(len(alphas), 4, figsize=(16, 4*len(alphas)))
            
            for i, res in enumerate(results):
                title = f"Power Law (a={res['alpha']})"
                d, f, fw, fb = make_flow_images(res['rgb_lin'], center, flows)
                
                axes[i,0].imshow(d); axes[i,0].axis('off')
                axes[i,1].imshow(f); axes[i,1].axis('off')
                axes[i,2].imshow(fw); axes[i,2].axis('off')
                axes[i,3].imshow(fb); axes[i,3].axis('off')
                
                axes[i,0].set_ylabel(title, fontsize=12, rotation=90, labelpad=20)
                
                if i == 0:
                    axes[i,0].set_title("Hue Disk")
                    axes[i,1].set_title("Flow")
                    axes[i,2].set_title("White BG")
                    axes[i,3].set_title("Black BG")
            
            plt.tight_layout()
            plt.show()
            
        except Exception as e: print(e)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from scipy.interpolate import interp1d
import os
from pathlib import Path

# Check for Omnipose
try:
    import omnirefactor
    from omnirefactor import io
    import fastremap
    OMNIPOSE_AVAILABLE = True
except ImportError:
    OMNIPOSE_AVAILABLE = False

torch.manual_seed(42)
np.random.seed(42)
dtype = torch.float32

# ============================================================
# 0. DEVICE
# ============================================================
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# ============================================================
# 1. Oklab Color Math
# ============================================================
M1 = torch.tensor([[0.8189330101, 0.3618667424, -0.1288597137],
                   [0.0329845436, 0.9293118715, 0.0361456387],
                   [0.0482003018, 0.2643662703, 0.6338517070]], dtype=dtype, device=device)
M2 = torch.tensor([[0.2104542553, 0.7936177850, -0.0040720468],
                   [1.9779984951, -2.4285922050, 0.4505937099],
                   [0.0259040371, 0.7827717662, -0.8086757660]], dtype=dtype, device=device)
M1_inv = torch.linalg.inv(M1)
M2_inv = torch.linalg.inv(M2)

class ColorConverter(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer('m1_inv', M1_inv)
        self.register_buffer('m2_inv', M2_inv)
        self.register_buffer('m1', M1)
        self.register_buffer('m2', M2)

    def oklab_to_linear_srgb(self, lab):
        lms_prime = lab @ self.m2_inv.T
        lms = torch.pow(lms_prime, 3)
        xyz = lms @ self.m1_inv.T
        r = 3.2404542*xyz[:,0] - 1.5371385*xyz[:,1] - 0.4985314*xyz[:,2]
        g = -0.9692660*xyz[:,0] + 1.8760108*xyz[:,1] + 0.0415560*xyz[:,2]
        b = 0.0556434*xyz[:,0] - 0.2040259*xyz[:,1] + 1.0572252*xyz[:,2]
        return torch.stack([r, g, b], dim=1)

    def linear_srgb_to_oklab(self, rgb):
        r = rgb[:,0]; g = rgb[:,1]; b = rgb[:,2]
        x = 0.4124564*r + 0.3575761*g + 0.1804375*b
        y = 0.2126729*r + 0.7151522*g + 0.0721750*b
        z = 0.0193339*r + 0.1191920*g + 0.9503041*b
        xyz = torch.stack([x,y,z], dim=1)
        lms = xyz @ self.m1.T
        lms = torch.clamp(lms, min=1e-8) 
        lms_prime = torch.pow(lms, 1.0/3.0)
        return lms_prime @ self.m2.T

converter = ColorConverter().to(device)

def linear_to_srgb_np(rgb):
    rgb = np.asarray(rgb, dtype=float); a = 0.055
    return np.where(rgb <= 0.0031308, 12.92 * rgb, (1 + a) * np.power(np.clip(rgb, 0, None), 1/2.4) - a)

def srgb_to_linear_np(rgb):
    rgb = np.asarray(rgb, dtype=float); a = 0.055
    return np.where(rgb <= 0.04045, rgb / 12.92, ((rgb + a) / (1 + a))**2.4)

def linear_to_gamma_srgb(rgb_lin):
    mask = rgb_lin > 0.0031308
    rgb_gamma = torch.zeros_like(rgb_lin)
    rgb_gamma[mask] = 1.055 * torch.pow(rgb_lin[mask], 1/2.4) - 0.055
    rgb_gamma[~mask] = 12.92 * rgb_lin[~mask]
    return torch.clamp(rgb_gamma, 0.0, 1.0)

# ============================================================
# 2. The Model (Stiff Wire)
# ============================================================
class HKLoop(nn.Module):
    def __init__(self, start_L=0.60):
        super().__init__()
        # Lightness Order 3 (Saddle)
        self.order_L = 3
        self.L_dc = nn.Parameter(torch.tensor(start_L, device=device)) 
        self.L_coeffs = nn.Parameter(torch.zeros((self.order_L, 2), device=device) * 0.01)
        
        # Chroma Order 2 (Ellipse)
        self.order_C = 2
        self.C_dc = nn.Parameter(torch.tensor(0.15, device=device)) 
        self.C_coeffs = nn.Parameter(torch.zeros((self.order_C, 2), device=device) * 0.01)

        with torch.no_grad():
             self.L_coeffs[0, 0] = 0.15 

    def forward(self, t):
        L = self.L_dc.expand_as(t); C = self.C_dc.expand_as(t)
        sin_t = torch.sin(t); cos_t = torch.cos(t)
        
        for k in range(1, self.order_L + 1):
            L = L + self.L_coeffs[k-1,0]*torch.sin(k*t) + self.L_coeffs[k-1,1]*torch.cos(k*t)
        for k in range(1, self.order_C + 1):
            C = C + self.C_coeffs[k-1,0]*torch.sin(k*t) + self.C_coeffs[k-1,1]*torch.cos(k*t)
        
        C = torch.abs(C)
        a = C * cos_t; b = C * sin_t
        return torch.stack([L, a, b], dim=1)

# ============================================================
# 3. Optimization (H-K Balance)
# ============================================================
def optimize_hk_balance(target_brightness=0.65, iterations=3000):
    model = HKLoop(start_L=target_brightness).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.015)
    t_grid = torch.linspace(0, 2*np.pi, 256, device=device)
    
    print(f"⚡ Optimizing H-K Target Q={target_brightness}...")
    
    for i in range(iterations):
        optimizer.zero_grad(set_to_none=True)
        lab_out = model(t_grid)
        rgb_lin = converter.oklab_to_linear_srgb(lab_out)
        
        # 1. Stable Barrier
        k = 100.0 
        barrier_0 = F.softplus(-k * rgb_lin) / k
        barrier_1 = F.softplus(k * (rgb_lin - 1.0)) / k
        gamut_loss = torch.sum((barrier_0 + barrier_1)**2) * 2000.0
        
        # 2. H-K Metric: Q = L + 0.14*C
        L = lab_out[:, 0]
        C = torch.norm(lab_out[:, 1:], dim=1)
        Q = L + 0.14 * C
        
        # Target Loss
        target_loss = torch.mean((Q - target_brightness)**2) * 500.0
        
        # Uniformity
        iso_loss = torch.var(Q) * 5000.0
        
        # 3. Expansion (Maximize C)
        expansion_loss = -torch.mean(C) * 100.0
        
        # 4. Repulsion
        repulsion_loss = torch.sum(F.relu(0.14 - C)**2) * 100.0
        
        # 5. Stiffness
        reg_loss = (torch.sum(model.L_coeffs**2) + torch.sum(model.C_coeffs**2)) * 5.0
        
        total_loss = gamut_loss + target_loss + iso_loss + expansion_loss + repulsion_loss + reg_loss
        
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
            
    return model

# ============================================================
# 4. Vis & Run
# ============================================================
def reparameterize_by_arclength(lab_points_np):
    dists = np.linalg.norm(lab_points_np - np.roll(lab_points_np, 1, axis=0), axis=1)
    cumulative = np.cumsum(dists); total = cumulative[-1]
    t_curr = np.concatenate(([0], cumulative / total))
    pts = np.vstack([lab_points_np[-1:], lab_points_np])
    iL = interp1d(t_curr, pts[:,0], kind='cubic')
    ia = interp1d(t_curr, pts[:,1], kind='cubic')
    ib = interp1d(t_curr, pts[:,2], kind='cubic')
    t_targ = np.linspace(0, 1, len(lab_points_np) + 1)[:-1]
    return np.stack([iL(t_targ), ia(t_targ), ib(t_targ)], axis=1)

def align_red(lab_path):
    lab_tensor = torch.from_numpy(lab_path).float().to(device)
    rgb_lin = converter.oklab_to_linear_srgb(lab_tensor)
    rgb_srgb = linear_to_srgb_np(rgb_lin.cpu().numpy())
    target = np.array([1.0, 0.0, 0.0])
    idx = np.argmin(np.linalg.norm(rgb_srgb - target, axis=1))
    return np.roll(lab_path, -idx, axis=0)

def get_srgb_wireframe():
    res = 20; t = torch.linspace(0, 1, res, device=device)
    def add(s, d): l = s.repeat(res, 1); l[:, d] = t; return l
    c = torch.tensor([[0,0,0],[1,0,0],[0,1,0],[0,0,1],[1,1,0],[1,0,1],[0,1,1],[1,1,1]], dtype=dtype, device=device)
    edges = []
    edges += [add(c[0],0), add(c[0],1), add(c[0],2), add(c[7],0), add(c[7],1), add(c[7],2)]
    edges += [add(c[1],1), add(c[1],2), add(c[2],0), add(c[2],2), add(c[3],0), add(c[3],1)]
    rgb = torch.cat(edges, dim=0)
    lab = converter.linear_srgb_to_oklab(rgb).cpu().numpy()
    out = []
    for i in range(0, len(lab), res):
        out.append(lab[i:i+res]); out.append(np.array([[None, None, None]]))
    return np.concatenate(out, axis=0)

def plot_3d_multi(results_list):
    data = []
    
    # 1. Gamut Wireframe
    gamut = get_srgb_wireframe()
    data.append(go.Scatter3d(
        x=gamut[:,1], y=gamut[:,2], z=gamut[:,0], 
        mode='lines', line=dict(color='rgba(150,150,150,0.2)', width=2), 
        name='sRGB', hoverinfo='none'
    ))
    
    # 2. Plot each result
    colors = ['cyan', 'magenta', 'yellow']
    for i, res in enumerate(results_list):
        lab = res['lab']
        target_q = res['Q']
        data.append(go.Scatter3d(
            x=lab[:,1], y=lab[:,2], z=lab[:,0],
            mode='lines', line=dict(width=6, color=colors[i % len(colors)]),
            name=f'H-K Target {target_q}'
        ))

    layout = go.Layout(
        template='plotly_dark', 
        title='H-K Brightness Sweep',
        scene=dict(xaxis_title='a', yaxis_title='b', zaxis_title='L', aspectmode='data'),
        margin=dict(l=0,r=0,b=0,t=40), height=700
    )
    
    fig = go.Figure(data=data, layout=layout)
    
    try:
        import google.colab; pio.renderers.default = "colab"
    except:
        pio.renderers.default = "notebook_connected"
    fig.show()

def make_flow_images(rgb_path, center, flows):
    n = len(rgb_path); N=512
    y,x = np.mgrid[0:N, 0:N]
    x = (x+0.5)/N*2-1; y = (y+0.5)/N*2-1
    r = np.sqrt(x*x+y*y); th = np.mod(np.arctan2(y,x), 2*np.pi)
    
    def map(ang, mag):
        idx0 = (ang * n / (2*np.pi)).astype(int) % n
        idx1 = (idx0 + 1) % n
        t = (ang * n / (2*np.pi)) % 1
        col = (1-t[...,None])*rgb_path[idx0] + t[...,None]*rgb_path[idx1]
        out = (1-mag[...,None])*center + mag[...,None]*col
        return np.clip(linear_to_srgb_np(out), 0, 1)

    disk = map(th, np.clip(r,0,1))
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    ang = np.mod(np.arctan2(v,u), 2*np.pi)
    mag = np.sqrt(u*u+v*v); mag /= (mag.max()+1e-8)
    flow = map(ang, mag)
    alpha = mag[...,None]
    fw = alpha*flow + (1-alpha)
    fb = alpha*flow
    return disk, flow, fw, fb

if __name__ == "__main__":
    targets = [0.60, 0.65, 0.70]
    results = []
    
    # 1. Optimize All
    for t in targets:
        model = optimize_hk_balance(target_brightness=t)
        with torch.no_grad():
            tt = torch.linspace(0, 2*np.pi, 512, device=device)
            lab = model(tt).cpu().numpy()
        
        lab_bal = reparameterize_by_arclength(lab)
        lab_final = align_red(lab_bal)
        
        lab_t = torch.from_numpy(lab_final).float().to(device)
        rgb_lin = converter.oklab_to_linear_srgb(lab_t).cpu().numpy().clip(0,1)
        
        results.append({
            'Q': t,
            'lab': lab_final,
            'rgb_lin': rgb_lin
        })
    
    # 2. Plot all 3 in 3D
    plot_3d_multi(results)
    
    # 3. Omnipose Vis
    if OMNIPOSE_AVAILABLE:
        try:
            omnidir = Path(omnirefactor.__file__).parent.parent.parent
            basedir = os.path.join(omnidir, "docs", "_static")
            img = io.imread(os.path.join(basedir, "ecoli.tif"))
            msk = io.imread(os.path.join(basedir, "ecoli_labels.tif"))
            slc = omnirefactor.measure.crop_bbox(msk>0, pad=0)[0]
            flows = omnirefactor.core.masks_to_flows(fastremap.renumber(msk[slc])[0], device="cpu")[4].to("cpu")
            
            center = srgb_to_linear_np(np.array([0.5,0.5,0.5]))
            
            fig, axes = plt.subplots(len(targets), 4, figsize=(16, 4*len(targets)))
            for i, res in enumerate(results):
                d, f, fw, fb = make_flow_images(res['rgb_lin'], center, flows)
                axes[i,0].imshow(d); axes[i,1].imshow(f); axes[i,2].imshow(fw); axes[i,3].imshow(fb)
                for ax in axes[i]: ax.axis('off')
                axes[i,0].set_ylabel(f"H-K Target {res['Q']}", fontsize=12)
                if i == 0:
                    axes[i,0].set_title("Hue Disk")
                    axes[i,1].set_title("Flow")
                    axes[i,2].set_title("White BG")
                    axes[i,3].set_title("Black BG")
            
            plt.tight_layout(); plt.show()
            
        except Exception as e: print(e)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from scipy.interpolate import interp1d
import os
from pathlib import Path

# Optional imports
try:
    import omnirefactor
    from omnirefactor import io
    import fastremap
    OMNIPOSE_AVAILABLE = True
except ImportError:
    OMNIPOSE_AVAILABLE = False

torch.manual_seed(42)
np.random.seed(42)
dtype = torch.float32

# ============================================================
# 0. DEVICE
# ============================================================
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"🚀 Running on GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🚀 Running on Apple MPS (Metal)")
else:
    device = torch.device("cpu")
    print("⚠️ Running on CPU (GPU not found)")

# ============================================================
# 1. Oklab Color Math (Vectorized)
# ============================================================
M1 = torch.tensor([[0.8189330101, 0.3618667424, -0.1288597137],
                   [0.0329845436, 0.9293118715, 0.0361456387],
                   [0.0482003018, 0.2643662703, 0.6338517070]], dtype=dtype, device=device)
M2 = torch.tensor([[0.2104542553, 0.7936177850, -0.0040720468],
                   [1.9779984951, -2.4285922050, 0.4505937099],
                   [0.0259040371, 0.7827717662, -0.8086757660]], dtype=dtype, device=device)
M1_inv = torch.linalg.inv(M1)
M2_inv = torch.linalg.inv(M2)

class ColorConverter(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer('m1_inv', M1_inv)
        self.register_buffer('m2_inv', M2_inv)
        self.register_buffer('m1', M1)
        self.register_buffer('m2', M2)

    def oklab_to_linear_srgb(self, lab):
        lms_prime = lab @ self.m2_inv.T
        lms = torch.pow(lms_prime, 3)
        xyz = lms @ self.m1_inv.T
        r = 3.2404542*xyz[:,0] - 1.5371385*xyz[:,1] - 0.4985314*xyz[:,2]
        g = -0.9692660*xyz[:,0] + 1.8760108*xyz[:,1] + 0.0415560*xyz[:,2]
        b = 0.0556434*xyz[:,0] - 0.2040259*xyz[:,1] + 1.0572252*xyz[:,2]
        return torch.stack([r, g, b], dim=1)

    def linear_srgb_to_oklab(self, rgb):
        r = rgb[:,0]; g = rgb[:,1]; b = rgb[:,2]
        x = 0.4124564*r + 0.3575761*g + 0.1804375*b
        y = 0.2126729*r + 0.7151522*g + 0.0721750*b
        z = 0.0193339*r + 0.1191920*g + 0.9503041*b
        xyz = torch.stack([x,y,z], dim=1)
        lms = xyz @ self.m1.T
        lms = torch.clamp(lms, min=1e-8) 
        lms_prime = torch.pow(lms, 1.0/3.0)
        return lms_prime @ self.m2.T

converter = ColorConverter().to(device)

def linear_to_srgb_np(rgb):
    rgb = np.asarray(rgb, dtype=float); a = 0.055
    return np.where(rgb <= 0.0031308, 12.92 * rgb, (1 + a) * np.power(np.clip(rgb, 0, None), 1/2.4) - a)

def srgb_to_linear_np(rgb):
    rgb = np.asarray(rgb, dtype=float); a = 0.055
    return np.where(rgb <= 0.04045, rgb / 12.92, ((rgb + a) / (1 + a))**2.4)

def linear_to_gamma_srgb(rgb_lin):
    mask = rgb_lin > 0.0031308
    rgb_gamma = torch.zeros_like(rgb_lin)
    rgb_gamma[mask] = 1.055 * torch.pow(rgb_lin[mask], 1/2.4) - 0.055
    rgb_gamma[~mask] = 12.92 * rgb_lin[~mask]
    return torch.clamp(rgb_gamma, 0.0, 1.0)

# ============================================================
# 2. The Model (Sphere Walker)
# ============================================================
class SphereWalker(nn.Module):
    def __init__(self, start_L=0.60):
        super().__init__()
        # Lightness: Order 3 (Allows saddle/dip)
        self.order_L = 3
        self.L_dc = nn.Parameter(torch.tensor(start_L, device=device)) 
        self.L_coeffs = nn.Parameter(torch.zeros((self.order_L, 2), device=device) * 0.01)
        
        # Chroma: Order 3 (Allows rounded square/circle)
        self.order_C = 3
        self.C_dc = nn.Parameter(torch.tensor(0.15, device=device)) 
        self.C_coeffs = nn.Parameter(torch.zeros((self.order_C, 2), device=device) * 0.01)

        with torch.no_grad():
             self.L_coeffs[0, 0] = 0.15 

    def forward(self, t):
        L = self.L_dc.expand_as(t); C = self.C_dc.expand_as(t)
        sin_t = torch.sin(t); cos_t = torch.cos(t)
        
        for k in range(1, self.order_L + 1):
            L = L + self.L_coeffs[k-1,0]*torch.sin(k*t) + self.L_coeffs[k-1,1]*torch.cos(k*t)
        for k in range(1, self.order_C + 1):
            C = C + self.C_coeffs[k-1,0]*torch.sin(k*t) + self.C_coeffs[k-1,1]*torch.cos(k*t)
        
        C = torch.abs(C)
        a = C * cos_t; b = C * sin_t
        return torch.stack([L, a, b], dim=1)

# ============================================================
# 3. Optimization (Iso-Distance Sphere)
# ============================================================
def optimize_sphere_dist(center_L=0.60, iterations=3000):
    model = SphereWalker(start_L=center_L).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.015)
    t_grid = torch.linspace(0, 2*np.pi, 256, device=device)
    
    # THE CENTER OF THE SPHERE (Fixed)
    P_CENTER = torch.tensor([[center_L, 0.0, 0.0]], dtype=dtype, device=device)
    
    print(f"⚡ Inflating Sphere centered at L={center_L}...")
    
    for i in range(iterations):
        optimizer.zero_grad(set_to_none=True)
        lab_out = model(t_grid)
        rgb_lin = converter.oklab_to_linear_srgb(lab_out)
        
        # 1. Stable Barrier
        k = 100.0 
        barrier_0 = F.softplus(-k * rgb_lin) / k
        barrier_1 = F.softplus(k * (rgb_lin - 1.0)) / k
        gamut_loss = torch.sum((barrier_0 + barrier_1)**2) * 2000.0
        
        # 2. SPHERICAL DISTANCE
        dist_from_center = torch.norm(lab_out - P_CENTER, dim=1)
        
        # A. Uniformity (Iso-Distance)
        sphere_loss = torch.var(dist_from_center) * 10000.0
        
        # B. Expansion (Maximize radius)
        expansion_loss = -torch.mean(dist_from_center) * 500.0
        
        # 3. Repulsion (Pillar)
        C = torch.norm(lab_out[:, 1:], dim=1)
        repulsion_loss = torch.sum(F.relu(0.14 - C)**2) * 100.0
        
        # 4. Stiffness
        reg_loss = (torch.sum(model.L_coeffs**2) + torch.sum(model.C_coeffs**2)) * 5.0
        
        total_loss = gamut_loss + sphere_loss + expansion_loss + repulsion_loss + reg_loss
        
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        if i % 1000 == 0:
            print(f"Iter {i:04d} | Sphere Radius: {dist_from_center.mean().item():.3f}")
            
    return model

# ============================================================
# 4. Vis & Run
# ============================================================
def reparameterize_by_arclength(lab_points_np):
    dists = np.linalg.norm(lab_points_np - np.roll(lab_points_np, 1, axis=0), axis=1)
    cumulative = np.cumsum(dists); total = cumulative[-1]
    t_curr = np.concatenate(([0], cumulative / total))
    pts = np.vstack([lab_points_np[-1:], lab_points_np])
    iL = interp1d(t_curr, pts[:,0], kind='cubic')
    ia = interp1d(t_curr, pts[:,1], kind='cubic')
    ib = interp1d(t_curr, pts[:,2], kind='cubic')
    t_targ = np.linspace(0, 1, len(lab_points_np) + 1)[:-1]
    return np.stack([iL(t_targ), ia(t_targ), ib(t_targ)], axis=1)

def align_red(lab_path):
    lab_tensor = torch.from_numpy(lab_path).float().to(device)
    rgb_lin = converter.oklab_to_linear_srgb(lab_tensor)
    rgb_srgb = linear_to_srgb_np(rgb_lin.cpu().numpy())
    target = np.array([1.0, 0.0, 0.0])
    idx = np.argmin(np.linalg.norm(rgb_srgb - target, axis=1))
    return np.roll(lab_path, -idx, axis=0)

def get_srgb_wireframe():
    res = 20; t = torch.linspace(0, 1, res, device=device)
    def add(s, d): l = s.repeat(res, 1); l[:, d] = t; return l
    c = torch.tensor([[0,0,0],[1,0,0],[0,1,0],[0,0,1],[1,1,0],[1,0,1],[0,1,1],[1,1,1]], dtype=dtype, device=device)
    edges = []
    edges += [add(c[0],0), add(c[0],1), add(c[0],2), add(c[7],0), add(c[7],1), add(c[7],2)]
    edges += [add(c[1],1), add(c[1],2), add(c[2],0), add(c[2],2), add(c[3],0), add(c[3],1)]
    rgb = torch.cat(edges, dim=0)
    lab = converter.linear_srgb_to_oklab(rgb).cpu().numpy()
    out = []
    for i in range(0, len(lab), res):
        out.append(lab[i:i+res]); out.append(np.array([[None, None, None]]))
    return np.concatenate(out, axis=0)

def plot_3d_multi(results_list):
    data = []
    gamut = get_srgb_wireframe()
    data.append(go.Scatter3d(x=gamut[:,1], y=gamut[:,2], z=gamut[:,0], 
                             mode='lines', line=dict(color='rgba(150,150,150,0.2)', width=2), name='sRGB', hoverinfo='none'))
    
    colors = ['cyan', 'magenta', 'yellow']
    for i, res in enumerate(results_list):
        lab = res['lab']
        data.append(go.Scatter3d(
            x=lab[:,1], y=lab[:,2], z=lab[:,0],
            mode='lines', line=dict(width=6, color=colors[i % len(colors)]),
            name=f"Center L={res['Lc']}"
        ))
        data.append(go.Scatter3d(
            x=[0], y=[0], z=[res['Lc']],
            mode='markers', marker=dict(size=4, color=colors[i % len(colors)]),
            showlegend=False
        ))

    layout = go.Layout(template='plotly_dark', title='Spherical Iso-Distance Sweep', 
                      scene=dict(xaxis_title='a', yaxis_title='b', zaxis_title='L', aspectmode='data'),
                      margin=dict(l=0,r=0,b=0,t=40), height=700)
    try: import google.colab; pio.renderers.default = "colab"
    except: pio.renderers.default = "notebook_connected"
    fig = go.Figure(data=data, layout=layout)
    fig.show()

def make_flow_images(rgb_path, center, flows):
    n = len(rgb_path); N=512
    y,x = np.mgrid[0:N, 0:N]
    x = (x+0.5)/N*2-1; y = (y+0.5)/N*2-1
    r = np.sqrt(x*x+y*y); th = np.mod(np.arctan2(y,x), 2*np.pi)
    def map(ang, mag):
        idx0 = (ang * n / (2*np.pi)).astype(int) % n
        idx1 = (idx0 + 1) % n
        t = (ang * n / (2*np.pi)) % 1
        col = (1-t[...,None])*rgb_path[idx0] + t[...,None]*rgb_path[idx1]
        out = (1-mag[...,None])*center + mag[...,None]*col
        return np.clip(linear_to_srgb_np(out), 0, 1)
    disk = map(th, np.clip(r,0,1))
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    ang = np.mod(np.arctan2(v,u), 2*np.pi)
    mag = np.sqrt(u*u+v*v); mag /= (mag.max()+1e-8)
    flow = map(ang, mag)
    alpha = mag[...,None]
    fw = alpha*flow + (1-alpha)
    fb = alpha*flow
    return disk, flow, fw, fb

if __name__ == "__main__":
    centers = [0.4, 0.5, 0.6] # Sweep Sphere Centers
    results = []
    
    for lc in centers:
        model = optimize_sphere_dist(center_L=lc)
        with torch.no_grad():
            t = torch.linspace(0, 2*np.pi, 512, device=device)
            lab = model(t).cpu().numpy()
        
        lab_bal = reparameterize_by_arclength(lab)
        lab_final = align_red(lab_bal)
        
        lab_t = torch.from_numpy(lab_final).float().to(device)
        rgb_lin = converter.oklab_to_linear_srgb(lab_t).cpu().numpy().clip(0,1)
        
        results.append({'Lc': lc, 'lab': lab_final, 'rgb_lin': rgb_lin})
    
    plot_3d_multi(results)
    
    if OMNIPOSE_AVAILABLE:
        try:
            omnidir = Path(omnirefactor.__file__).parent.parent.parent
            basedir = os.path.join(omnidir, "docs", "_static")
            img = io.imread(os.path.join(basedir, "ecoli.tif"))
            msk = io.imread(os.path.join(basedir, "ecoli_labels.tif"))
            slc = omnirefactor.measure.crop_bbox(msk>0, pad=0)[0]
            flows = omnirefactor.core.masks_to_flows(fastremap.renumber(msk[slc])[0], device="cpu")[4].to("cpu")
            center = srgb_to_linear_np(np.array([0.5,0.5,0.5]))
            
            fig, axes = plt.subplots(len(centers), 4, figsize=(16, 4*len(centers)))
            for i, res in enumerate(results):
                d, f, fw, fb = make_flow_images(res['rgb_lin'], center, flows)
                axes[i,0].imshow(d); axes[i,1].imshow(f); axes[i,2].imshow(fw); axes[i,3].imshow(fb)
                for ax in axes[i]: ax.axis('off')
                # Add clear labels to the Y-axis
                axes[i,0].set_ylabel(f"Sphere Center L={res['Lc']}", fontsize=14, rotation=90, labelpad=20)
                
                if i == 0:
                    axes[i,0].set_title("Hue Disk")
                    axes[i,1].set_title("Flow")
                    axes[i,2].set_title("White BG")
                    axes[i,3].set_title("Black BG")
            plt.tight_layout(); plt.show()
        except Exception as e: print(e)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from scipy.interpolate import interp1d
import os
from pathlib import Path

# Omnipose Imports
try:
    import omnirefactor
    from omnirefactor.plot import imshow
    from omnirefactor import io
    import fastremap
    OMNIPOSE_AVAILABLE = True
except ImportError:
    OMNIPOSE_AVAILABLE = False
    print("⚠️ WARNING: Omnipose/Cellpose-omni not installed. Flow visualization will fail.")

torch.manual_seed(42)
np.random.seed(42)
dtype = torch.float32

# ============================================================
# 0. DEVICE
# ============================================================
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"🚀 Running on GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🚀 Running on Apple MPS (Metal)")
else:
    device = torch.device("cpu")
    print("⚠️ Running on CPU (GPU not found)")

# ============================================================
# 1. Oklab Color Math
# ============================================================
M1 = torch.tensor([[0.8189330101, 0.3618667424, -0.1288597137],
                   [0.0329845436, 0.9293118715, 0.0361456387],
                   [0.0482003018, 0.2643662703, 0.6338517070]], dtype=dtype, device=device)
M2 = torch.tensor([[0.2104542553, 0.7936177850, -0.0040720468],
                   [1.9779984951, -2.4285922050, 0.4505937099],
                   [0.0259040371, 0.7827717662, -0.8086757660]], dtype=dtype, device=device)
M1_inv = torch.linalg.inv(M1)
M2_inv = torch.linalg.inv(M2)

class ColorConverter(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer('m1_inv', M1_inv)
        self.register_buffer('m2_inv', M2_inv)
        self.register_buffer('m1', M1)
        self.register_buffer('m2', M2)

    def oklab_to_linear_srgb(self, lab):
        lms_prime = lab @ self.m2_inv.T
        lms = torch.pow(lms_prime, 3)
        xyz = lms @ self.m1_inv.T
        r = 3.2404542*xyz[:,0] - 1.5371385*xyz[:,1] - 0.4985314*xyz[:,2]
        g = -0.9692660*xyz[:,0] + 1.8760108*xyz[:,1] + 0.0415560*xyz[:,2]
        b = 0.0556434*xyz[:,0] - 0.2040259*xyz[:,1] + 1.0572252*xyz[:,2]
        return torch.stack([r, g, b], dim=1)

    def linear_srgb_to_oklab(self, rgb):
        r = rgb[:,0]; g = rgb[:,1]; b = rgb[:,2]
        x = 0.4124564*r + 0.3575761*g + 0.1804375*b
        y = 0.2126729*r + 0.7151522*g + 0.0721750*b
        z = 0.0193339*r + 0.1191920*g + 0.9503041*b
        xyz = torch.stack([x,y,z], dim=1)
        lms = xyz @ self.m1.T
        lms = torch.clamp(lms, min=1e-8) 
        lms_prime = torch.pow(lms, 1.0/3.0)
        return lms_prime @ self.m2.T

converter = ColorConverter().to(device)

def linear_to_srgb_np(rgb):
    rgb = np.asarray(rgb, dtype=float); a = 0.055
    return np.where(rgb <= 0.0031308, 12.92 * rgb, (1 + a) * np.power(np.clip(rgb, 0, None), 1/2.4) - a)

def srgb_to_linear_np(rgb):
    rgb = np.asarray(rgb, dtype=float); a = 0.055
    return np.where(rgb <= 0.04045, rgb / 12.92, ((rgb + a) / (1 + a))**2.4)

# ============================================================
# 2. The "Smooth Max-Min" Model (Oklab)
# ============================================================
class SmoothMaxMinLoop(nn.Module):
    def __init__(self):
        super().__init__()
        # Lightness: Order 3 (Saddle)
        self.order_L = 3
        self.L_dc = nn.Parameter(torch.tensor(0.65, device=device)) 
        self.L_coeffs = nn.Parameter(torch.zeros((self.order_L, 2), device=device) * 0.01)
        
        # Chroma: Order 2 (Strict Ellipse = No Cusps)
        self.order_C = 2
        self.C_dc = nn.Parameter(torch.tensor(0.15, device=device)) 
        self.C_coeffs = nn.Parameter(torch.zeros((self.order_C, 2), device=device) * 0.01)

        with torch.no_grad():
             self.L_coeffs[0, 0] = 0.15 

    def forward(self, t):
        L = self.L_dc.expand_as(t); C = self.C_dc.expand_as(t)
        sin_t = torch.sin(t); cos_t = torch.cos(t)
        
        for k in range(1, self.order_L + 1):
            L = L + self.L_coeffs[k-1,0]*torch.sin(k*t) + self.L_coeffs[k-1,1]*torch.cos(k*t)
        for k in range(1, self.order_C + 1):
            C = C + self.C_coeffs[k-1,0]*torch.sin(k*t) + self.C_coeffs[k-1,1]*torch.cos(k*t)
        
        C = torch.abs(C)
        a = C * cos_t; b = C * sin_t
        return torch.stack([L, a, b], dim=1)

# ============================================================
# 3. Optimization
# ============================================================
def optimize_smooth_maxmin(iterations=4000, device=device):
    model = SmoothMaxMinLoop().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.015)
    t_grid = torch.linspace(0, 2*np.pi, 256, device=device)
    
    print("⚡ Optimizing Oklab Max-Min Ellipse...")
    
    for i in range(iterations):
        optimizer.zero_grad(set_to_none=True)
        lab_out = model(t_grid)
        rgb_lin = converter.oklab_to_linear_srgb(lab_out)
        
        # 1. Stable Barrier
        k = 100.0 
        barrier_0 = F.softplus(-k * rgb_lin) / k
        barrier_1 = F.softplus(k * (rgb_lin - 1.0)) / k
        gamut_loss = torch.sum((barrier_0 + barrier_1)**2) * 5000.0
        
        # 2. Max-Min Chroma
        C_vals = torch.norm(lab_out[:, 1:], dim=1)
        min_chroma_loss = -torch.min(C_vals) * 1000.0
        
        # 3. Lightness Floor
        L_vals = lab_out[:, 0]
        floor_loss = torch.sum(F.relu(0.40 - L_vals)**2) * 50.0
        
        # 4. Stiffness
        reg_loss = (torch.sum(model.L_coeffs**2) + torch.sum(model.C_coeffs**2)) * 2.0
        
        total_loss = gamut_loss + min_chroma_loss + floor_loss + reg_loss
        
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        if i % 1000 == 0:
            print(f"Iter {i:04d} | Min C: {C_vals.min().item():.3f}")
            
    return model

# ============================================================
# 4. Visualization Helpers
# ============================================================
def reparameterize_by_arclength(lab_points_np):
    dists = np.linalg.norm(lab_points_np - np.roll(lab_points_np, 1, axis=0), axis=1)
    cumulative = np.cumsum(dists); total = cumulative[-1]
    t_curr = np.concatenate(([0], cumulative / total))
    pts = np.vstack([lab_points_np[-1:], lab_points_np])
    iL = interp1d(t_curr, pts[:,0], kind='cubic')
    ia = interp1d(t_curr, pts[:,1], kind='cubic')
    ib = interp1d(t_curr, pts[:,2], kind='cubic')
    t_targ = np.linspace(0, 1, len(lab_points_np) + 1)[:-1]
    return np.stack([iL(t_targ), ia(t_targ), ib(t_targ)], axis=1)

def align_red(lab_path):
    lab_tensor = torch.from_numpy(lab_path).float().to(device)
    rgb_lin = converter.oklab_to_linear_srgb(lab_tensor)
    rgb_srgb = linear_to_srgb_np(rgb_lin.cpu().numpy())
    target = np.array([1.0, 0.0, 0.0])
    idx = np.argmin(np.linalg.norm(rgb_srgb - target, axis=1))
    return np.roll(lab_path, -idx, axis=0)

def get_srgb_wireframe():
    res = 20; t = torch.linspace(0, 1, res, device=device)
    def add(s, d): l = s.repeat(res, 1); l[:, d] = t; return l
    c = torch.tensor([[0,0,0],[1,0,0],[0,1,0],[0,0,1],[1,1,0],[1,0,1],[0,1,1],[1,1,1]], dtype=dtype, device=device)
    edges = []
    edges += [add(c[0],0), add(c[0],1), add(c[0],2), add(c[7],0), add(c[7],1), add(c[7],2)]
    edges += [add(c[1],1), add(c[1],2), add(c[2],0), add(c[2],2), add(c[3],0), add(c[3],1)]
    rgb = torch.cat(edges, dim=0)
    lab = converter.linear_srgb_to_oklab(rgb).cpu().numpy()
    out = []
    for i in range(0, len(lab), res):
        out.append(lab[i:i+res]); out.append(np.array([[None, None, None]]))
    return np.concatenate(out, axis=0)

def plot_3d(lab_path):
    lab_t = torch.from_numpy(lab_path).float().to(device)
    rgb_lin = converter.oklab_to_linear_srgb(lab_t)
    rgb_srgb = linear_to_srgb_np(rgb_lin.cpu().numpy())
    rgb_np = np.clip(rgb_srgb, 0, 1) * 255
    colors = [f'rgb({r:.0f},{g:.0f},{b:.0f})' for r,g,b in rgb_np]
    gamut = get_srgb_wireframe()
    
    fig = go.Figure(data=[
        go.Scatter3d(x=gamut[:,1], y=gamut[:,2], z=gamut[:,0], mode='lines', 
                     line=dict(color='rgba(150,150,150,0.2)', width=2), name='sRGB', hoverinfo='none'),
        go.Scatter3d(x=lab_path[:,1], y=lab_path[:,2], z=lab_path[:,0], mode='markers', 
                     marker=dict(size=6, color=colors), name='Optimal',
                     hovertemplate='L: %{z:.4f}<br>a: %{x:.4f}<br>b: %{y:.4f}<extra></extra>')
    ])
    fig.update_layout(template='plotly_dark', title='Smooth Max-Min Loop (Oklab)', 
                      scene=dict(xaxis_title='a', yaxis_title='b', zaxis_title='L', aspectmode='data'),
                      margin=dict(l=0,r=0,b=0,t=40), height=700)
    try: import google.colab; pio.renderers.default = "colab"
    except: pio.renderers.default = "notebook_connected"
    fig.show()

def make_flow_images(rgb_path, center, flows):
    n = len(rgb_path); N=512
    y,x = np.mgrid[0:N, 0:N]
    x = (x+0.5)/N*2-1; y = (y+0.5)/N*2-1
    r = np.sqrt(x*x+y*y); th = np.mod(np.arctan2(y,x), 2*np.pi)
    
    def map(ang, mag):
        idx0 = (ang * n / (2*np.pi)).astype(int) % n
        idx1 = (idx0 + 1) % n
        t = (ang * n / (2*np.pi)) % 1
        col = (1-t[...,None])*rgb_path[idx0] + t[...,None]*rgb_path[idx1]
        out = (1-mag[...,None])*center + mag[...,None]*col
        return np.clip(linear_to_srgb_np(out), 0, 1)

    disk = map(th, np.clip(r,0,1))
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    ang = np.mod(np.arctan2(v,u), 2*np.pi)
    mag = np.sqrt(u*u+v*v); mag /= (mag.max()+1e-8)
    flow = map(ang, mag)
    alpha = mag[...,None]
    fw = alpha*flow + (1-alpha)
    fb = alpha*flow
    return disk, flow, fw, fb

if __name__ == "__main__":
    # 1. Optimize
    model = optimize_smooth_maxmin(device=device)
    
    # 2. Sample & Align
    with torch.no_grad():
        t = torch.linspace(0, 2*np.pi, 512, device=device)
        lab = model(t).cpu().numpy()
        
    lab_balanced = reparameterize_by_arclength(lab)
    lab_final = align_red(lab_balanced)
    plot_3d(lab_final)
    
    # 3. Load User Data
    if OMNIPOSE_AVAILABLE:
        # --- YOUR DATA LOADING CODE HERE ---
        try:
            print("Loading Omnipose Data...")
            omnidir = Path(omnirefactor.__file__).parent.parent.parent
            basedir = os.path.join(omnidir, "docs", "_static")
            name = "ecoli"
            ext = ".tif"

            image_path = os.path.join(basedir, name + ext)
            mask_path = os.path.join(basedir, name + "_labels" + ext)
            
            # Strict check
            if not os.path.exists(image_path):
                print(f"❌ File not found: {image_path}")
                raise FileNotFoundError(f"Cannot find {image_path}")

            image = io.imread(image_path)
            masks = io.imread(mask_path)
            slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
            masks = fastremap.renumber(masks[slc])[0]
            image = image[slc]

            # Calculate Flows
            # Note: user code says [4], but verify omni version.
            # Standard omnipose returns (flows, styles, etc). 
            # We assume user code is correct for their version.
            flows_all = omnirefactor.core.masks_to_flows(masks, device="cpu")[4]
            # Ensure we move to the correct device for our viz code? 
            # Actually, our vis code expects numpy cpu or torch cpu.
            # We will keep it as a torch tensor on CPU for make_flow_images compatibility
            flows = torch.tensor(flows_all, device='cpu')

            # Prepare Colormap
            lab_t = torch.from_numpy(lab_final).float().to(device)
            rgb_lin = converter.oklab_to_linear_srgb(lab_t).cpu().numpy().clip(0,1)
            center = srgb_to_linear_np(np.array([0.5,0.5,0.5]))
            
            # Create Rotations
            N_HUES = len(rgb_lin)
            maps = [
                (rgb_lin, "Smooth Max-Min"),
                (np.roll(rgb_lin, N_HUES//3, 0), "Rot 1/3"),
                (np.roll(rgb_lin, N_HUES//4, 0), "Rot 1/4")
            ]
            
            fig, axes = plt.subplots(3,4, figsize=(16,12))
            for i, (m, t) in enumerate(maps):
                d, f, fw, fb = make_flow_images(m, center, flows)
                axes[i,0].imshow(d); axes[i,1].imshow(f); axes[i,2].imshow(fw); axes[i,3].imshow(fb)
                for ax in axes[i]: ax.axis('off')
                axes[i,0].set_ylabel(t, fontsize=12)
            
            axes[0,0].set_title("Hue Disk")
            axes[0,1].set_title("Flow")
            axes[0,2].set_title("White BG")
            axes[0,3].set_title("Black BG")
            plt.tight_layout(); plt.show()
            
        except Exception as e:
            print(f"❌ Error loading/plotting Omnipose data: {e}")
            import traceback
            traceback.print_exc()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from scipy.interpolate import interp1d
import os
from pathlib import Path

# Optional imports
try:
    import omnirefactor
    from omnirefactor import io
    import fastremap
    OMNIPOSE_AVAILABLE = True
except ImportError:
    OMNIPOSE_AVAILABLE = False

torch.manual_seed(42)
np.random.seed(42)
dtype = torch.float32

# ============================================================
# 0. DEVICE
# ============================================================
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"🚀 Running on GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🚀 Running on Apple MPS (Metal)")
else:
    device = torch.device("cpu")
    print("⚠️ Running on CPU (GPU not found)")

# ============================================================
# 1. Oklab Color Math
# ============================================================
M1 = torch.tensor([[0.8189330101, 0.3618667424, -0.1288597137],
                   [0.0329845436, 0.9293118715, 0.0361456387],
                   [0.0482003018, 0.2643662703, 0.6338517070]], dtype=dtype, device=device)
M2 = torch.tensor([[0.2104542553, 0.7936177850, -0.0040720468],
                   [1.9779984951, -2.4285922050, 0.4505937099],
                   [0.0259040371, 0.7827717662, -0.8086757660]], dtype=dtype, device=device)
M1_inv = torch.linalg.inv(M1)
M2_inv = torch.linalg.inv(M2)

class ColorConverter(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer('m1_inv', M1_inv)
        self.register_buffer('m2_inv', M2_inv)
        self.register_buffer('m1', M1)
        self.register_buffer('m2', M2)

    def oklab_to_linear_srgb(self, lab):
        lms_prime = lab @ self.m2_inv.T
        lms = torch.pow(lms_prime, 3)
        xyz = lms @ self.m1_inv.T
        r = 3.2404542*xyz[:,0] - 1.5371385*xyz[:,1] - 0.4985314*xyz[:,2]
        g = -0.9692660*xyz[:,0] + 1.8760108*xyz[:,1] + 0.0415560*xyz[:,2]
        b = 0.0556434*xyz[:,0] - 0.2040259*xyz[:,1] + 1.0572252*xyz[:,2]
        return torch.stack([r, g, b], dim=1)

    def linear_srgb_to_oklab(self, rgb):
        r = rgb[:,0]; g = rgb[:,1]; b = rgb[:,2]
        x = 0.4124564*r + 0.3575761*g + 0.1804375*b
        y = 0.2126729*r + 0.7151522*g + 0.0721750*b
        z = 0.0193339*r + 0.1191920*g + 0.9503041*b
        xyz = torch.stack([x,y,z], dim=1)
        lms = xyz @ self.m1.T
        lms = torch.clamp(lms, min=1e-8) 
        lms_prime = torch.pow(lms, 1.0/3.0)
        return lms_prime @ self.m2.T

converter = ColorConverter().to(device)
P_BLACK = torch.tensor([[0.0, 0.0, 0.0]], dtype=dtype, device=device)

# Helpers
def linear_to_srgb_np(rgb):
    rgb = np.asarray(rgb, dtype=float); a = 0.055
    return np.where(rgb <= 0.0031308, 12.92 * rgb, (1 + a) * np.power(np.clip(rgb, 0, None), 1/2.4) - a)
def srgb_to_linear_np(rgb):
    rgb = np.asarray(rgb, dtype=float); a = 0.055
    return np.where(rgb <= 0.04045, rgb / 12.92, ((rgb + a) / (1 + a))**2.4)

# ============================================================
# 2. The Stiff Wire Model (Saddle)
# ============================================================
class CompositeSalienceLoop(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Lightness: Order 3 (Allows saddle shape to dip for purple)
        self.order_L = 3
        self.L_dc = nn.Parameter(torch.tensor(0.60, device=device)) 
        self.L_coeffs = nn.Parameter(torch.zeros((self.order_L, 2), device=device) * 0.01)
        
        # Chroma: Order 2 (Ellipse)
        self.order_C = 2
        self.C_dc = nn.Parameter(torch.tensor(0.15, device=device)) 
        self.C_coeffs = nn.Parameter(torch.zeros((self.order_C, 2), device=device) * 0.01)

        with torch.no_grad():
             self.L_coeffs[0, 0] = 0.15 

    def forward(self, t):
        L = self.L_dc.expand_as(t); C = self.C_dc.expand_as(t)
        sin_t = torch.sin(t); cos_t = torch.cos(t)
        
        for k in range(1, self.order_L + 1):
            L = L + self.L_coeffs[k-1,0]*torch.sin(k*t) + self.L_coeffs[k-1,1]*torch.cos(k*t)
        for k in range(1, self.order_C + 1):
            C = C + self.C_coeffs[k-1,0]*torch.sin(k*t) + self.C_coeffs[k-1,1]*torch.cos(k*t)
        
        C = torch.abs(C)
        a = C * cos_t; b = C * sin_t
        return torch.stack([L, a, b], dim=1)

# ============================================================
# 3. Optimization (Composite Metric)
# ============================================================
def optimize_composite_salience(gamma=0.5, iterations=4000):
    """
    gamma: Weight for Chroma (Distance to Gray). 
           1-gamma is weight for Brightness (Distance to Black).
    """
    model = CompositeSalienceLoop().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.015)
    t_grid = torch.linspace(0, 2*np.pi, 256, device=device)
    
    print(f"⚡ Optimizing Composite Salience (Chroma Weight={gamma})...")
    
    for i in range(iterations):
        optimizer.zero_grad(set_to_none=True)
        lab_out = model(t_grid)
        rgb_lin = converter.oklab_to_linear_srgb(lab_out)
        
        # 1. Stable Barrier
        k = 100.0 
        barrier_0 = F.softplus(-k * rgb_lin) / k
        barrier_1 = F.softplus(k * (rgb_lin - 1.0)) / k
        gamut_loss = torch.sum((barrier_0 + barrier_1)**2) * 2000.0
        
        # 2. COMPOSITE SALIENCE
        # Distance to Black (Brightness + Chroma)
        d_black = torch.norm(lab_out - P_BLACK, dim=1)
        # Distance to Gray (Pure Chroma)
        # Gray in Oklab is just the L channel (distance from L-axis is sqrt(a^2+b^2))
        d_gray = torch.norm(lab_out[:, 1:], dim=1)
        
        # The Metric: Trade-off between Brightness and Colorfulness
        salience = (d_black ** (1.0 - gamma)) * (d_gray ** gamma)
        
        # A. Maximize
        expansion_loss = -torch.mean(salience) * 500.0
        
        # B. Uniformity (Iso-Salience)
        iso_loss = torch.var(salience) * 5000.0
        
        # 3. Repulsion (Pillar)
        repulsion_loss = torch.sum(F.relu(0.14 - d_gray)**2) * 100.0
        
        # 4. Stiffness
        reg_loss = (torch.sum(model.L_coeffs**2) + torch.sum(model.C_coeffs**2)) * 5.0
        
        # 5. Floor (Lowered slightly to allow purple dip)
        L_vals = lab_out[:, 0]
        floor_loss = torch.sum(F.relu(0.35 - L_vals)**2) * 50.0
        
        total_loss = gamut_loss + expansion_loss + iso_loss + repulsion_loss + floor_loss + reg_loss
        
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
            
    return model

# ============================================================
# 4. Vis & Run
# ============================================================
def reparameterize_by_arclength(lab_points_np):
    dists = np.linalg.norm(lab_points_np - np.roll(lab_points_np, 1, axis=0), axis=1)
    cumulative = np.cumsum(dists); total = cumulative[-1]
    t_curr = np.concatenate(([0], cumulative / total))
    pts = np.vstack([lab_points_np[-1:], lab_points_np])
    iL = interp1d(t_curr, pts[:,0], kind='cubic')
    ia = interp1d(t_curr, pts[:,1], kind='cubic')
    ib = interp1d(t_curr, pts[:,2], kind='cubic')
    t_targ = np.linspace(0, 1, len(lab_points_np) + 1)[:-1]
    return np.stack([iL(t_targ), ia(t_targ), ib(t_targ)], axis=1)

def align_red(lab_path):
    lab_tensor = torch.from_numpy(lab_path).float().to(device)
    rgb_lin = converter.oklab_to_linear_srgb(lab_tensor)
    rgb_srgb = linear_to_srgb_np(rgb_lin.cpu().numpy())
    target = np.array([1.0, 0.0, 0.0])
    idx = np.argmin(np.linalg.norm(rgb_srgb - target, axis=1))
    return np.roll(lab_path, -idx, axis=0)

def get_srgb_wireframe():
    res = 20; t = torch.linspace(0, 1, res, device=device)
    def add(s, d): l = s.repeat(res, 1); l[:, d] = t; return l
    c = torch.tensor([[0,0,0],[1,0,0],[0,1,0],[0,0,1],[1,1,0],[1,0,1],[0,1,1],[1,1,1]], dtype=dtype, device=device)
    edges = []
    edges += [add(c[0],0), add(c[0],1), add(c[0],2), add(c[7],0), add(c[7],1), add(c[7],2)]
    edges += [add(c[1],1), add(c[1],2), add(c[2],0), add(c[2],2), add(c[3],0), add(c[3],1)]
    rgb = torch.cat(edges, dim=0)
    lab = converter.linear_srgb_to_oklab(rgb).cpu().numpy()
    out = []
    for i in range(0, len(lab), res):
        out.append(lab[i:i+res]); out.append(np.array([[None, None, None]]))
    return np.concatenate(out, axis=0)

def plot_3d_multi(results_list):
    data = []
    gamut = get_srgb_wireframe()
    data.append(go.Scatter3d(x=gamut[:,1], y=gamut[:,2], z=gamut[:,0], 
                             mode='lines', line=dict(color='rgba(150,150,150,0.2)', width=2), name='sRGB', hoverinfo='none'))
    
    colors = ['cyan', 'magenta', 'yellow']
    for i, res in enumerate(results_list):
        lab = res['lab']
        data.append(go.Scatter3d(
            x=lab[:,1], y=lab[:,2], z=lab[:,0],
            mode='lines', line=dict(width=6, color=colors[i % len(colors)]),
            name=f"Gamma={res['g']}"
        ))

    layout = go.Layout(template='plotly_dark', title='Composite Salience (Brightness vs Chroma)', 
                      scene=dict(xaxis_title='a', yaxis_title='b', zaxis_title='L', aspectmode='data'),
                      margin=dict(l=0,r=0,b=0,t=40), height=700)
    try: import google.colab; pio.renderers.default = "colab"
    except: pio.renderers.default = "notebook_connected"
    fig = go.Figure(data=data, layout=layout)
    fig.show()

def make_flow_images(rgb_path, center, flows):
    n = len(rgb_path); N=512
    y,x = np.mgrid[0:N, 0:N]
    x = (x+0.5)/N*2-1; y = (y+0.5)/N*2-1
    r = np.sqrt(x*x+y*y); th = np.mod(np.arctan2(y,x), 2*np.pi)
    def map(ang, mag):
        idx0 = (ang * n / (2*np.pi)).astype(int) % n
        idx1 = (idx0 + 1) % n
        t = (ang * n / (2*np.pi)) % 1
        col = (1-t[...,None])*rgb_path[idx0] + t[...,None]*rgb_path[idx1]
        out = (1-mag[...,None])*center + mag[...,None]*col
        return np.clip(linear_to_srgb_np(out), 0, 1)
    disk = map(th, np.clip(r,0,1))
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    ang = np.mod(np.arctan2(v,u), 2*np.pi)
    mag = np.sqrt(u*u+v*v); mag /= (mag.max()+1e-8)
    flow = map(ang, mag)
    alpha = mag[...,None]
    fw = alpha*flow + (1-alpha)
    fb = alpha*flow
    return disk, flow, fw, fb

if __name__ == "__main__":
    gammas = [0.4, 0.5, 0.6] # 0.4 = More Brightness, 0.6 = More Chroma
    results = []
    
    for g in gammas:
        model = optimize_composite_salience(gamma=g)
        with torch.no_grad():
            t = torch.linspace(0, 2*np.pi, 512, device=device)
            lab = model(t).cpu().numpy()
        
        lab_bal = reparameterize_by_arclength(lab)
        lab_final = align_red(lab_bal)
        
        lab_t = torch.from_numpy(lab_final).float().to(device)
        rgb_lin = converter.oklab_to_linear_srgb(lab_t).cpu().numpy().clip(0,1)
        
        results.append({'g': g, 'lab': lab_final, 'rgb_lin': rgb_lin})
    
    plot_3d_multi(results)
    
    if OMNIPOSE_AVAILABLE:
        try:
            omnidir = Path(omnirefactor.__file__).parent.parent.parent
            basedir = os.path.join(omnidir, "docs", "_static")
            img = io.imread(os.path.join(basedir, "ecoli.tif"))
            msk = io.imread(os.path.join(basedir, "ecoli_labels.tif"))
            slc = omnirefactor.measure.crop_bbox(msk>0, pad=0)[0]
            flows = omnirefactor.core.masks_to_flows(fastremap.renumber(msk[slc])[0], device="cpu")[4].to("cpu")
            center = srgb_to_linear_np(np.array([0.5,0.5,0.5]))
            
            fig, axes = plt.subplots(len(gammas), 4, figsize=(16, 4*len(gammas)))
            for i, res in enumerate(results):
                d, f, fw, fb = make_flow_images(res['rgb_lin'], center, flows)
                axes[i,0].imshow(d); axes[i,1].imshow(f); axes[i,2].imshow(fw); axes[i,3].imshow(fb)
                for ax in axes[i]: ax.axis('off')
                axes[i,0].set_ylabel(f"Chroma Weight {res['g']}", fontsize=12, rotation=90, labelpad=20)
                
                if i == 0:
                    axes[i,0].set_title("Hue Disk")
                    axes[i,1].set_title("Flow")
                    axes[i,2].set_title("White BG")
                    axes[i,3].set_title("Black BG")
            plt.tight_layout(); plt.show()
            
        except Exception as e: print(e)

In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import hsv_to_rgb
from scipy.interpolate import interp1d

# Plotly imports for 3D interactivity
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, display

import omnirefactor
from omnirefactor.plot import imshow
from omnirefactor import io
import fastremap
from pathlib import Path
import os
import itertools

# ============================================================
# 1. Color Math & JzAzBz Transforms
# ============================================================

def srgb_to_linear_np(rgb):
    rgb = np.asarray(rgb, dtype=float)
    a = 0.055
    thresh = 0.04045
    low  = rgb <= thresh
    high = ~low
    out = np.zeros_like(rgb)
    out[low]  = rgb[low] / 12.92
    base = (rgb[high] + a) / (1 + a)
    base = np.maximum(base, 0.0)
    out[high] = base ** 2.4
    return out

def linear_to_srgb_np(rgb):
    rgb = np.asarray(rgb, dtype=float)
    a = 0.055
    return np.where(
        rgb <= 0.0031308,
        12.92 * rgb,
        (1 + a) * np.power(np.clip(rgb, 0, None), 1 / 2.4) - a,
    )

M_RGB_TO_XYZ_np = np.array([
    [0.4124564, 0.3575761, 0.1804375],
    [0.2126729, 0.7151522, 0.0721750],
    [0.0193339, 0.1191920, 0.9503041],
], dtype=float)
M_XYZ_TO_RGB_np = np.linalg.inv(M_RGB_TO_XYZ_np)

def srgb_to_xyz_np(srgb):
    rgb_lin = srgb_to_linear_np(srgb)
    return rgb_lin @ M_RGB_TO_XYZ_np.T

def xyz_to_srgb_np(xyz):
    rgb_lin = xyz @ M_XYZ_TO_RGB_np.T
    return linear_to_srgb_np(rgb_lin)

# JzAzBz Constants
b_  = 1.15; g_  = 0.66
c1_ = 3424 / 2**12; c2_ = 2413 / 2**7; c3_ = 2392 / 2**7
n__ = 2610 / 2**14; p__ = 1.7 * 2523 / 2**5
d_  = -0.56; d0_ = 1.6295499532821566e-11

M1_np = np.array([
    [ 0.41478972,  0.579999,    0.0146480],
    [-0.2015100,   1.120649,    0.0531008],
    [-0.0166008,   0.264800,    0.6684799],
], dtype=float)

M2_np = np.array([
    [ 0.5,         0.5,         0.0        ],
    [ 3.524000,   -4.066708,    0.542708  ],
    [ 0.199076,    1.096799,   -1.295875  ],
], dtype=float)

M2_INV_np = np.linalg.inv(M2_np)
M1_INV_np = np.linalg.inv(M1_np)

def xyz_to_jzazbz_np(xyz):
    xyz = np.asarray(xyz, dtype=float)
    orig_shape = xyz.shape
    if xyz.ndim == 1: xyz = xyz[None, :]
    
    X, Y, Z = xyz[...,0], xyz[...,1], xyz[...,2]
    Xp = b_ * X - (b_ - 1.0) * Z
    Yp = g_ * Y - (g_ - 1.0) * X
    Zp = Z
    XYZ = np.stack([Xp, Yp, Zp], axis=-1)
    XYZ_4 = XYZ * 1e4
    LMS = XYZ_4 @ M1_np.T
    LMS_ratio = np.clip(LMS / 1e4, 1e-10, None)
    LMS_ratio_n = np.power(LMS_ratio, n__)
    num = c1_ + c2_ * LMS_ratio_n
    den = 1.0 + c3_ * LMS_ratio_n
    ratio = np.clip(num / den, 1e-10, None)
    LMS_p = np.power(ratio, p__)
    Izazbz = LMS_p @ M2_np.T
    Jz = ((1 + d_) * Izazbz[...,0]) / (1 + d_ * Izazbz[...,0]) - d0_
    res = np.stack([Jz, Izazbz[...,1], Izazbz[...,2]], axis=-1)
    
    if len(orig_shape) == 1: return res[0]
    return res

def jzazbz_to_xyz_np(jzazbz):
    jzazbz = np.asarray(jzazbz, dtype=float)
    orig_shape = jzazbz.shape
    if jzazbz.ndim == 1: jzazbz = jzazbz[None, :]

    Jz = jzazbz[...,0]
    Iz = (Jz + d0_) / (1 + d_ - d_ * (Jz + d0_))
    Izazbz = np.stack([Iz, jzazbz[...,1], jzazbz[...,2]], axis=-1)
    LMS_p = Izazbz @ M2_INV_np.T
    LMS_p = np.clip(LMS_p, 1e-10, None)
    LMS_c = np.power(LMS_p, 1.0 / p__)
    num = np.clip(LMS_c - c1_, 1e-10, None)
    den = np.clip(c2_ - c3_ * LMS_c, 1e-10, None)
    LMS_ratio = np.power(num / den, 1.0 / n__)
    LMS = LMS_ratio * 1e4
    XYZ_4 = LMS @ M1_INV_np.T
    XYZ = XYZ_4 / 1e4
    X = (XYZ[...,0] + (b_ - 1.0) * XYZ[...,2]) / b_
    Y = (XYZ[...,1] + (g_ - 1.0) * X) / g_
    res = np.stack([X, Y, XYZ[...,2]], axis=-1)

    if len(orig_shape) == 1: return res[0]
    return res

# ============================================================
# 2. Lighter Symmetric Ring Logic
# ============================================================

def build_lighter_symmetric_ring(n_hues: int, center_val: float = 0.68):
    # Center
    center_srgb = np.array([center_val, center_val, center_val])
    gray_xyz = srgb_to_xyz_np(np.array([center_srgb]))
    gray_jz = xyz_to_jzazbz_np(gray_xyz)[0]
    J_center = gray_jz[0]
    
    # Sample angles (Half circle 0-180)
    n_samples = 90
    angles = np.linspace(0, np.pi, n_samples, endpoint=False)
    
    optimal_chromas = []
    optimal_dJs = []
    
    print(f"Optimizing Symmetric Ring centered at sRGB {center_val} (Jz={J_center:.3f})...")
    
    for theta in angles:
        vec_a = np.cos(theta)
        vec_b = np.sin(theta)
        
        best_C = 0.0
        best_dJ = 0.0
        
        # Tilt search (dJ)
        dJ_range = np.linspace(-0.12, 0.12, 40) 
        
        for dJ in dJ_range:
            def check_c(c_val):
                p1 = np.array([J_center + dJ, c_val * vec_a, c_val * vec_b])
                p2 = np.array([J_center - dJ, -c_val * vec_a, -c_val * vec_b])
                pts = np.stack([p1, p2])
                rgb = xyz_to_srgb_np(jzazbz_to_xyz_np(pts))
                eps = 0.005 
                return np.all((rgb >= -eps) & (rgb <= 1.0 + eps))

            # Binary search for Radius (Chroma)
            low, high = 0.0, 0.35 
            if not check_c(0.0): continue 
            for _ in range(12):
                mid = (low + high) / 2
                if check_c(mid): low = mid
                else: high = mid
            
            if low > best_C:
                best_C = low
                best_dJ = dJ
                
        optimal_chromas.append(best_C)
        optimal_dJs.append(best_dJ)
        
    # Reconstruct 360 loop
    chromas = np.array(optimal_chromas)
    dJs = np.array(optimal_dJs)
    full_chromas = np.concatenate([chromas, chromas])
    full_dJs = np.concatenate([dJs, -dJs])
    full_angles = np.concatenate([angles, angles + np.pi])
    
    # Fourier Smoothing
    def smooth_periodic(y, num_harmonics=4):
        n = len(y)
        coeffs = np.fft.rfft(y)
        coeffs[num_harmonics+1:] = 0
        return np.fft.irfft(coeffs, n)
    
    smooth_C = smooth_periodic(full_chromas, num_harmonics=4)
    smooth_dJ = smooth_periodic(full_dJs, num_harmonics=4)
    
    # High Res Interpolation
    N_hi = 1000
    t_hi = np.linspace(0, 2*np.pi, N_hi, endpoint=False)
    
    ext_angles = np.concatenate([full_angles - 2*np.pi, full_angles, full_angles + 2*np.pi])
    ext_C = np.concatenate([smooth_C, smooth_C, smooth_C])
    ext_dJ = np.concatenate([smooth_dJ, smooth_dJ, smooth_dJ])
    
    interp_C = interp1d(ext_angles, ext_C, kind='cubic')
    interp_dJ = interp1d(ext_angles, ext_dJ, kind='cubic')
    
    vals_C = interp_C(t_hi)
    vals_dJ = interp_dJ(t_hi)
    
    J_hi = J_center + vals_dJ
    A_hi = vals_C * np.cos(t_hi)
    B_hi = vals_C * np.sin(t_hi)
    jz_hi = np.stack([J_hi, A_hi, B_hi], axis=-1)
    
    # Arc Length Reparameterization (Constant Delta E)
    diffs = jz_hi[(np.arange(N_hi) + 1) % N_hi] - jz_hi
    dists = np.linalg.norm(diffs, axis=1)
    cum_dist = np.concatenate([[0.0], np.cumsum(dists)])
    total_len = cum_dist[-1]
    
    target_dists = np.linspace(0, total_len, n_hues, endpoint=False)
    jz_final = np.zeros((n_hues, 3))
    
    for i, d in enumerate(target_dists):
        idx = np.searchsorted(cum_dist, d, side='right') - 1
        idx = np.clip(idx, 0, N_hi-1)
        d0 = cum_dist[idx]
        d1 = cum_dist[idx+1]
        alpha = (d - d0) / (d1 - d0) if d1 > d0 else 0
        p0 = jz_hi[idx]
        p1 = jz_hi[(idx+1)%N_hi]
        jz_final[i] = (1 - alpha) * p0 + alpha * p1
        
    rgb_final = np.clip(xyz_to_srgb_np(jzazbz_to_xyz_np(jz_final)), 0, 1)
    
    return jz_final, rgb_final

# ============================================================
# 3. Plotly Visualization Utils (Restored)
# ============================================================

def make_face_jz(face: str, n: int = 64):
    t = np.linspace(0.0, 1.0, n, dtype=float)
    u, v = np.meshgrid(t, t, indexing="ij")
    srgb = np.zeros((n, n, 3), dtype=float)

    if   face == "R0": srgb[...,0]=0.0; srgb[...,1]=u; srgb[...,2]=v
    elif face == "R1": srgb[...,0]=1.0; srgb[...,1]=u; srgb[...,2]=v
    elif face == "G0": srgb[...,1]=0.0; srgb[...,0]=u; srgb[...,2]=v
    elif face == "G1": srgb[...,1]=1.0; srgb[...,0]=u; srgb[...,2]=v
    elif face == "B0": srgb[...,2]=0.0; srgb[...,0]=u; srgb[...,1]=v
    elif face == "B1": srgb[...,2]=1.0; srgb[...,0]=u; srgb[...,1]=v
    else: raise ValueError("bad face name")

    xyz = srgb_to_xyz_np(srgb.reshape(-1,3)).reshape(n,n,3)
    jz = xyz_to_jzazbz_np(xyz.reshape(-1,3)).reshape(n,n,3)
    Jg, Az, Bz = jz[...,0], jz[...,1], jz[...,2]
    return Jg, Az, Bz

faces = ["R0","R1","G0","G1","B0","B1"]
face_surfaces_jz = {f: make_face_jz(f, n=64) for f in faces}

def plot_jz_trajectory_comparison_plotly(
    jz_sym: np.ndarray, rgb_sym: np.ndarray,
    jz_sine: np.ndarray, rgb_sine: np.ndarray,
    jz_hsv: np.ndarray, rgb_hsv: np.ndarray,
    center_val: float
):
    fig = go.Figure()

    # 1. Background: Gamut Faces
    for f in faces:
        Jg, Az, Bz = face_surfaces_jz[f]
        fig.add_trace(go.Surface(
            x=Jg, y=Az, z=Bz,
            surfacecolor=np.zeros_like(Jg),
            colorscale=[[0,"rgb(120,120,120)"],[1,"rgb(120,120,120)"]],
            showscale=False, opacity=0.1, hoverinfo="skip",
            lighting=dict(ambient=1.0, diffuse=0.0, specular=0.0),
            showlegend=False,
        ))

    # 2. Gamut Edges
    verts = np.array(list(itertools.product([0.0,1.0],[0.0,1.0],[0.0,1.0])), dtype=float)
    edges = [(i,j) for i in range(8) for j in range(i+1,8)
             if np.sum(np.abs(verts[i]-verts[j])) == 1.0]
    t_edge = np.linspace(0.0,1.0,64)
    for i,j in edges:
        p0 = verts[i]; p1 = verts[j]
        srgb_line = p0 + (p1-p0)[None,:] * t_edge[:,None]
        xyz_line = srgb_to_xyz_np(srgb_line)
        jz_line = xyz_to_jzazbz_np(xyz_line)
        fig.add_trace(go.Scatter3d(
            x=jz_line[:,0], y=jz_line[:,1], z=jz_line[:,2],
            mode="lines", line=dict(width=2, color="rgb(100,100,100)"),
            hoverinfo="skip", showlegend=False,
        ))

    # 3. Symmetric Ring
    fig.add_trace(go.Scatter3d(
        x=jz_sym[:,0], y=jz_sym[:,1], z=jz_sym[:,2],
        mode="lines+markers",
        line=dict(width=8, color="cyan"),
        marker=dict(size=4, color=rgb_sym),
        name=f"Lighter Symmetric (Center={center_val})",
    ))

    # 4. Sinebow
    fig.add_trace(go.Scatter3d(
        x=jz_sine[:,0], y=jz_sine[:,1], z=jz_sine[:,2],
        mode="lines",
        line=dict(width=5, color="#888888"),
        name="Sinebow",
    ))

    # 5. HSV
    fig.add_trace(go.Scatter3d(
        x=jz_hsv[:,0], y=jz_hsv[:,1], z=jz_hsv[:,2],
        mode="lines",
        line=dict(width=3, color="#444444", dash="dot"),
        name="HSV",
    ))

    gray = "rgb(200,200,200)"
    axis_style = dict(
        showbackground=False, gridcolor="rgba(200,200,200,0.25)",
        zerolinecolor="rgba(200,200,200,0.35)", tickfont=dict(color=gray),
    )

    fig.update_layout(
        width=1000, height=800,
        paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
        scene=dict(
            xaxis=dict(axis_style, title="Jz"),
            yaxis=dict(axis_style, title="Az"),
            zaxis=dict(axis_style, title="Bz"),
            aspectmode="data",
        ),
        title=dict(text="Lighter Symmetric Ring vs Sinebow vs HSV in JzAzBz", font=dict(color=gray, size=20)),
        legend=dict(font=dict(color=gray)),
    )
    return fig

# ============================================================
# 4. Image Generation Helpers
# ============================================================

def build_hue_disk_from_ring(rgb_ring_lin, center_lin, size=512):
    n_hues = rgb_ring_lin.shape[0]
    N = size
    y, x = np.mgrid[0:N, 0:N]
    x = (x + 0.5) / N * 2 - 1
    y = (y + 0.5) / N * 2 - 1
    r = np.sqrt(x * x + y * y)
    theta = np.mod(np.arctan2(y, x), 2 * np.pi)

    hue_f = theta / (2 * np.pi) * n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)

    ring0 = rgb_ring_lin[idx0]
    ring1 = rgb_ring_lin[idx1]
    ring_interp = (1 - t[..., None]) * ring0 + t[..., None] * ring1

    r_exp = r[..., None]
    rgb_lin = (1 - r_exp) * center_lin + r_exp * ring_interp
    rgb_lin[r > 1] = 1.0

    return np.clip(linear_to_srgb_np(rgb_lin), 0, 1)

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    n_hues = rgb_ring_lin.shape[0]
    disk = build_hue_disk_from_ring(rgb_ring_lin, center_lin, size=512)

    u = flows[0].cpu().numpy()
    v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2 * np.pi)
    mag = np.sqrt(u * u + v * v)
    mag /= (mag.max() + 1e-8)

    hue_f = angle / (2 * np.pi) * n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)

    ring0 = rgb_ring_lin[idx0]
    ring1 = rgb_ring_lin[idx1]
    ring_interp = (1 - t[..., None]) * ring0 + t[..., None] * ring1

    r = mag[..., None]
    rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow), 0, 1)

    alpha = mag[..., None]
    white = np.ones_like(flow_rgb)
    black = np.zeros_like(flow_rgb)
    flow_white = alpha * flow_rgb + (1 - alpha) * white
    flow_black = alpha * flow_rgb + (1 - alpha) * black

    return disk, flow_rgb, flow_white, flow_black

def find_reddest_index(rgb_ring_srgb):
    rgb = np.asarray(rgb_ring_srgb)
    score = rgb[:,0] - 0.5*(rgb[:,1] + rgb[:,2])
    return int(np.argmax(score))

def find_greenest_index(rgb_ring_srgb):
    rgb = np.asarray(rgb_ring_srgb)
    score = rgb[:,1] - 0.5*(rgb[:,0] + rgb[:,2])
    return int(np.argmax(score))

def align_ring_to_reference(rgb_ring_srgb, ref_green_idx):
    N = rgb_ring_srgb.shape[0]
    ring = np.asarray(rgb_ring_srgb)
    idx_red = find_reddest_index(ring)
    ring_r0 = np.roll(ring, -idx_red, axis=0)
    
    g1 = find_greenest_index(ring_r0)
    ring_rev = ring_r0[::-1].copy()
    g2 = find_greenest_index(ring_rev)
    
    def circ_dist(a, b):
        d = abs(a - b)
        return min(d, N - d)
    
    if circ_dist(g1, ref_green_idx) <= circ_dist(g2, ref_green_idx):
        return ring_r0
    else:
        return ring_rev

# ============================================================
# Main Execution Block
# ============================================================

if __name__ == "__main__":
    device = torch.device("cpu")
    N_HUES = 72
    CENTER_VAL = 0.68

    # 1. Symmetric Ring
    jz_sym_raw, rgb_sym_raw = build_lighter_symmetric_ring(N_HUES, center_val=CENTER_VAL)
    
    # 2. HSV (Reference)
    t = np.linspace(0, 1, N_HUES, endpoint=False)
    hsv_vals = np.stack([t, np.ones_like(t), np.ones_like(t)], axis=1)
    rgb_hsv_raw = hsv_to_rgb(hsv_vals)
    
    # 3. Sinebow
    sine_r = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0/3))
    sine_g = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1/3))
    sine_b = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2/3))
    rgb_sine_raw = np.stack([sine_r, sine_g, sine_b], axis=1)
    
    # 4. Alignment
    idx_red_h = find_reddest_index(rgb_hsv_raw)
    rgb_hsv_r0 = np.roll(rgb_hsv_raw, -idx_red_h, axis=0)
    g_h = find_greenest_index(rgb_hsv_r0)
    if g_h > N_HUES // 2:
        rgb_hsv_r0 = rgb_hsv_r0[::-1].copy()
        g_h = find_greenest_index(rgb_hsv_r0)
    rgb_hsv = rgb_hsv_r0
    
    rgb_sym = align_ring_to_reference(rgb_sym_raw, g_h)
    rgb_sine = align_ring_to_reference(rgb_sine_raw, g_h)
    
    # Recalculate Jz for plotting
    jz_sym = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_sym))
    jz_sine = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_sine))
    jz_hsv = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_hsv))
    
    # --------------------------------------------------------
    # Part B: Interactive 3D Trajectory Plot (Plotly)
    # --------------------------------------------------------
    fig = plot_jz_trajectory_comparison_plotly(
        jz_sym, rgb_sym,
        jz_sine, rgb_sine,
        jz_hsv, rgb_hsv,
        center_val=CENTER_VAL
    )
    # Use display(HTML(...)) if in a notebook, or just fig.show() locally
    try:
        display(HTML(pio.to_html(fig, include_plotlyjs="cdn", full_html=False)))
    except:
        fig.show()

    # --------------------------------------------------------
    # Part C: Omnipose Flow Generation
    # --------------------------------------------------------
    omnidir = Path(omnirefactor.__file__).parent.parent.parent
    basedir = os.path.join(omnidir, "docs", "_static")
    name = "ecoli"
    image = io.imread(os.path.join(basedir, name + ".tif"))
    masks = io.imread(os.path.join(basedir, name + "_labels.tif"))
    slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
    masks = fastremap.renumber(masks[slc])[0]
    flows = omnirefactor.core.masks_to_flows(masks, device="cpu")[4].to(device)
    
    center_lin = srgb_to_linear_np(np.array([CENTER_VAL, CENTER_VAL, CENTER_VAL]))
    rgb_sym_lin = srgb_to_linear_np(rgb_sym)
    rgb_sine_lin = srgb_to_linear_np(rgb_sine)
    rgb_hsv_lin = srgb_to_linear_np(rgb_hsv)
    
    datasets = [("Lighter Symmetric", rgb_sym_lin), ("Sinebow", rgb_sine_lin), ("HSV", rgb_hsv_lin)]
    shift = N_HUES // 4 
    
    fig, axes = plt.subplots(3, 6, figsize=(26, 8))
    for i, (title, ring_lin) in enumerate(datasets):
        disk, flow, flow_w, flow_b = make_flow_images_for_ring(ring_lin, center_lin, flows)
        ring_lin_rot = np.roll(ring_lin, shift, axis=0)
        _, _, flow_w_rot, flow_b_rot = make_flow_images_for_ring(ring_lin_rot, center_lin, flows)
        
        axes[i,0].imshow(disk);        axes[i,0].axis("off")
        axes[i,1].imshow(flow);        axes[i,1].axis("off")
        axes[i,2].imshow(flow_w);      axes[i,2].axis("off")
        axes[i,3].imshow(flow_b);      axes[i,3].axis("off")
        axes[i,4].imshow(flow_w_rot);  axes[i,4].axis("off")
        axes[i,5].imshow(flow_b_rot);  axes[i,5].axis("off")
        axes[i,0].set_ylabel(title, fontsize=12)
        
    axes[0,0].set_title("Hue disk"); axes[0,1].set_title("Flow Chroma")
    axes[0,2].set_title("White Bg"); axes[0,3].set_title("Black Bg")
    plt.tight_layout()
    plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from scipy.interpolate import interp1d
import os
from pathlib import Path

# Check for Omnipose
try:
    import omnirefactor
    from omnirefactor import io
    import fastremap
    OMNIPOSE_AVAILABLE = True
except ImportError:
    OMNIPOSE_AVAILABLE = False

torch.manual_seed(42)
np.random.seed(42)
dtype = torch.float32

# ============================================================
# 0. DEVICE
# ============================================================
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"🚀 Running on GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🚀 Running on Apple MPS (Metal)")
else:
    device = torch.device("cpu")
    print("⚠️ Running on CPU (GPU not found)")

# ============================================================
# 1. Oklab Color Math (Vectorized)
# ============================================================
M1 = torch.tensor([[0.8189330101, 0.3618667424, -0.1288597137],
                   [0.0329845436, 0.9293118715, 0.0361456387],
                   [0.0482003018, 0.2643662703, 0.6338517070]], dtype=dtype, device=device)
M2 = torch.tensor([[0.2104542553, 0.7936177850, -0.0040720468],
                   [1.9779984951, -2.4285922050, 0.4505937099],
                   [0.0259040371, 0.7827717662, -0.8086757660]], dtype=dtype, device=device)
M1_inv = torch.linalg.inv(M1)
M2_inv = torch.linalg.inv(M2)

class ColorConverter(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer('m1_inv', M1_inv)
        self.register_buffer('m2_inv', M2_inv)
        self.register_buffer('m1', M1)
        self.register_buffer('m2', M2)

    def oklab_to_linear_srgb(self, lab):
        lms_prime = lab @ self.m2_inv.T
        lms = torch.pow(lms_prime, 3)
        xyz = lms @ self.m1_inv.T
        r = 3.2404542*xyz[:,0] - 1.5371385*xyz[:,1] - 0.4985314*xyz[:,2]
        g = -0.9692660*xyz[:,0] + 1.8760108*xyz[:,1] + 0.0415560*xyz[:,2]
        b = 0.0556434*xyz[:,0] - 0.2040259*xyz[:,1] + 1.0572252*xyz[:,2]
        return torch.stack([r, g, b], dim=1)

    def linear_srgb_to_oklab(self, rgb):
        r = rgb[:,0]; g = rgb[:,1]; b = rgb[:,2]
        x = 0.4124564*r + 0.3575761*g + 0.1804375*b
        y = 0.2126729*r + 0.7151522*g + 0.0721750*b
        z = 0.0193339*r + 0.1191920*g + 0.9503041*b
        xyz = torch.stack([x,y,z], dim=1)
        lms = xyz @ self.m1.T
        lms = torch.clamp(lms, min=1e-8) 
        lms_prime = torch.pow(lms, 1.0/3.0)
        return lms_prime @ self.m2.T

converter = ColorConverter().to(device)

# --- NumPy Helpers ---
def linear_to_srgb_np(rgb):
    rgb = np.asarray(rgb, dtype=float); a = 0.055
    return np.where(rgb <= 0.0031308, 12.92 * rgb, (1 + a) * np.power(np.clip(rgb, 0, None), 1/2.4) - a)

def srgb_to_linear_np(rgb):
    rgb = np.asarray(rgb, dtype=float); a = 0.055
    return np.where(rgb <= 0.04045, rgb / 12.92, ((rgb + a) / (1 + a))**2.4)

# ============================================================
# 2. The Parametric Model (RepulsionLoop)
# ============================================================
class RepulsionLoop(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Lightness: Needs to be flexible to ride the roller coaster
        self.order_L = 4
        self.L_dc = nn.Parameter(torch.tensor(0.65, device=device)) 
        self.L_coeffs = nn.Parameter(torch.zeros((self.order_L, 2), device=device) * 0.02)
        
        # Chroma: Keep reasonable flex
        self.order_C = 3
        self.C_dc = nn.Parameter(torch.tensor(0.15, device=device)) 
        self.C_coeffs = nn.Parameter(torch.zeros((self.order_C, 2), device=device) * 0.02)

        with torch.no_grad():
             self.L_coeffs[0, 0] = 0.20 

    def forward(self, t):
        L = self.L_dc.expand_as(t); C = self.C_dc.expand_as(t)
        sin_t = torch.sin(t); cos_t = torch.cos(t)
        
        # Vectorized Fourier Sum
        for k in range(1, self.order_L + 1):
            L = L + self.L_coeffs[k-1,0]*torch.sin(k*t) + self.L_coeffs[k-1,1]*torch.cos(k*t)
            
        for k in range(1, self.order_C + 1):
            C = C + self.C_coeffs[k-1,0]*torch.sin(k*t) + self.C_coeffs[k-1,1]*torch.cos(k*t)
        
        C = torch.abs(C)
        a = C * cos_t
        b = C * sin_t
        return torch.stack([L, a, b], dim=1)

# ============================================================
# 3. Optimization (Hard Repulsion)
# ============================================================
def optimize_repulsion_loop(iterations=3000, device=device):
    model = RepulsionLoop().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.015)
    t_grid = torch.linspace(0, 2*np.pi, 256, device=device)
    
    print("⚡ Optimizing Repulsion Loop...")
    
    for i in range(iterations):
        optimizer.zero_grad(set_to_none=True)
        lab_out = model(t_grid)
        rgb_lin = converter.oklab_to_linear_srgb(lab_out)
        
        # 1. Gamut (Stable Barrier)
        k = 100.0 
        barrier_0 = F.softplus(-k * rgb_lin) / k
        barrier_1 = F.softplus(k * (rgb_lin - 1.0)) / k
        gamut_loss = torch.sum((barrier_0 + barrier_1)**2) * 2000.0
        
        # 2. Maximize Chroma
        C_vals = torch.norm(lab_out[:, 1:], dim=1)
        chroma_loss = -torch.mean(C_vals)
        
        # 3. Anti-Mud Floor
        L_vals = lab_out[:, 0]
        darkness_penalty = torch.sum(F.relu(0.35 - L_vals)**2) * 50.0
        
        # 4. HARD REPULSION (Pillar)
        # Target C > 0.14
        repulsion_loss = torch.sum(F.relu(0.14 - C_vals)**2) * 100.0
        
        # 5. Regularization
        reg_loss = 0.0
        for k in range(1, model.order_C + 1):
            reg_loss += (k**3) * torch.sum(model.C_coeffs[k-1]**2)
            
        total_loss = gamut_loss + chroma_loss + darkness_penalty + repulsion_loss + reg_loss
        
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        if i % 500 == 0:
            print(f"Iter {i:04d} | G: {gamut_loss.item():.2f} | Min C: {C_vals.min().item():.3f}")
            
    return model

# ============================================================
# 4. Vis Helpers
# ============================================================

def reparameterize_by_arclength(lab_points_np):
    dists = np.linalg.norm(lab_points_np - np.roll(lab_points_np, 1, axis=0), axis=1)
    cumulative = np.cumsum(dists); total = cumulative[-1]
    t_curr = np.concatenate(([0], cumulative / total))
    pts = np.vstack([lab_points_np[-1:], lab_points_np])
    iL = interp1d(t_curr, pts[:,0], kind='cubic')
    ia = interp1d(t_curr, pts[:,1], kind='cubic')
    ib = interp1d(t_curr, pts[:,2], kind='cubic')
    t_targ = np.linspace(0, 1, len(lab_points_np) + 1)[:-1]
    return np.stack([iL(t_targ), ia(t_targ), ib(t_targ)], axis=1)

def align_red(lab_path):
    lab_tensor = torch.from_numpy(lab_path).float().to(device)
    rgb_lin = converter.oklab_to_linear_srgb(lab_tensor)
    rgb_srgb = linear_to_srgb_np(rgb_lin.cpu().numpy())
    target = np.array([1.0, 0.0, 0.0])
    idx = np.argmin(np.linalg.norm(rgb_srgb - target, axis=1))
    return np.roll(lab_path, -idx, axis=0)

def get_srgb_wireframe():
    res = 20; t = torch.linspace(0, 1, res, device=device)
    def add(s, d): l = s.repeat(res, 1); l[:, d] = t; return l
    c = torch.tensor([[0,0,0],[1,0,0],[0,1,0],[0,0,1],[1,1,0],[1,0,1],[0,1,1],[1,1,1]], dtype=dtype, device=device)
    edges = []
    edges += [add(c[0],0), add(c[0],1), add(c[0],2), add(c[7],0), add(c[7],1), add(c[7],2)]
    edges += [add(c[1],1), add(c[1],2), add(c[2],0), add(c[2],2), add(c[3],0), add(c[3],1)]
    rgb = torch.cat(edges, dim=0)
    lab = converter.linear_srgb_to_oklab(rgb).cpu().numpy()
    out = []
    for i in range(0, len(lab), res):
        out.append(lab[i:i+res]); out.append(np.array([[None, None, None]]))
    return np.concatenate(out, axis=0)

def plot_3d(lab_path):
    lab_t = torch.from_numpy(lab_path).float().to(device)
    rgb_lin = converter.oklab_to_linear_srgb(lab_t)
    rgb_np = np.clip(linear_to_srgb_np(rgb_lin.cpu().numpy()), 0, 1) * 255
    colors = [f'rgb({r:.0f},{g:.0f},{b:.0f})' for r,g,b in rgb_np]
    gamut = get_srgb_wireframe()
    
    fig = go.Figure(data=[
        go.Scatter3d(x=gamut[:,1], y=gamut[:,2], z=gamut[:,0], mode='lines', 
                     line=dict(color='rgba(150,150,150,0.2)', width=2), name='sRGB', hoverinfo='none'),
        go.Scatter3d(x=lab_path[:,1], y=lab_path[:,2], z=lab_path[:,0], mode='markers', 
                     marker=dict(size=6, color=colors), name='Optimal',
                     hovertemplate='L: %{z:.4f}<br>a: %{x:.4f}<br>b: %{y:.4f}<extra></extra>')
    ])
    fig.update_layout(template='plotly_dark', title='Repulsion Loop', 
                      scene=dict(xaxis_title='a', yaxis_title='b', zaxis_title='L', aspectmode='data'),
                      margin=dict(l=0,r=0,b=0,t=40), height=700)
    try: import google.colab; pio.renderers.default = "colab"
    except: pio.renderers.default = "notebook_connected"
    fig.show()

# === THE OMNIPOSE VISUALIZATION HELPERS ===
def build_hue_disk_from_ring(rgb_ring_lin, center_lin, size=512):
    n_hues = rgb_ring_lin.shape[0]; N = size
    y, x = np.mgrid[0:N, 0:N]
    x = (x + 0.5)/N*2 - 1; y = (y + 0.5)/N*2 - 1
    r = np.sqrt(x*x + y*y); theta = np.mod(np.arctan2(y, x), 2*np.pi)
    
    hue_f = theta/(2*np.pi)*n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)
    
    ring0 = rgb_ring_lin[idx0]; ring1 = rgb_ring_lin[idx1]
    ring_interp = (1 - t[...,None])*ring0 + t[...,None]*ring1
    
    r_exp = r[...,None]
    rgb_lin = (1-r_exp)*center_lin + r_exp*ring_interp
    rgb_lin[r > 1] = 0.0 # Black outside disk
    
    return np.clip(linear_to_srgb_np(rgb_lin), 0, 1)

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    n_hues = rgb_ring_lin.shape[0]
    disk = build_hue_disk_from_ring(rgb_ring_lin, center_lin, size=512)
    
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi)
    mag = np.sqrt(u*u + v*v); mag /= (mag.max() + 1e-8)
    
    hue_f = angle/(2*np.pi)*n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)
    
    ring0 = rgb_ring_lin[idx0]; ring1 = rgb_ring_lin[idx1]
    ring_interp = (1 - t[...,None])*ring0 + t[...,None]*ring1
    
    r = mag[...,None]
    rgb_lin_flow = (1-r)*center_lin + r*ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow), 0, 1)
    
    alpha = mag[...,None]
    white = np.ones_like(flow_rgb)
    black = np.zeros_like(flow_rgb)
    flow_white = alpha*flow_rgb + (1-alpha)*white
    flow_black = alpha*flow_rgb + (1-alpha)*black
    
    return disk, flow_rgb, flow_white, flow_black

# ============================================================
# Main
# ============================================================
if __name__ == "__main__":
    # 1. Optimize (RepulsionLoop)
    model = optimize_repulsion_loop(device=device)
    
    # 2. Sample & Align
    with torch.no_grad():
        t = torch.linspace(0, 2*np.pi, 512, device=device)
        lab_out = model(t)
        lab_raw = lab_out.cpu().numpy()
        
    lab_balanced = reparameterize_by_arclength(lab_raw)
    lab_final = align_red(lab_balanced)
    plot_3d(lab_final)
    
    # 3. Omnipose Vis
    if OMNIPOSE_AVAILABLE:
        try:
            omnidir = Path(omnirefactor.__file__).parent.parent.parent
            basedir = os.path.join(omnidir, "docs", "_static")
            img_p = os.path.join(basedir, "ecoli.tif")
            msk_p = os.path.join(basedir, "ecoli_labels.tif")
            
            if os.path.exists(img_p):
                masks = io.imread(msk_p)
                slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
                masks = fastremap.renumber(masks[slc])[0]
                flows = omnirefactor.core.masks_to_flows(masks, device="cpu")[4].to("cpu")
                
                # Prepare Colors
                lab_t = torch.from_numpy(lab_final).float().to(device)
                rgb_lin = converter.oklab_to_linear_srgb(lab_t).cpu().numpy().clip(0,1)
                
                # Center Gray
                center_lin = srgb_to_linear_np(np.array([0.5, 0.5, 0.5]))
                
                # Rotations
                N_HUES = len(rgb_lin)
                maps = [
                    (rgb_lin, "Repulsion Loop"),
                    (np.roll(rgb_lin, N_HUES//3, 0), "Rot 1/3"),
                    (np.roll(rgb_lin, N_HUES//4, 0), "Rot 1/4")
                ]
                
                fig, axes = plt.subplots(3,4, figsize=(16,12))
                for i, (m, t) in enumerate(maps):
                    d, f, fw, fb = make_flow_images_for_ring(m, center_lin, flows)
                    axes[i,0].imshow(d); axes[i,1].imshow(f); axes[i,2].imshow(fw); axes[i,3].imshow(fb)
                    for ax in axes[i]: ax.axis('off')
                    axes[i,0].set_ylabel(t, fontsize=12)
                
                axes[0,0].set_title("Hue Disk")
                axes[0,1].set_title("Flow")
                axes[0,2].set_title("White BG")
                axes[0,3].set_title("Black BG")
                plt.tight_layout(); plt.show()
                
        except Exception as e: print(f"Vis Error: {e}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from scipy.interpolate import interp1d
import os
from pathlib import Path

# Optional imports
try:
    import omnirefactor
    from omnirefactor import io
    import fastremap
    OMNIPOSE_AVAILABLE = True
except ImportError:
    OMNIPOSE_AVAILABLE = False

torch.manual_seed(42)
np.random.seed(42)
dtype = torch.float32

# ============================================================
# 0. DEVICE
# ============================================================
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"🚀 Running on GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🚀 Running on Apple MPS (Metal)")
else:
    device = torch.device("cpu")
    print("⚠️ Running on CPU (GPU not found)")

# ============================================================
# 1. Oklab Color Math
# ============================================================
M1 = torch.tensor([[0.8189330101, 0.3618667424, -0.1288597137],
                   [0.0329845436, 0.9293118715, 0.0361456387],
                   [0.0482003018, 0.2643662703, 0.6338517070]], dtype=dtype, device=device)
M2 = torch.tensor([[0.2104542553, 0.7936177850, -0.0040720468],
                   [1.9779984951, -2.4285922050, 0.4505937099],
                   [0.0259040371, 0.7827717662, -0.8086757660]], dtype=dtype, device=device)
M1_inv = torch.linalg.inv(M1)
M2_inv = torch.linalg.inv(M2)

class ColorConverter(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer('m1_inv', M1_inv)
        self.register_buffer('m2_inv', M2_inv)
        self.register_buffer('m1', M1)
        self.register_buffer('m2', M2)

    def oklab_to_linear_srgb(self, lab):
        lms_prime = lab @ self.m2_inv.T
        lms = torch.pow(lms_prime, 3)
        xyz = lms @ self.m1_inv.T
        r = 3.2404542*xyz[:,0] - 1.5371385*xyz[:,1] - 0.4985314*xyz[:,2]
        g = -0.9692660*xyz[:,0] + 1.8760108*xyz[:,1] + 0.0415560*xyz[:,2]
        b = 0.0556434*xyz[:,0] - 0.2040259*xyz[:,1] + 1.0572252*xyz[:,2]
        return torch.stack([r, g, b], dim=1)

    def linear_srgb_to_oklab(self, rgb):
        r = rgb[:,0]; g = rgb[:,1]; b = rgb[:,2]
        x = 0.4124564*r + 0.3575761*g + 0.1804375*b
        y = 0.2126729*r + 0.7151522*g + 0.0721750*b
        z = 0.0193339*r + 0.1191920*g + 0.9503041*b
        xyz = torch.stack([x,y,z], dim=1)
        lms = xyz @ self.m1.T
        lms = torch.clamp(lms, min=1e-8) 
        lms_prime = torch.pow(lms, 1.0/3.0)
        return lms_prime @ self.m2.T

converter = ColorConverter().to(device)
P_WHITE = torch.tensor([[1.0, 0.0, 0.0]], dtype=dtype, device=device)
P_BLACK = torch.tensor([[0.0, 0.0, 0.0]], dtype=dtype, device=device)

# Helpers
def linear_to_srgb_np(rgb):
    rgb = np.asarray(rgb, dtype=float); a = 0.055
    return np.where(rgb <= 0.0031308, 12.92 * rgb, (1 + a) * np.power(np.clip(rgb, 0, None), 1/2.4) - a)

def srgb_to_linear_np(rgb):
    rgb = np.asarray(rgb, dtype=float); a = 0.055
    return np.where(rgb <= 0.04045, rgb / 12.92, ((rgb + a) / (1 + a))**2.4)

# ============================================================
# 2. The Star-Convex Model (Impossible to Knot)
# ============================================================
class StarConvexLoop(nn.Module):
    def __init__(self):
        super().__init__()
        
        # LIGHTNESS: Order 3 (Saddle)
        self.order_L = 3
        self.L_dc = nn.Parameter(torch.tensor(0.65, device=device)) 
        self.L_coeffs = nn.Parameter(torch.zeros((self.order_L, 2), device=device) * 0.01)
        
        # RADIUS: Order 3 (Allows lobes but forces valid topology)
        self.order_R = 3
        self.R_dc = nn.Parameter(torch.tensor(0.15, device=device)) 
        self.R_coeffs = nn.Parameter(torch.zeros((self.order_R, 2), device=device) * 0.01)

        # Init Tilt
        with torch.no_grad():
             self.L_coeffs[0, 0] = 0.15 

    def forward(self, t):
        L = self.L_dc.expand_as(t)
        R = self.R_dc.expand_as(t)
        
        sin_t = torch.sin(t); cos_t = torch.cos(t)
        
        # Fourier Series for Lightness
        for k in range(1, self.order_L + 1):
            L = L + self.L_coeffs[k-1,0]*torch.sin(k*t) + self.L_coeffs[k-1,1]*torch.cos(k*t)
            
        # Fourier Series for Radius
        for k in range(1, self.order_R + 1):
            R = R + self.R_coeffs[k-1,0]*torch.sin(k*t) + self.R_coeffs[k-1,1]*torch.cos(k*t)
        
        # Force Radius > 0
        R = F.softplus(R)
        
        # Convert to Cartesian (Star-Convex Constraint)
        a = R * cos_t
        b = R * sin_t
        
        return torch.stack([L, a, b], dim=1)

# ============================================================
# 3. Optimization (Weighted Salience)
# ============================================================
def optimize_star_salience(alpha_black=0.7, iterations=4000):
    alpha_white = 1.0 - alpha_black
    
    model = StarConvexLoop().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.015)
    t_grid = torch.linspace(0, 2*np.pi, 256, device=device)
    
    print(f"⚡ Optimizing Star-Convex Salience (Black Weight={alpha_black})...")
    
    for i in range(iterations):
        optimizer.zero_grad(set_to_none=True)
        lab_out = model(t_grid)
        rgb_lin = converter.oklab_to_linear_srgb(lab_out)
        
        # 1. Stable Barrier
        k = 100.0 
        barrier_0 = F.softplus(-k * rgb_lin) / k
        barrier_1 = F.softplus(k * (rgb_lin - 1.0)) / k
        gamut_loss = torch.sum((barrier_0 + barrier_1)**2) * 5000.0
        
        # 2. WEIGHTED SALIENCE
        d_black = torch.norm(lab_out - P_BLACK, dim=1)
        d_white = torch.norm(lab_out - P_WHITE, dim=1)
        
        # Metric
        salience = (d_black ** alpha_black) * (d_white ** alpha_white)
        
        # A. Maximize
        expansion_loss = -torch.mean(salience) * 500.0
        
        # B. Uniformity
        iso_loss = torch.var(salience) * 5000.0
        
        # 3. Safety Rails
        radii = torch.norm(lab_out[:, 1:], dim=1)
        repulsion_loss = torch.sum(F.relu(0.14 - radii)**2) * 100.0
        
        L_vals = lab_out[:, 0]
        floor_loss = torch.sum(F.relu(0.40 - L_vals)**2) * 100.0
        
        # 4. Stiffness
        reg_loss = (torch.sum(model.L_coeffs**2) + torch.sum(model.R_coeffs**2)) * 5.0
        
        total_loss = gamut_loss + expansion_loss + iso_loss + repulsion_loss + floor_loss + reg_loss
        
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        if i % 1000 == 0:
             print(f"Iter {i:04d} | Salience: {salience.mean().item():.4f}")
            
    return model

# ============================================================
# 4. Vis & Run
# ============================================================
def reparameterize_by_arclength(lab_points_np):
    dists = np.linalg.norm(lab_points_np - np.roll(lab_points_np, 1, axis=0), axis=1)
    cumulative = np.cumsum(dists); total = cumulative[-1]
    t_curr = np.concatenate(([0], cumulative / total))
    pts = np.vstack([lab_points_np[-1:], lab_points_np])
    iL = interp1d(t_curr, pts[:,0], kind='cubic')
    ia = interp1d(t_curr, pts[:,1], kind='cubic')
    ib = interp1d(t_curr, pts[:,2], kind='cubic')
    t_targ = np.linspace(0, 1, len(lab_points_np) + 1)[:-1]
    return np.stack([iL(t_targ), ia(t_targ), ib(t_targ)], axis=1)

def align_red(lab_path):
    lab_tensor = torch.from_numpy(lab_path).float().to(device)
    rgb_lin = converter.oklab_to_linear_srgb(lab_tensor)
    rgb_srgb = linear_to_srgb_np(rgb_lin.cpu().numpy())
    target = np.array([1.0, 0.0, 0.0])
    idx = np.argmin(np.linalg.norm(rgb_srgb - target, axis=1))
    return np.roll(lab_path, -idx, axis=0)

def get_srgb_wireframe():
    res = 20; t = torch.linspace(0, 1, res, device=device)
    def add(s, d): l = s.repeat(res, 1); l[:, d] = t; return l
    c = torch.tensor([[0,0,0],[1,0,0],[0,1,0],[0,0,1],[1,1,0],[1,0,1],[0,1,1],[1,1,1]], dtype=dtype, device=device)
    edges = []
    edges += [add(c[0],0), add(c[0],1), add(c[0],2), add(c[7],0), add(c[7],1), add(c[7],2)]
    edges += [add(c[1],1), add(c[1],2), add(c[2],0), add(c[2],2), add(c[3],0), add(c[3],1)]
    rgb = torch.cat(edges, dim=0)
    lab = converter.linear_srgb_to_oklab(rgb).cpu().numpy()
    out = []
    for i in range(0, len(lab), res):
        out.append(lab[i:i+res]); out.append(np.array([[None, None, None]]))
    return np.concatenate(out, axis=0)

def plot_3d_multi(results_list):
    data = []
    gamut = get_srgb_wireframe()
    data.append(go.Scatter3d(x=gamut[:,1], y=gamut[:,2], z=gamut[:,0], 
                             mode='lines', line=dict(color='rgba(150,150,150,0.2)', width=2), name='sRGB', hoverinfo='none'))
    
    colors = ['cyan', 'magenta', 'yellow']
    for i, res in enumerate(results_list):
        lab = res['lab']
        data.append(go.Scatter3d(
            x=lab[:,1], y=lab[:,2], z=lab[:,0],
            mode='lines', line=dict(width=6, color=colors[i % len(colors)]),
            name=f"a={res['a']}"
        ))
        
    layout = go.Layout(template='plotly_dark', title='Star-Convex Salience Loop', 
                      scene=dict(xaxis_title='a', yaxis_title='b', zaxis_title='L', aspectmode='data'),
                      margin=dict(l=0,r=0,b=0,t=40), height=700)
    try: import google.colab; pio.renderers.default = "colab"
    except: pio.renderers.default = "notebook_connected"
    fig = go.Figure(data=data, layout=layout)
    fig.show()

def make_flow_images(rgb_path, center, flows):
    n = len(rgb_path); N=512
    y,x = np.mgrid[0:N, 0:N]
    x = (x+0.5)/N*2-1; y = (y+0.5)/N*2-1
    r = np.sqrt(x*x+y*y); th = np.mod(np.arctan2(y,x), 2*np.pi)
    def map(ang, mag):
        idx0 = (ang * n / (2*np.pi)).astype(int) % n
        idx1 = (idx0 + 1) % n
        t = (ang * n / (2*np.pi)) % 1
        col = (1-t[...,None])*rgb_path[idx0] + t[...,None]*rgb_path[idx1]
        out = (1-mag[...,None])*center + mag[...,None]*col
        return np.clip(linear_to_srgb_np(out), 0, 1)
    disk = map(th, np.clip(r,0,1))
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    ang = np.mod(np.arctan2(v,u), 2*np.pi)
    mag = np.sqrt(u*u+v*v); mag /= (mag.max()+1e-8)
    flow = map(ang, mag)
    alpha = mag[...,None]
    fw = alpha*flow + (1-alpha)
    fb = alpha*flow
    return disk, flow, fw, fb

if __name__ == "__main__":
    alphas = [0.6, 0.7, 0.8] # Biased towards Black (Brightness)
    results = []
    
    for a in alphas:
        model = optimize_star_salience(alpha_black=a)
        with torch.no_grad():
            t = torch.linspace(0, 2*np.pi, 512, device=device)
            lab = model(t).cpu().numpy()
        
        lab_bal = reparameterize_by_arclength(lab)
        lab_final = align_red(lab_bal)
        
        lab_t = torch.from_numpy(lab_final).float().to(device)
        rgb_lin = converter.oklab_to_linear_srgb(lab_t).cpu().numpy().clip(0,1)
        
        results.append({'a': a, 'lab': lab_final, 'rgb_lin': rgb_lin})
    
    plot_3d_multi(results)
    
    if OMNIPOSE_AVAILABLE:
        try:
            omnidir = Path(omnirefactor.__file__).parent.parent.parent
            basedir = os.path.join(omnidir, "docs", "_static")
            img = io.imread(os.path.join(basedir, "ecoli.tif"))
            msk = io.imread(os.path.join(basedir, "ecoli_labels.tif"))
            slc = omnirefactor.measure.crop_bbox(msk>0, pad=0)[0]
            flows = omnirefactor.core.masks_to_flows(fastremap.renumber(msk[slc])[0], device="cpu")[4].to("cpu")
            center = srgb_to_linear_np(np.array([0.5,0.5,0.5]))
            
            fig, axes = plt.subplots(len(alphas), 4, figsize=(16, 4*len(alphas)))
            for i, res in enumerate(results):
                d, f, fw, fb = make_flow_images(res['rgb_lin'], center, flows)
                axes[i,0].imshow(d); axes[i,1].imshow(f); axes[i,2].imshow(fw); axes[i,3].imshow(fb)
                for ax in axes[i]: ax.axis('off')
                axes[i,0].set_ylabel(f"Salience (Black={res['a']})", fontsize=12, rotation=90, labelpad=20)
                if i == 0:
                    axes[i,0].set_title("Hue Disk")
                    axes[i,1].set_title("Flow")
                    axes[i,2].set_title("White BG")
                    axes[i,3].set_title("Black BG")
            plt.tight_layout(); plt.show()
            
        except Exception as e: print(e)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from scipy.interpolate import interp1d
import os
from pathlib import Path

# Check for Omnipose
try:
    import omnirefactor
    from omnirefactor import io
    import fastremap
    OMNIPOSE_AVAILABLE = True
except ImportError:
    OMNIPOSE_AVAILABLE = False

torch.manual_seed(42)
np.random.seed(42)
dtype = torch.float32

# ============================================================
# 0. DEVICE
# ============================================================
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"🚀 Running on GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🚀 Running on Apple MPS (Metal)")
else:
    device = torch.device("cpu")
    print("⚠️ Running on CPU (GPU not found)")

# ============================================================
# 1. Oklab Color Math
# ============================================================
M1 = torch.tensor([[0.8189330101, 0.3618667424, -0.1288597137],
                   [0.0329845436, 0.9293118715, 0.0361456387],
                   [0.0482003018, 0.2643662703, 0.6338517070]], dtype=dtype, device=device)
M2 = torch.tensor([[0.2104542553, 0.7936177850, -0.0040720468],
                   [1.9779984951, -2.4285922050, 0.4505937099],
                   [0.0259040371, 0.7827717662, -0.8086757660]], dtype=dtype, device=device)
M1_inv = torch.linalg.inv(M1)
M2_inv = torch.linalg.inv(M2)

class ColorConverter(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer('m1_inv', M1_inv)
        self.register_buffer('m2_inv', M2_inv)
        self.register_buffer('m1', M1)
        self.register_buffer('m2', M2)

    def oklab_to_linear_srgb(self, lab):
        lms_prime = lab @ self.m2_inv.T
        lms = torch.pow(lms_prime, 3)
        xyz = lms @ self.m1_inv.T
        r = 3.2404542*xyz[:,0] - 1.5371385*xyz[:,1] - 0.4985314*xyz[:,2]
        g = -0.9692660*xyz[:,0] + 1.8760108*xyz[:,1] + 0.0415560*xyz[:,2]
        b = 0.0556434*xyz[:,0] - 0.2040259*xyz[:,1] + 1.0572252*xyz[:,2]
        return torch.stack([r, g, b], dim=1)

    def linear_srgb_to_oklab(self, rgb):
        r = rgb[:,0]; g = rgb[:,1]; b = rgb[:,2]
        x = 0.4124564*r + 0.3575761*g + 0.1804375*b
        y = 0.2126729*r + 0.7151522*g + 0.0721750*b
        z = 0.0193339*r + 0.1191920*g + 0.9503041*b
        xyz = torch.stack([x,y,z], dim=1)
        lms = xyz @ self.m1.T
        lms = torch.clamp(lms, min=1e-8) 
        lms_prime = torch.pow(lms, 1.0/3.0)
        return lms_prime @ self.m2.T

converter = ColorConverter().to(device)

def linear_to_srgb_np(rgb):
    rgb = np.asarray(rgb, dtype=float); a = 0.055
    return np.where(rgb <= 0.0031308, 12.92 * rgb, (1 + a) * np.power(np.clip(rgb, 0, None), 1/2.4) - a)

def srgb_to_linear_np(rgb):
    rgb = np.asarray(rgb, dtype=float); a = 0.055
    return np.where(rgb <= 0.04045, rgb / 12.92, ((rgb + a) / (1 + a))**2.4)

# ============================================================
# 2. The Model (Stiff Wire)
# ============================================================
class HKWireLoop(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Lightness: Order 3 (Saddle)
        # We start at L=0.60
        self.order_L = 3
        self.L_dc = nn.Parameter(torch.tensor(0.60, device=device)) 
        self.L_coeffs = nn.Parameter(torch.zeros((self.order_L, 2), device=device) * 0.01)
        
        # Chroma: Order 2 (Oval)
        # NOTE: We use Order 2 to prevent the "Clover" lobes, keeping it round-ish.
        self.order_C = 2
        self.C_dc = nn.Parameter(torch.tensor(0.15, device=device)) 
        self.C_coeffs = nn.Parameter(torch.zeros((self.order_C, 2), device=device) * 0.01)

        # Init Tilt
        with torch.no_grad():
             self.L_coeffs[0, 0] = 0.15 

    def forward(self, t):
        L = self.L_dc.expand_as(t); C = self.C_dc.expand_as(t)
        sin_t = torch.sin(t); cos_t = torch.cos(t)
        
        for k in range(1, self.order_L + 1):
            L = L + self.L_coeffs[k-1,0]*torch.sin(k*t) + self.L_coeffs[k-1,1]*torch.cos(k*t)
        for k in range(1, self.order_C + 1):
            C = C + self.C_coeffs[k-1,0]*torch.sin(k*t) + self.C_coeffs[k-1,1]*torch.cos(k*t)
        
        C = torch.abs(C)
        a = C * cos_t; b = C * sin_t
        return torch.stack([L, a, b], dim=1)

# ============================================================
# 3. Optimization (H-K Balance)
# ============================================================
def optimize_hk_vibrant(hk_k=0.16, iterations=4000):
    model = HKWireLoop().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.015)
    t_grid = torch.linspace(0, 2*np.pi, 256, device=device)
    
    print(f"⚡ Optimizing H-K Vibrancy (k={hk_k})...")
    
    for i in range(iterations):
        optimizer.zero_grad(set_to_none=True)
        lab_out = model(t_grid)
        rgb_lin = converter.oklab_to_linear_srgb(lab_out)
        
        # 1. Stable Barrier
        k_bar = 100.0 
        barrier_0 = F.softplus(-k_bar * rgb_lin) / k_bar
        barrier_1 = F.softplus(k_bar * (rgb_lin - 1.0)) / k_bar
        gamut_loss = torch.sum((barrier_0 + barrier_1)**2) * 5000.0
        
        # 2. H-K METRIC
        L = lab_out[:, 0]
        C = torch.norm(lab_out[:, 1:], dim=1)
        
        # Perceived Brightness Q
        Q = L + (hk_k * C)
        
        # A. Iso-Brightness (Uniformity)
        # We want the *Perceived Brightness* to be constant around the ring.
        # This allows L to drop if C is high.
        iso_loss = torch.var(Q) * 20000.0
        
        # B. Expansion (Maximize Q)
        # Push the ring to be as "Bright+Colorful" as possible
        expansion_loss = -torch.mean(Q) * 500.0
        
        # 3. Repulsion (Safety)
        repulsion_loss = torch.sum(F.relu(0.14 - C)**2) * 100.0
        
        # 4. Stiffness
        reg_loss = (torch.sum(model.L_coeffs**2) + torch.sum(model.C_coeffs**2)) * 10.0
        
        total_loss = gamut_loss + iso_loss + expansion_loss + repulsion_loss + reg_loss
        
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        if i % 1000 == 0:
            print(f"Iter {i:04d} | Mean Q: {Q.mean().item():.4f} | Var Q: {iso_loss.item():.5f}")
            
    return model

# ============================================================
# 4. Vis & Run
# ============================================================
def reparameterize_by_arclength(lab_points_np):
    dists = np.linalg.norm(lab_points_np - np.roll(lab_points_np, 1, axis=0), axis=1)
    cumulative = np.cumsum(dists); total = cumulative[-1]
    t_curr = np.concatenate(([0], cumulative / total))
    pts = np.vstack([lab_points_np[-1:], lab_points_np])
    iL = interp1d(t_curr, pts[:,0], kind='cubic')
    ia = interp1d(t_curr, pts[:,1], kind='cubic')
    ib = interp1d(t_curr, pts[:,2], kind='cubic')
    t_targ = np.linspace(0, 1, len(lab_points_np) + 1)[:-1]
    return np.stack([iL(t_targ), ia(t_targ), ib(t_targ)], axis=1)

def align_red(lab_path):
    lab_tensor = torch.from_numpy(lab_path).float().to(device)
    rgb_lin = converter.oklab_to_linear_srgb(lab_tensor)
    rgb_srgb = linear_to_srgb_np(rgb_lin.cpu().numpy())
    target = np.array([1.0, 0.0, 0.0])
    idx = np.argmin(np.linalg.norm(rgb_srgb - target, axis=1))
    return np.roll(lab_path, -idx, axis=0)

def get_srgb_wireframe():
    res = 20; t = torch.linspace(0, 1, res, device=device)
    def add(s, d): l = s.repeat(res, 1); l[:, d] = t; return l
    c = torch.tensor([[0,0,0],[1,0,0],[0,1,0],[0,0,1],[1,1,0],[1,0,1],[0,1,1],[1,1,1]], dtype=dtype, device=device)
    edges = []
    edges += [add(c[0],0), add(c[0],1), add(c[0],2), add(c[7],0), add(c[7],1), add(c[7],2)]
    edges += [add(c[1],1), add(c[1],2), add(c[2],0), add(c[2],2), add(c[3],0), add(c[3],1)]
    rgb = torch.cat(edges, dim=0)
    lab = converter.linear_srgb_to_oklab(rgb).cpu().numpy()
    out = []
    for i in range(0, len(lab), res):
        out.append(lab[i:i+res]); out.append(np.array([[None, None, None]]))
    return np.concatenate(out, axis=0)

def plot_3d_multi(results_list):
    data = []
    gamut = get_srgb_wireframe()
    data.append(go.Scatter3d(x=gamut[:,1], y=gamut[:,2], z=gamut[:,0], 
                             mode='lines', line=dict(color='rgba(150,150,150,0.2)', width=2), name='sRGB', hoverinfo='none'))
    
    colors = ['cyan', 'magenta', 'yellow']
    for i, res in enumerate(results_list):
        lab = res['lab']
        data.append(go.Scatter3d(
            x=lab[:,1], y=lab[:,2], z=lab[:,0],
            mode='lines', line=dict(width=6, color=colors[i % len(colors)]),
            name=f"H-K k={res['k']}"
        ))

    layout = go.Layout(template='plotly_dark', title='H-K Iso-Brightness Optimization', 
                      scene=dict(xaxis_title='a', yaxis_title='b', zaxis_title='L', aspectmode='data'),
                      margin=dict(l=0,r=0,b=0,t=40), height=700)
    try: import google.colab; pio.renderers.default = "colab"
    except: pio.renderers.default = "notebook_connected"
    fig = go.Figure(data=data, layout=layout)
    fig.show()

def make_flow_images(rgb_path, center, flows):
    n = len(rgb_path); N=512
    y,x = np.mgrid[0:N, 0:N]
    x = (x+0.5)/N*2-1; y = (y+0.5)/N*2-1
    r = np.sqrt(x*x+y*y); th = np.mod(np.arctan2(y,x), 2*np.pi)
    def map(ang, mag):
        idx0 = (ang * n / (2*np.pi)).astype(int) % n
        idx1 = (idx0 + 1) % n
        t = (ang * n / (2*np.pi)) % 1
        col = (1-t[...,None])*rgb_path[idx0] + t[...,None]*rgb_path[idx1]
        out = (1-mag[...,None])*center + mag[...,None]*col
        return np.clip(linear_to_srgb_np(out), 0, 1)
    disk = map(th, np.clip(r,0,1))
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    ang = np.mod(np.arctan2(v,u), 2*np.pi)
    mag = np.sqrt(u*u+v*v); mag /= (mag.max()+1e-8)
    flow = map(ang, mag)
    alpha = mag[...,None]
    fw = alpha*flow + (1-alpha)
    fb = alpha*flow
    return disk, flow, fw, fb

if __name__ == "__main__":
    # Sweep reasonable H-K factors
    # k=0.14 is standard, k=0.20 is high adaptation (more vibrancy weight)
    factors = [0.14, 0.17, 0.20]
    results = []
    
    for k in factors:
        model = optimize_hk_vibrant(hk_k=k)
        with torch.no_grad():
            t = torch.linspace(0, 2*np.pi, 512, device=device)
            lab = model(t).cpu().numpy()
        
        lab_bal = reparameterize_by_arclength(lab)
        lab_final = align_red(lab_bal)
        
        lab_t = torch.from_numpy(lab_final).float().to(device)
        rgb_lin = converter.oklab_to_linear_srgb(lab_t).cpu().numpy().clip(0,1)
        
        results.append({'k': k, 'lab': lab_final, 'rgb_lin': rgb_lin})
    
    plot_3d_multi(results)
    
    if OMNIPOSE_AVAILABLE:
        try:
            omnidir = Path(omnirefactor.__file__).parent.parent.parent
            basedir = os.path.join(omnidir, "docs", "_static")
            img = io.imread(os.path.join(basedir, "ecoli.tif"))
            msk = io.imread(os.path.join(basedir, "ecoli_labels.tif"))
            slc = omnirefactor.measure.crop_bbox(msk>0, pad=0)[0]
            flows = omnirefactor.core.masks_to_flows(fastremap.renumber(msk[slc])[0], device="cpu")[4].to("cpu")
            
            # Use standard Gray for display background reference
            center = srgb_to_linear_np(np.array([0.5,0.5,0.5]))
            
            fig, axes = plt.subplots(len(factors), 4, figsize=(16, 4*len(factors)))
            # Handle single row case if needed, though we have 3
            axes = np.atleast_2d(axes)
            
            for i, res in enumerate(results):
                d, f, fw, fb = make_flow_images(res['rgb_lin'], center, flows)
                axes[i,0].imshow(d); axes[i,1].imshow(f); axes[i,2].imshow(fw); axes[i,3].imshow(fb)
                for ax in axes[i]: ax.axis('off')
                axes[i,0].set_ylabel(f"H-K (k={res['k']})", fontsize=12, rotation=90, labelpad=20)
                
                if i == 0:
                    axes[i,0].set_title("Hue Disk")
                    axes[i,1].set_title("Flow")
                    axes[i,2].set_title("White BG")
                    axes[i,3].set_title("Black BG")
            
            plt.tight_layout(); plt.show()
            
        except Exception as e: print(e)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from scipy.interpolate import interp1d
import os
from pathlib import Path

# Optional imports for Flow Vis
try:
    import omnirefactor
    from omnirefactor import io
    import fastremap
    OMNIPOSE_AVAILABLE = True
except ImportError:
    OMNIPOSE_AVAILABLE = False

torch.manual_seed(42)
np.random.seed(42)
dtype = torch.float32

# ============================================================
# 0. DEVICE CONFIGURATION
# ============================================================
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"🚀 Running on GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🚀 Running on Apple MPS (Metal)")
else:
    device = torch.device("cpu")
    print("⚠️ Running on CPU (GPU not found)")

# ============================================================
# 1. Oklab Color Math (Fixed Class)
# ============================================================
M1 = torch.tensor([[0.8189330101, 0.3618667424, -0.1288597137],
                   [0.0329845436, 0.9293118715, 0.0361456387],
                   [0.0482003018, 0.2643662703, 0.6338517070]], dtype=dtype, device=device)
M2 = torch.tensor([[0.2104542553, 0.7936177850, -0.0040720468],
                   [1.9779984951, -2.4285922050, 0.4505937099],
                   [0.0259040371, 0.7827717662, -0.8086757660]], dtype=dtype, device=device)
M1_inv = torch.linalg.inv(M1)
M2_inv = torch.linalg.inv(M2)

class ColorConverter(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer('m1_inv', M1_inv)
        self.register_buffer('m2_inv', M2_inv)
        self.register_buffer('m1', M1)
        self.register_buffer('m2', M2)

    def oklab_to_linear_srgb(self, lab):
        lms_prime = lab @ self.m2_inv.T
        lms = torch.pow(lms_prime, 3)
        xyz = lms @ self.m1_inv.T
        r = 3.2404542*xyz[:,0] - 1.5371385*xyz[:,1] - 0.4985314*xyz[:,2]
        g = -0.9692660*xyz[:,0] + 1.8760108*xyz[:,1] + 0.0415560*xyz[:,2]
        b = 0.0556434*xyz[:,0] - 0.2040259*xyz[:,1] + 1.0572252*xyz[:,2]
        return torch.stack([r, g, b], dim=1)

    # --- FIX: Method Added ---
    def linear_srgb_to_oklab(self, rgb):
        r = rgb[:,0]; g = rgb[:,1]; b = rgb[:,2]
        x = 0.4124564*r + 0.3575761*g + 0.1804375*b
        y = 0.2126729*r + 0.7151522*g + 0.0721750*b
        z = 0.0193339*r + 0.1191920*g + 0.9503041*b
        xyz = torch.stack([x,y,z], dim=1)
        lms = xyz @ self.m1.T
        lms = torch.clamp(lms, min=1e-8) 
        lms_prime = torch.pow(lms, 1.0/3.0)
        return lms_prime @ self.m2.T

converter = ColorConverter().to(device)

def linear_to_srgb_np(rgb):
    rgb = np.asarray(rgb, dtype=float); a = 0.055
    return np.where(rgb <= 0.0031308, 12.92 * rgb, (1 + a) * np.power(np.clip(rgb, 0, None), 1/2.4) - a)

def srgb_to_linear_np(rgb):
    rgb = np.asarray(rgb, dtype=float); a = 0.055
    return np.where(rgb <= 0.04045, rgb / 12.92, ((rgb + a) / (1 + a))**2.4)

def linear_to_gamma_srgb(rgb_lin):
    mask = rgb_lin > 0.0031308
    rgb_gamma = torch.zeros_like(rgb_lin)
    rgb_gamma[mask] = 1.055 * torch.pow(rgb_lin[mask], 1/2.4) - 0.055
    rgb_gamma[~mask] = 12.92 * rgb_lin[~mask]
    return torch.clamp(rgb_gamma, 0.0, 1.0)

# ============================================================
# 2. The Phase-Locked Model
# ============================================================
class PhaseLockedLoop(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Lightness: Order 3 (Flexible Saddle)
        self.order_L = 3
        self.L_dc = nn.Parameter(torch.tensor(0.65, device=device)) 
        self.L_coeffs = nn.Parameter(torch.zeros((self.order_L, 2), device=device) * 0.01)
        
        # Chroma: Order 2 (Strict Ellipse - No Kinks)
        self.order_C = 2
        self.C_dc = nn.Parameter(torch.tensor(0.15, device=device)) 
        self.C_coeffs = nn.Parameter(torch.zeros((self.order_C, 2), device=device) * 0.01)

        with torch.no_grad():
             self.L_coeffs[0, 0] = 0.20 

    def forward(self, t):
        L = self.L_dc.expand_as(t); C = self.C_dc.expand_as(t)
        sin_t = torch.sin(t); cos_t = torch.cos(t)
        
        for k in range(1, self.order_L + 1):
            L = L + self.L_coeffs[k-1,0]*torch.sin(k*t) + self.L_coeffs[k-1,1]*torch.cos(k*t)
        for k in range(1, self.order_C + 1):
            C = C + self.C_coeffs[k-1,0]*torch.sin(k*t) + self.C_coeffs[k-1,1]*torch.cos(k*t)
        
        C = torch.abs(C)
        a = C * cos_t; b = C * sin_t
        return torch.stack([L, a, b], dim=1)

# ============================================================
# 3. Optimization (Opponent Phase Locking)
# ============================================================
def optimize_opponent_phase(iterations=4000, device=device):
    model = PhaseLockedLoop().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.015)
    t_grid = torch.linspace(0, 2*np.pi, 256, device=device)
    
    print("⚡ Optimizing Opponent Phase-Locked Loop...")
    
    for i in range(iterations):
        optimizer.zero_grad(set_to_none=True)
        lab_out = model(t_grid)
        rgb_lin = converter.oklab_to_linear_srgb(lab_out)
        
        # 1. Concrete Gamut Wall
        k = 500.0 
        barrier_0 = F.softplus(-k * rgb_lin) / k
        barrier_1 = F.softplus(k * (rgb_lin - 1.0)) / k
        gamut_loss = torch.sum((barrier_0 + barrier_1)**2) * 5000.0
        # Max Violation Penalty
        viol_max = torch.max(F.relu(-rgb_lin) + F.relu(rgb_lin - 1.0))
        gamut_loss += viol_max * 1_000_000.0
        
        # 2. OPPONENT PHASE LOCKING
        L = lab_out[:, 0]
        a = lab_out[:, 1] # Red-Green
        b = lab_out[:, 2] # Blue-Yellow
        
        # A. Lock L to b (Maximize Correlation)
        # Lightness should track Yellow (+b) and Blue (-b)
        L_cent = L - L.mean()
        b_cent = b - b.mean()
        cov_Lb = torch.mean(L_cent * b_cent)
        phase_lock_loss = -cov_Lb * 5000.0 
        
        # B. Decouple L from a (Minimize Correlation)
        # Red and Green should have roughly equal lightness
        a_cent = a - a.mean()
        cov_La = torch.mean(L_cent * a_cent)
        ortho_loss = (cov_La**2) * 10000.0
        
        # 3. Expansion
        C = torch.norm(lab_out[:, 1:], dim=1)
        expansion_loss = -torch.mean(C) * 500.0
        
        # 4. Safety Rails
        floor_loss = torch.sum(F.relu(0.35 - L)**2) * 100.0
        repulsion_loss = torch.sum(F.relu(0.14 - C)**2) * 100.0
        reg_loss = (torch.sum(model.L_coeffs**2) + torch.sum(model.C_coeffs**2)) * 5.0
        
        total_loss = gamut_loss + phase_lock_loss + ortho_loss + expansion_loss + floor_loss + repulsion_loss + reg_loss
        
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        if i % 1000 == 0:
            print(f"Iter {i:04d} | Cov(L,b): {cov_Lb.item():.4f} | Min C: {C.min().item():.3f}")
            
    return model

# ============================================================
# 4. Visualization Helpers
# ============================================================

def reparameterize_by_arclength(lab_points_np):
    dists = np.linalg.norm(lab_points_np - np.roll(lab_points_np, 1, axis=0), axis=1)
    cumulative = np.cumsum(dists); total = cumulative[-1]
    t_curr = np.concatenate(([0], cumulative / total))
    pts = np.vstack([lab_points_np[-1:], lab_points_np])
    iL = interp1d(t_curr, pts[:,0], kind='cubic')
    ia = interp1d(t_curr, pts[:,1], kind='cubic')
    ib = interp1d(t_curr, pts[:,2], kind='cubic')
    t_targ = np.linspace(0, 1, len(lab_points_np) + 1)[:-1]
    return np.stack([iL(t_targ), ia(t_targ), ib(t_targ)], axis=1)

def align_red(lab_path):
    lab_tensor = torch.from_numpy(lab_path).float().to(device)
    rgb_lin = converter.oklab_to_linear_srgb(lab_tensor)
    rgb_srgb = linear_to_srgb_np(rgb_lin.cpu().numpy())
    target = np.array([1.0, 0.0, 0.0])
    idx = np.argmin(np.linalg.norm(rgb_srgb - target, axis=1))
    return np.roll(lab_path, -idx, axis=0)

def get_srgb_wireframe():
    res = 20; t = torch.linspace(0, 1, res, device=device)
    def add(s, d): l = s.repeat(res, 1); l[:, d] = t; return l
    c = torch.tensor([[0,0,0],[1,0,0],[0,1,0],[0,0,1],[1,1,0],[1,0,1],[0,1,1],[1,1,1]], dtype=dtype, device=device)
    edges = []
    edges += [add(c[0],0), add(c[0],1), add(c[0],2), add(c[7],0), add(c[7],1), add(c[7],2)]
    edges += [add(c[1],1), add(c[1],2), add(c[2],0), add(c[2],2), add(c[3],0), add(c[3],1)]
    rgb = torch.cat(edges, dim=0)
    # FIXED: Using method on converter instance
    lab = converter.linear_srgb_to_oklab(rgb).cpu().numpy()
    out = []
    for i in range(0, len(lab), res):
        out.append(lab[i:i+res]); out.append(np.array([[None, None, None]]))
    return np.concatenate(out, axis=0)

def plot_3d(lab_path):
    lab_t = torch.from_numpy(lab_path).float().to(device)
    rgb_lin = converter.oklab_to_linear_srgb(lab_t)
    rgb_np = np.clip(linear_to_srgb_np(rgb_lin.cpu().numpy()), 0, 1) * 255
    colors = [f'rgb({r:.0f},{g:.0f},{b:.0f})' for r,g,b in rgb_np]
    gamut = get_srgb_wireframe()
    
    fig = go.Figure(data=[
        go.Scatter3d(x=gamut[:,1], y=gamut[:,2], z=gamut[:,0], mode='lines', 
                     line=dict(color='rgba(150,150,150,0.2)', width=2), name='sRGB', hoverinfo='none'),
        go.Scatter3d(x=lab_path[:,1], y=lab_path[:,2], z=lab_path[:,0], mode='markers', 
                     marker=dict(size=6, color=colors), name='Optimal',
                     hovertemplate='L: %{z:.4f}<br>a: %{x:.4f}<br>b: %{y:.4f}<extra></extra>')
    ])
    fig.update_layout(template='plotly_dark', title='Opponent Phase-Locked Loop', 
                      scene=dict(xaxis_title='a', yaxis_title='b', zaxis_title='L', aspectmode='data'),
                      margin=dict(l=0,r=0,b=0,t=40), height=700)
    try: import google.colab; pio.renderers.default = "colab"
    except: pio.renderers.default = "notebook_connected"
    fig.show()

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    n_hues = rgb_ring_lin.shape[0]; N=512
    y, x = np.mgrid[0:N, 0:N]
    x = (x + 0.5)/N*2 - 1; y = (y + 0.5)/N*2 - 1
    r = np.sqrt(x*x + y*y); theta = np.mod(np.arctan2(y, x), 2*np.pi)
    
    def apply_cmap(angle_norm, mag_norm):
        hue_f = angle_norm * n_hues
        idx0 = np.floor(hue_f).astype(int) % n_hues
        idx1 = (idx0 + 1) % n_hues
        t = hue_f - np.floor(hue_f)
        ring_interp = (1 - t[...,None])*rgb_ring_lin[idx0] + t[...,None]*rgb_ring_lin[idx1]
        r = mag_norm[..., None]
        rgb = (1 - r) * center_lin + r * ring_interp
        return np.clip(linear_to_srgb_np(rgb), 0, 1)

    disk = apply_cmap(theta / (2*np.pi), np.clip(r, 0, 1))
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi) / (2*np.pi)
    mag = np.sqrt(u*u + v*v); mag /= (mag.max() + 1e-8)
    flow_rgb = apply_cmap(angle, mag)
    alpha = mag[...,None]
    fw = alpha*flow_rgb + (1-alpha)
    fb = alpha*flow_rgb
    return disk, flow_rgb, fw, fb

if __name__ == "__main__":
    # 1. Optimize (Opponent Phase)
    model = optimize_opponent_phase(device=device)
    
    # 2. Sample & Align
    with torch.no_grad():
        t = torch.linspace(0, 2*np.pi, 512, device=device)
        lab_out = model(t)
        lab_raw = lab_out.cpu().numpy()
        
    lab_balanced = reparameterize_by_arclength(lab_raw)
    lab_final = align_red(lab_balanced)
    plot_3d(lab_final)
    
    # 3. Omnipose Vis
    if OMNIPOSE_AVAILABLE:
        try:
            omnidir = Path(omnirefactor.__file__).parent.parent.parent
            basedir = os.path.join(omnidir, "docs", "_static")
            img = io.imread(os.path.join(basedir, "ecoli.tif"))
            msk = io.imread(os.path.join(basedir, "ecoli_labels.tif"))
            slc = omnirefactor.measure.crop_bbox(msk>0, pad=0)[0]
            flows = omnirefactor.core.masks_to_flows(fastremap.renumber(msk[slc])[0], device="cpu")[4].to("cpu")
            center = srgb_to_linear_np(np.array([0.5,0.5,0.5]))
            
            lab_t = torch.from_numpy(lab_final).float().to(device)
            rgb_lin = converter.oklab_to_linear_srgb(lab_t).cpu().numpy().clip(0,1)
            N_HUES = len(rgb_lin)
            maps = [
                (rgb_lin, "Opponent Phase-Locked"),
                (np.roll(rgb_lin, N_HUES//3, 0), "Rot 1/3"),
                (np.roll(rgb_lin, N_HUES//4, 0), "Rot 1/4")
            ]
            
            fig, axes = plt.subplots(3,4, figsize=(16,12))
            # Axes safety
            axes = np.atleast_2d(axes)
            
            for i, (m, t) in enumerate(maps):
                d, f, fw, fb = make_flow_images_for_ring(m, center, flows)
                axes[i,0].imshow(d); axes[i,1].imshow(f); axes[i,2].imshow(fw); axes[i,3].imshow(fb)
                for ax in axes[i]: ax.axis('off')
                axes[i,0].set_ylabel(t, fontsize=12)
            
            axes[0,0].set_title("Hue Disk")
            axes[0,1].set_title("Flow")
            axes[0,2].set_title("White BG")
            axes[0,3].set_title("Black BG")
            plt.tight_layout(); plt.show()
        except Exception as e: print(f"Vis Error: {e}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from scipy.interpolate import interp1d
import os
from pathlib import Path

# Optional imports
try:
    import omnirefactor
    from omnirefactor import io
    import fastremap
    OMNIPOSE_AVAILABLE = True
except ImportError:
    OMNIPOSE_AVAILABLE = False

torch.manual_seed(42)
np.random.seed(42)
dtype = torch.float32

# ============================================================
# 0. DEVICE
# ============================================================
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"🚀 Running on GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🚀 Running on Apple MPS (Metal)")
else:
    device = torch.device("cpu")
    print("⚠️ Running on CPU (GPU not found)")

# ============================================================
# 1. Oklab Color Math
# ============================================================
M1 = torch.tensor([[0.8189330101, 0.3618667424, -0.1288597137],
                   [0.0329845436, 0.9293118715, 0.0361456387],
                   [0.0482003018, 0.2643662703, 0.6338517070]], dtype=dtype, device=device)
M2 = torch.tensor([[0.2104542553, 0.7936177850, -0.0040720468],
                   [1.9779984951, -2.4285922050, 0.4505937099],
                   [0.0259040371, 0.7827717662, -0.8086757660]], dtype=dtype, device=device)
M1_inv = torch.linalg.inv(M1)
M2_inv = torch.linalg.inv(M2)

class ColorConverter(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer('m1_inv', M1_inv)
        self.register_buffer('m2_inv', M2_inv)
        self.register_buffer('m1', M1)
        self.register_buffer('m2', M2)

    def oklab_to_linear_srgb(self, lab):
        lms_prime = lab @ self.m2_inv.T
        lms = torch.pow(lms_prime, 3)
        xyz = lms @ self.m1_inv.T
        r = 3.2404542*xyz[:,0] - 1.5371385*xyz[:,1] - 0.4985314*xyz[:,2]
        g = -0.9692660*xyz[:,0] + 1.8760108*xyz[:,1] + 0.0415560*xyz[:,2]
        b = 0.0556434*xyz[:,0] - 0.2040259*xyz[:,1] + 1.0572252*xyz[:,2]
        return torch.stack([r, g, b], dim=1)

    def linear_srgb_to_oklab(self, rgb):
        r = rgb[:,0]; g = rgb[:,1]; b = rgb[:,2]
        x = 0.4124564*r + 0.3575761*g + 0.1804375*b
        y = 0.2126729*r + 0.7151522*g + 0.0721750*b
        z = 0.0193339*r + 0.1191920*g + 0.9503041*b
        xyz = torch.stack([x,y,z], dim=1)
        lms = xyz @ self.m1.T
        lms = torch.clamp(lms, min=1e-8) 
        lms_prime = torch.pow(lms, 1.0/3.0)
        return lms_prime @ self.m2.T

converter = ColorConverter().to(device)

def linear_to_srgb_np(rgb):
    rgb = np.asarray(rgb, dtype=float); a = 0.055
    return np.where(rgb <= 0.0031308, 12.92 * rgb, (1 + a) * np.power(np.clip(rgb, 0, None), 1/2.4) - a)

def srgb_to_linear_np(rgb):
    rgb = np.asarray(rgb, dtype=float); a = 0.055
    return np.where(rgb <= 0.04045, rgb / 12.92, ((rgb + a) / (1 + a))**2.4)

# ============================================================
# 2. The HK-Elastic Model
# ============================================================
class HKElasticLoop(nn.Module):
    def __init__(self):
        super().__init__()
        
        # LIGHTNESS: Order 3 Fourier (Flexible Saddle)
        # Allows it to dip/rise to find the gamut fit.
        self.order_L = 3
        self.L_dc = nn.Parameter(torch.tensor(0.65, device=device)) 
        self.L_coeffs = nn.Parameter(torch.zeros((self.order_L, 2), device=device) * 0.01)
        
        # CHROMA: Single Scalar Radius (Strict Circle)
        # This forces Red/Green to be just as saturated as Blue/Yellow
        self.radius = nn.Parameter(torch.tensor(0.15, device=device)) 
        
        # PHASE: A learnable rotation for the Chroma circle
        # Allows the circle to rotate to align with the best gamut fit
        self.phase = nn.Parameter(torch.tensor(0.0, device=device))

        # Init Tilt (Start with a reasonable slope)
        with torch.no_grad():
             self.L_coeffs[0, 0] = 0.15 

    def forward(self, t):
        L = self.L_dc.expand_as(t)
        
        # Scalar Radius
        R = F.softplus(self.radius).expand_as(t)
        
        # Fourier L
        for k in range(1, self.order_L + 1):
            L = L + self.L_coeffs[k-1,0]*torch.sin(k*t) + self.L_coeffs[k-1,1]*torch.cos(k*t)
            
        # Rotated Circle for a/b
        theta = t + self.phase
        a = R * torch.cos(theta)
        b = R * torch.sin(theta)
        
        return torch.stack([L, a, b], dim=1)

# ============================================================
# 3. Optimization
# ============================================================
def optimize_hk_elastic(hk_k=0.20, iterations=4000):
    model = HKElasticLoop().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.015)
    t_grid = torch.linspace(0, 2*np.pi, 256, device=device)
    
    print(f"⚡ Optimizing HK-Elastic Circle (k={hk_k})...")
    
    for i in range(iterations):
        optimizer.zero_grad(set_to_none=True)
        lab_out = model(t_grid)
        rgb_lin = converter.oklab_to_linear_srgb(lab_out)
        
        # 1. Stable Gamut Wall
        k_bar = 100.0 
        barrier = F.softplus(-k_bar * rgb_lin) + F.softplus(k_bar * (rgb_lin - 1.0))
        gamut_loss = torch.sum(barrier**2) * 5000.0
        
        # 2. Maximize Radius (Vibrancy)
        R = F.softplus(model.radius)
        expansion_loss = -R * 2000.0
        
        # 3. H-K Uniformity (Salience Balance)
        # Q = L + k*C. We want Q to be constant.
        # Since C is constant (by design), this actually forces L to be constant!
        # WAIT. If C is constant, Q-variance is just L-variance.
        # WE DO NOT WANT L TO BE CONSTANT.
        # We want Equal Salience relative to Gray vs Black.
        
        # REVISED METRIC: Opponent Phase Lock
        # We want L to correlate with b (Yellow-Blue axis)
        # Yellow (+b) should be light, Blue (-b) should be dark.
        b = lab_out[:, 2]
        L = lab_out[:, 0]
        # Maximize correlation between L and b
        # Minimize correlation between L and a (Red-Green)
        
        cov_Lb = torch.mean((L - L.mean()) * (b - b.mean()))
        cov_La = torch.mean((L - L.mean()) * (lab_out[:,1] - lab_out[:,1].mean()))
        
        phase_loss = -cov_Lb * 5000.0 + (cov_La**2) * 10000.0
        
        # 4. Safety Floor
        floor_loss = torch.sum(F.relu(0.35 - L)**2) * 100.0
        
        # 5. Regularization (Smooth L)
        reg_loss = torch.sum(model.L_coeffs**2) * 5.0
        
        total_loss = gamut_loss + expansion_loss + phase_loss + floor_loss + reg_loss
        
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        if i % 1000 == 0:
            print(f"Iter {i:04d} | Radius: {R.item():.4f} | Phase Score: {cov_Lb.item():.4f}")
            
    return model

# ============================================================
# 4. Vis & Run
# ============================================================
def reparameterize_by_arclength(lab_points_np):
    dists = np.linalg.norm(lab_points_np - np.roll(lab_points_np, 1, axis=0), axis=1)
    cumulative = np.cumsum(dists); total = cumulative[-1]
    t_curr = np.concatenate(([0], cumulative / total))
    pts = np.vstack([lab_points_np[-1:], lab_points_np])
    iL = interp1d(t_curr, pts[:,0], kind='cubic')
    ia = interp1d(t_curr, pts[:,1], kind='cubic')
    ib = interp1d(t_curr, pts[:,2], kind='cubic')
    t_targ = np.linspace(0, 1, len(lab_points_np) + 1)[:-1]
    return np.stack([iL(t_targ), ia(t_targ), ib(t_targ)], axis=1)

def align_red(lab_path):
    lab_tensor = torch.from_numpy(lab_path).float().to(device)
    rgb_lin = converter.oklab_to_linear_srgb(lab_tensor)
    rgb_srgb = linear_to_srgb_np(rgb_lin.cpu().numpy())
    target = np.array([1.0, 0.0, 0.0])
    idx = np.argmin(np.linalg.norm(rgb_srgb - target, axis=1))
    return np.roll(lab_path, -idx, axis=0)

def get_srgb_wireframe():
    res = 20; t = torch.linspace(0, 1, res, device=device)
    def add(s, d): l = s.repeat(res, 1); l[:, d] = t; return l
    c = torch.tensor([[0,0,0],[1,0,0],[0,1,0],[0,0,1],[1,1,0],[1,0,1],[0,1,1],[1,1,1]], dtype=dtype, device=device)
    edges = []
    edges += [add(c[0],0), add(c[0],1), add(c[0],2), add(c[7],0), add(c[7],1), add(c[7],2)]
    edges += [add(c[1],1), add(c[1],2), add(c[2],0), add(c[2],2), add(c[3],0), add(c[3],1)]
    rgb = torch.cat(edges, dim=0)
    lab = converter.linear_srgb_to_oklab(rgb).cpu().numpy()
    out = []
    for i in range(0, len(lab), res):
        out.append(lab[i:i+res]); out.append(np.array([[None, None, None]]))
    return np.concatenate(out, axis=0)

def plot_3d(lab_path):
    lab_t = torch.from_numpy(lab_path).float().to(device)
    rgb_lin = converter.oklab_to_linear_srgb(lab_t)
    rgb_np = np.clip(linear_to_srgb_np(rgb_lin.cpu().numpy()), 0, 1) * 255
    colors = [f'rgb({r:.0f},{g:.0f},{b:.0f})' for r,g,b in rgb_np]
    gamut = get_srgb_wireframe()
    
    fig = go.Figure(data=[
        go.Scatter3d(x=gamut[:,1], y=gamut[:,2], z=gamut[:,0], mode='lines', 
                     line=dict(color='rgba(150,150,150,0.2)', width=2), name='sRGB', hoverinfo='none'),
        go.Scatter3d(x=lab_path[:,1], y=lab_path[:,2], z=lab_path[:,0], mode='markers', 
                     marker=dict(size=6, color=colors), name='Optimal',
                     hovertemplate='L: %{z:.4f}<br>a: %{x:.4f}<br>b: %{y:.4f}<extra></extra>')
    ])
    fig.update_layout(template='plotly_dark', title='H-K Elastic Circle', 
                      scene=dict(xaxis_title='a', yaxis_title='b', zaxis_title='L', aspectmode='data'),
                      margin=dict(l=0,r=0,b=0,t=40), height=700)
    try: import google.colab; pio.renderers.default = "colab"
    except: pio.renderers.default = "notebook_connected"
    fig.show()

def make_flow_images(rgb_path, center, flows):
    n = len(rgb_path); N=512
    y,x = np.mgrid[0:N, 0:N]
    x = (x+0.5)/N*2-1; y = (y+0.5)/N*2-1
    r = np.sqrt(x*x+y*y); th = np.mod(np.arctan2(y,x), 2*np.pi)
    
    def map(ang, mag):
        idx0 = (ang * n / (2*np.pi)).astype(int) % n
        idx1 = (idx0 + 1) % n
        t = (ang * n / (2*np.pi)) % 1
        col = (1-t[...,None])*rgb_path[idx0] + t[...,None]*rgb_path[idx1]
        out = (1-mag[...,None])*center + mag[...,None]*col
        return np.clip(linear_to_srgb_np(out), 0, 1)

    disk = map(th, np.clip(r,0,1))
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    ang = np.mod(np.arctan2(v,u), 2*np.pi)
    mag = np.sqrt(u*u+v*v); mag /= (mag.max()+1e-8)
    flow = map(ang, mag)
    alpha = mag[...,None]
    fw = alpha*flow + (1-alpha)
    fb = alpha*flow
    return disk, flow, fw, fb

if __name__ == "__main__":
    model = optimize_hk_elastic(hk_k=0.20)
    
    with torch.no_grad():
        t = torch.linspace(0, 2*np.pi, 512, device=device)
        lab = model(t).cpu().numpy()
        
    lab_balanced = reparameterize_by_arclength(lab)
    lab_final = align_red(lab_balanced)
    plot_3d(lab_final)
    
    if OMNIPOSE_AVAILABLE:
        try:
            omnidir = Path(omnirefactor.__file__).parent.parent.parent
            basedir = os.path.join(omnidir, "docs", "_static")
            img = io.imread(os.path.join(basedir, "ecoli.tif"))
            msk = io.imread(os.path.join(basedir, "ecoli_labels.tif"))
            slc = omnirefactor.measure.crop_bbox(msk>0, pad=0)[0]
            flows = omnirefactor.core.masks_to_flows(fastremap.renumber(msk[slc])[0], device="cpu")[4].to("cpu")
            
            lab_t = torch.from_numpy(lab_final).float().to(device)
            rgb_lin = converter.oklab_to_linear_srgb(lab_t).cpu().numpy().clip(0,1)
            center = srgb_to_linear_np(np.array([0.5,0.5,0.5]))
            
            maps = [
                (rgb_lin, "HK-Elastic Circle"),
                (np.roll(rgb_lin, 512//3, 0), "Rot 1/3"),
                (np.roll(rgb_lin, 512//4, 0), "Rot 1/4")
            ]
            
            fig, axes = plt.subplots(3,4, figsize=(16,12))
            for i, (m, t) in enumerate(maps):
                d, f, fw, fb = make_flow_images(m, center, flows)
                axes[i,0].imshow(d); axes[i,1].imshow(f); axes[i,2].imshow(fw); axes[i,3].imshow(fb)
                for ax in axes[i]: ax.axis('off')
                axes[i,0].set_ylabel(t, fontsize=12)
            plt.tight_layout(); plt.show()
        except Exception as e: print(f"Vis error: {e}")

In [ ]:
cmponent color theory notrices that yelow blue should have birght yellow and dark blue
green red can be more muted
so this is what we shohld expect for a more balanced wheeel 

a circle will never work, must fill the gamut 
we nees some isosurface 

In [ ]:
import numpy as np
import torch
import os
from pathlib import Path
from skimage import io
import fastremap
import matplotlib.pyplot as plt

import plotly.graph_objects as go
from plotly.offline import plot
from IPython.display import HTML, display

import omnirefactor

# Expect these helpers to exist in the environment:
#   build_fourier_hsv_hex_ring_jz
#   make_flow_images_for_ring

dtype = torch.float64

# -------------------------------------------------------------------------
# 0. Pure NumPy Color Conversion Helpers
# -------------------------------------------------------------------------

def hsv_to_rgb(hsv):
    """Vectorized HSV to sRGB (0..1)."""
    hsv = np.asarray(hsv)
    h, s, v = hsv[..., 0], hsv[..., 1], hsv[..., 2]
    
    i = (h * 6.0).astype(int)
    f = (h * 6.0) - i
    p = v * (1.0 - s)
    q = v * (1.0 - s * f)
    t = v * (1.0 - s * (1.0 - f))
    
    i = i % 6
    
    rgb = np.zeros_like(hsv)
    
    idx = (i == 0)
    rgb[idx] = np.stack([v[idx], t[idx], p[idx]], axis=-1)
    
    idx = (i == 1)
    rgb[idx] = np.stack([q[idx], v[idx], p[idx]], axis=-1)
    
    idx = (i == 2)
    rgb[idx] = np.stack([p[idx], v[idx], t[idx]], axis=-1)
    
    idx = (i == 3)
    rgb[idx] = np.stack([p[idx], q[idx], v[idx]], axis=-1)
    
    idx = (i == 4)
    rgb[idx] = np.stack([t[idx], p[idx], v[idx]], axis=-1)
    
    idx = (i == 5)
    rgb[idx] = np.stack([v[idx], p[idx], q[idx]], axis=-1)
    
    return rgb

def srgb_to_linear_np(srgb):
    srgb = np.asarray(srgb, dtype=np.float64)
    mask = srgb <= 0.04045
    lin = np.empty_like(srgb)
    lin[mask] = srgb[mask] / 12.92
    lin[~mask] = ((srgb[~mask] + 0.055) / 1.055) ** 2.4
    return lin

def linear_to_srgb_np(lin):
    lin = np.asarray(lin, dtype=np.float64)
    mask = lin <= 0.0031308
    srgb = np.empty_like(lin)
    srgb[mask] = lin[mask] * 12.92
    srgb[~mask] = 1.055 * (lin[~mask] ** (1.0 / 2.4)) - 0.055
    return srgb

# sRGB <-> XYZ matrices
M_srgb_xyz = np.array([
    [0.4124564, 0.3575761, 0.1804375],
    [0.2126729, 0.7151522, 0.0721750],
    [0.0193339, 0.1191920, 0.9503041]
])
M_xyz_srgb = np.linalg.inv(M_srgb_xyz)

def srgb_to_xyz_np(srgb):
    lin = srgb_to_linear_np(srgb)
    return lin @ M_srgb_xyz.T

def xyz_to_srgb_np(xyz):
    lin = xyz @ M_xyz_srgb.T
    return linear_to_srgb_np(lin)

# JzAzBz constants (pure NumPy)
b_ = 1.15
g_ = 0.66
c1 = 3424 / 2**12
c2 = 2413 / 2**7
c3 = 2392 / 2**7
n  = 2610 / 2**14
p  = 1.7 * 2523 / 2**5
d  = -0.56

M1 = np.array([
    [0.41478972, 0.579999, 0.0146480],
    [-0.2015100, 1.120649, 0.0531008],
    [-0.0166008, 0.264800, 0.6684799]
])
M2 = np.array([
    [0.5, 0.5, 0],
    [3.524000, -4.066708, 0.542708],
    [0.199076, 1.096799, -1.295875]
])

def xyz_to_jzazbz_np(xyz):
    xyz = np.asarray(xyz, dtype=np.float64)
    x = xyz[..., 0]
    y = xyz[..., 1]
    z = xyz[..., 2]
    
    xp = b_ * x - (b_ - 1) * z
    yp = g_ * y - (g_ - 1) * x
    
    xyz_p = np.stack([xp, yp, z], axis=-1)
    lms = xyz_p @ M1.T
    
    lms_prime = np.power(np.maximum(lms * 1e-4, 1e-10), n) # avoid div by zero/neg
    
    iz = (c1 + c2 * lms_prime) / (1 + c3 * lms_prime)
    iz_p = np.power(iz, p)
    
    jab = iz_p @ M2.T
    
    Jz = jab[..., 0] + d
    Az = jab[..., 1]
    Bz = jab[..., 2]
    
    return np.stack([Jz, Az, Bz], axis=-1)

def jzazbz_to_xyz_np(jzazbz):
    jzazbz = np.asarray(jzazbz, dtype=np.float64)
    Jz, Az, Bz = jzazbz[..., 0], jzazbz[..., 1], jzazbz[..., 2]
    
    jab = np.stack([Jz - d, Az, Bz], axis=-1)
    iz_p = jab @ np.linalg.inv(M2).T
    
    iz = np.power(iz_p, 1.0/p)
    
    num = iz - c1
    den = c2 - c3 * iz
    # Avoid numerical instability
    lms_prime = num / (den + 1e-12)
    
    lms = np.power(lms_prime, 1.0/n)
    lms = lms / 1e-4 # restore scale
    
    xyz_p = lms @ np.linalg.inv(M1).T
    xp, yp, zp = xyz_p[..., 0], xyz_p[..., 1], xyz_p[..., 2]
    
    x = (xp + (b_ - 1)*zp) / b_
    y = (yp + (g_ - 1)*x) / g_
    z = zp
    
    return np.stack([x, y, z], axis=-1)


# -------------------------------------------------------------------------
# 1. Alignment helpers
# -------------------------------------------------------------------------

def find_reddest_index(rgb_ring_srgb: np.ndarray) -> int:
    rgb = np.asarray(rgb_ring_srgb)
    r = rgb[:, 0]
    g = rgb[:, 1]
    b = rgb[:, 2]
    score = r - 0.5 * (g + b)
    return int(np.argmax(score))


def find_greenest_index(rgb_ring_srgb: np.ndarray) -> int:
    rgb = np.asarray(rgb_ring_srgb)
    r = rgb[:, 0]
    g = rgb[:, 1]
    b = rgb[:, 2]
    score = g - 0.5 * (r + b)
    return int(np.argmax(score))


def align_ring_to_reference(rgb_ring_srgb: np.ndarray,
                            ref_green_idx: int) -> np.ndarray:
    ring = np.asarray(rgb_ring_srgb)
    N = ring.shape[0]

    idx_red = find_reddest_index(ring)
    ring_r0 = np.roll(ring, -idx_red, axis=0)

    g1 = find_greenest_index(ring_r0)
    ring_rev = ring_r0[::-1].copy()
    g2 = find_greenest_index(ring_rev)

    def circ_dist(a, b):
        d = abs(a - b)
        return min(d, N - d)

    if circ_dist(g1, ref_green_idx) <= circ_dist(g2, ref_green_idx):
        return ring_r0
    else:
        return ring_rev


# -------------------------------------------------------------------------
# 2. Resampling / Arc-length helper
# -------------------------------------------------------------------------

def resample_jz_ring_uniform(jz_ring: np.ndarray, n_samples: int) -> np.ndarray:
    """Resample JzAzBz ring to have uniform Euclidean step sizes."""
    ring_cyclic = np.concatenate([jz_ring, jz_ring[:1]], axis=0)
    diffs = np.linalg.norm(np.diff(ring_cyclic, axis=0), axis=1)
    cum_dist = np.concatenate([[0.0], np.cumsum(diffs)])
    total_len = cum_dist[-1]
    
    t_original = cum_dist / total_len
    t_uniform = np.linspace(0, 1, n_samples, endpoint=False)
    
    new_J = np.interp(t_uniform, t_original, ring_cyclic[:, 0])
    new_az = np.interp(t_uniform, t_original, ring_cyclic[:, 1])
    new_bz = np.interp(t_uniform, t_original, ring_cyclic[:, 2])
    
    return np.stack([new_J, new_az, new_bz], axis=1)


# -------------------------------------------------------------------------
# 3. Iso-Jz ring
# -------------------------------------------------------------------------

def build_isoJ_from_fourier(jz_fourier: np.ndarray,
                            J0: float = 0.5) -> np.ndarray:
    jz_iso = jz_fourier.copy()
    jz_iso[:, 0] = J0

    jz_t = torch.from_numpy(jz_iso).to(dtype=dtype)
    center = jz_t.mean(dim=0)
    offsets = jz_t - center[None, :]

    def is_safe(scale: float) -> bool:
        jz_scaled = center[None, :] + scale * offsets
        jz_np = jz_scaled.cpu().numpy()
        xyz = jzazbz_to_xyz_np(jz_np)
        srgb = xyz_to_srgb_np(xyz)
        rgb_lin = srgb_to_linear_np(srgb)
        return np.all((rgb_lin >= -1e-6) & (rgb_lin <= 1.0 + 1e-6))

    s_lo, s_hi = 0.0, 1.0
    if not is_safe(1.0):
        for _ in range(40):
            mid = 0.5 * (s_lo + s_hi)
            if is_safe(float(mid)):
                s_lo = mid
            else:
                s_hi = mid
        s_opt = s_lo
    else:
        s_opt = 1.0

    jz_iso_safe_t = center[None, :] + s_opt * offsets
    jz_iso_safe = jz_iso_safe_t.cpu().numpy()
    xyz_iso_safe = jzazbz_to_xyz_np(jz_iso_safe)
    rgb_iso_srgb = np.clip(xyz_to_srgb_np(xyz_iso_safe), 0, 1)
    return rgb_iso_srgb


# -------------------------------------------------------------------------
# 4. Salience versus gray backgrounds (global warp in J and C)
# -------------------------------------------------------------------------

def relative_luminance_from_linear(rgb_lin: np.ndarray) -> np.ndarray:
    rgb_lin = np.asarray(rgb_lin)
    r = rgb_lin[..., 0]
    g = rgb_lin[..., 1]
    b = rgb_lin[..., 2]
    return 0.2126 * r + 0.7152 * g + 0.0722 * b


def pc_contrast(Y_fg: float, Y_bg: float, exp: float = 0.4, eps: float = 1e-4) -> float:
    Y_fg = float(Y_fg)
    Y_bg = float(Y_bg)
    L_fg = max(Y_fg, eps) ** exp
    L_bg = max(Y_bg, eps) ** exp
    return abs(L_fg - L_bg) / (L_fg + L_bg)


def build_contrast_balanced_ring_from_fourier(
    jz_fourier: np.ndarray,
    C_min: float = 0.04,
    J_center_min: float = 0.45,
    J_center_max: float = 0.75,
    J_center_step: float = 0.02,
    C_scale_min: float = 0.6,
    C_scale_max: float = 1.35,
    C_scale_step: float = 0.05,
    w_mean: float = 1.0,
    w_varbg: float = 1.0,
    w_varhue: float = 0.3,
    w_chr: float = 0.6,
    w_cvar: float = 0.4,
    exp_pc: float = 0.4,
) -> tuple[np.ndarray, np.ndarray]:
    jz_fourier = np.asarray(jz_fourier, dtype=np.float64)
    J_base = jz_fourier[:, 0]
    az_base = jz_fourier[:, 1]
    bz_base = jz_fourier[:, 2]

    C_base = np.hypot(az_base, bz_base)
    eps = 1e-6
    u_az = az_base / np.maximum(C_base, eps)
    u_bz = bz_base / np.maximum(C_base, eps)

    J_mean = float(J_base.mean())

    Y_bgs = np.linspace(0.0, 1.0, 13, dtype=np.float64)

    best_score = -1e9
    best_jz = None
    best_rgb = None

    J_centers = np.arange(J_center_min, J_center_max + 1e-9, J_center_step)
    C_scales = np.arange(C_scale_min, C_scale_max + 1e-9, C_scale_step)

    for J_center in J_centers:
        J_new = J_center + (J_base - J_mean)
        if np.any(J_new < 0.0) or np.any(J_new > 1.0):
            continue

        for C_scale in C_scales:
            C_new = C_scale * C_base
            if np.min(C_new) < C_min:
                continue

            az_new = C_new * u_az
            bz_new = C_new * u_bz
            jz_candidate = np.stack([J_new, az_new, bz_new], axis=1)

            xyz = jzazbz_to_xyz_np(jz_candidate)
            srgb = xyz_to_srgb_np(xyz)
            rgb_clip = np.clip(srgb, 0.0, 1.0)
            rgb_lin = srgb_to_linear_np(rgb_clip)

            if np.any(rgb_lin < -1e-4) or np.any(rgb_lin > 1.0 + 1e-4):
                continue

            Y_fg = relative_luminance_from_linear(rgb_lin)

            n_h = jz_candidate.shape[0]
            n_bg = Y_bgs.size
            pc_mat = np.empty((n_h, n_bg), dtype=np.float64)
            for j, Y_bg in enumerate(Y_bgs):
                for i in range(n_h):
                    pc_mat[i, j] = pc_contrast(float(Y_fg[i]), float(Y_bg), exp=exp_pc)

            mean_sal = float(pc_mat.mean())
            var_bg = float(pc_mat.var(axis=1).mean())
            var_hue = float(pc_mat.var(axis=0).mean())
            mean_C = float(C_new.mean())
            std_C = float(C_new.std())

            score = (
                w_mean * mean_sal
                - w_varbg * var_bg
                - w_varhue * var_hue
                + w_chr * mean_C
                - w_cvar * std_C
            )

            if score > best_score:
                best_score = score
                best_jz = jz_candidate
                best_rgb = rgb_clip

    if best_jz is None:
        best_jz = jz_fourier
        xyz = jzazbz_to_xyz_np(best_jz)
        best_rgb = np.clip(xyz_to_srgb_np(xyz), 0.0, 1.0)

    return best_rgb, best_jz


# -------------------------------------------------------------------------
# 5. Plotly 3D trajectory with warped sRGB cube surfaces
# -------------------------------------------------------------------------

def _srgb_face_to_jz_surface(n: int, fixed_channel: int, fixed_value: float):
    grid = np.linspace(0.0, 1.0, n)
    G, B = np.meshgrid(grid, grid)

    if fixed_channel == 0:
        R = np.full_like(G, fixed_value)
        srgb = np.stack([R, G, B], axis=-1)
    elif fixed_channel == 1:
        Gf = np.full_like(G, fixed_value)
        srgb = np.stack([G, Gf, B], axis=-1)
    else:
        Bf = np.full_like(G, fixed_value)
        srgb = np.stack([G, B, Bf], axis=-1)

    xyz = srgb_to_xyz_np(srgb.reshape(-1, 3))
    jz = xyz_to_jzazbz_np(xyz).reshape(srgb.shape)
    J = jz[..., 0]
    az = jz[..., 1]
    bz = jz[..., 2]
    return az, bz, J


def plot_jz_trajectory_hex_fourier_sine_hsv(
    jz_fourier: np.ndarray, rgb_fourier: np.ndarray,
    jz_arc: np.ndarray,     rgb_arc: np.ndarray,
    jz_sine_arc: np.ndarray, rgb_sine_arc: np.ndarray,
    jz_sine: np.ndarray,     rgb_sine: np.ndarray,
    jz_hsv: np.ndarray,     rgb_hsv: np.ndarray,
    jz_iso: np.ndarray,     rgb_iso: np.ndarray,
    jz_bal: np.ndarray,     rgb_bal: np.ndarray,
):
    fig = go.Figure()

    # sRGB cube faces as translucent surfaces
    n_face = 17
    face_specs = [
        (0, 0.0, "R=0"), (0, 1.0, "R=1"),
        (1, 0.0, "G=0"), (1, 1.0, "G=1"),
        (2, 0.0, "B=0"), (2, 1.0, "B=1"),
    ]
    for fixed_channel, fixed_value, name in face_specs:
        az, bz, J = _srgb_face_to_jz_surface(n_face, fixed_channel, fixed_value)
        fig.add_trace(go.Surface(x=az, y=bz, z=J, opacity=0.18, showscale=False, colorscale="Viridis", name=name, hoverinfo="skip"))

    def add_curve(jz, name, width=7):
        jz = np.asarray(jz)
        fig.add_trace(go.Scatter3d(x=jz[:, 1], y=jz[:, 2], z=jz[:, 0], mode="lines", name=name, line=dict(width=width)))

    add_curve(jz_fourier, "Fourier",        width=7)
    add_curve(jz_arc,      "Arc-length HSV", width=5)
    add_curve(jz_iso,      "Iso-Jz",        width=5)
    add_curve(jz_bal,      "Balanced",      width=8)
    add_curve(jz_sine_arc, "Arc-length Sine", width=5)
    add_curve(jz_sine,     "sinebow",       width=4)
    add_curve(jz_hsv,      "HSV",           width=4)

    fig.update_layout(
        template="plotly_dark",
        paper_bgcolor="black",
        scene=dict(xaxis_title="az", yaxis_title="bz", zaxis_title="Jz", bgcolor="black", aspectmode="data"),
        margin=dict(l=0, r=0, b=0, t=40),
        title="JzAzBz trajectories with warped sRGB cube and balanced ring",
        showlegend=True,
    )

    html = plot(fig, output_type="div", include_plotlyjs="cdn")
    return HTML(html)


# -------------------------------------------------------------------------
# 6. Main visualization
# -------------------------------------------------------------------------

def visualize_fourier_hex_vs_sinebow_hsv_on_omnipose_jz(
    n_hues: int = 72,
    n_harmonics: int = 1,
    device_str: str = "cpu",
):
    dev = torch.device(device_str)

    # 1) Base HSV
    t = np.linspace(0, 1, n_hues, endpoint=False)
    hsv_vals = np.stack([t, np.ones_like(t), np.ones_like(t)], axis=1)
    rgb_hsv_srgb_raw = hsv_to_rgb(hsv_vals)
    xyz_hsv_raw = srgb_to_xyz_np(rgb_hsv_srgb_raw)
    jz_hsv_raw = xyz_to_jzazbz_np(xyz_hsv_raw)

    # ---------------------------------------------------------------------
    # Sinebow (Raw) and Arc-Length Reparameterized Sinebow
    # ---------------------------------------------------------------------
    t_high = np.linspace(0, 1, n_hues * 8, endpoint=False)
    sine_r_h = 0.5 + 0.5 * np.sin(2 * np.pi * (t_high + 0 / 3))
    sine_g_h = 0.5 + 0.5 * np.sin(2 * np.pi * (t_high + 1 / 3))
    sine_b_h = 0.5 + 0.5 * np.sin(2 * np.pi * (t_high + 2 / 3))
    rgb_sine_high = np.stack([sine_r_h, sine_g_h, sine_b_h], axis=1)
    jz_sine_high = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_sine_high))

    jz_sine_arc_raw = resample_jz_ring_uniform(jz_sine_high, n_hues)
    rgb_sine_arc_srgb_raw = np.clip(xyz_to_srgb_np(jzazbz_to_xyz_np(jz_sine_arc_raw)), 0.0, 1.0)

    sine_r = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0 / 3))
    sine_g = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1 / 3))
    sine_b = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2 / 3))
    rgb_sine_srgb_raw = np.stack([sine_r, sine_g, sine_b], axis=1)

    # 2) Fourier
    jz_fourier_raw, rgb_fourier_srgb_raw, jz_arc_raw, rgb_arc_srgb_raw = build_fourier_hsv_hex_ring_jz(
        n_hues=n_hues, n_harmonics=n_harmonics, oversample=8
    )

    # ---------------------------------------------------------------------
    # Canonical Alignment
    # ---------------------------------------------------------------------
    idx_red_h = find_reddest_index(rgb_hsv_srgb_raw)
    rgb_hsv_r0 = np.roll(rgb_hsv_srgb_raw, -idx_red_h, axis=0)
    g_h = find_greenest_index(rgb_hsv_r0)
    
    if g_h > n_hues // 2:
        rgb_hsv_r0 = rgb_hsv_r0[::-1].copy()
        g_h = find_greenest_index(rgb_hsv_r0)
    rgb_hsv_srgb = rgb_hsv_r0
    jz_hsv = xyz_to_jzazbz_np(srgb_to_xyz_np(rgb_hsv_srgb))

    def align_to_hsv(rgb_raw):
        ring_aligned = align_ring_to_reference(rgb_raw, ref_green_idx=g_h)
        jz_aligned = xyz_to_jzazbz_np(srgb_to_xyz_np(ring_aligned))
        return ring_aligned, jz_aligned

    rgb_arc_srgb, jz_arc = align_to_hsv(rgb_arc_srgb_raw)
    rgb_fourier_srgb, jz_fourier = align_to_hsv(rgb_fourier_srgb_raw)
    rgb_sine_srgb, jz_sine = align_to_hsv(rgb_sine_srgb_raw)
    rgb_sine_arc_srgb, jz_sine_arc = align_to_hsv(rgb_sine_arc_srgb_raw)

    rgb_iso_srgb_raw = build_isoJ_from_fourier(jz_fourier, J0=0.65)
    rgb_iso_srgb, jz_iso = align_to_hsv(rgb_iso_srgb_raw)

    rgb_bal_srgb_raw, jz_bal_raw = build_contrast_balanced_ring_from_fourier(
        jz_fourier, C_min=0.05, J_center_min=0.6, J_center_max=0.9,
        J_center_step=0.02, C_scale_min=0.6, C_scale_max=1.5, C_scale_step=0.05,
    )
    rgb_bal_srgb, jz_bal = align_to_hsv(rgb_bal_srgb_raw)

    # ---------------------------------------------------------------------
    # Plotting
    # ---------------------------------------------------------------------
    traj_html = plot_jz_trajectory_hex_fourier_sine_hsv(
        jz_fourier, rgb_fourier_srgb,
        jz_arc, rgb_arc_srgb,
        jz_sine_arc, rgb_sine_arc_srgb,
        jz_sine, rgb_sine_srgb,
        jz_hsv, rgb_hsv_srgb,
        jz_iso, rgb_iso_srgb,
        jz_bal, rgb_bal_srgb,
    )
    display(traj_html)

    rgb_fourier_lin = srgb_to_linear_np(rgb_fourier_srgb)
    rgb_arc_lin = srgb_to_linear_np(rgb_arc_srgb)
    rgb_iso_lin = srgb_to_linear_np(rgb_iso_srgb)
    rgb_bal_lin = srgb_to_linear_np(rgb_bal_srgb)
    rgb_sine_lin = srgb_to_linear_np(rgb_sine_srgb)
    rgb_sine_arc_lin = srgb_to_linear_np(rgb_sine_arc_srgb)
    rgb_hsv_lin = srgb_to_linear_np(rgb_hsv_srgb)

    center_srgb = np.array([0.5, 0.5, 0.5], dtype=float)
    center_lin = srgb_to_linear_np(center_srgb)

    omnidir = Path(omnirefactor.__file__).parent.parent.parent
    basedir = os.path.join(omnidir, "docs", "_static")
    name = "ecoli"
    ext = ".tif"

    image = io.imread(os.path.join(basedir, name + ext))
    masks = io.imread(os.path.join(basedir, name + "_labels" + ext))
    slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
    masks = fastremap.renumber(masks[slc])[0]
    image = image[slc]

    flows_all = omnirefactor.core.masks_to_flows(masks, device="cpu")[4].to(dev)
    flows = flows_all

    def get_flow_imgs(rgb_lin):
        return make_flow_images_for_ring(rgb_lin, center_lin, flows)

    disk_fourier, flow_fourier, flow_fourier_white, flow_fourier_black = get_flow_imgs(rgb_fourier_lin)
    disk_arc, flow_arc, flow_arc_white, flow_arc_black = get_flow_imgs(rgb_arc_lin)
    disk_iso, flow_iso, flow_iso_white, flow_iso_black = get_flow_imgs(rgb_iso_lin)
    disk_bal, flow_bal, flow_bal_white, flow_bal_black = get_flow_imgs(rgb_bal_lin)
    disk_sine_arc, flow_sine_arc, flow_sine_arc_white, flow_sine_arc_black = get_flow_imgs(rgb_sine_arc_lin)
    disk_sine, flow_sine, flow_sine_white, flow_sine_black = get_flow_imgs(rgb_sine_lin)
    disk_hsv, flow_hsv, flow_hsv_white, flow_hsv_black = get_flow_imgs(rgb_hsv_lin)

    shift = n_hues // 4

    def get_rot_imgs(rgb_lin):
        rgb_lin_rot = np.roll(rgb_lin, shift, axis=0)
        # make_flow_images_for_ring returns (disk, flow, white, black)
        # We only need white and black here
        _, _, w, b = make_flow_images_for_ring(rgb_lin_rot, center_lin, flows)
        return w, b

    # FIXED UNPACKING HERE: expects 2 values, now receives 2 values
    flow_fourier_white_rot, flow_fourier_black_rot = get_rot_imgs(rgb_fourier_lin)
    flow_arc_white_rot, flow_arc_black_rot = get_rot_imgs(rgb_arc_lin)
    flow_iso_white_rot, flow_iso_black_rot = get_rot_imgs(rgb_iso_lin)
    flow_bal_white_rot, flow_bal_black_rot = get_rot_imgs(rgb_bal_lin)
    flow_sine_arc_white_rot, flow_sine_arc_black_rot = get_rot_imgs(rgb_sine_arc_lin)
    flow_sine_white_rot, flow_sine_black_rot = get_rot_imgs(rgb_sine_lin)
    flow_hsv_white_rot, flow_hsv_black_rot = get_rot_imgs(rgb_hsv_lin)

    fig, axes = plt.subplots(7, 6, figsize=(26, 16))
    row_titles = [
        "Fourier", "Arc-length HSV", "Iso-Jz", "Balanced",
        "Arc-length Sine", "sinebow", "HSV",
    ]

    def _strip_axes(ax):
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values(): spine.set_visible(False)
    
    def plot_row(row_idx, disk, flow, fw, fb, fwr, fbr):
        axes[row_idx, 0].imshow(disk); _strip_axes(axes[row_idx, 0])
        axes[row_idx, 1].imshow(flow); axes[row_idx, 1].axis("off")
        axes[row_idx, 2].imshow(fw);   axes[row_idx, 2].axis("off")
        axes[row_idx, 3].imshow(fb);   axes[row_idx, 3].axis("off")
        axes[row_idx, 4].imshow(fwr);  axes[row_idx, 4].axis("off")
        axes[row_idx, 5].imshow(fbr);  axes[row_idx, 5].axis("off")

    plot_row(0, disk_fourier, flow_fourier, flow_fourier_white, flow_fourier_black, flow_fourier_white_rot, flow_fourier_black_rot)
    plot_row(1, disk_arc, flow_arc, flow_arc_white, flow_arc_black, flow_arc_white_rot, flow_arc_black_rot)
    plot_row(2, disk_iso, flow_iso, flow_iso_white, flow_iso_black, flow_iso_white_rot, flow_iso_black_rot)
    plot_row(3, disk_bal, flow_bal, flow_bal_white, flow_bal_black, flow_bal_white_rot, flow_bal_black_rot)
    plot_row(4, disk_sine_arc, flow_sine_arc, flow_sine_arc_white, flow_sine_arc_black, flow_sine_arc_white_rot, flow_sine_arc_black_rot)
    plot_row(5, disk_sine, flow_sine, flow_sine_white, flow_sine_black, flow_sine_white_rot, flow_sine_black_rot)
    plot_row(6, disk_hsv, flow_hsv, flow_hsv_white, flow_hsv_black, flow_hsv_white_rot, flow_hsv_black_rot)

    for i, title in enumerate(row_titles):
        axes[i, 0].set_ylabel(title, fontsize=12)

    axes[0, 0].set_title("Hue disk (0°)")
    axes[0, 1].set_title("Flow → chroma (0°)")
    axes[0, 2].set_title("Alpha on white (0°)")
    axes[0, 3].set_title("Alpha on black (0°)")
    axes[0, 4].set_title("Alpha on white (+90°)")
    axes[0, 5].set_title("Alpha on black (+90°)")

    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    N_HUES = 72
    N_HARMONICS = 1
    visualize_fourier_hex_vs_sinebow_hsv_on_omnipose_jz(n_hues=N_HUES, n_harmonics=N_HARMONICS, device_str="cpu")

In [ ]:
import numpy as np
import torch
import os
from pathlib import Path
from skimage import io
import fastremap
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

import plotly.graph_objects as go
from plotly.offline import plot
from IPython.display import HTML, display

import omnirefactor

dtype = torch.float64

# -------------------------------------------------------------------------
# 0. Robust Color Conversion (JzAzBz - 200 nits)
# -------------------------------------------------------------------------

def srgb_to_linear_np(srgb):
    srgb = np.asarray(srgb, dtype=np.float64)
    srgb = np.clip(srgb, 0.0, 1.0)
    mask = srgb <= 0.04045
    lin = np.empty_like(srgb)
    lin[mask] = srgb[mask] / 12.92
    lin[~mask] = ((srgb[~mask] + 0.055) / 1.055) ** 2.4
    return lin

def linear_to_srgb_np(lin):
    lin = np.asarray(lin, dtype=np.float64)
    # We use a tolerant conversion that handles slight out-of-gamut for probing
    sign = np.sign(lin)
    val = np.abs(lin)
    mask = val <= 0.0031308
    srgb = np.empty_like(lin)
    srgb[mask] = val[mask] * 12.92
    srgb[~mask] = 1.055 * (val[~mask] ** (1.0 / 2.4)) - 0.055
    return srgb * sign

M_srgb_xyz = np.array([[0.4124564, 0.3575761, 0.1804375],[0.2126729, 0.7151522, 0.0721750],[0.0193339, 0.1191920, 0.9503041]])
M_xyz_srgb = np.linalg.inv(M_srgb_xyz)
def srgb_to_xyz_np(srgb): return srgb_to_linear_np(srgb) @ M_srgb_xyz.T
def xyz_to_srgb_np(xyz): return linear_to_srgb_np(xyz @ M_xyz_srgb.T)

# Safdar 2017 Constants
JZ_B = 1.15; JZ_G = 0.66
JZ_C1 = 3424/4096; JZ_C2 = 2413/128; JZ_C3 = 2392/128
JZ_N = 2610/16384; JZ_P = 1.7 * 2523/32; JZ_D = -0.56
M1 = np.array([[0.41478972, 0.579999, 0.0146480],[-0.2015100, 1.120649, 0.0531008],[-0.0166008, 0.264800, 0.6684799]])
M2 = np.array([[0.5, 0.5, 0],[3.524000, -4.066708, 0.542708],[0.199076, 1.096799, -1.295875]])
M1_INV = np.linalg.inv(M1); M2_INV = np.linalg.inv(M2)
DISPLAY_PEAK_NITS = 200.0 

def xyz_to_jzazbz(xyz):
    xyz_abs = xyz * DISPLAY_PEAK_NITS
    lms = np.maximum((np.stack([JZ_B*xyz_abs[...,0]-(JZ_B-1)*xyz_abs[...,2], JZ_G*xyz_abs[...,1]-(JZ_G-1)*xyz_abs[...,0], xyz_abs[...,2]], axis=-1)) @ M1.T, 1e-12)
    lms_p = np.power(lms, JZ_N)
    iz = np.maximum((JZ_C1 + JZ_C2 * lms_p) / (1 + JZ_C3 * lms_p), 1e-12)
    jab = np.power(iz, JZ_P) @ M2.T
    return np.stack([jab[..., 0] + JZ_D, jab[..., 1], jab[..., 2]], axis=-1)

def jzazbz_to_xyz(jab):
    iz_pow = np.maximum(np.stack([jab[...,0]-JZ_D, jab[...,1], jab[...,2]], axis=-1) @ M2_INV.T, 1e-12)
    iz = np.power(iz_pow, 1.0/JZ_P)
    lms_p = np.maximum((iz - JZ_C1) / (JZ_C2 - JZ_C3 * iz + 1e-12), 0)
    lms = np.power(lms_p, 1.0/JZ_N)
    xyz_p = lms @ M1_INV.T
    X = (xyz_p[...,0] + (JZ_B-1)*xyz_p[...,2]) / JZ_B
    Y = (xyz_p[...,1] + (JZ_G-1)*X) / JZ_G
    return np.stack([X, Y, xyz_p[...,2]], axis=-1) / DISPLAY_PEAK_NITS

# -------------------------------------------------------------------------
# 1. Geometric Construction: Ray Scanning
# -------------------------------------------------------------------------

def is_in_gamut(jz_point):
    xyz = jzazbz_to_xyz(jz_point)
    srgb = xyz_to_srgb_np(xyz)
    lin = srgb_to_linear_np(srgb)
    # Check strict 0..1 bounds
    return np.all((lin >= -0.001) & (lin <= 1.001))

def scan_wall_distance(J, angle_rad):
    """
    Shoots a ray from (J, 0, 0) at angle `angle_rad` in the ab-plane.
    Returns the maximum Chroma (radius) before hitting the gamut wall.
    """
    low = 0.0
    high = 0.40 # Max possible chroma in JzAzBz is roughly 0.3
    
    da = np.cos(angle_rad)
    db = np.sin(angle_rad)
    
    # Binary Search for the wall
    for _ in range(16):
        mid = (low + high) * 0.5
        pt = np.array([J, mid*da, mid*db])
        if is_in_gamut(pt):
            low = mid
        else:
            high = mid
    return low

def harmonic_2_fit(x, c0, c1, s1, c2, s2):
    """2nd Order Fourier Series (Ellipse-like)"""
    return c0 + c1*np.cos(x) + s1*np.sin(x) + c2*np.cos(2*x) + s2*np.sin(2*x)

def construct_maximal_ellipse_loop(n_scan=72):
    print("Constructing Maximal Ellipse...")
    
    angles = np.linspace(0, 2*np.pi, n_scan, endpoint=False)
    
    # 1. Define Fixed Lightness Tilt (The "Bone Structure")
    # We anchor this to the perceptual reality of sRGB:
    # Yellow (~90 deg) is Bright. Blue (~270 deg) is Dark.
    # J=0.17 is mid-grey. Amplitude 0.05 swings from 0.12 (Dark) to 0.22 (Bright).
    # This range is safe for sRGB.
    J_vals = 0.17 + 0.05 * np.sin(angles - np.deg2rad(10)) 

    # 2. Laser Scan the Walls
    # For every angle, find exactly how far we can push saturation.
    C_raw = []
    for i, theta in enumerate(angles):
        dist = scan_wall_distance(J_vals[i], theta)
        C_raw.append(dist)
    C_raw = np.array(C_raw)
    
    # 3. Fit a Smooth Ellipse (Harmonic 2)
    # We take the jagged scan data and fit a perfect mathematical ellipse to it.
    popt, _ = curve_fit(harmonic_2_fit, angles, C_raw)
    C_smooth = harmonic_2_fit(angles, *popt)
    
    # 4. Buffer (The "Safety Margin")
    # We contract the perfect ellipse by 10% to ensure it floats inside the gamut.
    C_final = C_smooth * 0.90
    
    # 5. Construct Points
    az = C_final * np.cos(angles)
    bz = C_final * np.sin(angles)
    
    return np.stack([J_vals, az, bz], axis=1)

# -------------------------------------------------------------------------
# 2. Resampling & Alignment
# -------------------------------------------------------------------------

def resample_uniform(jz_ring, n_target):
    ring_cyc = np.concatenate([jz_ring, jz_ring[:1]], axis=0)
    dists = np.linalg.norm(np.diff(ring_cyc, axis=0), axis=1)
    cum_dist = np.concatenate([[0.0], np.cumsum(dists)])
    t_orig = cum_dist / cum_dist[-1]
    t_new = np.linspace(0, 1, n_target, endpoint=False)
    out = np.zeros((n_target, 3))
    for i in range(3):
        out[:,i] = np.interp(t_new, t_orig, ring_cyc[:,i])
    return out

def align_to_green(rgb, n_hues):
    # Standard helper to align Green to ~120 deg pos
    r = rgb[:,0]; g = rgb[:,1]; b = rgb[:,2]
    ridx = np.argmax(r - 0.5*(g+b))
    r0 = np.roll(rgb, -ridx, axis=0)
    gidx = np.argmax(r0[:,1] - 0.5*(r0[:,0]+r0[:,2]))
    r_rev = r0[::-1].copy()
    gidx_rev = np.argmax(r_rev[:,1] - 0.5*(r_rev[:,0]+r_rev[:,2]))
    
    # Ref target: roughly 1/3rd of the way around
    ref = n_hues // 3
    d1 = min(abs(gidx - ref), n_hues - abs(gidx - ref))
    d2 = min(abs(gidx_rev - ref), n_hues - abs(gidx_rev - ref))
    return r0 if d1 <= d2 else r_rev

# -------------------------------------------------------------------------
# 3. Main Routine
# -------------------------------------------------------------------------

def run_maximal_ellipse(n_hues=72, device_str="cpu"):
    dev = torch.device(device_str)
    
    # 1. Construct Geometric Loop
    jz_raw = construct_maximal_ellipse_loop(n_scan=180)
    
    # 2. Resample for constant perceptual velocity
    jz_opt = resample_uniform(jz_raw, n_hues)
    
    # 3. Convert to RGB
    xyz_opt = jzazbz_to_xyz(jz_opt)
    rgb_opt = np.clip(xyz_to_srgb_np(xyz_opt), 0, 1)

    # --- Comparators ---
    t = np.linspace(0, 1, n_hues, endpoint=False)
    
    # Sinebow
    sr = 0.5 + 0.5*np.sin(2*np.pi*(t+0/3))
    sg = 0.5 + 0.5*np.sin(2*np.pi*(t+1/3))
    sb = 0.5 + 0.5*np.sin(2*np.pi*(t+2/3))
    rgb_sine = np.stack([sr, sg, sb], axis=1)
    
    # HSV
    hsv_vals = np.stack([t, np.ones_like(t), np.ones_like(t)], axis=1)
    import matplotlib.colors as mcolors
    rgb_hsv = mcolors.hsv_to_rgb(hsv_vals)

    # Align all to Green
    rgb_hsv_a = align_to_green(rgb_hsv, n_hues)
    
    # Find alignment index
    gidx = np.argmax(rgb_hsv_a[:,1] - 0.5*(rgb_hsv_a[:,0]+rgb_hsv_a[:,2]))
    
    def align_ref(rgb, ref):
        g = np.argmax(rgb[:,1] - 0.5*(rgb[:,0]+rgb[:,2]))
        shift = ref - g
        return np.roll(rgb, shift, axis=0)

    rgb_sine_a = align_ref(rgb_sine, gidx)
    rgb_opt_a = align_ref(rgb_opt, gidx)

    jz_hsv_a = xyz_to_jzazbz(srgb_to_xyz_np(rgb_hsv_a))
    jz_sine_a = xyz_to_jzazbz(srgb_to_xyz_np(rgb_sine_a))
    jz_opt_a = xyz_to_jzazbz(srgb_to_xyz_np(rgb_opt_a))

    # --- Plotly 3D ---
    def plot_traj(trajs):
        fig = go.Figure()
        grid = np.linspace(0,1,17); G, B = np.meshgrid(grid, grid)
        def sface(ch, v, n):
            if ch==0: s=np.stack([np.full_like(G,v),G,B],axis=-1)
            elif ch==1: s=np.stack([G,np.full_like(G,v),B],axis=-1)
            else: s=np.stack([G,B,np.full_like(G,v)],axis=-1)
            jz = xyz_to_jzazbz(srgb_to_xyz_np(s.reshape(-1,3))).reshape(17,17,3)
            fig.add_trace(go.Surface(x=jz[...,1],y=jz[...,2],z=jz[...,0],opacity=0.1,showscale=False,name=n,hoverinfo='skip'))
        
        for c,v,n in [(0,0,'R0'),(0,1,'R1'),(1,0,'G0'),(1,1,'G1'),(2,0,'B0'),(2,1,'B1')]: sface(c,v,n)
        
        for n, jz, w in trajs:
            fig.add_trace(go.Scatter3d(x=jz[:,1], y=jz[:,2], z=jz[:,0], mode='lines', name=n, line=dict(width=w)))
            
        fig.update_layout(template="plotly_dark", height=600, title="JzAzBz Maximal Ellipse (Geometric Fit)")
        return HTML(plot(fig, output_type='div', include_plotlyjs='cdn'))
    
    display(plot_traj([("HSV", jz_hsv_a, 3), ("Sinebow", jz_sine_a, 3), ("Max-Ellipse", jz_opt_a, 10)]))
    
    # --- Omnipose Grid ---
    omnidir = Path(omnirefactor.__file__).parent.parent.parent
    basedir = os.path.join(omnidir, "docs", "_static")
    masks = io.imread(os.path.join(basedir, "ecoli_labels.tif"))
    slc = omnirefactor.measure.crop_bbox(masks>0, pad=0)[0]
    masks = fastremap.renumber(masks[slc])[0]
    flows = omnirefactor.core.masks_to_flows(masks, device="cpu")[4].to(dev)
    center = srgb_to_linear_np(np.array([0.5,0.5,0.5]))
    
    def get_row(rgb):
        lin = srgb_to_linear_np(rgb)
        d,f,w,b = make_flow_images_for_ring(lin, center, flows)
        rot = np.roll(lin, n_hues//4, axis=0)
        _,_,wr,br = make_flow_images_for_ring(rot, center, flows)
        return [d,f,w,b,wr,br]

    r_hsv = get_row(rgb_hsv_a)
    r_sin = get_row(rgb_sine_a)
    r_opt = get_row(rgb_opt_a)
    
    fig, ax = plt.subplots(3,6, figsize=(24,10))
    def clean(a): a.axis('off')
    names = ["HSV", "Sinebow", "Max-Ellipse"]
    data = [r_hsv, r_sin, r_opt]
    cols = ["Disk", "Flow", "Alpha/W", "Alpha/B", "Rot/W", "Rot/B"]
    
    for i in range(3):
        for j in range(6):
            ax[i,j].imshow(data[i][j])
            clean(ax[i,j])
            if i==0: ax[i,j].set_title(cols[j])
            if j==0: ax[i,j].text(-0.2, 0.5, names[i], transform=ax[i,j].transAxes, va='center', ha='right', fontsize=12, rotation=90)
            
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    if 'make_flow_images_for_ring' not in locals():
        def make_flow_images_for_ring(a,b,c): return np.zeros((10,10,3)), np.zeros((10,10,3)), np.zeros((10,10,3)), np.zeros((10,10,3))
    run_maximal_ellipse(72)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Utility Functions ---

def y_to_srgb(y):
    """Converts Physical Luminance Y (0-100) to sRGB (0-1)."""
    # Normalize Y to 0-1 range
    xyz = np.array([0.95047, 1.00000, 1.08883]) * (y / 100.0)
    
    # sRGB D65 Matrix Transform
    m_inv = np.array([[ 3.2404542, -1.5371385, -0.4985314],
                      [-0.9692660,  1.8760108,  0.0415560],
                      [ 0.0556434, -0.2040259,  1.0572252]])
    rgb_lin = np.dot(m_inv, xyz)
    
    # Gamma Correction (sRGB Companding)
    # If val <= 0.0031308, val * 12.92
    # Else, 1.055 * val^(1/2.4) - 0.055
    rgb = np.where(rgb_lin <= 0.0031308, 
                   12.92 * rgb_lin, 
                   1.055 * (np.maximum(rgb_lin, 0)**(1/2.4)) - 0.055)
    
    return np.clip(rgb, 0, 1)

def srgb_encoded_to_y(v_encoded):
    """Converts an encoded sRGB value (0-1) to physical Luminance Y (0-100)."""
    # Inverse Gamma Companding
    v_lin = np.where(v_encoded <= 0.04045, 
                     v_encoded / 12.92, 
                     ((v_encoded + 0.055) / 1.055) ** 2.4)
    # For neutral gray (R=G=B), Y is proportional to the linear value.
    # Y = 100 * Linear_Intensity
    return v_lin * 100.0

# --- 2. Define the Gray Values ---

# A. Standard sRGB 50% (Pixel value 128)
val_srgb_mid = 127.5 / 255.0  # Exact 0.5 float
y_srgb_mid = srgb_encoded_to_y(val_srgb_mid) # ~21.4%

# B. ZCAM (Calculated previously for Jz=50)
y_zcam = 23.17

# C. Hellwig 2022 (Calculated previously for Universal Contrast)
y_hellwig = 34.15

candidates = {
    "sRGB 50% (Pixel=128)": y_srgb_mid,
    "ZCAM (Jz=50)": y_zcam,
    "Hellwig '22 (Univ.)": y_hellwig
}

# --- 3. Plotting Logic ---

fig, axes = plt.subplots(3, 2, figsize=(10, 12), constrained_layout=True)

# Helper: Draws the isolated square checkerboard
def draw_checkerboard(ax, y_val, title):
    rgb = y_to_srgb(y_val)
    rgb_255 = (rgb * 255).astype(int)
    
    # 4x4 Grid
    for i in range(4):
        for j in range(4):
            is_white = (i + j) % 2 == 0
            bg_color = 'white' if is_white else 'black'
            
            # Background Tile
            ax.add_patch(plt.Rectangle((i, j), 1, 1, color=bg_color))
            
            # Isolated Gray Center Square (Size 0.4)
            ax.add_patch(plt.Rectangle((i+0.3, j+0.3), 0.4, 0.4, color=rgb))
            
    ax.set_xlim(0, 4)
    ax.set_ylim(0, 4)
    ax.axis('off')
    # Title with Y% and RGB values
    ax.set_title(f"{title}\nRGB: {tuple(rgb_255)} | Y={y_val:.1f}%", fontsize=11, fontweight='bold')

# Helper: Draws the Text Legibility Grid
def draw_text_grid(ax, y_val):
    rgb = y_to_srgb(y_val)
    
    # Split Background
    ax.add_patch(plt.Rectangle((0, 0), 0.5, 1, color='black'))
    ax.add_patch(plt.Rectangle((0.5, 0), 0.5, 1, color='white'))
    
    # Grid of text
    rows, cols = 4, 6
    for r in range(rows):
        for c in range(cols):
            x = (c + 0.5) / cols
            y = (r + 0.5) / rows
            ax.text(x, y, "Text", ha='center', va='center', 
                    color=rgb, fontsize=12, fontweight='bold', fontfamily='sans-serif')

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    
    # Labels for bottom of text grid
    ax.text(0.25, -0.05, "On Black", ha='center', fontsize=9, color='gray')
    ax.text(0.75, -0.05, "On White", ha='center', fontsize=9, color='gray')

# --- 4. Execution Loop ---

row = 0
for name, y_val in candidates.items():
    # Left Column: Checkerboard
    draw_checkerboard(axes[row, 0], y_val, name)
    # Right Column: Text Grid
    draw_text_grid(axes[row, 1], y_val)
    row += 1

plt.suptitle("Comparison: sRGB 50% vs ZCAM vs Hellwig\n(Grid & Text Legibility)", fontsize=14)

# Save or Show
plt.savefig('three_way_gray_test.png', dpi=120)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# --- 1. Color Science Utilities ---
def y_to_srgb(y):
    """
    Converts physical Luminance Y (0-100) to sRGB (0-1).
    Includes standard D65 matrix transformation and Gamma companding.
    """
    # Normalize Y to 0-1 range based on D65 White
    xyz = np.array([0.95047, 1.00000, 1.08883]) * (y / 100.0)
    
    # XYZ to Linear RGB (sRGB D65 Matrix)
    m_inv = np.array([[ 3.2404542, -1.5371385, -0.4985314],
                      [-0.9692660,  1.8760108,  0.0415560],
                      [ 0.0556434, -0.2040259,  1.0572252]])
    rgb_lin = np.dot(m_inv, xyz)
    
    # Gamma Correction (sRGB Companding)
    # If val <= 0.0031308, val * 12.92
    # Else, 1.055 * val^(1/2.4) - 0.055
    rgb = np.where(rgb_lin <= 0.0031308, 
                   12.92 * rgb_lin, 
                   1.055 * (np.maximum(rgb_lin, 0)**(1/2.4)) - 0.055)
    
    return np.clip(rgb, 0, 1)

# --- 2. Configuration ---
# Set seed for reproducible "noise"
np.random.seed(42)

candidates = {
    "sRGB 50% (Y=21.4)": 21.4,
    "ZCAM (Y=23.2)": 23.17,
    # "Compromise (Y=27.0)": 27.0,  # <-- Uncomment to test the 'Goldilocks' value
    "Hellwig '22 (Y=34.2)": 34.15
}

# --- 3. Generate Data ---
# A decaying sine wave with noise to simulate real scientific data
t = np.linspace(0, 10, 1000)
signal = np.sin(t) * np.exp(-t/5) + np.random.normal(0, 0.05, 1000)

# --- 4. Plotting ---
fig, axes = plt.subplots(len(candidates), 1, figsize=(10, 4 * len(candidates)), constrained_layout=True)

# Ensure axes is iterable even if only 1 candidate
if len(candidates) == 1: axes = [axes]

for ax, (name, y_val) in zip(axes, candidates.items()):
    rgb = y_to_srgb(y_val)
    rgb_tuple = tuple(rgb) # Matplotlib prefers tuples for explicit colors
    
    # A. Draw Backgrounds
    ax.set_facecolor('black') 
    # Draw White Rectangle covering the right half
    # (Coordinates: x=5 to 10, y=-2 to 2 covers the visible area)
    rect_white = patches.Rectangle((5, -2), 6, 4, facecolor='white', zorder=0)
    ax.add_patch(rect_white)
    
    # B. Draw Grids (The Stress Test)
    # Major Grid
    ax.grid(True, which='major', color=rgb_tuple, linestyle='-', linewidth=0.8, alpha=1.0)
    # Minor Grid (Dense)
    ax.minorticks_on()
    ax.grid(True, which='minor', color=rgb_tuple, linestyle=':', linewidth=0.5, alpha=1.0)
    
    # C. Plot Data
    ax.plot(t, signal, color=rgb_tuple, linewidth=0.8, label='Signal Data')
    
    # D. Annotations & Text
    # Tiny text to test "Irradiation" vs "Disappearance"
    ax.text(1, 0.8, "Fine Print on Black", color=rgb_tuple, fontsize=8, ha='center', weight='normal')
    ax.text(9, 0.8, "Fine Print on White", color=rgb_tuple, fontsize=8, ha='center', weight='normal')
    
    # Axis Labels
    ax.set_ylabel("Amplitude", color=rgb_tuple, fontsize=10, fontweight='bold')
    ax.set_xlabel("Time (s) - Crossing the Contrast Threshold", color=rgb_tuple, fontsize=10, fontweight='bold')
    
    # E. Styling the Container (Spines & Ticks)
    ax.tick_params(axis='x', colors=rgb_tuple, labelsize=9)
    ax.tick_params(axis='y', colors=rgb_tuple, labelsize=9)
    
    for spine in ax.spines.values():
        spine.set_color(rgb_tuple)
        spine.set_linewidth(1.0)

    # Title Label
    ax.text(5, 1.25, f"{name}\nRGB: {(rgb*255).astype(int)}", 
            color='black', fontsize=11, fontweight='bold', ha='center', va='center',
            bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.4'))
    
    # Set Limits
    ax.set_xlim(0, 10)
    ax.set_ylim(-1.2, 1.2)

plt.suptitle("Dense Grid & Fine Line Stress Test\n(ZCAM vs Hellwig vs sRGB)", fontsize=16, y=1.02)

# Save and Show
plt.savefig('dense_grid_test_local.png', dpi=150, bbox_inches='tight')
print("Plot generated and saved as 'dense_grid_test_local.png'")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

# --- 1. Utilities ---
def y_to_srgb_val(y):
    # D65 XYZ to sRGB (Linear)
    xyz = np.array([0.95047, 1.00000, 1.08883]) * (y / 100.0)
    m_inv = np.array([[ 3.2404542, -1.5371385, -0.4985314],
                      [-0.9692660,  1.8760108,  0.0415560],
                      [ 0.0556434, -0.2040259,  1.0572252]])
    rgb_lin = np.dot(m_inv, xyz)[1] # Green channel approx is enough for neutral
    
    # Gamma
    if rgb_lin <= 0.0031308:
        v = 12.92 * rgb_lin
    else:
        v = 1.055 * (max(rgb_lin, 0)**(1/2.4)) - 0.055
    return int(np.clip(v * 255, 0, 255))

# --- 2. Contrast Models ---

def get_apca_contrast(y_txt, y_bg):
    # Simplified SAPC-0.98 implementation for Neutral Colors
    # Constants
    scale = 1.14
    
    # Clamp for noise floor (Black level)
    y_txt = max(y_txt, 0.5)
    y_bg = max(y_bg, 0.5)
    
    if y_bg > y_txt: # Dark text on Light BG
        # SAPC: (Bg^0.56 - Txt^0.57)
        C = (y_bg**0.56) - (y_txt**0.57)
        return C * scale
    else: # Light text on Dark BG (Polarity Asymmetry)
        # SAPC: (Bg^0.65 - Txt^0.62)
        C = (y_bg**0.65) - (y_txt**0.62)
        # For calculation, we want positive magnitude comparison
        return abs(C) * scale

def get_hellwig_contrast(y_target, y_bg_surround):
    # Approximating the J (Lightness) calculation of Hellwig/CAM16
    # This is a simplified "power law" fit for speed, 
    # matching the J-behavior we saw in previous plots.
    # Hellwig J roughly follows a power function dependent on background.
    
    # On White BG, J is suppressed (Darker appearance)
    if y_bg_surround > 50: 
        j_val = 100 * (y_target / 100)**0.58 
    # On Black BG, J is boosted (Crispening)
    else:
        j_val = 100 * (y_target / 100)**0.42
        
    j_bg = 100 if y_bg_surround > 50 else 0
    return abs(j_val - j_bg)

# --- 3. The Solver ---
def solve_base_gray(opacity, model='APCA'):
    
    def objective(y_base):
        # 1. Blend Physics
        y_on_white = y_base * opacity + 100.0 * (1 - opacity)
        y_on_black = y_base * opacity + 0.5 * (1 - opacity)
        
        # 2. Contrast Math
        if model == 'APCA':
            c_white = get_apca_contrast(y_on_white, 100.0)
            c_black = get_apca_contrast(y_on_black, 0.5)
        else: # Hellwig
            c_white = get_hellwig_contrast(y_on_white, 100.0)
            c_black = get_hellwig_contrast(y_on_black, 0.5)
            
        return c_white - c_black

    try:
        # Search range 0 (Black Ink) to 100 (White Ink)
        opt_y = brentq(objective, 0, 100)
    except ValueError:
        opt_y = 0.0 # Fail
        
    return opt_y

# --- 4. Execution ---
# We test Opacities from 30% to 70%
opacities = [0.3, 0.4, 0.5, 0.6, 0.7]

print(f"{'Opacity':<10} | {'Model':<10} | {'Base Y':<10} | {'Base RGB':<15} | {'Result on White (Y)':<20} | {'Result on Black (Y)':<20}")
print("-" * 100)

for alpha in opacities:
    # Solve for APCA
    y_opt = solve_base_gray(alpha, 'APCA')
    rgb_opt = y_to_srgb_val(y_opt)
    res_w = y_opt * alpha + 100 * (1 - alpha)
    res_b = y_opt * alpha + 0.5 * (1 - alpha)
    
    print(f"{alpha*100:.0f}%{'':<6} | APCA       | {y_opt:.2f}{'':<6} | {rgb_opt:<15} | {res_w:.2f}{'':<16} | {res_b:.2f}")

# Visualizing the 50% case
opt_50 = solve_base_gray(0.5, 'APCA')
rgb_50 = y_to_srgb_val(opt_50) / 255.0

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
# Backgrounds
ax.add_patch(plt.Rectangle((0, 0), 0.5, 1, color='black'))
ax.add_patch(plt.Rectangle((0.5, 0), 0.5, 1, color='white'))

# The Translucent Squares
# Note: Matplotlib alpha blending works in sRGB space, not linear light physics.
# To be accurate, we manually calculated the result Y and convert to RGB.
res_w_50 = opt_50 * 0.5 + 100 * 0.5
res_b_50 = opt_50 * 0.5 + 0.5 * 0.5

def get_rgb_tuple(y):
    v = y_to_srgb_val(y)/255.0
    return (v, v, v)

# Draw resulting colors (simulating the blend)
rect_black = plt.Rectangle((0.15, 0.3), 0.2, 0.4, color=get_rgb_tuple(res_b_50))
rect_white = plt.Rectangle((0.65, 0.3), 0.2, 0.4, color=get_rgb_tuple(res_w_50))

ax.add_patch(rect_black)
ax.add_patch(rect_white)

# Labels
ax.text(0.25, 0.8, f"Base Gray\nY={opt_50:.1f}\nAlpha=50%", ha='center', color='gray', fontsize=10)
ax.text(0.25, 0.15, f"Result on Black\nY={res_b_50:.1f}", ha='center', color='white')
ax.text(0.75, 0.15, f"Result on White\nY={res_w_50:.1f}", ha='center', color='black')

ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
ax.set_title(f"The Optimized Translucent Gray (APCA Balanced)\nBase Ink: RGB({y_to_srgb_val(opt_50)}) @ 50% Opacity", fontsize=13)
plt.savefig('translucent_gray_solution.png')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

# --- 1. APCA Math (SAPC-0.98) ---
def get_apca_contrast(y_txt, y_bg):
    """
    Calculates APCA Lc score (SAPC-0.98).
    Inputs: Luminance Y (0.0 - 100.0)
    """
    y_txt = max(y_txt, 0.0)
    y_bg = max(y_bg, 0.0)
    
    scale = 1.14
    
    if y_bg > y_txt: # Dark on Light
        return (y_bg**0.56 - y_txt**0.57) * scale * 100
    else: # Light on Dark
        return (y_bg**0.65 - y_txt**0.62) * scale * 100

def y_to_srgb(y):
    """Converts Y to sRGB (0-255) tuple for display."""
    if y <= 0: return (0, 0, 0)
    if y >= 100: return (1, 1, 1)
    v = y / 100.0
    if v <= 0.0031308:
        val = 12.92 * v
    else:
        val = 1.055 * (v**(1/2.4)) - 0.055
    val = np.clip(val, 0, 1)
    return (val, val, val)

# --- 2. Optimization ---
def balance_objective(y):
    cb = abs(get_apca_contrast(y, 0.5))   # Contrast on Black
    cw = abs(get_apca_contrast(y, 100.0)) # Contrast on White
    return -min(cb, cw) # Negative for minimization

# Find exact peak
res = minimize_scalar(balance_objective, bounds=(0, 100), method='bounded')
y_optimal = res.x
lc_optimal = -res.fun
rgb_optimal = y_to_srgb(y_optimal)
srgb_int = int(rgb_optimal[0] * 255)

# --- 3. Numerical Proof ---
print(f"{'Luminance (Y)':<15} | {'sRGB':<10} | {'Lc on Black':<15} | {'Lc on White':<15} | {'Min Score':<20}")
print("-" * 80)

test_range = np.sort(np.append(np.linspace(20, 35, 16), y_optimal))

for y in test_range:
    srgb = int(y_to_srgb(y)[0] * 255)
    cb = abs(get_apca_contrast(y, 0.5))
    cw = abs(get_apca_contrast(y, 100.0))
    min_score = min(cb, cw)
    
    prefix = ">>> " if y == y_optimal else "    "
    print(f"{prefix}{y:<11.2f} | {srgb:<10} | {cb:<15.2f} | {cw:<15.2f} | {min_score:<20.2f}")

print("-" * 80)
print(f"OPTIMUM FOUND: Y={y_optimal:.4f}, sRGB=({srgb_int},{srgb_int},{srgb_int})")

# --- 4. Visual Check ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))

# Left: Black Background
ax1.add_patch(plt.Rectangle((0, 0), 1, 1, facecolor='black', zorder=-1)) # Explicit BG
ax1.add_patch(plt.Circle((0.5, 0.6), 0.25, color=rgb_optimal))
ax1.text(0.5, 0.2, "Text Sample", ha='center', va='center', color=rgb_optimal, fontweight='bold', fontsize=16)
ax1.set_xlim(0, 1); ax1.set_ylim(0, 1)
ax1.axis('off')
ax1.set_title(f"On Black\n(Lc {abs(get_apca_contrast(y_optimal, 0.5)):.1f})")

# Right: White Background
ax2.add_patch(plt.Rectangle((0, 0), 1, 1, facecolor='white', zorder=-1)) # Explicit BG
ax2.add_patch(plt.Circle((0.5, 0.6), 0.25, color=rgb_optimal))
ax2.text(0.5, 0.2, "Text Sample", ha='center', va='center', color=rgb_optimal, fontweight='bold', fontsize=16)
ax2.set_xlim(0, 1); ax2.set_ylim(0, 1)
ax2.axis('off')
ax2.set_title(f"On White\n(Lc {abs(get_apca_contrast(y_optimal, 100)):.1f})")

plt.tight_layout()
plt.savefig('optimal_gray_check.png')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

# --- Utilities ---
def get_apca_contrast(y_txt, y_bg):
    y_txt = max(y_txt, 0.0); y_bg = max(y_bg, 0.0); scale = 1.14
    if y_bg > y_txt: return (y_bg**0.56 - y_txt**0.57) * scale * 100
    else: return (y_bg**0.65 - y_txt**0.62) * scale * 100

def y_to_srgb(y):
    if y <= 0: return (0,0,0)
    if y >= 100: return (1,1,1)
    v = y / 100.0
    val = 12.92 * v if v <= 0.0031308 else 1.055 * (v**(1/2.4)) - 0.055
    return (val, val, val)

# --- Candidates ---
# 1. APCA Optimum (Calculated just now)
y_apca = 25.61
rgb_apca = y_to_srgb(y_apca)

# 2. Hellwig Optimum (Calculated in previous turn)
y_hellwig = 34.15
rgb_hellwig = y_to_srgb(y_hellwig)

# 3. Standard 128 (Middle Gray)
y_128 = 21.58
rgb_128 = (0.5, 0.5, 0.5)

candidates = [
    {"name": "APCA (Text Optimized)", "y": y_apca, "rgb": rgb_apca},
    {"name": "Hellwig (Shape Optimized)", "y": y_hellwig, "rgb": rgb_hellwig},
    {"name": "Standard Gray (128)", "y": y_128, "rgb": rgb_128}
]

# --- Plotting ---
plt.rcParams['font.family'] = 'monospace'
fig, axes = plt.subplots(3, 2, figsize=(8, 10))

for i, c in enumerate(candidates):
    # Calculate scores
    lc_b = abs(get_apca_contrast(c['y'], 0.5))
    lc_w = abs(get_apca_contrast(c['y'], 100.0))
    rgb_int = int(c['rgb'][0]*255)
    
    # Row Title
    # axes[i, 0].text(-0.1, 0.5, c['name'], transform=axes[i,0].transAxes, 
    #                 rotation=90, va='center', ha='right', fontweight='bold', fontsize=12)

    # Left: Black
    ax_b = axes[i, 0]
    ax_b.add_patch(plt.Rectangle((0,0), 1, 1, facecolor='black', zorder=-1))
    ax_b.add_patch(plt.Circle((0.5, 0.6), 0.25, color=c['rgb']))
    ax_b.text(0.5, 0.2, "Text Sample", ha='center', color=c['rgb'], fontsize=14, fontweight='bold')
    ax_b.axis('off'); ax_b.set_xlim(0,1); ax_b.set_ylim(0,1)
    ax_b.set_title(f"{c['name']}\nRGB {rgb_int} on Black\nLc {lc_b:.1f}")

    # Right: White
    ax_w = axes[i, 1]
    ax_w.add_patch(plt.Rectangle((0,0), 1, 1, facecolor='white', zorder=-1))
    ax_w.add_patch(plt.Circle((0.5, 0.6), 0.25, color=c['rgb']))
    ax_w.text(0.5, 0.2, "Text Sample", ha='center', color=c['rgb'], fontsize=14, fontweight='bold')
    ax_w.axis('off'); ax_w.set_xlim(0,1); ax_w.set_ylim(0,1)
    ax_w.set_title(f"{c['name']}\nRGB {rgb_int} on White\nLc {lc_w:.1f}")

plt.tight_layout()
plt.savefig('final_gray_shootout.png')

In [ ]:
import numpy as np
import torch
import os
from pathlib import Path
from skimage import io
import fastremap
import matplotlib.pyplot as plt

import plotly.graph_objects as go
from plotly.offline import plot
from IPython.display import HTML, display

import omnirefactor

dtype = torch.float64

# -------------------------------------------------------------------------
# 0. Helpers (Standard)
# -------------------------------------------------------------------------
def srgb_to_linear_np(srgb):
    srgb = np.asarray(srgb, dtype=np.float64)
    srgb = np.clip(srgb, 0.0, 1.0)
    mask = srgb <= 0.04045
    lin = np.empty_like(srgb)
    lin[mask] = srgb[mask] / 12.92
    lin[~mask] = ((srgb[~mask] + 0.055) / 1.055) ** 2.4
    return lin

def linear_to_srgb_np(lin):
    lin = np.asarray(lin, dtype=np.float64)
    sign = np.sign(lin); val = np.abs(lin)
    mask = val <= 0.0031308
    srgb = np.empty_like(lin)
    srgb[mask] = val[mask] * 12.92
    srgb[~mask] = 1.055 * (val[~mask] ** (1.0 / 2.4)) - 0.055
    return srgb * sign

M_srgb_xyz = np.array([[0.4124564, 0.3575761, 0.1804375],[0.2126729, 0.7151522, 0.0721750],[0.0193339, 0.1191920, 0.9503041]])
M_xyz_srgb = np.linalg.inv(M_srgb_xyz)
def srgb_to_xyz_np(srgb): return srgb_to_linear_np(srgb) @ M_srgb_xyz.T
def xyz_to_srgb_np(xyz): return linear_to_srgb_np(xyz @ M_xyz_srgb.T)

JZ_B = 1.15; JZ_G = 0.66
JZ_C1 = 3424/4096; JZ_C2 = 2413/128; JZ_C3 = 2392/128
JZ_N = 2610/16384; JZ_P = 1.7 * 2523/32; JZ_D = -0.56
M1 = np.array([[0.41478972, 0.579999, 0.0146480],[-0.2015100, 1.120649, 0.0531008],[-0.0166008, 0.264800, 0.6684799]])
M2 = np.array([[0.5, 0.5, 0],[3.524000, -4.066708, 0.542708],[0.199076, 1.096799, -1.295875]])
M1_INV = np.linalg.inv(M1); M2_INV = np.linalg.inv(M2)
DISPLAY_PEAK_NITS = 200.0 

def xyz_to_jzazbz(xyz):
    xyz_abs = xyz * DISPLAY_PEAK_NITS
    lms = np.maximum((np.stack([JZ_B*xyz_abs[...,0]-(JZ_B-1)*xyz_abs[...,2], JZ_G*xyz_abs[...,1]-(JZ_G-1)*xyz_abs[...,0], xyz_abs[...,2]], axis=-1)) @ M1.T, 1e-12)
    lms_p = np.power(lms, JZ_N)
    iz = np.maximum((JZ_C1 + JZ_C2 * lms_p) / (1 + JZ_C3 * lms_p), 1e-12)
    jab = np.power(iz, JZ_P) @ M2.T
    return np.stack([jab[..., 0] + JZ_D, jab[..., 1], jab[..., 2]], axis=-1)

def jzazbz_to_xyz(jab):
    iz_pow = np.maximum(np.stack([jab[...,0]-JZ_D, jab[...,1], jab[...,2]], axis=-1) @ M2_INV.T, 1e-12)
    iz = np.power(iz_pow, 1.0/JZ_P)
    lms_p = np.maximum((iz - JZ_C1) / (JZ_C2 - JZ_C3 * iz + 1e-12), 0)
    lms = np.power(lms_p, 1.0/JZ_N)
    xyz_p = lms @ M1_INV.T
    X = (xyz_p[...,0] + (JZ_B-1)*xyz_p[...,2]) / JZ_B
    Y = (xyz_p[...,1] + (JZ_G-1)*X) / JZ_G
    return np.stack([X, Y, xyz_p[...,2]], axis=-1) / DISPLAY_PEAK_NITS

# -------------------------------------------------------------------------
# 1. Adjustable Smoothness Logic
# -------------------------------------------------------------------------

def is_in_gamut(jz_point):
    xyz = jzazbz_to_xyz(jz_point)
    srgb = xyz_to_srgb_np(xyz)
    lin = srgb_to_linear_np(srgb)
    return np.all((lin >= -0.002) & (lin <= 1.002))

def get_rgb_hull_jz(n_samples=360):
    corners_rgb = np.array([[1,0,0],[1,1,0],[0,1,0],[0,1,1],[0,0,1],[1,0,1],[1,0,0]], dtype=float)
    rgb_path = []
    points_per_seg = n_samples // 6
    for i in range(6):
        start, end = corners_rgb[i], corners_rgb[i+1]
        for t in np.linspace(0, 1, points_per_seg, endpoint=False):
            rgb_path.append(start + t * (end - start))
    return xyz_to_jzazbz(srgb_to_xyz_np(np.array(rgb_path)))

def fft_smooth_hull(jz_hull, n_harmonics):
    """
    Applies a brick-wall low-pass filter in frequency domain.
    """
    N = len(jz_hull)
    # FFT each channel
    fft_J = np.fft.rfft(jz_hull[:,0])
    fft_a = np.fft.rfft(jz_hull[:,1])
    fft_b = np.fft.rfft(jz_hull[:,2])
    
    # Filter
    cutoff = n_harmonics + 1
    if cutoff < len(fft_J):
        fft_J[cutoff:] = 0
        fft_a[cutoff:] = 0
        fft_b[cutoff:] = 0
        
    # Inverse FFT
    return np.stack([
        np.fft.irfft(fft_J, n=N),
        np.fft.irfft(fft_a, n=N),
        np.fft.irfft(fft_b, n=N)
    ], axis=1)

def shrink_wrap(jz_smooth, iterations=50):
    """Shrinks the loop until it fits safely inside gamut."""
    center_J = np.mean(jz_smooth[:,0])
    curr = jz_smooth.copy()
    sC, sJ = 1.0, 1.0
    
    for i in range(iterations):
        if np.all([is_in_gamut(pt) for pt in curr]):
            return curr
        sC *= 0.98; sJ *= 0.99
        curr[:,1] = jz_smooth[:,1] * sC
        curr[:,2] = jz_smooth[:,2] * sC
        curr[:,0] = center_J + (jz_smooth[:,0] - center_J) * sJ
    return curr

def resample_uniform(jz_ring, n_target):
    ring_cyc = np.concatenate([jz_ring, jz_ring[:1]], axis=0)
    dists = np.linalg.norm(np.diff(ring_cyc, axis=0), axis=1)
    cum = np.concatenate([[0.0], np.cumsum(dists)])
    t_orig = cum / cum[-1]
    t_new = np.linspace(0, 1, n_target, endpoint=False)
    out = np.zeros((n_target, 3))
    for i in range(3): out[:,i] = np.interp(t_new, t_orig, ring_cyc[:,i])
    return out

# -------------------------------------------------------------------------
# 2. Main
# -------------------------------------------------------------------------

def run_order1_ellipse(n_hues=72, device_str="cpu"):
    dev = torch.device(device_str)
    
    # 1. Get Jagged Hull
    jz_hull = get_rgb_hull_jz(360)
    
    # 2. Generate Order 1 (Ellipse)
    # This keeps only the Fundamental frequency (Oval) and Mean.
    jz_h1 = shrink_wrap(fft_smooth_hull(jz_hull, n_harmonics=1))
    jz_h1 = resample_uniform(jz_h1, n_hues)
    
    # 3. Generate Order 2 for comparison (Squircle-ish)
    jz_h2 = shrink_wrap(fft_smooth_hull(jz_hull, n_harmonics=2))
    jz_h2 = resample_uniform(jz_h2, n_hues)
    
    # Convert
    rgb_h1 = np.clip(xyz_to_srgb_np(jzazbz_to_xyz(jz_h1)), 0, 1)
    rgb_h2 = np.clip(xyz_to_srgb_np(jzazbz_to_xyz(jz_h2)), 0, 1)
    
    # 4. Sinebow Baseline
    t = np.linspace(0, 1, n_hues, endpoint=False)
    sr = 0.5 + 0.5*np.sin(2*np.pi*(t+0/3))
    sg = 0.5 + 0.5*np.sin(2*np.pi*(t+1/3))
    sb = 0.5 + 0.5*np.sin(2*np.pi*(t+2/3))
    rgb_sine = np.stack([sr, sg, sb], axis=1)

    # 5. Alignment
    def align(rgb, ref_idx):
        r = rgb[:,0]; g = rgb[:,1]; b = rgb[:,2]
        ridx = np.argmax(r - 0.5*(g+b))
        r0 = np.roll(rgb, -ridx, axis=0)
        gidx = np.argmax(r0[:,1] - 0.5*(r0[:,0]+r0[:,2]))
        r_rev = r0[::-1].copy()
        gidx_rev = np.argmax(r_rev[:,1] - 0.5*(r_rev[:,0]+r_rev[:,2]))
        N = rgb.shape[0]
        d1 = min(abs(gidx - ref_idx), N - abs(gidx - ref_idx))
        d2 = min(abs(gidx_rev - ref_idx), N - abs(gidx_rev - ref_idx))
        return r0 if d1 <= d2 else r_rev
    
    # Align Sinebow to itself to find "Green"
    rgb_sine_a = align(rgb_sine, n_hues//3)
    g_loc = np.argmax(rgb_sine_a[:,1] - 0.5*(rgb_sine_a[:,0]+rgb_sine_a[:,2]))
    
    rgb_h1_a = align(rgb_h1, g_loc)
    rgb_h2_a = align(rgb_h2, g_loc)
    
    # 6. Omnipose Visuals
    omnidir = Path(omnirefactor.__file__).parent.parent.parent
    basedir = os.path.join(omnidir, "docs", "_static")
    masks = io.imread(os.path.join(basedir, "ecoli_labels.tif"))
    slc = omnirefactor.measure.crop_bbox(masks>0, pad=0)[0]
    masks = fastremap.renumber(masks[slc])[0]
    flows = omnirefactor.core.masks_to_flows(masks, device="cpu")[4].to(dev)
    center = srgb_to_linear_np(np.array([0.5,0.5,0.5]))
    
    def get_row(rgb):
        lin = srgb_to_linear_np(rgb)
        d,f,w,b = make_flow_images_for_ring(lin, center, flows)
        rot = np.roll(lin, n_hues//4, axis=0)
        _,_,wr,br = make_flow_images_for_ring(rot, center, flows)
        return [d,f,w,b,wr,br]

    r_sin = get_row(rgb_sine_a)
    r_h1 = get_row(rgb_h1_a)
    r_h2 = get_row(rgb_h2_a)
    
    fig, ax = plt.subplots(3,6, figsize=(24,10))
    def clean(a): a.axis('off')
    names = ["Sinebow", "Order 1 (Ellipse)", "Order 2 (Squircle)"]
    data = [r_sin, r_h1, r_h2]
    cols = ["Disk", "Flow", "Alpha/W", "Alpha/B", "Rot/W", "Rot/B"]
    
    for i in range(3):
        for j in range(6):
            ax[i,j].imshow(data[i][j])
            clean(ax[i,j])
            if i==0: ax[i,j].set_title(cols[j])
            if j==0: ax[i,j].text(-0.2, 0.5, names[i], transform=ax[i,j].transAxes, va='center', ha='right', fontsize=12, rotation=90)
            
    plt.tight_layout()
    plt.show()
    
    # 7. Plot 3D Trajectories
    jz_sin = xyz_to_jzazbz(srgb_to_xyz_np(rgb_sine_a))
    jz_h1 = xyz_to_jzazbz(srgb_to_xyz_np(rgb_h1_a))
    jz_h2 = xyz_to_jzazbz(srgb_to_xyz_np(rgb_h2_a))
    
    fig3d = go.Figure()
    # Cube
    grid = np.linspace(0,1,17); G, B = np.meshgrid(grid, grid)
    def sface(ch, v, n):
        if ch==0: s=np.stack([np.full_like(G,v),G,B],axis=-1)
        elif ch==1: s=np.stack([G,np.full_like(G,v),B],axis=-1)
        else: s=np.stack([G,B,np.full_like(G,v)],axis=-1)
        jz = xyz_to_jzazbz(srgb_to_xyz_np(s.reshape(-1,3))).reshape(17,17,3)
        fig3d.add_trace(go.Surface(x=jz[...,1],y=jz[...,2],z=jz[...,0],opacity=0.1,showscale=False,hoverinfo='skip'))
    for c,v,n in [(0,0,''),(0,1,''),(1,0,''),(1,1,''),(2,0,''),(2,1,'')]: sface(c,v,n)

    fig3d.add_trace(go.Scatter3d(x=jz_sin[:,1],y=jz_sin[:,2],z=jz_sin[:,0], mode='lines', name='Sinebow', line=dict(width=3, color='yellow')))
    fig3d.add_trace(go.Scatter3d(x=jz_h1[:,1],y=jz_h1[:,2],z=jz_h1[:,0], mode='lines', name='Order 1 (Ellipse)', line=dict(width=10, color='cyan')))
    fig3d.add_trace(go.Scatter3d(x=jz_h2[:,1],y=jz_h2[:,2],z=jz_h2[:,0], mode='lines', name='Order 2 (Squircle)', line=dict(width=5, color='magenta')))
    
    fig3d.update_layout(template="plotly_dark", height=600, title="Order 1 vs Order 2 JzAzBz Smoothing")
    display(HTML(plot(fig3d, output_type='div', include_plotlyjs='cdn')))

if __name__ == "__main__":
    if 'make_flow_images_for_ring' not in locals():
        def make_flow_images_for_ring(a,b,c): return np.zeros((10,10,3)), np.zeros((10,10,3)), np.zeros((10,10,3)), np.zeros((10,10,3))
    run_order1_ellipse(72)

In [ ]:
import numpy as np
import torch
import os
from pathlib import Path
from skimage import io
import fastremap
import matplotlib.pyplot as plt

import plotly.graph_objects as go
from plotly.offline import plot
from IPython.display import HTML, display

import omnirefactor

dtype = torch.float64

# -------------------------------------------------------------------------
# 0. JzAzBz Color Conversion (200 nits)
# -------------------------------------------------------------------------

def srgb_to_linear_np(srgb):
    srgb = np.clip(np.asarray(srgb, dtype=np.float64), 0.0, 1.0)
    mask = srgb <= 0.04045
    lin = np.empty_like(srgb)
    lin[mask] = srgb[mask] / 12.92
    lin[~mask] = ((srgb[~mask] + 0.055) / 1.055) ** 2.4
    return lin

def linear_to_srgb_np(lin):
    lin = np.asarray(lin, dtype=np.float64)
    # Strict check for validity
    valid_mask = (lin >= -0.0005) & (lin <= 1.0005)
    
    lin_clamped = np.clip(lin, 0.0, 1.0)
    mask = lin_clamped <= 0.0031308
    srgb = np.empty_like(lin_clamped)
    srgb[mask] = lin_clamped[mask] * 12.92
    srgb[~mask] = 1.055 * (lin_clamped[~mask] ** (1.0 / 2.4)) - 0.055
    
    return srgb, np.all(valid_mask, axis=-1)

M_srgb_xyz = np.array([[0.4124564, 0.3575761, 0.1804375],[0.2126729, 0.7151522, 0.0721750],[0.0193339, 0.1191920, 0.9503041]])
M_xyz_srgb = np.linalg.inv(M_srgb_xyz)
def srgb_to_xyz_np(srgb): return srgb_to_linear_np(srgb) @ M_srgb_xyz.T
def xyz_to_srgb_np(xyz): return linear_to_srgb_np(xyz @ M_xyz_srgb.T)

# Safdar 2017 Constants
JZ_B = 1.15; JZ_G = 0.66
JZ_C1 = 3424/4096; JZ_C2 = 2413/128; JZ_C3 = 2392/128
JZ_N = 2610/16384; JZ_P = 1.7 * 2523/32; JZ_D = -0.56
M1 = np.array([[0.41478972, 0.579999, 0.0146480],[-0.2015100, 1.120649, 0.0531008],[-0.0166008, 0.264800, 0.6684799]])
M2 = np.array([[0.5, 0.5, 0],[3.524000, -4.066708, 0.542708],[0.199076, 1.096799, -1.295875]])
DISPLAY_PEAK_NITS = 200.0 

def xyz_to_jzazbz(xyz):
    xyz_abs = xyz * DISPLAY_PEAK_NITS
    lms = np.maximum((np.stack([JZ_B*xyz_abs[...,0]-(JZ_B-1)*xyz_abs[...,2], JZ_G*xyz_abs[...,1]-(JZ_G-1)*xyz_abs[...,0], xyz_abs[...,2]], axis=-1)) @ M1.T, 1e-12)
    iz = np.maximum((JZ_C1 + JZ_C2 * np.power(lms, JZ_N)) / (1 + JZ_C3 * np.power(lms, JZ_N)), 1e-12)
    jab = np.power(iz, JZ_P) @ M2.T
    return np.stack([jab[..., 0] + JZ_D, jab[..., 1], jab[..., 2]], axis=-1)

def jzazbz_to_xyz(jab):
    iz = np.power(np.maximum(np.stack([jab[...,0]-JZ_D, jab[...,1], jab[...,2]], axis=-1) @ np.linalg.inv(M2).T, 1e-12), 1.0/JZ_P)
    lms = np.power(np.maximum((iz - JZ_C1) / (JZ_C2 - JZ_C3 * iz + 1e-12), 0), 1.0/JZ_N)
    xyz_p = lms @ np.linalg.inv(M1).T
    X = (xyz_p[...,0] + (JZ_B-1)*xyz_p[...,2]) / JZ_B
    Y = (xyz_p[...,1] + (JZ_G-1)*X) / JZ_G
    return np.stack([X, Y, xyz_p[...,2]], axis=-1) / DISPLAY_PEAK_NITS

# -------------------------------------------------------------------------
# 1. HSV Boundary Logic (Fixed)
# -------------------------------------------------------------------------

def get_hsv_boundary(n=360):
    """
    Generates the precise JzAzBz coordinates of the HSV boundary (S=1, V=1).
    This defines the 'ridges' of the sRGB gamut.
    """
    h = np.linspace(0, 1, n, endpoint=False)
    i = (h * 6).astype(int)
    f = h * 6 - i
    
    # Pre-calculate components as full arrays to avoid shape mismatch
    v_ones = np.ones_like(h)
    p_zeros = np.zeros_like(h)
    q = 1 - f
    t = f
    
    rgb = np.zeros((n, 3))
    i = i % 6
    
    # Explicit stacking for each sector
    mask = i == 0
    rgb[mask] = np.stack([v_ones[mask], t[mask], p_zeros[mask]], axis=1)
    
    mask = i == 1
    rgb[mask] = np.stack([q[mask], v_ones[mask], p_zeros[mask]], axis=1)
    
    mask = i == 2
    rgb[mask] = np.stack([p_zeros[mask], v_ones[mask], t[mask]], axis=1)
    
    mask = i == 3
    rgb[mask] = np.stack([p_zeros[mask], q[mask], v_ones[mask]], axis=1)
    
    mask = i == 4
    rgb[mask] = np.stack([t[mask], p_zeros[mask], v_ones[mask]], axis=1)
    
    mask = i == 5
    rgb[mask] = np.stack([v_ones[mask], p_zeros[mask], q[mask]], axis=1)
    
    return xyz_to_jzazbz(srgb_to_xyz_np(rgb))

def fourier_approx(curve, n_harmonics):
    """Applies a brick-wall low-pass filter to the J, a, b coords."""
    N = len(curve)
    smooth = np.zeros_like(curve)
    for i in range(3):
        fft = np.fft.rfft(curve[:,i])
        # Zero out everything above the nth harmonic
        # rfft array indices: 0=DC, 1=Freq1, etc.
        if n_harmonics + 1 < len(fft):
            fft[n_harmonics+1:] = 0
        smooth[:,i] = np.fft.irfft(fft, n=N)
    return smooth

def clip_to_gamut(jz_curve):
    """
    Pulls invalid points back to the gamut wall along the chroma vector.
    Does NOT shrink the entire ring, only the parts that bulge out.
    """
    xyz = jzazbz_to_xyz(jz_curve)
    _, valid = xyz_to_srgb_np(xyz)
    
    if np.all(valid): return jz_curve
    
    # Only process invalid points
    clipped = jz_curve.copy()
    idxs = np.where(~valid)[0]
    
    print(f"  Clipping {len(idxs)} points to gamut wall...")
    
    for i in idxs:
        J, a, b = jz_curve[i]
        # Binary search scale factor [0, 1]
        low, high = 0.0, 1.0
        for _ in range(12):
            mid = (low + high) / 2
            pt = np.array([J, a*mid, b*mid])
            xyz_pt = jzazbz_to_xyz(pt)
            lin_pt = srgb_to_linear_np(xyz_to_srgb_np(xyz_pt)[0])
            
            # Check strict bounds
            if np.all((lin_pt >= -0.001) & (lin_pt <= 1.001)):
                low = mid
            else:
                high = mid
        
        scale = low
        clipped[i, 1] *= scale
        clipped[i, 2] *= scale
        
    return clipped

def resample_arc(curve, n_target):
    # Cyclic closure for distance calculation
    closed = np.vstack([curve, curve[0]])
    dist = np.cumsum(np.linalg.norm(np.diff(closed, axis=0), axis=1))
    dist = np.insert(dist, 0, 0)
    
    # Uniform targets
    t_uni = np.linspace(0, dist[-1], n_target, endpoint=False)
    
    out = np.zeros((n_target, 3))
    for i in range(3):
        out[:,i] = np.interp(t_uni, dist, closed[:,i])
    return out

# -------------------------------------------------------------------------
# 2. Visualization (Normalized Flow)
# -------------------------------------------------------------------------

def generate_vis_row(rgb_ring, flows_np, n_hues):
    # 1. Disk
    y, x = np.ogrid[-1:1:complex(0, 256), -1:1:complex(0, 256)]
    mask = x**2 + y**2 > 1.0
    ang = np.arctan2(y, x)
    idx = (((ang + np.pi) / (2*np.pi)) * len(rgb_ring)).astype(int) % len(rgb_ring)
    disk = rgb_ring[idx]
    disk[mask] = 0
    
    # 2. Flow (Normalized for visibility)
    dy, dx = flows_np[0], flows_np[1]
    mag = np.sqrt(dy**2 + dx**2)
    
    # Robust normalization to handle outliers
    p99 = np.percentile(mag, 99)
    if p99 < 1e-5: p99 = 1.0
    alpha = np.clip(mag / p99, 0, 1)[..., None]
    
    ang_f = np.arctan2(dy, dx)
    
    def get_col(r, a): 
        ix = (((a + np.pi) / (2*np.pi)) * len(r)).astype(int) % len(r)
        return r[ix]
    
    c = get_col(rgb_ring, ang_f)
    
    f = c * alpha + 0.1 * (1-alpha) # Slight grey background for flow
    ow = c * alpha + (1.0 - alpha)
    ob = c * alpha
    
    # 3. Rotated
    rot = np.roll(rgb_ring, n_hues//4, axis=0)
    cr = get_col(rot, ang_f)
    rw = cr * alpha + (1.0 - alpha)
    rb = cr * alpha
    
    return [disk, f, ow, ob, rw, rb]

# -------------------------------------------------------------------------
# 3. Main
# -------------------------------------------------------------------------

def run_final_fourier(n_hues=72, harmonics=5):
    # 1. Get Truth
    jz_raw = get_hsv_boundary(360)
    
    # 2. Smooth
    jz_smooth = fourier_approx(jz_raw, harmonics)
    
    # 3. Clip
    jz_clipped = clip_to_gamut(jz_smooth)
    
    # 4. Resample
    jz_opt = resample_arc(jz_clipped, n_hues)
    
    # 5. RGB
    xyz = jzazbz_to_xyz(jz_opt)
    rgb_opt, _ = xyz_to_srgb_np(xyz)
    rgb_opt = np.clip(rgb_opt, 0, 1)
    
    # --- Comparators ---
    t = np.linspace(0, 1, n_hues, endpoint=False)
    # Sinebow
    sr = 0.5 + 0.5*np.sin(2*np.pi*(t+0/3))
    sg = 0.5 + 0.5*np.sin(2*np.pi*(t+1/3))
    sb = 0.5 + 0.5*np.sin(2*np.pi*(t+2/3))
    rgb_sine = np.stack([sr, sg, sb], axis=1)
    
    # Align
    def get_g(r): return np.argmax(r[:,1] - 0.5*(r[:,0]+r[:,2]))
    g_ref = get_g(rgb_sine)
    curr = get_g(rgb_opt)
    rgb_opt = np.roll(rgb_opt, g_ref - curr, axis=0)
    
    # --- Vis ---
    omnidir = Path(omnirefactor.__file__).parent.parent.parent
    masks_path = os.path.join(omnidir, "docs", "_static", "ecoli_labels.tif")
    
    # Fallback if file missing
    if not os.path.exists(masks_path):
        print("Omnipose data not found, using synthetic flow.")
        y, x = np.ogrid[-100:100, -100:100]
        flows_np = np.stack([y/100.0, x/100.0], axis=0)
    else:
        masks = io.imread(masks_path)
        slc = omnirefactor.measure.crop_bbox(masks>0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        # Assuming gpu available, else cpu
        dev = "cpu"
        flows_tensor = omnirefactor.core.masks_to_flows(masks, device=dev)[4]
        flows_np = flows_tensor.cpu().numpy() if hasattr(flows_tensor, 'cpu') else flows_tensor

    row_s = generate_vis_row(rgb_sine, flows_np, n_hues)
    row_o = generate_vis_row(rgb_opt, flows_np, n_hues)
    
    fig, ax = plt.subplots(2, 6, figsize=(24, 8))
    cols = ["Disk", "Flow", "Alpha/W", "Alpha/B", "Rot/W", "Rot/B"]
    data = [row_s, row_o]
    names = ["Sinebow", f"Fourier HSV (H={harmonics})"]
    
    for i in range(2):
        for j in range(6):
            ax[i,j].imshow(data[i][j])
            ax[i,j].axis('off')
            if i==0: ax[i,j].set_title(cols[j])
            if j==0: ax[i,j].text(-0.1, 0.5, names[i], transform=ax[i,j].transAxes, va='center', ha='right', rotation=90, fontsize=12)
            
    plt.tight_layout()
    plt.show()
    
    # 3D
    jz_sin = xyz_to_jzazbz(srgb_to_xyz_np(rgb_sine))
    jz_opt_plot = xyz_to_jzazbz(srgb_to_xyz_np(rgb_opt))
    jz_hull = get_hsv_boundary(360)
    
    fig3d = go.Figure()
    
    # Simple Grid Surface
    grid = np.linspace(0,1,17); G, B = np.meshgrid(grid, grid)
    def sface(ch, v):
        if ch==0: s=np.stack([np.full_like(G,v),G,B],axis=-1)
        elif ch==1: s=np.stack([G,np.full_like(G,v),B],axis=-1)
        else: s=np.stack([G,B,np.full_like(G,v)],axis=-1)
        jz = xyz_to_jzazbz(srgb_to_xyz_np(s.reshape(-1,3))).reshape(17,17,3)
        fig3d.add_trace(go.Surface(x=jz[...,1],y=jz[...,2],z=jz[...,0],opacity=0.1,showscale=False,hoverinfo='skip'))
    for c in range(3): 
        for v in [0,1]: sface(c,v)

    fig3d.add_trace(go.Scatter3d(x=jz_hull[:,1],y=jz_hull[:,2],z=jz_hull[:,0], mode='lines', name='Raw Gamut Edge', line=dict(width=2, color='white', dash='dot')))
    fig3d.add_trace(go.Scatter3d(x=jz_sin[:,1],y=jz_sin[:,2],z=jz_sin[:,0], mode='lines', name='Sinebow', line=dict(width=4, color='yellow')))
    fig3d.add_trace(go.Scatter3d(x=jz_opt_plot[:,1],y=jz_opt_plot[:,2],z=jz_opt_plot[:,0], mode='lines', name=f'Fourier (H={harmonics})', line=dict(width=10, color='cyan')))
    
    fig3d.update_layout(template="plotly_dark", height=600, title=f"Fourier HSV Approximation (H={harmonics})")
    display(HTML(plot(fig3d, output_type='div', include_plotlyjs='cdn')))

if __name__ == "__main__":
    run_final_fourier(150, harmonics=1)

In [ ]:
import numpy as np
import torch
import os
from pathlib import Path
from skimage import io
import fastremap
import matplotlib.pyplot as plt

import plotly.graph_objects as go
from plotly.offline import plot
from IPython.display import HTML, display

import omnirefactor

dtype = torch.float64

# -------------------------------------------------------------------------
# 0. JzAzBz Conversion
# -------------------------------------------------------------------------
def srgb_to_linear_np(srgb):
    srgb = np.clip(np.asarray(srgb, dtype=np.float64), 0.0, 1.0)
    mask = srgb <= 0.04045
    lin = np.empty_like(srgb)
    lin[mask] = srgb[mask] / 12.92
    lin[~mask] = ((srgb[~mask] + 0.055) / 1.055) ** 2.4
    return lin

def linear_to_srgb_np(lin):
    lin = np.asarray(lin, dtype=np.float64)
    valid_mask = (lin >= -0.0001) & (lin <= 1.0001)
    lin_clamped = np.clip(lin, 0.0, 1.0)
    mask = lin_clamped <= 0.0031308
    srgb = np.empty_like(lin_clamped)
    srgb[mask] = lin_clamped[mask] * 12.92
    srgb[~mask] = 1.055 * (lin_clamped[~mask] ** (1.0 / 2.4)) - 0.055
    return srgb, np.all(valid_mask, axis=-1)

M_srgb_xyz = np.array([[0.4124564, 0.3575761, 0.1804375],[0.2126729, 0.7151522, 0.0721750],[0.0193339, 0.1191920, 0.9503041]])
M_xyz_srgb = np.linalg.inv(M_srgb_xyz)
def srgb_to_xyz_np(srgb): return srgb_to_linear_np(srgb) @ M_srgb_xyz.T
def xyz_to_srgb_np(xyz): return linear_to_srgb_np(xyz @ M_xyz_srgb.T)

JZ_B = 1.15; JZ_G = 0.66
JZ_C1 = 3424/4096; JZ_C2 = 2413/128; JZ_C3 = 2392/128
JZ_N = 2610/16384; JZ_P = 1.7 * 2523/32; JZ_D = -0.56
M1 = np.array([[0.41478972, 0.579999, 0.0146480],[-0.2015100, 1.120649, 0.0531008],[-0.0166008, 0.264800, 0.6684799]])
M2 = np.array([[0.5, 0.5, 0],[3.524000, -4.066708, 0.542708],[0.199076, 1.096799, -1.295875]])
DISPLAY_PEAK_NITS = 200.0 

def xyz_to_jzazbz(xyz):
    xyz_abs = xyz * DISPLAY_PEAK_NITS
    lms = np.maximum((np.stack([JZ_B*xyz_abs[...,0]-(JZ_B-1)*xyz_abs[...,2], JZ_G*xyz_abs[...,1]-(JZ_G-1)*xyz_abs[...,0], xyz_abs[...,2]], axis=-1)) @ M1.T, 1e-12)
    iz = np.maximum((JZ_C1 + JZ_C2 * np.power(lms, JZ_N)) / (1 + JZ_C3 * np.power(lms, JZ_N)), 1e-12)
    jab = np.power(iz, JZ_P) @ M2.T
    return np.stack([jab[..., 0] + JZ_D, jab[..., 1], jab[..., 2]], axis=-1)

def jzazbz_to_xyz(jab):
    iz = np.power(np.maximum(np.stack([jab[...,0]-JZ_D, jab[...,1], jab[...,2]], axis=-1) @ np.linalg.inv(M2).T, 1e-12), 1.0/JZ_P)
    lms = np.power(np.maximum((iz - JZ_C1) / (JZ_C2 - JZ_C3 * iz + 1e-12), 0), 1.0/JZ_N)
    xyz_p = lms @ np.linalg.inv(M1).T
    X = (xyz_p[...,0] + (JZ_B-1)*xyz_p[...,2]) / JZ_B
    Y = (xyz_p[...,1] + (JZ_G-1)*X) / JZ_G
    return np.stack([X, Y, xyz_p[...,2]], axis=-1) / DISPLAY_PEAK_NITS

# -------------------------------------------------------------------------
# 1. Geometry Helpers
# -------------------------------------------------------------------------

def get_hsv_boundary(n=360):
    h = np.linspace(0, 1, n, endpoint=False)
    i = (h * 6).astype(int); f = h * 6 - i
    q = 1 - f; t = f
    rgb = np.zeros((n, 3))
    mask = i%6==0; rgb[mask] = np.stack([np.ones(mask.sum()), t[mask], np.zeros(mask.sum())], axis=1)
    mask = i%6==1; rgb[mask] = np.stack([q[mask], np.ones(mask.sum()), np.zeros(mask.sum())], axis=1)
    mask = i%6==2; rgb[mask] = np.stack([np.zeros(mask.sum()), np.ones(mask.sum()), t[mask]], axis=1)
    mask = i%6==3; rgb[mask] = np.stack([np.zeros(mask.sum()), q[mask], np.ones(mask.sum())], axis=1)
    mask = i%6==4; rgb[mask] = np.stack([t[mask], np.zeros(mask.sum()), np.ones(mask.sum())], axis=1)
    mask = i%6==5; rgb[mask] = np.stack([np.ones(mask.sum()), np.zeros(mask.sum()), q[mask]], axis=1)
    return xyz_to_jzazbz(srgb_to_xyz_np(rgb))

def get_srgb_wireframe_jz(n_per_edge=20):
    edges = [
        ([0,0,0],[1,0,0]), ([1,0,0],[1,1,0]), ([1,1,0],[0,1,0]), ([0,1,0],[0,0,0]),
        ([0,0,1],[1,0,1]), ([1,0,1],[1,1,1]), ([1,1,1],[0,1,1]), ([0,1,1],[0,0,1]),
        ([0,0,0],[0,0,1]), ([1,0,0],[1,0,1]), ([1,1,0],[1,1,1]), ([0,1,0],[0,1,1])
    ]
    x, y, z = [], [], []
    for start, end in edges:
        t = np.linspace(0, 1, n_per_edge)
        rgb = np.array(start) + t[:,None] * (np.array(end) - np.array(start))
        jz = xyz_to_jzazbz(srgb_to_xyz_np(rgb))
        x.extend(jz[:,1]); y.extend(jz[:,2]); z.extend(jz[:,0])
        x.append(None); y.append(None); z.append(None)
    return x, y, z

def fourier_approx(curve, n_harmonics):
    N = len(curve)
    smooth = np.zeros_like(curve)
    for i in range(3):
        fft = np.fft.rfft(curve[:,i])
        if n_harmonics + 1 < len(fft):
            fft[n_harmonics+1:] = 0
        smooth[:,i] = np.fft.irfft(fft, n=N)
    return smooth

def shrink_to_fit(jz_curve):
    center = np.mean(jz_curve, axis=0)
    centered = jz_curve - center
    low, high = 0.0, 1.5
    best_scale = 0.0
    for _ in range(20):
        scale = (low + high) / 2
        test = center + centered * scale
        _, valid = xyz_to_srgb_np(jzazbz_to_xyz(test))
        if np.all(valid):
            best_scale = scale
            low = scale
        else:
            high = scale
    print(f"  Optimization: Centered Ellipse Scaled by {best_scale:.3f}")
    return center + centered * best_scale

def resample_arc(curve, n_target):
    closed = np.vstack([curve, curve[0]])
    dist = np.cumsum(np.linalg.norm(np.diff(closed, axis=0), axis=1))
    dist = np.insert(dist, 0, 0)
    t_uni = np.linspace(0, dist[-1], n_target, endpoint=False)
    out = np.zeros((n_target, 3))
    for i in range(3):
        out[:,i] = np.interp(t_uni, dist, closed[:,i])
    return out

# -------------------------------------------------------------------------
# 2. Visualization
# -------------------------------------------------------------------------

def generate_vis_row(rgb_ring, flows_np, n_hues):
    y, x = np.ogrid[-1:1:complex(0, 256), -1:1:complex(0, 256)]
    mask = x**2 + y**2 > 1.0
    ang = np.arctan2(y, x)
    idx = (((ang + np.pi) / (2*np.pi)) * len(rgb_ring)).astype(int) % len(rgb_ring)
    disk = rgb_ring[idx].copy()
    disk[mask] = 0
    
    dy, dx = flows_np[0], flows_np[1]
    mag = np.sqrt(dy**2 + dx**2)
    p99 = np.percentile(mag, 99)
    if p99 < 1e-5: p99 = 1.0
    alpha = np.clip(mag / p99, 0, 1)[..., None]
    
    ang_f = np.arctan2(dy, dx)
    def get_c(r, a): return r[(((a + np.pi) / (2*np.pi)) * len(r)).astype(int) % len(r)]
    
    c = get_c(rgb_ring, ang_f)
    f = c * alpha + 0.1*(1-alpha)
    ow = c * alpha + (1.0 - alpha)
    ob = c * alpha
    
    rot = np.roll(rgb_ring, n_hues//4, axis=0)
    cr = get_c(rot, ang_f)
    rw = cr * alpha + (1.0 - alpha)
    rb = cr * alpha
    return [disk, f, ow, ob, rw, rb]

# -------------------------------------------------------------------------
# 3. Main
# -------------------------------------------------------------------------

def run_true_ellipse(n_hues=72, harmonics=1):
    # 1. Data Gen
    jz_raw = get_hsv_boundary(360)
    # RGB for dots
    xyz_raw = jzazbz_to_xyz(jz_raw)
    rgb_raw_safe, _ = xyz_to_srgb_np(xyz_raw)
    rgb_raw_safe = np.clip(rgb_raw_safe, 0, 1)
    
    # 2. Geometry
    jz_geo = fourier_approx(jz_raw, harmonics)
    
    # 3. Shrink
    jz_opt = shrink_to_fit(jz_geo)
    
    # 4. Resample
    jz_final = resample_arc(jz_opt, n_hues)
    
    # 5. Final RGB
    xyz = jzazbz_to_xyz(jz_final)
    rgb_opt, _ = xyz_to_srgb_np(xyz)
    rgb_opt = np.clip(rgb_opt, 0, 1)
    
    # --- Comparators ---
    t = np.linspace(0, 1, n_hues, endpoint=False)
    sr = 0.5 + 0.5*np.sin(2*np.pi*(t+0/3))
    sg = 0.5 + 0.5*np.sin(2*np.pi*(t+1/3))
    sb = 0.5 + 0.5*np.sin(2*np.pi*(t+2/3))
    rgb_sine = np.stack([sr, sg, sb], axis=1)
    
    def get_g(r): return np.argmax(r[:,1] - 0.5*(r[:,0]+r[:,2]))
    g_ref = get_g(rgb_sine)
    rgb_opt_a = np.roll(rgb_opt, g_ref - get_g(rgb_opt), axis=0)
    
    # --- Vis ---
    omnidir = Path(omnirefactor.__file__).parent.parent.parent
    masks = io.imread(os.path.join(omnidir, "docs", "_static", "ecoli_labels.tif"))
    slc = omnirefactor.measure.crop_bbox(masks>0, pad=0)[0]
    masks = fastremap.renumber(masks[slc])[0]
    flows_tensor = omnirefactor.core.masks_to_flows(masks, device="cpu")[4]
    flows_np = flows_tensor.cpu().numpy() if hasattr(flows_tensor, 'cpu') else flows_tensor

    row_s = generate_vis_row(rgb_sine, flows_np, n_hues)
    row_o = generate_vis_row(rgb_opt_a, flows_np, n_hues)
    
    fig, ax = plt.subplots(2, 6, figsize=(24, 8))
    cols = ["Disk", "Flow", "Alpha/W", "Alpha/B", "Rot/W", "Rot/B"]
    data = [row_s, row_o]
    names = ["Sinebow", f"True Ellipse (H={harmonics})"]
    
    for i in range(2):
        for j in range(6):
            ax[i,j].imshow(data[i][j]); ax[i,j].axis('off')
            if i==0: ax[i,j].set_title(cols[j])
            if j==0: ax[i,j].text(-0.1, 0.5, names[i], transform=ax[i,j].transAxes, va='center', ha='right', rotation=90, fontsize=12)
    plt.tight_layout()
    plt.show()
    
    # --- 3D Plot ---
    jz_final_plot = xyz_to_jzazbz(srgb_to_xyz_np(rgb_opt_a))
    
    fig3d = go.Figure()
    
    # 1. Faces (Translucent)
    grid = np.linspace(0,1,17); G, B = np.meshgrid(grid, grid)
    def sface(ch, v):
        if ch==0: s=np.stack([np.full_like(G,v),G,B],axis=-1)
        elif ch==1: s=np.stack([G,np.full_like(G,v),B],axis=-1)
        else: s=np.stack([G,B,np.full_like(G,v)],axis=-1)
        jz = xyz_to_jzazbz(srgb_to_xyz_np(s.reshape(-1,3))).reshape(17,17,3)
        fig3d.add_trace(go.Surface(x=jz[...,1],y=jz[...,2],z=jz[...,0],opacity=0.1,showscale=False,hoverinfo='skip'))
    for c in range(3): 
        for v in [0,1]: sface(c,v)
        
    # 2. Wireframe (Edges)
    wx, wy, wz = get_srgb_wireframe_jz()
    fig3d.add_trace(go.Scatter3d(x=wx, y=wy, z=wz, mode='lines', name='sRGB Wireframe', line=dict(color='gray', width=3), hoverinfo='skip'))
    
    # 3. HSV Truth (Dots)
    
    fig3d.add_trace(go.Scatter3d(x=jz_raw[:,1], y=jz_raw[:,2], z=jz_raw[:,0], mode='markers', name='HSV Truth', marker=dict(size=3, color=rgb_raw_safe)))
    
    # 4. Loop (Dots)
    fig3d.add_trace(go.Scatter3d(x=jz_final_plot[:,1], y=jz_final_plot[:,2], z=jz_final_plot[:,0], mode='markers', name=f'True Ellipse (H={harmonics})', marker=dict(size=6, color=rgb_opt_a)))
    
    fig3d.update_layout(template="plotly_dark", height=600, title=f"True Geometric Ellipse (H={harmonics}) in sRGB")
    display(HTML(plot(fig3d, output_type='div', include_plotlyjs='cdn')))

if __name__ == "__main__":
    run_true_ellipse(72, harmonics=1)

In [ ]:
import numpy as np
import torch
import os
from pathlib import Path
from skimage import io
import fastremap
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

import plotly.graph_objects as go
from plotly.offline import plot
from IPython.display import HTML, display

import omnirefactor

dtype = torch.float64

# -------------------------------------------------------------------------
# 0. JzAzBz Conversion
# -------------------------------------------------------------------------
def srgb_to_linear_np(srgb):
    srgb = np.clip(np.asarray(srgb, dtype=np.float64), 0.0, 1.0)
    mask = srgb <= 0.04045
    lin = np.empty_like(srgb)
    lin[mask] = srgb[mask] / 12.92
    lin[~mask] = ((srgb[~mask] + 0.055) / 1.055) ** 2.4
    return lin

def linear_to_srgb_np(lin):
    lin = np.asarray(lin, dtype=np.float64)
    valid_mask = (lin >= -0.0001) & (lin <= 1.0001)
    lin_clamped = np.clip(lin, 0.0, 1.0)
    mask = lin_clamped <= 0.0031308
    srgb = np.empty_like(lin_clamped)
    srgb[mask] = lin_clamped[mask] * 12.92
    srgb[~mask] = 1.055 * (lin_clamped[~mask] ** (1.0 / 2.4)) - 0.055
    return srgb, np.all(valid_mask, axis=-1)

M_srgb_xyz = np.array([[0.4124564, 0.3575761, 0.1804375],[0.2126729, 0.7151522, 0.0721750],[0.0193339, 0.1191920, 0.9503041]])
M_xyz_srgb = np.linalg.inv(M_srgb_xyz)
def srgb_to_xyz_np(srgb): return srgb_to_linear_np(srgb) @ M_srgb_xyz.T
def xyz_to_srgb_np(xyz): return linear_to_srgb_np(xyz @ M_xyz_srgb.T)

JZ_B = 1.15; JZ_G = 0.66
JZ_C1 = 3424/4096; JZ_C2 = 2413/128; JZ_C3 = 2392/128
JZ_N = 2610/16384; JZ_P = 1.7 * 2523/32; JZ_D = -0.56
M1 = np.array([[0.41478972, 0.579999, 0.0146480],[-0.2015100, 1.120649, 0.0531008],[-0.0166008, 0.264800, 0.6684799]])
M2 = np.array([[0.5, 0.5, 0],[3.524000, -4.066708, 0.542708],[0.199076, 1.096799, -1.295875]])
DISPLAY_PEAK_NITS = 200.0 

def xyz_to_jzazbz(xyz):
    xyz_abs = xyz * DISPLAY_PEAK_NITS
    lms = np.maximum((np.stack([JZ_B*xyz_abs[...,0]-(JZ_B-1)*xyz_abs[...,2], JZ_G*xyz_abs[...,1]-(JZ_G-1)*xyz_abs[...,0], xyz_abs[...,2]], axis=-1)) @ M1.T, 1e-12)
    iz = np.maximum((JZ_C1 + JZ_C2 * np.power(lms, JZ_N)) / (1 + JZ_C3 * np.power(lms, JZ_N)), 1e-12)
    jab = np.power(iz, JZ_P) @ M2.T
    return np.stack([jab[..., 0] + JZ_D, jab[..., 1], jab[..., 2]], axis=-1)

def jzazbz_to_xyz(jab):
    iz = np.power(np.maximum(np.stack([jab[...,0]-JZ_D, jab[...,1], jab[...,2]], axis=-1) @ np.linalg.inv(M2).T, 1e-12), 1.0/JZ_P)
    lms = np.power(np.maximum((iz - JZ_C1) / (JZ_C2 - JZ_C3 * iz + 1e-12), 0), 1.0/JZ_N)
    xyz_p = lms @ M1_INV.T
    X = (xyz_p[...,0] + (JZ_B-1)*xyz_p[...,2]) / JZ_B
    Y = (xyz_p[...,1] + (JZ_G-1)*X) / JZ_G
    return np.stack([X, Y, xyz_p[...,2]], axis=-1) / DISPLAY_PEAK_NITS

# -------------------------------------------------------------------------
# 1. Uniform Reparameterization & Smoothing
# -------------------------------------------------------------------------

def get_hsv_boundary(n=1000):
    """High-res trace of the RGB cube edges."""
    h = np.linspace(0, 1, n, endpoint=False)
    i = (h * 6).astype(int); f = h * 6 - i
    q = 1 - f; t = f
    rgb = np.zeros((n, 3))
    mask = i%6==0; rgb[mask] = np.stack([np.ones(mask.sum()), t[mask], np.zeros(mask.sum())], axis=1)
    mask = i%6==1; rgb[mask] = np.stack([q[mask], np.ones(mask.sum()), np.zeros(mask.sum())], axis=1)
    mask = i%6==2; rgb[mask] = np.stack([np.zeros(mask.sum()), np.ones(mask.sum()), t[mask]], axis=1)
    mask = i%6==3; rgb[mask] = np.stack([np.zeros(mask.sum()), q[mask], np.ones(mask.sum())], axis=1)
    mask = i%6==4; rgb[mask] = np.stack([t[mask], np.zeros(mask.sum()), np.ones(mask.sum())], axis=1)
    mask = i%6==5; rgb[mask] = np.stack([np.ones(mask.sum()), np.zeros(mask.sum()), q[mask]], axis=1)
    return xyz_to_jzazbz(srgb_to_xyz_np(rgb))

def resample_uniform_arc_length(curve, n_target):
    """
    Reparameterize curve so points are equidistant in Euclidean JzAzBz space.
    This is crucial for the Gaussian blur to work symmetrically.
    """
    # Close the loop for distance calculation
    closed = np.vstack([curve, curve[0]])
    dists = np.linalg.norm(np.diff(closed, axis=0), axis=1)
    cum_dist = np.concatenate([[0.0], np.cumsum(dists)])
    total_len = cum_dist[-1]
    
    t_orig = cum_dist / total_len
    t_new = np.linspace(0, 1, n_target, endpoint=False)
    
    out = np.zeros((n_target, 3))
    for i in range(3):
        # Interpolate over the periodic boundary
        # We append the first point to the end of data for interp
        data = np.concatenate([curve[:,i], [curve[0,i]]])
        out[:,i] = np.interp(t_new, t_orig, data)
        
    return out

def gaussian_smooth_loop(curve, sigma):
    """
    Applies a Gaussian filter with wrapping (cyclic).
    This rounds off corners without adding ringing artifacts.
    """
    smoothed = np.zeros_like(curve)
    for i in range(3):
        smoothed[:,i] = gaussian_filter1d(curve[:,i], sigma=sigma, mode='wrap')
    return smoothed

def fit_to_gamut(jz_curve):
    """
    Scales the curve towards its perceptual center until it fits safely.
    Because Gaussian smoothing inherently shrinks the convex hull,
    this scaling factor should remain close to 1.0 (large loop).
    """
    center = np.mean(jz_curve, axis=0)
    centered = jz_curve - center
    
    # Binary search for max scale
    low, high = 0.0, 1.2
    best_scale = 0.0
    
    for _ in range(20):
        scale = (low + high) / 2
        test = center + centered * scale
        
        xyz = jzazbz_to_xyz(test)
        _, valid = xyz_to_srgb_np(xyz)
        
        if np.all(valid):
            best_scale = scale
            low = scale
        else:
            high = scale
            
    print(f"  Final Scale Factor: {best_scale:.3f}")
    return center + centered * best_scale

# -------------------------------------------------------------------------
# 2. Visualization
# -------------------------------------------------------------------------

def generate_vis_row(rgb_ring, flows_np, n_hues):
    y, x = np.ogrid[-1:1:complex(0, 256), -1:1:complex(0, 256)]
    mask = x**2 + y**2 > 1.0
    ang = np.arctan2(y, x)
    idx = (((ang + np.pi) / (2*np.pi)) * len(rgb_ring)).astype(int) % len(rgb_ring)
    disk = rgb_ring[idx].copy()
    disk[mask] = 0
    
    dy, dx = flows_np[0], flows_np[1]
    mag = np.sqrt(dy**2 + dx**2)
    p99 = np.percentile(mag, 99)
    if p99 < 1e-5: p99 = 1.0
    alpha = np.clip(mag / p99, 0, 1)[..., None]
    
    ang_f = np.arctan2(dy, dx)
    def get_c(r, a): return r[(((a + np.pi) / (2*np.pi)) * len(r)).astype(int) % len(r)]
    
    c = get_c(rgb_ring, ang_f)
    f = c * alpha + 0.1*(1-alpha)
    ow = c * alpha + (1.0 - alpha)
    ob = c * alpha
    
    rot = np.roll(rgb_ring, n_hues//4, axis=0)
    cr = get_c(rot, ang_f)
    rw = cr * alpha + (1.0 - alpha)
    rb = cr * alpha
    
    return [disk, f, ow, ob, rw, rb]

def get_srgb_wireframe_jz(n_per_edge=20):
    edges = [([0,0,0],[1,0,0]), ([1,0,0],[1,1,0]), ([1,1,0],[0,1,0]), ([0,1,0],[0,0,0]),
             ([0,0,1],[1,0,1]), ([1,0,1],[1,1,1]), ([1,1,1],[0,1,1]), ([0,1,1],[0,0,1]),
             ([0,0,0],[0,0,1]), ([1,0,0],[1,0,1]), ([1,1,0],[1,1,1]), ([0,1,0],[0,1,1])]
    x, y, z = [], [], []
    for start, end in edges:
        t = np.linspace(0, 1, n_per_edge)
        rgb = np.array(start) + t[:,None] * (np.array(end) - np.array(start))
        jz = xyz_to_jzazbz(srgb_to_xyz_np(rgb))
        x.extend(jz[:,1]); y.extend(jz[:,2]); z.extend(jz[:,0])
        x.append(None); y.append(None); z.append(None)
    return x, y, z

# -------------------------------------------------------------------------
# 3. Main
# -------------------------------------------------------------------------

def run_smooth_uniform_hsv(n_hues=72, blur_sigma=5.0):
    # 1. Get Raw HSV Truth (High Res)
    jz_raw = get_hsv_boundary(n=720)
    
    # 2. Reparameterize to Uniform Perceptual Velocity
    # (This creates the "Uniform Space" you requested)
    jz_uniform = resample_uniform_arc_length(jz_raw, n_target=720)
    
    # 3. Blur (Gaussian Smoothing)
    # Sigma controls the "corner cutting". 
    # Higher sigma = Rounder, more ellipse-like. Lower sigma = More hexagonal.
    jz_smooth = gaussian_smooth_loop(jz_uniform, sigma=blur_sigma)
    
    # 4. Fit to Gamut (Maximal Expansion)
    jz_fitted = fit_to_gamut(jz_smooth)
    
    # 5. Final Resample to target size
    jz_final = resample_uniform_arc_length(jz_fitted, n_hues)
    
    # 6. RGB
    xyz = jzazbz_to_xyz(jz_final)
    rgb_opt, _ = xyz_to_srgb_np(xyz)
    rgb_opt = np.clip(rgb_opt, 0, 1)
    
    # --- Comparators ---
    t = np.linspace(0, 1, n_hues, endpoint=False)
    sr = 0.5 + 0.5*np.sin(2*np.pi*(t+0/3))
    sg = 0.5 + 0.5*np.sin(2*np.pi*(t+1/3))
    sb = 0.5 + 0.5*np.sin(2*np.pi*(t+2/3))
    rgb_sine = np.stack([sr, sg, sb], axis=1)
    
    # Align
    def get_g(r): return np.argmax(r[:,1] - 0.5*(r[:,0]+r[:,2]))
    g_ref = get_g(rgb_sine)
    rgb_opt = np.roll(rgb_opt, g_ref - get_g(rgb_opt), axis=0)
    
    # --- Vis ---
    omnidir = Path(omnirefactor.__file__).parent.parent.parent
    masks = io.imread(os.path.join(omnidir, "docs", "_static", "ecoli_labels.tif"))
    slc = omnirefactor.measure.crop_bbox(masks>0, pad=0)[0]
    masks = fastremap.renumber(masks[slc])[0]
    flows_tensor = omnirefactor.core.masks_to_flows(masks, device="cpu")[4]
    flows_np = flows_tensor.cpu().numpy() if hasattr(flows_tensor, 'cpu') else flows_tensor

    row_s = generate_vis_row(rgb_sine, flows_np, n_hues)
    row_o = generate_vis_row(rgb_opt, flows_np, n_hues)
    
    fig, ax = plt.subplots(2, 6, figsize=(24, 8))
    cols = ["Disk", "Flow", "Alpha/W", "Alpha/B", "Rot/W", "Rot/B"]
    data = [row_s, row_o]
    names = ["Sinebow", f"Uniform-Gaussian (Sigma={blur_sigma})"]
    
    for i in range(2):
        for j in range(6):
            ax[i,j].imshow(data[i][j]); ax[i,j].axis('off')
            if i==0: ax[i,j].set_title(cols[j])
            if j==0: ax[i,j].text(-0.1, 0.5, names[i], transform=ax[i,j].transAxes, va='center', ha='right', rotation=90, fontsize=12)
    plt.tight_layout()
    plt.show()
    
    # --- 3D Plot ---
    jz_sin = xyz_to_jzazbz(srgb_to_xyz_np(rgb_sine))
    jz_final_plot = xyz_to_jzazbz(srgb_to_xyz_np(rgb_opt))
    jz_truth = get_hsv_boundary(360)
    # Get colors for truth dots
    truth_xyz = jzazbz_to_xyz(jz_truth)
    truth_rgb = np.clip(xyz_to_srgb_np(truth_xyz)[0], 0, 1)
    
    fig3d = go.Figure()
    
    # Faces
    grid = np.linspace(0,1,17); G, B = np.meshgrid(grid, grid)
    def sface(ch, v):
        if ch==0: s=np.stack([np.full_like(G,v),G,B],axis=-1)
        elif ch==1: s=np.stack([G,np.full_like(G,v),B],axis=-1)
        else: s=np.stack([G,B,np.full_like(G,v)],axis=-1)
        jz = xyz_to_jzazbz(srgb_to_xyz_np(s.reshape(-1,3))).reshape(17,17,3)
        fig3d.add_trace(go.Surface(x=jz[...,1],y=jz[...,2],z=jz[...,0],opacity=0.1,showscale=False,hoverinfo='skip'))
    for c in range(3): 
        for v in [0,1]: sface(c,v)
    
    # Wireframe
    wx, wy, wz = get_srgb_wireframe_jz()
    fig3d.add_trace(go.Scatter3d(x=wx, y=wy, z=wz, mode='lines', name='sRGB Wireframe', line=dict(color='gray', width=3), hoverinfo='skip'))
    
    # HSV Truth
    fig3d.add_trace(go.Scatter3d(x=jz_truth[:,1], y=jz_truth[:,2], z=jz_truth[:,0], mode='markers', name='HSV Truth (Uniform)', marker=dict(size=3, color=truth_rgb)))
    
    # Result
    fig3d.add_trace(go.Scatter3d(x=jz_final_plot[:,1], y=jz_final_plot[:,2], z=jz_final_plot[:,0], mode='lines', name='Smoothed Result', line=dict(width=10, color='cyan')))
    
    fig3d.update_layout(template="plotly_dark", height=600, title=f"Uniform Reparam + Gaussian Blur (Sigma={blur_sigma})")
    display(HTML(plot(fig3d, output_type='div', include_plotlyjs='cdn')))

if __name__ == "__main__":
    # Sigma=25.0 provides a very smooth, near-elliptical shape that still fills the volume well.
    # Sigma=5.0 would be more hexagonal.
    run_smooth_uniform_hsv(300, blur_sigma=25)

In [ ]:
import numpy as np
import torch
import os
from pathlib import Path
from skimage import io
import fastremap
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

import plotly.graph_objects as go
from plotly.offline import plot
from IPython.display import HTML, display

import omnirefactor

dtype = torch.float64

# -------------------------------------------------------------------------
# 0. Robust JzAzBz Conversion (200 nits)
# -------------------------------------------------------------------------
def srgb_to_linear_np(srgb):
    srgb = np.clip(np.asarray(srgb, dtype=np.float64), 0.0, 1.0)
    mask = srgb <= 0.04045
    lin = np.empty_like(srgb)
    lin[mask] = srgb[mask] / 12.92
    lin[~mask] = ((srgb[~mask] + 0.055) / 1.055) ** 2.4
    return lin

def linear_to_srgb_np(lin):
    lin = np.asarray(lin, dtype=np.float64)
    # Strict check
    valid_mask = (lin >= -0.0005) & (lin <= 1.0005)
    
    lin_clamped = np.clip(lin, 0.0, 1.0)
    mask = lin_clamped <= 0.0031308
    srgb = np.empty_like(lin_clamped)
    srgb[mask] = lin_clamped[mask] * 12.92
    srgb[~mask] = 1.055 * (lin_clamped[~mask] ** (1.0 / 2.4)) - 0.055
    
    return srgb, np.all(valid_mask, axis=-1)

M_srgb_xyz = np.array([[0.4124564, 0.3575761, 0.1804375],[0.2126729, 0.7151522, 0.0721750],[0.0193339, 0.1191920, 0.9503041]])
M_xyz_srgb = np.linalg.inv(M_srgb_xyz)
def srgb_to_xyz_np(srgb): return srgb_to_linear_np(srgb) @ M_srgb_xyz.T
def xyz_to_srgb_np(xyz): return linear_to_srgb_np(xyz @ M_xyz_srgb.T)

JZ_B = 1.15; JZ_G = 0.66
JZ_C1 = 3424/4096; JZ_C2 = 2413/128; JZ_C3 = 2392/128
JZ_N = 2610/16384; JZ_P = 1.7 * 2523/32; JZ_D = -0.56
M1 = np.array([[0.41478972, 0.579999, 0.0146480],[-0.2015100, 1.120649, 0.0531008],[-0.0166008, 0.264800, 0.6684799]])
M2 = np.array([[0.5, 0.5, 0],[3.524000, -4.066708, 0.542708],[0.199076, 1.096799, -1.295875]])
DISPLAY_PEAK_NITS = 200.0 

def xyz_to_jzazbz(xyz):
    xyz_abs = xyz * DISPLAY_PEAK_NITS
    lms = np.maximum((np.stack([JZ_B*xyz_abs[...,0]-(JZ_B-1)*xyz_abs[...,2], JZ_G*xyz_abs[...,1]-(JZ_G-1)*xyz_abs[...,0], xyz_abs[...,2]], axis=-1)) @ M1.T, 1e-12)
    iz = np.maximum((JZ_C1 + JZ_C2 * np.power(lms, JZ_N)) / (1 + JZ_C3 * np.power(lms, JZ_N)), 1e-12)
    jab = np.power(iz, JZ_P) @ M2.T
    return np.stack([jab[..., 0] + JZ_D, jab[..., 1], jab[..., 2]], axis=-1)

def jzazbz_to_xyz(jab):
    iz = np.power(np.maximum(np.stack([jab[...,0]-JZ_D, jab[...,1], jab[...,2]], axis=-1) @ np.linalg.inv(M2).T, 1e-12), 1.0/JZ_P)
    lms = np.power(np.maximum((iz - JZ_C1) / (JZ_C2 - JZ_C3 * iz + 1e-12), 0), 1.0/JZ_N)
    xyz_p = lms @ M1_INV.T
    X = (xyz_p[...,0] + (JZ_B-1)*xyz_p[...,2]) / JZ_B
    Y = (xyz_p[...,1] + (JZ_G-1)*X) / JZ_G
    return np.stack([X, Y, xyz_p[...,2]], axis=-1) / DISPLAY_PEAK_NITS

# -------------------------------------------------------------------------
# 1. Analytic Ridge Construction (The "Truth")
# -------------------------------------------------------------------------

def get_analytic_hsv_ridge(n=360):
    """
    Generates the exact edges of the sRGB cube in order.
    This IS the maximum saturation ridge.
    """
    h = np.linspace(0, 1, n, endpoint=False)
    i = (h * 6).astype(int); f = h * 6 - i
    q = 1 - f; t = f
    rgb = np.zeros((n, 3))
    
    # Explicitly construct the 6 sectors of the HSV boundary
    mask = i%6==0; rgb[mask] = np.stack([np.ones(mask.sum()), t[mask], np.zeros(mask.sum())], axis=1)
    mask = i%6==1; rgb[mask] = np.stack([q[mask], np.ones(mask.sum()), np.zeros(mask.sum())], axis=1)
    mask = i%6==2; rgb[mask] = np.stack([np.zeros(mask.sum()), np.ones(mask.sum()), t[mask]], axis=1)
    mask = i%6==3; rgb[mask] = np.stack([np.zeros(mask.sum()), q[mask], np.ones(mask.sum())], axis=1)
    mask = i%6==4; rgb[mask] = np.stack([t[mask], np.zeros(mask.sum()), np.ones(mask.sum())], axis=1)
    mask = i%6==5; rgb[mask] = np.stack([np.ones(mask.sum()), np.zeros(mask.sum()), q[mask]], axis=1)
    
    # Convert to JzAzBz
    return xyz_to_jzazbz(srgb_to_xyz_np(rgb))

# -------------------------------------------------------------------------
# 2. Smoothing Pipeline
# -------------------------------------------------------------------------

def resample_uniform_arc_length(curve, n_target):
    """
    Crucial step: Distribute points evenly in JzAzBz space.
    This prevents the Gaussian blur from being biased by the varying speed of hue in sRGB.
    """
    closed = np.vstack([curve, curve[0]])
    dists = np.linalg.norm(np.diff(closed, axis=0), axis=1)
    cum = np.concatenate([[0.0], np.cumsum(dists)])
    t_orig = cum / cum[-1]
    t_new = np.linspace(0, 1, n_target, endpoint=False)
    
    out = np.zeros((n_target, 3))
    # Cyclic interpolation
    for i in range(3):
        data = np.concatenate([curve[:,i], [curve[0,i]]])
        out[:,i] = np.interp(t_new, t_orig, data)
    return out

def gaussian_smooth(curve, sigma):
    """
    Applies Gaussian blur.
    This rounds off the sharp corners of the RGB hexagon naturally.
    """
    smooth = np.zeros_like(curve)
    for i in range(3):
        smooth[:,i] = gaussian_filter1d(curve[:,i], sigma=sigma, mode='wrap')
    return smooth

def clip_to_gamut(jz_curve):
    """
    Point-wise clipping. If a point smoothed out of bounds, pull it back.
    Does NOT shrink the valid parts of the ring.
    """
    xyz = jzazbz_to_xyz(jz_curve)
    _, valid = xyz_to_srgb_np(xyz)
    
    if np.all(valid): return jz_curve
    
    # print(f"  Clipping {np.sum(~valid)} points...")
    clipped = jz_curve.copy()
    idxs = np.where(~valid)[0]
    
    for i in idxs:
        J, a, b = jz_curve[i]
        # Binary search back to gamut wall
        low, high = 0.0, 1.0
        for _ in range(10):
            mid = (low + high) / 2
            pt = np.array([J, a*mid, b*mid])
            # Fast check
            xyz_pt = jzazbz_to_xyz(pt)
            lin_pt = xyz_pt @ M_xyz_srgb.T
            if np.all((lin_pt >= -0.001) & (lin_pt <= 1.001)):
                low = mid
            else:
                high = mid
        clipped[i, 1] *= low
        clipped[i, 2] *= low
    return clipped

# -------------------------------------------------------------------------
# 3. Visualization
# -------------------------------------------------------------------------

def generate_vis_row(rgb_ring, flows_np, n_hues):
    # Disk
    y, x = np.ogrid[-1:1:complex(0, 256), -1:1:complex(0, 256)]
    mask = x**2 + y**2 > 1.0
    ang = np.arctan2(y, x)
    idx = (((ang + np.pi) / (2*np.pi)) * len(rgb_ring)).astype(int) % len(rgb_ring)
    disk = rgb_ring[idx].copy()
    disk[mask] = 0
    
    # Flow
    dy, dx = flows_np[0], flows_np[1]
    mag = np.sqrt(dy**2 + dx**2)
    p99 = np.percentile(mag, 99)
    if p99 < 1e-5: p99 = 1.0
    alpha = np.clip(mag / p99, 0, 1)[..., None]
    
    ang_f = np.arctan2(dy, dx)
    def get_c(r, a): 
        ix = (((a + np.pi) / (2*np.pi)) * len(r)).astype(int) % len(r)
        return r[ix]
    
    c = get_c(rgb_ring, ang_f)
    f = c * alpha + 0.1*(1-alpha)
    ow = c * alpha + (1.0 - alpha)
    ob = c * alpha
    
    rot = np.roll(rgb_ring, n_hues//4, axis=0)
    cr = get_c(rot, ang_f)
    rw = cr * alpha + (1.0 - alpha)
    rb = cr * alpha
    return [disk, f, ow, ob, rw, rb]

def get_srgb_wireframe_jz(n=20):
    edges = [([0,0,0],[1,0,0]), ([1,0,0],[1,1,0]), ([1,1,0],[0,1,0]), ([0,1,0],[0,0,0]),
             ([0,0,1],[1,0,1]), ([1,0,1],[1,1,1]), ([1,1,1],[0,1,1]), ([0,1,1],[0,0,1]),
             ([0,0,0],[0,0,1]), ([1,0,0],[1,0,1]), ([1,1,0],[1,1,1]), ([0,1,0],[0,1,1])]
    x, y, z = [], [], []
    for start, end in edges:
        t = np.linspace(0, 1, n)
        rgb = np.array(start) + t[:,None] * (np.array(end) - np.array(start))
        jz = xyz_to_jzazbz(srgb_to_xyz_np(rgb)) # Direct array return
        x.extend(jz[:,1]); y.extend(jz[:,2]); z.extend(jz[:,0])
        x.append(None); y.append(None); z.append(None)
    return x, y, z

# -------------------------------------------------------------------------
# 4. Main
# -------------------------------------------------------------------------

def run_analytic_ridge_smoothing(n_hues=72, blur_sigma=75.0):
    # 1. Analytic Truth (High Res)
    jz_ridge = get_analytic_hsv_ridge(n=720)
    
    # 2. Uniform Reparameterization (Crucial for even smoothing)
    jz_uni = resample_uniform_arc_length(jz_ridge, n_target=720)
    
    # 3. Gaussian Blur (The Smoothing)
    jz_smooth = gaussian_smooth(jz_uni, sigma=blur_sigma)
    
    # 4. Clip (Safe)
    jz_safe = clip_to_gamut(jz_smooth)
    
    # 5. Final Resample
    jz_final = resample_uniform_arc_length(jz_safe, n_hues)
    
    # 6. RGB
    xyz = jzazbz_to_xyz(jz_final)
    rgb_opt, _ = xyz_to_srgb_np(xyz)
    rgb_opt = np.clip(rgb_opt, 0, 1)
    
    # --- Comparators ---
    t = np.linspace(0, 1, n_hues, endpoint=False)
    sr = 0.5 + 0.5*np.sin(2*np.pi*(t+0/3))
    sg = 0.5 + 0.5*np.sin(2*np.pi*(t+1/3))
    sb = 0.5 + 0.5*np.sin(2*np.pi*(t+2/3))
    rgb_sine = np.stack([sr, sg, sb], axis=1)
    
    # Align
    def get_g(r): return np.argmax(r[:,1] - 0.5*(r[:,0]+r[:,2]))
    def align_to(rgb, target):
        curr = get_g(rgb)
        return np.roll(rgb, target - curr, axis=0)
    
    g_ref = get_g(rgb_sine)
    rgb_opt_a = align_to(rgb_opt, g_ref)
    
    # --- Visuals ---
    omnidir = Path(omnirefactor.__file__).parent.parent.parent
    masks_path = os.path.join(omnidir, "docs", "_static", "ecoli_labels.tif")
    
    if os.path.exists(masks_path):
        masks = io.imread(masks_path)
        slc = omnirefactor.measure.crop_bbox(masks>0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows_tensor = omnirefactor.core.masks_to_flows(masks, device="cpu")[4]
        flows_np = flows_tensor.cpu().numpy() if hasattr(flows_tensor, 'cpu') else flows_tensor
    else:
        y, x = np.mgrid[-10:10:0.1, -10:10:0.1]
        flows_np = np.stack([np.sin(x/3), np.cos(y/3)]) * 5

    row_s = generate_vis_row(rgb_sine, flows_np, n_hues)
    row_o = generate_vis_row(rgb_opt_a, flows_np, n_hues)
    
    fig, ax = plt.subplots(2, 6, figsize=(24, 8))
    cols = ["Disk", "Flow", "Alpha/W", "Alpha/B", "Rot/W", "Rot/B"]
    data = [row_s, row_o]
    names = ["Sinebow", f"Analytic Ridge (Sigma={blur_sigma})"]
    
    for i in range(2):
        for j in range(6):
            ax[i,j].imshow(data[i][j]); ax[i,j].axis('off')
            if i==0: ax[i,j].set_title(cols[j])
            if j==0: ax[i,j].text(-0.1, 0.5, names[i], transform=ax[i,j].transAxes, va='center', ha='right', rotation=90, fontsize=12)
    plt.tight_layout()
    plt.show()
    
    # --- 3D Plot ---
    # Prepare colors for truth dots
    xyz_truth = jzazbz_to_xyz(jz_ridge)
    rgb_truth = np.clip(xyz_to_srgb_np(xyz_truth)[0], 0, 1)
    
    jz_sin = xyz_to_jzazbz(srgb_to_xyz_np(rgb_sine))
    jz_final_plot = xyz_to_jzazbz(srgb_to_xyz_np(rgb_opt_a))
    
    fig3d = go.Figure()
    
    # 1. Faces
    grid = np.linspace(0,1,17); G, B = np.meshgrid(grid, grid)
    def sface(ch, v):
        if ch==0: s=np.stack([np.full_like(G,v),G,B],axis=-1)
        elif ch==1: s=np.stack([G,np.full_like(G,v),B],axis=-1)
        else: s=np.stack([G,B,np.full_like(G,v)],axis=-1)
        jz = xyz_to_jzazbz(srgb_to_xyz_np(s.reshape(-1,3))).reshape(17,17,3)
        fig3d.add_trace(go.Surface(x=jz[...,1],y=jz[...,2],z=jz[...,0],opacity=0.1,showscale=False,hoverinfo='skip'))
    for c in range(3): 
        for v in [0,1]: sface(c,v)
        
    # 2. Wireframe
    wx, wy, wz = get_srgb_wireframe_jz()
    fig3d.add_trace(go.Scatter3d(x=wx, y=wy, z=wz, mode='lines', name='sRGB Wireframe', line=dict(color='gray', width=3), hoverinfo='skip'))
    
    # 3. Analytic Truth     fig3d.add_trace(go.Scatter3d(x=jz_ridge[:,1], y=jz_ridge[:,2], z=jz_ridge[:,0], mode='markers', name='Analytic Ridge (Truth)', marker=dict(size=3, color=rgb_truth)))
    
    # 4. Loop
    fig3d.add_trace(go.Scatter3d(x=jz_final_plot[:,1], y=jz_final_plot[:,2], z=jz_final_plot[:,0], mode='lines', name='Smoothed Result', line=dict(width=10, color='cyan')))
    
    fig3d.update_layout(template="plotly_dark", height=600, title=f"Analytic Ridge + Gaussian Smooth (S={blur_sigma})")
    display(HTML(plot(fig3d, output_type='div', include_plotlyjs='cdn')))

if __name__ == "__main__":
    # Sigma 70 on 720 points is approx 10% of the loop length
    run_analytic_ridge_smoothing(72, blur_sigma=70.0)

In [ ]:
import numpy as np
import torch
import os
from pathlib import Path
from skimage import io
import fastremap
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.interpolate import interp1d

import plotly.graph_objects as go
from plotly.offline import plot
from IPython.display import HTML, display

import omnirefactor

dtype = torch.float64

# =========================================================================
# 0. Oklab Color Space (Standard)
# =========================================================================

def srgb_to_linear_np(srgb):
    srgb = np.clip(np.asarray(srgb, dtype=np.float64), 0.0, 1.0)
    mask = srgb <= 0.04045
    lin = np.empty_like(srgb)
    lin[mask] = srgb[mask] / 12.92
    lin[~mask] = ((srgb[~mask] + 0.055) / 1.055) ** 2.4
    return lin

def linear_to_srgb_np(lin):
    lin = np.asarray(lin, dtype=np.float64)
    valid_mask = (lin >= -0.0005) & (lin <= 1.0005)
    lin = np.clip(lin, 0.0, 1.0)
    mask = lin <= 0.0031308
    srgb = np.empty_like(lin)
    srgb[mask] = lin[mask] * 12.92
    srgb[~mask] = 1.055 * (lin[~mask] ** (1.0 / 2.4)) - 0.055
    return srgb, np.all(valid_mask, axis=-1)

M_srgb_xyz = np.array([[0.4124564, 0.3575761, 0.1804375],
                       [0.2126729, 0.7151522, 0.0721750],
                       [0.0193339, 0.1191920, 0.9503041]])
M_xyz_srgb = np.linalg.inv(M_srgb_xyz)

# Oklab Matrices (Ottosson 2020)
M1 = np.array([[0.8189330101, 0.3618667424, -0.1288597137],
               [0.0329845436, 0.9293118715, 0.0361456387],
               [0.0482003018, 0.2643662691, 0.6338517070]])
M2 = np.array([[0.2104542553, 0.7936177850, -0.0040720468],
               [1.9779984951, -2.4285922050, 0.4505937099],
               [0.0259040371, 0.7827717662, -0.8086757660]])
M1_INV = np.linalg.inv(M1)
M2_INV = np.linalg.inv(M2)

def xyz_to_oklab(xyz):
    lms = xyz @ M1.T
    lms_p = np.cbrt(np.maximum(lms, 0))
    return lms_p @ M2.T

def oklab_to_xyz(lab):
    lms_p = lab @ M2_INV.T
    lms = lms_p**3
    return lms @ M1_INV.T

def srgb_to_oklab(srgb):
    lin = srgb_to_linear_np(srgb)
    xyz = lin @ M_srgb_xyz.T
    return xyz_to_oklab(xyz)

def oklab_to_srgb_check(lab):
    xyz = oklab_to_xyz(lab)
    lin = xyz @ M_xyz_srgb.T
    return linear_to_srgb_np(lin)

# =========================================================================
# 1. Geometric "Equator" Calculation
# =========================================================================

def fit_plane_to_points(points):
    """
    Fits a plane to a set of 3D points using SVD.
    Returns: centroid, normal, u, v (basis vectors)
    """
    centroid = np.mean(points, axis=0)
    # Center the points
    centered = points - centroid
    # SVD
    u, s, vh = np.linalg.svd(centered)
    # The normal is the last column of V (or last row of Vh) corresponding to smallest singular value
    normal = vh[2, :]
    
    # Ensure normal points "up" (Positive L)
    if normal[0] < 0: normal = -normal
    
    # Create planar basis
    # u_vec corresponds to largest variance (major axis)
    u_vec = vh[0, :]
    v_vec = np.cross(normal, u_vec)
    
    return centroid, normal, u_vec, v_vec

def get_gamut_equator_plane():
    """
    Finds the plane that best fits the 6 corners of sRGB:
    R, Y, G, C, B, M
    """
    corners_rgb = np.array([
        [1,0,0], [1,1,0], [0,1,0], 
        [0,1,1], [0,0,1], [1,0,1]
    ], dtype=float)
    
    corners_lab = srgb_to_oklab(corners_rgb)
    
    # Fit plane
    center, normal, u, v = fit_plane_to_points(corners_lab)
    
    print(f"Equator Plane Found:")
    print(f"  Center L={center[0]:.3f}")
    print(f"  Normal={normal}")
    
    return center, normal, u, v

# =========================================================================
# 2. Planar Intersection (Tracing the Slice)
# =========================================================================

def ray_cast_on_plane(center, u, v, angle_rad):
    """Finds the distance to the gamut wall on the defined plane."""
    direction = np.cos(angle_rad) * u + np.sin(angle_rad) * v
    
    low, high = 0.0, 0.50 # Max possible radius
    
    for _ in range(12):
        mid = (low + high) * 0.5
        pt = center + mid * direction
        
        _, valid = oklab_to_srgb_check(pt[None, :])
        if valid[0]:
            low = mid
        else:
            high = mid
            
    return low

def trace_planar_slice(n_scan=360):
    """
    1. Get Best Fit Plane
    2. Scan 360 degrees on that plane to find the polygon boundary.
    """
    center, normal, u, v = get_gamut_equator_plane()
    
    angles = np.linspace(0, 2*np.pi, n_scan, endpoint=False)
    radii = []
    
    for theta in angles:
        r = ray_cast_on_plane(center, u, v, theta)
        radii.append(r)
        
    radii = np.array(radii)
    
    # Reconstruct 3D points
    # points = center + radius * (cos*u + sin*v)
    pts = center[None, :] + radii[:, None] * (np.cos(angles)[:, None] * u[None, :] + np.sin(angles)[:, None] * v[None, :])
    
    return pts

# =========================================================================
# 3. Smoothing & Resampling
# =========================================================================

def gaussian_smooth_loop(curve, sigma=5.0):
    """Applies Gaussian smoothing to the 3D coordinates."""
    smooth = np.zeros_like(curve)
    for i in range(3):
        smooth[:, i] = gaussian_filter1d(curve[:, i], sigma=sigma, mode='wrap')
    return smooth

def shrink_to_fit(curve):
    """Ensures the smoothed curve didn't bulge out."""
    center = np.mean(curve, axis=0)
    centered = curve - center
    
    low, high = 0.0, 1.1
    best_scale = 0.0
    
    for _ in range(15):
        scale = (low + high) / 2
        test = center + centered * scale
        _, valid = oklab_to_srgb_check(test)
        
        if np.all(valid):
            best_scale = scale
            low = scale
        else:
            high = scale
            
    print(f"  Final Fit Scale: {best_scale:.3f}")
    return center + centered * best_scale

def resample_uniform(curve, n_target):
    # Cyclic closure
    closed = np.vstack([curve, curve[0]])
    dists = np.linalg.norm(np.diff(closed, axis=0), axis=1)
    cum = np.concatenate([[0.0], np.cumsum(dists)])
    
    t_orig = cum / cum[-1]
    t_new = np.linspace(0, 1, n_target, endpoint=False)
    
    out = np.zeros((n_target, 3))
    for i in range(3):
        out[:, i] = np.interp(t_new, t_orig, closed[:, i])
    return out

# =========================================================================
# 4. Vis
# =========================================================================

def generate_vis_row(rgb_ring, flows_np, n_hues):
    y, x = np.ogrid[-1:1:complex(0, 256), -1:1:complex(0, 256)]
    mask = x**2 + y**2 > 1.0
    ang = np.arctan2(y, x)
    idx = (((ang + np.pi) / (2*np.pi)) * len(rgb_ring)).astype(int) % len(rgb_ring)
    disk = rgb_ring[idx].copy()
    disk[mask] = 0
    
    dy, dx = flows_np[0], flows_np[1]
    mag = np.sqrt(dy**2 + dx**2)
    p99 = np.percentile(mag, 99)
    if p99 < 1e-5: p99 = 1.0
    alpha = np.clip(mag / p99, 0, 1)[..., None]
    
    ang_f = np.arctan2(dy, dx)
    def get_c(r, a): return r[(((a + np.pi) / (2*np.pi)) * len(r)).astype(int) % len(r)]
    
    c = get_c(rgb_ring, ang_f)
    f = c * alpha + 0.1*(1-alpha)
    ow = c * alpha + (1.0 - alpha)
    ob = c * alpha
    
    rot = np.roll(rgb_ring, n_hues//4, axis=0)
    cr = get_c(rot, ang_f)
    rw = cr * alpha + (1.0 - alpha)
    rb = cr * alpha
    return [disk, f, ow, ob, rw, rb]

def get_srgb_wireframe_oklab(n=20):
    edges = [([0,0,0],[1,0,0]), ([1,0,0],[1,1,0]), ([1,1,0],[0,1,0]), ([0,1,0],[0,0,0]),
             ([0,0,1],[1,0,1]), ([1,0,1],[1,1,1]), ([1,1,1],[0,1,1]), ([0,1,1],[0,0,1]),
             ([0,0,0],[0,0,1]), ([1,0,0],[1,0,1]), ([1,1,0],[1,1,1]), ([0,1,0],[0,1,1])]
    x, y, z = [], [], []
    for s, e in edges:
        t = np.linspace(0, 1, n)
        rgb = np.array(s) + t[:,None]*(np.array(e)-np.array(s))
        lab = srgb_to_oklab(rgb)
        x.extend(lab[:,1]); y.extend(lab[:,2]); z.extend(lab[:,0])
        x.append(None); y.append(None); z.append(None)
    return x, y, z

# =========================================================================
# 5. Main
# =========================================================================

def run_equator_slice(n_hues=72, smooth_sigma=20.0):
    # 1. Trace the "Equator" (Best Fit Plane intersection)
    # Use high res for accuracy
    lab_raw = trace_planar_slice(n_scan=720)
    
    # 2. Uniform Resample (Before smoothing)
    lab_uni = resample_uniform(lab_raw, n_target=720)
    
    # 3. Smooth the Polygon
    # This rounds off the corners of the slice
    lab_smooth = gaussian_smooth_loop(lab_uni, sigma=smooth_sigma)
    
    # 4. Ensure Validity
    lab_safe = shrink_to_fit(lab_smooth)
    
    # 5. Final Resample
    lab_final = resample_uniform(lab_safe, n_hues)
    
    # 6. RGB
    rgb_opt, _ = oklab_to_srgb_check(lab_final)
    rgb_opt = np.clip(rgb_opt, 0, 1)
    
    # --- Comparators ---
    t = np.linspace(0, 1, n_hues, endpoint=False)
    sr = 0.5 + 0.5*np.sin(2*np.pi*(t+0/3))
    sg = 0.5 + 0.5*np.sin(2*np.pi*(t+1/3))
    sb = 0.5 + 0.5*np.sin(2*np.pi*(t+2/3))
    rgb_sine = np.stack([sr, sg, sb], axis=1)
    
    def get_g(r): return np.argmax(r[:,1] - 0.5*(r[:,0]+r[:,2]))
    g_ref = get_g(rgb_sine)
    curr = get_g(rgb_opt)
    rgb_opt_a = np.roll(rgb_opt, g_ref - curr, axis=0)
    
    # --- Vis ---
    omnidir = Path(omnirefactor.__file__).parent.parent.parent
    masks_path = os.path.join(omnidir, "docs", "_static", "ecoli_labels.tif")
    if os.path.exists(masks_path):
        masks = io.imread(masks_path)
        slc = omnirefactor.measure.crop_bbox(masks>0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows_tensor = omnirefactor.core.masks_to_flows(masks, device="cpu")[4]
        flows_np = flows_tensor.cpu().numpy() if hasattr(flows_tensor, 'cpu') else flows_tensor
    else:
        y, x = np.mgrid[-10:10:0.1, -10:10:0.1]
        flows_np = np.stack([np.sin(x/3), np.cos(y/3)]) * 5

    row_s = generate_vis_row(rgb_sine, flows_np, n_hues)
    row_o = generate_vis_row(rgb_opt_a, flows_np, n_hues)
    
    fig, ax = plt.subplots(2, 6, figsize=(24, 8))
    cols = ["Disk", "Flow", "Alpha/W", "Alpha/B", "Rot/W", "Rot/B"]
    data = [row_s, row_o]
    names = ["Sinebow", "Oklab Equator Slice"]
    
    for i in range(2):
        for j in range(6):
            ax[i,j].imshow(data[i][j]); ax[i,j].axis('off')
            if i==0: ax[i,j].set_title(cols[j])
            if j==0: ax[i,j].text(-0.1, 0.5, names[i], transform=ax[i,j].transAxes, va='center', ha='right', rotation=90, fontsize=12)
    plt.tight_layout()
    plt.show()
    
    # --- 3D ---
    lab_sin = srgb_to_oklab(rgb_sine)
    lab_plot = srgb_to_oklab(rgb_opt_a)
    
    fig3d = go.Figure()
    wx, wy, wz = get_srgb_wireframe_oklab()
    fig3d.add_trace(go.Scatter3d(x=wx, y=wy, z=wz, mode='lines', name='sRGB Wireframe', line=dict(color='gray', width=3), hoverinfo='skip'))
    fig3d.add_trace(go.Scatter3d(x=lab_sin[:,1], y=lab_sin[:,2], z=lab_sin[:,0], mode='lines', name='Sinebow', line=dict(width=4, color='yellow')))
    fig3d.add_trace(go.Scatter3d(x=lab_plot[:,1], y=lab_plot[:,2], z=lab_plot[:,0], mode='lines', name='Equator Slice', line=dict(width=10, color='cyan')))
    
    # Plot the raw intersection too
    fig3d.add_trace(go.Scatter3d(x=lab_raw[:,1], y=lab_raw[:,2], z=lab_raw[:,0], mode='lines', name='Raw Plane Intersection', line=dict(width=2, color='white', dash='dot')))
    
    fig3d.update_layout(template="plotly_dark", height=600, title="Oklab Equator Slice (Average RGB Plane)", scene=dict(xaxis_title='a', yaxis_title='b', zaxis_title='L'))
    display(HTML(plot(fig3d, output_type='div', include_plotlyjs='cdn')))

if __name__ == "__main__":
    # Sigma 20 on 720 points is a gentle rounding of corners
    run_equator_slice(360, smooth_sigma=100.0)

In [ ]:
import numpy as np
import torch
import os
from pathlib import Path
from skimage import io
import fastremap
import matplotlib.pyplot as plt
from scipy.optimize import minimize

import plotly.graph_objects as go
from plotly.offline import plot
from IPython.display import HTML, display

import omnirefactor

dtype = torch.float64

# =========================================================================
# 0. Oklab Color Space (Standard)
# =========================================================================

def srgb_to_linear_np(srgb):
    srgb = np.clip(np.asarray(srgb, dtype=np.float64), 0.0, 1.0)
    mask = srgb <= 0.04045
    lin = np.empty_like(srgb)
    lin[mask] = srgb[mask] / 12.92
    lin[~mask] = ((srgb[~mask] + 0.055) / 1.055) ** 2.4
    return lin

def linear_to_srgb_np(lin):
    lin = np.asarray(lin, dtype=np.float64)
    valid_mask = (lin >= -0.0005) & (lin <= 1.0005)
    lin = np.clip(lin, 0.0, 1.0)
    mask = lin <= 0.0031308
    srgb = np.empty_like(lin)
    srgb[mask] = lin[mask] * 12.92
    srgb[~mask] = 1.055 * (lin[~mask] ** (1.0 / 2.4)) - 0.055
    return srgb, np.all(valid_mask, axis=-1)

M_srgb_xyz = np.array([[0.4124564, 0.3575761, 0.1804375],
                       [0.2126729, 0.7151522, 0.0721750],
                       [0.0193339, 0.1191920, 0.9503041]])
M_xyz_srgb = np.linalg.inv(M_srgb_xyz)

M1 = np.array([[0.8189330101, 0.3618667424, -0.1288597137],
               [0.0329845436, 0.9293118715, 0.0361456387],
               [0.0482003018, 0.2643662691, 0.6338517070]])
M2 = np.array([[0.2104542553, 0.7936177850, -0.0040720468],
               [1.9779984951, -2.4285922050, 0.4505937099],
               [0.0259040371, 0.7827717662, -0.8086757660]])
M1_INV = np.linalg.inv(M1)
M2_INV = np.linalg.inv(M2)

def xyz_to_oklab(xyz):
    lms = xyz @ M1.T
    lms_p = np.cbrt(np.maximum(lms, 0))
    return lms_p @ M2.T

def oklab_to_xyz(lab):
    lms_p = lab @ M2_INV.T
    lms = lms_p**3
    return lms @ M1_INV.T

def srgb_to_oklab(srgb):
    lin = srgb_to_linear_np(srgb)
    xyz = lin @ M_srgb_xyz.T
    return xyz_to_oklab(xyz)

def oklab_to_srgb_check(lab):
    xyz = oklab_to_xyz(lab)
    lin = xyz @ M_xyz_srgb.T
    return linear_to_srgb_np(lin)

# =========================================================================
# 1. Plane & Boundary Logic
# =========================================================================

def get_gamut_equator_plane():
    """Fits plane to 6 sRGB corners."""
    corners_rgb = np.array([
        [1,0,0], [1,1,0], [0,1,0], 
        [0,1,1], [0,0,1], [1,0,1]
    ], dtype=float)
    corners_lab = srgb_to_oklab(corners_rgb)
    
    # SVD for best fit plane
    centroid = np.mean(corners_lab, axis=0)
    u, s, vh = np.linalg.svd(corners_lab - centroid)
    normal = vh[2, :]
    if normal[0] < 0: normal = -normal # Face Up
    
    u_vec = vh[0, :]
    v_vec = np.cross(normal, u_vec)
    return centroid, normal, u_vec, v_vec

def ray_cast_on_plane(center, u, v, angle_rad):
    """Returns max radius at this angle on the plane."""
    direction = np.cos(angle_rad) * u + np.sin(angle_rad) * v
    low, high = 0.0, 0.50
    for _ in range(12):
        mid = (low + high) * 0.5
        pt = center + mid * direction
        _, valid = oklab_to_srgb_check(pt[None, :])
        if valid[0]: low = mid
        else: high = mid
    return low

def get_plane_boundary_polygon(n_scan=180):
    center, normal, u, v = get_gamut_equator_plane()
    angles = np.linspace(0, 2*np.pi, n_scan, endpoint=False)
    
    boundary_uv = [] # Store (u, v) coords relative to center
    
    for theta in angles:
        r = ray_cast_on_plane(center, u, v, theta)
        boundary_uv.append([r * np.cos(theta), r * np.sin(theta)])
        
    return np.array(boundary_uv), center, u, v

# =========================================================================
# 2. Ellipse Fitting
# =========================================================================

def check_ellipse_in_polygon(ellipse_params, polygon_uv):
    """
    Checks if the ellipse defined by params fits inside the origin-centered
    distance map defined by polygon_uv.
    """
    cx, cy, ra, rb, theta = ellipse_params
    
    # We check N points on the ellipse against the polygon boundary
    # polygon_uv is essentially a function R_poly(angle)
    # This is approximate but fast.
    
    t = np.linspace(0, 2*np.pi, 64, endpoint=False)
    cos_t, sin_t = np.cos(t), np.sin(t)
    
    # Ellipse points in UV plane
    rot = np.radians(theta)
    cr, sr = np.cos(rot), np.sin(rot)
    
    eu = cx + ra*cos_t*cr - rb*sin_t*sr
    ev = cy + ra*cos_t*sr + rb*sin_t*cr
    
    # Convert ellipse points to polar (r, ang)
    e_dist = np.sqrt(eu**2 + ev**2)
    e_ang = np.arctan2(ev, eu) # -pi to pi
    
    # Convert e_ang to 0..2pi index for lookup
    # Polygon is uniformly sampled in angle
    n_poly = len(polygon_uv)
    
    # To compare efficiently, we interpolate the polygon boundary radius at ellipse angles
    # Since polygon_uv was generated by uniform ray casting, we know the angles.
    poly_radii = np.linalg.norm(polygon_uv, axis=1)
    poly_angles = np.linspace(0, 2*np.pi, n_poly, endpoint=False)
    # Adjust angles to be monotonic for interp if needed (0..2pi)
    e_ang_mod = np.mod(e_ang, 2*np.pi)
    
    # Extend polygon data for wrap-around interp
    p_ang_wrap = np.concatenate([poly_angles, [2*np.pi]])
    p_rad_wrap = np.concatenate([poly_radii, [poly_radii[0]]])
    
    limit_radii = np.interp(e_ang_mod, p_ang_wrap, p_rad_wrap)
    
    # Check violation: Ellipse radius > Limit radius
    # We use a small epsilon buffer
    diff = limit_radii - e_dist
    
    if np.all(diff >= 0):
        return True, 0.0
    else:
        # Return penalty
        return False, np.sum(diff[diff < 0]**2)

def maximize_inscribed_ellipse(polygon_uv):
    """Finds largest ellipse (cx, cy, ra, rb, theta) inside polygon."""
    
    def loss(params):
        cx, cy, ra, rb, theta = params
        if ra < 0.01 or rb < 0.01: return 1e6
        
        fits, penalty = check_ellipse_in_polygon(params, polygon_uv)
        
        if fits:
            return -(ra * rb) # Maximize area
        else:
            return 1.0 + penalty * 1000.0
            
    # Init guess: centered circle
    # Estimate size from min radius of polygon
    r_est = np.min(np.linalg.norm(polygon_uv, axis=1)) * 0.8
    x0 = [0.0, 0.0, r_est, r_est, 0.0]
    
    bounds = [
        (-0.1, 0.1), (-0.1, 0.1), # Center drift small
        (0.01, 0.4), (0.01, 0.4), # Radii
        (-180, 180)
    ]
    
    res = minimize(loss, x0, bounds=bounds, method='Nelder-Mead', tol=1e-4)
    return res.x

def reconstruct_ellipse_3d(params, center_3d, u, v, n_points=72):
    cx, cy, ra, rb, theta_deg = params
    t = np.linspace(0, 2*np.pi, n_points, endpoint=False)
    
    rot = np.radians(theta_deg)
    cr, sr = np.cos(rot), np.sin(rot)
    cost, sint = np.cos(t), np.sin(t)
    
    # UV coords
    eu = cx + ra*cost*cr - rb*sint*sr
    ev = cy + ra*cost*sr + rb*sint*cr
    
    # 3D Coords
    pts = center_3d[None, :] + eu[:, None]*u[None, :] + ev[:, None]*v[None, :]
    return pts

# =========================================================================
# 3. Utils & Vis
# =========================================================================

def resample_uniform_arc(curve, n_target):
    closed = np.vstack([curve, curve[0]])
    dist = np.cumsum(np.linalg.norm(np.diff(closed, axis=0), axis=1))
    dist = np.insert(dist, 0, 0)
    t_uni = np.linspace(0, dist[-1], n_target, endpoint=False)
    out = np.zeros((n_target, 3))
    for i in range(3):
        out[:,i] = np.interp(t_uni, dist, closed[:,i])
    return out

def generate_vis_row(rgb_ring, flows_np, n_hues):
    y, x = np.ogrid[-1:1:complex(0, 256), -1:1:complex(0, 256)]
    mask = x**2 + y**2 > 1.0
    ang = np.arctan2(y, x)
    idx = (((ang + np.pi) / (2*np.pi)) * len(rgb_ring)).astype(int) % len(rgb_ring)
    disk = rgb_ring[idx].copy()
    disk[mask] = 0
    
    dy, dx = flows_np[0], flows_np[1]
    mag = np.sqrt(dy**2 + dx**2)
    p99 = np.percentile(mag, 99)
    if p99 < 1e-5: p99 = 1.0
    alpha = np.clip(mag / p99, 0, 1)[..., None]
    
    ang_f = np.arctan2(dy, dx)
    def get_c(r, a): return r[(((a + np.pi) / (2*np.pi)) * len(r)).astype(int) % len(r)]
    
    c = get_c(rgb_ring, ang_f)
    f = c * alpha + 0.1*(1-alpha)
    ow = c * alpha + (1.0 - alpha)
    ob = c * alpha
    
    rot = np.roll(rgb_ring, n_hues//4, axis=0)
    cr = get_c(rot, ang_f)
    rw = cr * alpha + (1.0 - alpha)
    rb = cr * alpha
    return [disk, f, ow, ob, rw, rb]

def get_srgb_wireframe_oklab(n=20):
    edges = [([0,0,0],[1,0,0]), ([1,0,0],[1,1,0]), ([1,1,0],[0,1,0]), ([0,1,0],[0,0,0]),
             ([0,0,1],[1,0,1]), ([1,0,1],[1,1,1]), ([1,1,1],[0,1,1]), ([0,1,1],[0,0,1]),
             ([0,0,0],[0,0,1]), ([1,0,0],[1,0,1]), ([1,1,0],[1,1,1]), ([0,1,0],[0,1,1])]
    x, y, z = [], [], []
    for s, e in edges:
        t = np.linspace(0, 1, n)
        rgb = np.array(s) + t[:,None]*(np.array(e)-np.array(s))
        lab = srgb_to_oklab(rgb)
        x.extend(lab[:,1]); y.extend(lab[:,2]); z.extend(lab[:,0])
        x.append(None); y.append(None); z.append(None)
    return x, y, z

# =========================================================================
# 4. Main
# =========================================================================

def run_equator_inscribed_ellipse(n_hues=72):
    # 1. Get the Polygon on the "Average Plane"
    poly_uv, center_3d, u, v = get_plane_boundary_polygon(n_scan=180)
    
    # 2. Fit Maximum Ellipse inside that polygon
    params = maximize_inscribed_ellipse(poly_uv)
    
    # 3. Reconstruct 3D Loop
    lab_raw = reconstruct_ellipse_3d(params, center_3d, u, v, n_points=720)
    
    # 4. Resample
    lab_uni = resample_uniform_arc(lab_raw, n_hues)
    
    # 5. Convert
    rgb_opt, _ = oklab_to_srgb_check(lab_uni)
    rgb_opt = np.clip(rgb_opt, 0, 1)
    
    # --- Vis ---
    t = np.linspace(0, 1, n_hues, endpoint=False)
    sr = 0.5 + 0.5*np.sin(2*np.pi*(t+0/3))
    sg = 0.5 + 0.5*np.sin(2*np.pi*(t+1/3))
    sb = 0.5 + 0.5*np.sin(2*np.pi*(t+2/3))
    rgb_sine = np.stack([sr, sg, sb], axis=1)
    
    def get_g(r): return np.argmax(r[:,1] - 0.5*(r[:,0]+r[:,2]))
    g_ref = get_g(rgb_sine)
    rgb_opt_a = np.roll(rgb_opt, g_ref - get_g(rgb_opt), axis=0)
    
    omnidir = Path(omnirefactor.__file__).parent.parent.parent
    masks_path = os.path.join(omnidir, "docs", "_static", "ecoli_labels.tif")
    if os.path.exists(masks_path):
        masks = io.imread(masks_path)
        slc = omnirefactor.measure.crop_bbox(masks>0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows_tensor = omnirefactor.core.masks_to_flows(masks, device="cpu")[4]
        flows_np = flows_tensor.cpu().numpy() if hasattr(flows_tensor, 'cpu') else flows_tensor
    else:
        y, x = np.mgrid[-10:10:0.1, -10:10:0.1]
        flows_np = np.stack([np.sin(x/3), np.cos(y/3)]) * 5

    row_s = generate_vis_row(rgb_sine, flows_np, n_hues)
    row_o = generate_vis_row(rgb_opt_a, flows_np, n_hues)
    
    fig, ax = plt.subplots(2, 6, figsize=(24, 8))
    cols = ["Disk", "Flow", "Alpha/W", "Alpha/B", "Rot/W", "Rot/B"]
    data = [row_s, row_o]
    names = ["Sinebow", "Inscribed Equator Ellipse"]
    
    for i in range(2):
        for j in range(6):
            ax[i,j].imshow(data[i][j]); ax[i,j].axis('off')
            if i==0: ax[i,j].set_title(cols[j])
            if j==0: ax[i,j].text(-0.1, 0.5, names[i], transform=ax[i,j].transAxes, va='center', ha='right', rotation=90, fontsize=12)
    plt.tight_layout()
    plt.show()
    
    # --- 3D ---
    lab_sin = srgb_to_oklab(rgb_sine)
    lab_plot = srgb_to_oklab(rgb_opt_a)
    
    # Generate raw intersection points for plotting
    pts_poly = center_3d[None,:] + poly_uv[:,0][:,None]*u[None,:] + poly_uv[:,1][:,None]*v[None,:]
    
    fig3d = go.Figure()
    wx, wy, wz = get_srgb_wireframe_oklab()
    fig3d.add_trace(go.Scatter3d(x=wx, y=wy, z=wz, mode='lines', name='sRGB Wireframe', line=dict(color='gray', width=3), hoverinfo='skip'))
    fig3d.add_trace(go.Scatter3d(x=pts_poly[:,1], y=pts_poly[:,2], z=pts_poly[:,0], mode='lines', name='Plane Intersection', line=dict(width=2, color='white', dash='dot')))
    fig3d.add_trace(go.Scatter3d(x=lab_sin[:,1], y=lab_sin[:,2], z=lab_sin[:,0], mode='lines', name='Sinebow', line=dict(width=4, color='yellow')))
    fig3d.add_trace(go.Scatter3d(x=lab_plot[:,1], y=lab_plot[:,2], z=lab_plot[:,0], mode='lines', name='Inscribed Ellipse', line=dict(width=10, color='cyan')))
    
    fig3d.update_layout(template="plotly_dark", height=600, title="Inscribed Ellipse on Equator Plane", scene=dict(xaxis_title='a', yaxis_title='b', zaxis_title='L'))
    display(HTML(plot(fig3d, output_type='div', include_plotlyjs='cdn')))

if __name__ == "__main__":
    run_equator_inscribed_ellipse(72)

In [ ]:
import numpy as np
import torch
import os
from pathlib import Path
from skimage import io
import fastremap
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

import plotly.graph_objects as go
from plotly.offline import plot
from IPython.display import HTML, display

import omnirefactor

dtype = torch.float64

# =========================================================================
# 0. Oklab Color Space
# =========================================================================

def srgb_to_linear_np(srgb):
    srgb = np.clip(np.asarray(srgb, dtype=np.float64), 0.0, 1.0)
    mask = srgb <= 0.04045
    lin = np.empty_like(srgb)
    lin[mask] = srgb[mask] / 12.92
    lin[~mask] = ((srgb[~mask] + 0.055) / 1.055) ** 2.4
    return lin

def linear_to_srgb_np(lin):
    lin = np.asarray(lin, dtype=np.float64)
    valid_mask = (lin >= -0.0005) & (lin <= 1.0005)
    lin = np.clip(lin, 0.0, 1.0)
    mask = lin <= 0.0031308
    srgb = np.empty_like(lin)
    srgb[mask] = lin[mask] * 12.92
    srgb[~mask] = 1.055 * (lin[~mask] ** (1.0 / 2.4)) - 0.055
    return srgb, np.all(valid_mask, axis=-1)

M_srgb_xyz = np.array([[0.4124564, 0.3575761, 0.1804375],
                       [0.2126729, 0.7151522, 0.0721750],
                       [0.0193339, 0.1191920, 0.9503041]])
M_xyz_srgb = np.linalg.inv(M_srgb_xyz)

M1 = np.array([[0.8189330101, 0.3618667424, -0.1288597137],
               [0.0329845436, 0.9293118715, 0.0361456387],
               [0.0482003018, 0.2643662691, 0.6338517070]])
M2 = np.array([[0.2104542553, 0.7936177850, -0.0040720468],
               [1.9779984951, -2.4285922050, 0.4505937099],
               [0.0259040371, 0.7827717662, -0.8086757660]])
M1_INV = np.linalg.inv(M1)
M2_INV = np.linalg.inv(M2)

def xyz_to_oklab(xyz):
    lms = xyz @ M1.T
    lms_p = np.cbrt(np.maximum(lms, 0))
    return lms_p @ M2.T

def oklab_to_xyz(lab):
    lms_p = lab @ M2_INV.T
    lms = lms_p**3
    return lms @ M1_INV.T

def srgb_to_oklab(srgb):
    lin = srgb_to_linear_np(srgb)
    xyz = lin @ M_srgb_xyz.T
    return xyz_to_oklab(xyz)

def oklab_to_srgb_check(lab):
    xyz = oklab_to_xyz(lab)
    lin = xyz @ M_xyz_srgb.T
    return linear_to_srgb_np(lin)

# =========================================================================
# 1. Smooth Gaussian Loop
# =========================================================================

def get_hsv_boundary_oklab(n=1000):
    h = np.linspace(0, 1, n, endpoint=False)
    i = (h * 6).astype(int); f = h * 6 - i
    q = 1 - f; t = f
    rgb = np.zeros((n, 3))
    
    m = (i % 6 == 0); rgb[m] = np.stack([np.ones(m.sum()), t[m], np.zeros(m.sum())], axis=1)
    m = (i % 6 == 1); rgb[m] = np.stack([q[m], np.ones(m.sum()), np.zeros(m.sum())], axis=1)
    m = (i % 6 == 2); rgb[m] = np.stack([np.zeros(m.sum()), np.ones(m.sum()), t[m]], axis=1)
    m = (i % 6 == 3); rgb[m] = np.stack([np.zeros(m.sum()), q[m], np.ones(m.sum())], axis=1)
    m = (i % 6 == 4); rgb[m] = np.stack([t[m], np.zeros(m.sum()), np.ones(m.sum())], axis=1)
    m = (i % 6 == 5); rgb[m] = np.stack([np.ones(m.sum()), np.zeros(m.sum()), q[m]], axis=1)
    
    return srgb_to_oklab(rgb)

def resample_uniform_arc(curve, n_target):
    """Reparameterize to ensure Gaussian blur is geometric, not temporal."""
    closed = np.vstack([curve, curve[0]])
    dists = np.linalg.norm(np.diff(closed, axis=0), axis=1)
    cum = np.concatenate([[0.0], np.cumsum(dists)])
    
    t_orig = cum / cum[-1]
    t_new = np.linspace(0, 1, n_target, endpoint=False)
    
    out = np.zeros((n_target, 3))
    for i in range(3):
        # Append for cyclic interpolation
        data = np.concatenate([curve[:,i], [curve[0,i]]])
        out[:,i] = np.interp(t_new, t_orig, data)
    return out

def gaussian_smooth_loop(curve, sigma):
    """
    Applies cyclic Gaussian smoothing.
    This mathematically rounds off corners without ringing.
    """
    smooth = np.zeros_like(curve)
    for i in range(3):
        smooth[:,i] = gaussian_filter1d(curve[:,i], sigma=sigma, mode='wrap')
    return smooth

def clip_gamut_if_needed(lab_curve):
    """
    Pull back any points that drifted out (rare with Gaussian smoothing on convex hulls,
    but sRGB has slight concavities in Oklab).
    """
    _, valid = oklab_to_srgb_check(lab_curve)
    
    if np.all(valid): return lab_curve
    
    clipped = lab_curve.copy()
    idxs = np.where(~valid)[0]
    # print(f"  Clipping {len(idxs)} points...")
    
    for i in idxs:
        L, a, b = lab_curve[i]
        low, high = 0.0, 1.0
        for _ in range(8):
            mid = (low + high) / 2
            # Pull towards L axis (desaturate)
            pt = np.array([L, a*mid, b*mid])
            # Fast check
            xyz = oklab_to_xyz(pt)
            lin = xyz @ M_xyz_srgb.T
            if np.all((lin >= -0.001) & (lin <= 1.001)):
                low = mid
            else:
                high = mid
        clipped[i, 1] *= low
        clipped[i, 2] *= low
        
    return clipped

# =========================================================================
# 2. Visualization
# =========================================================================

def generate_vis_row(rgb_ring, flows_np, n_hues):
    y, x = np.ogrid[-1:1:complex(0, 256), -1:1:complex(0, 256)]
    mask = x**2 + y**2 > 1.0
    ang = np.arctan2(y, x)
    idx = (((ang + np.pi) / (2*np.pi)) * len(rgb_ring)).astype(int) % len(rgb_ring)
    disk = rgb_ring[idx].copy()
    disk[mask] = 0
    
    dy, dx = flows_np[0], flows_np[1]
    mag = np.sqrt(dy**2 + dx**2)
    p99 = np.percentile(mag, 99)
    if p99 < 1e-5: p99 = 1.0
    alpha = np.clip(mag / p99, 0, 1)[..., None]
    
    ang_f = np.arctan2(dy, dx)
    def get_c(r, a): 
        ix = (((a + np.pi) / (2*np.pi)) * len(r)).astype(int) % len(r)
        return r[ix]
    
    c = get_c(rgb_ring, ang_f)
    f = c * alpha + 0.1*(1-alpha)
    ow = c * alpha + (1.0 - alpha)
    ob = c * alpha
    
    rot = np.roll(rgb_ring, n_hues//4, axis=0)
    cr = get_c(rot, ang_f)
    rw = cr * alpha + (1.0 - alpha)
    rb = cr * alpha
    return [disk, f, ow, ob, rw, rb]

def get_srgb_wireframe_oklab(n=20):
    edges = [([0,0,0],[1,0,0]), ([1,0,0],[1,1,0]), ([1,1,0],[0,1,0]), ([0,1,0],[0,0,0]),
             ([0,0,1],[1,0,1]), ([1,0,1],[1,1,1]), ([1,1,1],[0,1,1]), ([0,1,1],[0,0,1]),
             ([0,0,0],[0,0,1]), ([1,0,0],[1,0,1]), ([1,1,0],[1,1,1]), ([0,1,0],[0,1,1])]
    x, y, z = [], [], []
    for s, e in edges:
        t = np.linspace(0, 1, n)
        rgb = np.array(s) + t[:,None]*(np.array(e)-np.array(s))
        lab = srgb_to_oklab(rgb)
        x.extend(lab[:,1]); y.extend(lab[:,2]); z.extend(lab[:,0])
        x.append(None); y.append(None); z.append(None)
    return x, y, z

# =========================================================================
# 3. Main
# =========================================================================

def run_oklab_gaussian(n_hues=72, blur_sigma=40):
    # 1. Truth (High Res)
    lab_raw = get_hsv_boundary_oklab(n=720)
    
    # 2. Uniform Resample (Critical for even smoothing)
    lab_uni = resample_uniform_arc(lab_raw, n_target=720)
    
    # 3. Gaussian Blur
    # Sigma determines how much we "melt" the corners.
    # 40 on 720 points is a very strong smooth (5.5% of circumference)
    lab_smooth = gaussian_smooth_loop(lab_uni, sigma=blur_sigma)
    
    # 4. Clip (Safety)
    lab_safe = clip_gamut_if_needed(lab_smooth)
    
    # 5. Final Resample
    lab_final = resample_uniform_arc(lab_safe, n_hues)
    
    # 6. RGB
    rgb_opt, _ = oklab_to_srgb_check(lab_final)
    rgb_opt = np.clip(rgb_opt, 0, 1)
    
    # --- Vis Setup ---
    t = np.linspace(0, 1, n_hues, endpoint=False)
    sr = 0.5 + 0.5*np.sin(2*np.pi*(t+0/3))
    sg = 0.5 + 0.5*np.sin(2*np.pi*(t+1/3))
    sb = 0.5 + 0.5*np.sin(2*np.pi*(t+2/3))
    rgb_sine = np.stack([sr, sg, sb], axis=1)
    
    def get_g(r): return np.argmax(r[:,1] - 0.5*(r[:,0]+r[:,2]))
    g_ref = get_g(rgb_sine)
    rgb_opt_a = np.roll(rgb_opt, g_ref - get_g(rgb_opt), axis=0)
    
    omnidir = Path(omnirefactor.__file__).parent.parent.parent
    masks_path = os.path.join(omnidir, "docs", "_static", "ecoli_labels.tif")
    if os.path.exists(masks_path):
        masks = io.imread(masks_path)
        slc = omnirefactor.measure.crop_bbox(masks>0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows_tensor = omnirefactor.core.masks_to_flows(masks, device="cpu")[4]
        flows_np = flows_tensor.cpu().numpy() if hasattr(flows_tensor, 'cpu') else flows_tensor
    else:
        y, x = np.mgrid[-10:10:0.1, -10:10:0.1]
        flows_np = np.stack([np.sin(x/3), np.cos(y/3)]) * 5

    row_s = generate_vis_row(rgb_sine, flows_np, n_hues)
    row_o = generate_vis_row(rgb_opt_a, flows_np, n_hues)
    
    fig, ax = plt.subplots(2, 6, figsize=(24, 8))
    cols = ["Disk", "Flow", "Alpha/W", "Alpha/B", "Rot/W", "Rot/B"]
    data = [row_s, row_o]
    names = ["Sinebow", f"Oklab Gaussian (S={blur_sigma})"]
    
    for i in range(2):
        for j in range(6):
            ax[i,j].imshow(data[i][j]); ax[i,j].axis('off')
            if i==0: ax[i,j].set_title(cols[j])
            if j==0: ax[i,j].text(-0.1, 0.5, names[i], transform=ax[i,j].transAxes, va='center', ha='right', rotation=90, fontsize=12)
    plt.tight_layout()
    plt.show()
    
    # --- 3D ---
    lab_sin = srgb_to_oklab(rgb_sine)
    lab_plot = srgb_to_oklab(rgb_opt_a)
    
    fig3d = go.Figure()
    wx, wy, wz = get_srgb_wireframe_oklab()
    fig3d.add_trace(go.Scatter3d(x=wx, y=wy, z=wz, mode='lines', name='sRGB Wireframe', line=dict(color='gray', width=3), hoverinfo='skip'))
    
    # Truth Dots
    fig3d.add_trace(go.Scatter3d(x=lab_raw[:,1], y=lab_raw[:,2], z=lab_raw[:,0], mode='markers', name='HSV Truth', marker=dict(size=2, color='white')))
    
    fig3d.add_trace(go.Scatter3d(x=lab_sin[:,1], y=lab_sin[:,2], z=lab_sin[:,0], mode='lines', name='Sinebow', line=dict(width=4, color='yellow')))
    fig3d.add_trace(go.Scatter3d(x=lab_plot[:,1], y=lab_plot[:,2], z=lab_plot[:,0], mode='lines', name='Gaussian Loop', line=dict(width=10, color='cyan')))
    
    fig3d.update_layout(template="plotly_dark", height=600, title=f"Oklab Gaussian Smoothing (Sigma={blur_sigma})", scene=dict(xaxis_title='a', yaxis_title='b', zaxis_title='L'))
    display(HTML(plot(fig3d, output_type='div', include_plotlyjs='cdn')))

if __name__ == "__main__":
    # Sigma 40 on 720 points gives a very pleasing "Rounded Hexagon"
    run_oklab_gaussian(150, blur_sigma=70)

In [ ]:
import numpy as np
import torch
import os
from pathlib import Path
from skimage import io
import fastremap
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

import plotly.graph_objects as go
from plotly.offline import plot
from IPython.display import HTML, display

import omnirefactor

dtype = torch.float64

# =========================================================================
# 0. Robust JzAzBz (Safdar 2017, 200 nits)
# =========================================================================

def srgb_to_linear_np(srgb):
    srgb = np.clip(np.asarray(srgb, dtype=np.float64), 0.0, 1.0)
    mask = srgb <= 0.04045
    lin = np.empty_like(srgb)
    lin[mask] = srgb[mask] / 12.92
    lin[~mask] = ((srgb[~mask] + 0.055) / 1.055) ** 2.4
    return lin

def linear_to_srgb_np(lin):
    lin = np.asarray(lin, dtype=np.float64)
    valid_mask = (lin >= -0.0005) & (lin <= 1.0005)
    lin_clamped = np.clip(lin, 0.0, 1.0)
    mask = lin_clamped <= 0.0031308
    srgb = np.empty_like(lin_clamped)
    srgb[mask] = lin_clamped[mask] * 12.92
    srgb[~mask] = 1.055 * (lin_clamped[~mask] ** (1.0 / 2.4)) - 0.055
    return srgb, np.all(valid_mask, axis=-1)

M_srgb_xyz = np.array([[0.4124564, 0.3575761, 0.1804375],
                       [0.2126729, 0.7151522, 0.0721750],
                       [0.0193339, 0.1191920, 0.9503041]])
M_xyz_srgb = np.linalg.inv(M_srgb_xyz)

# Safdar 2017 Constants
JZ_B, JZ_G = 1.15, 0.66
JZ_C1, JZ_C2, JZ_C3 = 3424/4096, 2413/128, 2392/128
JZ_N, JZ_P, JZ_D = 2610/16384, 1.7 * 2523/32, -0.56
M1 = np.array([[0.41478972, 0.579999, 0.0146480],[-0.2015100, 1.120649, 0.0531008],[-0.0166008, 0.264800, 0.6684799]])
M2 = np.array([[0.5, 0.5, 0],[3.524000, -4.066708, 0.542708],[0.199076, 1.096799, -1.295875]])
M1_INV = np.linalg.inv(M1); M2_INV = np.linalg.inv(M2)
DISPLAY_PEAK_NITS = 200.0 

def xyz_to_jzazbz(xyz):
    xyz_abs = xyz * DISPLAY_PEAK_NITS
    lms = np.maximum((np.stack([JZ_B*xyz_abs[...,0]-(JZ_B-1)*xyz_abs[...,2], JZ_G*xyz_abs[...,1]-(JZ_G-1)*xyz_abs[...,0], xyz_abs[...,2]], axis=-1)) @ M1.T, 1e-12)
    iz = np.maximum((JZ_C1 + JZ_C2 * np.power(lms, JZ_N)) / (1 + JZ_C3 * np.power(lms, JZ_N)), 1e-12)
    jab = np.power(iz, JZ_P) @ M2.T
    return np.stack([jab[..., 0] + JZ_D, jab[..., 1], jab[..., 2]], axis=-1)

def jzazbz_to_xyz(jab):
    iz = np.power(np.maximum(np.stack([jab[...,0]-JZ_D, jab[...,1], jab[...,2]], axis=-1) @ M2_INV.T, 1e-12), 1.0/JZ_P)
    lms = np.power(np.maximum((iz - JZ_C1) / (JZ_C2 - JZ_C3 * iz + 1e-12), 0), 1.0/JZ_N)
    xyz_p = lms @ M1_INV.T
    X = (xyz_p[...,0] + (JZ_B-1)*xyz_p[...,2]) / JZ_B
    Y = (xyz_p[...,1] + (JZ_G-1)*X) / JZ_G
    return np.stack([X, Y, xyz_p[...,2]], axis=-1) / DISPLAY_PEAK_NITS

def srgb_to_jzazbz(srgb):
    return xyz_to_jzazbz(srgb_to_linear_np(srgb) @ M_srgb_xyz.T)

def jzazbz_to_srgb_check(jz):
    xyz = jzazbz_to_xyz(jz)
    lin = xyz @ M_xyz_srgb.T
    return linear_to_srgb_np(lin)

# =========================================================================
# 1. Geometry Pipeline
# =========================================================================

def get_hsv_boundary_jz(n=720):
    """
    Trace the RGB Cube edges (The "Truth").
    """
    h = np.linspace(0, 1, n, endpoint=False)
    i = (h * 6).astype(int); f = h * 6 - i
    q = 1 - f; t = f
    rgb = np.zeros((n, 3))
    
    m = (i%6==0); rgb[m] = np.stack([np.ones(m.sum()), t[m], np.zeros(m.sum())], axis=1)
    m = (i%6==1); rgb[m] = np.stack([q[m], np.ones(m.sum()), np.zeros(m.sum())], axis=1)
    m = (i%6==2); rgb[m] = np.stack([np.zeros(m.sum()), np.ones(m.sum()), t[m]], axis=1)
    m = (i%6==3); rgb[m] = np.stack([np.zeros(m.sum()), q[m], np.ones(m.sum())], axis=1)
    m = (i%6==4); rgb[m] = np.stack([t[m], np.zeros(m.sum()), np.ones(m.sum())], axis=1)
    m = (i%6==5); rgb[m] = np.stack([np.ones(m.sum()), np.zeros(m.sum()), q[m]], axis=1)
    
    return srgb_to_jzazbz(rgb)

def resample_uniform_arc(curve, n_target):
    """
    Reparameterize curve so points are equidistant in Euclidean JzAzBz space.
    Crucial for uniform smoothing.
    """
    closed = np.vstack([curve, curve[0]])
    dists = np.linalg.norm(np.diff(closed, axis=0), axis=1)
    cum = np.concatenate([[0.0], np.cumsum(dists)])
    
    t_orig = cum / cum[-1]
    t_new = np.linspace(0, 1, n_target, endpoint=False)
    
    out = np.zeros((n_target, 3))
    for i in range(3):
        data = np.concatenate([curve[:,i], [curve[0,i]]])
        out[:,i] = np.interp(t_new, t_orig, data)
    return out

def gaussian_smooth_loop(curve, sigma):
    """
    Applies cyclic Gaussian smoothing.
    This rounds off corners purely by averaging neighbors.
    """
    smooth = np.zeros_like(curve)
    for i in range(3):
        smooth[:,i] = gaussian_filter1d(curve[:,i], sigma=sigma, mode='wrap')
    return smooth

def clip_to_gamut(jz_curve):
    """
    If smoothing pushed concave areas (Cyan, Magenta) out, pull them back.
    """
    _, valid = jzazbz_to_srgb_check(jz_curve)
    
    if np.all(valid): return jz_curve
    
    clipped = jz_curve.copy()
    idx = np.where(~valid)[0]
    
    for i in idx:
        J, a, b = jz_curve[i]
        low, high = 0.0, 1.0
        for _ in range(10):
            mid = (low + high) / 2
            pt = np.array([J, a*mid, b*mid])
            
            xyz = jzazbz_to_xyz(pt)
            lin = xyz @ M_xyz_srgb.T
            if np.all((lin >= -0.001) & (lin <= 1.001)):
                low = mid
            else:
                high = mid
        clipped[i, 1:] *= low
    return clipped

# =========================================================================
# 2. Visualization
# =========================================================================

def generate_vis_row(rgb_ring, flows_np, n_hues):
    y, x = np.ogrid[-1:1:complex(0, 256), -1:1:complex(0, 256)]
    mask = x**2 + y**2 > 1.0
    ang = np.arctan2(y, x)
    idx = (((ang + np.pi) / (2*np.pi)) * len(rgb_ring)).astype(int) % len(rgb_ring)
    disk = rgb_ring[idx].copy()
    disk[mask] = 0
    
    dy, dx = flows_np[0], flows_np[1]
    mag = np.sqrt(dy**2 + dx**2)
    p99 = np.percentile(mag, 99)
    if p99 < 1e-5: p99 = 1.0
    alpha = np.clip(mag / p99, 0, 1)[..., None]
    
    ang_f = np.arctan2(dy, dx)
    def get_c(r, a): return r[(((a + np.pi) / (2*np.pi)) * len(r)).astype(int) % len(r)]
    
    c = get_c(rgb_ring, ang_f)
    f = c * alpha + 0.1*(1-alpha)
    ow = c * alpha + (1.0 - alpha)
    ob = c * alpha
    
    rot = np.roll(rgb_ring, n_hues//4, axis=0)
    cr = get_c(rot, ang_f)
    rw = cr * alpha + (1.0 - alpha)
    rb = cr * alpha
    return [disk, f, ow, ob, rw, rb]

def get_srgb_wireframe_jz(n=20):
    edges = [([0,0,0],[1,0,0]), ([1,0,0],[1,1,0]), ([1,1,0],[0,1,0]), ([0,1,0],[0,0,0]),
             ([0,0,1],[1,0,1]), ([1,0,1],[1,1,1]), ([1,1,1],[0,1,1]), ([0,1,1],[0,0,1]),
             ([0,0,0],[0,0,1]), ([1,0,0],[1,0,1]), ([1,1,0],[1,1,1]), ([0,1,0],[0,1,1])]
    x, y, z = [], [], []
    for s, e in edges:
        t = np.linspace(0, 1, n)
        rgb = np.array(s) + t[:,None]*(np.array(e)-np.array(s))
        jz = srgb_to_jzazbz(rgb)
        x.extend(jz[:,1]); y.extend(jz[:,2]); z.extend(jz[:,0])
        x.append(None); y.append(None); z.append(None)
    return x, y, z

# =========================================================================
# 3. Main
# =========================================================================

def run_jz_gaussian_smooth(n_hues=72, blur_sigma=40.0):
    # 1. Trace Truth (High Res)
    jz_raw = get_hsv_boundary_jz(n=720)
    
    # 2. Uniform Resample (Critical)
    jz_uni = resample_uniform_arc(jz_raw, n_target=720)
    
    # 3. Gaussian Blur (The Smoothing)
    jz_smooth = gaussian_smooth_loop(jz_uni, sigma=blur_sigma)
    
    # 4. Clip (Safety)
    jz_safe = clip_to_gamut(jz_smooth)
    
    # 5. Final Resample
    jz_final = resample_uniform_arc(jz_safe, n_target=n_hues)
    
    # 6. RGB
    rgb_opt, _ = jzazbz_to_srgb_check(jz_final)
    rgb_opt = np.clip(rgb_opt, 0, 1)
    
    # --- Vis ---
    t = np.linspace(0, 1, n_hues, endpoint=False)
    sr = 0.5 + 0.5*np.sin(2*np.pi*(t+0/3))
    sg = 0.5 + 0.5*np.sin(2*np.pi*(t+1/3))
    sb = 0.5 + 0.5*np.sin(2*np.pi*(t+2/3))
    rgb_sine = np.stack([sr, sg, sb], axis=1)
    
    def get_g(r): return np.argmax(r[:,1] - 0.5*(r[:,0]+r[:,2]))
    g_ref = get_g(rgb_sine)
    curr = get_g(rgb_opt)
    rgb_opt_a = np.roll(rgb_opt, g_ref - curr, axis=0)
    
    omnidir = Path(omnirefactor.__file__).parent.parent.parent
    masks_path = os.path.join(omnidir, "docs", "_static", "ecoli_labels.tif")
    if os.path.exists(masks_path):
        masks = io.imread(masks_path)
        slc = omnirefactor.measure.crop_bbox(masks>0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows_tensor = omnirefactor.core.masks_to_flows(masks, device="cpu")[4]
        flows_np = flows_tensor.cpu().numpy() if hasattr(flows_tensor, 'cpu') else flows_tensor
    else:
        y, x = np.mgrid[-10:10:0.1, -10:10:0.1]
        flows_np = np.stack([np.sin(x/3), np.cos(y/3)]) * 5

    row_s = generate_vis_row(rgb_sine, flows_np, n_hues)
    row_o = generate_vis_row(rgb_opt_a, flows_np, n_hues)
    
    fig, ax = plt.subplots(2, 6, figsize=(24, 8))
    cols = ["Disk", "Flow", "Alpha/W", "Alpha/B", "Rot/W", "Rot/B"]
    data = [row_s, row_o]
    names = ["Sinebow", f"JzAzBz Gaussian (S={blur_sigma})"]
    
    for i in range(2):
        for j in range(6):
            ax[i,j].imshow(data[i][j]); ax[i,j].axis('off')
            if i==0: ax[i,j].set_title(cols[j])
            if j==0: ax[i,j].text(-0.1, 0.5, names[i], transform=ax[i,j].transAxes, va='center', ha='right', rotation=90, fontsize=12)
    plt.tight_layout()
    plt.show()
    
    # --- 3D Plot ---
    
    jz_sin = srgb_to_jzazbz(rgb_sine)
    jz_plot = srgb_to_jzazbz(rgb_opt_a)
    
    # Get Truth Colors
    rgb_truth, _ = jzazbz_to_srgb_check(jz_raw)
    rgb_truth = np.clip(rgb_truth, 0, 1)
    
    fig3d = go.Figure()
    wx, wy, wz = get_srgb_wireframe_jz()
    fig3d.add_trace(go.Scatter3d(x=wx, y=wy, z=wz, mode='lines', name='sRGB Wireframe', line=dict(color='gray', width=3), hoverinfo='skip'))
    fig3d.add_trace(go.Scatter3d(x=jz_raw[:,1], y=jz_raw[:,2], z=jz_raw[:,0], mode='markers', name='Raw HSV Truth', marker=dict(size=2, color=rgb_truth)))
    fig3d.add_trace(go.Scatter3d(x=jz_plot[:,1], y=jz_plot[:,2], z=jz_plot[:,0], mode='lines', name='Gaussian Result', line=dict(width=10, color='cyan')))
    fig3d.add_trace(go.Scatter3d(x=jz_sin[:,1], y=jz_sin[:,2], z=jz_sin[:,0], mode='lines', name='Sinebow', line=dict(width=4, color='yellow')))
    
    fig3d.update_layout(template="plotly_dark", height=600, title=f"JzAzBz Gaussian Smoothing (S={blur_sigma})", scene=dict(xaxis_title='az', yaxis_title='bz', zaxis_title='Jz'))
    display(HTML(plot(fig3d, output_type='div', include_plotlyjs='cdn')))

if __name__ == "__main__":
    # Sigma 40 on 720 points is a strong smooth that removes the hex corners
    run_jz_gaussian_smooth(72, blur_sigma=66.0)

In [ ]:
new idea to try: reparametrize hsv in unifomr space, then blur the colors a bit around the edge
or do that with spectral colors and then shrink uniformly until inside the gamut. 

    plane could be the one maximizing distance from all 6 orimaries, 
or the average location of the HSV loop 


maybe choose a blue such that the lopp is equidistant from primaries?

In [ ]:
import numpy as np
import torch
import os
from pathlib import Path
from skimage import io
import fastremap
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

import plotly.graph_objects as go
from plotly.offline import plot
from IPython.display import HTML, display

import omnirefactor

dtype = torch.float64

# =========================================================================
# 0. JzAzBz Implementation (Exact Safdar 2017 with 1e4 Scaling)
# =========================================================================

def srgb_to_linear_np(srgb):
    srgb = np.clip(np.asarray(srgb, dtype=np.float64), 0.0, 1.0)
    mask = srgb <= 0.04045
    lin = np.empty_like(srgb)
    lin[mask] = srgb[mask] / 12.92
    lin[~mask] = ((srgb[~mask] + 0.055) / 1.055) ** 2.4
    return lin

def linear_to_srgb_np(lin):
    lin = np.asarray(lin, dtype=np.float64)
    # Validity Check
    valid_mask = (lin >= -0.0005) & (lin <= 1.0005)
    
    lin_clamped = np.clip(lin, 0.0, 1.0)
    mask = lin_clamped <= 0.0031308
    srgb = np.empty_like(lin_clamped)
    srgb[mask] = lin_clamped[mask] * 12.92
    srgb[~mask] = 1.055 * (lin_clamped[~mask] ** (1.0 / 2.4)) - 0.055
    return srgb, np.all(valid_mask, axis=-1)

M_srgb_xyz = np.array([[0.4124564, 0.3575761, 0.1804375],
                       [0.2126729, 0.7151522, 0.0721750],
                       [0.0193339, 0.1191920, 0.9503041]])
M_xyz_srgb = np.linalg.inv(M_srgb_xyz)

# Constants (Safdar 2017)
b_ = 1.15
g_ = 0.66
c1 = 3424 / 2**12
c2 = 2413 / 2**7
c3 = 2392 / 2**7
n  = 2610 / 2**14
p  = 1.7 * 2523 / 2**5
d  = -0.56

M1 = np.array([[0.41478972, 0.579999, 0.0146480],
               [-0.2015100, 1.120649, 0.0531008],
               [-0.0166008, 0.264800, 0.6684799]])
M2 = np.array([[0.5, 0.5, 0],
               [3.524000, -4.066708, 0.542708],
               [0.199076, 1.096799, -1.295875]])
M1_INV = np.linalg.inv(M1)
M2_INV = np.linalg.inv(M2)

def xyz_to_jzazbz(xyz):
    x, y, z = xyz[...,0], xyz[...,1], xyz[...,2]
    
    # 1. Domain Transform
    xp = b_ * x - (b_ - 1) * z
    yp = g_ * y - (g_ - 1) * x
    
    # 2. M1 + Scaling (1e4 is critical for PQ)
    xyz_p = np.stack([xp, yp, z], axis=-1)
    lms = (xyz_p * 10000.0) @ M1.T
    
    # 3. PQ
    lms_p = np.power(np.maximum(lms, 1e-12), n)
    
    # 4. Iz
    iz = (c1 + c2 * lms_p) / (1 + c3 * lms_p)
    
    # 5. M2
    iz_p = np.power(np.maximum(iz, 1e-12), p)
    jab = iz_p @ M2.T
    
    Jz = jab[..., 0] + d
    az = jab[..., 1]
    bz = jab[..., 2]
    return np.stack([Jz, az, bz], axis=-1)

def jzazbz_to_xyz(jab):
    Jz, az, bz = jab[..., 0], jab[..., 1], jab[..., 2]
    
    jab_shifted = np.stack([Jz - d, az, bz], axis=-1)
    iz_p = jab_shifted @ M2_INV.T
    
    iz = np.power(np.maximum(iz_p, 1e-12), 1.0/p)
    
    num = iz - c1
    den = c2 - c3 * iz
    lms_p = np.maximum(num / (den + 1e-12), 0)
    lms = np.power(lms_p, 1.0/n)
    
    # Inverse Scaling (1e-4)
    xyz_p = (lms @ M1_INV.T) / 10000.0
    xp, yp, zp = xyz_p[..., 0], xyz_p[..., 1], xyz_p[..., 2]
    
    x = (xp + (b_ - 1)*zp) / b_
    y = (yp + (g_ - 1)*x) / g_
    
    return np.stack([x, y, zp], axis=-1)

def srgb_to_jzazbz(srgb):
    return xyz_to_jzazbz(srgb_to_linear_np(srgb) @ M_srgb_xyz.T)

def jzazbz_to_srgb_check(jz):
    xyz = jzazbz_to_xyz(jz)
    lin = xyz @ M_xyz_srgb.T
    return linear_to_srgb_np(lin)

# =========================================================================
# 1. Geometry Pipeline
# =========================================================================

def get_hsv_boundary_jz(n=1000):
    """Trace the HSV S=1 V=1 boundary in JzAzBz."""
    h = np.linspace(0, 1, n, endpoint=False)
    i = (h * 6).astype(int); f = h * 6 - i
    q = 1 - f; t = f
    
    rgb = np.zeros((n, 3))
    m = (i % 6 == 0); rgb[m] = np.stack([np.ones(m.sum()), t[m], np.zeros(m.sum())], axis=1)
    m = (i % 6 == 1); rgb[m] = np.stack([q[m], np.ones(m.sum()), np.zeros(m.sum())], axis=1)
    m = (i % 6 == 2); rgb[m] = np.stack([np.zeros(m.sum()), np.ones(m.sum()), t[m]], axis=1)
    m = (i % 6 == 3); rgb[m] = np.stack([np.zeros(m.sum()), q[m], np.ones(m.sum())], axis=1)
    m = (i % 6 == 4); rgb[m] = np.stack([t[m], np.zeros(m.sum()), np.ones(m.sum())], axis=1)
    m = (i % 6 == 5); rgb[m] = np.stack([np.ones(m.sum()), np.zeros(m.sum()), q[m]], axis=1)
    
    return srgb_to_jzazbz(rgb)

def resample_uniform_arc(curve, n_target):
    closed = np.vstack([curve, curve[0]])
    dists = np.linalg.norm(np.diff(closed, axis=0), axis=1)
    cum = np.concatenate([[0.0], np.cumsum(dists)])
    
    t_orig = cum / cum[-1]
    t_new = np.linspace(0, 1, n_target, endpoint=False)
    
    out = np.zeros((n_target, 3))
    for i in range(3):
        data = np.concatenate([curve[:,i], [curve[0,i]]])
        out[:,i] = np.interp(t_new, t_orig, data)
    return out

def gaussian_smooth_loop(curve, sigma):
    """Cyclic Gaussian Smooth."""
    smooth = np.zeros_like(curve)
    for i in range(3):
        smooth[:,i] = gaussian_filter1d(curve[:,i], sigma=sigma, mode='wrap')
    return smooth

def clip_to_gamut(jz_curve):
    """
    If smoothing pushes points outside, pull them back along the chroma vector.
    """
    _, valid = jzazbz_to_srgb_check(jz_curve)
    if np.all(valid): return jz_curve
    
    clipped = jz_curve.copy()
    idxs = np.where(~valid)[0]
    
    for i in idxs:
        J, a, b = jz_curve[i]
        low, high = 0.0, 1.0
        for _ in range(10):
            mid = (low + high) / 2
            pt = np.array([J, a*mid, b*mid])
            xyz = jzazbz_to_xyz(pt)
            lin = xyz @ M_xyz_srgb.T
            if np.all((lin >= -0.0005) & (lin <= 1.0005)):
                low = mid
            else:
                high = mid
        clipped[i, 1] *= low
        clipped[i, 2] *= low
    return clipped

# =========================================================================
# 2. Visualization
# =========================================================================

def generate_vis_row(rgb_ring, flows_np, n_hues):
    y, x = np.ogrid[-1:1:complex(0, 256), -1:1:complex(0, 256)]
    mask = x**2 + y**2 > 1.0
    ang = np.arctan2(y, x)
    idx = (((ang + np.pi) / (2*np.pi)) * len(rgb_ring)).astype(int) % len(rgb_ring)
    disk = rgb_ring[idx].copy()
    disk[mask] = 0
    
    dy, dx = flows_np[0], flows_np[1]
    mag = np.sqrt(dy**2 + dx**2)
    p99 = np.percentile(mag, 99)
    if p99 < 1e-5: p99 = 1.0
    alpha = np.clip(mag / p99, 0, 1)[..., None]
    
    ang_f = np.arctan2(dy, dx)
    def get_c(r, a): 
        ix = (((a + np.pi) / (2*np.pi)) * len(r)).astype(int) % len(r)
        return r[ix]
    
    c = get_c(rgb_ring, ang_f)
    f = c * alpha + 0.1*(1-alpha)
    ow = c * alpha + (1.0 - alpha)
    ob = c * alpha
    
    rot = np.roll(rgb_ring, n_hues//4, axis=0)
    cr = get_c(rot, ang_f)
    rw = cr * alpha + (1.0 - alpha)
    rb = cr * alpha
    return [disk, f, ow, ob, rw, rb]

def get_srgb_wireframe_jz(norm_J):
    edges = [([0,0,0],[1,0,0]), ([1,0,0],[1,1,0]), ([1,1,0],[0,1,0]), ([0,1,0],[0,0,0]),
             ([0,0,1],[1,0,1]), ([1,0,1],[1,1,1]), ([1,1,1],[0,1,1]), ([0,1,1],[0,0,1]),
             ([0,0,0],[0,0,1]), ([1,0,0],[1,0,1]), ([1,1,0],[1,1,1]), ([0,1,0],[0,1,1])]
    x, y, z = [], [], []
    for s, e in edges:
        t = np.linspace(0, 1, 20)
        rgb = np.array(s) + t[:,None]*(np.array(e)-np.array(s))
        jz = srgb_to_jzazbz(rgb)
        x.extend(jz[:,1]); y.extend(jz[:,2]); z.extend(jz[:,0] / norm_J)
        x.append(None); y.append(None); z.append(None)
    return x, y, z

# =========================================================================
# 3. Main
# =========================================================================

def run_gaussian_hsv_final(n_hues=72, blur_sigma=40.0):
    # 1. Trace Truth (High Res)
    jz_raw = get_hsv_boundary_jz(n=720)
    
    # 2. Uniform Resample
    jz_uni = resample_uniform_arc(jz_raw, n_target=720)
    
    # 3. Gaussian Smooth
    jz_smooth = gaussian_smooth_loop(jz_uni, sigma=blur_sigma)
    
    # 4. Clip
    jz_safe = clip_to_gamut(jz_smooth)
    
    # 5. Final Resample
    jz_final = resample_uniform_arc(jz_safe, n_target=n_hues)
    
    # 6. RGB
    rgb_opt, _ = jzazbz_to_srgb_check(jz_final)
    rgb_opt = np.clip(rgb_opt, 0, 1)
    
    # --- Vis ---
    # Calc J_white for normalization
    white_jz = srgb_to_jzazbz(np.array([[[1.0, 1.0, 1.0]]]))[0,0,0]
    
    t = np.linspace(0, 1, n_hues, endpoint=False)
    sr = 0.5 + 0.5*np.sin(2*np.pi*(t+0/3))
    sg = 0.5 + 0.5*np.sin(2*np.pi*(t+1/3))
    sb = 0.5 + 0.5*np.sin(2*np.pi*(t+2/3))
    rgb_sine = np.stack([sr, sg, sb], axis=1)
    
    def get_g(r): return np.argmax(r[:,1] - 0.5*(r[:,0]+r[:,2]))
    g_ref = get_g(rgb_sine)
    rgb_opt_a = np.roll(rgb_opt, g_ref - get_g(rgb_opt), axis=0)
    
    omnidir = Path(omnirefactor.__file__).parent.parent.parent
    masks_path = os.path.join(omnidir, "docs", "_static", "ecoli_labels.tif")
    
    if os.path.exists(masks_path):
        masks = io.imread(masks_path)
        slc = omnirefactor.measure.crop_bbox(masks>0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows_tensor = omnirefactor.core.masks_to_flows(masks, device="cpu")[4]
        flows_np = flows_tensor.cpu().numpy() if hasattr(flows_tensor, 'cpu') else flows_tensor
    else:
        y, x = np.mgrid[-10:10:0.1, -10:10:0.1]
        flows_np = np.stack([np.sin(x/3), np.cos(y/3)]) * 5

    row_s = generate_vis_row(rgb_sine, flows_np, n_hues)
    row_o = generate_vis_row(rgb_opt_a, flows_np, n_hues)
    
    fig, ax = plt.subplots(2, 6, figsize=(24, 8))
    cols = ["Disk", "Flow", "Alpha/W", "Alpha/B", "Rot/W", "Rot/B"]
    data = [row_s, row_o]
    names = ["Sinebow", f"JzAzBz Gaussian (S={blur_sigma})"]
    
    for i in range(2):
        for j in range(6):
            ax[i,j].imshow(data[i][j]); ax[i,j].axis('off')
            if i==0: ax[i,j].set_title(cols[j])
            if j==0: ax[i,j].text(-0.1, 0.5, names[i], transform=ax[i,j].transAxes, va='center', ha='right', rotation=90, fontsize=12)
    plt.tight_layout()
    plt.show()
    
    # --- 3D Plot ---
    jz_sin = srgb_to_jzazbz(rgb_sine)
    jz_plot = srgb_to_jzazbz(rgb_opt_a)
    jz_truth = srgb_to_jzazbz(np.clip(xyz_to_srgb_np(jzazbz_to_xyz(jz_raw))[0],0,1))
    
    fig3d = go.Figure()
    
    # Wireframe
    wx, wy, wz = get_srgb_wireframe_jz(white_jz)
    fig3d.add_trace(go.Scatter3d(x=wx, y=wy, z=wz, mode='lines', name='sRGB Wireframe', line=dict(color='gray', width=3), hoverinfo='skip'))
    
    # Truth
    fig3d.add_trace(go.Scatter3d(x=jz_truth[:,1], y=jz_truth[:,2], z=jz_truth[:,0]/white_jz, mode='markers', name='HSV Truth', marker=dict(size=2, color='white')))
    
    # Sinebow (Ref)
    fig3d.add_trace(go.Scatter3d(x=jz_sin[:,1], y=jz_sin[:,2], z=jz_sin[:,0]/white_jz, mode='lines', name='Sinebow', line=dict(width=4, color='yellow')))
    
    # Result
    fig3d.add_trace(go.Scatter3d(x=jz_plot[:,1], y=jz_plot[:,2], z=jz_plot[:,0]/white_jz, mode='lines', name='Smoothed', line=dict(width=10, color='cyan')))
    
    fig3d.update_layout(template="plotly_dark", height=600, title=f"Corrected JzAzBz (10k nits) Gaussian Smooth (S={blur_sigma})", scene=dict(xaxis_title='az', yaxis_title='bz', zaxis_title='J (Norm)'))
    display(HTML(plot(fig3d, output_type='div', include_plotlyjs='cdn')))

if __name__ == "__main__":
    run_gaussian_hsv_final(72, blur_sigma=40.0)

In [ ]:
import numpy as np
import scipy.optimize as opt
from scipy.spatial import ConvexHull
import plotly.graph_objects as go

# ==========================================
# 1. Validated Jzazbz Transformation
# ==========================================

def srgb_to_xyz(rgb):
    """
    Standard sRGB to XYZ (D65).
    Input: RGB usually 0-1.
    """
    # Inverse Gamma
    mask = rgb > 0.04045
    linear = np.zeros_like(rgb)
    linear[mask] = ((rgb[mask] + 0.055) / 1.055) ** 2.4
    linear[~mask] = rgb[~mask] / 12.92
    
    # sRGB -> XYZ matrix
    M = np.array([
        [0.4124, 0.3576, 0.1805],
        [0.2126, 0.7152, 0.0722],
        [0.0193, 0.1192, 0.9505]
    ])
    return linear @ M.T

def xyz_to_jzazbz(xyz):
    """
    XYZ to Jzazbz implementation (Safdar 2017).
    Correctly handles PQ normalization to avoid overflow.
    """
    # Constants
    b, g = 1.15, 0.66
    c1 = 3424 / 4096
    c2 = 2413 / 128
    c3 = 2392 / 128
    n = 2610 / 16384
    p = 1.7 * 2523 / 32
    d = -0.56
    d0 = 1.6295499532821566e-11

    # 1. XYZ -> LMS
    M1 = np.array([
        [0.674207838, 0.382799340, -0.047570458],
        [0.149284160, 0.739628340, 0.083327300],
        [0.070941080, 0.174768000, 0.670970020]
    ])
    lms = xyz @ M1.T
    
    # 2. PQ Function (Perceptual Quantizer)
    # CRITICAL FIX: Scale to absolute luminance then normalize by 10,000 nits.
    # We assume the sRGB display peak white is 200 nits for a healthy volume.
    display_peak_nits = 200 
    lms_abs = lms * display_peak_nits
    
    # Normalize by PQ ceiling (10,000 nits)
    lms_norm = lms_abs / 10000.0 
    
    # Avoid negative bases for powers
    sign_lms = np.sign(lms_norm)
    abs_lms = np.abs(lms_norm)
    
    # Apply curve
    # ((c1 + c2 * Y^n) / (1 + c3 * Y^n))^p
    num = c1 + c2 * (abs_lms ** n)
    den = 1 + c3 * (abs_lms ** n)
    lms_prime = sign_lms * ((num / den) ** p)
    
    # 3. LMS' -> Izazbz
    M2 = np.array([
        [0.5, 0.5, 0],
        [3.524000, -4.066708, 0.542708],
        [0.199076, 1.096799, -1.295875]
    ])
    izazbz = lms_prime @ M2.T
    
    # 4. Iz -> Jz
    Iz = izazbz[:, 0]
    Jz = ((1 + d) * Iz) / (1 + d * Iz) - d0
    
    return np.column_stack((Jz, izazbz[:, 1], izazbz[:, 2]))

def get_gamut_surface(res=12):
    """Generate points on the surface of the RGB cube."""
    t = np.linspace(0, 1, res)
    g = np.meshgrid(t, t)
    # 6 Faces
    faces = []
    # z=0, z=1
    faces.append(np.stack([g[0], g[1], np.zeros_like(g[0])], -1))
    faces.append(np.stack([g[0], g[1], np.ones_like(g[0])], -1))
    # y=0, y=1
    faces.append(np.stack([g[0], np.zeros_like(g[0]), g[1]], -1))
    faces.append(np.stack([g[0], np.ones_like(g[0]), g[1]], -1))
    # x=0, x=1
    faces.append(np.stack([np.zeros_like(g[0]), g[0], g[1]], -1))
    faces.append(np.stack([np.ones_like(g[0]), g[0], g[1]], -1))
    
    return np.vstack([f.reshape(-1, 3) for f in faces])

# ==========================================
# 2. Optimization (Maximal Inscribed Ellipsoid)
# ==========================================

def fit_ellipsoid(points):
    """
    Fits maximum volume ellipsoid (Lowner-John) inside the Convex Hull of points.
    Uses 'scipy.optimize' with linear inequality constraints.
    """
    # 1. Convex Hull Inequalities: A*x <= b
    hull = ConvexHull(points)
    A = hull.equations[:, :3]
    b_const = -hull.equations[:, 3]

    # SCALING TRICK: Optimization struggles with tiny numbers (Jz ~ 0.01).
    # Scale space up by 100 for calculation, scale back down later.
    scale_factor = 100.0
    A_scaled = A / scale_factor
    # b stays same because A*x <= b => (A/s)*(x*s) <= b
    
    # Variables: Center 'd' (3) and Matrix 'M' (6 for symmetric 3x3)
    # Ellipsoid: x = M*u + d, where ||u|| <= 1
    
    # Initial Guess: Centroid and small sphere
    centroid = np.mean(points, axis=0) * scale_factor
    # Identity matrix (sphere)
    M_init = np.eye(3).flatten() 
    x0 = np.concatenate([centroid, M_init])

    def objective(x):
        # Maximize log(det(M)) -> Minimize -log(det(M))
        M = x[3:].reshape(3, 3)
        # Symmetrize roughly (though we just use full matrix for simplicity in scipy)
        sign, val = np.linalg.slogdet(M)
        if sign <= 0: return 1e6 # Barrier
        return -val

    def constraints(x):
        d = x[:3]
        M = x[3:].reshape(3, 3)
        # Constraint: || A_scaled @ M @ u || + A_scaled @ d <= b
        # Max value of LHS over unit u is || M.T @ A_scaled.T ||
        
        # A_scaled: (N, 3)
        # M: (3, 3)
        # Norm term: row-wise norm of (A @ M)
        
        # (N, 3) @ (3, 3) -> (N, 3)
        AM = A_scaled @ M 
        norm_AM = np.linalg.norm(AM, axis=1)
        
        # Linear term
        Ad = A_scaled @ d
        
        return b_const - (norm_AM + Ad)

    # Optimization
    # Tolerance must be tight.
    res = opt.minimize(
        objective, x0, 
        method='SLSQP', 
        constraints={'type': 'ineq', 'fun': constraints},
        options={'maxiter': 200, 'ftol': 1e-6}
    )

    # Unpack and rescale back
    d_opt = res.x[:3] / scale_factor
    M_opt = res.x[3:].reshape(3, 3) / scale_factor
    
    return d_opt, M_opt

# ==========================================
# 3. Execution & Visualization
# ==========================================

# A. Data Gen
rgb = get_gamut_surface(res=15)
xyz = srgb_to_xyz(rgb)
jz = xyz_to_jzazbz(xyz)

# B. Fit Ellipsoid
center, axes_matrix = fit_ellipsoid(jz)

# C. Create Mesh for Ellipsoid
u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
x_sphere = np.cos(u)*np.sin(v)
y_sphere = np.sin(u)*np.sin(v)
z_sphere = np.cos(v)
sphere_pts = np.stack([x_sphere, y_sphere, z_sphere], axis=-1) # (40, 20, 3)
sphere_flat = sphere_pts.reshape(-1, 3)

# Transform sphere to ellipsoid: E = M*u + d
ell_flat = sphere_flat @ axes_matrix.T + center
ex = ell_flat[:, 0].reshape(x_sphere.shape)
ey = ell_flat[:, 1].reshape(y_sphere.shape)
ez = ell_flat[:, 2].reshape(z_sphere.shape)

# D. Plot
fig = go.Figure()

# Hull Mesh
hull = ConvexHull(jz)
fig.add_trace(go.Mesh3d(
    x=jz[:, 0], y=jz[:, 1], z=jz[:, 2],
    i=hull.simplices[:, 0], j=hull.simplices[:, 1], k=hull.simplices[:, 2],
    color='gray', opacity=0.15, name='sRGB Gamut'
))

# Ellipsoid Surface
fig.add_trace(go.Surface(
    x=ex, y=ey, z=ez,
    colorscale='Plasma', opacity=0.8, name='Max Inscribed Ellipsoid',
    showscale=False
))

# Markers for Center
fig.add_trace(go.Scatter3d(
    x=[center[0]], y=[center[1]], z=[center[2]],
    mode='markers', marker=dict(size=4, color='red'), name='Center'
))

fig.update_layout(
    title='sRGB Gamut (Jzazbz) with Maximal Inscribed Ellipsoid',
    scene=dict(
        xaxis_title='Jz (Lightness)',
        yaxis_title='az (Red-Green)',
        zaxis_title='bz (Blue-Yellow)',
        aspectmode='data' # Ensures true perceptual proportions
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()

In [ ]:
import numpy as np
import scipy.optimize as opt
from scipy.spatial import ConvexHull
import plotly.graph_objects as go

# ==========================================
# 1. Jzazbz Transformation
# ==========================================
def srgb_to_xyz(rgb):
    mask = rgb > 0.04045
    linear = np.zeros_like(rgb)
    linear[mask] = ((rgb[mask] + 0.055) / 1.055) ** 2.4
    linear[~mask] = rgb[~mask] / 12.92
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    return linear @ M.T

def xyz_to_jzazbz(xyz):
    b, g, d, d0 = 1.15, 0.66, -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    
    # 1. XYZ -> LMS
    lms = xyz @ M1.T
    
    # 2. PQ Function (Absolute Luminance)
    # We map sRGB White (1.0) to 200 nits.
    display_peak_nits = 200 
    lms_norm = (lms * display_peak_nits) / 10000.0 
    
    lms_prime = np.sign(lms_norm) * (((c1 + c2 * (np.abs(lms_norm) ** n)) / (1 + c3 * (np.abs(lms_norm) ** n))) ** p)
    
    # 3. LMS -> Izazbz
    izazbz = lms_prime @ M2.T
    
    # 4. Iz -> Jz
    Jz = ((1 + d) * izazbz[:, 0]) / (1 + d * izazbz[:, 0]) - d0
    
    return np.column_stack((Jz, izazbz[:, 1], izazbz[:, 2]))

def get_gamut_surface(res=25):
    t = np.linspace(0, 1, res)
    g = np.meshgrid(t, t)
    faces = [
        np.stack([g[0], g[1], np.zeros_like(g[0])], -1), np.stack([g[0], g[1], np.ones_like(g[0])], -1),
        np.stack([g[0], np.zeros_like(g[0]), g[1]], -1), np.stack([g[0], np.ones_like(g[0]), g[1]], -1),
        np.stack([np.zeros_like(g[0]), g[0], g[1]], -1), np.stack([np.ones_like(g[0]), g[0], g[1]], -1)
    ]
    return np.vstack([f.reshape(-1, 3) for f in faces])

# ==========================================
# 2. Optimization
# ==========================================
def fit_ellipsoid(points):
    hull = ConvexHull(points)
    A = hull.equations[:, :3]
    b_const = -hull.equations[:, 3]
    
    # Upscale for numerical stability during optimization
    scale = 100.0
    A_scaled = A / scale
    centroid = np.mean(points, axis=0) * scale
    x0 = np.concatenate([centroid, np.eye(3).flatten() * 0.1])

    def objective(x):
        M = x[3:].reshape(3, 3)
        sign, val = np.linalg.slogdet(M)
        return -val if sign > 0 else 1e9

    def constraints(x):
        d, M = x[:3], x[3:].reshape(3, 3)
        return b_const - (np.linalg.norm(A_scaled @ M, axis=1) + A_scaled @ d)

    print("Optimizing Maximal Inscribed Ellipsoid...")
    res = opt.minimize(objective, x0, method='SLSQP', constraints={'type': 'ineq', 'fun': constraints}, options={'maxiter': 500})
    
    return res.x[:3]/scale, res.x[3:].reshape(3, 3)/scale, hull

def calculate_shadow_boundary(center, M, view_vector):
    # Solve for normal where n dot v = 0 -> u dot (M^-1 v) = 0
    w = np.linalg.solve(M, view_vector)
    w = w / np.linalg.norm(w)
    
    # Basis perpendicular to w
    tmp = np.array([0.0, 1.0, 0.0]) if np.abs(w[1]) < 0.9 else np.array([0.0, 0.0, 1.0])
    s1 = np.cross(w, tmp); s1 /= np.linalg.norm(s1)
    s2 = np.cross(w, s1)
    
    # Circle
    theta = np.linspace(0, 2*np.pi, 200)
    u_circle = np.outer(np.cos(theta), s1) + np.outer(np.sin(theta), s2)
    return center + u_circle @ M.T

# ==========================================
# 3. Visualization
# ==========================================

# A. Generate Raw Data
print("Generating Gamut...")
# Note: Peak white is at index -1 usually, or we can find max Jz
raw_points = xyz_to_jzazbz(srgb_to_xyz(get_gamut_surface(25)))

# B. Fit Ellipsoid (In the correct physical space)
center, M, hull = fit_ellipsoid(raw_points)

# C. Calculate Geometry (In the correct physical space)
silhouette = calculate_shadow_boundary(center, M, view_vector=np.array([1.0, 0.0, 0.0])) # View down Jz
sil_closed = np.vstack([silhouette, silhouette[0]])

# Generate Ellipsoid Mesh
u, v = np.mgrid[0:2*np.pi:60j, 0:np.pi:30j]
sph = np.stack([np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v)], axis=-1).reshape(-1, 3)
ell = sph @ M.T + center

# D. NORMALIZE FOR PLOTTING
# We find the Peak White Jz value (approx 0.22) and scale Jz axis by 1/Peak
max_jz = np.max(raw_points[:, 0])
scale_vec = np.array([1.0/max_jz, 1.0, 1.0]) # Scale Jz, keep az/bz absolute

# Apply scaling to all geometric entities
points_norm = raw_points * scale_vec
ell_norm = ell * scale_vec
sil_norm = sil_closed * scale_vec
center_norm = center * scale_vec

# Helper to map to X, Y, Z for Plotly (Jz=Z, az=X, bz=Y)
def map_coords(arr):
    return arr[:, 1], arr[:, 2], arr[:, 0]

x_hull, y_hull, z_hull = map_coords(points_norm)
x_ell, y_ell, z_ell = map_coords(ell_norm)
x_sil, y_sil, z_sil = map_coords(sil_norm)
cx, cy, cz = center_norm[1], center_norm[2], center_norm[0]

fig = go.Figure()

# 1. Gamut Hull
fig.add_trace(go.Mesh3d(
    x=x_hull, y=y_hull, z=z_hull,
    i=hull.simplices[:,0], j=hull.simplices[:,1], k=hull.simplices[:,2],
    color='gray', opacity=0.1, name='sRGB Gamut'
))

# 2. Ellipsoid
fig.add_trace(go.Surface(
    x=x_ell.reshape(u.shape), y=y_ell.reshape(u.shape), z=z_ell.reshape(u.shape),
    colorscale='Viridis', opacity=0.85, showscale=False, name='Ellipsoid'
))

# 3. Silhouette Curve
fig.add_trace(go.Scatter3d(
    x=x_sil, y=y_sil, z=z_sil,
    mode='lines', line=dict(color='magenta', width=10),
    name='Widest Profile (ab plane)'
))

# 4. Center
fig.add_trace(go.Scatter3d(
    x=[cx], y=[cy], z=[cz],
    mode='markers', marker=dict(color='red', size=5), name='Center'
))

fig.update_layout(
    template="plotly_dark",
    title="sRGB in Jzazbz (Normalized Jz 0-1)",
    scene=dict(
        xaxis_title='az (Red-Green)',
        yaxis_title='bz (Blue-Yellow)',
        zaxis_title='Jz (Lightness)',
        aspectmode='manual',
        # We manually stretch Z to look nice, since we distorted the physical space
        aspectratio=dict(x=1, y=1, z=1), 
        camera=dict(eye=dict(x=1.6, y=1.6, z=0.6))
    )
)

fig.show()

In [ ]:
import numpy as np
import scipy.optimize as opt
from scipy.spatial import ConvexHull
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==========================================
# 1. Forward Color Transformation (RGB -> Jzazbz)
# ==========================================
def srgb_to_xyz(rgb):
    mask = rgb > 0.04045
    linear = np.zeros_like(rgb)
    linear[mask] = ((rgb[mask] + 0.055) / 1.055) ** 2.4
    linear[~mask] = rgb[~mask] / 12.92
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    return linear @ M.T

def xyz_to_jzazbz(xyz):
    b, g, d, d0 = 1.15, 0.66, -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    
    lms = xyz @ M1.T
    lms_norm = (lms * 200) / 10000.0 # Normalize to 200 nits peak
    lms_prime = np.sign(lms_norm) * (((c1 + c2 * (np.abs(lms_norm) ** n)) / (1 + c3 * (np.abs(lms_norm) ** n))) ** p)
    izazbz = lms_prime @ M2.T
    Jz = ((1 + d) * izazbz[:, 0]) / (1 + d * izazbz[:, 0]) - d0
    return np.column_stack((Jz, izazbz[:, 1], izazbz[:, 2]))

# ==========================================
# 2. Inverse Color Transformation (Jzazbz -> RGB)
#    Needed to color the dots on the hoop
# ==========================================
def jzazbz_to_xyz(jzazbz):
    Jz, az, bz = jzazbz[:,0], jzazbz[:,1], jzazbz[:,2]
    d, d0 = -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])

    # 1. Jz -> Iz
    Jp = Jz + d0
    Iz = Jp / (1 + d - d * Jp)

    # 2. Izazbz -> LMS'
    izazbz_vec = np.stack([Iz, az, bz], axis=1)
    M2_inv = np.linalg.inv(M2)
    lms_prime = izazbz_vec @ M2_inv.T

    # 3. LMS' -> LMS_norm (Inverse PQ)
    sign_lms_prime = np.sign(lms_prime)
    Y = np.abs(lms_prime) ** (1/p)
    # Numerical stability check for division
    num = Y - c1
    den = c2 - Y * c3
    # Clip negative results arising from numerical noise near zero
    An = np.clip(num / den, 0, None) 
    A = An ** (1/n)
    lms_norm = sign_lms_prime * A

    # 4. Denormalize LMS and -> XYZ
    lms = (lms_norm * 10000.0) / 200.0
    M1_inv = np.linalg.inv(M1)
    xyz = lms @ M1_inv.T
    return xyz

def xyz_to_srgb(xyz):
    M_inv = np.linalg.inv(np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]]))
    linear = xyz @ M_inv.T
    # Gamma Compression
    mask = linear > 0.0031308
    srgb = np.zeros_like(linear)
    srgb[mask] = 1.055 * (linear[mask] ** (1/2.4)) - 0.055
    srgb[~mask] = 12.92 * linear[~mask]
    # Clip to ensure valid RGB range
    return np.clip(srgb, 0, 1)

# ==========================================
# 3. Geometry & Optimization
# ==========================================
def get_gamut_surface(res=25):
    t = np.linspace(0, 1, res)
    g = np.meshgrid(t, t)
    faces = [np.stack([g[0], g[1], np.zeros_like(g[0])], -1), np.stack([g[0], g[1], np.ones_like(g[0])], -1),
             np.stack([g[0], np.zeros_like(g[0]), g[1]], -1), np.stack([g[0], np.ones_like(g[0]), g[1]], -1),
             np.stack([np.zeros_like(g[0]), g[0], g[1]], -1), np.stack([np.ones_like(g[0]), g[0], g[1]], -1)]
    return np.vstack([f.reshape(-1, 3) for f in faces])

def fit_ellipsoid(points):
    hull = ConvexHull(points)
    A = hull.equations[:, :3]; b_const = -hull.equations[:, 3]
    scale = 100.0
    A_scaled = A / scale
    x0 = np.concatenate([np.mean(points, axis=0) * scale, np.eye(3).flatten() * 0.1])

    def objective(x):
        M = x[3:].reshape(3, 3)
        sign, val = np.linalg.slogdet(M)
        return -val if sign > 0 else 1e9

    def constraints(x):
        d, M = x[:3], x[3:].reshape(3, 3)
        return b_const - (np.linalg.norm(A_scaled @ M, axis=1) + A_scaled @ d)

    print("Optimizing Ellipsoid...")
    res = opt.minimize(objective, x0, method='SLSQP', constraints={'type': 'ineq', 'fun': constraints}, options={'maxiter': 500})
    return res.x[:3]/scale, res.x[3:].reshape(3, 3)/scale, hull

def calculate_shadow_boundary(center, M, view_vector, res=200):
    w = np.linalg.solve(M, view_vector)
    w = w / np.linalg.norm(w)
    tmp = np.array([0.0, 1.0, 0.0]) if np.abs(w[1]) < 0.9 else np.array([0.0, 0.0, 1.0])
    s1 = np.cross(w, tmp); s1 /= np.linalg.norm(s1)
    s2 = np.cross(w, s1)
    theta = np.linspace(0, 2*np.pi, res)
    return center + (np.outer(np.cos(theta), s1) + np.outer(np.sin(theta), s2)) @ M.T

# ==========================================
# 4. Execution & Plotting
# ==========================================
# A. Generate Data & Fit
raw_points = xyz_to_jzazbz(srgb_to_xyz(get_gamut_surface(25)))
center, M, hull = fit_ellipsoid(raw_points)

# B. Calculate Silhouette and its Colors
# Use slightly fewer points for dots so they are distinct
silhouette = calculate_shadow_boundary(center, M, view_vector=np.array([1.0, 0.0, 0.0]), res=100)
sil_rgb = xyz_to_srgb(jzazbz_to_xyz(silhouette))
sil_colors = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in sil_rgb]

# C. Ellipsoid Mesh
u, v = np.mgrid[0:2*np.pi:60j, 0:np.pi:30j]
sph = np.stack([np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v)], axis=-1).reshape(-1, 3)
ell = sph @ M.T + center

# D. Normalization for 3D Plotting (Jz 0-1)
max_jz = np.max(raw_points[:, 0])
scale_vec = np.array([1.0/max_jz, 1.0, 1.0])
points_norm = raw_points * scale_vec
ell_norm = ell * scale_vec
sil_norm = silhouette * scale_vec
center_norm = center * scale_vec

# Map for Plotly (Jz->Z, az->X, bz->Y)
def map_coords(arr): return arr[:, 1], arr[:, 2], arr[:, 0]
x_hull, y_hull, z_hull = map_coords(points_norm)
x_ell, y_ell, z_ell = map_coords(ell_norm)
x_sil, y_sil, z_sil = map_coords(sil_norm)

# --- Figure 1: 3D View ---
fig3d = go.Figure()
# Gamut Hull
fig3d.add_trace(go.Mesh3d(x=x_hull, y=y_hull, z=z_hull, i=hull.simplices[:,0], j=hull.simplices[:,1], k=hull.simplices[:,2],
                        color='white', opacity=0.05, name='sRGB Gamut'))
# Gray Ellipsoid
fig3d.add_trace(go.Surface(x=x_ell.reshape(u.shape), y=y_ell.reshape(u.shape), z=z_ell.reshape(u.shape),
                         colorscale=[(0, '#aaaaaa'), (1, '#aaaaaa')], opacity=0.8, showscale=False, name='Ellipsoid'))
# Colored Silhouette Dots
fig3d.add_trace(go.Scatter3d(x=x_sil, y=y_sil, z=z_sil, mode='markers',
                           marker=dict(color=sil_colors, size=6, line=dict(width=1, color='rgb(50,50,50)')),
                           name='Widest Profile Colors'))
fig3d.update_layout(template="plotly_dark", title="3D: sRGB in Jzazbz (Gray Ellipsoid, Colored Profile)",
                  scene=dict(xaxis_title='az', yaxis_title='bz', zaxis_title='Jz (Normalized)',
                             aspectmode='manual', aspectratio=dict(x=1, y=1, z=1), camera=dict(eye=dict(x=1.6, y=1.6, z=0.8))))
fig3d.show()

# --- Figure 2: 2D Hue Disk ---
fig2d = go.Figure()
fig2d.add_trace(go.Scatter(
    x=silhouette[:, 1], # az (raw units)
    y=silhouette[:, 2], # bz (raw units)
    mode='markers',
    marker=dict(color=sil_colors, size=12, line=dict(width=1, color='rgb(50,50,50)')),
    text=[f"az: {a:.3f}, bz: {b:.3f}" for a,b in zip(silhouette[:,1], silhouette[:,2])],
    name='Hue Hoop'
))
# Add center marker
fig2d.add_trace(go.Scatter(x=[center[1]], y=[center[2]], mode='markers', marker=dict(color='white', size=8, symbol='cross'), name='Center'))

# Force square aspect ratio for correct circle representation
range_max = max(np.max(np.abs(silhouette[:,1])), np.max(np.abs(silhouette[:,2]))) * 1.1
fig2d.update_layout(
    template="plotly_dark", title="2D: ab Plane Projection (Hue Disk)",
    xaxis=dict(title='az (Red-Green)', range=[-range_max, range_max], constrain='domain', gridcolor='rgb(50,50,50)'),
    yaxis=dict(title='bz (Blue-Yellow)', range=[-range_max, range_max], scaleanchor='x', scaleratio=1, gridcolor='rgb(50,50,50)'),
    width=700, height=700 # Ensure figure is square
)
fig2d.show()

In [ ]:
import numpy as np
import scipy.optimize as opt
from scipy.spatial import ConvexHull
import plotly.graph_objects as go
import matplotlib.pyplot as plt

# ==========================================
# 1. Color Transformation Logic (Jzazbz)
# ==========================================
def srgb_to_xyz(rgb):
    mask = rgb > 0.04045
    linear = np.zeros_like(rgb)
    linear[mask] = ((rgb[mask] + 0.055) / 1.055) ** 2.4
    linear[~mask] = rgb[~mask] / 12.92
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    return linear @ M.T

def xyz_to_jzazbz(xyz):
    b, g, d, d0 = 1.15, 0.66, -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    
    lms = xyz @ M1.T
    lms_norm = (lms * 200) / 10000.0 # Normalize 200 nit peak to 10k ceiling
    lms_prime = np.sign(lms_norm) * (((c1 + c2 * (np.abs(lms_norm) ** n)) / (1 + c3 * (np.abs(lms_norm) ** n))) ** p)
    izazbz = lms_prime @ M2.T
    Jz = ((1 + d) * izazbz[:, 0]) / (1 + d * izazbz[:, 0]) - d0
    return np.column_stack((Jz, izazbz[:, 1], izazbz[:, 2]))

def jzazbz_to_xyz(jzazbz):
    Jz, az, bz = jzazbz[:,0], jzazbz[:,1], jzazbz[:,2]
    d, d0 = -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])

    Jp = Jz + d0
    Iz = Jp / (1 + d - d * Jp)
    izazbz_vec = np.stack([Iz, az, bz], axis=1)
    lms_prime = izazbz_vec @ np.linalg.inv(M2).T
    
    sign_lms = np.sign(lms_prime)
    Y = np.abs(lms_prime) ** (1/p)
    num = Y - c1
    den = c2 - Y * c3
    An = np.clip(num / den, 0, None) 
    lms_norm = sign_lms * (An ** (1/n))
    
    lms = (lms_norm * 10000.0) / 200.0
    return lms @ np.linalg.inv(M1).T

def xyz_to_srgb(xyz):
    M_inv = np.linalg.inv(np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]]))
    linear = xyz @ M_inv.T
    mask = linear > 0.0031308
    srgb = np.zeros_like(linear)
    srgb[mask] = 1.055 * (linear[mask] ** (1/2.4)) - 0.055
    srgb[~mask] = 12.92 * linear[~mask]
    return np.clip(srgb, 0, 1)

# ==========================================
# 2. Geometry & Optimization
# ==========================================
def get_gamut_edges_and_surface(res=16):
    # 1. Surface (for Hull reference)
    t = np.linspace(0, 1, res)
    g = np.meshgrid(t, t)
    faces = [
        np.stack([g[0], g[1], np.zeros_like(g[0])], -1), np.stack([g[0], g[1], np.ones_like(g[0])], -1),
        np.stack([g[0], np.zeros_like(g[0]), g[1]], -1), np.stack([g[0], np.ones_like(g[0]), g[1]], -1),
        np.stack([np.zeros_like(g[0]), g[0], g[1]], -1), np.stack([np.ones_like(g[0]), g[0], g[1]], -1)
    ]
    surface_pts = np.vstack([f.reshape(-1, 3) for f in faces])

    # 2. Explicit Edges (The 12 lines of the cube)
    # Define corners
    corners = [[0,0,0], [1,0,0], [1,1,0], [0,1,0], [0,0,1], [1,0,1], [1,1,1], [0,1,1]]
    lines_idx = [[0,1], [1,2], [2,3], [3,0], [4,5], [5,6], [6,7], [7,4], [0,4], [1,5], [2,6], [3,7]]
    
    edges_list = []
    line_t = np.linspace(0, 1, 50)
    for (start, end) in lines_idx:
        p_start = np.array(corners[start])
        p_end = np.array(corners[end])
        # Interpolate
        line_pts = p_start[None, :] * (1 - line_t[:, None]) + p_end[None, :] * line_t[:, None]
        edges_list.append(line_pts)
        
    return surface_pts, edges_list

def fit_ellipsoid(points):
    hull = ConvexHull(points)
    A = hull.equations[:, :3]; b_const = -hull.equations[:, 3]
    scale = 100.0; A_scaled = A / scale
    x0 = np.concatenate([np.mean(points, axis=0) * scale, np.eye(3).flatten() * 0.1])

    def objective(x):
        M = x[3:].reshape(3, 3); sign, val = np.linalg.slogdet(M)
        return -val if sign > 0 else 1e9
    def constraints(x):
        d, M = x[:3], x[3:].reshape(3, 3)
        return b_const - (np.linalg.norm(A_scaled @ M, axis=1) + A_scaled @ d)

    print("Optimizing Ellipsoid...")
    res = opt.minimize(objective, x0, method='SLSQP', constraints={'type': 'ineq', 'fun': constraints}, options={'maxiter': 500})
    return res.x[:3]/scale, res.x[3:].reshape(3, 3)/scale, hull

def calculate_shadow_boundary(center, M, view_vector, res=256):
    w = np.linalg.solve(M, view_vector); w = w / np.linalg.norm(w)
    tmp = np.array([0.0, 1.0, 0.0]) if np.abs(w[1]) < 0.9 else np.array([0.0, 0.0, 1.0])
    s1 = np.cross(w, tmp); s1 /= np.linalg.norm(s1); s2 = np.cross(w, s1)
    theta = np.linspace(0, 2*np.pi, res)
    return center + (np.outer(np.cos(theta), s1) + np.outer(np.sin(theta), s2)) @ M.T

# ==========================================
# 3. Main Data Generation
# ==========================================
print("Generating Geometry...")
surf_rgb, edges_rgb_list = get_gamut_edges_and_surface(res=12)
surf_jz = xyz_to_jzazbz(srgb_to_xyz(surf_rgb))
edges_jz_list = [xyz_to_jzazbz(srgb_to_xyz(e)) for e in edges_rgb_list]

center, M, hull = fit_ellipsoid(surf_jz)
silhouette_jz = calculate_shadow_boundary(center, M, np.array([1.0, 0.0, 0.0]), res=256)
silhouette_rgb = xyz_to_srgb(jzazbz_to_xyz(silhouette_jz))

# Scale normalize Jz for 3D plotting
max_jz = np.max(surf_jz[:, 0])
scale_vec = np.array([1.0/max_jz, 1.0, 1.0])

# ==========================================
# 4. Visualization 1: 3D Plotly (Opaque Edges)
# ==========================================
def map_coords(arr): return arr[:, 1], arr[:, 2], arr[:, 0]

fig3d = go.Figure()

# A. Gamut Faces (Transparent)
x_h, y_h, z_h = map_coords(surf_jz * scale_vec)
fig3d.add_trace(go.Mesh3d(x=x_h, y=y_h, z=z_h, i=hull.simplices[:,0], j=hull.simplices[:,1], k=hull.simplices[:,2],
                          color='gray', opacity=0.08, name='Gamut Faces', hoverinfo='skip'))

# B. Gamut Edges (Opaque Lines)
for edge in edges_jz_list:
    xe, ye, ze = map_coords(edge * scale_vec)
    fig3d.add_trace(go.Scatter3d(x=xe, y=ye, z=ze, mode='lines', 
                                 line=dict(color='white', width=4), hoverinfo='skip', showlegend=False))
# Add one dummy trace for legend
fig3d.add_trace(go.Scatter3d(x=[None], y=[None], z=[None], mode='lines', 
                             line=dict(color='white', width=4), name='Gamut Edges'))

# C. Ellipsoid (Gray)
u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
sph = np.stack([np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v)], axis=-1).reshape(-1, 3)
ell = (sph @ M.T + center) * scale_vec
ex, ey, ez = map_coords(ell)
fig3d.add_trace(go.Surface(x=ex.reshape(u.shape), y=ey.reshape(u.shape), z=ez.reshape(u.shape),
                         colorscale=[(0, '#888888'), (1, '#888888')], opacity=0.5, showscale=False, name='Ellipsoid'))

# D. Silhouette Loop (Colored Dots)
xs, ys, zs = map_coords(silhouette_jz * scale_vec)
sil_hex = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in silhouette_rgb]
fig3d.add_trace(go.Scatter3d(x=xs, y=ys, z=zs, mode='markers',
                           marker=dict(color=sil_hex, size=5, line=dict(width=0)), name='Ellipsoid Loop'))

fig3d.update_layout(template="plotly_dark", title="3D View: Opaque Edges & Inscribed Ellipsoid",
                  scene=dict(xaxis_title='az', yaxis_title='bz', zaxis_title='Jz (Norm)',
                             aspectmode='manual', aspectratio=dict(x=1, y=1, z=1), 
                             camera=dict(eye=dict(x=1.6, y=1.6, z=0.8))))
fig3d.show()

# ==========================================
# 5. Visualization 2: Matplotlib Comparison
#    (Sinebow vs Ellipsoid Rainbow)
# ==========================================

def compare_sinebow_vs_ellipsoid():
    n_hues = 256
    
    # 1. Generate Sinebow
    t = np.linspace(0, 1, n_hues)
    sr = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0 / 3))
    sg = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1 / 3))
    sb = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2 / 3))
    rgb_sine = np.stack([sr, sg, sb], axis=1)
    # Get Jzazbz coords for plotting sinebow on ab plane
    jz_sine = xyz_to_jzazbz(srgb_to_xyz(rgb_sine))

    # 2. Ellipsoid Loop (Already calc, just ensuring size matches)
    # Re-calc to ensure exactly n_hues and cyclic
    # Note: calculate_shadow_boundary is strictly geometric. 
    # To match hue orientation of sinebow, we'd typically rotate, but raw comparison is safer.
    ell_jz = calculate_shadow_boundary(center, M, np.array([1.0,0.0,0.0]), res=n_hues)
    rgb_ell = xyz_to_srgb(jzazbz_to_xyz(ell_jz))

    # 3. Plotting (Matplotlib - Styled like user request)
    fig, ax = plt.subplots(2, 2, figsize=(12, 10), facecolor='#111111')
    
    # Setup styles
    for a_row in ax:
        for a in a_row:
            a.set_facecolor('#222222')
            a.tick_params(colors='white')
            for spine in a.spines.values(): spine.set_edgecolor('#555555')
            a.xaxis.label.set_color('white')
            a.yaxis.label.set_color('white')
            a.title.set_color('white')

    # A. Hue Disks (ab plane)
    # Sinebow Disk
    ax[0,0].scatter(jz_sine[:,1], jz_sine[:,2], c=rgb_sine, s=20)
    ax[0,0].set_title("Sinebow (Jzazbz Projection)")
    ax[0,0].set_xlabel("az")
    ax[0,0].set_ylabel("bz")
    ax[0,0].axis('equal')

    # Ellipsoid Disk
    ax[0,1].scatter(ell_jz[:,1], ell_jz[:,2], c=rgb_ell, s=20)
    ax[0,1].set_title("Ellipsoid Max Profile (Jzazbz Projection)")
    ax[0,1].set_xlabel("az")
    ax[0,1].set_ylabel("bz")
    ax[0,1].axis('equal')
    
    # Force same scale
    max_range = max(np.max(np.abs(jz_sine[:,1:])), np.max(np.abs(ell_jz[:,1:]))) * 1.1
    ax[0,0].set_xlim(-max_range, max_range); ax[0,0].set_ylim(-max_range, max_range)
    ax[0,1].set_xlim(-max_range, max_range); ax[0,1].set_ylim(-max_range, max_range)

    # B. Color Strips
    # Helper to plot strip
    def plot_strip(axis, rgb_data, title):
        img = rgb_data[np.newaxis, :, :] # (1, N, 3)
        axis.imshow(img, aspect='auto')
        axis.set_title(title)
        axis.set_yticks([])
        axis.set_xlabel("Index")

    plot_strip(ax[1,0], rgb_sine, "Sinebow Gradient")
    plot_strip(ax[1,1], rgb_ell, "Ellipsoid Profile Gradient")

    plt.tight_layout()
    plt.show()

compare_sinebow_vs_ellipsoid()

In [ ]:
import numpy as np
import scipy.optimize as opt
from scipy.spatial import ConvexHull
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pathlib import Path

# Omnipose / Imaging Imports
import torch
import omnirefactor.core
from skimage import io
import fastremap

# ==========================================
# 1. Color Math & Conversions
# ==========================================

def srgb_to_xyz(rgb):
    mask = rgb > 0.04045
    linear = np.zeros_like(rgb)
    linear[mask] = ((rgb[mask] + 0.055) / 1.055) ** 2.4
    linear[~mask] = rgb[~mask] / 12.92
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    return linear @ M.T

def xyz_to_jzazbz(xyz):
    # Safdar 2017 Constants
    b, g, d, d0 = 1.15, 0.66, -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    
    lms = xyz @ M1.T
    lms_norm = (lms * 200) / 10000.0 
    lms_prime = np.sign(lms_norm) * (((c1 + c2 * (np.abs(lms_norm) ** n)) / (1 + c3 * (np.abs(lms_norm) ** n))) ** p)
    izazbz = lms_prime @ M2.T
    Jz = ((1 + d) * izazbz[:, 0]) / (1 + d * izazbz[:, 0]) - d0
    return np.column_stack((Jz, izazbz[:, 1], izazbz[:, 2]))

def jzazbz_to_xyz(jzazbz):
    Jz, az, bz = jzazbz[:,0], jzazbz[:,1], jzazbz[:,2]
    d, d0 = -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])

    Jp = Jz + d0; Iz = Jp / (1 + d - d * Jp)
    izazbz_vec = np.stack([Iz, az, bz], axis=1)
    lms_prime = izazbz_vec @ np.linalg.inv(M2).T
    sign_lms = np.sign(lms_prime)
    Y = np.abs(lms_prime) ** (1/p)
    num = Y - c1; den = c2 - Y * c3
    An = np.clip(num / den, 0, None) 
    lms_norm = sign_lms * (An ** (1/n))
    lms = (lms_norm * 10000.0) / 200.0
    return lms @ np.linalg.inv(M1).T

def xyz_to_srgb(xyz):
    M_inv = np.linalg.inv(np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]]))
    linear = xyz @ M_inv.T
    return linear_to_srgb_np(linear)

def srgb_to_linear_np(srgb):
    return np.where(srgb <= 0.04045, srgb / 12.92, ((srgb + 0.055) / 1.055) ** 2.4)

def linear_to_srgb_np(lin):
    s = np.zeros_like(lin)
    mask = lin > 0.0031308
    s[mask] = 1.055 * (lin[mask]**(1/2.4)) - 0.055
    s[~mask] = 12.92 * lin[~mask]
    return np.clip(s, 0, 1)

# ==========================================
# 2. Optimization & Geometry
# ==========================================

def get_gamut_surface(res=16):
    t = np.linspace(0, 1, res)
    g = np.meshgrid(t, t)
    faces = [np.stack([g[0], g[1], np.zeros_like(g[0])], -1), np.stack([g[0], g[1], np.ones_like(g[0])], -1),
             np.stack([g[0], np.zeros_like(g[0]), g[1]], -1), np.stack([g[0], np.ones_like(g[0]), g[1]], -1),
             np.stack([np.zeros_like(g[0]), g[0], g[1]], -1), np.stack([np.ones_like(g[0]), g[0], g[1]], -1)]
    return np.vstack([f.reshape(-1, 3) for f in faces])

def fit_ellipsoid(points):
    hull = ConvexHull(points)
    A = hull.equations[:, :3]; b_const = -hull.equations[:, 3]
    scale = 100.0; A_scaled = A / scale
    x0 = np.concatenate([np.mean(points, axis=0) * scale, np.eye(3).flatten() * 0.1])
    def objective(x):
        M = x[3:].reshape(3, 3); sign, val = np.linalg.slogdet(M)
        return -val if sign > 0 else 1e9
    def constraints(x):
        d, M = x[:3], x[3:].reshape(3, 3)
        return b_const - (np.linalg.norm(A_scaled @ M, axis=1) + A_scaled @ d)
    print("Optimizing Maximal Inscribed Ellipsoid...")
    res = opt.minimize(objective, x0, method='SLSQP', constraints={'type': 'ineq', 'fun': constraints}, options={'maxiter': 500})
    return res.x[:3]/scale, res.x[3:].reshape(3, 3)/scale, hull

def calculate_shadow_boundary(center, M, view_vector, res=256):
    w = np.linalg.solve(M, view_vector); w = w / np.linalg.norm(w)
    tmp = np.array([0.0, 1.0, 0.0]) if np.abs(w[1]) < 0.9 else np.array([0.0, 0.0, 1.0])
    s1 = np.cross(w, tmp); s1 /= np.linalg.norm(s1); s2 = np.cross(w, s1)
    theta = np.linspace(0, 2*np.pi, res)
    return center + (np.outer(np.cos(theta), s1) + np.outer(np.sin(theta), s2)) @ M.T

# ==========================================
# 3. Omnipose Helpers
# ==========================================

def build_hue_disk_from_ring(rgb_ring_lin, center_lin, size=256):
    y, x = np.mgrid[-1:1:size*1j, -1:1:size*1j]
    mag = np.sqrt(x*x + y*y)
    angle = np.mod(np.arctan2(y, x), 2*np.pi)
    
    n_hues = rgb_ring_lin.shape[0]
    hue_f = angle/(2*np.pi)*n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)
    
    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    rgb_lin = (1 - mag[..., None]) * center_lin + mag[..., None] * ring_interp
    rgb_lin[mag > 1] = 0
    return np.clip(linear_to_srgb_np(rgb_lin), 0, 1)

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    n_hues = rgb_ring_lin.shape[0]
    disk = build_hue_disk_from_ring(rgb_ring_lin, center_lin)

    u = flows[0].cpu().numpy()
    v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi)
    mag = np.sqrt(u*u + v*v)
    mag /= (mag.max() + 1e-8)

    hue_f = angle/(2*np.pi)*n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)

    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    r = mag[..., None]
    rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow), 0, 1)

    alpha = mag[..., None]
    white = np.ones_like(flow_rgb)
    black = np.zeros_like(flow_rgb)
    flow_white = alpha * flow_rgb + (1 - alpha) * white
    flow_black = alpha * flow_rgb + (1 - alpha) * black
    return disk, flow_rgb, flow_white, flow_black

def align_ring_red_to_green(ring_lin):
    n = len(ring_lin)
    red_ref = np.array([1.0, 0.0, 0.0])
    dists_r = np.linalg.norm(ring_lin - red_ref, axis=1)
    idx_r = np.argmin(dists_r)
    ring_aligned = np.roll(ring_lin, -idx_r, axis=0)
    
    green_ref = np.array([0.0, 1.0, 0.0])
    check_idx_1 = n // 3
    check_idx_2 = (2 * n) // 3
    dist_1 = np.linalg.norm(ring_aligned[check_idx_1] - green_ref)
    dist_2 = np.linalg.norm(ring_aligned[check_idx_2] - green_ref)
    
    if dist_2 < dist_1:
        ring_aligned = ring_aligned[::-1]
    return ring_aligned

# ==========================================
# 4. Main Execution
# ==========================================

def main():
    device_str = "cpu"
    dev = torch.device(device_str)
    
    print("1. Calculating Geometries...")
    surf_pts = get_gamut_surface(16)
    jz_gamut = xyz_to_jzazbz(srgb_to_xyz(surf_pts))
    center, M, hull = fit_ellipsoid(jz_gamut)
    
    n_hues = 72
    # Jzazbz Hoop
    jz_hoop = calculate_shadow_boundary(center, M, np.array([1.0, 0.0, 0.0]), res=n_hues)
    rgb_hoop_lin = srgb_to_linear_np(xyz_to_srgb(jzazbz_to_xyz(jz_hoop)))
    
    # Sinebow (sRGB math)
    t = np.linspace(0, 1, n_hues, endpoint=False)
    sr = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0/3))
    sg = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1/3))
    sb = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2/3))
    rgb_sine_lin = srgb_to_linear_np(np.stack([sr, sg, sb], axis=1))

    # Align both
    rgb_hoop_lin = align_ring_red_to_green(rgb_hoop_lin)
    rgb_sine_lin = align_ring_red_to_green(rgb_sine_lin)

    print("2. Loading Omnipose Flows...")
    omnidir = Path(omnirefactor.__file__).resolve().parent.parent.parent
    basedir = omnidir / "docs" / "_static"
    name = "ecoli"
    ext = ".tif"

    try:
        masks = io.imread(str(basedir / f"{name}_labels{ext}"))
        slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows = omnirefactor.core.masks_to_flows(masks, device=device_str)[4].to(dev)
        center_lin = srgb_to_linear_np(np.array([0.5, 0.5, 0.5]))

        # Generate Sets
        def gen_set(ring, rotation_steps=0):
            r = np.roll(ring, rotation_steps, axis=0)
            disk, _, w, b = make_flow_images_for_ring(r, center_lin, flows)
            return disk, w, b

        # 0 Degrees
        disk_s_0, w_s_0, b_s_0 = gen_set(rgb_sine_lin, 0)
        disk_e_0, w_e_0, b_e_0 = gen_set(rgb_hoop_lin, 0)
        # 90 Degrees
        rot = n_hues // 4
        _, w_s_90, b_s_90 = gen_set(rgb_sine_lin, rot)
        _, w_e_90, b_e_90 = gen_set(rgb_hoop_lin, rot)

        # Plot 2D
        print("3. Plotting 2D Comparisons...")
        plt.style.use('dark_background')
        fig, axes = plt.subplots(2, 5, figsize=(20, 8))
        rows = [("Sinebow", disk_s_0, b_s_0, w_s_0, b_s_90, w_s_90),
                ("Ellipsoid", disk_e_0, b_e_0, w_e_0, b_e_90, w_e_90)]
        titles = ["Hue Disk", "Black (0°)", "White (0°)", "Black (90°)", "White (90°)"]

        for i, (name, disk, b0, w0, b90, w90) in enumerate(rows):
            ax_row = axes[i]
            ax_row[0].imshow(disk)
            ax_row[0].set_ylabel(name, fontsize=14, color='white', rotation=90, labelpad=20)
            ax_row[1].imshow(b0); ax_row[2].imshow(w0)
            ax_row[3].imshow(b90); ax_row[4].imshow(w90)
            for j, ax in enumerate(ax_row):
                if i == 0: ax.set_title(titles[j], fontsize=12, color='#dddddd')
                ax.set_xticks([]); ax.set_yticks([])
                for spine in ax.spines.values(): spine.set_edgecolor('#333333')

        plt.tight_layout()
        plt.show()
        
        # Plot 3D (WITH EDGES + SINEBOW)
        print("4. Generating 3D Context (With Edges + Sinebow)...")
        fig3d = go.Figure()
        
        max_jz = np.max(jz_gamut[:,0])
        sc = np.array([1.0/max_jz, 1.0, 1.0])
        def mc(a): return a[:,1], a[:,2], a[:,0] # az, bz, Jz
        
        # A. Gamut Faces (Transparent)
        x_h, y_h, z_h = mc(jz_gamut * sc)
        fig3d.add_trace(go.Mesh3d(x=x_h, y=y_h, z=z_h, i=hull.simplices[:,0], j=hull.simplices[:,1], k=hull.simplices[:,2],
                                color='gray', opacity=0.08, name='Gamut Faces', hoverinfo='skip'))

        # B. Gamut Edges (Opaque Lines)
        corners = [[0,0,0], [1,0,0], [1,1,0], [0,1,0], [0,0,1], [1,0,1], [1,1,1], [0,1,1]]
        edges_idx = [[0,1], [1,2], [2,3], [3,0], [4,5], [5,6], [6,7], [7,4], [0,4], [1,5], [2,6], [3,7]]
        for (s, e) in edges_idx:
            line_pts_rgb = np.linspace(corners[s], corners[e], 25)
            line_pts_jz = xyz_to_jzazbz(srgb_to_xyz(line_pts_rgb)) * sc
            xl, yl, zl = mc(line_pts_jz)
            fig3d.add_trace(go.Scatter3d(x=xl, y=yl, z=zl, mode='lines', line=dict(color='white', width=4), showlegend=False))
        fig3d.add_trace(go.Scatter3d(x=[None], y=[None], z=[None], mode='lines', line=dict(color='white', width=4), name='RGB Gamut Edges'))

        # C. Ellipsoid Surface
        u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
        sph = np.stack([np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v)], axis=-1).reshape(-1, 3)
        ell = (sph @ M.T + center) * sc
        ex, ey, ez = mc(ell)
        fig3d.add_trace(go.Surface(x=ex.reshape(u.shape), y=ey.reshape(u.shape), z=ez.reshape(u.shape),
                                colorscale=[(0, '#888888'), (1, '#888888')], opacity=0.3, showscale=False, name='Ellipsoid'))

        # D. The Ellipsoid Hoop
        hx, hy, hz = mc(jz_hoop * sc)
        # Use natural colors for the geometric path
        cols_hoop = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in linear_to_srgb_np(rgb_hoop_lin)]
        fig3d.add_trace(go.Scatter3d(x=hx, y=hy, z=hz, mode='markers', marker=dict(color=cols_hoop, size=5), name='Ellipsoid Loop'))

        # E. THE SINEBOW LOOP (NEW)
        # Convert the Aligned Sinebow RGB -> Jzazbz -> Scaled
        rgb_sine_srgb = linear_to_srgb_np(rgb_sine_lin)
        jz_sine = xyz_to_jzazbz(srgb_to_xyz(rgb_sine_srgb)) * sc
        sx, sy, sz = mc(jz_sine)
        cols_sine = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in rgb_sine_srgb]
        
        fig3d.add_trace(go.Scatter3d(x=sx, y=sy, z=sz, mode='markers', 
                                     marker=dict(color=cols_sine, size=4, symbol='diamond'), name='Sinebow Loop'))
        
        fig3d.update_layout(template="plotly_dark", title="3D Jzazbz Gamut with Loops", 
                            scene=dict(xaxis_title='az', yaxis_title='bz', zaxis_title='Jz (Norm)',
                                       # aspectmode='manual', aspectratio=dict(x=1,y=1,z=1), 
                                       aspectmode='data'))

        
        fig3d.show()

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import scipy.optimize as opt
from scipy.spatial import ConvexHull
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pathlib import Path

# Omnipose / Imaging Imports
import torch
import omnirefactor.core
from skimage import io
import fastremap

# ==========================================
# 1. Color Math (Jzazbz)
# ==========================================
def srgb_to_xyz(rgb):
    mask = rgb > 0.04045
    linear = np.zeros_like(rgb)
    linear[mask] = ((rgb[mask] + 0.055) / 1.055) ** 2.4
    linear[~mask] = rgb[~mask] / 12.92
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    return linear @ M.T

def xyz_to_jzazbz(xyz):
    b, g, d, d0 = 1.15, 0.66, -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    lms = xyz @ M1.T
    lms_norm = (lms * 200) / 10000.0 
    lms_prime = np.sign(lms_norm) * (((c1 + c2 * (np.abs(lms_norm) ** n)) / (1 + c3 * (np.abs(lms_norm) ** n))) ** p)
    izazbz = lms_prime @ M2.T
    Jz = ((1 + d) * izazbz[:, 0]) / (1 + d * izazbz[:, 0]) - d0
    return np.column_stack((Jz, izazbz[:, 1], izazbz[:, 2]))

def jzazbz_to_xyz(jzazbz):
    Jz, az, bz = jzazbz[:,0], jzazbz[:,1], jzazbz[:,2]
    d, d0 = -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    Jp = Jz + d0; Iz = Jp / (1 + d - d * Jp)
    izazbz_vec = np.stack([Iz, az, bz], axis=1)
    lms_prime = izazbz_vec @ np.linalg.inv(M2).T
    sign_lms = np.sign(lms_prime)
    Y = np.abs(lms_prime) ** (1/p)
    num = Y - c1; den = c2 - Y * c3
    An = np.clip(num / den, 0, None) 
    lms_norm = sign_lms * (An ** (1/n))
    lms = (lms_norm * 10000.0) / 200.0
    return lms @ np.linalg.inv(M1).T

def xyz_to_srgb(xyz):
    M_inv = np.linalg.inv(np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]]))
    linear = xyz @ M_inv.T
    return linear_to_srgb_np(linear)

def srgb_to_linear_np(srgb):
    return np.where(srgb <= 0.04045, srgb / 12.92, ((srgb + 0.055) / 1.055) ** 2.4)

def linear_to_srgb_np(lin):
    s = np.zeros_like(lin)
    mask = lin > 0.0031308
    s[mask] = 1.055 * (lin[mask]**(1/2.4)) - 0.055
    s[~mask] = 12.92 * lin[~mask]
    return np.clip(s, 0, 1)

# ==========================================
# 2. Optimization (SPHERE)
# ==========================================

def get_gamut_surface(res=16):
    t = np.linspace(0, 1, res)
    g = np.meshgrid(t, t)
    faces = [np.stack([g[0], g[1], np.zeros_like(g[0])], -1), np.stack([g[0], g[1], np.ones_like(g[0])], -1),
             np.stack([g[0], np.zeros_like(g[0]), g[1]], -1), np.stack([g[0], np.ones_like(g[0]), g[1]], -1),
             np.stack([np.zeros_like(g[0]), g[0], g[1]], -1), np.stack([np.ones_like(g[0]), g[0], g[1]], -1)]
    return np.vstack([f.reshape(-1, 3) for f in faces])

def fit_sphere(points):
    """
    Finds the largest inscribed sphere (Chebyshev Center).
    Optimization Variable: x = [center_J, center_a, center_b, radius]
    """
    hull = ConvexHull(points)
    A = hull.equations[:, :3] # Normal vectors
    b_const = -hull.equations[:, 3] # Offsets
    
    # Scale up for numerical stability
    scale = 100.0
    A_scaled = A / scale
    
    # Initial Guess: Centroid, r=0.01
    c0 = np.mean(points, axis=0) * scale
    x0 = np.append(c0, 0.01)

    # Objective: Maximize radius (Minimize -r)
    def objective(x): return -x[3]

    # Constraint: Distance to every face >= radius
    # For plane n.x <= d, distance is (d - n.c) / ||n||. 
    # Here equations are A.x + b <= 0 -> A.x <= -b
    # Distance slack = (-b - A.c) / ||A|| >= r
    def constraints(x):
        center = x[:3]
        radius = x[3]
        # b_const - (A @ center + radius * ||A||) >= 0
        return b_const - (A_scaled @ center + radius * np.linalg.norm(A_scaled, axis=1))

    print("Optimizing Inscribed Sphere...")
    res = opt.minimize(objective, x0, method='SLSQP', constraints={'type': 'ineq', 'fun': constraints}, options={'maxiter': 500})
    
    center_opt = res.x[:3] / scale
    radius_opt = res.x[3] / scale
    return center_opt, radius_opt, hull

# ==========================================
# 3. Helpers
# ==========================================

def build_hue_disk_from_ring(rgb_ring_lin, center_lin, size=256):
    y, x = np.mgrid[-1:1:size*1j, -1:1:size*1j]
    mag = np.sqrt(x*x + y*y)
    angle = np.mod(np.arctan2(y, x), 2*np.pi)
    
    n_hues = rgb_ring_lin.shape[0]
    hue_f = angle/(2*np.pi)*n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)
    
    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    rgb_lin = (1 - mag[..., None]) * center_lin + mag[..., None] * ring_interp
    rgb_lin[mag > 1] = 0
    return np.clip(linear_to_srgb_np(rgb_lin), 0, 1)

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    n_hues = rgb_ring_lin.shape[0]
    disk = build_hue_disk_from_ring(rgb_ring_lin, center_lin)
    u = flows[0].cpu().numpy()
    v = flows[1].cpu().numpy()
    
    angle = np.mod(np.arctan2(v, u), 2*np.pi)
    mag = np.sqrt(u*u + v*v)
    mag /= (mag.max() + 1e-8)

    hue_f = angle/(2*np.pi)*n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)

    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    r = mag[..., None]
    rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow), 0, 1)

    alpha = mag[..., None]
    white = np.ones_like(flow_rgb)
    black = np.zeros_like(flow_rgb)
    flow_white = alpha * flow_rgb + (1 - alpha) * white
    flow_black = alpha * flow_rgb + (1 - alpha) * black
    return disk, flow_rgb, flow_white, flow_black

def align_ring_red_to_green(ring_lin):
    n = len(ring_lin)
    red_ref = np.array([1.0, 0.0, 0.0])
    idx_r = np.argmin(np.linalg.norm(ring_lin - red_ref, axis=1))
    ring_aligned = np.roll(ring_lin, -idx_r, axis=0)
    
    green_ref = np.array([0.0, 1.0, 0.0])
    dist_1 = np.linalg.norm(ring_aligned[n // 3] - green_ref)
    dist_2 = np.linalg.norm(ring_aligned[(2 * n) // 3] - green_ref)
    if dist_2 < dist_1: ring_aligned = ring_aligned[::-1]
    return ring_aligned

# ==========================================
# 4. Main
# ==========================================

def main():
    device_str = "cpu"
    dev = torch.device(device_str)
    
    print("1. Calculating Geometries...")
    surf_pts = get_gamut_surface(16)
    jz_gamut = xyz_to_jzazbz(srgb_to_xyz(surf_pts))
    
    # Fit Sphere
    center, radius, hull = fit_sphere(jz_gamut)
    print(f"   Max Inscribed Sphere Radius: {radius:.4f}")
    
    n_hues = 72
    
    # 1. Sphere Equator (Constant Jz)
    theta = np.linspace(0, 2*np.pi, n_hues)
    jz_hoop = np.column_stack([
        np.full_like(theta, center[0]),
        center[1] + radius * np.cos(theta),
        center[2] + radius * np.sin(theta)
    ])
    rgb_hoop_lin = srgb_to_linear_np(xyz_to_srgb(jzazbz_to_xyz(jz_hoop)))

    # 2. Sinebow
    t = np.linspace(0, 1, n_hues, endpoint=False)
    sr = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0/3))
    sg = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1/3))
    sb = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2/3))
    rgb_sine_lin = srgb_to_linear_np(np.stack([sr, sg, sb], axis=1))

    # Align
    rgb_hoop_lin = align_ring_red_to_green(rgb_hoop_lin)
    rgb_sine_lin = align_ring_red_to_green(rgb_sine_lin)

    print("2. Loading Omnipose Flows...")
    omnidir = Path(omnirefactor.__file__).resolve().parent.parent.parent
    basedir = omnidir / "docs" / "_static"
    name = "ecoli"
    ext = ".tif"

    try:
        masks = io.imread(str(basedir / f"{name}_labels{ext}"))
        slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows = omnirefactor.core.masks_to_flows(masks, device=device_str)[4].to(dev)
        center_lin = srgb_to_linear_np(np.array([0.5, 0.5, 0.5]))

        def gen_set(ring, rotation_steps=0):
            r = np.roll(ring, rotation_steps, axis=0)
            disk, _, w, b = make_flow_images_for_ring(r, center_lin, flows)
            return disk, w, b

        # 0 Deg
        disk_s_0, w_s_0, b_s_0 = gen_set(rgb_sine_lin, 0)
        disk_e_0, w_e_0, b_e_0 = gen_set(rgb_hoop_lin, 0)
        # 90 Deg
        rot = n_hues // 4
        _, w_s_90, b_s_90 = gen_set(rgb_sine_lin, rot)
        _, w_e_90, b_e_90 = gen_set(rgb_hoop_lin, rot)

        print("3. Plotting 2D...")
        plt.style.use('dark_background')
        fig, axes = plt.subplots(2, 5, figsize=(20, 8))
        rows = [("Sinebow", disk_s_0, b_s_0, w_s_0, b_s_90, w_s_90),
                ("Sphere", disk_e_0, b_e_0, w_e_0, b_e_90, w_e_90)]
        titles = ["Hue Disk", "Black (0°)", "White (0°)", "Black (90°)", "White (90°)"]

        for i, (name, disk, b0, w0, b90, w90) in enumerate(rows):
            ax_row = axes[i]
            ax_row[0].imshow(disk)
            ax_row[0].set_ylabel(name, fontsize=14, color='white', rotation=90, labelpad=20)
            ax_row[1].imshow(b0); ax_row[2].imshow(w0)
            ax_row[3].imshow(b90); ax_row[4].imshow(w90)
            for j, ax in enumerate(ax_row):
                if i == 0: ax.set_title(titles[j], fontsize=12, color='#dddddd')
                ax.set_xticks([]); ax.set_yticks([])
                for spine in ax.spines.values(): spine.set_edgecolor('#333333')
        plt.tight_layout()
        plt.show()
        
        # 3D Plot
        print("4. Generating 3D...")
        fig3d = go.Figure()
        
        # NOTE: NO SCALING APPLIED HERE. 
        # Plotting in raw Jzazbz data units to prove it is a sphere.
        # It will look like a pancake because the sRGB gamut IS a pancake in Jzazbz.
        def mc(a): return a[:,1], a[:,2], a[:,0] # az, bz, Jz
        
        # Gamut Edges
        corners = [[0,0,0], [1,0,0], [1,1,0], [0,1,0], [0,0,1], [1,0,1], [1,1,1], [0,1,1]]
        edges_idx = [[0,1], [1,2], [2,3], [3,0], [4,5], [5,6], [6,7], [7,4], [0,4], [1,5], [2,6], [3,7]]
        for (s, e) in edges_idx:
            line_pts = np.linspace(corners[s], corners[e], 25)
            line_jz = xyz_to_jzazbz(srgb_to_xyz(line_pts))
            xl, yl, zl = mc(line_jz)
            fig3d.add_trace(go.Scatter3d(x=xl, y=yl, z=zl, mode='lines', line=dict(color='white', width=4), showlegend=False))
        fig3d.add_trace(go.Scatter3d(x=[None], y=[None], z=[None], mode='lines', line=dict(color='white', width=4), name='RGB Edges'))

        # Sphere
        u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
        sx = center[0] + radius * np.cos(u) * np.sin(v)
        sy = center[1] + radius * np.sin(u) * np.sin(v)
        sz = center[2] + radius * np.cos(v)
        # Map: Jz(sx)->Z, az(sy)->X, bz(sz)->Y
        fig3d.add_trace(go.Surface(x=sy, y=sz, z=sx, colorscale='gray', opacity=0.3, showscale=False, name='Sphere'))

        # Loops
        hx, hy, hz = mc(jz_hoop)
        c_hoop = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in linear_to_srgb_np(rgb_hoop_lin)]
        fig3d.add_trace(go.Scatter3d(x=hx, y=hy, z=hz, mode='markers', marker=dict(color=c_hoop, size=5), name='Sphere Equator'))

        rgb_sine_srgb = linear_to_srgb_np(rgb_sine_lin)
        jz_sine = xyz_to_jzazbz(srgb_to_xyz(rgb_sine_srgb))
        sx, sy, sz = mc(jz_sine)
        c_sine = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in rgb_sine_srgb]
        fig3d.add_trace(go.Scatter3d(x=sx, y=sy, z=sz, mode='markers', marker=dict(color=c_sine, size=4, symbol='diamond'), name='Sinebow Loop'))
        
        # IMPORTANT: Aspect mode = data ensures 1 unit Jz = 1 unit az = 1 unit bz
        fig3d.update_layout(template="plotly_dark", title="3D Jzazbz (Raw Scale: True Sphere)", 
                            scene=dict(xaxis_title='az', yaxis_title='bz', zaxis_title='Jz',
                                       aspectmode='data'))
        fig3d.show()

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import scipy.optimize as opt
from scipy.spatial import ConvexHull
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pathlib import Path

# Omnipose / Imaging Imports
import torch
import omnirefactor.core
from skimage import io
import fastremap

# ==========================================
# 1. Color Math
# ==========================================
def srgb_to_xyz(rgb):
    mask = rgb > 0.04045
    linear = np.zeros_like(rgb)
    linear[mask] = ((rgb[mask] + 0.055) / 1.055) ** 2.4
    linear[~mask] = rgb[~mask] / 12.92
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    return linear @ M.T

def xyz_to_jzazbz(xyz):
    b, g, d, d0 = 1.15, 0.66, -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    lms = xyz @ M1.T
    lms_norm = (lms * 200) / 10000.0 
    lms_prime = np.sign(lms_norm) * (((c1 + c2 * (np.abs(lms_norm) ** n)) / (1 + c3 * (np.abs(lms_norm) ** n))) ** p)
    izazbz = lms_prime @ M2.T
    Jz = ((1 + d) * izazbz[:, 0]) / (1 + d * izazbz[:, 0]) - d0
    return np.column_stack((Jz, izazbz[:, 1], izazbz[:, 2]))

def jzazbz_to_xyz(jzazbz):
    Jz, az, bz = jzazbz[:,0], jzazbz[:,1], jzazbz[:,2]
    d, d0 = -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    Jp = Jz + d0; Iz = Jp / (1 + d - d * Jp)
    izazbz_vec = np.stack([Iz, az, bz], axis=1)
    lms_prime = izazbz_vec @ np.linalg.inv(M2).T
    sign_lms = np.sign(lms_prime)
    Y = np.abs(lms_prime) ** (1/p)
    num = Y - c1; den = c2 - Y * c3
    An = np.clip(num / den, 0, None) 
    lms_norm = sign_lms * (An ** (1/n))
    lms = (lms_norm * 10000.0) / 200.0
    return lms @ np.linalg.inv(M1).T

def xyz_to_srgb(xyz):
    M_inv = np.linalg.inv(np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]]))
    linear = xyz @ M_inv.T
    return linear_to_srgb_np(linear)

def srgb_to_linear_np(srgb):
    return np.where(srgb <= 0.04045, srgb / 12.92, ((srgb + 0.055) / 1.055) ** 2.4)

def linear_to_srgb_np(lin):
    s = np.zeros_like(lin)
    mask = lin > 0.0031308
    s[mask] = 1.055 * (lin[mask]**(1/2.4)) - 0.055
    s[~mask] = 12.92 * lin[~mask]
    return np.clip(s, 0, 1)

# ==========================================
# 2. Optimization (ANCHOR POINT STRATEGY)
# ==========================================

def get_gamut_surface(res=16):
    t = np.linspace(0, 1, res)
    g = np.meshgrid(t, t)
    faces = [np.stack([g[0], g[1], np.zeros_like(g[0])], -1), np.stack([g[0], g[1], np.ones_like(g[0])], -1),
             np.stack([g[0], np.zeros_like(g[0]), g[1]], -1), np.stack([g[0], np.ones_like(g[0]), g[1]], -1),
             np.stack([np.zeros_like(g[0]), g[0], g[1]], -1), np.stack([np.ones_like(g[0]), g[0], g[1]], -1)]
    return np.vstack([f.reshape(-1, 3) for f in faces])

def fit_ellipsoid_anchored(points, vertex_target, anchor_strength=0.75):
    """
    Maximizes Volume while ensuring the ellipsoid contains a specific 'Anchor Point'.
    Anchor Point = Centroid + strength * (GreenVertex - Centroid)
    
    This forces the ellipse to reach towards green, but allows the center to float
    backwards to maintain width (touching other sides).
    """
    hull = ConvexHull(points)
    A = hull.equations[:, :3]
    b_const = -hull.equations[:, 3]
    scale = 100.0
    A_scaled = A / scale
    v_scaled = vertex_target * scale
    
    # Calculate Anchor Point (The point the ellipse MUST enclose)
    centroid = np.mean(points, axis=0) * scale
    anchor = centroid + anchor_strength * (v_scaled - centroid)
    
    # Init: Center at Centroid, Sphere shape
    c0 = centroid
    M0 = np.eye(3).flatten() * 0.1
    x0 = np.concatenate([c0, M0])

    def objective(x):
        M = x[3:].reshape(3, 3)
        sign, val = np.linalg.slogdet(M)
        if sign <= 0: return 1e9
        return -val # Maximize Volume

    def constraints(x):
        d = x[:3]
        M = x[3:].reshape(3, 3)
        
        # 1. Hull Constraint: Ellipsoid inside Gamut
        # b - (Ad + ||AM||) >= 0
        norm_AM = np.linalg.norm(A_scaled @ M, axis=1)
        Ad = A_scaled @ d
        hull_con = b_const - (Ad + norm_AM)
        
        # 2. Anchor Constraint: Anchor point inside Ellipsoid
        # (p - d)^T (M M^T)^-1 (p - d) <= 1
        # Equivalent to: || M^-1 (p - d) || <= 1
        # or: 1 - || M^-1 (p - d) || >= 0
        try:
            diff = anchor - d
            # Solve M*u = diff for u
            u = np.linalg.solve(M, diff)
            dist = np.linalg.norm(u)
            anchor_con = 1.0 - dist
        except np.linalg.LinAlgError:
            anchor_con = -1.0 # Fail
            
        return np.concatenate([hull_con, [anchor_con]])

    print(f"Optimizing (Anchor Strength={anchor_strength})...")
    res = opt.minimize(objective, x0, method='SLSQP', constraints={'type': 'ineq', 'fun': constraints}, 
                       options={'maxiter': 500, 'ftol': 1e-4})
    
    return res.x[:3]/scale, res.x[3:].reshape(3, 3)/scale, hull

def calculate_shadow_boundary(center, M, view_vector, res=256):
    w = np.linalg.solve(M, view_vector); w = w / np.linalg.norm(w)
    tmp = np.array([0.0, 1.0, 0.0]) if np.abs(w[1]) < 0.9 else np.array([0.0, 0.0, 1.0])
    s1 = np.cross(w, tmp); s1 /= np.linalg.norm(s1); s2 = np.cross(w, s1)
    theta = np.linspace(0, 2*np.pi, res)
    return center + (np.outer(np.cos(theta), s1) + np.outer(np.sin(theta), s2)) @ M.T

# ==========================================
# 3. Omnipose Helpers
# ==========================================
def build_hue_disk_from_ring(rgb_ring_lin, center_lin, size=256):
    y, x = np.mgrid[-1:1:size*1j, -1:1:size*1j]
    mag = np.sqrt(x*x + y*y)
    angle = np.mod(np.arctan2(y, x), 2*np.pi)
    n_hues = rgb_ring_lin.shape[0]
    hue_f = angle/(2*np.pi)*n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)
    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    rgb_lin = (1 - mag[..., None]) * center_lin + mag[..., None] * ring_interp
    rgb_lin[mag > 1] = 0
    return np.clip(linear_to_srgb_np(rgb_lin), 0, 1)

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    n_hues = rgb_ring_lin.shape[0]
    disk = build_hue_disk_from_ring(rgb_ring_lin, center_lin)
    u = flows[0].cpu().numpy()
    v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi)
    mag = np.sqrt(u*u + v*v)
    mag /= (mag.max() + 1e-8)
    hue_f = angle/(2*np.pi)*n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)
    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    r = mag[..., None]
    rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow), 0, 1)
    alpha = mag[..., None]
    white = np.ones_like(flow_rgb)
    black = np.zeros_like(flow_rgb)
    flow_white = alpha * flow_rgb + (1 - alpha) * white
    flow_black = alpha * flow_rgb + (1 - alpha) * black
    return disk, flow_rgb, flow_white, flow_black

def align_ring_red_to_green(ring_lin):
    n = len(ring_lin)
    red_ref = np.array([1.0, 0.0, 0.0])
    idx_r = np.argmin(np.linalg.norm(ring_lin - red_ref, axis=1))
    ring_aligned = np.roll(ring_lin, -idx_r, axis=0)
    green_ref = np.array([0.0, 1.0, 0.0])
    if np.linalg.norm(ring_aligned[(2*n)//3]-green_ref) < np.linalg.norm(ring_aligned[n//3]-green_ref):
        ring_aligned = ring_aligned[::-1]
    return ring_aligned

# ==========================================
# 4. Main
# ==========================================

def main():
    device_str = "cpu"
    dev = torch.device(device_str)
    
    print("1. Calculating Geometries...")
    surf_pts = get_gamut_surface(18)
    jz_gamut = xyz_to_jzazbz(srgb_to_xyz(surf_pts))
    
    # Target: Green Vertex
    green_jz = xyz_to_jzazbz(srgb_to_xyz(np.array([[0.0, 1.0, 0.0]])))[0]
    
    # *** ANCHOR STRATEGY: Reach 75% of the way to Green, but maximize width ***
    # This prevents the needle collapse because volume maximization forces width
    center, M, hull = fit_ellipsoid_anchored(jz_gamut, green_jz, anchor_strength=0.75)
    
    n_hues = 72
    # Ellipsoid Hoop
    jz_hoop = calculate_shadow_boundary(center, M, np.array([1.0, 0.0, 0.0]), res=n_hues)
    rgb_hoop_lin = srgb_to_linear_np(xyz_to_srgb(jzazbz_to_xyz(jz_hoop)))
    
    # Sinebow
    t = np.linspace(0, 1, n_hues, endpoint=False)
    sr = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0/3))
    sg = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1/3))
    sb = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2/3))
    rgb_sine_lin = srgb_to_linear_np(np.stack([sr, sg, sb], axis=1))

    # Align
    rgb_hoop_lin = align_ring_red_to_green(rgb_hoop_lin)
    rgb_sine_lin = align_ring_red_to_green(rgb_sine_lin)

    print("2. Loading Omnipose Flows...")
    omnidir = Path(omnirefactor.__file__).resolve().parent.parent.parent
    basedir = omnidir / "docs" / "_static"
    name = "ecoli"
    ext = ".tif"

    try:
        masks = io.imread(str(basedir / f"{name}_labels{ext}"))
        slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows = omnirefactor.core.masks_to_flows(masks, device=device_str)[4].to(dev)
        center_lin = srgb_to_linear_np(np.array([0.5, 0.5, 0.5]))

        def gen_set(ring, rotation_steps=0):
            r = np.roll(ring, rotation_steps, axis=0)
            disk, _, w, b = make_flow_images_for_ring(r, center_lin, flows)
            return disk, w, b

        disk_s_0, w_s_0, b_s_0 = gen_set(rgb_sine_lin, 0)
        disk_e_0, w_e_0, b_e_0 = gen_set(rgb_hoop_lin, 0)
        rot = n_hues // 4
        _, w_s_90, b_s_90 = gen_set(rgb_sine_lin, rot)
        _, w_e_90, b_e_90 = gen_set(rgb_hoop_lin, rot)

        print("3. Plotting 2D...")
        plt.style.use('dark_background')
        fig, axes = plt.subplots(2, 5, figsize=(20, 8))
        rows = [("Sinebow", disk_s_0, b_s_0, w_s_0, b_s_90, w_s_90),
                ("Anchor Ellipse", disk_e_0, b_e_0, w_e_0, b_e_90, w_e_90)]
        titles = ["Hue Disk", "Black (0°)", "White (0°)", "Black (90°)", "White (90°)"]

        for i, (name, disk, b0, w0, b90, w90) in enumerate(rows):
            ax_row = axes[i]
            ax_row[0].imshow(disk)
            ax_row[0].set_ylabel(name, fontsize=14, color='white', rotation=90, labelpad=20)
            ax_row[1].imshow(b0); ax_row[2].imshow(w0)
            ax_row[3].imshow(b90); ax_row[4].imshow(w90)
            for j, ax in enumerate(ax_row):
                if i == 0: ax.set_title(titles[j], fontsize=12, color='#dddddd')
                ax.set_xticks([]); ax.set_yticks([])
                for spine in ax.spines.values(): spine.set_edgecolor('#333333')
        plt.tight_layout()
        plt.show()
        
        print("4. Generating 3D...")
        fig3d = go.Figure()
        
        max_jz = np.max(jz_gamut[:,0])
        sc = np.array([1.0/max_jz, 1.0, 1.0]) 
        def mc(a): return a[:,1], a[:,2], a[:,0]
        
        # Edges
        corners = [[0,0,0], [1,0,0], [1,1,0], [0,1,0], [0,0,1], [1,0,1], [1,1,1], [0,1,1]]
        edges_idx = [[0,1], [1,2], [2,3], [3,0], [4,5], [5,6], [6,7], [7,4], [0,4], [1,5], [2,6], [3,7]]
        for (s, e) in edges_idx:
            line_pts = np.linspace(corners[s], corners[e], 25)
            line_jz = xyz_to_jzazbz(srgb_to_xyz(line_pts)) * sc
            xl, yl, zl = mc(line_jz)
            fig3d.add_trace(go.Scatter3d(x=xl, y=yl, z=zl, mode='lines', line=dict(color='white', width=4), showlegend=False))
        fig3d.add_trace(go.Scatter3d(x=[None], y=[None], z=[None], mode='lines', line=dict(color='white', width=4), name='RGB Edges'))

        # Ellipsoid
        u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
        sph = np.stack([np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v)], axis=-1).reshape(-1, 3)
        ell = (sph @ M.T + center) * sc
        ex, ey, ez = mc(ell)
        fig3d.add_trace(go.Surface(x=ex.reshape(u.shape), y=ey.reshape(u.shape), z=ez.reshape(u.shape),
                                colorscale=[(0, '#88aa88'), (1, '#88aa88')], opacity=0.3, showscale=False, name='Anchored Ellipsoid'))

        # Loops
        hx, hy, hz = mc(jz_hoop * sc)
        c_hoop = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in linear_to_srgb_np(rgb_hoop_lin)]
        fig3d.add_trace(go.Scatter3d(x=hx, y=hy, z=hz, mode='markers', marker=dict(color=c_hoop, size=5), name='Green Profile'))

        rgb_sine_srgb = linear_to_srgb_np(rgb_sine_lin)
        jz_sine = xyz_to_jzazbz(srgb_to_xyz(rgb_sine_srgb)) * sc
        sx, sy, sz = mc(jz_sine)
        c_sine = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in rgb_sine_srgb]
        fig3d.add_trace(go.Scatter3d(x=sx, y=sy, z=sz, mode='markers', marker=dict(color=c_sine, size=4, symbol='diamond'), name='Sinebow'))
        
        fig3d.update_layout(template="plotly_dark", title="3D Jzazbz (Anchored to Green)", 
                            scene=dict(xaxis_title='az', yaxis_title='bz', zaxis_title='Jz',
                                       aspectmode='manual', aspectratio=dict(x=1,y=1,z=1)))
        fig3d.show()

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import scipy.optimize as opt
from scipy.spatial import ConvexHull
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import omnirefactor.core
from skimage import io
import fastremap

# ==========================================
# 1. Color Math
# ==========================================
def srgb_to_xyz(rgb):
    mask = rgb > 0.04045
    linear = np.zeros_like(rgb)
    linear[mask] = ((rgb[mask] + 0.055) / 1.055) ** 2.4
    linear[~mask] = rgb[~mask] / 12.92
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    return linear @ M.T

def xyz_to_jzazbz(xyz):
    b, g, d, d0 = 1.15, 0.66, -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    lms = xyz @ M1.T
    lms_norm = (lms * 200) / 10000.0 
    lms_prime = np.sign(lms_norm) * (((c1 + c2 * (np.abs(lms_norm) ** n)) / (1 + c3 * (np.abs(lms_norm) ** n))) ** p)
    izazbz = lms_prime @ M2.T
    Jz = ((1 + d) * izazbz[:, 0]) / (1 + d * izazbz[:, 0]) - d0
    return np.column_stack((Jz, izazbz[:, 1], izazbz[:, 2]))

def jzazbz_to_xyz(jzazbz):
    Jz, az, bz = jzazbz[:,0], jzazbz[:,1], jzazbz[:,2]
    d, d0 = -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    Jp = Jz + d0; Iz = Jp / (1 + d - d * Jp)
    izazbz_vec = np.stack([Iz, az, bz], axis=1)
    lms_prime = izazbz_vec @ np.linalg.inv(M2).T
    sign_lms = np.sign(lms_prime)
    Y = np.abs(lms_prime) ** (1/p)
    num = Y - c1; den = c2 - Y * c3
    An = np.clip(num / den, 0, None) 
    lms_norm = sign_lms * (An ** (1/n))
    lms = (lms_norm * 10000.0) / 200.0
    return lms @ np.linalg.inv(M1).T

def xyz_to_srgb(xyz):
    M_inv = np.linalg.inv(np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]]))
    linear = xyz @ M_inv.T
    return linear_to_srgb_np(linear)

def srgb_to_linear_np(srgb):
    return np.where(srgb <= 0.04045, srgb / 12.92, ((srgb + 0.055) / 1.055) ** 2.4)

def linear_to_srgb_np(lin):
    s = np.zeros_like(lin)
    mask = lin > 0.0031308
    s[mask] = 1.055 * (lin[mask]**(1/2.4)) - 0.055
    s[~mask] = 12.92 * lin[~mask]
    return np.clip(s, 0, 1)

# ==========================================
# 2. Optimization Logic
# ==========================================
def get_gamut_surface(res=16):
    t = np.linspace(0, 1, res)
    g = np.meshgrid(t, t)
    faces = [np.stack([g[0], g[1], np.zeros_like(g[0])], -1), np.stack([g[0], g[1], np.ones_like(g[0])], -1),
             np.stack([g[0], np.zeros_like(g[0]), g[1]], -1), np.stack([g[0], np.ones_like(g[0]), g[1]], -1),
             np.stack([np.zeros_like(g[0]), g[0], g[1]], -1), np.stack([np.ones_like(g[0]), g[0], g[1]], -1)]
    return np.vstack([f.reshape(-1, 3) for f in faces])

def fit_ellipsoid_anchored(points, anchor_point):
    hull = ConvexHull(points)
    A = hull.equations[:, :3]; b_const = -hull.equations[:, 3]
    scale = 100.0; A_scaled = A / scale; anchor_scaled = anchor_point * scale
    
    c0 = np.mean(points, axis=0) * scale
    c0 = 0.5 * c0 + 0.5 * anchor_scaled
    x0 = np.concatenate([c0, np.eye(3).flatten() * 0.1])

    def objective(x):
        M = x[3:].reshape(3, 3)
        sign, val = np.linalg.slogdet(M)
        if sign <= 0: return 1e9
        return -val 

    def constraints(x):
        d, M = x[:3], x[3:].reshape(3, 3)
        hull_con = b_const - (A_scaled @ d + np.linalg.norm(A_scaled @ M, axis=1))
        try:
            u = np.linalg.solve(M, anchor_scaled - d)
            anchor_con = 1.0 - np.linalg.norm(u)
        except: anchor_con = -1.0
        return np.concatenate([hull_con, [anchor_con]])

    res = opt.minimize(objective, x0, method='SLSQP', constraints={'type': 'ineq', 'fun': constraints}, options={'maxiter': 500})
    return res.x[:3]/scale, res.x[3:].reshape(3, 3)/scale

def calculate_shadow_boundary(center, M, view_vector, res=72):
    w = np.linalg.solve(M, view_vector); w = w / np.linalg.norm(w)
    tmp = np.array([0.0, 1.0, 0.0]) if np.abs(w[1]) < 0.9 else np.array([0.0, 0.0, 1.0])
    s1 = np.cross(w, tmp); s1 /= np.linalg.norm(s1); s2 = np.cross(w, s1)
    theta = np.linspace(0, 2*np.pi, res)
    return center + (np.outer(np.cos(theta), s1) + np.outer(np.sin(theta), s2)) @ M.T

# ==========================================
# 3. Omnipose Helpers
# ==========================================
def build_hue_disk_from_ring(rgb_ring_lin, center_lin, size=256):
    y, x = np.mgrid[-1:1:size*1j, -1:1:size*1j]
    mag = np.sqrt(x*x + y*y)
    angle = np.mod(np.arctan2(y, x), 2*np.pi)
    n_hues = rgb_ring_lin.shape[0]
    hue_f = angle/(2*np.pi)*n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)
    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    rgb_lin = (1 - mag[..., None]) * center_lin + mag[..., None] * ring_interp
    rgb_lin[mag > 1] = 0
    return np.clip(linear_to_srgb_np(rgb_lin), 0, 1)

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    disk = build_hue_disk_from_ring(rgb_ring_lin, center_lin)
    u = flows[0].cpu().numpy()
    v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi)
    mag = np.sqrt(u*u + v*v)
    mag /= (mag.max() + 1e-8)
    n_hues = rgb_ring_lin.shape[0]
    hue_f = angle/(2*np.pi)*n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)
    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    r = mag[..., None]
    rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow), 0, 1)
    alpha = mag[..., None]
    white = np.ones_like(flow_rgb)
    black = np.zeros_like(flow_rgb)
    flow_white = alpha * flow_rgb + (1 - alpha) * white
    flow_black = alpha * flow_rgb + (1 - alpha) * black
    return disk, flow_rgb, flow_white, flow_black

def align_ring_red_to_green(ring_lin):
    n = len(ring_lin)
    red_ref = np.array([1.0, 0.0, 0.0])
    idx_r = np.argmin(np.linalg.norm(ring_lin - red_ref, axis=1))
    ring_aligned = np.roll(ring_lin, -idx_r, axis=0)
    green_ref = np.array([0.0, 1.0, 0.0])
    if np.linalg.norm(ring_aligned[(2*n)//3]-green_ref) < np.linalg.norm(ring_aligned[n//3]-green_ref):
        ring_aligned = ring_aligned[::-1]
    return ring_aligned

# ==========================================
# 4. Main Execution
# ==========================================

def main():
    device_str = "cpu"
    dev = torch.device(device_str)
    
    print("1. Calculating Geometries & Tuning...")
    surf_pts = get_gamut_surface(18)
    jz_gamut = xyz_to_jzazbz(srgb_to_xyz(surf_pts))
    
    centroid = np.mean(jz_gamut, axis=0)
    green_vertex = xyz_to_jzazbz(srgb_to_xyz(np.array([[0.0, 1.0, 0.0]])))[0]
    
    configs = [
        {"name": "Safe (50%)", "str": 0.50, "color": "cyan"},
        {"name": "Vivid (75%)", "str": 0.75, "color": "yellow"},
        {"name": "Extreme (90%)", "str": 0.90, "color": "magenta"},
    ]
    
    results = []
    
    for cfg in configs:
        print(f"   Processing {cfg['name']}...")
        anchor = centroid + cfg['str'] * (green_vertex - centroid)
        center, M = fit_ellipsoid_anchored(jz_gamut, anchor)
        
        # Calculate Loop (Hoop)
        hoop = calculate_shadow_boundary(center, M, np.array([1.0, 0.0, 0.0]), res=72)
        rgb_hoop_lin = srgb_to_linear_np(xyz_to_srgb(jzazbz_to_xyz(hoop)))
        rgb_hoop_lin = align_ring_red_to_green(rgb_hoop_lin)
        
        results.append({
            "name": cfg['name'],
            "color": cfg['color'],
            "ring": rgb_hoop_lin,
            "anchor": anchor,
            "hoop_jz": hoop
        })

    print("2. Loading Omnipose Flows...")
    omnidir = Path(omnirefactor.__file__).resolve().parent.parent.parent
    basedir = omnidir / "docs" / "_static"
    name = "ecoli"
    ext = ".tif"

    try:
        masks = io.imread(str(basedir / f"{name}_labels{ext}"))
        slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows = omnirefactor.core.masks_to_flows(masks, device=device_str)[4].to(dev)
        center_lin = srgb_to_linear_np(np.array([0.5, 0.5, 0.5]))

        # --- PLOT 1: 2D MATPLOTLIB GRID ---
        print("3. Plotting 2D Flow Grid...")
        plt.style.use('dark_background')
        fig, axes = plt.subplots(3, 4, figsize=(16, 12))
        
        for i, res in enumerate(results):
            disk, _, w, b = make_flow_images_for_ring(res['ring'], center_lin, flows)
            ax_row = axes[i]
            
            # 1. Gradient
            ax_row[0].imshow(linear_to_srgb_np(res['ring'])[None, ...], aspect='auto')
            ax_row[0].set_ylabel(res['name'], fontsize=14, color='white', rotation=90, labelpad=20)
            if i==0: ax_row[0].set_title("Gradient")
            
            # 2. Disk
            ax_row[1].imshow(disk)
            if i==0: ax_row[1].set_title("Hue Disk")
            
            # 3. Black BG
            ax_row[2].imshow(b)
            if i==0: ax_row[2].set_title("Flow (Black BG)")
            
            # 4. White BG
            ax_row[3].imshow(w)
            if i==0: ax_row[3].set_title("Flow (White BG)")
            
            for ax in ax_row:
                ax.set_xticks([]); ax.set_yticks([])
                for spine in ax.spines.values(): spine.set_edgecolor('#444444')

        plt.suptitle("Impact of Green Anchor Strength on Omnipose Flows", fontsize=16, color='white')
        plt.tight_layout()
        plt.show()

        # --- PLOT 2: 3D PLOTLY GEOMETRY ---
        print("4. Plotting 3D Geometry...")
        fig3d = go.Figure()
        
        max_jz = np.max(jz_gamut[:,0])
        sc = np.array([1.0/max_jz, 1.0, 1.0])
        def mc(a): return a[:,1], a[:,2], a[:,0] # az, bz, Jz
        
        # A. Gamut Edges
        corners = [[0,0,0], [1,0,0], [1,1,0], [0,1,0], [0,0,1], [1,0,1], [1,1,1], [0,1,1]]
        edges_idx = [[0,1], [1,2], [2,3], [3,0], [4,5], [5,6], [6,7], [7,4], [0,4], [1,5], [2,6], [3,7]]
        for (s, e) in edges_idx:
            line_pts = np.linspace(corners[s], corners[e], 25)
            line_jz = xyz_to_jzazbz(srgb_to_xyz(line_pts)) * sc
            xl, yl, zl = mc(line_jz)
            fig3d.add_trace(go.Scatter3d(x=xl, y=yl, z=zl, mode='lines', line=dict(color='gray', width=2), showlegend=False))
        fig3d.add_trace(go.Scatter3d(x=[None], y=[None], z=[None], mode='lines', line=dict(color='gray', width=2), name='RGB Edges'))

        # B. Plot Results (Loops + Anchors)
        for res in results:
            # Loop
            hx, hy, hz = mc(res['hoop_jz'] * sc)
            fig3d.add_trace(go.Scatter3d(
                x=hx, y=hy, z=hz, mode='lines', 
                line=dict(color=res['color'], width=6),
                name=f"{res['name']} Loop"
            ))
            # Anchor
            ax, ay, az = mc(np.array([res['anchor']]) * sc)
            fig3d.add_trace(go.Scatter3d(
                x=ax, y=ay, z=az, mode='markers',
                marker=dict(color=res['color'], size=5, symbol='x'),
                name=f"{res['name']} Anchor", showlegend=False
            ))
            
        # C. Green Vertex
        gx, gy, gz = mc(np.array([green_vertex]) * sc)
        fig3d.add_trace(go.Scatter3d(x=gx, y=gy, z=gz, mode='markers', marker=dict(color='green', size=8), name='Green Vertex'))

        fig3d.update_layout(
            template="plotly_dark", 
            title="3D Jzazbz: Anchor Strength Comparison",
            scene=dict(
                xaxis_title='az', yaxis_title='bz', zaxis_title='Jz',
                aspectmode='manual', aspectratio=dict(x=1,y=1,z=1),
                camera=dict(eye=dict(x=1.5, y=1.5, z=0.5))
            )
        )
        fig3d.show()

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import scipy.optimize as opt
from scipy.spatial import ConvexHull
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import omnirefactor.core
from skimage import io
import fastremap

# ==========================================
# 1. Color Math (Strict & Raw)
# ==========================================

def srgb_to_xyz(rgb):
    mask = rgb > 0.04045
    linear = np.zeros_like(rgb)
    linear[mask] = ((rgb[mask] + 0.055) / 1.055) ** 2.4
    linear[~mask] = rgb[~mask] / 12.92
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    return linear @ M.T

def xyz_to_jzazbz(xyz):
    b, g, d, d0 = 1.15, 0.66, -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    lms = xyz @ M1.T
    lms_norm = (lms * 200) / 10000.0 
    lms_prime = np.sign(lms_norm) * (((c1 + c2 * (np.abs(lms_norm) ** n)) / (1 + c3 * (np.abs(lms_norm) ** n))) ** p)
    izazbz = lms_prime @ M2.T
    Jz = ((1 + d) * izazbz[:, 0]) / (1 + d * izazbz[:, 0]) - d0
    return np.column_stack((Jz, izazbz[:, 1], izazbz[:, 2]))

def jzazbz_to_xyz(jzazbz):
    Jz, az, bz = jzazbz[:,0], jzazbz[:,1], jzazbz[:,2]
    d, d0 = -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    Jp = Jz + d0; Iz = Jp / (1 + d - d * Jp)
    izazbz_vec = np.stack([Iz, az, bz], axis=1)
    lms_prime = izazbz_vec @ np.linalg.inv(M2).T
    sign_lms = np.sign(lms_prime)
    Y = np.abs(lms_prime) ** (1/p)
    num = Y - c1; den = c2 - Y * c3
    An = np.clip(num / den, 0, None) 
    lms_norm = sign_lms * (An ** (1/n))
    lms = (lms_norm * 10000.0) / 200.0
    return lms @ np.linalg.inv(M1).T

def xyz_to_srgb_raw(xyz):
    M_inv = np.linalg.inv(np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]]))
    linear = xyz @ M_inv.T
    srgb = np.zeros_like(linear)
    mask = linear > 0.0031308
    srgb[mask] = 1.055 * (linear[mask] ** (1/2.4)) - 0.055
    srgb[~mask] = 12.92 * linear[~mask]
    return srgb

def xyz_to_srgb(xyz): return np.clip(xyz_to_srgb_raw(xyz), 0, 1)
def srgb_to_linear_np(srgb): return np.where(srgb <= 0.04045, srgb / 12.92, ((srgb + 0.055) / 1.055) ** 2.4)
def linear_to_srgb_np(lin):
    s = np.zeros_like(lin)
    mask = lin > 0.0031308
    s[mask] = 1.055 * (lin[mask]**(1/2.4)) - 0.055
    s[~mask] = 12.92 * lin[~mask]
    return np.clip(s, 0, 1)

def get_standard_spectral_locus():
    # 2-deg observer, 380-730nm
    cmf = np.array([
        [0.0014,0.0000,0.0065], [0.0042,0.0001,0.0201], [0.0143,0.0004,0.0679], [0.0435,0.0012,0.2074],
        [0.1344,0.0040,0.6456], [0.2839,0.0116,1.3856], [0.3483,0.0230,1.7471], [0.3362,0.0380,1.7721],
        [0.2908,0.0600,1.6692], [0.1954,0.0910,1.2876], [0.0956,0.1390,0.8130], [0.0320,0.2080,0.4652],
        [0.0049,0.3230,0.2720], [0.0093,0.5030,0.1582], [0.0633,0.7100,0.0782], [0.1655,0.8620,0.0422],
        [0.2904,0.9540,0.0203], [0.4334,0.9950,0.0087], [0.5945,0.9950,0.0039], [0.7621,0.9520,0.0021],
        [0.9163,0.8700,0.0017], [1.0263,0.7570,0.0011], [1.0622,0.6310,0.0008], [1.0026,0.5030,0.0006],
        [0.8544,0.3810,0.0002], [0.6424,0.2650,0.0000], [0.4479,0.1750,0.0000], [0.2835,0.1070,0.0000],
        [0.1649,0.0610,0.0000], [0.0874,0.0320,0.0000], [0.0468,0.0170,0.0000], [0.0227,0.0082,0.0000],
        [0.0114,0.0041,0.0000], [0.0058,0.0021,0.0000], [0.0029,0.0010,0.0000], [0.0014,0.0005,0.0000]
    ])
    start_pt, end_pt = cmf[-1], cmf[0]
    purples = np.linspace(start_pt, end_pt, 20)
    return np.vstack([cmf, purples])

# ==========================================
# 2. Optimization & Inflation
# ==========================================

def get_gamut_surface(res=16):
    t = np.linspace(0, 1, res)
    g = np.meshgrid(t, t)
    faces = [np.stack([g[0], g[1], np.zeros_like(g[0])], -1), np.stack([g[0], g[1], np.ones_like(g[0])], -1),
             np.stack([g[0], np.zeros_like(g[0]), g[1]], -1), np.stack([g[0], np.ones_like(g[0]), g[1]], -1),
             np.stack([np.zeros_like(g[0]), g[0], g[1]], -1), np.stack([np.ones_like(g[0]), g[0], g[1]], -1)]
    return np.vstack([f.reshape(-1, 3) for f in faces])

def fit_ellipsoid_anchored_solver(points, anchor_point):
    # 1. Standard Convex Optimization
    hull = ConvexHull(points)
    A = hull.equations[:, :3]; b_const = -hull.equations[:, 3]
    scale = 100.0; A_scaled = A / scale; anchor_scaled = anchor_point * scale
    c0 = np.mean(points, axis=0) * scale
    c0 = 0.5 * c0 + 0.5 * anchor_scaled 
    x0 = np.concatenate([c0, np.eye(3).flatten() * 0.05])

    def objective(x):
        M = x[3:].reshape(3, 3); sign, val = np.linalg.slogdet(M)
        return -val if sign > 0 else 1e9

    def constraints(x):
        d, M = x[:3], x[3:].reshape(3, 3)
        hull_con = b_const - (A_scaled @ d + np.linalg.norm(A_scaled @ M, axis=1))
        try: u = np.linalg.solve(M, anchor_scaled - d); anchor_con = 1.0 - np.linalg.norm(u)
        except: anchor_con = -1.0
        return np.concatenate([hull_con, [anchor_con]])

    res = opt.minimize(objective, x0, method='SLSQP', constraints={'type': 'ineq', 'fun': constraints}, options={'maxiter': 1000})
    return res.x[:3]/scale, res.x[3:].reshape(3, 3)/scale

def calculate_shadow_boundary(center, M, view_vector, res=100):
    w = np.linalg.solve(M, view_vector); w = w / np.linalg.norm(w)
    tmp = np.array([0.0, 1.0, 0.0]) if np.abs(w[1]) < 0.9 else np.array([0.0, 0.0, 1.0])
    s1 = np.cross(w, tmp); s1 /= np.linalg.norm(s1); s2 = np.cross(w, s1)
    theta = np.linspace(0, 2*np.pi, res)
    return center + (np.outer(np.cos(theta), s1) + np.outer(np.sin(theta), s2)) @ M.T

def inflate_to_max_gamut(center, M_init):
    """
    Takes a safe, valid ellipsoid and blows it up until the Hoop actually hits the sRGB wall.
    Uses raw RGB math, bypassing the inaccuracies of the convex hull solver.
    """
    print("--- Inflating Ellipsoid to True Gamut ---")
    scale = 1.0
    step = 0.05
    best_M = M_init.copy()
    
    # 1. Coarse Expansion
    for _ in range(50):
        test_M = M_init * (scale + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        
        # Check strict gamut: -0.001 to 1.001
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001:
            break # Hit the wall
        
        scale += step
        best_M = test_M
    
    # 2. Fine Expansion
    step = 0.005
    current_M = best_M
    for _ in range(20):
        test_M = current_M * (1.0 + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001:
            break 
        current_M = test_M
        
    print(f"   Final Inflation Factor: {scale:.2f}x (approx)")
    return current_M

def check_gamut_compliance(rgb_loop):
    min_val = np.min(rgb_loop)
    max_val = np.max(rgb_loop)
    print(f"   Final RGB Check: [{min_val:.4f}, {max_val:.4f}]")
    if min_val < -0.002 or max_val > 1.002:
        print("   >>> WARNING: Slightly out of gamut (expected at max boundary)")
    else:
        print("   >>> PASS: Perfectly maximal")

# ==========================================
# 3. Omnipose Helpers
# ==========================================
def align_ring_red_to_green(ring_lin):
    n = len(ring_lin)
    red_ref = np.array([1.0, 0.0, 0.0])
    idx_r = np.argmin(np.linalg.norm(ring_lin - red_ref, axis=1))
    ring_aligned = np.roll(ring_lin, -idx_r, axis=0)
    green_ref = np.array([0.0, 1.0, 0.0])
    if np.linalg.norm(ring_aligned[(2*n)//3]-green_ref) < np.linalg.norm(ring_aligned[n//3]-green_ref):
        ring_aligned = ring_aligned[::-1]
    return ring_aligned

def build_hue_disk_from_ring(rgb_ring_lin, center_lin, size=256):
    y, x = np.mgrid[-1:1:size*1j, -1:1:size*1j]
    mag = np.sqrt(x*x + y*y)
    angle = np.mod(np.arctan2(y, x), 2*np.pi)
    n_hues = rgb_ring_lin.shape[0]
    hue_f = angle/(2*np.pi)*n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)
    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    rgb_lin = (1 - mag[..., None]) * center_lin + mag[..., None] * ring_interp
    rgb_lin[mag > 1] = 0
    return np.clip(linear_to_srgb_np(rgb_lin), 0, 1)

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    disk = build_hue_disk_from_ring(rgb_ring_lin, center_lin)
    u = flows[0].cpu().numpy()
    v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi)
    mag = np.sqrt(u*u + v*v)
    mag /= (mag.max() + 1e-8)
    n_hues = rgb_ring_lin.shape[0]
    hue_f = angle/(2*np.pi)*n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)
    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    r = mag[..., None]
    rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow), 0, 1)
    alpha = mag[..., None]
    white = np.ones_like(flow_rgb)
    black = np.zeros_like(flow_rgb)
    flow_white = alpha * flow_rgb + (1 - alpha) * white
    flow_black = alpha * flow_rgb + (1 - alpha) * black
    return disk, flow_rgb, flow_white, flow_black

# ==========================================
# 4. Main Execution
# ==========================================

def main():
    device_str = "cpu"
    dev = torch.device(device_str)
    
    print("1. Calculating Geometries...")
    surf_pts = get_gamut_surface(40)
    jz_gamut = xyz_to_jzazbz(srgb_to_xyz(surf_pts))
    
    centroid = np.mean(jz_gamut, axis=0)
    green_vertex = xyz_to_jzazbz(srgb_to_xyz(np.array([[0.0, 1.0, 0.0]])))[0]
    
    # Use strong anchor to orient, but rely on inflation for size
    anchor = centroid + 0.60 * (green_vertex - centroid)
    
    print("2. Solving Geometry (Convex Hull)...")
    center, M_solver = fit_ellipsoid_anchored_solver(jz_gamut, anchor)
    
    print("3. Inflating to True sRGB Wall...")
    M_max = inflate_to_max_gamut(center, M_solver)
    
    # Generate Final Hoop
    hoop = calculate_shadow_boundary(center, M_max, np.array([1.0, 0.0, 0.0]), res=72)
    rgb_hoop_raw = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
    check_gamut_compliance(rgb_hoop_raw)
    
    rgb_hoop_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(rgb_hoop_raw, 0, 1)))
    
    # Sinebow
    t = np.linspace(0, 1, 72, endpoint=False)
    sr = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0/3))
    sg = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1/3))
    sb = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2/3))
    rgb_sine_lin = align_ring_red_to_green(srgb_to_linear_np(np.stack([sr, sg, sb], axis=1)))

    print("4. Omnipose Rendering...")
    omnidir = Path(omnirefactor.__file__).resolve().parent.parent.parent
    basedir = omnidir / "docs" / "_static"
    name = "ecoli"
    ext = ".tif"

    try:
        masks = io.imread(str(basedir / f"{name}_labels{ext}"))
        slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows = omnirefactor.core.masks_to_flows(masks, device=device_str)[4].to(dev)
        center_lin = srgb_to_linear_np(np.array([0.5, 0.5, 0.5]))

        def gen_set(ring, rotation_steps=0):
            r = np.roll(ring, rotation_steps, axis=0)
            return make_flow_images_for_ring(r, center_lin, flows)

        disk_s_0, _, w_s_0, b_s_0 = gen_set(rgb_sine_lin, 0)
        disk_e_0, _, w_e_0, b_e_0 = gen_set(rgb_hoop_lin, 0)
        rot = 18
        _, _, w_s_90, b_s_90 = gen_set(rgb_sine_lin, rot)
        _, _, w_e_90, b_e_90 = gen_set(rgb_hoop_lin, rot)

        print("5. Plotting 2D...")
        plt.style.use('dark_background')
        fig, axes = plt.subplots(2, 5, figsize=(20, 8))
        rows = [("Sinebow", disk_s_0, b_s_0, w_s_0, b_s_90, w_s_90),
                ("Maximal Ellipse", disk_e_0, b_e_0, w_e_0, b_e_90, w_e_90)]
        titles = ["Hue Disk", "Black (0°)", "White (0°)", "Black (90°)", "White (90°)"]

        for i, (name, disk, b0, w0, b90, w90) in enumerate(rows):
            ax_row = axes[i]
            ax_row[0].imshow(disk)
            ax_row[0].set_ylabel(name, fontsize=14, color='white', rotation=90, labelpad=20)
            ax_row[1].imshow(b0); ax_row[2].imshow(w0)
            ax_row[3].imshow(b90); ax_row[4].imshow(w90)
            for j, ax in enumerate(ax_row):
                if i == 0: ax.set_title(titles[j], fontsize=12, color='#dddddd')
                ax.set_xticks([]); ax.set_yticks([])
                for spine in ax.spines.values(): spine.set_edgecolor('#333333')
        plt.tight_layout()
        plt.show()
        
        print("6. Generating 3D...")
        fig3d = go.Figure()
        
        max_jz = np.max(jz_gamut[:,0])
        sc = np.array([1.0/max_jz, 1.0, 1.0]) 
        def mc(a): return a[:,1], a[:,2], a[:,0]

        # A. Shaded Faces
        N = 10; t = np.linspace(0, 1, N); g = np.meshgrid(t, t)
        rgb_faces = [np.stack([g[0], g[1], np.zeros_like(g[0])], -1), np.stack([g[0], g[1], np.ones_like(g[0])], -1),
                     np.stack([g[0], np.zeros_like(g[0]), g[1]], -1), np.stack([g[0], np.ones_like(g[0]), g[1]], -1),
                     np.stack([np.zeros_like(g[0]), g[0], g[1]], -1), np.stack([np.ones_like(g[0]), g[0], g[1]], -1)]
        i_idx, j_idx, k_idx = [], [], []
        for r in range(N-1):
            for c in range(N-1):
                p1 = r*N+c; p2 = r*N+(c+1); p3 = (r+1)*N+c; p4 = (r+1)*N+(c+1)
                i_idx.extend([p1, p2]); j_idx.extend([p2, p4]); k_idx.extend([p3, p3])

        for i, face in enumerate(rgb_faces):
            jz_face = xyz_to_jzazbz(srgb_to_xyz(face.reshape(-1, 3))) * sc
            x_f, y_f, z_f = mc(jz_face)
            fig3d.add_trace(go.Mesh3d(x=x_f, y=y_f, z=z_f, i=i_idx, j=j_idx, k=k_idx, color='gray', opacity=0.15, flatshading=False,
                lighting=dict(ambient=0.1, diffuse=0.8, specular=0.2), hoverinfo='skip'))

        # B. Edges
        corners = [[0,0,0], [1,0,0], [1,1,0], [0,1,0], [0,0,1], [1,0,1], [1,1,1], [0,1,1]]
        edges_idx = [[0,1], [1,2], [2,3], [3,0], [4,5], [5,6], [6,7], [7,4], [0,4], [1,5], [2,6], [3,7]]
        for (s, e) in edges_idx:
            line_pts = np.linspace(corners[s], corners[e], 25)
            line_jz = xyz_to_jzazbz(srgb_to_xyz(line_pts)) * sc
            xl, yl, zl = mc(line_jz)
            fig3d.add_trace(go.Scatter3d(x=xl, y=yl, z=zl, mode='lines', line=dict(color='white', width=3), showlegend=False))

        # C. Ellipsoid (Maximal)
        u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
        sph = np.stack([np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v)], axis=-1).reshape(-1, 3)
        ell = (sph @ M_max.T + center) * sc
        ex, ey, ez = mc(ell)
        fig3d.add_trace(go.Surface(x=ex.reshape(u.shape), y=ey.reshape(u.shape), z=ez.reshape(u.shape),
                                colorscale=[(0, '#88aa88'), (1, '#88aa88')], opacity=0.4, showscale=False, name='Maximal Ellipsoid'))

        # D. Loops
        hx, hy, hz = mc(hoop * sc)
        c_hoop = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in linear_to_srgb_np(np.clip(rgb_hoop_raw,0,1))]
        fig3d.add_trace(go.Scatter3d(x=hx, y=hy, z=hz, mode='markers', marker=dict(color=c_hoop, size=5), name='Ellipsoid Loop'))

        rgb_sine_srgb = linear_to_srgb_np(rgb_sine_lin)
        jz_sine = xyz_to_jzazbz(srgb_to_xyz(rgb_sine_srgb)) * sc
        sx, sy, sz = mc(jz_sine)
        c_sine = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in rgb_sine_srgb]
        fig3d.add_trace(go.Scatter3d(x=sx, y=sy, z=sz, mode='markers', marker=dict(color=c_sine, size=4, symbol='diamond'), name='Sinebow'))
        
        # E. Locus
        locus_xyz = get_standard_spectral_locus()
        locus_jz = xyz_to_jzazbz(locus_xyz) * sc
        lx, ly, lz = mc(locus_jz)
        fig3d.add_trace(go.Scatter3d(x=lx, y=ly, z=lz, mode='lines', line=dict(color='cyan', width=5), name='Spectral Locus'))

        fig3d.update_layout(template="plotly_dark", title="3D Jzazbz (Maximal + Inflated)", 
                            scene=dict(xaxis_title='az', yaxis_title='bz', zaxis_title='Jz',
                                       aspectmode='manual', aspectratio=dict(x=1,y=1,z=1)))
        fig3d.show()

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import scipy.optimize as opt
from scipy.spatial import ConvexHull
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import omnirefactor.core
from skimage import io
import fastremap

# ==========================================
# 1. Color Math
# ==========================================
def srgb_to_xyz(rgb):
    mask = rgb > 0.04045
    linear = np.zeros_like(rgb)
    linear[mask] = ((rgb[mask] + 0.055) / 1.055) ** 2.4
    linear[~mask] = rgb[~mask] / 12.92
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    return linear @ M.T

def xyz_to_jzazbz(xyz):
    # Safdar 2017
    b, g, d, d0 = 1.15, 0.66, -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    lms = xyz @ M1.T
    lms_norm = (lms * 200) / 10000.0 
    lms_prime = np.sign(lms_norm) * (((c1 + c2 * (np.abs(lms_norm) ** n)) / (1 + c3 * (np.abs(lms_norm) ** n))) ** p)
    izazbz = lms_prime @ M2.T
    Jz = ((1 + d) * izazbz[:, 0]) / (1 + d * izazbz[:, 0]) - d0
    return np.column_stack((Jz, izazbz[:, 1], izazbz[:, 2]))

def jzazbz_to_xyz(jzazbz):
    Jz, az, bz = jzazbz[:,0], jzazbz[:,1], jzazbz[:,2]
    d, d0 = -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    Jp = Jz + d0; Iz = Jp / (1 + d - d * Jp)
    izazbz_vec = np.stack([Iz, az, bz], axis=1)
    lms_prime = izazbz_vec @ np.linalg.inv(M2).T
    sign_lms = np.sign(lms_prime)
    Y = np.abs(lms_prime) ** (1/p)
    num = Y - c1; den = c2 - Y * c3
    An = np.clip(num / den, 0, None) 
    lms_norm = sign_lms * (An ** (1/n))
    lms = (lms_norm * 10000.0) / 200.0
    return lms @ np.linalg.inv(M1).T

def xyz_to_srgb_raw(xyz):
    M_inv = np.linalg.inv(np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]]))
    linear = xyz @ M_inv.T
    srgb = np.zeros_like(linear)
    mask = linear > 0.0031308
    srgb[mask] = 1.055 * (linear[mask] ** (1/2.4)) - 0.055
    srgb[~mask] = 12.92 * linear[~mask]
    return srgb

def xyz_to_srgb(xyz): return np.clip(xyz_to_srgb_raw(xyz), 0, 1)
def srgb_to_linear_np(srgb): return np.where(srgb <= 0.04045, srgb / 12.92, ((srgb + 0.055) / 1.055) ** 2.4)
def linear_to_srgb_np(lin):
    s = np.zeros_like(lin)
    mask = lin > 0.0031308
    s[mask] = 1.055 * (lin[mask]**(1/2.4)) - 0.055
    s[~mask] = 12.92 * lin[~mask]
    return np.clip(s, 0, 1)

def get_standard_spectral_locus():
    # CIE 1931 2-deg (380-730nm)
    cmf = np.array([
        [0.0014,0.0000,0.0065], [0.0042,0.0001,0.0201], [0.0143,0.0004,0.0679], [0.0435,0.0012,0.2074],
        [0.1344,0.0040,0.6456], [0.2839,0.0116,1.3856], [0.3483,0.0230,1.7471], [0.3362,0.0380,1.7721],
        [0.2908,0.0600,1.6692], [0.1954,0.0910,1.2876], [0.0956,0.1390,0.8130], [0.0320,0.2080,0.4652],
        [0.0049,0.3230,0.2720], [0.0093,0.5030,0.1582], [0.0633,0.7100,0.0782], [0.1655,0.8620,0.0422],
        [0.2904,0.9540,0.0203], [0.4334,0.9950,0.0087], [0.5945,0.9950,0.0039], [0.7621,0.9520,0.0021],
        [0.9163,0.8700,0.0017], [1.0263,0.7570,0.0011], [1.0622,0.6310,0.0008], [1.0026,0.5030,0.0006],
        [0.8544,0.3810,0.0002], [0.6424,0.2650,0.0000], [0.4479,0.1750,0.0000], [0.2835,0.1070,0.0000],
        [0.1649,0.0610,0.0000], [0.0874,0.0320,0.0000], [0.0468,0.0170,0.0000], [0.0227,0.0082,0.0000],
        [0.0114,0.0041,0.0000], [0.0058,0.0021,0.0000], [0.0029,0.0010,0.0000], [0.0014,0.0005,0.0000]
    ])
    # Close the loop (Line of Purples)
    start_pt = cmf[-1]; end_pt = cmf[0]
    purples = np.linspace(start_pt, end_pt, 20)
    return np.vstack([cmf, purples])

# ==========================================
# 2. Optimization (Maximal + Inflated)
# ==========================================

def get_gamut_surface(res=16):
    t = np.linspace(0, 1, res)
    g = np.meshgrid(t, t)
    faces = [np.stack([g[0], g[1], np.zeros_like(g[0])], -1), np.stack([g[0], g[1], np.ones_like(g[0])], -1),
             np.stack([g[0], np.zeros_like(g[0]), g[1]], -1), np.stack([g[0], np.ones_like(g[0]), g[1]], -1),
             np.stack([np.zeros_like(g[0]), g[0], g[1]], -1), np.stack([np.ones_like(g[0]), g[0], g[1]], -1)]
    pts = np.vstack([f.reshape(-1, 3) for f in faces])
    corners = np.array([[0,0,0], [1,0,0], [0,1,0], [0,0,1], [1,1,0], [1,0,1], [0,1,1], [1,1,1]])
    return np.vstack([pts, corners])

def fit_ellipsoid_anchored_solver(points, anchor_point):
    hull = ConvexHull(points)
    A = hull.equations[:, :3]; b_const = -hull.equations[:, 3]
    scale = 100.0; A_scaled = A / scale; anchor_scaled = anchor_point * scale
    c0 = np.mean(points, axis=0) * scale
    c0 = 0.5 * c0 + 0.5 * anchor_scaled 
    x0 = np.concatenate([c0, np.eye(3).flatten() * 0.05])

    def objective(x):
        M = x[3:].reshape(3, 3); sign, val = np.linalg.slogdet(M)
        return -val if sign > 0 else 1e9

    def constraints(x):
        d, M = x[:3], x[3:].reshape(3, 3)
        norm_AM = np.linalg.norm(A_scaled @ M, axis=1)
        hull_con = b_const - (A_scaled @ d + norm_AM)
        try: u = np.linalg.solve(M, anchor_scaled - d); anchor_con = 1.0 - np.linalg.norm(u)
        except: anchor_con = -1.0
        return np.concatenate([hull_con, [anchor_con]])

    res = opt.minimize(objective, x0, method='SLSQP', constraints={'type': 'ineq', 'fun': constraints}, options={'maxiter': 1000})
    return res.x[:3]/scale, res.x[3:].reshape(3, 3)/scale

def calculate_shadow_boundary(center, M, view_vector, res=100):
    w = np.linalg.solve(M, view_vector); w = w / np.linalg.norm(w)
    tmp = np.array([0.0, 1.0, 0.0]) if np.abs(w[1]) < 0.9 else np.array([0.0, 0.0, 1.0])
    s1 = np.cross(w, tmp); s1 /= np.linalg.norm(s1); s2 = np.cross(w, s1)
    theta = np.linspace(0, 2*np.pi, res)
    return center + (np.outer(np.cos(theta), s1) + np.outer(np.sin(theta), s2)) @ M.T

def inflate_to_max_gamut(center, M_init):
    scale = 1.0; step = 0.05; best_M = M_init.copy()
    for _ in range(50):
        test_M = M_init * (scale + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        scale += step; best_M = test_M
    step = 0.005; current_M = best_M
    for _ in range(20):
        test_M = current_M * (1.0 + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        current_M = test_M
    return current_M

def fit_locus_in_gamut_jz(locus_xyz_raw, center_jz):
    """
    Shrinks the spectral locus (XYZ) towards the gamut centroid (Jzazbz)
    until it fits inside sRGB.
    """
    locus_jz = xyz_to_jzazbz(locus_xyz_raw)
    
    scale = 1.0
    step = 0.02
    
    for _ in range(100):
        # Interpolate in Jzazbz space
        shrunk_jz = center_jz + scale * (locus_jz - center_jz)
        
        # Check Validity
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(shrunk_jz))
        if np.min(rgb) >= -0.001 and np.max(rgb) <= 1.001:
            print(f"   Locus Fit Scale: {scale:.3f}x")
            return shrunk_jz # Return Jz coordinates
            
        scale -= step
    return locus_jz

def check_gamut_compliance(rgb_loop):
    min_val = np.min(rgb_loop); max_val = np.max(rgb_loop)
    tol = 1e-3
    if min_val < -tol or max_val > 1.0 + tol:
        print(f"   >>> FAIL: Loop range [{min_val:.4f}, {max_val:.4f}]")
    else:
        print(f"   >>> PASS: Loop is within sRGB")

# ==========================================
# 3. Omnipose Helpers
# ==========================================
def align_ring_red_to_green(ring_lin):
    n = len(ring_lin)
    red_ref = np.array([1.0, 0.0, 0.0])
    idx_r = np.argmin(np.linalg.norm(ring_lin - red_ref, axis=1))
    ring_aligned = np.roll(ring_lin, -idx_r, axis=0)
    green_ref = np.array([0.0, 1.0, 0.0])
    if np.linalg.norm(ring_aligned[(2*n)//3]-green_ref) < np.linalg.norm(ring_aligned[n//3]-green_ref):
        ring_aligned = ring_aligned[::-1]
    return ring_aligned

def build_hue_disk_from_ring(rgb_ring_lin, center_lin, size=256):
    y, x = np.mgrid[-1:1:size*1j, -1:1:size*1j]
    mag = np.sqrt(x*x + y*y)
    angle = np.mod(np.arctan2(y, x), 2*np.pi)
    n_hues = rgb_ring_lin.shape[0]
    hue_f = angle/(2*np.pi)*n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)
    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    rgb_lin = (1 - mag[..., None]) * center_lin + mag[..., None] * ring_interp
    rgb_lin[mag > 1] = 0
    return np.clip(linear_to_srgb_np(rgb_lin), 0, 1)

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    disk = build_hue_disk_from_ring(rgb_ring_lin, center_lin)
    u = flows[0].cpu().numpy()
    v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi)
    mag = np.sqrt(u*u + v*v)
    mag /= (mag.max() + 1e-8)
    n_hues = rgb_ring_lin.shape[0]
    hue_f = angle/(2*np.pi)*n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)
    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    r = mag[..., None]
    rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow), 0, 1)
    alpha = mag[..., None]
    white = np.ones_like(flow_rgb)
    black = np.zeros_like(flow_rgb)
    flow_white = alpha * flow_rgb + (1 - alpha) * white
    flow_black = alpha * flow_rgb + (1 - alpha) * black
    return disk, flow_rgb, flow_white, flow_black

# ==========================================
# 4. Main Execution
# ==========================================

def main():
    device_str = "cpu"
    dev = torch.device(device_str)
    
    print("1. Calculating Geometries...")
    surf_pts_opt = get_gamut_surface(40) 
    jz_gamut_opt = xyz_to_jzazbz(srgb_to_xyz(surf_pts_opt))
    
    centroid = np.mean(jz_gamut_opt, axis=0)
    green_vertex = xyz_to_jzazbz(srgb_to_xyz(np.array([[0.0, 1.0, 0.0]])))[0]
    anchor = centroid + 0.60 * (green_vertex - centroid)
    
    print("2. Fitting Maximal Ellipsoid...")
    center, M_solver = fit_ellipsoid_anchored_solver(jz_gamut_opt, anchor)
    M_max = inflate_to_max_gamut(center, M_solver)
    
    hoop = calculate_shadow_boundary(center, M_max, np.array([1.0, 0.0, 0.0]), res=72)
    rgb_hoop_raw = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
    check_gamut_compliance(rgb_hoop_raw)
    rgb_hoop_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(rgb_hoop_raw, 0, 1)))
    
    t = np.linspace(0, 1, 72, endpoint=False)
    sr = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0/3))
    sg = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1/3))
    sb = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2/3))
    rgb_sine_lin = align_ring_red_to_green(srgb_to_linear_np(np.stack([sr, sg, sb], axis=1)))

    # --- Shrunk Locus (Jz Shrink) ---
    print("3. Calculating Shrunk Locus...")
    locus_xyz = get_standard_spectral_locus()
    shrunk_locus_jz = fit_locus_in_gamut_jz(locus_xyz, centroid) # Use Gamut Centroid
    rgb_locus_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(shrunk_locus_jz)), 0, 1)))

    print("4. Loading Omnipose Flows...")
    omnidir = Path(omnirefactor.__file__).resolve().parent.parent.parent
    basedir = omnidir / "docs" / "_static"
    name = "ecoli"
    ext = ".tif"

    try:
        masks = io.imread(str(basedir / f"{name}_labels{ext}"))
        slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows = omnirefactor.core.masks_to_flows(masks, device=device_str)[4].to(dev)
        center_lin = srgb_to_linear_np(np.array([0.5, 0.5, 0.5]))

        def gen_set(ring, rotation_steps=0):
            r = np.roll(ring, rotation_steps, axis=0)
            return make_flow_images_for_ring(r, center_lin, flows)

        disk_s_0, _, w_s_0, b_s_0 = gen_set(rgb_sine_lin, 0)
        disk_e_0, _, w_e_0, b_e_0 = gen_set(rgb_hoop_lin, 0)
        disk_l_0, _, w_l_0, b_l_0 = gen_set(rgb_locus_lin, 0)

        rot = 18
        _, _, w_s_90, b_s_90 = gen_set(rgb_sine_lin, rot)
        _, _, w_e_90, b_e_90 = gen_set(rgb_hoop_lin, rot)
        _, _, w_l_90, b_l_90 = gen_set(rgb_locus_lin, rot)

        print("5. Plotting 2D...")
        plt.style.use('dark_background')
        fig, axes = plt.subplots(3, 5, figsize=(20, 12))
        rows = [("Sinebow", disk_s_0, b_s_0, w_s_0, b_s_90, w_s_90),
                ("Anchor Ellipse", disk_e_0, b_e_0, w_e_0, b_e_90, w_e_90),
                ("Shrunk Locus", disk_l_0, b_l_0, w_l_0, b_l_90, w_l_90)]
        titles = ["Hue Disk", "Black (0°)", "White (0°)", "Black (90°)", "White (90°)"]

        for i, (name, disk, b0, w0, b90, w90) in enumerate(rows):
            ax_row = axes[i]
            ax_row[0].imshow(disk)
            ax_row[0].set_ylabel(name, fontsize=14, color='white', rotation=90, labelpad=20)
            ax_row[1].imshow(b0); ax_row[2].imshow(w0)
            ax_row[3].imshow(b90); ax_row[4].imshow(w90)
            for j, ax in enumerate(ax_row):
                if i == 0: ax.set_title(titles[j], fontsize=12, color='#dddddd')
                ax.set_xticks([]); ax.set_yticks([])
        plt.tight_layout()
        plt.show()
        
        print("6. Generating 3D...")
        fig3d = go.Figure()
        max_jz = np.max(jz_gamut_opt[:,0]); sc = np.array([1.0/max_jz, 1.0, 1.0]); 
        def mc(a): return a[:,1], a[:,2], a[:,0]

        # Faces
        N=10; t=np.linspace(0,1,N); g=np.meshgrid(t,t)
        rgb_faces = [np.stack([g[0],g[1],np.zeros_like(g[0])],-1), np.stack([g[0],g[1],np.ones_like(g[0])],-1),
                     np.stack([g[0],np.zeros_like(g[0]),g[1]],-1), np.stack([g[0],np.ones_like(g[0]),g[1]],-1),
                     np.stack([np.zeros_like(g[0]),g[0],g[1]],-1), np.stack([np.ones_like(g[0]),g[0],g[1]],-1)]
        i_idx, j_idx, k_idx = [], [], []
        for r in range(N-1):
            for c in range(N-1):
                p1=r*N+c; p2=r*N+(c+1); p3=(r+1)*N+c; p4=(r+1)*N+(c+1)
                i_idx.extend([p1,p2]); j_idx.extend([p2,p4]); k_idx.extend([p3,p3])
        for i, face in enumerate(rgb_faces):
            jz_face = xyz_to_jzazbz(srgb_to_xyz(face.reshape(-1,3))) * sc
            x,y,z = mc(jz_face)
            fig3d.add_trace(go.Mesh3d(x=x, y=y, z=z, i=i_idx, j=j_idx, k=k_idx, color='gray', opacity=0.15, flatshading=False, hoverinfo='skip'))

        # Edges
        corners = [[0,0,0],[1,0,0],[1,1,0],[0,1,0],[0,0,1],[1,0,1],[1,1,1],[0,1,1]]
        edges_idx = [[0,1],[1,2],[2,3],[3,0],[4,5],[5,6],[6,7],[7,4],[0,4],[1,5],[2,6],[3,7]]
        for (s,e) in edges_idx:
            pts = np.linspace(corners[s],corners[e],25)
            line = xyz_to_jzazbz(srgb_to_xyz(pts)) * sc
            x,y,z = mc(line)
            fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', line=dict(color='white', width=3), showlegend=False))

        # Ellipsoid
        u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
        sph = np.stack([np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v)], axis=-1).reshape(-1, 3)
        ell = (sph @ M_max.T + center) * sc
        ex, ey, ez = mc(ell)
        fig3d.add_trace(go.Surface(x=ex.reshape(u.shape), y=ey.reshape(u.shape), z=ez.reshape(u.shape),
                                colorscale=[(0,'#88aa88'),(1,'#88aa88')], opacity=0.4, showscale=False, name='Anchored Ellipsoid'))

        # Loops
        hx, hy, hz = mc(hoop * sc)
        c_hoop = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in linear_to_srgb_np(np.clip(rgb_hoop_raw,0,1))]
        fig3d.add_trace(go.Scatter3d(x=hx, y=hy, z=hz, mode='markers', marker=dict(color=c_hoop, size=5), name='Ellipsoid Loop'))

        sx, sy, sz = mc(xyz_to_jzazbz(srgb_to_xyz(rgb_sine_lin)) * sc)
        c_sine = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in rgb_sine_srgb]
        fig3d.add_trace(go.Scatter3d(x=sx, y=sy, z=sz, mode='markers', marker=dict(color=c_sine, size=4, symbol='diamond'), name='Sinebow'))
        
        lx, ly, lz = mc(shrunk_locus_jz * sc)
        fig3d.add_trace(go.Scatter3d(x=lx, y=ly, z=lz, mode='lines', line=dict(color='cyan', width=5), name='Shrunk Locus'))

        fig3d.update_layout(template="plotly_dark", title="3D Jzazbz (Maximal + Shrunk Locus)", 
                            scene=dict(xaxis_title='az', yaxis_title='bz', zaxis_title='Jz', aspectmode='manual', aspectratio=dict(x=1,y=1,z=1)))
        fig3d.show()

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import scipy.optimize as opt
from scipy.spatial import ConvexHull
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import omnirefactor.core
from skimage import io
import fastremap

# ==========================================
# 1. Color Math
# ==========================================
def srgb_to_xyz(rgb):
    # Expects sRGB (gamma-encoded 0-1)
    mask = rgb > 0.04045
    linear = np.zeros_like(rgb)
    linear[mask] = ((rgb[mask] + 0.055) / 1.055) ** 2.4
    linear[~mask] = rgb[~mask] / 12.92
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    return linear @ M.T

def xyz_to_jzazbz(xyz):
    b, g, d, d0 = 1.15, 0.66, -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    lms = xyz @ M1.T
    lms_norm = (lms * 200) / 10000.0 
    lms_prime = np.sign(lms_norm) * (((c1 + c2 * (np.abs(lms_norm) ** n)) / (1 + c3 * (np.abs(lms_norm) ** n))) ** p)
    izazbz = lms_prime @ M2.T
    Jz = ((1 + d) * izazbz[:, 0]) / (1 + d * izazbz[:, 0]) - d0
    return np.column_stack((Jz, izazbz[:, 1], izazbz[:, 2]))

def jzazbz_to_xyz(jzazbz):
    Jz, az, bz = jzazbz[:,0], jzazbz[:,1], jzazbz[:,2]
    d, d0 = -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    Jp = Jz + d0; Iz = Jp / (1 + d - d * Jp)
    izazbz_vec = np.stack([Iz, az, bz], axis=1)
    lms_prime = izazbz_vec @ np.linalg.inv(M2).T
    sign_lms = np.sign(lms_prime)
    Y = np.abs(lms_prime) ** (1/p)
    num = Y - c1; den = c2 - Y * c3
    An = np.clip(num / den, 0, None) 
    lms_norm = sign_lms * (An ** (1/n))
    lms = (lms_norm * 10000.0) / 200.0
    return lms @ np.linalg.inv(M1).T

def xyz_to_srgb_raw(xyz):
    M_inv = np.linalg.inv(np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]]))
    linear = xyz @ M_inv.T
    srgb = np.zeros_like(linear)
    mask = linear > 0.0031308
    srgb[mask] = 1.055 * (linear[mask] ** (1/2.4)) - 0.055
    srgb[~mask] = 12.92 * linear[~mask]
    return srgb

def xyz_to_srgb(xyz): return np.clip(xyz_to_srgb_raw(xyz), 0, 1)
def srgb_to_linear_np(srgb): return np.where(srgb <= 0.04045, srgb / 12.92, ((srgb + 0.055) / 1.055) ** 2.4)
def linear_to_srgb_np(lin):
    s = np.zeros_like(lin)
    mask = lin > 0.0031308
    s[mask] = 1.055 * (lin[mask]**(1/2.4)) - 0.055
    s[~mask] = 12.92 * lin[~mask]
    return np.clip(s, 0, 1)

def get_full_spectral_locus():
    # Full 380-780nm CMF
    cmf = np.array([
        [0.0014,0.0000,0.0065], [0.0042,0.0001,0.0201], [0.0143,0.0004,0.0679], [0.0435,0.0012,0.2074],
        [0.1344,0.0040,0.6456], [0.2839,0.0116,1.3856], [0.3483,0.0230,1.7471], [0.3362,0.0380,1.7721],
        [0.2908,0.0600,1.6692], [0.1954,0.0910,1.2876], [0.0956,0.1390,0.8130], [0.0320,0.2080,0.4652],
        [0.0049,0.3230,0.2720], [0.0093,0.5030,0.1582], [0.0633,0.7100,0.0782], [0.1655,0.8620,0.0422],
        [0.2904,0.9540,0.0203], [0.4334,0.9950,0.0087], [0.5945,0.9950,0.0039], [0.7621,0.9520,0.0021],
        [0.9163,0.8700,0.0017], [1.0263,0.7570,0.0011], [1.0622,0.6310,0.0008], [1.0026,0.5030,0.0006],
        [0.8544,0.3810,0.0002], [0.6424,0.2650,0.0000], [0.4479,0.1750,0.0000], [0.2835,0.1070,0.0000],
        [0.1649,0.0610,0.0000], [0.0874,0.0320,0.0000], [0.0468,0.0170,0.0000], [0.0227,0.0082,0.0000],
        [0.0114,0.0041,0.0000], [0.0058,0.0021,0.0000], [0.0029,0.0010,0.0000], [0.0014,0.0005,0.0000]
    ])
    # Close Line of Purples
    start_pt, end_pt = cmf[-1], cmf[0]
    purples = np.linspace(start_pt, end_pt, 25)
    return np.vstack([cmf, purples])

# ==========================================
# 2. Optimization
# ==========================================

def get_gamut_surface(res=16):
    t = np.linspace(0, 1, res)
    g = np.meshgrid(t, t)
    faces = [np.stack([g[0], g[1], np.zeros_like(g[0])], -1), np.stack([g[0], g[1], np.ones_like(g[0])], -1),
             np.stack([g[0], np.zeros_like(g[0]), g[1]], -1), np.stack([g[0], np.ones_like(g[0]), g[1]], -1),
             np.stack([np.zeros_like(g[0]), g[0], g[1]], -1), np.stack([np.ones_like(g[0]), g[0], g[1]], -1)]
    pts = np.vstack([f.reshape(-1, 3) for f in faces])
    corners = np.array([[0,0,0], [1,0,0], [0,1,0], [0,0,1], [1,1,0], [1,0,1], [0,1,1], [1,1,1]])
    return np.vstack([pts, corners])

def fit_ellipsoid_anchored_solver(points, anchor_point):
    hull = ConvexHull(points)
    A = hull.equations[:, :3]; b_const = -hull.equations[:, 3]
    scale = 100.0; A_scaled = A / scale; anchor_scaled = anchor_point * scale
    c0 = np.mean(points, axis=0) * scale
    c0 = 0.5 * c0 + 0.5 * anchor_scaled 
    x0 = np.concatenate([c0, np.eye(3).flatten() * 0.05])

    # Strict Hull, No Margin
    def objective(x):
        M = x[3:].reshape(3, 3); sign, val = np.linalg.slogdet(M)
        return -val if sign > 0 else 1e9

    def constraints(x):
        d, M = x[:3], x[3:].reshape(3, 3)
        norm_AM = np.linalg.norm(A_scaled @ M, axis=1)
        hull_con = b_const - (A_scaled @ d + norm_AM)
        try: u = np.linalg.solve(M, anchor_scaled - d); anchor_con = 1.0 - np.linalg.norm(u)
        except: anchor_con = -1.0
        return np.concatenate([hull_con, [anchor_con]])

    res = opt.minimize(objective, x0, method='SLSQP', constraints={'type': 'ineq', 'fun': constraints}, options={'maxiter': 1000})
    return res.x[:3]/scale, res.x[3:].reshape(3, 3)/scale

def calculate_shadow_boundary(center, M, view_vector, res=72):
    w = np.linalg.solve(M, view_vector); w = w / np.linalg.norm(w)
    tmp = np.array([0.0, 1.0, 0.0]) if np.abs(w[1]) < 0.9 else np.array([0.0, 0.0, 1.0])
    s1 = np.cross(w, tmp); s1 /= np.linalg.norm(s1); s2 = np.cross(w, s1)
    theta = np.linspace(0, 2*np.pi, res)
    return center + (np.outer(np.cos(theta), s1) + np.outer(np.sin(theta), s2)) @ M.T

def inflate_to_max_gamut(center, M_init):
    scale = 1.0; step = 0.05; best_M = M_init.copy()
    for _ in range(50):
        test_M = M_init * (scale + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        scale += step; best_M = test_M
    step = 0.005; current_M = best_M
    for _ in range(20):
        test_M = current_M * (1.0 + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        current_M = test_M
    return current_M

def fit_locus_chroma_shrink(locus_xyz_raw, gamut_points_jz):
    """
    1. Maps Locus to Jzazbz.
    2. Adjusts Locus Lightness (Jz) to fit within Gamut range (so it isn't invisible/white).
    3. Shrinks Chroma (az, bz) towards the neutral axis until in Gamut.
    """
    locus_jz_full = xyz_to_jzazbz(locus_xyz_raw)
    
    # 1. Vertical Fit: Normalize Jz to match Gamut Max Jz
    max_gamut_J = np.max(gamut_points_jz[:,0])
    max_locus_J = np.max(locus_jz_full[:,0])
    
    # Scale down lightness if locus is brighter than monitor max
    scaling_J = min(1.0, max_gamut_J / max_locus_J)
    locus_jz_full[:, 0] *= scaling_J * 0.98 # 0.98 margin
    
    # 2. Chroma Shrink (towards 0,0)
    locus_J = locus_jz_full[:, 0]
    locus_C = locus_jz_full[:, 1:] # az, bz
    
    scale = 1.0
    step = 0.01
    
    for _ in range(200):
        shrunk_C = locus_C * scale # Scale towards 0,0 (Neutral)
        shrunk_jz = np.column_stack((locus_J, shrunk_C))
        
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(shrunk_jz))
        
        # Tolerance: We allow small violations for the dark tails (Line of Purples)
        # but enforce strict compliance for the bright spectral parts.
        valid_pts = (rgb >= -0.005) & (rgb <= 1.005)
        compliance = np.mean(np.all(valid_pts, axis=1))
        
        # If 95% of points are valid (allowing for dark noise), we accept
        if compliance > 0.95:
            print(f"   Locus Chroma Scale: {scale:.3f}x (Compliance: {compliance:.1%})")
            return shrunk_jz
            
        scale -= step
        
    return locus_jz_full * 0.1 # Fallback

def check_gamut_compliance(rgb_loop):
    min_val = np.min(rgb_loop); max_val = np.max(rgb_loop)
    tol = 2e-3
    if min_val < -tol or max_val > 1.0 + tol:
        print(f"   >>> FAIL: Range [{min_val:.4f}, {max_val:.4f}]")
    else:
        print(f"   >>> PASS: Loop is within sRGB")

# ==========================================
# 3. Omnipose Helpers
# ==========================================
def align_ring_red_to_green(ring_lin):
    n = len(ring_lin)
    red_ref = np.array([1.0, 0.0, 0.0])
    idx_r = np.argmin(np.linalg.norm(ring_lin - red_ref, axis=1))
    ring_aligned = np.roll(ring_lin, -idx_r, axis=0)
    green_ref = np.array([0.0, 1.0, 0.0])
    if np.linalg.norm(ring_aligned[(2*n)//3]-green_ref) < np.linalg.norm(ring_aligned[n//3]-green_ref):
        ring_aligned = ring_aligned[::-1]
    return ring_aligned

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    def build_disk(ring, center):
        y, x = np.mgrid[-1:1:256j, -1:1:256j]
        mag = np.sqrt(x*x + y*y)
        angle = np.mod(np.arctan2(y, x), 2*np.pi)
        n = ring.shape[0]
        hue_f = angle/(2*np.pi)*n
        idx0 = np.floor(hue_f).astype(int) % n
        idx1 = (idx0 + 1) % n
        t = hue_f - np.floor(hue_f)
        interp = (1-t[...,None])*ring[idx0] + t[...,None]*ring[idx1]
        rgb = (1-mag[...,None])*center + mag[...,None]*interp
        return np.clip(linear_to_srgb_np(rgb),0,1)

    disk = build_disk(rgb_ring_lin, center_lin)
    
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi)
    mag = np.sqrt(u*u + v*v)
    mag /= (mag.max() + 1e-8)
    n_hues = rgb_ring_lin.shape[0]
    hue_f = angle/(2*np.pi)*n_hues
    idx0 = np.floor(hue_f).astype(int) % n_hues
    idx1 = (idx0 + 1) % n_hues
    t = hue_f - np.floor(hue_f)
    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    r = mag[..., None]
    rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow),0,1)
    
    alpha = mag[...,None]
    white = np.ones_like(flow_rgb)
    black = np.zeros_like(flow_rgb)
    flow_white = alpha*flow_rgb + (1-alpha)*white
    flow_black = alpha*flow_rgb + (1-alpha)*black
    return disk, flow_rgb, flow_white, flow_black

# ==========================================
# 4. Main Execution
# ==========================================

def main():
    device_str = "cpu"
    dev = torch.device(device_str)
    
    print("1. Calculating Geometries...")
    surf_pts_opt = get_gamut_surface(40) 
    jz_gamut_opt = xyz_to_jzazbz(srgb_to_xyz(surf_pts_opt))
    
    centroid = np.mean(jz_gamut_opt, axis=0)
    green_vertex = xyz_to_jzazbz(srgb_to_xyz(np.array([[0.0, 1.0, 0.0]])))[0]
    anchor = centroid + 0.60 * (green_vertex - centroid)
    
    print("2. Fitting Maximal Ellipsoid (Solver + Inflation)...")
    center, M_solver = fit_ellipsoid_anchored_solver(jz_gamut_opt, anchor)
    M_max = inflate_to_max_gamut(center, M_solver)
    
    hoop = calculate_shadow_boundary(center, M_max, np.array([1.0, 0.0, 0.0]), res=72)
    rgb_hoop_raw = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
    check_gamut_compliance(rgb_hoop_raw)
    rgb_hoop_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(rgb_hoop_raw,0,1)))
    
    # Sinebow Fix: Generate sRGB first (gamma encoded), then convert
    t = np.linspace(0, 1, 72, endpoint=False)
    sr = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0/3))
    sg = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1/3))
    sb = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2/3))
    rgb_sine_srgb = np.stack([sr, sg, sb], axis=1) # Standard sRGB definition
    rgb_sine_lin = align_ring_red_to_green(srgb_to_linear_np(rgb_sine_srgb))

    # --- Shrunk Locus (Chroma Only) ---
    print("3. Calculating Shrunk Locus...")
    locus_xyz = get_full_spectral_locus()
    shrunk_locus_jz = fit_locus_chroma_shrink(locus_xyz, jz_gamut_opt)
    rgb_locus_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(shrunk_locus_jz)), 0, 1)))

    print("4. Loading Omnipose Flows...")
    omnidir = Path(omnirefactor.__file__).resolve().parent.parent.parent
    basedir = omnidir / "docs" / "_static"
    name = "ecoli"
    ext = ".tif"

    try:
        masks = io.imread(str(basedir / f"{name}_labels{ext}"))
        slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows = omnirefactor.core.masks_to_flows(masks, device=device_str)[4].to(dev)
        center_lin = srgb_to_linear_np(np.array([0.5, 0.5, 0.5]))

        def gen_set(ring, rotation_steps=0):
            r = np.roll(ring, rotation_steps, axis=0)
            return make_flow_images_for_ring(r, center_lin, flows)

        disk_s_0, _, w_s_0, b_s_0 = gen_set(rgb_sine_lin, 0)
        disk_e_0, _, w_e_0, b_e_0 = gen_set(rgb_hoop_lin, 0)
        disk_l_0, _, w_l_0, b_l_0 = gen_set(rgb_locus_lin, 0)

        rot = 18
        _, _, w_s_90, b_s_90 = gen_set(rgb_sine_lin, rot)
        _, _, w_e_90, b_e_90 = gen_set(rgb_hoop_lin, rot)
        _, _, w_l_90, b_l_90 = gen_set(rgb_locus_lin, rot)

        print("5. Plotting 2D...")
        plt.style.use('dark_background')
        fig, axes = plt.subplots(3, 5, figsize=(20, 12))
        rows = [("Sinebow", disk_s_0, b_s_0, w_s_0, b_s_90, w_s_90),
                ("Anchor Ellipse", disk_e_0, b_e_0, w_e_0, b_e_90, w_e_90),
                ("Shrunk Locus", disk_l_0, b_l_0, w_l_0, b_l_90, w_l_90)]
        titles = ["Hue Disk", "Black (0°)", "White (0°)", "Black (90°)", "White (90°)"]

        for i, (name, disk, b0, w0, b90, w90) in enumerate(rows):
            ax_row = axes[i]
            ax_row[0].imshow(disk)
            ax_row[0].set_ylabel(name, fontsize=14, color='white', rotation=90, labelpad=20)
            ax_row[1].imshow(b0); ax_row[2].imshow(w0)
            ax_row[3].imshow(b90); ax_row[4].imshow(w90)
            for j, ax in enumerate(ax_row):
                if i == 0: ax.set_title(titles[j], fontsize=12, color='#dddddd')
                ax.set_xticks([]); ax.set_yticks([])
        plt.tight_layout()
        plt.show()
        
        print("6. Generating 3D...")
        fig3d = go.Figure()
        max_jz = np.max(jz_gamut_opt[:,0]); sc = np.array([1.0/max_jz, 1.0, 1.0]); 
        def mc(a): return a[:,1], a[:,2], a[:,0]

        # Faces
        N=10; t=np.linspace(0,1,N); g=np.meshgrid(t,t)
        rgb_faces = [np.stack([g[0],g[1],np.zeros_like(g[0])],-1), np.stack([g[0],g[1],np.ones_like(g[0])],-1),
                     np.stack([g[0],np.zeros_like(g[0]),g[1]],-1), np.stack([g[0],np.ones_like(g[0]),g[1]],-1),
                     np.stack([np.zeros_like(g[0]),g[0],g[1]],-1), np.stack([np.ones_like(g[0]),g[0],g[1]],-1)]
        i_idx, j_idx, k_idx = [], [], []
        for r in range(N-1):
            for c in range(N-1):
                p1=r*N+c; p2=r*N+(c+1); p3=(r+1)*N+c; p4=(r+1)*N+(c+1)
                i_idx.extend([p1,p2]); j_idx.extend([p2,p4]); k_idx.extend([p3,p3])
        for i, face in enumerate(rgb_faces):
            jz_face = xyz_to_jzazbz(srgb_to_xyz(face.reshape(-1,3))) * sc
            x,y,z = mc(jz_face)
            fig3d.add_trace(go.Mesh3d(x=x, y=y, z=z, i=i_idx, j=j_idx, k=k_idx, color='gray', opacity=0.15, flatshading=False, hoverinfo='skip'))

        # Edges
        corners = [[0,0,0],[1,0,0],[1,1,0],[0,1,0],[0,0,1],[1,0,1],[1,1,1],[0,1,1]]
        edges_idx = [[0,1],[1,2],[2,3],[3,0],[4,5],[5,6],[6,7],[7,4],[0,4],[1,5],[2,6],[3,7]]
        for (s,e) in edges_idx:
            pts = np.linspace(corners[s],corners[e],25)
            line = xyz_to_jzazbz(srgb_to_xyz(pts)) * sc
            x,y,z = mc(line)
            fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', line=dict(color='white', width=3), showlegend=False))

        # Ellipsoid
        u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
        sph = np.stack([np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v)], axis=-1).reshape(-1, 3)
        ell = (sph @ M_max.T + center) * sc
        ex, ey, ez = mc(ell)
        fig3d.add_trace(go.Surface(x=ex.reshape(u.shape), y=ey.reshape(u.shape), z=ez.reshape(u.shape),
                                colorscale=[(0,'#88aa88'),(1,'#88aa88')], opacity=0.4, showscale=False, name='Anchored Ellipsoid'))

        # Loops
        hx, hy, hz = mc(hoop * sc)
        c_hoop = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in linear_to_srgb_np(np.clip(rgb_hoop_raw,0,1))]
        fig3d.add_trace(go.Scatter3d(x=hx, y=hy, z=hz, mode='markers', marker=dict(color=c_hoop, size=5), name='Ellipsoid Loop'))

        # Sinebow Corrected
        # rgb_sine_srgb is the "Source", so we transform IT to Jzazbz for the 3D plot
        jz_sine = xyz_to_jzazbz(srgb_to_xyz(rgb_sine_srgb)) * sc
        sx, sy, sz = mc(jz_sine)
        c_sine = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in rgb_sine_srgb]
        fig3d.add_trace(go.Scatter3d(x=sx, y=sy, z=sz, mode='markers', marker=dict(color=c_sine, size=4, symbol='diamond'), name='Sinebow'))
        
        # Shrunk Locus
        shrunk_jz = shrunk_locus_jz * sc
        lx, ly, lz = mc(shrunk_jz)
        fig3d.add_trace(go.Scatter3d(x=lx, y=ly, z=lz, mode='lines', line=dict(color='cyan', width=5), name='Shrunk Locus'))

        # Full Locus (Reference)
        full_locus_jz = xyz_to_jzazbz(locus_xyz) * sc
        fx, fy, fz = mc(full_locus_jz)
        fig3d.add_trace(go.Scatter3d(x=fx, y=fy, z=fz, mode='lines', line=dict(color='magenta', width=3, dash='dash'), name='Full Locus (Ref)'))

        fig3d.update_layout(template="plotly_dark", title="3D Jzazbz (Maximal + Shrunk Locus + Fixed Sinebow)", 
                            scene=dict(xaxis_title='az', yaxis_title='bz', zaxis_title='Jz', aspectmode='manual', aspectratio=dict(x=1,y=1,z=1)))
        fig3d.show()

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
or poroject the spectral locus into srgb in this space and then fourer approximate it? 

In [ ]:
import numpy as np
import scipy.optimize as opt
from scipy.spatial import ConvexHull
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import omnirefactor.core
from skimage import io
import fastremap

# ==========================================
# 1. Color Math
# ==========================================
def srgb_to_xyz(rgb):
    # Input: Gamma-encoded sRGB (0-1)
    mask = rgb > 0.04045
    linear = np.zeros_like(rgb)
    linear[mask] = ((rgb[mask] + 0.055) / 1.055) ** 2.4
    linear[~mask] = rgb[~mask] / 12.92
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    return linear @ M.T

def xyz_to_jzazbz(xyz):
    b, g, d, d0 = 1.15, 0.66, -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    lms = xyz @ M1.T
    lms_norm = (lms * 200) / 10000.0 
    lms_prime = np.sign(lms_norm) * (((c1 + c2 * (np.abs(lms_norm) ** n)) / (1 + c3 * (np.abs(lms_norm) ** n))) ** p)
    izazbz = lms_prime @ M2.T
    Jz = ((1 + d) * izazbz[:, 0]) / (1 + d * izazbz[:, 0]) - d0
    return np.column_stack((Jz, izazbz[:, 1], izazbz[:, 2]))

def jzazbz_to_xyz(jzazbz):
    Jz, az, bz = jzazbz[:,0], jzazbz[:,1], jzazbz[:,2]
    d, d0 = -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    Jp = Jz + d0; Iz = Jp / (1 + d - d * Jp)
    izazbz_vec = np.stack([Iz, az, bz], axis=1)
    lms_prime = izazbz_vec @ np.linalg.inv(M2).T
    sign_lms = np.sign(lms_prime)
    Y = np.abs(lms_prime) ** (1/p)
    num = Y - c1; den = c2 - Y * c3
    An = np.clip(num / den, 0, None) 
    lms_norm = sign_lms * (An ** (1/n))
    lms = (lms_norm * 10000.0) / 200.0
    return lms @ np.linalg.inv(M1).T

def xyz_to_srgb_raw(xyz):
    M_inv = np.linalg.inv(np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]]))
    linear = xyz @ M_inv.T
    srgb = np.zeros_like(linear)
    mask = linear > 0.0031308
    srgb[mask] = 1.055 * (linear[mask] ** (1/2.4)) - 0.055
    srgb[~mask] = 12.92 * linear[~mask]
    return srgb

def xyz_to_srgb(xyz): return np.clip(xyz_to_srgb_raw(xyz), 0, 1)
def srgb_to_linear_np(srgb): return np.where(srgb <= 0.04045, srgb / 12.92, ((srgb + 0.055) / 1.055) ** 2.4)
def linear_to_srgb_np(lin):
    s = np.zeros_like(lin)
    mask = lin > 0.0031308
    s[mask] = 1.055 * (lin[mask]**(1/2.4)) - 0.055
    s[~mask] = 12.92 * lin[~mask]
    return np.clip(s, 0, 1)

def get_full_spectral_locus():
    # 380-780nm
    cmf = np.array([
        [0.0014,0.0000,0.0065], [0.0042,0.0001,0.0201], [0.0143,0.0004,0.0679], [0.0435,0.0012,0.2074],
        [0.1344,0.0040,0.6456], [0.2839,0.0116,1.3856], [0.3483,0.0230,1.7471], [0.3362,0.0380,1.7721],
        [0.2908,0.0600,1.6692], [0.1954,0.0910,1.2876], [0.0956,0.1390,0.8130], [0.0320,0.2080,0.4652],
        [0.0049,0.3230,0.2720], [0.0093,0.5030,0.1582], [0.0633,0.7100,0.0782], [0.1655,0.8620,0.0422],
        [0.2904,0.9540,0.0203], [0.4334,0.9950,0.0087], [0.5945,0.9950,0.0039], [0.7621,0.9520,0.0021],
        [0.9163,0.8700,0.0017], [1.0263,0.7570,0.0011], [1.0622,0.6310,0.0008], [1.0026,0.5030,0.0006],
        [0.8544,0.3810,0.0002], [0.6424,0.2650,0.0000], [0.4479,0.1750,0.0000], [0.2835,0.1070,0.0000],
        [0.1649,0.0610,0.0000], [0.0874,0.0320,0.0000], [0.0468,0.0170,0.0000], [0.0227,0.0082,0.0000],
        [0.0114,0.0041,0.0000], [0.0058,0.0021,0.0000], [0.0029,0.0010,0.0000], [0.0014,0.0005,0.0000]
    ])
    # Close Line of Purples
    start_pt = cmf[-1]; end_pt = cmf[0]
    purples = np.linspace(start_pt, end_pt, 25)
    return np.vstack([cmf, purples])

# ==========================================
# 2. Optimization
# ==========================================

def get_gamut_surface(res=16):
    t = np.linspace(0, 1, res)
    g = np.meshgrid(t, t)
    faces = [np.stack([g[0], g[1], np.zeros_like(g[0])], -1), np.stack([g[0], g[1], np.ones_like(g[0])], -1),
             np.stack([g[0], np.zeros_like(g[0]), g[1]], -1), np.stack([g[0], np.ones_like(g[0]), g[1]], -1),
             np.stack([np.zeros_like(g[0]), g[0], g[1]], -1), np.stack([np.ones_like(g[0]), g[0], g[1]], -1)]
    pts = np.vstack([f.reshape(-1, 3) for f in faces])
    corners = np.array([[0,0,0], [1,0,0], [0,1,0], [0,0,1], [1,1,0], [1,0,1], [0,1,1], [1,1,1]])
    return np.vstack([pts, corners])

def fit_ellipsoid_anchored_solver(points, anchor_point):
    hull = ConvexHull(points)
    A = hull.equations[:, :3]; b_const = -hull.equations[:, 3]
    scale = 100.0; A_scaled = A / scale; anchor_scaled = anchor_point * scale
    c0 = np.mean(points, axis=0) * scale
    c0 = 0.5 * c0 + 0.5 * anchor_scaled 
    x0 = np.concatenate([c0, np.eye(3).flatten() * 0.05])

    def objective(x):
        M = x[3:].reshape(3, 3); sign, val = np.linalg.slogdet(M)
        return -val if sign > 0 else 1e9

    def constraints(x):
        d, M = x[:3], x[3:].reshape(3, 3)
        norm_AM = np.linalg.norm(A_scaled @ M, axis=1)
        hull_con = b_const - (A_scaled @ d + norm_AM)
        try: u = np.linalg.solve(M, anchor_scaled - d); anchor_con = 1.0 - np.linalg.norm(u)
        except: anchor_con = -1.0
        return np.concatenate([hull_con, [anchor_con]])

    res = opt.minimize(objective, x0, method='SLSQP', constraints={'type': 'ineq', 'fun': constraints}, options={'maxiter': 1000})
    return res.x[:3]/scale, res.x[3:].reshape(3, 3)/scale

def calculate_shadow_boundary(center, M, view_vector, res=72):
    w = np.linalg.solve(M, view_vector); w = w / np.linalg.norm(w)
    tmp = np.array([0.0, 1.0, 0.0]) if np.abs(w[1]) < 0.9 else np.array([0.0, 0.0, 1.0])
    s1 = np.cross(w, tmp); s1 /= np.linalg.norm(s1); s2 = np.cross(w, s1)
    theta = np.linspace(0, 2*np.pi, res)
    return center + (np.outer(np.cos(theta), s1) + np.outer(np.sin(theta), s2)) @ M.T

def inflate_to_max_gamut(center, M_init):
    scale = 1.0; step = 0.05; best_M = M_init.copy()
    for _ in range(50):
        test_M = M_init * (scale + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        scale += step; best_M = test_M
    step = 0.005; current_M = best_M
    for _ in range(20):
        test_M = current_M * (1.0 + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        current_M = test_M
    return current_M

def project_locus_chroma_binary_search(locus_xyz_raw, gamut_max_J):
    """
    Projects the spectral locus onto the sRGB gamut surface by
    shrinking chroma (az, bz) via binary search.
    Keeps Lightness (Jz) constant unless it exceeds gamut max.
    """
    locus_jz = xyz_to_jzazbz(locus_xyz_raw)
    projected_jz = []
    
    print("   Projecting Locus to Gamut Surface...")
    
    for point in locus_jz:
        J, a, b = point
        
        # 1. Clip Lightness if it's above sRGB max (approx 0.22)
        # We assume some margin to avoid top-face issues
        J = min(J, gamut_max_J * 0.99)
        
        # 2. Binary Search for Chroma Scale
        low = 0.0
        high = 1.0
        best_scale = 0.0
        
        # Vector direction in chroma
        for _ in range(15): # 15 iterations is plenty precision
            mid = (low + high) / 2.0
            test_pt = np.array([[J, a * mid, b * mid]])
            rgb = xyz_to_srgb_raw(jzazbz_to_xyz(test_pt))
            
            # Check validity
            if np.min(rgb) >= -0.001 and np.max(rgb) <= 1.001:
                best_scale = mid
                low = mid # Try pushing further out
            else:
                high = mid # Pull back
        
        projected_jz.append([J, a * best_scale, b * best_scale])
        
    return np.array(projected_jz)

def check_gamut_compliance(rgb_loop, name="Loop"):
    min_val = np.min(rgb_loop); max_val = np.max(rgb_loop)
    tol = 2e-3
    if min_val < -tol or max_val > 1.0 + tol:
        print(f"   >>> {name} FAIL: [{min_val:.4f}, {max_val:.4f}]")
    else:
        print(f"   >>> {name} PASS: Inside sRGB")

# ==========================================
# 3. Omnipose Helpers
# ==========================================
def align_ring_red_to_green(ring_lin):
    n = len(ring_lin)
    red_ref = np.array([1.0, 0.0, 0.0])
    idx_r = np.argmin(np.linalg.norm(ring_lin - red_ref, axis=1))
    ring_aligned = np.roll(ring_lin, -idx_r, axis=0)
    green_ref = np.array([0.0, 1.0, 0.0])
    if np.linalg.norm(ring_aligned[(2*n)//3]-green_ref) < np.linalg.norm(ring_aligned[n//3]-green_ref):
        ring_aligned = ring_aligned[::-1]
    return ring_aligned

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    def build_disk(ring, center):
        y, x = np.mgrid[-1:1:256j, -1:1:256j]
        mag = np.sqrt(x*x + y*y)
        angle = np.mod(np.arctan2(y, x), 2*np.pi)
        n = ring.shape[0]
        hue_f = angle/(2*np.pi)*n
        idx0 = np.floor(hue_f).astype(int) % n
        idx1 = (idx0 + 1) % n
        t = hue_f - np.floor(hue_f)
        interp = (1-t[...,None])*ring[idx0] + t[...,None]*ring[idx1]
        rgb = (1-mag[...,None])*center + mag[...,None]*interp
        return np.clip(linear_to_srgb_np(rgb),0,1)

    disk = build_disk(rgb_ring_lin, center_lin)
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi)
    mag = np.sqrt(u*u + v*v); mag /= (mag.max() + 1e-8)
    n = rgb_ring_lin.shape[0]
    hue_f = angle/(2*np.pi)*n
    idx0 = np.floor(hue_f).astype(int) % n
    idx1 = (idx0 + 1) % n
    t = hue_f - np.floor(hue_f)
    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    r = mag[..., None]
    rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow),0,1)
    alpha = mag[...,None]
    white = np.ones_like(flow_rgb); black = np.zeros_like(flow_rgb)
    flow_white = alpha*flow_rgb + (1-alpha)*white
    flow_black = alpha*flow_rgb + (1-alpha)*black
    return disk, flow_rgb, flow_white, flow_black

# ==========================================
# 4. Main Execution
# ==========================================

def main():
    device_str = "cpu"
    dev = torch.device(device_str)
    
    print("1. Calculating Geometries...")
    surf_pts_opt = get_gamut_surface(40) 
    jz_gamut_opt = xyz_to_jzazbz(srgb_to_xyz(surf_pts_opt))
    
    centroid = np.mean(jz_gamut_opt, axis=0)
    green_vertex = xyz_to_jzazbz(srgb_to_xyz(np.array([[0.0, 1.0, 0.0]])))[0]
    anchor = centroid + 0.60 * (green_vertex - centroid)
    
    print("2. Fitting Maximal Ellipsoid...")
    center, M_solver = fit_ellipsoid_anchored_solver(jz_gamut_opt, anchor)
    M_max = inflate_to_max_gamut(center, M_solver)
    
    hoop = calculate_shadow_boundary(center, M_max, np.array([1.0, 0.0, 0.0]), res=72)
    rgb_hoop_raw = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
    check_gamut_compliance(rgb_hoop_raw, "Ellipsoid")
    rgb_hoop_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(rgb_hoop_raw,0,1)))
    
    # Fix Sinebow Trajectory (Use standard sRGB, not linear)
    t = np.linspace(0, 1, 72, endpoint=False)
    sr = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0/3))
    sg = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1/3))
    sb = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2/3))
    rgb_sine_std = np.stack([sr, sg, sb], axis=1) # Gamma-encoded
    rgb_sine_lin = align_ring_red_to_green(srgb_to_linear_np(rgb_sine_std))

    # --- Projected Spectral Locus ---
    print("3. Projecting Spectral Locus...")
    locus_xyz = get_full_spectral_locus()
    max_J = np.max(jz_gamut_opt[:,0])
    projected_locus_jz = project_locus_chroma_binary_search(locus_xyz, max_J)
    
    rgb_locus_raw = xyz_to_srgb_raw(jzazbz_to_xyz(projected_locus_jz))
    check_gamut_compliance(rgb_locus_raw, "Projected Locus")
    rgb_locus_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(rgb_locus_raw, 0, 1)))

    print("4. Loading Omnipose Flows...")
    omnidir = Path(omnirefactor.__file__).resolve().parent.parent.parent
    basedir = omnidir / "docs" / "_static"
    name = "ecoli"
    ext = ".tif"

    try:
        masks = io.imread(str(basedir / f"{name}_labels{ext}"))
        slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows = omnirefactor.core.masks_to_flows(masks, device=device_str)[4].to(dev)
        center_lin = srgb_to_linear_np(np.array([0.5, 0.5, 0.5]))

        def gen_set(ring, rotation_steps=0):
            r = np.roll(ring, rotation_steps, axis=0)
            return make_flow_images_for_ring(r, center_lin, flows)

        disk_s_0, _, w_s_0, b_s_0 = gen_set(rgb_sine_lin, 0)
        disk_e_0, _, w_e_0, b_e_0 = gen_set(rgb_hoop_lin, 0)
        disk_l_0, _, w_l_0, b_l_0 = gen_set(rgb_locus_lin, 0)

        rot = 18
        _, _, w_s_90, b_s_90 = gen_set(rgb_sine_lin, rot)
        _, _, w_e_90, b_e_90 = gen_set(rgb_hoop_lin, rot)
        _, _, w_l_90, b_l_90 = gen_set(rgb_locus_lin, rot)

        print("5. Plotting 2D...")
        plt.style.use('dark_background')
        fig, axes = plt.subplots(3, 5, figsize=(20, 12))
        rows = [("Sinebow", disk_s_0, b_s_0, w_s_0, b_s_90, w_s_90),
                ("Anchor Ellipse", disk_e_0, b_e_0, w_e_0, b_e_90, w_e_90),
                ("Projected Locus", disk_l_0, b_l_0, w_l_0, b_l_90, w_l_90)]
        titles = ["Hue Disk", "Black (0°)", "White (0°)", "Black (90°)", "White (90°)"]

        for i, (name, disk, b0, w0, b90, w90) in enumerate(rows):
            ax_row = axes[i]
            ax_row[0].imshow(disk)
            ax_row[0].set_ylabel(name, fontsize=14, color='white', rotation=90, labelpad=20)
            ax_row[1].imshow(b0); ax_row[2].imshow(w0)
            ax_row[3].imshow(b90); ax_row[4].imshow(w90)
            for j, ax in enumerate(ax_row):
                if i == 0: ax.set_title(titles[j], fontsize=12, color='#dddddd')
                ax.set_xticks([]); ax.set_yticks([])
        plt.tight_layout()
        plt.show()
        
        print("6. Generating 3D...")
        fig3d = go.Figure()
        max_jz = np.max(jz_gamut_opt[:,0]); sc = np.array([1.0/max_jz, 1.0, 1.0]); 
        def mc(a): return a[:,1], a[:,2], a[:,0]

        # Faces
        N=10; t=np.linspace(0,1,N); g=np.meshgrid(t,t)
        rgb_faces = [np.stack([g[0],g[1],np.zeros_like(g[0])],-1), np.stack([g[0],g[1],np.ones_like(g[0])],-1),
                     np.stack([g[0],np.zeros_like(g[0]),g[1]],-1), np.stack([g[0],np.ones_like(g[0]),g[1]],-1),
                     np.stack([np.zeros_like(g[0]),g[0],g[1]],-1), np.stack([np.ones_like(g[0]),g[0],g[1]],-1)]
        i_idx, j_idx, k_idx = [], [], []
        for r in range(N-1):
            for c in range(N-1):
                p1=r*N+c; p2=r*N+(c+1); p3=(r+1)*N+c; p4=(r+1)*N+(c+1)
                i_idx.extend([p1,p2]); j_idx.extend([p2,p4]); k_idx.extend([p3,p3])
        for i, face in enumerate(rgb_faces):
            jz_face = xyz_to_jzazbz(srgb_to_xyz(face.reshape(-1,3))) * sc
            x,y,z = mc(jz_face)
            fig3d.add_trace(go.Mesh3d(x=x, y=y, z=z, i=i_idx, j=j_idx, k=k_idx, color='gray', opacity=0.15, flatshading=False, hoverinfo='skip'))

        # Edges
        corners = [[0,0,0],[1,0,0],[1,1,0],[0,1,0],[0,0,1],[1,0,1],[1,1,1],[0,1,1]]
        edges_idx = [[0,1],[1,2],[2,3],[3,0],[4,5],[5,6],[6,7],[7,4],[0,4],[1,5],[2,6],[3,7]]
        for (s,e) in edges_idx:
            pts = np.linspace(corners[s],corners[e],25)
            line = xyz_to_jzazbz(srgb_to_xyz(pts)) * sc
            x,y,z = mc(line)
            fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', line=dict(color='white', width=3), showlegend=False))

        # Ellipsoid
        u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
        sph = np.stack([np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v)], axis=-1).reshape(-1, 3)
        ell = (sph @ M_max.T + center) * sc
        ex, ey, ez = mc(ell)
        fig3d.add_trace(go.Surface(x=ex.reshape(u.shape), y=ey.reshape(u.shape), z=ez.reshape(u.shape),
                                colorscale=[(0,'#88aa88'),(1,'#88aa88')], opacity=0.4, showscale=False, name='Anchored Ellipsoid'))

        # Loops
        hx, hy, hz = mc(hoop * sc)
        c_hoop = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in linear_to_srgb_np(np.clip(rgb_hoop_raw,0,1))]
        fig3d.add_trace(go.Scatter3d(x=hx, y=hy, z=hz, mode='markers', marker=dict(color=c_hoop, size=5), name='Ellipsoid Loop'))

        # Fix Sinebow 3D: Use the Standard sRGB (gamma encoded) values we generated
        sx, sy, sz = mc(xyz_to_jzazbz(srgb_to_xyz(rgb_sine_std)) * sc)
        c_sine = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in rgb_sine_std]
        fig3d.add_trace(go.Scatter3d(x=sx, y=sy, z=sz, mode='markers', marker=dict(color=c_sine, size=4, symbol='diamond'), name='Sinebow Loop'))
        
        # Projected Locus
        lx, ly, lz = mc(projected_locus_jz * sc)
        fig3d.add_trace(go.Scatter3d(x=lx, y=ly, z=lz, mode='lines', line=dict(color='cyan', width=5), name='Projected Locus'))

        # Full Locus (Ref)
        lx_f, ly_f, lz_f = mc(xyz_to_jzazbz(locus_xyz) * sc)
        fig3d.add_trace(go.Scatter3d(x=lx_f, y=ly_f, z=lz_f, mode='lines', line=dict(color='magenta', width=3, dash='dash'), name='Full Locus (Ref)'))

        fig3d.update_layout(template="plotly_dark", title="3D Jzazbz (Maximal + Projected Locus)", 
                            scene=dict(xaxis_title='az', yaxis_title='bz', zaxis_title='Jz', aspectmode='manual', aspectratio=dict(x=1,y=1,z=1)))
        fig3d.show()

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import scipy.optimize as opt
from scipy.spatial import ConvexHull
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import omnirefactor.core
from skimage import io
import fastremap

# ==========================================
# 1. Color Math
# ==========================================
def srgb_to_xyz(rgb):
    # Input: Gamma-encoded sRGB (0-1)
    mask = rgb > 0.04045
    linear = np.zeros_like(rgb)
    linear[mask] = ((rgb[mask] + 0.055) / 1.055) ** 2.4
    linear[~mask] = rgb[~mask] / 12.92
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    return linear @ M.T

def xyz_to_jzazbz(xyz):
    b, g, d, d0 = 1.15, 0.66, -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    lms = xyz @ M1.T
    lms_norm = (lms * 200) / 10000.0 
    lms_prime = np.sign(lms_norm) * (((c1 + c2 * (np.abs(lms_norm) ** n)) / (1 + c3 * (np.abs(lms_norm) ** n))) ** p)
    izazbz = lms_prime @ M2.T
    Jz = ((1 + d) * izazbz[:, 0]) / (1 + d * izazbz[:, 0]) - d0
    return np.column_stack((Jz, izazbz[:, 1], izazbz[:, 2]))

def jzazbz_to_xyz(jzazbz):
    Jz, az, bz = jzazbz[:,0], jzazbz[:,1], jzazbz[:,2]
    d, d0 = -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    Jp = Jz + d0; Iz = Jp / (1 + d - d * Jp)
    izazbz_vec = np.stack([Iz, az, bz], axis=1)
    lms_prime = izazbz_vec @ np.linalg.inv(M2).T
    sign_lms = np.sign(lms_prime)
    Y = np.abs(lms_prime) ** (1/p)
    num = Y - c1; den = c2 - Y * c3
    An = np.clip(num / den, 0, None) 
    lms_norm = sign_lms * (An ** (1/n))
    lms = (lms_norm * 10000.0) / 200.0
    return lms @ np.linalg.inv(M1).T

def xyz_to_srgb_raw(xyz):
    M_inv = np.linalg.inv(np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]]))
    linear = xyz @ M_inv.T
    srgb = np.zeros_like(linear)
    mask = linear > 0.0031308
    srgb[mask] = 1.055 * (linear[mask] ** (1/2.4)) - 0.055
    srgb[~mask] = 12.92 * linear[~mask]
    return srgb

def xyz_to_srgb(xyz): return np.clip(xyz_to_srgb_raw(xyz), 0, 1)
def srgb_to_linear_np(srgb): return np.where(srgb <= 0.04045, srgb / 12.92, ((srgb + 0.055) / 1.055) ** 2.4)
def linear_to_srgb_np(lin):
    s = np.zeros_like(lin)
    mask = lin > 0.0031308
    s[mask] = 1.055 * (lin[mask]**(1/2.4)) - 0.055
    s[~mask] = 12.92 * lin[~mask]
    return np.clip(s, 0, 1)

def get_full_spectral_locus():
    cmf = np.array([
        [0.0014,0.0000,0.0065], [0.0042,0.0001,0.0201], [0.0143,0.0004,0.0679], [0.0435,0.0012,0.2074],
        [0.1344,0.0040,0.6456], [0.2839,0.0116,1.3856], [0.3483,0.0230,1.7471], [0.3362,0.0380,1.7721],
        [0.2908,0.0600,1.6692], [0.1954,0.0910,1.2876], [0.0956,0.1390,0.8130], [0.0320,0.2080,0.4652],
        [0.0049,0.3230,0.2720], [0.0093,0.5030,0.1582], [0.0633,0.7100,0.0782], [0.1655,0.8620,0.0422],
        [0.2904,0.9540,0.0203], [0.4334,0.9950,0.0087], [0.5945,0.9950,0.0039], [0.7621,0.9520,0.0021],
        [0.9163,0.8700,0.0017], [1.0263,0.7570,0.0011], [1.0622,0.6310,0.0008], [1.0026,0.5030,0.0006],
        [0.8544,0.3810,0.0002], [0.6424,0.2650,0.0000], [0.4479,0.1750,0.0000], [0.2835,0.1070,0.0000],
        [0.1649,0.0610,0.0000], [0.0874,0.0320,0.0000], [0.0468,0.0170,0.0000], [0.0227,0.0082,0.0000],
        [0.0114,0.0041,0.0000], [0.0058,0.0021,0.0000], [0.0029,0.0010,0.0000], [0.0014,0.0005,0.0000]
    ])
    start_pt, end_pt = cmf[-1], cmf[0]
    purples = np.linspace(start_pt, end_pt, 25)
    return np.vstack([cmf, purples])

# ==========================================
# 2. Optimization
# ==========================================

def get_gamut_surface(res=16):
    t = np.linspace(0, 1, res)
    g = np.meshgrid(t, t)
    faces = [np.stack([g[0], g[1], np.zeros_like(g[0])], -1), np.stack([g[0], g[1], np.ones_like(g[0])], -1),
             np.stack([g[0], np.zeros_like(g[0]), g[1]], -1), np.stack([g[0], np.ones_like(g[0]), g[1]], -1),
             np.stack([np.zeros_like(g[0]), g[0], g[1]], -1), np.stack([np.ones_like(g[0]), g[0], g[1]], -1)]
    pts = np.vstack([f.reshape(-1, 3) for f in faces])
    corners = np.array([[0,0,0], [1,0,0], [0,1,0], [0,0,1], [1,1,0], [1,0,1], [0,1,1], [1,1,1]])
    return np.vstack([pts, corners])

def fit_ellipsoid_anchored_solver(points, anchor_point):
    hull = ConvexHull(points)
    A = hull.equations[:, :3]; b_const = -hull.equations[:, 3]
    scale = 100.0; A_scaled = A / scale; anchor_scaled = anchor_point * scale
    c0 = np.mean(points, axis=0) * scale
    c0 = 0.5 * c0 + 0.5 * anchor_scaled 
    x0 = np.concatenate([c0, np.eye(3).flatten() * 0.05])

    def objective(x):
        M = x[3:].reshape(3, 3); sign, val = np.linalg.slogdet(M)
        return -val if sign > 0 else 1e9

    def constraints(x):
        d, M = x[:3], x[3:].reshape(3, 3)
        norm_AM = np.linalg.norm(A_scaled @ M, axis=1)
        hull_con = b_const - (A_scaled @ d + norm_AM)
        try: u = np.linalg.solve(M, anchor_scaled - d); anchor_con = 1.0 - np.linalg.norm(u)
        except: anchor_con = -1.0
        return np.concatenate([hull_con, [anchor_con]])

    res = opt.minimize(objective, x0, method='SLSQP', constraints={'type': 'ineq', 'fun': constraints}, options={'maxiter': 1000})
    return res.x[:3]/scale, res.x[3:].reshape(3, 3)/scale

def calculate_shadow_boundary(center, M, view_vector, res=72):
    w = np.linalg.solve(M, view_vector); w = w / np.linalg.norm(w)
    tmp = np.array([0.0, 1.0, 0.0]) if np.abs(w[1]) < 0.9 else np.array([0.0, 0.0, 1.0])
    s1 = np.cross(w, tmp); s1 /= np.linalg.norm(s1); s2 = np.cross(w, s1)
    theta = np.linspace(0, 2*np.pi, res)
    return center + (np.outer(np.cos(theta), s1) + np.outer(np.sin(theta), s2)) @ M.T

def inflate_to_max_gamut(center, M_init):
    scale = 1.0; step = 0.05; best_M = M_init.copy()
    for _ in range(50):
        test_M = M_init * (scale + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        scale += step; best_M = test_M
    step = 0.005; current_M = best_M
    for _ in range(20):
        test_M = current_M * (1.0 + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        current_M = test_M
    return current_M

def project_locus_conical(locus_xyz_raw, center_point_jz):
    """
    Projects the locus towards 'center_point_jz' until it hits the sRGB boundary.
    Using Binary Search along the ray.
    """
    locus_jz = xyz_to_jzazbz(locus_xyz_raw)
    projected_jz = []
    
    print("   Projecting Locus (Conical)...")
    
    for P in locus_jz:
        # Vector from Center to Point
        V = P - center_point_jz
        
        # Binary Search for t in [0, 2.0] (Assuming P is outside, but check range)
        # Ray = C + t * V
        # We want max t such that Ray(t) is in Gamut
        
        low = 0.0
        high = 1.5 # Start assuming we need to shrink (t < 1), but allow slight growth if needed
        
        best_t = 0.0
        
        for _ in range(15):
            mid = (low + high) / 2.0
            test_pt = center_point_jz + mid * V
            
            # Check sRGB validity
            rgb = xyz_to_srgb_raw(jzazbz_to_xyz(np.array([test_pt])))
            
            if np.min(rgb) >= -0.001 and np.max(rgb) <= 1.001:
                best_t = mid
                low = mid # Try pushing out further
            else:
                high = mid # Pull back
                
        projected_jz.append(center_point_jz + best_t * V)
        
    return np.array(projected_jz)

def check_gamut_compliance(rgb_loop, name="Loop"):
    min_val = np.min(rgb_loop); max_val = np.max(rgb_loop)
    tol = 2e-3
    if min_val < -tol or max_val > 1.0 + tol:
        print(f"   >>> {name} FAIL: [{min_val:.4f}, {max_val:.4f}]")
    else:
        print(f"   >>> {name} PASS: Inside sRGB")

# ==========================================
# 3. Omnipose Helpers
# ==========================================
def align_ring_red_to_green(ring_lin):
    n = len(ring_lin)
    red_ref = np.array([1.0, 0.0, 0.0])
    idx_r = np.argmin(np.linalg.norm(ring_lin - red_ref, axis=1))
    ring_aligned = np.roll(ring_lin, -idx_r, axis=0)
    green_ref = np.array([0.0, 1.0, 0.0])
    if np.linalg.norm(ring_aligned[(2*n)//3]-green_ref) < np.linalg.norm(ring_aligned[n//3]-green_ref):
        ring_aligned = ring_aligned[::-1]
    return ring_aligned

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    def build_disk(ring, center):
        y, x = np.mgrid[-1:1:256j, -1:1:256j]
        mag = np.sqrt(x*x + y*y)
        angle = np.mod(np.arctan2(y, x), 2*np.pi)
        n = ring.shape[0]
        hue_f = angle/(2*np.pi)*n
        idx0 = np.floor(hue_f).astype(int) % n
        idx1 = (idx0 + 1) % n
        t = hue_f - np.floor(hue_f)
        interp = (1-t[...,None])*ring[idx0] + t[...,None]*ring[idx1]
        rgb = (1-mag[...,None])*center + mag[...,None]*interp
        return np.clip(linear_to_srgb_np(rgb),0,1)

    disk = build_disk(rgb_ring_lin, center_lin)
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi)
    mag = np.sqrt(u*u + v*v); mag /= (mag.max() + 1e-8)
    n = rgb_ring_lin.shape[0]
    hue_f = angle/(2*np.pi)*n
    idx0 = np.floor(hue_f).astype(int) % n
    idx1 = (idx0 + 1) % n
    t = hue_f - np.floor(hue_f)
    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    r = mag[..., None]
    rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow),0,1)
    alpha = mag[...,None]
    white = np.ones_like(flow_rgb); black = np.zeros_like(flow_rgb)
    flow_white = alpha*flow_rgb + (1-alpha)*white
    flow_black = alpha*flow_rgb + (1-alpha)*black
    return disk, flow_rgb, flow_white, flow_black

# ==========================================
# 4. Main Execution
# ==========================================

def main():
    device_str = "cpu"
    dev = torch.device(device_str)
    
    print("1. Calculating Geometries...")
    surf_pts_opt = get_gamut_surface(40) 
    jz_gamut_opt = xyz_to_jzazbz(srgb_to_xyz(surf_pts_opt))
    
    centroid = np.mean(jz_gamut_opt, axis=0)
    green_vertex = xyz_to_jzazbz(srgb_to_xyz(np.array([[0.0, 1.0, 0.0]])))[0]
    anchor = centroid + 0.60 * (green_vertex - centroid)
    
    print("2. Fitting Maximal Ellipsoid...")
    center, M_solver = fit_ellipsoid_anchored_solver(jz_gamut_opt, anchor)
    M_max = inflate_to_max_gamut(center, M_solver)
    
    hoop = calculate_shadow_boundary(center, M_max, np.array([1.0, 0.0, 0.0]), res=72)
    rgb_hoop_raw = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
    check_gamut_compliance(rgb_hoop_raw, "Ellipsoid")
    rgb_hoop_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(rgb_hoop_raw,0,1)))
    
    t = np.linspace(0, 1, 72, endpoint=False)
    sr = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0/3))
    sg = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1/3))
    sb = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2/3))
    rgb_sine_std = np.stack([sr, sg, sb], axis=1) # Gamma encoded
    rgb_sine_lin = align_ring_red_to_green(srgb_to_linear_np(rgb_sine_std))

    # --- Conical Projection ---
    print("3. Projecting Spectral Locus (Conical)...")
    locus_xyz = get_full_spectral_locus()
    
    # Target Point: sRGB Middle Gray (0.5, 0.5, 0.5) -> Jzazbz
    # This is a safe, valid center inside the volume.
    target_rgb = np.array([[0.5, 0.5, 0.5]]) 
    target_jz = xyz_to_jzazbz(srgb_to_xyz(target_rgb))[0]
    
    projected_locus_jz = project_locus_conical(locus_xyz, target_jz)
    rgb_locus_raw = xyz_to_srgb_raw(jzazbz_to_xyz(projected_locus_jz))
    check_gamut_compliance(rgb_locus_raw, "Projected Locus")
    rgb_locus_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(rgb_locus_raw, 0, 1)))

    print("4. Loading Omnipose Flows...")
    omnidir = Path(omnirefactor.__file__).resolve().parent.parent.parent
    basedir = omnidir / "docs" / "_static"
    name = "ecoli"
    ext = ".tif"

    try:
        masks = io.imread(str(basedir / f"{name}_labels{ext}"))
        slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows = omnirefactor.core.masks_to_flows(masks, device=device_str)[4].to(dev)
        center_lin = srgb_to_linear_np(np.array([0.5, 0.5, 0.5]))

        def gen_set(ring, rotation_steps=0):
            r = np.roll(ring, rotation_steps, axis=0)
            return make_flow_images_for_ring(r, center_lin, flows)

        disk_s_0, _, w_s_0, b_s_0 = gen_set(rgb_sine_lin, 0)
        disk_e_0, _, w_e_0, b_e_0 = gen_set(rgb_hoop_lin, 0)
        disk_l_0, _, w_l_0, b_l_0 = gen_set(rgb_locus_lin, 0)

        rot = 18
        _, _, w_s_90, b_s_90 = gen_set(rgb_sine_lin, rot)
        _, _, w_e_90, b_e_90 = gen_set(rgb_hoop_lin, rot)
        _, _, w_l_90, b_l_90 = gen_set(rgb_locus_lin, rot)

        print("5. Plotting 2D...")
        plt.style.use('dark_background')
        fig, axes = plt.subplots(3, 5, figsize=(20, 12))
        rows = [("Sinebow", disk_s_0, b_s_0, w_s_0, b_s_90, w_s_90),
                ("Anchor Ellipse", disk_e_0, b_e_0, w_e_0, b_e_90, w_e_90),
                ("Projected Locus", disk_l_0, b_l_0, w_l_0, b_l_90, w_l_90)]
        titles = ["Hue Disk", "Black (0°)", "White (0°)", "Black (90°)", "White (90°)"]

        for i, (name, disk, b0, w0, b90, w90) in enumerate(rows):
            ax_row = axes[i]
            ax_row[0].imshow(disk)
            ax_row[0].set_ylabel(name, fontsize=14, color='white', rotation=90, labelpad=20)
            ax_row[1].imshow(b0); ax_row[2].imshow(w0)
            ax_row[3].imshow(b90); ax_row[4].imshow(w90)
            for j, ax in enumerate(ax_row):
                if i == 0: ax.set_title(titles[j], fontsize=12, color='#dddddd')
                ax.set_xticks([]); ax.set_yticks([])
        plt.tight_layout()
        plt.show()
        
        print("6. Generating 3D...")
        fig3d = go.Figure()
        max_jz = np.max(jz_gamut_opt[:,0]); sc = np.array([1.0/max_jz, 1.0, 1.0]); 
        def mc(a): return a[:,1], a[:,2], a[:,0]

        # Faces
        N=10; t=np.linspace(0,1,N); g=np.meshgrid(t,t)
        rgb_faces = [np.stack([g[0],g[1],np.zeros_like(g[0])],-1), np.stack([g[0],g[1],np.ones_like(g[0])],-1),
                     np.stack([g[0],np.zeros_like(g[0]),g[1]],-1), np.stack([g[0],np.ones_like(g[0]),g[1]],-1),
                     np.stack([np.zeros_like(g[0]),g[0],g[1]],-1), np.stack([np.ones_like(g[0]),g[0],g[1]],-1)]
        i_idx, j_idx, k_idx = [], [], []
        for r in range(N-1):
            for c in range(N-1):
                p1=r*N+c; p2=r*N+(c+1); p3=(r+1)*N+c; p4=(r+1)*N+(c+1)
                i_idx.extend([p1,p2]); j_idx.extend([p2,p4]); k_idx.extend([p3,p3])
        for i, face in enumerate(rgb_faces):
            jz_face = xyz_to_jzazbz(srgb_to_xyz(face.reshape(-1,3))) * sc
            x,y,z = mc(jz_face)
            fig3d.add_trace(go.Mesh3d(x=x, y=y, z=z, i=i_idx, j=j_idx, k=k_idx, color='gray', opacity=0.15, flatshading=False, hoverinfo='skip'))

        # Edges
        corners = [[0,0,0],[1,0,0],[1,1,0],[0,1,0],[0,0,1],[1,0,1],[1,1,1],[0,1,1]]
        edges_idx = [[0,1],[1,2],[2,3],[3,0],[4,5],[5,6],[6,7],[7,4],[0,4],[1,5],[2,6],[3,7]]
        for (s,e) in edges_idx:
            pts = np.linspace(corners[s],corners[e],25)
            line = xyz_to_jzazbz(srgb_to_xyz(pts)) * sc
            x,y,z = mc(line)
            fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', line=dict(color='white', width=3), showlegend=False))

        # Ellipsoid
        u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
        sph = np.stack([np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v)], axis=-1).reshape(-1, 3)
        ell = (sph @ M_max.T + center) * sc
        ex, ey, ez = mc(ell)
        fig3d.add_trace(go.Surface(x=ex.reshape(u.shape), y=ey.reshape(u.shape), z=ez.reshape(u.shape),
                                colorscale=[(0,'#88aa88'),(1,'#88aa88')], opacity=0.4, showscale=False, name='Anchored Ellipsoid'))

        # Loops
        hx, hy, hz = mc(hoop * sc)
        c_hoop = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in linear_to_srgb_np(np.clip(rgb_hoop_raw,0,1))]
        fig3d.add_trace(go.Scatter3d(x=hx, y=hy, z=hz, mode='markers', marker=dict(color=c_hoop, size=5), name='Ellipsoid Loop'))

        sx, sy, sz = mc(xyz_to_jzazbz(srgb_to_xyz(rgb_sine_std)) * sc)
        c_sine = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in rgb_sine_std]
        fig3d.add_trace(go.Scatter3d(x=sx, y=sy, z=sz, mode='markers', marker=dict(color=c_sine, size=4, symbol='diamond'), name='Sinebow'))
        
        # Projected Locus (Cyan)
        lx, ly, lz = mc(projected_locus_jz * sc)
        fig3d.add_trace(go.Scatter3d(x=lx, y=ly, z=lz, mode='lines', line=dict(color='cyan', width=5), name='Projected Locus'))

        # Full Locus (Ref)
        lx_f, ly_f, lz_f = mc(xyz_to_jzazbz(locus_xyz) * sc)
        fig3d.add_trace(go.Scatter3d(x=lx_f, y=ly_f, z=lz_f, mode='lines', line=dict(color='magenta', width=3, dash='dash'), name='Full Locus (Ref)'))

        fig3d.update_layout(template="plotly_dark", title="3D Jzazbz (Maximal + Projected Locus)", 
                            scene=dict(xaxis_title='az', yaxis_title='bz', zaxis_title='Jz', aspectmode='manual', aspectratio=dict(x=1,y=1,z=1)))
        fig3d.show()

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import scipy.optimize as opt
from scipy.spatial import ConvexHull
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import omnirefactor.core
from skimage import io
import fastremap

# ==========================================
# 1. Color Math
# ==========================================
def srgb_to_xyz(rgb):
    mask = rgb > 0.04045
    linear = np.zeros_like(rgb)
    linear[mask] = ((rgb[mask] + 0.055) / 1.055) ** 2.4
    linear[~mask] = rgb[~mask] / 12.92
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    return linear @ M.T

def xyz_to_jzazbz(xyz):
    b, g, d, d0 = 1.15, 0.66, -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    lms = xyz @ M1.T
    lms_norm = (lms * 200) / 10000.0 
    lms_prime = np.sign(lms_norm) * (((c1 + c2 * (np.abs(lms_norm) ** n)) / (1 + c3 * (np.abs(lms_norm) ** n))) ** p)
    izazbz = lms_prime @ M2.T
    Jz = ((1 + d) * izazbz[:, 0]) / (1 + d * izazbz[:, 0]) - d0
    return np.column_stack((Jz, izazbz[:, 1], izazbz[:, 2]))

def jzazbz_to_xyz(jzazbz):
    Jz, az, bz = jzazbz[:,0], jzazbz[:,1], jzazbz[:,2]
    d, d0 = -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    Jp = Jz + d0; Iz = Jp / (1 + d - d * Jp)
    izazbz_vec = np.stack([Iz, az, bz], axis=1)
    lms_prime = izazbz_vec @ np.linalg.inv(M2).T
    sign_lms = np.sign(lms_prime)
    Y = np.abs(lms_prime) ** (1/p)
    num = Y - c1; den = c2 - Y * c3
    An = np.clip(num / den, 0, None) 
    lms_norm = sign_lms * (An ** (1/n))
    lms = (lms_norm * 10000.0) / 200.0
    return lms @ np.linalg.inv(M1).T

def xyz_to_srgb_raw(xyz):
    M_inv = np.linalg.inv(np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]]))
    linear = xyz @ M_inv.T
    srgb = np.zeros_like(linear)
    mask = linear > 0.0031308
    srgb[mask] = 1.055 * (linear[mask] ** (1/2.4)) - 0.055
    srgb[~mask] = 12.92 * linear[~mask]
    return srgb

def xyz_to_srgb(xyz): return np.clip(xyz_to_srgb_raw(xyz), 0, 1)
def srgb_to_linear_np(srgb): return np.where(srgb <= 0.04045, srgb / 12.92, ((srgb + 0.055) / 1.055) ** 2.4)
def linear_to_srgb_np(lin):
    s = np.zeros_like(lin)
    mask = lin > 0.0031308
    s[mask] = 1.055 * (lin[mask]**(1/2.4)) - 0.055
    s[~mask] = 12.92 * lin[~mask]
    return np.clip(s, 0, 1)

def get_full_spectral_locus():
    # 380-780nm CMF
    cmf = np.array([
        [0.0014,0.0000,0.0065], [0.0042,0.0001,0.0201], [0.0143,0.0004,0.0679], [0.0435,0.0012,0.2074],
        [0.1344,0.0040,0.6456], [0.2839,0.0116,1.3856], [0.3483,0.0230,1.7471], [0.3362,0.0380,1.7721],
        [0.2908,0.0600,1.6692], [0.1954,0.0910,1.2876], [0.0956,0.1390,0.8130], [0.0320,0.2080,0.4652],
        [0.0049,0.3230,0.2720], [0.0093,0.5030,0.1582], [0.0633,0.7100,0.0782], [0.1655,0.8620,0.0422],
        [0.2904,0.9540,0.0203], [0.4334,0.9950,0.0087], [0.5945,0.9950,0.0039], [0.7621,0.9520,0.0021],
        [0.9163,0.8700,0.0017], [1.0263,0.7570,0.0011], [1.0622,0.6310,0.0008], [1.0026,0.5030,0.0006],
        [0.8544,0.3810,0.0002], [0.6424,0.2650,0.0000], [0.4479,0.1750,0.0000], [0.2835,0.1070,0.0000],
        [0.1649,0.0610,0.0000], [0.0874,0.0320,0.0000], [0.0468,0.0170,0.0000], [0.0227,0.0082,0.0000],
        [0.0114,0.0041,0.0000], [0.0058,0.0021,0.0000], [0.0029,0.0010,0.0000], [0.0014,0.0005,0.0000]
    ])
    # Close loop (Line of Purples)
    start_pt = cmf[-1]; end_pt = cmf[0]
    purples = np.linspace(start_pt, end_pt, 25)
    return np.vstack([cmf, purples])

# ==========================================
# 2. Optimization
# ==========================================

def get_gamut_surface(res=40):
    t = np.linspace(0, 1, res)
    g = np.meshgrid(t, t)
    faces = [np.stack([g[0], g[1], np.zeros_like(g[0])], -1), np.stack([g[0], g[1], np.ones_like(g[0])], -1),
             np.stack([g[0], np.zeros_like(g[0]), g[1]], -1), np.stack([g[0], np.ones_like(g[0]), g[1]], -1),
             np.stack([np.zeros_like(g[0]), g[0], g[1]], -1), np.stack([np.ones_like(g[0]), g[0], g[1]], -1)]
    pts = np.vstack([f.reshape(-1, 3) for f in faces])
    corners = np.array([[0,0,0], [1,0,0], [0,1,0], [0,0,1], [1,1,0], [1,0,1], [0,1,1], [1,1,1]])
    return np.vstack([pts, corners])

def fit_ellipsoid_anchored_solver(points, anchor_point):
    hull = ConvexHull(points)
    A = hull.equations[:, :3]; b_const = -hull.equations[:, 3]
    scale = 100.0; A_scaled = A / scale; anchor_scaled = anchor_point * scale
    c0 = np.mean(points, axis=0) * scale
    c0 = 0.5 * c0 + 0.5 * anchor_scaled 
    x0 = np.concatenate([c0, np.eye(3).flatten() * 0.05])

    def objective(x):
        M = x[3:].reshape(3, 3); sign, val = np.linalg.slogdet(M)
        return -val if sign > 0 else 1e9

    def constraints(x):
        d, M = x[:3], x[3:].reshape(3, 3)
        norm_AM = np.linalg.norm(A_scaled @ M, axis=1)
        hull_con = b_const - (A_scaled @ d + norm_AM)
        try: u = np.linalg.solve(M, anchor_scaled - d); anchor_con = 1.0 - np.linalg.norm(u)
        except: anchor_con = -1.0
        return np.concatenate([hull_con, [anchor_con]])

    res = opt.minimize(objective, x0, method='SLSQP', constraints={'type': 'ineq', 'fun': constraints}, options={'maxiter': 1000})
    return res.x[:3]/scale, res.x[3:].reshape(3, 3)/scale

def calculate_shadow_boundary(center, M, view_vector, res=72):
    w = np.linalg.solve(M, view_vector); w = w / np.linalg.norm(w)
    tmp = np.array([0.0, 1.0, 0.0]) if np.abs(w[1]) < 0.9 else np.array([0.0, 0.0, 1.0])
    s1 = np.cross(w, tmp); s1 /= np.linalg.norm(s1); s2 = np.cross(w, s1)
    theta = np.linspace(0, 2*np.pi, res)
    return center + (np.outer(np.cos(theta), s1) + np.outer(np.sin(theta), s2)) @ M.T

def inflate_to_max_gamut(center, M_init):
    scale = 1.0; step = 0.05; best_M = M_init.copy()
    for _ in range(50):
        test_M = M_init * (scale + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        scale += step; best_M = test_M
    step = 0.005; current_M = best_M
    for _ in range(20):
        test_M = current_M * (1.0 + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        current_M = test_M
    return current_M

def project_locus_onto_ellipsoid_surface(locus_xyz_raw, center, M):
    """
    Projects the spectral locus (XYZ) onto the surface of the Ellipsoid (center, M).
    Logic: Transform locus to 'Unit Sphere Space' relative to M, normalize to radius 1, transform back.
    """
    # 1. Convert Locus to Jzazbz
    locus_jz = xyz_to_jzazbz(locus_xyz_raw)
    
    # 2. Shift to Ellipsoid Local Space
    vecs = locus_jz - center
    
    # 3. Transform to Unit Sphere Space (u = M^-1 * v)
    M_inv = np.linalg.inv(M)
    # M is (3,3), vecs is (N,3). We want u = M_inv @ v.T
    # Transpose approach: u.T = vecs @ M_inv.T
    u_vecs = vecs @ M_inv.T
    
    # 4. Normalize to Surface (Radius = 1)
    norms = np.linalg.norm(u_vecs, axis=1, keepdims=True)
    u_surface = u_vecs / norms
    
    # 5. Transform back to World Space (p = M * u + center)
    projected_jz = (u_surface @ M.T) + center
    
    print("   Projected Locus onto Ellipsoid Surface.")
    return projected_jz

def check_gamut_compliance(rgb_loop, name="Loop"):
    min_val = np.min(rgb_loop); max_val = np.max(rgb_loop)
    tol = 2e-3
    if min_val < -tol or max_val > 1.0 + tol:
        print(f"   >>> {name} FAIL: [{min_val:.4f}, {max_val:.4f}]")
    else:
        print(f"   >>> {name} PASS: Inside sRGB")

# ==========================================
# 3. Omnipose Helpers
# ==========================================
def align_ring_red_to_green(ring_lin):
    n = len(ring_lin)
    red_ref = np.array([1.0, 0.0, 0.0])
    idx_r = np.argmin(np.linalg.norm(ring_lin - red_ref, axis=1))
    ring_aligned = np.roll(ring_lin, -idx_r, axis=0)
    green_ref = np.array([0.0, 1.0, 0.0])
    if np.linalg.norm(ring_aligned[(2*n)//3]-green_ref) < np.linalg.norm(ring_aligned[n//3]-green_ref):
        ring_aligned = ring_aligned[::-1]
    return ring_aligned

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    def build_disk(ring, center):
        y, x = np.mgrid[-1:1:256j, -1:1:256j]
        mag = np.sqrt(x*x + y*y)
        angle = np.mod(np.arctan2(y, x), 2*np.pi)
        n = ring.shape[0]
        hue_f = angle/(2*np.pi)*n
        idx0 = np.floor(hue_f).astype(int) % n
        idx1 = (idx0 + 1) % n
        t = hue_f - np.floor(hue_f)
        interp = (1-t[...,None])*ring[idx0] + t[...,None]*ring[idx1]
        rgb = (1-mag[...,None])*center + mag[...,None]*interp
        return np.clip(linear_to_srgb_np(rgb),0,1)

    disk = build_disk(rgb_ring_lin, center_lin)
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi)
    mag = np.sqrt(u*u + v*v); mag /= (mag.max() + 1e-8)
    n = rgb_ring_lin.shape[0]
    hue_f = angle/(2*np.pi)*n
    idx0 = np.floor(hue_f).astype(int) % n
    idx1 = (idx0 + 1) % n
    t = hue_f - np.floor(hue_f)
    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    r = mag[..., None]
    rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow),0,1)
    alpha = mag[...,None]
    white = np.ones_like(flow_rgb); black = np.zeros_like(flow_rgb)
    flow_white = alpha*flow_rgb + (1-alpha)*white
    flow_black = alpha*flow_rgb + (1-alpha)*black
    return disk, flow_rgb, flow_white, flow_black

# ==========================================
# 4. Main Execution
# ==========================================

def main():
    device_str = "cpu"
    dev = torch.device(device_str)
    
    print("1. Calculating Geometries...")
    surf_pts_opt = get_gamut_surface(40) 
    jz_gamut_opt = xyz_to_jzazbz(srgb_to_xyz(surf_pts_opt))
    
    centroid = np.mean(jz_gamut_opt, axis=0)
    green_vertex = xyz_to_jzazbz(srgb_to_xyz(np.array([[0.0, 1.0, 0.0]])))[0]
    
    # Anchor Strategy
    anchor = centroid + 0.60 * (green_vertex - centroid)
    
    print("2. Fitting Maximal Ellipsoid...")
    center, M_solver = fit_ellipsoid_anchored_solver(jz_gamut_opt, anchor)
    M_max = inflate_to_max_gamut(center, M_solver)
    
    # Loop A: Shadow Boundary (Geodesic Equator)
    hoop = calculate_shadow_boundary(center, M_max, np.array([1.0, 0.0, 0.0]), res=72)
    rgb_hoop_raw = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
    check_gamut_compliance(rgb_hoop_raw, "Ellipsoid Loop")
    rgb_hoop_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(rgb_hoop_raw,0,1)))
    
    # Loop B: Sinebow
    t = np.linspace(0, 1, 72, endpoint=False)
    sr = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0/3))
    sg = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1/3))
    sb = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2/3))
    rgb_sine_std = np.stack([sr, sg, sb], axis=1) 
    rgb_sine_lin = align_ring_red_to_green(srgb_to_linear_np(rgb_sine_std))

    # Loop C: Projected Spectral Locus (onto Ellipsoid Surface)
    print("3. Projecting Locus onto Ellipsoid Surface...")
    locus_xyz = get_full_spectral_locus()
    # Project onto the fitted M_max ellipsoid
    proj_locus_jz = project_locus_onto_ellipsoid_surface(locus_xyz, center, M_max)
    
    rgb_locus_raw = xyz_to_srgb_raw(jzazbz_to_xyz(proj_locus_jz))
    check_gamut_compliance(rgb_locus_raw, "Locus on Ellipsoid")
    # Note: Because ellipsoid is inside gamut, this projection is guaranteed valid/clipped safely
    rgb_locus_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(rgb_locus_raw, 0, 1)))

    print("4. Loading Omnipose Flows...")
    omnidir = Path(omnirefactor.__file__).resolve().parent.parent.parent
    basedir = omnidir / "docs" / "_static"
    name = "ecoli"
    ext = ".tif"

    try:
        masks = io.imread(str(basedir / f"{name}_labels{ext}"))
        slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows = omnirefactor.core.masks_to_flows(masks, device=device_str)[4].to(dev)
        center_lin = srgb_to_linear_np(np.array([0.5, 0.5, 0.5]))

        def gen_set(ring, rotation_steps=0):
            r = np.roll(ring, rotation_steps, axis=0)
            return make_flow_images_for_ring(r, center_lin, flows)

        disk_s_0, _, w_s_0, b_s_0 = gen_set(rgb_sine_lin, 0)
        disk_e_0, _, w_e_0, b_e_0 = gen_set(rgb_hoop_lin, 0)
        disk_l_0, _, w_l_0, b_l_0 = gen_set(rgb_locus_lin, 0)

        rot = 18
        _, _, w_s_90, b_s_90 = gen_set(rgb_sine_lin, rot)
        _, _, w_e_90, b_e_90 = gen_set(rgb_hoop_lin, rot)
        _, _, w_l_90, b_l_90 = gen_set(rgb_locus_lin, rot)

        print("5. Plotting 2D...")
        plt.style.use('dark_background')
        fig, axes = plt.subplots(3, 5, figsize=(20, 12))
        rows = [("Sinebow", disk_s_0, b_s_0, w_s_0, b_s_90, w_s_90),
                ("Ellipsoid Loop", disk_e_0, b_e_0, w_e_0, b_e_90, w_e_90),
                ("Locus on Ellipsoid", disk_l_0, b_l_0, w_l_0, b_l_90, w_l_90)]
        titles = ["Hue Disk", "Black (0°)", "White (0°)", "Black (90°)", "White (90°)"]

        for i, (name, disk, b0, w0, b90, w90) in enumerate(rows):
            ax_row = axes[i]
            ax_row[0].imshow(disk)
            ax_row[0].set_ylabel(name, fontsize=14, color='white', rotation=90, labelpad=20)
            ax_row[1].imshow(b0); ax_row[2].imshow(w0)
            ax_row[3].imshow(b90); ax_row[4].imshow(w90)
            for j, ax in enumerate(ax_row):
                if i == 0: ax.set_title(titles[j], fontsize=12, color='#dddddd')
                ax.set_xticks([]); ax.set_yticks([])
        plt.tight_layout()
        plt.show()
        
        print("6. Generating 3D...")
        fig3d = go.Figure()
        max_jz = np.max(jz_gamut_opt[:,0]); sc = np.array([1.0/max_jz, 1.0, 1.0]); 
        def mc(a): return a[:,1], a[:,2], a[:,0]

        # Faces
        N=10; t=np.linspace(0,1,N); g=np.meshgrid(t,t)
        rgb_faces = [np.stack([g[0],g[1],np.zeros_like(g[0])],-1), np.stack([g[0],g[1],np.ones_like(g[0])],-1),
                     np.stack([g[0],np.zeros_like(g[0]),g[1]],-1), np.stack([g[0],np.ones_like(g[0]),g[1]],-1),
                     np.stack([np.zeros_like(g[0]),g[0],g[1]],-1), np.stack([np.ones_like(g[0]),g[0],g[1]],-1)]
        i_idx, j_idx, k_idx = [], [], []
        for r in range(N-1):
            for c in range(N-1):
                p1=r*N+c; p2=r*N+(c+1); p3=(r+1)*N+c; p4=(r+1)*N+(c+1)
                i_idx.extend([p1,p2]); j_idx.extend([p2,p4]); k_idx.extend([p3,p3])
        for i, face in enumerate(rgb_faces):
            jz_face = xyz_to_jzazbz(srgb_to_xyz(face.reshape(-1,3))) * sc
            x,y,z = mc(jz_face)
            fig3d.add_trace(go.Mesh3d(x=x, y=y, z=z, i=i_idx, j=j_idx, k=k_idx, color='gray', opacity=0.15, flatshading=False, hoverinfo='skip'))

        # Edges
        corners = [[0,0,0],[1,0,0],[1,1,0],[0,1,0],[0,0,1],[1,0,1],[1,1,1],[0,1,1]]
        edges_idx = [[0,1],[1,2],[2,3],[3,0],[4,5],[5,6],[6,7],[7,4],[0,4],[1,5],[2,6],[3,7]]
        for (s,e) in edges_idx:
            pts = np.linspace(corners[s],corners[e],25)
            line = xyz_to_jzazbz(srgb_to_xyz(pts)) * sc
            x,y,z = mc(line)
            fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', line=dict(color='white', width=3), showlegend=False))

        # Ellipsoid
        u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
        sph = np.stack([np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v)], axis=-1).reshape(-1, 3)
        ell = (sph @ M_max.T + center) * sc
        ex, ey, ez = mc(ell)
        fig3d.add_trace(go.Surface(x=ex.reshape(u.shape), y=ey.reshape(u.shape), z=ez.reshape(u.shape),
                                colorscale=[(0,'#88aa88'),(1,'#88aa88')], opacity=0.4, showscale=False, name='Anchored Ellipsoid'))

        # Loops
        hx, hy, hz = mc(hoop * sc)
        c_hoop = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in linear_to_srgb_np(np.clip(rgb_hoop_raw,0,1))]
        fig3d.add_trace(go.Scatter3d(x=hx, y=hy, z=hz, mode='markers', marker=dict(color=c_hoop, size=5), name='Ellipsoid Loop'))

        sx, sy, sz = mc(xyz_to_jzazbz(srgb_to_xyz(rgb_sine_std)) * sc)
        c_sine = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in rgb_sine_std]
        fig3d.add_trace(go.Scatter3d(x=sx, y=sy, z=sz, mode='markers', marker=dict(color=c_sine, size=4, symbol='diamond'), name='Sinebow Loop'))
        
        # Locus on Ellipsoid
        lx, ly, lz = mc(proj_locus_jz * sc)
        fig3d.add_trace(go.Scatter3d(x=lx, y=ly, z=lz, mode='lines', line=dict(color='cyan', width=5), name='Locus on Ellipsoid'))

        # Full Locus (Ref)
        lx_f, ly_f, lz_f = mc(xyz_to_jzazbz(locus_xyz) * sc)
        fig3d.add_trace(go.Scatter3d(x=lx_f, y=ly_f, z=lz_f, mode='lines', line=dict(color='magenta', width=3, dash='dash'), name='Full Locus (Ref)'))

        fig3d.update_layout(template="plotly_dark", title="3D Jzazbz (Maximal + Locus on Ellipsoid)", 
                            scene=dict(xaxis_title='az', yaxis_title='bz', zaxis_title='Jz', aspectmode='manual', aspectratio=dict(x=1,y=1,z=1)))
        fig3d.show()

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
wrap pruples u somehow first, then fourer approximate? then project? 
some kind of projection from the zero boint ito the ellipse or soemthing could define the lightening

In [ ]:
import numpy as np
import scipy.optimize as opt
from scipy.spatial import ConvexHull
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import omnirefactor.core
from skimage import io
import fastremap

# ==========================================
# 1. Color Math
# ==========================================
def srgb_to_xyz(rgb):
    mask = rgb > 0.04045
    linear = np.zeros_like(rgb)
    linear[mask] = ((rgb[mask] + 0.055) / 1.055) ** 2.4
    linear[~mask] = rgb[~mask] / 12.92
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    return linear @ M.T

def xyz_to_jzazbz(xyz):
    b, g, d, d0 = 1.15, 0.66, -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    lms = xyz @ M1.T
    lms_norm = (lms * 200) / 10000.0 
    lms_prime = np.sign(lms_norm) * (((c1 + c2 * (np.abs(lms_norm) ** n)) / (1 + c3 * (np.abs(lms_norm) ** n))) ** p)
    izazbz = lms_prime @ M2.T
    Jz = ((1 + d) * izazbz[:, 0]) / (1 + d * izazbz[:, 0]) - d0
    return np.column_stack((Jz, izazbz[:, 1], izazbz[:, 2]))

def jzazbz_to_xyz(jzazbz):
    Jz, az, bz = jzazbz[:,0], jzazbz[:,1], jzazbz[:,2]
    d, d0 = -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    Jp = Jz + d0; Iz = Jp / (1 + d - d * Jp)
    izazbz_vec = np.stack([Iz, az, bz], axis=1)
    lms_prime = izazbz_vec @ np.linalg.inv(M2).T
    sign_lms = np.sign(lms_prime)
    Y = np.abs(lms_prime) ** (1/p)
    num = Y - c1; den = c2 - Y * c3
    An = np.clip(num / den, 0, None) 
    lms_norm = sign_lms * (An ** (1/n))
    lms = (lms_norm * 10000.0) / 200.0
    return lms @ np.linalg.inv(M1).T

def xyz_to_srgb_raw(xyz):
    M_inv = np.linalg.inv(np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]]))
    linear = xyz @ M_inv.T
    srgb = np.zeros_like(linear)
    mask = linear > 0.0031308
    srgb[mask] = 1.055 * (linear[mask] ** (1/2.4)) - 0.055
    srgb[~mask] = 12.92 * linear[~mask]
    return srgb

def xyz_to_srgb(xyz): return np.clip(xyz_to_srgb_raw(xyz), 0, 1)
def srgb_to_linear_np(srgb): return np.where(srgb <= 0.04045, srgb / 12.92, ((srgb + 0.055) / 1.055) ** 2.4)
def linear_to_srgb_np(lin):
    s = np.zeros_like(lin)
    mask = lin > 0.0031308
    s[mask] = 1.055 * (lin[mask]**(1/2.4)) - 0.055
    s[~mask] = 12.92 * lin[~mask]
    return np.clip(s, 0, 1)

def get_full_spectral_locus():
    # CIE 1931 2-deg (380-780nm)
    cmf = np.array([
        [0.0014,0.0000,0.0065], [0.0042,0.0001,0.0201], [0.0143,0.0004,0.0679], [0.0435,0.0012,0.2074],
        [0.1344,0.0040,0.6456], [0.2839,0.0116,1.3856], [0.3483,0.0230,1.7471], [0.3362,0.0380,1.7721],
        [0.2908,0.0600,1.6692], [0.1954,0.0910,1.2876], [0.0956,0.1390,0.8130], [0.0320,0.2080,0.4652],
        [0.0049,0.3230,0.2720], [0.0093,0.5030,0.1582], [0.0633,0.7100,0.0782], [0.1655,0.8620,0.0422],
        [0.2904,0.9540,0.0203], [0.4334,0.9950,0.0087], [0.5945,0.9950,0.0039], [0.7621,0.9520,0.0021],
        [0.9163,0.8700,0.0017], [1.0263,0.7570,0.0011], [1.0622,0.6310,0.0008], [1.0026,0.5030,0.0006],
        [0.8544,0.3810,0.0002], [0.6424,0.2650,0.0000], [0.4479,0.1750,0.0000], [0.2835,0.1070,0.0000],
        [0.1649,0.0610,0.0000], [0.0874,0.0320,0.0000], [0.0468,0.0170,0.0000], [0.0227,0.0082,0.0000],
        [0.0114,0.0041,0.0000], [0.0058,0.0021,0.0000], [0.0029,0.0010,0.0000], [0.0014,0.0005,0.0000]
    ])
    # Close loop (Line of Purples)
    start_pt = cmf[-1]; end_pt = cmf[0]
    purples = np.linspace(start_pt, end_pt, 25)
    return np.vstack([cmf, purples])

# ==========================================
# 2. MacAdam Limit Calculation
# ==========================================
def get_cmf_interpolated(res=360):
    # Interpolate standard CMF to 1nm resolution (approx)
    base_cmf = get_full_spectral_locus()[:36] # Just spectral part
    x = np.linspace(0, 1, len(base_cmf))
    xi = np.linspace(0, 1, res)
    
    interp_cmf = np.zeros((res, 3))
    for k in range(3):
        interp_cmf[:,k] = np.interp(xi, x, base_cmf[:,k])
    return interp_cmf

def compute_macadam_solid(res=100):
    """
    Computes the Optimal Color Solid (MacAdam Limit) by integrating
    ideal block spectra.
    """
    cmf = get_cmf_interpolated(res)
    n_lambda = len(cmf)
    
    # Pre-calculate cumulative sum for fast integration
    cum_cmf = np.vstack([np.zeros(3), np.cumsum(cmf, axis=0)])
    total_white = cum_cmf[-1]
    
    optimal_colors = []
    
    # Iterate start and end wavelengths to create Band-Pass (Type 1) and Band-Stop (Type 2)
    # This creates the "surface" of the solid
    indices = np.linspace(0, n_lambda, 60).astype(int)
    
    for i in indices:
        for j in indices:
            if i == j: continue
            
            # Calculate integral from lambda_i to lambda_j
            if j > i:
                # One continuous band (Type 1)
                xyz = cum_cmf[j] - cum_cmf[i]
            else:
                # Wrap around (Type 2: Purple boundary)
                # Integral(0 to j) + Integral(i to end)
                xyz = cum_cmf[j] + (total_white - cum_cmf[i])
                
            optimal_colors.append(xyz)
            
    # Add Black and White explicitly
    optimal_colors.append([0,0,0])
    optimal_colors.append(total_white)
    
    # Normalize so Y max = 1 (or 100 if you prefer, here we use 0-1 range logic)
    opt_xyz = np.array(optimal_colors)
    # Scale so White Y = 1.0
    scale = 1.0 / total_white[1]
    return opt_xyz * scale

# ==========================================
# 3. Optimization
# ==========================================

def get_gamut_surface(res=16):
    t = np.linspace(0, 1, res)
    g = np.meshgrid(t, t)
    faces = [np.stack([g[0], g[1], np.zeros_like(g[0])], -1), np.stack([g[0], g[1], np.ones_like(g[0])], -1),
             np.stack([g[0], np.zeros_like(g[0]), g[1]], -1), np.stack([g[0], np.ones_like(g[0]), g[1]], -1),
             np.stack([np.zeros_like(g[0]), g[0], g[1]], -1), np.stack([np.ones_like(g[0]), g[0], g[1]], -1)]
    pts = np.vstack([f.reshape(-1, 3) for f in faces])
    corners = np.array([[0,0,0], [1,0,0], [0,1,0], [0,0,1], [1,1,0], [1,0,1], [0,1,1], [1,1,1]])
    return np.vstack([pts, corners])

def fit_ellipsoid_anchored_solver(points, anchor_point):
    hull = ConvexHull(points)
    A = hull.equations[:, :3]; b_const = -hull.equations[:, 3]
    scale = 100.0; A_scaled = A / scale; anchor_scaled = anchor_point * scale
    c0 = np.mean(points, axis=0) * scale
    c0 = 0.5 * c0 + 0.5 * anchor_scaled 
    x0 = np.concatenate([c0, np.eye(3).flatten() * 0.05])

    def objective(x):
        M = x[3:].reshape(3, 3); sign, val = np.linalg.slogdet(M)
        return -val if sign > 0 else 1e9

    def constraints(x):
        d, M = x[:3], x[3:].reshape(3, 3)
        norm_AM = np.linalg.norm(A_scaled @ M, axis=1)
        hull_con = b_const - (A_scaled @ d + norm_AM)
        try: u = np.linalg.solve(M, anchor_scaled - d); anchor_con = 1.0 - np.linalg.norm(u)
        except: anchor_con = -1.0
        return np.concatenate([hull_con, [anchor_con]])

    res = opt.minimize(objective, x0, method='SLSQP', constraints={'type': 'ineq', 'fun': constraints}, options={'maxiter': 1000})
    return res.x[:3]/scale, res.x[3:].reshape(3, 3)/scale

def calculate_shadow_boundary(center, M, view_vector, res=72):
    w = np.linalg.solve(M, view_vector); w = w / np.linalg.norm(w)
    tmp = np.array([0.0, 1.0, 0.0]) if np.abs(w[1]) < 0.9 else np.array([0.0, 0.0, 1.0])
    s1 = np.cross(w, tmp); s1 /= np.linalg.norm(s1); s2 = np.cross(w, s1)
    theta = np.linspace(0, 2*np.pi, res)
    return center + (np.outer(np.cos(theta), s1) + np.outer(np.sin(theta), s2)) @ M.T

def inflate_to_max_gamut(center, M_init):
    scale = 1.0; step = 0.05; best_M = M_init.copy()
    for _ in range(50):
        test_M = M_init * (scale + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        scale += step; best_M = test_M
    step = 0.005; current_M = best_M
    for _ in range(20):
        test_M = current_M * (1.0 + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        current_M = test_M
    return current_M

def project_locus_onto_ellipsoid_surface(locus_xyz_raw, center, M):
    locus_jz = xyz_to_jzazbz(locus_xyz_raw)
    vecs = locus_jz - center
    M_inv = np.linalg.inv(M)
    u_vecs = vecs @ M_inv.T
    norms = np.linalg.norm(u_vecs, axis=1, keepdims=True)
    u_surface = u_vecs / norms
    return (u_surface @ M.T) + center

# ==========================================
# 4. Omnipose Helpers
# ==========================================
def align_ring_red_to_green(ring_lin):
    n = len(ring_lin)
    red_ref = np.array([1.0, 0.0, 0.0])
    idx_r = np.argmin(np.linalg.norm(ring_lin - red_ref, axis=1))
    ring_aligned = np.roll(ring_lin, -idx_r, axis=0)
    green_ref = np.array([0.0, 1.0, 0.0])
    if np.linalg.norm(ring_aligned[(2*n)//3]-green_ref) < np.linalg.norm(ring_aligned[n//3]-green_ref):
        ring_aligned = ring_aligned[::-1]
    return ring_aligned

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    def build_disk(ring, center):
        y, x = np.mgrid[-1:1:256j, -1:1:256j]
        mag = np.sqrt(x*x + y*y)
        angle = np.mod(np.arctan2(y, x), 2*np.pi)
        n = ring.shape[0]
        hue_f = angle/(2*np.pi)*n
        idx0 = np.floor(hue_f).astype(int) % n
        idx1 = (idx0 + 1) % n
        t = hue_f - np.floor(hue_f)
        interp = (1-t[...,None])*ring[idx0] + t[...,None]*ring[idx1]
        rgb = (1-mag[...,None])*center + mag[...,None]*interp
        return np.clip(linear_to_srgb_np(rgb),0,1)

    disk = build_disk(rgb_ring_lin, center_lin)
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi)
    mag = np.sqrt(u*u + v*v); mag /= (mag.max() + 1e-8)
    n = rgb_ring_lin.shape[0]
    hue_f = angle/(2*np.pi)*n
    idx0 = np.floor(hue_f).astype(int) % n
    idx1 = (idx0 + 1) % n
    t = hue_f - np.floor(hue_f)
    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    r = mag[..., None]
    rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow),0,1)
    alpha = mag[...,None]
    white = np.ones_like(flow_rgb); black = np.zeros_like(flow_rgb)
    flow_white = alpha*flow_rgb + (1-alpha)*white
    flow_black = alpha*flow_rgb + (1-alpha)*black
    return disk, flow_rgb, flow_white, flow_black

# ==========================================
# 5. Main Execution
# ==========================================

def main():
    device_str = "cpu"
    dev = torch.device(device_str)
    
    print("1. Calculating Geometries...")
    surf_pts_opt = get_gamut_surface(40) 
    jz_gamut_opt = xyz_to_jzazbz(srgb_to_xyz(surf_pts_opt))
    centroid = np.mean(jz_gamut_opt, axis=0)
    green_vertex = xyz_to_jzazbz(srgb_to_xyz(np.array([[0.0, 1.0, 0.0]])))[0]
    anchor = centroid + 0.60 * (green_vertex - centroid)
    
    print("2. Fitting Maximal Ellipsoid...")
    center, M_solver = fit_ellipsoid_anchored_solver(jz_gamut_opt, anchor)
    M_max = inflate_to_max_gamut(center, M_solver)
    
    hoop = calculate_shadow_boundary(center, M_max, np.array([1.0, 0.0, 0.0]), res=72)
    rgb_hoop_raw = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
    rgb_hoop_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(rgb_hoop_raw,0,1)))
    
    # Sinebow
    t = np.linspace(0, 1, 72, endpoint=False)
    sr = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0/3))
    sg = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1/3))
    sb = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2/3))
    rgb_sine_std = np.stack([sr, sg, sb], axis=1) 
    rgb_sine_lin = align_ring_red_to_green(srgb_to_linear_np(rgb_sine_std))

    # Loop C: Lifted Locus (420-660nm, lifted) on Ellipsoid
    print("3. Projecting Lifted Locus...")
    
    # Re-using the lifted locus function logic inline for brevity/clarity since we removed it to fit the solid logic
    # Standard CMF (Truncated 420-660nm)
    cmf = np.array([
        [0.1344,0.0040,0.6456], [0.2839,0.0116,1.3856], [0.3483,0.0230,1.7471], [0.3362,0.0380,1.7721], 
        [0.2908,0.0600,1.6692], [0.1954,0.0910,1.2876], [0.0956,0.1390,0.8130], [0.0320,0.2080,0.4652],
        [0.0049,0.3230,0.2720], [0.0093,0.5030,0.1582], [0.0633,0.7100,0.0782], [0.1655,0.8620,0.0422],
        [0.2904,0.9540,0.0203], [0.4334,0.9950,0.0087], [0.5945,0.9950,0.0039], [0.7621,0.9520,0.0021],
        [0.9163,0.8700,0.0017], [1.0263,0.7570,0.0011], [1.0622,0.6310,0.0008], [1.0026,0.5030,0.0006],
        [0.8544,0.3810,0.0002], [0.6424,0.2650,0.0000], [0.4479,0.1750,0.0000], [0.2835,0.1070,0.0000],
        [0.1649,0.0610,0.0000]
    ])
    cmf = cmf / np.max(cmf[:,1]) * 0.8 
    locus_jz = xyz_to_jzazbz(cmf)
    start_pt, end_pt = locus_jz[-1], locus_jz[0]
    n_bridge = 30; t_b = np.linspace(0, 1, n_bridge)
    J1, a1, b1 = start_pt; J2, a2, b2 = end_pt
    # Simple linear bridge with sine lift
    J_bridge = (1-t_b)*J1 + t_b*J2 + 0.08*np.sin(np.pi*t_b)
    a_bridge = (1-t_b)*a1 + t_b*a2
    b_bridge = (1-t_b)*b1 + t_b*b2
    bridge_jz = np.column_stack([J_bridge, a_bridge, b_bridge])
    lifted_locus_jz = np.vstack([locus_jz, bridge_jz])
    
    # Project onto surface relative to center (M_max)
    # Note: We pass Jzazbz directly to project_locus_onto_ellipsoid_surface (modified)
    # Actually, the function expects XYZ, let's fix it or pass XYZ.
    # Let's modify project_locus to accept Jz directly for simplicity
    vecs = lifted_locus_jz - center
    M_inv = np.linalg.inv(M_max)
    u_vecs = vecs @ M_inv.T
    u_surf = u_vecs / np.linalg.norm(u_vecs, axis=1, keepdims=True)
    proj_locus_jz = (u_surf @ M_max.T) + center
    
    rgb_proj_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(proj_locus_jz)), 0, 1)))

    print("4. Loading Omnipose Flows...")
    omnidir = Path(omnirefactor.__file__).resolve().parent.parent.parent
    basedir = omnidir / "docs" / "_static"
    name = "ecoli"
    ext = ".tif"

    try:
        masks = io.imread(str(basedir / f"{name}_labels{ext}"))
        slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows = omnirefactor.core.masks_to_flows(masks, device=device_str)[4].to(dev)
        center_lin = srgb_to_linear_np(np.array([0.5, 0.5, 0.5]))

        def gen_set(ring, rotation_steps=0):
            r = np.roll(ring, rotation_steps, axis=0)
            return make_flow_images_for_ring(r, center_lin, flows)

        # 2D Rows
        rows = []
        d, f, w, b = gen_set(rgb_sine_lin, 0); d9, f9, w9, b9 = gen_set(rgb_sine_lin, 18)
        rows.append(("Sinebow", rgb_sine_lin, d, b, w, b9, w9))
        
        d, f, w, b = gen_set(rgb_hoop_lin, 0); d9, f9, w9, b9 = gen_set(rgb_hoop_lin, 18)
        rows.append(("Max Ellipse", rgb_hoop_lin, d, b, w, b9, w9))
        
        d, f, w, b = gen_set(rgb_proj_lin, 0); d9, f9, w9, b9 = gen_set(rgb_proj_lin, 18)
        rows.append(("Locus on Ellipse", rgb_proj_lin, d, b, w, b9, w9))

        print("5. Plotting 2D...")
        plt.style.use('dark_background')
        fig, axes = plt.subplots(3, 5, figsize=(20, 12))
        titles = ["Gradient", "Hue Disk", "Black (0°)", "White (0°)", "Black (90°)"]

        for i, (name, ring, disk, b0, w0, b90, w90) in enumerate(rows):
            ax_row = axes[i]
            disp_gradient = linear_to_srgb_np(ring)[None, ...]
            ax_row[0].imshow(disp_gradient, aspect='auto')
            ax_row[0].set_ylabel(name, fontsize=14, color='white', rotation=90, labelpad=20)
            ax_row[1].imshow(disk); ax_row[2].imshow(b0); ax_row[3].imshow(w0); ax_row[4].imshow(b90)
            for j, ax in enumerate(ax_row):
                if i==0: ax.set_title(titles[j], fontsize=12, color='#dddddd')
                ax.set_xticks([]); ax.set_yticks([])
        plt.tight_layout()
        plt.show()
        
        print("6. Generating 3D...")
        # Compute MacAdam Solid
        print("   Computing MacAdam Solid (this may take a moment)...")
        macadam_xyz = compute_macadam_solid(res=60) # Medium res for speed
        macadam_jz = xyz_to_jzazbz(macadam_xyz)
        # MacAdam Hull
        hull_macadam = ConvexHull(macadam_jz)
        
        fig3d = go.Figure()
        max_jz = np.max(jz_gamut_opt[:,0]); sc = np.array([1.0/max_jz, 1.0, 1.0]); 
        def mc(a): return a[:,1], a[:,2], a[:,0]

        # 1. MacAdam Solid (The True Limit)
        mx, my, mz = mc(macadam_jz * sc)
        fig3d.add_trace(go.Mesh3d(x=mx, y=my, z=mz,
            i=hull_macadam.simplices[:,0], j=hull_macadam.simplices[:,1], k=hull_macadam.simplices[:,2],
            color='cyan', opacity=0.05, name='MacAdam Solid (True Limit)'))

        # 2. sRGB Gamut (Wireframe + Faces)
        # Faces
        N=10; t=np.linspace(0,1,N); g=np.meshgrid(t,t)
        rgb_faces = [np.stack([g[0],g[1],np.zeros_like(g[0])],-1), np.stack([g[0],g[1],np.ones_like(g[0])],-1),
                     np.stack([g[0],np.zeros_like(g[0]),g[1]],-1), np.stack([g[0],np.ones_like(g[0]),g[1]],-1),
                     np.stack([np.zeros_like(g[0]),g[0],g[1]],-1), np.stack([np.ones_like(g[0]),g[0],g[1]],-1)]
        i_idx, j_idx, k_idx = [], [], []
        for r in range(N-1):
            for c in range(N-1):
                p1=r*N+c; p2=r*N+(c+1); p3=(r+1)*N+c; p4=(r+1)*N+(c+1)
                i_idx.extend([p1,p2]); j_idx.extend([p2,p4]); k_idx.extend([p3,p3])
        for i, face in enumerate(rgb_faces):
            jz_face = xyz_to_jzazbz(srgb_to_xyz(face.reshape(-1,3))) * sc
            x,y,z = mc(jz_face)
            fig3d.add_trace(go.Mesh3d(x=x, y=y, z=z, i=i_idx, j=j_idx, k=k_idx, color='gray', opacity=0.15, flatshading=False, hoverinfo='skip'))
        
        # 3. Ellipsoid
        u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
        sph = np.stack([np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v)], axis=-1).reshape(-1, 3)
        ell = (sph @ M_max.T + center) * sc
        ex, ey, ez = mc(ell)
        fig3d.add_trace(go.Surface(x=ex.reshape(u.shape), y=ey.reshape(u.shape), z=ez.reshape(u.shape),
                                colorscale=[(0,'#88aa88'),(1,'#88aa88')], opacity=0.3, showscale=False, name='Ellipsoid'))

        # 4. Loops
        hx, hy, hz = mc(hoop * sc)
        c_hoop = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in linear_to_srgb_np(np.clip(rgb_hoop_raw,0,1))]
        fig3d.add_trace(go.Scatter3d(x=hx, y=hy, z=hz, mode='markers', marker=dict(color=c_hoop, size=5), name='Ellipsoid Loop'))

        sx, sy, sz = mc(xyz_to_jzazbz(srgb_to_xyz(rgb_sine_std)) * sc)
        c_sine = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in rgb_sine_std]
        fig3d.add_trace(go.Scatter3d(x=sx, y=sy, z=sz, mode='markers', marker=dict(color=c_sine, size=4, symbol='diamond'), name='Sinebow Loop'))

        # 5. Projected Locus
        px, py, pz = mc(proj_locus_jz * sc)
        fig3d.add_trace(go.Scatter3d(x=px, y=py, z=pz, mode='lines', line=dict(color='cyan', width=4), name='Locus on Ellipsoid'))

        fig3d.update_layout(template="plotly_dark", title="3D Jzazbz: MacAdam Solid vs sRGB vs Ellipsoid", 
                            scene=dict(xaxis_title='az', yaxis_title='bz', zaxis_title='Jz', aspectmode='manual', aspectratio=dict(x=1,y=1,z=1)))
        fig3d.show()

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
maybe find the plane where spectral locs is most flat or convex, project onto that plan, then project that onto 
the ellipse
oooh, maybe the goal should be to get it so that is ab location is the same as the ellipse slice but the j can vary 

In [ ]:
import numpy as np
import scipy.optimize as opt
from scipy.spatial import ConvexHull
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import omnirefactor.core
from skimage import io
import fastremap

# ==========================================
# 1. Color Math
# ==========================================
def srgb_to_xyz(rgb):
    # Input: Gamma-encoded sRGB (0-1)
    mask = rgb > 0.04045
    linear = np.zeros_like(rgb)
    linear[mask] = ((rgb[mask] + 0.055) / 1.055) ** 2.4
    linear[~mask] = rgb[~mask] / 12.92
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    return linear @ M.T

def xyz_to_jzazbz(xyz):
    b, g, d, d0 = 1.15, 0.66, -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    lms = xyz @ M1.T
    lms_norm = (lms * 200) / 10000.0 
    lms_prime = np.sign(lms_norm) * (((c1 + c2 * (np.abs(lms_norm) ** n)) / (1 + c3 * (np.abs(lms_norm) ** n))) ** p)
    izazbz = lms_prime @ M2.T
    Jz = ((1 + d) * izazbz[:, 0]) / (1 + d * izazbz[:, 0]) - d0
    return np.column_stack((Jz, izazbz[:, 1], izazbz[:, 2]))

def jzazbz_to_xyz(jzazbz):
    Jz, az, bz = jzazbz[:,0], jzazbz[:,1], jzazbz[:,2]
    d, d0 = -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    Jp = Jz + d0; Iz = Jp / (1 + d - d * Jp)
    izazbz_vec = np.stack([Iz, az, bz], axis=1)
    lms_prime = izazbz_vec @ np.linalg.inv(M2).T
    sign_lms = np.sign(lms_prime)
    Y = np.abs(lms_prime) ** (1/p)
    num = Y - c1; den = c2 - Y * c3
    An = np.clip(num / den, 0, None) 
    lms_norm = sign_lms * (An ** (1/n))
    lms = (lms_norm * 10000.0) / 200.0
    return lms @ np.linalg.inv(M1).T

def xyz_to_srgb_raw(xyz):
    M_inv = np.linalg.inv(np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]]))
    linear = xyz @ M_inv.T
    srgb = np.zeros_like(linear)
    mask = linear > 0.0031308
    srgb[mask] = 1.055 * (linear[mask] ** (1/2.4)) - 0.055
    srgb[~mask] = 12.92 * linear[~mask]
    return srgb

def xyz_to_srgb(xyz): return np.clip(xyz_to_srgb_raw(xyz), 0, 1)
def srgb_to_linear_np(srgb): return np.where(srgb <= 0.04045, srgb / 12.92, ((srgb + 0.055) / 1.055) ** 2.4)
def linear_to_srgb_np(lin): 
    s = np.zeros_like(lin)
    mask = lin > 0.0031308
    s[mask] = 1.055 * (lin[mask]**(1/2.4)) - 0.055
    s[~mask] = 12.92 * lin[~mask]
    return np.clip(s, 0, 1)

def get_full_spectral_locus_raw(res=360):
    # CIE 1931 2-deg (380-780nm), approx 5nm steps
    cmf = np.array([
        [0.0014,0.0000,0.0065], [0.0042,0.0001,0.0201], [0.0143,0.0004,0.0679], [0.0435,0.0012,0.2074],
        [0.1344,0.0040,0.6456], [0.2839,0.0116,1.3856], [0.3483,0.0230,1.7471], [0.3362,0.0380,1.7721],
        [0.2908,0.0600,1.6692], [0.1954,0.0910,1.2876], [0.0956,0.1390,0.8130], [0.0320,0.2080,0.4652],
        [0.0049,0.3230,0.2720], [0.0093,0.5030,0.1582], [0.0633,0.7100,0.0782], [0.1655,0.8620,0.0422],
        [0.2904,0.9540,0.0203], [0.4334,0.9950,0.0087], [0.5945,0.9950,0.0039], [0.7621,0.9520,0.0021],
        [0.9163,0.8700,0.0017], [1.0263,0.7570,0.0011], [1.0622,0.6310,0.0008], [1.0026,0.5030,0.0006],
        [0.8544,0.3810,0.0002], [0.6424,0.2650,0.0000], [0.4479,0.1750,0.0000], [0.2835,0.1070,0.0000],
        [0.1649,0.0610,0.0000], [0.0874,0.0320,0.0000], [0.0468,0.0170,0.0000], [0.0227,0.0082,0.0000],
        [0.0114,0.0041,0.0000], [0.0058,0.0021,0.0000], [0.0029,0.0010,0.0000], [0.0014,0.0005,0.0000]
    ])
    # Interpolate to 1nm resolution for MacAdam Calculation
    x = np.linspace(0, 1, len(cmf))
    xi = np.linspace(0, 1, res)
    interp_cmf = np.zeros((res, 3))
    for k in range(3):
        interp_cmf[:,k] = np.interp(xi, x, cmf[:,k])
    return interp_cmf

def compute_macadam_solid(res=60):
    """
    Generates the Optimal Color Solid (MacAdam Limit) by integrating 
    block spectra (Band-pass and Band-stop) over the CMFs.
    """
    cmf = get_full_spectral_locus_raw(res=180) # Higher res for integration
    n_lambda = len(cmf)
    
    # Precompute integral
    cum_cmf = np.vstack([np.zeros(3), np.cumsum(cmf, axis=0)])
    total_white = cum_cmf[-1]
    
    optimal_colors = []
    
    # Subsample indices for speed
    indices = np.linspace(0, n_lambda, res).astype(int)
    
    for i in indices:
        for j in indices:
            if i == j: continue
            
            # Band Pass (Type 1)
            if j > i:
                xyz = cum_cmf[j] - cum_cmf[i]
            # Band Stop (Type 2 - Purple Boundary)
            else:
                xyz = cum_cmf[j] + (total_white - cum_cmf[i])
            optimal_colors.append(xyz)
            
    # Add Black and White
    optimal_colors.append([0,0,0])
    optimal_colors.append(total_white)
    
    opt_xyz = np.array(optimal_colors)
    # Normalize so White Y = 1.0
    return opt_xyz / total_white[1]

# ==========================================
# 2. Optimization
# ==========================================

def get_gamut_surface(res=40):
    t = np.linspace(0, 1, res)
    g = np.meshgrid(t, t)
    faces = [np.stack([g[0], g[1], np.zeros_like(g[0])], -1), np.stack([g[0], g[1], np.ones_like(g[0])], -1),
             np.stack([g[0], np.zeros_like(g[0]), g[1]], -1), np.stack([g[0], np.ones_like(g[0]), g[1]], -1),
             np.stack([np.zeros_like(g[0]), g[0], g[1]], -1), np.stack([np.ones_like(g[0]), g[0], g[1]], -1)]
    pts = np.vstack([f.reshape(-1, 3) for f in faces])
    corners = np.array([[0,0,0], [1,0,0], [0,1,0], [0,0,1], [1,1,0], [1,0,1], [0,1,1], [1,1,1]])
    return np.vstack([pts, corners])

def fit_ellipsoid_anchored_solver(points, anchor_point):
    hull = ConvexHull(points)
    A = hull.equations[:, :3]; b_const = -hull.equations[:, 3]
    scale = 100.0; A_scaled = A / scale; anchor_scaled = anchor_point * scale
    c0 = np.mean(points, axis=0) * scale
    c0 = 0.5 * c0 + 0.5 * anchor_scaled 
    x0 = np.concatenate([c0, np.eye(3).flatten() * 0.05])

    def objective(x):
        M = x[3:].reshape(3, 3); sign, val = np.linalg.slogdet(M)
        return -val if sign > 0 else 1e9

    def constraints(x):
        d, M = x[:3], x[3:].reshape(3, 3)
        norm_AM = np.linalg.norm(A_scaled @ M, axis=1)
        hull_con = b_const - (A_scaled @ d + norm_AM)
        try: u = np.linalg.solve(M, anchor_scaled - d); anchor_con = 1.0 - np.linalg.norm(u)
        except: anchor_con = -1.0
        return np.concatenate([hull_con, [anchor_con]])

    res = opt.minimize(objective, x0, method='SLSQP', constraints={'type': 'ineq', 'fun': constraints}, options={'maxiter': 1000})
    return res.x[:3]/scale, res.x[3:].reshape(3, 3)/scale

def calculate_shadow_boundary(center, M, view_vector, res=72):
    w = np.linalg.solve(M, view_vector); w = w / np.linalg.norm(w)
    tmp = np.array([0.0, 1.0, 0.0]) if np.abs(w[1]) < 0.9 else np.array([0.0, 0.0, 1.0])
    s1 = np.cross(w, tmp); s1 /= np.linalg.norm(s1); s2 = np.cross(w, s1)
    theta = np.linspace(0, 2*np.pi, res)
    return center + (np.outer(np.cos(theta), s1) + np.outer(np.sin(theta), s2)) @ M.T

def inflate_to_max_gamut(center, M_init):
    scale = 1.0; step = 0.05; best_M = M_init.copy()
    for _ in range(50):
        test_M = M_init * (scale + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        scale += step; best_M = test_M
    step = 0.005; current_M = best_M
    for _ in range(20):
        test_M = current_M * (1.0 + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        current_M = test_M
    return current_M

def project_locus_onto_ellipsoid_surface(locus_xyz_raw, center, M):
    locus_jz = xyz_to_jzazbz(locus_xyz_raw)
    vecs = locus_jz - center
    M_inv = np.linalg.inv(M)
    u_vecs = vecs @ M_inv.T
    norms = np.linalg.norm(u_vecs, axis=1, keepdims=True)
    u_surface = u_vecs / norms
    return (u_surface @ M.T) + center

def check_gamut_compliance(rgb_loop, name="Loop"):
    min_val = np.min(rgb_loop); max_val = np.max(rgb_loop)
    tol = 2e-3
    if min_val < -tol or max_val > 1.0 + tol:
        print(f"   >>> {name} FAIL: [{min_val:.4f}, {max_val:.4f}]")
    else:
        print(f"   >>> {name} PASS: Inside sRGB")

# ==========================================
# 3. Omnipose Helpers
# ==========================================
def align_ring_red_to_green(ring_lin):
    n = len(ring_lin)
    red_ref = np.array([1.0, 0.0, 0.0])
    idx_r = np.argmin(np.linalg.norm(ring_lin - red_ref, axis=1))
    ring_aligned = np.roll(ring_lin, -idx_r, axis=0)
    green_ref = np.array([0.0, 1.0, 0.0])
    if np.linalg.norm(ring_aligned[(2*n)//3]-green_ref) < np.linalg.norm(ring_aligned[n//3]-green_ref):
        ring_aligned = ring_aligned[::-1]
    return ring_aligned

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    def build_disk(ring, center):
        y, x = np.mgrid[-1:1:256j, -1:1:256j]
        mag = np.sqrt(x*x + y*y)
        angle = np.mod(np.arctan2(y, x), 2*np.pi)
        n = ring.shape[0]
        hue_f = angle/(2*np.pi)*n
        idx0 = np.floor(hue_f).astype(int) % n
        idx1 = (idx0 + 1) % n
        t = hue_f - np.floor(hue_f)
        interp = (1-t[...,None])*ring[idx0] + t[...,None]*ring[idx1]
        rgb = (1-mag[...,None])*center + mag[...,None]*interp
        return np.clip(linear_to_srgb_np(rgb),0,1)

    disk = build_disk(rgb_ring_lin, center_lin)
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi)
    mag = np.clip(np.sqrt(u*u + v*v), 0, 1)
    n = rgb_ring_lin.shape[0]
    hue_f = angle/(2*np.pi)*n
    idx0 = np.floor(hue_f).astype(int) % n
    idx1 = (idx0 + 1) % n
    t = hue_f - np.floor(hue_f)
    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    r = mag[..., None]
    rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow),0,1)
    alpha = mag[...,None]
    white = np.ones_like(flow_rgb); black = np.zeros_like(flow_rgb)
    flow_white = alpha*flow_rgb + (1-alpha)*white
    flow_black = alpha*flow_rgb + (1-alpha)*black
    return disk, flow_rgb, flow_white, flow_black

# ==========================================
# 4. Main Execution
# ==========================================

def main():
    device_str = "cpu"
    dev = torch.device(device_str)
    
    print("1. Calculating Geometries...")
    surf_pts_opt = get_gamut_surface(40) 
    jz_gamut_opt = xyz_to_jzazbz(srgb_to_xyz(surf_pts_opt))
    centroid = np.mean(jz_gamut_opt, axis=0)
    green_vertex = xyz_to_jzazbz(srgb_to_xyz(np.array([[0.0, 1.0, 0.0]])))[0]
    
    # Anchor Strategy
    anchor = centroid + 0.60 * (green_vertex - centroid)
    
    print("2. Fitting Maximal Ellipsoid...")
    center, M_solver = fit_ellipsoid_anchored_solver(jz_gamut_opt, anchor)
    M_max = inflate_to_max_gamut(center, M_solver)
    
    hoop = calculate_shadow_boundary(center, M_max, np.array([1.0, 0.0, 0.0]), res=72)
    rgb_hoop_raw = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
    rgb_hoop_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(rgb_hoop_raw,0,1)))
    
    # Sinebow
    t = np.linspace(0, 1, 72, endpoint=False)
    sr = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0/3))
    sg = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1/3))
    sb = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2/3))
    rgb_sine_std = np.stack([sr, sg, sb], axis=1) 
    rgb_sine_lin = align_ring_red_to_green(srgb_to_linear_np(rgb_sine_std))

    # Loop C: Projected Locus on Ellipsoid
    print("3. Projecting Locus onto Ellipsoid...")
    locus_xyz_raw = get_full_spectral_locus_raw(res=72)
    # Close loop for projection
    locus_xyz_closed = np.vstack([locus_xyz_raw, locus_xyz_raw[0]]) 
    proj_locus_jz = project_locus_onto_ellipsoid_surface(locus_xyz_closed, center, M_max)
    rgb_proj_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(proj_locus_jz)), 0, 1)))

    print("4. Loading Omnipose Flows...")
    omnidir = Path(omnirefactor.__file__).resolve().parent.parent.parent
    basedir = omnidir / "docs" / "_static"
    name = "ecoli"
    ext = ".tif"

    try:
        masks = io.imread(str(basedir / f"{name}_labels{ext}"))
        slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows = omnirefactor.core.masks_to_flows(masks, device=device_str)[4].to(dev)
        center_lin = srgb_to_linear_np(np.array([0.5, 0.5, 0.5]))

        def gen_set(ring, rotation_steps=0):
            r = np.roll(ring, rotation_steps, axis=0)
            return make_flow_images_for_ring(r, center_lin, flows)

        # 2D Rows
        rows = []
        d, f, w, b = gen_set(rgb_sine_lin, 0); d9, f9, w9, b9 = gen_set(rgb_sine_lin, 18)
        rows.append(("Sinebow", rgb_sine_lin, d, b, w, b9, w9))
        
        d, f, w, b = gen_set(rgb_hoop_lin, 0); d9, f9, w9, b9 = gen_set(rgb_hoop_lin, 18)
        rows.append(("Max Ellipse", rgb_hoop_lin, d, b, w, b9, w9))
        
        d, f, w, b = gen_set(rgb_proj_lin, 0); d9, f9, w9, b9 = gen_set(rgb_proj_lin, 18)
        rows.append(("Locus on Ellipse", rgb_proj_lin, d, b, w, b9, w9))

        print("5. Plotting 2D...")
        plt.style.use('dark_background')
        fig, axes = plt.subplots(3, 5, figsize=(20, 12))
        titles = ["Gradient", "Hue Disk", "Black (0°)", "White (0°)", "Black (90°)"]

        for i, (name, ring, disk, b0, w0, b90, w90) in enumerate(rows):
            ax_row = axes[i]
            disp_gradient = linear_to_srgb_np(ring)[None, ...]
            ax_row[0].imshow(disp_gradient, aspect='auto')
            ax_row[0].set_ylabel(name, fontsize=14, color='white', rotation=90, labelpad=20)
            ax_row[1].imshow(disk); ax_row[2].imshow(b0); ax_row[3].imshow(w0); ax_row[4].imshow(b90)
            for j, ax in enumerate(ax_row):
                if i==0: ax.set_title(titles[j], fontsize=12, color='#dddddd')
                ax.set_xticks([]); ax.set_yticks([])
        plt.tight_layout()
        plt.show()
        
        print("6. Generating 3D...")
        fig3d = go.Figure()
        max_jz = np.max(jz_gamut_opt[:,0]); sc = np.array([1.0/max_jz, 1.0, 1.0]); 
        def mc(a): return a[:,1], a[:,2], a[:,0]

        # TRUE MacAdam Solid (Visible Gamut Hull)
        print("   Computing True MacAdam Solid...")
        macadam_xyz = compute_macadam_solid(res=50) # Medium res
        macadam_jz = xyz_to_jzazbz(macadam_xyz)
        hull_macadam = ConvexHull(macadam_jz)
        mx, my, mz = mc(macadam_jz * sc)
        # Visible Gamut (White Shell)         
        fig3d.add_trace(go.Mesh3d(x=mx, y=my, z=mz,
            i=hull_macadam.simplices[:,0], j=hull_macadam.simplices[:,1], k=hull_macadam.simplices[:,2],
            color='white', opacity=0.05, name='Visible Gamut (MacAdam)'))

        # sRGB Faces
        N=10; t=np.linspace(0,1,N); g=np.meshgrid(t,t)
        rgb_faces = [np.stack([g[0],g[1],np.zeros_like(g[0])],-1), np.stack([g[0],g[1],np.ones_like(g[0])],-1),
                     np.stack([g[0],np.zeros_like(g[0]),g[1]],-1), np.stack([g[0],np.ones_like(g[0]),g[1]],-1),
                     np.stack([np.zeros_like(g[0]),g[0],g[1]],-1), np.stack([np.ones_like(g[0]),g[0],g[1]],-1)]
        i_idx, j_idx, k_idx = [], [], []
        for r in range(N-1):
            for c in range(N-1):
                p1=r*N+c; p2=r*N+(c+1); p3=(r+1)*N+c; p4=(r+1)*N+(c+1)
                i_idx.extend([p1,p2]); j_idx.extend([p2,p4]); k_idx.extend([p3,p3])
        for i, face in enumerate(rgb_faces):
            jz_face = xyz_to_jzazbz(srgb_to_xyz(face.reshape(-1,3))) * sc
            x,y,z = mc(jz_face)
            fig3d.add_trace(go.Mesh3d(x=x, y=y, z=z, i=i_idx, j=j_idx, k=k_idx, color='gray', opacity=0.15, flatshading=False, hoverinfo='skip'))

        # sRGB Edges
        corners = [[0,0,0],[1,0,0],[1,1,0],[0,1,0],[0,0,1],[1,0,1],[1,1,1],[0,1,1]]
        edges_idx = [[0,1],[1,2],[2,3],[3,0],[4,5],[5,6],[6,7],[7,4],[0,4],[1,5],[2,6],[3,7]]
        for (s,e) in edges_idx:
            pts = np.linspace(corners[s],corners[e],25)
            line = xyz_to_jzazbz(srgb_to_xyz(pts)) * sc
            x,y,z = mc(line)
            fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', line=dict(color='white', width=2), showlegend=False))

        # Ellipsoid
        u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
        sph = np.stack([np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v)], axis=-1).reshape(-1, 3)
        ell = (sph @ M_max.T + center) * sc
        ex, ey, ez = mc(ell)
        fig3d.add_trace(go.Surface(x=ex.reshape(u.shape), y=ey.reshape(u.shape), z=ez.reshape(u.shape),
                                colorscale=[(0,'#88aa88'),(1,'#88aa88')], opacity=0.3, showscale=False, name='Ellipsoid'))

        # Loops
        hx, hy, hz = mc(hoop * sc)
        c_hoop = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in linear_to_srgb_np(np.clip(rgb_hoop_raw,0,1))]
        fig3d.add_trace(go.Scatter3d(x=hx, y=hy, z=hz, mode='markers', marker=dict(color=c_hoop, size=5), name='Ellipsoid Loop'))

        sx, sy, sz = mc(xyz_to_jzazbz(srgb_to_xyz(rgb_sine_std)) * sc)
        c_sine = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in rgb_sine_std]
        fig3d.add_trace(go.Scatter3d(x=sx, y=sy, z=sz, mode='markers', marker=dict(color=c_sine, size=4, symbol='diamond'), name='Sinebow Loop'))

        # Project Locus
        px, py, pz = mc(proj_locus_jz * sc)
        fig3d.add_trace(go.Scatter3d(x=px, y=py, z=pz, mode='lines', line=dict(color='cyan', width=5), name='Locus on Ellipsoid'))

        # Full Locus Reference (Solid White)
        ref_xyz = get_full_spectral_locus_raw(res=100)
        ref_jz = xyz_to_jzazbz(ref_xyz) * sc
        rx, ry, rz = mc(ref_jz)
        fig3d.add_trace(go.Scatter3d(x=rx, y=ry, z=rz, mode='lines', line=dict(color='white', width=3), name='Full Locus (Ref)'))

        fig3d.update_layout(template="plotly_dark", title="3D Jzazbz: MacAdam Solid & sRGB", 
                            scene=dict(xaxis_title='az', yaxis_title='bz', zaxis_title='Jz', aspectmode='manual', aspectratio=dict(x=1,y=1,z=1)))
        fig3d.show()

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import scipy.optimize as opt
from scipy.spatial import ConvexHull
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import omnirefactor.core
from skimage import io
import fastremap

# ==========================================
# 1. Color Math
# ==========================================
def srgb_to_xyz(rgb):
    mask = rgb > 0.04045
    linear = np.zeros_like(rgb)
    linear[mask] = ((rgb[mask] + 0.055) / 1.055) ** 2.4
    linear[~mask] = rgb[~mask] / 12.92
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    return linear @ M.T

def xyz_to_jzazbz(xyz):
    b, g, d, d0 = 1.15, 0.66, -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    lms = xyz @ M1.T
    lms_norm = (lms * 200) / 10000.0 
    lms_prime = np.sign(lms_norm) * (((c1 + c2 * (np.abs(lms_norm) ** n)) / (1 + c3 * (np.abs(lms_norm) ** n))) ** p)
    izazbz = lms_prime @ M2.T
    Jz = ((1 + d) * izazbz[:, 0]) / (1 + d * izazbz[:, 0]) - d0
    return np.column_stack((Jz, izazbz[:, 1], izazbz[:, 2]))

def jzazbz_to_xyz(jzazbz):
    Jz, az, bz = jzazbz[:,0], jzazbz[:,1], jzazbz[:,2]
    d, d0 = -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    Jp = Jz + d0; Iz = Jp / (1 + d - d * Jp)
    izazbz_vec = np.stack([Iz, az, bz], axis=1)
    lms_prime = izazbz_vec @ np.linalg.inv(M2).T
    sign_lms = np.sign(lms_prime)
    Y = np.abs(lms_prime) ** (1/p)
    num = Y - c1; den = c2 - Y * c3
    An = np.clip(num / den, 0, None) 
    lms_norm = sign_lms * (An ** (1/n))
    lms = (lms_norm * 10000.0) / 200.0
    return lms @ np.linalg.inv(M1).T

def xyz_to_srgb_raw(xyz):
    M_inv = np.linalg.inv(np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]]))
    linear = xyz @ M_inv.T
    srgb = np.zeros_like(linear)
    mask = linear > 0.0031308
    srgb[mask] = 1.055 * (linear[mask] ** (1/2.4)) - 0.055
    srgb[~mask] = 12.92 * linear[~mask]
    return srgb

def xyz_to_srgb(xyz): return np.clip(xyz_to_srgb_raw(xyz), 0, 1)
def srgb_to_linear_np(srgb): return np.where(srgb <= 0.04045, srgb / 12.92, ((srgb + 0.055) / 1.055) ** 2.4)
def linear_to_srgb_np(lin): 
    s = np.zeros_like(lin)
    mask = lin > 0.0031308
    s[mask] = 1.055 * (lin[mask]**(1/2.4)) - 0.055
    s[~mask] = 12.92 * lin[~mask]
    return np.clip(s, 0, 1)

# ==========================================
# 2. MacAdam Limit (Visible Gamut)
# ==========================================

def get_cmf_1nm():
    # Standard CIE 1931 2-deg (380-780nm)
    base_cmf = np.array([
        [0.0014,0.0000,0.0065], [0.0042,0.0001,0.0201], [0.0143,0.0004,0.0679], [0.0435,0.0012,0.2074],
        [0.1344,0.0040,0.6456], [0.2839,0.0116,1.3856], [0.3483,0.0230,1.7471], [0.3362,0.0380,1.7721],
        [0.2908,0.0600,1.6692], [0.1954,0.0910,1.2876], [0.0956,0.1390,0.8130], [0.0320,0.2080,0.4652],
        [0.0049,0.3230,0.2720], [0.0093,0.5030,0.1582], [0.0633,0.7100,0.0782], [0.1655,0.8620,0.0422],
        [0.2904,0.9540,0.0203], [0.4334,0.9950,0.0087], [0.5945,0.9950,0.0039], [0.7621,0.9520,0.0021],
        [0.9163,0.8700,0.0017], [1.0263,0.7570,0.0011], [1.0622,0.6310,0.0008], [1.0026,0.5030,0.0006],
        [0.8544,0.3810,0.0002], [0.6424,0.2650,0.0000], [0.4479,0.1750,0.0000], [0.2835,0.1070,0.0000],
        [0.1649,0.0610,0.0000], [0.0874,0.0320,0.0000], [0.0468,0.0170,0.0000], [0.0227,0.0082,0.0000],
        [0.0114,0.0041,0.0000], [0.0058,0.0021,0.0000], [0.0029,0.0010,0.0000], [0.0014,0.0005,0.0000]
    ])
    # Interpolate to 1nm steps for integration
    x_base = np.linspace(380, 780, len(base_cmf))
    x_new = np.arange(380, 781, 1) # 1nm steps
    cmf_interp = np.zeros((len(x_new), 3))
    for k in range(3):
        cmf_interp[:,k] = np.interp(x_new, x_base, base_cmf[:,k])
    return cmf_interp

def compute_macadam_solid(res=60):
    """
    Computes the Optimal Color Solid (MacAdam Limit) by integrating 
    ideal block spectra (0/1 transitions).
    """
    cmf = get_cmf_1nm()
    n_lambda = len(cmf)
    
    # Cumulative sum for fast integration of bands
    cum_cmf = np.vstack([np.zeros(3), np.cumsum(cmf, axis=0)])
    white_xyz = cum_cmf[-1] # Sum of all wavelengths (Ideal White)
    
    optimal_colors = []
    
    # We sample start/end wavelengths
    indices = np.linspace(0, n_lambda, res).astype(int)
    
    for i in indices:
        for j in indices:
            if i == j: continue
            
            # Band-Pass (Type 1): Light ON between i and j
            if j > i:
                color = cum_cmf[j] - cum_cmf[i]
            # Band-Stop (Type 2): Light OFF between i and j (Purple line logic)
            else:
                color = white_xyz - (cum_cmf[i] - cum_cmf[j])
                
            optimal_colors.append(color)
            
    # Add extremes
    optimal_colors.append([0,0,0])
    optimal_colors.append(white_xyz)
    
    # Normalize so Y=1.0 (match sRGB White luminance scale approx)
    opt_xyz = np.array(optimal_colors)
    opt_xyz = opt_xyz / white_xyz[1] 
    
    return opt_xyz

# ==========================================
# 3. Optimization
# ==========================================

def get_gamut_surface(res=40):
    t = np.linspace(0, 1, res)
    g = np.meshgrid(t, t)
    faces = [np.stack([g[0], g[1], np.zeros_like(g[0])], -1), np.stack([g[0], g[1], np.ones_like(g[0])], -1),
             np.stack([g[0], np.zeros_like(g[0]), g[1]], -1), np.stack([g[0], np.ones_like(g[0]), g[1]], -1),
             np.stack([np.zeros_like(g[0]), g[0], g[1]], -1), np.stack([np.ones_like(g[0]), g[0], g[1]], -1)]
    pts = np.vstack([f.reshape(-1, 3) for f in faces])
    corners = np.array([[0,0,0], [1,0,0], [0,1,0], [0,0,1], [1,1,0], [1,0,1], [0,1,1], [1,1,1]])
    return np.vstack([pts, corners])

def fit_ellipsoid_anchored_solver(points, anchor_point):
    hull = ConvexHull(points)
    A = hull.equations[:, :3]; b_const = -hull.equations[:, 3]
    scale = 100.0; A_scaled = A / scale; anchor_scaled = anchor_point * scale
    c0 = np.mean(points, axis=0) * scale
    c0 = 0.5 * c0 + 0.5 * anchor_scaled 
    x0 = np.concatenate([c0, np.eye(3).flatten() * 0.05])

    def objective(x):
        M = x[3:].reshape(3, 3); sign, val = np.linalg.slogdet(M)
        return -val if sign > 0 else 1e9

    def constraints(x):
        d, M = x[:3], x[3:].reshape(3, 3)
        norm_AM = np.linalg.norm(A_scaled @ M, axis=1)
        hull_con = b_const - (A_scaled @ d + norm_AM)
        try: u = np.linalg.solve(M, anchor_scaled - d); anchor_con = 1.0 - np.linalg.norm(u)
        except: anchor_con = -1.0
        return np.concatenate([hull_con, [anchor_con]])

    res = opt.minimize(objective, x0, method='SLSQP', constraints={'type': 'ineq', 'fun': constraints}, options={'maxiter': 1000})
    return res.x[:3]/scale, res.x[3:].reshape(3, 3)/scale

def calculate_shadow_boundary(center, M, view_vector, res=72):
    w = np.linalg.solve(M, view_vector); w = w / np.linalg.norm(w)
    tmp = np.array([0.0, 1.0, 0.0]) if np.abs(w[1]) < 0.9 else np.array([0.0, 0.0, 1.0])
    s1 = np.cross(w, tmp); s1 /= np.linalg.norm(s1); s2 = np.cross(w, s1)
    theta = np.linspace(0, 2*np.pi, res)
    return center + (np.outer(np.cos(theta), s1) + np.outer(np.sin(theta), s2)) @ M.T

def inflate_to_max_gamut(center, M_init):
    scale = 1.0; step = 0.05; best_M = M_init.copy()
    for _ in range(50):
        test_M = M_init * (scale + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        scale += step; best_M = test_M
    step = 0.005; current_M = best_M
    for _ in range(20):
        test_M = current_M * (1.0 + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        current_M = test_M
    return current_M

def project_locus_onto_ellipsoid_surface(locus_xyz_raw, center, M):
    locus_jz = xyz_to_jzazbz(locus_xyz_raw)
    vecs = locus_jz - center
    M_inv = np.linalg.inv(M)
    u_vecs = vecs @ M_inv.T
    norms = np.linalg.norm(u_vecs, axis=1, keepdims=True)
    u_surface = u_vecs / norms
    return (u_surface @ M.T) + center

# ==========================================
# 4. Omnipose Helpers
# ==========================================
def align_ring_red_to_green(ring_lin):
    n = len(ring_lin)
    red_ref = np.array([1.0, 0.0, 0.0])
    idx_r = np.argmin(np.linalg.norm(ring_lin - red_ref, axis=1))
    ring_aligned = np.roll(ring_lin, -idx_r, axis=0)
    green_ref = np.array([0.0, 1.0, 0.0])
    if np.linalg.norm(ring_aligned[(2*n)//3]-green_ref) < np.linalg.norm(ring_aligned[n//3]-green_ref):
        ring_aligned = ring_aligned[::-1]
    return ring_aligned

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    def build_disk(ring, center):
        y, x = np.mgrid[-1:1:256j, -1:1:256j]
        mag = np.sqrt(x*x + y*y)
        angle = np.mod(np.arctan2(y, x), 2*np.pi)
        n = ring.shape[0]
        hue_f = angle/(2*np.pi)*n
        idx0 = np.floor(hue_f).astype(int) % n
        idx1 = (idx0 + 1) % n
        t = hue_f - np.floor(hue_f)
        interp = (1-t[...,None])*ring[idx0] + t[...,None]*ring[idx1]
        rgb = (1-mag[...,None])*center + mag[...,None]*interp
        return np.clip(linear_to_srgb_np(rgb),0,1)

    disk = build_disk(rgb_ring_lin, center_lin)
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi)
    mag = np.clip(np.sqrt(u*u + v*v), 0, 1)
    n = rgb_ring_lin.shape[0]
    hue_f = angle/(2*np.pi)*n
    idx0 = np.floor(hue_f).astype(int) % n
    idx1 = (idx0 + 1) % n
    t = hue_f - np.floor(hue_f)
    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    r = mag[..., None]
    rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow),0,1)
    alpha = mag[...,None]
    white = np.ones_like(flow_rgb); black = np.zeros_like(flow_rgb)
    flow_white = alpha*flow_rgb + (1-alpha)*white
    flow_black = alpha*flow_rgb + (1-alpha)*black
    return disk, flow_rgb, flow_white, flow_black

# ==========================================
# 5. Main Execution
# ==========================================

def main():
    device_str = "cpu"
    dev = torch.device(device_str)
    
    print("1. Calculating Geometries...")
    surf_pts_opt = get_gamut_surface(40) 
    jz_gamut_opt = xyz_to_jzazbz(srgb_to_xyz(surf_pts_opt))
    centroid = np.mean(jz_gamut_opt, axis=0)
    green_vertex = xyz_to_jzazbz(srgb_to_xyz(np.array([[0.0, 1.0, 0.0]])))[0]
    
    # Anchor
    anchor = centroid + 0.60 * (green_vertex - centroid)
    
    print("2. Fitting Maximal Ellipsoid...")
    center, M_solver = fit_ellipsoid_anchored_solver(jz_gamut_opt, anchor)
    M_max = inflate_to_max_gamut(center, M_solver)
    
    # Loop A: Ellipsoid Loop
    hoop = calculate_shadow_boundary(center, M_max, np.array([1.0, 0.0, 0.0]), res=72)
    rgb_hoop_raw = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
    rgb_hoop_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(rgb_hoop_raw,0,1)))
    
    # Loop B: Sinebow
    t = np.linspace(0, 1, 72, endpoint=False)
    sr = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0/3))
    sg = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1/3))
    sb = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2/3))
    rgb_sine_std = np.stack([sr, sg, sb], axis=1) 
    rgb_sine_lin = align_ring_red_to_green(srgb_to_linear_np(rgb_sine_std))

    # Loop C: Projected Locus on Ellipsoid (From Raw)
    print("3. Projecting Locus onto Ellipsoid...")
    locus_xyz_raw = get_cmf_1nm() # Get the pure spectral data (unclosed)
    # Close loop manually for projection
    locus_closed = np.vstack([locus_xyz_raw, locus_xyz_raw[0]])
    proj_locus_jz = project_locus_onto_ellipsoid_surface(locus_closed, center, M_max)
    rgb_proj_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(proj_locus_jz)), 0, 1)))

    print("4. Loading Omnipose Flows...")
    omnidir = Path(omnirefactor.__file__).resolve().parent.parent.parent
    basedir = omnidir / "docs" / "_static"
    name = "ecoli"
    ext = ".tif"

    try:
        masks = io.imread(str(basedir / f"{name}_labels{ext}"))
        slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows = omnirefactor.core.masks_to_flows(masks, device=device_str)[4].to(dev)
        center_lin = srgb_to_linear_np(np.array([0.5, 0.5, 0.5]))

        def gen_set(ring, rotation_steps=0):
            r = np.roll(ring, rotation_steps, axis=0)
            return make_flow_images_for_ring(r, center_lin, flows)

        # 2D Rows
        rows = []
        d, f, w, b = gen_set(rgb_sine_lin, 0); d9, f9, w9, b9 = gen_set(rgb_sine_lin, 18)
        rows.append(("Sinebow", rgb_sine_lin, d, b, w, b9, w9))
        
        d, f, w, b = gen_set(rgb_hoop_lin, 0); d9, f9, w9, b9 = gen_set(rgb_hoop_lin, 18)
        rows.append(("Max Ellipse", rgb_hoop_lin, d, b, w, b9, w9))
        
        d, f, w, b = gen_set(rgb_proj_lin, 0); d9, f9, w9, b9 = gen_set(rgb_proj_lin, 18)
        rows.append(("Locus on Ellipse", rgb_proj_lin, d, b, w, b9, w9))

        print("5. Plotting 2D...")
        plt.style.use('dark_background')
        fig, axes = plt.subplots(3, 5, figsize=(20, 12))
        titles = ["Gradient", "Hue Disk", "Black (0°)", "White (0°)", "Black (90°)"]

        for i, (name, ring, disk, b0, w0, b90, w90) in enumerate(rows):
            ax_row = axes[i]
            disp_gradient = linear_to_srgb_np(ring)[None, ...]
            ax_row[0].imshow(disp_gradient, aspect='auto')
            ax_row[0].set_ylabel(name, fontsize=14, color='white', rotation=90, labelpad=20)
            ax_row[1].imshow(disk); ax_row[2].imshow(b0); ax_row[3].imshow(w0); ax_row[4].imshow(b90)
            for j, ax in enumerate(ax_row):
                if i==0: ax.set_title(titles[j], fontsize=12, color='#dddddd')
                ax.set_xticks([]); ax.set_yticks([])
        plt.tight_layout()
        plt.show()
        
        print("6. Generating 3D...")
        fig3d = go.Figure()
        max_jz = np.max(jz_gamut_opt[:,0]); sc = np.array([1.0/max_jz, 1.0, 1.0]); 
        def mc(a): return a[:,1], a[:,2], a[:,0]

        # A. MacAdam Solid (Visible Gamut)
        print("   Computing MacAdam Solid...")
        macadam_xyz = compute_macadam_solid(res=50)
        macadam_jz = xyz_to_jzazbz(macadam_xyz)
        hull_macadam = ConvexHull(macadam_jz)
        mx, my, mz = mc(macadam_jz * sc)
        fig3d.add_trace(go.Mesh3d(x=mx, y=my, z=mz,
            i=hull_macadam.simplices[:,0], j=hull_macadam.simplices[:,1], k=hull_macadam.simplices[:,2],
            color='white', opacity=0.05, name='Visible Gamut (MacAdam)'))

        # B. sRGB Edges
        corners = [[0,0,0],[1,0,0],[1,1,0],[0,1,0],[0,0,1],[1,0,1],[1,1,1],[0,1,1]]
        edges_idx = [[0,1],[1,2],[2,3],[3,0],[4,5],[5,6],[6,7],[7,4],[0,4],[1,5],[2,6],[3,7]]
        for (s,e) in edges_idx:
            pts = np.linspace(corners[s],corners[e],25)
            line = xyz_to_jzazbz(srgb_to_xyz(pts)) * sc
            x,y,z = mc(line)
            fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', line=dict(color='gray', width=2), showlegend=False))
        fig3d.add_trace(go.Scatter3d(x=[None], y=[None], z=[None], mode='lines', line=dict(color='gray', width=2), name='sRGB Edges'))

        # C. Ellipsoid
        u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
        sph = np.stack([np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v)], axis=-1).reshape(-1, 3)
        ell = (sph @ M_max.T + center) * sc
        ex, ey, ez = mc(ell)
        fig3d.add_trace(go.Surface(x=ex.reshape(u.shape), y=ey.reshape(u.shape), z=ez.reshape(u.shape),
                                colorscale=[(0,'#88aa88'),(1,'#88aa88')], opacity=0.3, showscale=False, name='Ellipsoid'))

        # D. Loops
        hx, hy, hz = mc(hoop * sc)
        c_hoop = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in linear_to_srgb_np(np.clip(rgb_hoop_raw,0,1))]
        fig3d.add_trace(go.Scatter3d(x=hx, y=hy, z=hz, mode='markers', marker=dict(color=c_hoop, size=5), name='Ellipsoid Loop'))

        sx, sy, sz = mc(xyz_to_jzazbz(srgb_to_xyz(rgb_sine_std)) * sc)
        c_sine = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in rgb_sine_std]
        fig3d.add_trace(go.Scatter3d(x=sx, y=sy, z=sz, mode='markers', marker=dict(color=c_sine, size=4, symbol='diamond'), name='Sinebow Loop'))

        # E. Project Locus (Cyan)
        px, py, pz = mc(proj_locus_jz * sc)
        fig3d.add_trace(go.Scatter3d(x=px, y=py, z=pz, mode='lines', line=dict(color='cyan', width=6), name='Locus on Ellipsoid'))

        # F. Full Locus Ref (White)
        ref_xyz = get_cmf_1nm()
        ref_jz = xyz_to_jzazbz(ref_xyz) * sc
        rx, ry, rz = mc(ref_jz)
        fig3d.add_trace(go.Scatter3d(x=rx, y=ry, z=rz, mode='lines', line=dict(color='white', width=4), name='Full Locus (Ref)'))

        fig3d.update_layout(template="plotly_dark", title="3D Jzazbz: MacAdam Solid & sRGB", 
                            scene=dict(xaxis_title='az', yaxis_title='bz', zaxis_title='Jz', aspectmode='manual', aspectratio=dict(x=1,y=1,z=1)))
        fig3d.show()

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import scipy.optimize as opt
from scipy.spatial import ConvexHull
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import omnirefactor.core
from skimage import io
import fastremap

# ==========================================
# 1. Color Math
# ==========================================
def srgb_to_xyz(rgb):
    mask = rgb > 0.04045
    linear = np.zeros_like(rgb)
    linear[mask] = ((rgb[mask] + 0.055) / 1.055) ** 2.4
    linear[~mask] = rgb[~mask] / 12.92
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    return linear @ M.T

def xyz_to_jzazbz(xyz):
    b, g, d, d0 = 1.15, 0.66, -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    lms = xyz @ M1.T
    lms_norm = (lms * 200) / 10000.0 
    lms_prime = np.sign(lms_norm) * (((c1 + c2 * (np.abs(lms_norm) ** n)) / (1 + c3 * (np.abs(lms_norm) ** n))) ** p)
    izazbz = lms_prime @ M2.T
    Jz = ((1 + d) * izazbz[:, 0]) / (1 + d * izazbz[:, 0]) - d0
    return np.column_stack((Jz, izazbz[:, 1], izazbz[:, 2]))

def jzazbz_to_xyz(jzazbz):
    Jz, az, bz = jzazbz[:,0], jzazbz[:,1], jzazbz[:,2]
    d, d0 = -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    Jp = Jz + d0; Iz = Jp / (1 + d - d * Jp)
    izazbz_vec = np.stack([Iz, az, bz], axis=1)
    lms_prime = izazbz_vec @ np.linalg.inv(M2).T
    sign_lms = np.sign(lms_prime)
    Y = np.abs(lms_prime) ** (1/p)
    num = Y - c1; den = c2 - Y * c3
    An = np.clip(num / den, 0, None) 
    lms_norm = sign_lms * (An ** (1/n))
    lms = (lms_norm * 10000.0) / 200.0
    return lms @ np.linalg.inv(M1).T

def xyz_to_srgb_raw(xyz):
    M_inv = np.linalg.inv(np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]]))
    linear = xyz @ M_inv.T
    srgb = np.zeros_like(linear)
    mask = linear > 0.0031308
    srgb[mask] = 1.055 * (linear[mask] ** (1/2.4)) - 0.055
    srgb[~mask] = 12.92 * linear[~mask]
    return srgb

def xyz_to_srgb(xyz): return np.clip(xyz_to_srgb_raw(xyz), 0, 1)
def srgb_to_linear_np(srgb): return np.where(srgb <= 0.04045, srgb / 12.92, ((srgb + 0.055) / 1.055) ** 2.4)
def linear_to_srgb_np(lin):
    s = np.zeros_like(lin)
    mask = lin > 0.0031308
    s[mask] = 1.055 * (lin[mask]**(1/2.4)) - 0.055
    s[~mask] = 12.92 * lin[~mask]
    return np.clip(s, 0, 1)

# ==========================================
# 2. MacAdam Solid & Ridge Logic
# ==========================================

def get_cmf_high_res(res=360):
    # Base CIE 1931 2-deg
    base_cmf = np.array([
        [0.0014,0.0000,0.0065], [0.0042,0.0001,0.0201], [0.0143,0.0004,0.0679], [0.0435,0.0012,0.2074],
        [0.1344,0.0040,0.6456], [0.2839,0.0116,1.3856], [0.3483,0.0230,1.7471], [0.3362,0.0380,1.7721],
        [0.2908,0.0600,1.6692], [0.1954,0.0910,1.2876], [0.0956,0.1390,0.8130], [0.0320,0.2080,0.4652],
        [0.0049,0.3230,0.2720], [0.0093,0.5030,0.1582], [0.0633,0.7100,0.0782], [0.1655,0.8620,0.0422],
        [0.2904,0.9540,0.0203], [0.4334,0.9950,0.0087], [0.5945,0.9950,0.0039], [0.7621,0.9520,0.0021],
        [0.9163,0.8700,0.0017], [1.0263,0.7570,0.0011], [1.0622,0.6310,0.0008], [1.0026,0.5030,0.0006],
        [0.8544,0.3810,0.0002], [0.6424,0.2650,0.0000], [0.4479,0.1750,0.0000], [0.2835,0.1070,0.0000],
        [0.1649,0.0610,0.0000], [0.0874,0.0320,0.0000], [0.0468,0.0170,0.0000], [0.0227,0.0082,0.0000],
        [0.0114,0.0041,0.0000], [0.0058,0.0021,0.0000], [0.0029,0.0010,0.0000], [0.0014,0.0005,0.0000]
    ])
    x = np.linspace(0, 1, len(base_cmf))
    xi = np.linspace(0, 1, res)
    interp_cmf = np.zeros((res, 3))
    for k in range(3):
        interp_cmf[:,k] = np.interp(xi, x, base_cmf[:,k])
    return interp_cmf

def get_macadam_data(res=100):
    """
    Returns:
    1. solid_xyz: Cloud of Optimal Colors (Volume)
    2. ridge_xyz: The 'Edge' of the solid (Max saturation for surfaces)
    3. white_xyz: The peak white reference
    """
    cmf = get_cmf_high_res(360) # Use high res for integration
    n_lambda = len(cmf)
    
    cum_cmf = np.vstack([np.zeros(3), np.cumsum(cmf, axis=0)])
    white_xyz = cum_cmf[-1]
    
    # Subsample indices for output resolution
    indices = np.linspace(0, n_lambda, res).astype(int)
    
    solid_colors = []
    ridge_colors = [] # We will build this by taking optimal colors with specific bandwidths
    
    # 1. Generate Volume (Variable Bandwidth)
    for i in indices:
        for j in indices:
            if i == j: continue
            if j > i: xyz = cum_cmf[j] - cum_cmf[i] # Type 1
            else:     xyz = cum_cmf[j] + (white_xyz - cum_cmf[i]) # Type 2
            solid_colors.append(xyz)

    # 2. Generate Ridge (Fixed Minimal Bandwidth -> Surface Locus)
    # We approximate the surface locus by taking small bands around the spectrum
    # This corresponds to the sharpest possible transition physically realizable
    step = 10 # Bandwidth index width
    for i in range(n_lambda):
        # Band Pass Ridge
        start = i
        end = min(i + step, n_lambda)
        xyz = cum_cmf[end] - cum_cmf[start]
        ridge_colors.append(xyz)
        
        # Band Stop Ridge (Purples)
        xyz_stop = white_xyz - xyz
        ridge_colors.append(xyz_stop)
    
    # Add black/white to solid
    solid_colors.append([0,0,0])
    solid_colors.append(white_xyz)
    
    # Normalize everything to White Y=1.0
    scale = 1.0 / white_xyz[1]
    
    return np.array(solid_colors)*scale, np.array(ridge_colors)*scale, white_xyz*scale

# ==========================================
# 3. Optimization & Projection
# ==========================================

def get_gamut_surface(res=40):
    t = np.linspace(0, 1, res)
    g = np.meshgrid(t, t)
    faces = [np.stack([g[0], g[1], np.zeros_like(g[0])], -1), np.stack([g[0], g[1], np.ones_like(g[0])], -1),
             np.stack([g[0], np.zeros_like(g[0]), g[1]], -1), np.stack([g[0], np.ones_like(g[0]), g[1]], -1),
             np.stack([np.zeros_like(g[0]), g[0], g[1]], -1), np.stack([np.ones_like(g[0]), g[0], g[1]], -1)]
    pts = np.vstack([f.reshape(-1, 3) for f in faces])
    corners = np.array([[0,0,0], [1,0,0], [0,1,0], [0,0,1], [1,1,0], [1,0,1], [0,1,1], [1,1,1]])
    return np.vstack([pts, corners])

def fit_ellipsoid_anchored_solver(points, anchor_point):
    hull = ConvexHull(points)
    A = hull.equations[:, :3]; b_const = -hull.equations[:, 3]
    scale = 100.0; A_scaled = A / scale; anchor_scaled = anchor_point * scale
    c0 = np.mean(points, axis=0) * scale
    c0 = 0.5 * c0 + 0.5 * anchor_scaled 
    x0 = np.concatenate([c0, np.eye(3).flatten() * 0.05])

    def objective(x):
        M = x[3:].reshape(3, 3); sign, val = np.linalg.slogdet(M)
        return -val if sign > 0 else 1e9

    def constraints(x):
        d, M = x[:3], x[3:].reshape(3, 3)
        norm_AM = np.linalg.norm(A_scaled @ M, axis=1)
        hull_con = b_const - (A_scaled @ d + norm_AM)
        try: u = np.linalg.solve(M, anchor_scaled - d); anchor_con = 1.0 - np.linalg.norm(u)
        except: anchor_con = -1.0
        return np.concatenate([hull_con, [anchor_con]])

    res = opt.minimize(objective, x0, method='SLSQP', constraints={'type': 'ineq', 'fun': constraints}, options={'maxiter': 1000})
    return res.x[:3]/scale, res.x[3:].reshape(3, 3)/scale

def calculate_shadow_boundary(center, M, view_vector, res=72):
    w = np.linalg.solve(M, view_vector); w = w / np.linalg.norm(w)
    tmp = np.array([0.0, 1.0, 0.0]) if np.abs(w[1]) < 0.9 else np.array([0.0, 0.0, 1.0])
    s1 = np.cross(w, tmp); s1 /= np.linalg.norm(s1); s2 = np.cross(w, s1)
    theta = np.linspace(0, 2*np.pi, res)
    return center + (np.outer(np.cos(theta), s1) + np.outer(np.sin(theta), s2)) @ M.T

def inflate_to_max_gamut(center, M_init):
    scale = 1.0; step = 0.05; best_M = M_init.copy()
    for _ in range(50):
        test_M = M_init * (scale + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        scale += step; best_M = test_M
    step = 0.005; current_M = best_M
    for _ in range(20):
        test_M = current_M * (1.0 + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        current_M = test_M
    return current_M

def project_locus_onto_ellipsoid_surface(locus_xyz_raw, center, M):
    locus_jz = xyz_to_jzazbz(locus_xyz_raw)
    vecs = locus_jz - center
    M_inv = np.linalg.inv(M)
    u_vecs = vecs @ M_inv.T
    norms = np.linalg.norm(u_vecs, axis=1, keepdims=True)
    u_surface = u_vecs / norms
    return (u_surface @ M.T) + center

# ==========================================
# 4. Omnipose Helpers
# ==========================================
def align_ring_red_to_green(ring_lin):
    n = len(ring_lin)
    red_ref = np.array([1.0, 0.0, 0.0])
    idx_r = np.argmin(np.linalg.norm(ring_lin - red_ref, axis=1))
    ring_aligned = np.roll(ring_lin, -idx_r, axis=0)
    green_ref = np.array([0.0, 1.0, 0.0])
    if np.linalg.norm(ring_aligned[(2*n)//3]-green_ref) < np.linalg.norm(ring_aligned[n//3]-green_ref):
        ring_aligned = ring_aligned[::-1]
    return ring_aligned

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    def build_disk(ring, center):
        y, x = np.mgrid[-1:1:256j, -1:1:256j]
        mag = np.sqrt(x*x + y*y)
        angle = np.mod(np.arctan2(y, x), 2*np.pi)
        n = ring.shape[0]
        hue_f = angle/(2*np.pi)*n
        idx0 = np.floor(hue_f).astype(int) % n
        idx1 = (idx0 + 1) % n
        t = hue_f - np.floor(hue_f)
        interp = (1-t[...,None])*ring[idx0] + t[...,None]*ring[idx1]
        rgb = (1-mag[...,None])*center + mag[...,None]*interp
        return np.clip(linear_to_srgb_np(rgb),0,1)

    disk = build_disk(rgb_ring_lin, center_lin)
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi)
    mag = np.clip(np.sqrt(u*u + v*v), 0, 1)
    n = rgb_ring_lin.shape[0]
    hue_f = angle/(2*np.pi)*n
    idx0 = np.floor(hue_f).astype(int) % n
    idx1 = (idx0 + 1) % n
    t = hue_f - np.floor(hue_f)
    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    r = mag[..., None]
    rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow),0,1)
    alpha = mag[...,None]
    white = np.ones_like(flow_rgb); black = np.zeros_like(flow_rgb)
    flow_white = alpha*flow_rgb + (1-alpha)*white
    flow_black = alpha*flow_rgb + (1-alpha)*black
    return disk, flow_rgb, flow_white, flow_black

# ==========================================
# 5. Main Execution
# ==========================================

def main():
    device_str = "cpu"
    dev = torch.device(device_str)
    
    print("1. Calculating Geometries...")
    surf_pts_opt = get_gamut_surface(40) 
    jz_gamut_opt = xyz_to_jzazbz(srgb_to_xyz(surf_pts_opt))
    centroid = np.mean(jz_gamut_opt, axis=0)
    green_vertex = xyz_to_jzazbz(srgb_to_xyz(np.array([[0.0, 1.0, 0.0]])))[0]
    anchor = centroid + 0.60 * (green_vertex - centroid)
    
    print("2. Fitting Maximal Ellipsoid...")
    center, M_solver = fit_ellipsoid_anchored_solver(jz_gamut_opt, anchor)
    M_max = inflate_to_max_gamut(center, M_solver)
    
    hoop = calculate_shadow_boundary(center, M_max, np.array([1.0, 0.0, 0.0]), res=72)
    rgb_hoop_raw = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
    rgb_hoop_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(rgb_hoop_raw,0,1)))
    
    # Sinebow
    t = np.linspace(0, 1, 72, endpoint=False)
    sr = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0/3))
    sg = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1/3))
    sb = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2/3))
    rgb_sine_std = np.stack([sr, sg, sb], axis=1) 
    rgb_sine_lin = align_ring_red_to_green(srgb_to_linear_np(rgb_sine_std))

    # 3. MacAdam Solid Data
    print("3. Computing MacAdam Data...")
    macadam_xyz, macadam_ridge_xyz, white_xyz = get_macadam_data(res=40)
    
    # PROJECT THE LIFTED RIDGE onto the Ellipsoid
    # This is the "Lifted" version of the locus (the ridge of the solid)
    proj_locus_jz = project_locus_onto_ellipsoid_surface(macadam_ridge_xyz, center, M_max)
    rgb_proj_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(proj_locus_jz)), 0, 1)))

    print("4. Loading Omnipose Flows...")
    omnidir = Path(omnirefactor.__file__).resolve().parent.parent.parent
    basedir = omnidir / "docs" / "_static"
    name = "ecoli"
    ext = ".tif"

    try:
        masks = io.imread(str(basedir / f"{name}_labels{ext}"))
        slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows = omnirefactor.core.masks_to_flows(masks, device=device_str)[4].to(dev)
        center_lin = srgb_to_linear_np(np.array([0.5, 0.5, 0.5]))

        def gen_set(ring, rotation_steps=0):
            r = np.roll(ring, rotation_steps, axis=0)
            return make_flow_images_for_ring(r, center_lin, flows)

        d_s, _, w_s, b_s = gen_set(rgb_sine_lin, 0)
        d_e, _, w_e, b_e = gen_set(rgb_hoop_lin, 0)
        d_l, _, w_l, b_l = gen_set(rgb_proj_lin, 0)

        print("5. Plotting 2D...")
        plt.style.use('dark_background')
        fig, axes = plt.subplots(3, 5, figsize=(20, 12))
        rows = [("Sinebow", rgb_sine_lin, d_s, b_s, w_s),
                ("Max Ellipse", rgb_hoop_lin, d_e, b_e, w_e),
                ("Locus on Ellipse", rgb_proj_lin, d_l, b_l, w_l)]

        for i, (name, ring, disk, b, w) in enumerate(rows):
            ax_row = axes[i]
            disp_gradient = linear_to_srgb_np(ring)[None, ...]
            ax_row[0].imshow(disp_gradient, aspect='auto')
            ax_row[0].set_ylabel(name, fontsize=14, color='white', rotation=90, labelpad=20)
            ax_row[1].imshow(disk); ax_row[2].imshow(b); ax_row[3].imshow(w)
            for j, ax in enumerate(ax_row):
                ax.set_xticks([]); ax.set_yticks([])
        plt.tight_layout()
        plt.show()
        
        print("6. Generating 3D...")
        fig3d = go.Figure()
        max_jz = np.max(jz_gamut_opt[:,0]); sc = np.array([1.0/max_jz, 1.0, 1.0]); 
        def mc(a): return a[:,1], a[:,2], a[:,0]

        # MacAdam Solid Hull (White Faint) 
        macadam_jz = xyz_to_jzazbz(macadam_xyz)
        hull_macadam = ConvexHull(macadam_jz)
        mx, my, mz = mc(macadam_jz * sc)
        fig3d.add_trace(go.Mesh3d(x=mx, y=my, z=mz, i=hull_macadam.simplices[:,0], j=hull_macadam.simplices[:,1], k=hull_macadam.simplices[:,2],
                                  color='white', opacity=0.05, name='Visible Gamut (MacAdam)'))

        # sRGB Edges (White Wireframe)
        corners = [[0,0,0],[1,0,0],[1,1,0],[0,1,0],[0,0,1],[1,0,1],[1,1,1],[0,1,1]]
        edges_idx = [[0,1],[1,2],[2,3],[3,0],[4,5],[5,6],[6,7],[7,4],[0,4],[1,5],[2,6],[3,7]]
        for (s,e) in edges_idx:
            pts = np.linspace(corners[s],corners[e],25)
            line = xyz_to_jzazbz(srgb_to_xyz(pts)) * sc
            x,y,z = mc(line)
            fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', line=dict(color='white', width=2), showlegend=False))

        # Ellipsoid (Green)
        u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
        sph = np.stack([np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v)], axis=-1).reshape(-1, 3)
        ell = (sph @ M_max.T + center) * sc
        ex, ey, ez = mc(ell)
        fig3d.add_trace(go.Surface(x=ex.reshape(u.shape), y=ey.reshape(u.shape), z=ez.reshape(u.shape),
                                colorscale=[(0,'#88aa88'),(1,'#88aa88')], opacity=0.3, showscale=False, name='Ellipsoid'))

        # Loops
        hx, hy, hz = mc(hoop * sc)
        c_hoop = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in linear_to_srgb_np(np.clip(rgb_hoop_raw,0,1))]
        fig3d.add_trace(go.Scatter3d(x=hx, y=hy, z=hz, mode='markers', marker=dict(color=c_hoop, size=5), name='Ellipsoid Loop'))

        # Projected Ridge (Cyan)
        px, py, pz = mc(proj_locus_jz * sc)
        fig3d.add_trace(go.Scatter3d(x=px, y=py, z=pz, mode='lines', line=dict(color='cyan', width=6), name='Projected MacAdam Ridge'))

        # MacAdam Ridge Ref (White Solid)
        ridge_jz = xyz_to_jzazbz(macadam_ridge_xyz) * sc
        rx, ry, rz = mc(ridge_jz)
        fig3d.add_trace(go.Scatter3d(x=rx, y=ry, z=rz, mode='lines', line=dict(color='white', width=3), name='MacAdam Ridge (Ref)'))

        fig3d.update_layout(template="plotly_dark", title="3D Jzazbz: MacAdam Ridge Projection", 
                            scene=dict(xaxis_title='az', yaxis_title='bz', zaxis_title='Jz', aspectmode='manual', aspectratio=dict(x=1,y=1,z=1)))
        fig3d.show()

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import scipy.optimize as opt
from scipy.spatial import ConvexHull
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import omnirefactor.core
from skimage import io
import fastremap

# ==========================================
# 1. Color Math
# ==========================================
def srgb_to_xyz(rgb):
    mask = rgb > 0.04045
    linear = np.zeros_like(rgb)
    linear[mask] = ((rgb[mask] + 0.055) / 1.055) ** 2.4
    linear[~mask] = rgb[~mask] / 12.92
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    return linear @ M.T

def xyz_to_jzazbz(xyz):
    b, g, d, d0 = 1.15, 0.66, -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    lms = xyz @ M1.T
    lms_norm = (lms * 200) / 10000.0 
    lms_prime = np.sign(lms_norm) * (((c1 + c2 * (np.abs(lms_norm) ** n)) / (1 + c3 * (np.abs(lms_norm) ** n))) ** p)
    izazbz = lms_prime @ M2.T
    Jz = ((1 + d) * izazbz[:, 0]) / (1 + d * izazbz[:, 0]) - d0
    return np.column_stack((Jz, izazbz[:, 1], izazbz[:, 2]))

def jzazbz_to_xyz(jzazbz):
    Jz, az, bz = jzazbz[:,0], jzazbz[:,1], jzazbz[:,2]
    d, d0 = -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    Jp = Jz + d0; Iz = Jp / (1 + d - d * Jp)
    izazbz_vec = np.stack([Iz, az, bz], axis=1)
    lms_prime = izazbz_vec @ np.linalg.inv(M2).T
    sign_lms = np.sign(lms_prime)
    Y = np.abs(lms_prime) ** (1/p)
    num = Y - c1; den = c2 - Y * c3
    An = np.clip(num / den, 0, None) 
    lms_norm = sign_lms * (An ** (1/n))
    lms = (lms_norm * 10000.0) / 200.0
    return lms @ np.linalg.inv(M1).T

def xyz_to_srgb_raw(xyz):
    M_inv = np.linalg.inv(np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]]))
    linear = xyz @ M_inv.T
    srgb = np.zeros_like(linear)
    mask = linear > 0.0031308
    srgb[mask] = 1.055 * (linear[mask] ** (1/2.4)) - 0.055
    srgb[~mask] = 12.92 * linear[~mask]
    return srgb

def xyz_to_srgb(xyz): return np.clip(xyz_to_srgb_raw(xyz), 0, 1)
def srgb_to_linear_np(srgb): return np.where(srgb <= 0.04045, srgb / 12.92, ((srgb + 0.055) / 1.055) ** 2.4)
def linear_to_srgb_np(lin): 
    s = np.zeros_like(lin)
    mask = lin > 0.0031308
    s[mask] = 1.055 * (lin[mask]**(1/2.4)) - 0.055
    s[~mask] = 12.92 * lin[~mask]
    return np.clip(s, 0, 1)

# ==========================================
# 2. MacAdam Solid & Ridge (Smoothed)
# ==========================================

def get_cmf_high_res(res=360):
    base_cmf = np.array([
        [0.0014,0.0000,0.0065], [0.0042,0.0001,0.0201], [0.0143,0.0004,0.0679], [0.0435,0.0012,0.2074],
        [0.1344,0.0040,0.6456], [0.2839,0.0116,1.3856], [0.3483,0.0230,1.7471], [0.3362,0.0380,1.7721],
        [0.2908,0.0600,1.6692], [0.1954,0.0910,1.2876], [0.0956,0.1390,0.8130], [0.0320,0.2080,0.4652],
        [0.0049,0.3230,0.2720], [0.0093,0.5030,0.1582], [0.0633,0.7100,0.0782], [0.1655,0.8620,0.0422],
        [0.2904,0.9540,0.0203], [0.4334,0.9950,0.0087], [0.5945,0.9950,0.0039], [0.7621,0.9520,0.0021],
        [0.9163,0.8700,0.0017], [1.0263,0.7570,0.0011], [1.0622,0.6310,0.0008], [1.0026,0.5030,0.0006],
        [0.8544,0.3810,0.0002], [0.6424,0.2650,0.0000], [0.4479,0.1750,0.0000], [0.2835,0.1070,0.0000],
        [0.1649,0.0610,0.0000], [0.0874,0.0320,0.0000], [0.0468,0.0170,0.0000], [0.0227,0.0082,0.0000],
        [0.0114,0.0041,0.0000], [0.0058,0.0021,0.0000], [0.0029,0.0010,0.0000], [0.0014,0.0005,0.0000]
    ])
    x = np.linspace(0, 1, len(base_cmf))
    xi = np.linspace(0, 1, res)
    interp_cmf = np.zeros((res, 3))
    for k in range(3):
        interp_cmf[:,k] = np.interp(xi, x, base_cmf[:,k])
    return interp_cmf

def compute_macadam_solid(res=90): # Higher res for smoothness
    cmf = get_cmf_high_res(360)
    n_lambda = len(cmf)
    cum_cmf = np.vstack([np.zeros(3), np.cumsum(cmf, axis=0)])
    white_xyz = cum_cmf[-1]
    
    optimal_colors = []
    indices = np.linspace(0, n_lambda, res).astype(int)
    
    for i in indices:
        for j in indices:
            if i == j: continue
            if j > i: xyz = cum_cmf[j] - cum_cmf[i]
            else:     xyz = cum_cmf[j] + (white_xyz - cum_cmf[i])
            optimal_colors.append(xyz)
    
    optimal_colors.append([0,0,0])
    optimal_colors.append(white_xyz)
    return np.array(optimal_colors) / white_xyz[1]

def extract_macadam_ridge_smoothed(macadam_jz, res=72):
    """
    Extracts max chroma ridge and applies Gaussian smoothing to fix noise.
    """
    az = macadam_jz[:, 1]
    bz = macadam_jz[:, 2]
    chroma = np.sqrt(az**2 + bz**2)
    angle = np.arctan2(bz, az)
    
    bins = np.linspace(-np.pi, np.pi, res+1)
    ridge_points = []
    
    for i in range(res):
        # Wrap around logic for angle
        mask = (angle >= bins[i]) & (angle < bins[i+1])
        if np.sum(mask) == 0: 
            # Fallback: if bin empty, look wider or duplicate prev
            if len(ridge_points) > 0: ridge_points.append(ridge_points[-1])
            else: ridge_points.append(macadam_jz[0]) # Fail safe
            continue
            
        slice_pts = macadam_jz[mask]
        slice_chroma = chroma[mask]
        idx_max = np.argmax(slice_chroma)
        ridge_points.append(slice_pts[idx_max])
        
    raw_ridge = np.array(ridge_points)
    
    # --- SMOOTHING ---
    # Circular convolution with Gaussian kernel
    def smooth_channel(data, sigma=2):
        # Pad for circularity
        padded = np.concatenate([data[-5:], data, data[:5]])
        # Simple gaussian kernel
        kernel = np.exp(-np.linspace(-2,2,5)**2 / (2*sigma**2))
        kernel /= kernel.sum()
        smoothed = np.convolve(padded, kernel, mode='same')
        return smoothed[5:-5]

    J_smooth = smooth_channel(raw_ridge[:,0])
    a_smooth = smooth_channel(raw_ridge[:,1])
    b_smooth = smooth_channel(raw_ridge[:,2])
    
    return np.column_stack([J_smooth, a_smooth, b_smooth])

# ==========================================
# 3. Optimization
# ==========================================

def get_gamut_surface(res=40):
    t = np.linspace(0, 1, res)
    g = np.meshgrid(t, t)
    faces = [np.stack([g[0], g[1], np.zeros_like(g[0])], -1), np.stack([g[0], g[1], np.ones_like(g[0])], -1),
             np.stack([g[0], np.zeros_like(g[0]), g[1]], -1), np.stack([g[0], np.ones_like(g[0]), g[1]], -1),
             np.stack([np.zeros_like(g[0]), g[0], g[1]], -1), np.stack([np.ones_like(g[0]), g[0], g[1]], -1)]
    pts = np.vstack([f.reshape(-1, 3) for f in faces])
    corners = np.array([[0,0,0], [1,0,0], [0,1,0], [0,0,1], [1,1,0], [1,0,1], [0,1,1], [1,1,1]])
    return np.vstack([pts, corners])

def fit_ellipsoid_anchored_solver(points, anchor_point):
    hull = ConvexHull(points)
    A = hull.equations[:, :3]; b_const = -hull.equations[:, 3]
    scale = 100.0; A_scaled = A / scale; anchor_scaled = anchor_point * scale
    c0 = np.mean(points, axis=0) * scale
    c0 = 0.5 * c0 + 0.5 * anchor_scaled 
    x0 = np.concatenate([c0, np.eye(3).flatten() * 0.05])

    def objective(x):
        M = x[3:].reshape(3, 3); sign, val = np.linalg.slogdet(M)
        return -val if sign > 0 else 1e9

    def constraints(x):
        d, M = x[:3], x[3:].reshape(3, 3)
        norm_AM = np.linalg.norm(A_scaled @ M, axis=1)
        hull_con = b_const - (A_scaled @ d + norm_AM)
        try: u = np.linalg.solve(M, anchor_scaled - d); anchor_con = 1.0 - np.linalg.norm(u)
        except: anchor_con = -1.0
        return np.concatenate([hull_con, [anchor_con]])

    res = opt.minimize(objective, x0, method='SLSQP', constraints={'type': 'ineq', 'fun': constraints}, options={'maxiter': 1000})
    return res.x[:3]/scale, res.x[3:].reshape(3, 3)/scale

def calculate_shadow_boundary(center, M, view_vector, res=72):
    w = np.linalg.solve(M, view_vector); w = w / np.linalg.norm(w)
    tmp = np.array([0.0, 1.0, 0.0]) if np.abs(w[1]) < 0.9 else np.array([0.0, 0.0, 1.0])
    s1 = np.cross(w, tmp); s1 /= np.linalg.norm(s1); s2 = np.cross(w, s1)
    theta = np.linspace(0, 2*np.pi, res)
    return center + (np.outer(np.cos(theta), s1) + np.outer(np.sin(theta), s2)) @ M.T

def inflate_to_max_gamut(center, M_init):
    scale = 1.0; step = 0.05; best_M = M_init.copy()
    for _ in range(50):
        test_M = M_init * (scale + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        scale += step; best_M = test_M
    step = 0.005; current_M = best_M
    for _ in range(20):
        test_M = current_M * (1.0 + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        current_M = test_M
    return current_M

def project_ridge_onto_ellipsoid(ridge_jz, center, M):
    vecs = ridge_jz - center
    M_inv = np.linalg.inv(M)
    u_vecs = vecs @ M_inv.T
    norms = np.linalg.norm(u_vecs, axis=1, keepdims=True)
    u_surface = u_vecs / norms
    return (u_surface @ M.T) + center

# ==========================================
# 4. Omnipose Helpers
# ==========================================
def align_ring_red_to_green(ring_lin):
    n = len(ring_lin)
    red_ref = np.array([1.0, 0.0, 0.0])
    idx_r = np.argmin(np.linalg.norm(ring_lin - red_ref, axis=1))
    ring_aligned = np.roll(ring_lin, -idx_r, axis=0)
    green_ref = np.array([0.0, 1.0, 0.0])
    if np.linalg.norm(ring_aligned[(2*n)//3]-green_ref) < np.linalg.norm(ring_aligned[n//3]-green_ref):
        ring_aligned = ring_aligned[::-1]
    return ring_aligned

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    def build_disk(ring, center):
        y, x = np.mgrid[-1:1:256j, -1:1:256j]
        mag = np.sqrt(x*x + y*y)
        angle = np.mod(np.arctan2(y, x), 2*np.pi)
        n = ring.shape[0]
        hue_f = angle/(2*np.pi)*n
        idx0 = np.floor(hue_f).astype(int) % n
        idx1 = (idx0 + 1) % n
        t = hue_f - np.floor(hue_f)
        interp = (1-t[...,None])*ring[idx0] + t[...,None]*ring[idx1]
        rgb = (1-mag[...,None])*center + mag[...,None]*interp
        return np.clip(linear_to_srgb_np(rgb),0,1)

    disk = build_disk(rgb_ring_lin, center_lin)
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi)
    mag = np.clip(np.sqrt(u*u + v*v), 0, 1)
    n = rgb_ring_lin.shape[0]
    hue_f = angle/(2*np.pi)*n
    idx0 = np.floor(hue_f).astype(int) % n
    idx1 = (idx0 + 1) % n
    t = hue_f - np.floor(hue_f)
    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    r = mag[..., None]
    rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow),0,1)
    alpha = mag[...,None]
    white = np.ones_like(flow_rgb); black = np.zeros_like(flow_rgb)
    flow_white = alpha*flow_rgb + (1-alpha)*white
    flow_black = alpha*flow_rgb + (1-alpha)*black
    return disk, flow_rgb, flow_white, flow_black

# ==========================================
# 5. Main Execution
# ==========================================

def main():
    device_str = "cpu"
    dev = torch.device(device_str)
    
    print("1. Calculating Geometries...")
    surf_pts_opt = get_gamut_surface(40) 
    jz_gamut_opt = xyz_to_jzazbz(srgb_to_xyz(surf_pts_opt))
    centroid = np.mean(jz_gamut_opt, axis=0)
    green_vertex = xyz_to_jzazbz(srgb_to_xyz(np.array([[0.0, 1.0, 0.0]])))[0]
    
    print("2. Fitting Maximal Ellipsoid...")
    anchor = centroid + 0.60 * (green_vertex - centroid)
    center, M_solver = fit_ellipsoid_anchored_solver(jz_gamut_opt, anchor)
    M_max = inflate_to_max_gamut(center, M_solver)
    
    # Loops
    hoop = calculate_shadow_boundary(center, M_max, np.array([1.0, 0.0, 0.0]), res=72)
    rgb_hoop_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(hoop)),0,1)))
    
    t = np.linspace(0, 1, 72, endpoint=False)
    sr = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0/3))
    sg = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1/3))
    sb = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2/3))
    rgb_sine_lin = align_ring_red_to_green(srgb_to_linear_np(np.stack([sr, sg, sb], axis=1)))
    
    # MacAdam Ridge
    print("3. Computing MacAdam Ridge (Smoothed)...")
    macadam_xyz = compute_macadam_solid(res=90) # High res for smoothness
    macadam_jz = xyz_to_jzazbz(macadam_xyz)
    ridge_jz = extract_macadam_ridge_smoothed(macadam_jz, res=72)
    
    proj_ridge_jz = project_ridge_onto_ellipsoid(ridge_jz, center, M_max)
    rgb_ridge_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(proj_ridge_jz)), 0, 1)))

    print("4. Loading Omnipose Flows...")
    omnidir = Path(omnirefactor.__file__).resolve().parent.parent.parent
    basedir = omnidir / "docs" / "_static"
    name = "ecoli"
    ext = ".tif"

    try:
        masks = io.imread(str(basedir / f"{name}_labels{ext}"))
        slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows = omnirefactor.core.masks_to_flows(masks, device=device_str)[4].to(dev)
        center_lin = srgb_to_linear_np(np.array([0.5, 0.5, 0.5]))

        def gen_set(ring, rotation_steps=0):
            r = np.roll(ring, rotation_steps, axis=0)
            return make_flow_images_for_ring(r, center_lin, flows)

        d_s, _, w_s, b_s = gen_set(rgb_sine_lin, 0)
        d_e, _, w_e, b_e = gen_set(rgb_hoop_lin, 0)
        d_r, _, w_r, b_r = gen_set(rgb_ridge_lin, 0)

        print("5. Plotting 2D...")
        plt.style.use('dark_background')
        fig, axes = plt.subplots(3, 5, figsize=(20, 12))
        rows = [("Sinebow", rgb_sine_lin, d_s, b_s, w_s),
                ("Ellipsoid Loop", rgb_hoop_lin, d_e, b_e, w_e),
                ("Projected Ridge", rgb_ridge_lin, d_r, b_r, w_r)]
        
        titles = ["Gradient", "Hue Disk", "Black (0°)", "White (0°)", "Black (90°)"]
        for i, (name, ring, disk, b, w) in enumerate(rows):
            ax_row = axes[i]
            ax_row[0].imshow(linear_to_srgb_np(ring)[None, ...], aspect='auto')
            ax_row[0].set_ylabel(name, fontsize=14, color='white', rotation=90, labelpad=20)
            ax_row[1].imshow(disk); ax_row[2].imshow(b); ax_row[3].imshow(w)
            for j, ax in enumerate(ax_row):
                if i==0: ax.set_title(titles[j], fontsize=12, color='#dddddd')
                ax.set_xticks([]); ax.set_yticks([])
        plt.tight_layout()
        plt.show()
        
        print("6. Generating 3D...")
        fig3d = go.Figure()
        max_jz = np.max(jz_gamut_opt[:,0]); sc = np.array([1.0/max_jz, 1.0, 1.0]); 
        def mc(a): return a[:,1], a[:,2], a[:,0]

        # MacAdam Hull
        hull_macadam = ConvexHull(macadam_jz)
        mx, my, mz = mc(macadam_jz * sc)
        fig3d.add_trace(go.Mesh3d(x=mx, y=my, z=mz, i=hull_macadam.simplices[:,0], j=hull_macadam.simplices[:,1], k=hull_macadam.simplices[:,2],
                                  color='white', opacity=0.05, name='MacAdam Solid'))

        # sRGB Wireframe
        corners = [[0,0,0],[1,0,0],[1,1,0],[0,1,0],[0,0,1],[1,0,1],[1,1,1],[0,1,1]]
        edges_idx = [[0,1],[1,2],[2,3],[3,0],[4,5],[5,6],[6,7],[7,4],[0,4],[1,5],[2,6],[3,7]]
        for (s,e) in edges_idx:
            line = xyz_to_jzazbz(srgb_to_xyz(np.linspace(corners[s],corners[e],25))) * sc
            x,y,z = mc(line)
            fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', line=dict(color='gray', width=2), showlegend=False))

        # Ellipsoid
        u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
        sph = np.stack([np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v)], axis=-1).reshape(-1, 3)
        ell = (sph @ M_max.T + center) * sc
        ex, ey, ez = mc(ell)
        fig3d.add_trace(go.Surface(x=ex.reshape(u.shape), y=ey.reshape(u.shape), z=ez.reshape(u.shape),
                                colorscale=[(0,'#88aa88'),(1,'#88aa88')], opacity=0.3, showscale=False, name='Ellipsoid'))

        # Loops
        hx, hy, hz = mc(hoop * sc)
        fig3d.add_trace(go.Scatter3d(x=hx, y=hy, z=hz, mode='markers', marker=dict(color='orange', size=4), name='Ellipsoid Loop'))

        rx, ry, rz = mc(proj_ridge_jz * sc)
        fig3d.add_trace(go.Scatter3d(x=rx, y=ry, z=rz, mode='lines', line=dict(color='cyan', width=6), name='Projected MacAdam Ridge'))

        rrx, rry, rrz = mc(ridge_jz * sc)
        fig3d.add_trace(go.Scatter3d(x=rrx, y=rry, z=rrz, mode='lines', line=dict(color='white', width=3), name='MacAdam Ridge (Ref)'))

        fig3d.update_layout(template="plotly_dark", title="3D Jzazbz: MacAdam Ridge Projection", 
                            scene=dict(xaxis_title='az', yaxis_title='bz', zaxis_title='Jz', aspectmode='manual', aspectratio=dict(x=1,y=1,z=1)))
        fig3d.show()

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import scipy.optimize as opt
from scipy.spatial import ConvexHull
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import omnirefactor.core
from skimage import io
import fastremap

# ==========================================
# 1. Color Math
# ==========================================
def srgb_to_xyz(rgb):
    mask = rgb > 0.04045
    linear = np.zeros_like(rgb)
    linear[mask] = ((rgb[mask] + 0.055) / 1.055) ** 2.4
    linear[~mask] = rgb[~mask] / 12.92
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    return linear @ M.T

def xyz_to_jzazbz(xyz):
    b, g, d, d0 = 1.15, 0.66, -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    lms = xyz @ M1.T
    lms_norm = (lms * 200) / 10000.0 
    lms_prime = np.sign(lms_norm) * (((c1 + c2 * (np.abs(lms_norm) ** n)) / (1 + c3 * (np.abs(lms_norm) ** n))) ** p)
    izazbz = lms_prime @ M2.T
    Jz = ((1 + d) * izazbz[:, 0]) / (1 + d * izazbz[:, 0]) - d0
    return np.column_stack((Jz, izazbz[:, 1], izazbz[:, 2]))

def jzazbz_to_xyz(jzazbz):
    Jz, az, bz = jzazbz[:,0], jzazbz[:,1], jzazbz[:,2]
    d, d0 = -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    Jp = Jz + d0; Iz = Jp / (1 + d - d * Jp)
    izazbz_vec = np.stack([Iz, az, bz], axis=1)
    lms_prime = izazbz_vec @ np.linalg.inv(M2).T
    sign_lms = np.sign(lms_prime)
    Y = np.abs(lms_prime) ** (1/p)
    num = Y - c1; den = c2 - Y * c3
    An = np.clip(num / den, 0, None) 
    lms_norm = sign_lms * (An ** (1/n))
    lms = (lms_norm * 10000.0) / 200.0
    return lms @ np.linalg.inv(M1).T

def xyz_to_srgb_raw(xyz):
    M_inv = np.linalg.inv(np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]]))
    linear = xyz @ M_inv.T
    srgb = np.zeros_like(linear)
    mask = linear > 0.0031308
    srgb[mask] = 1.055 * (linear[mask] ** (1/2.4)) - 0.055
    srgb[~mask] = 12.92 * linear[~mask]
    return srgb

def xyz_to_srgb(xyz): return np.clip(xyz_to_srgb_raw(xyz), 0, 1)
def srgb_to_linear_np(srgb): return np.where(srgb <= 0.04045, srgb / 12.92, ((srgb + 0.055) / 1.055) ** 2.4)
def linear_to_srgb_np(lin): 
    s = np.zeros_like(lin)
    mask = lin > 0.0031308
    s[mask] = 1.055 * (lin[mask]**(1/2.4)) - 0.055
    s[~mask] = 12.92 * lin[~mask]
    return np.clip(s, 0, 1)

# ==========================================
# 2. MacAdam Solid & Ridge (Smoothed)
# ==========================================

def get_cmf_high_res(res=360):
    base_cmf = np.array([
        [0.0014,0.0000,0.0065], [0.0042,0.0001,0.0201], [0.0143,0.0004,0.0679], [0.0435,0.0012,0.2074],
        [0.1344,0.0040,0.6456], [0.2839,0.0116,1.3856], [0.3483,0.0230,1.7471], [0.3362,0.0380,1.7721],
        [0.2908,0.0600,1.6692], [0.1954,0.0910,1.2876], [0.0956,0.1390,0.8130], [0.0320,0.2080,0.4652],
        [0.0049,0.3230,0.2720], [0.0093,0.5030,0.1582], [0.0633,0.7100,0.0782], [0.1655,0.8620,0.0422],
        [0.2904,0.9540,0.0203], [0.4334,0.9950,0.0087], [0.5945,0.9950,0.0039], [0.7621,0.9520,0.0021],
        [0.9163,0.8700,0.0017], [1.0263,0.7570,0.0011], [1.0622,0.6310,0.0008], [1.0026,0.5030,0.0006],
        [0.8544,0.3810,0.0002], [0.6424,0.2650,0.0000], [0.4479,0.1750,0.0000], [0.2835,0.1070,0.0000],
        [0.1649,0.0610,0.0000], [0.0874,0.0320,0.0000], [0.0468,0.0170,0.0000], [0.0227,0.0082,0.0000],
        [0.0114,0.0041,0.0000], [0.0058,0.0021,0.0000], [0.0029,0.0010,0.0000], [0.0014,0.0005,0.0000]
    ])
    x = np.linspace(0, 1, len(base_cmf))
    xi = np.linspace(0, 1, res)
    interp_cmf = np.zeros((res, 3))
    for k in range(3):
        interp_cmf[:,k] = np.interp(xi, x, base_cmf[:,k])
    return interp_cmf

def compute_macadam_solid(res=90):
    cmf = get_cmf_high_res(360)
    n_lambda = len(cmf)
    cum_cmf = np.vstack([np.zeros(3), np.cumsum(cmf, axis=0)])
    white_xyz = cum_cmf[-1]
    
    optimal_colors = []
    indices = np.linspace(0, n_lambda, res).astype(int)
    
    for i in indices:
        for j in indices:
            if i == j: continue
            if j > i: xyz = cum_cmf[j] - cum_cmf[i]
            else:     xyz = cum_cmf[j] + (white_xyz - cum_cmf[i])
            optimal_colors.append(xyz)
    
    optimal_colors.append([0,0,0])
    optimal_colors.append(white_xyz)
    return np.array(optimal_colors) / white_xyz[1]

def extract_macadam_ridge_smoothed(macadam_jz, res=72):
    az = macadam_jz[:, 1]
    bz = macadam_jz[:, 2]
    chroma = np.sqrt(az**2 + bz**2)
    angle = np.arctan2(bz, az)
    bins = np.linspace(-np.pi, np.pi, res+1)
    ridge_points = []
    
    for i in range(res):
        mask = (angle >= bins[i]) & (angle < bins[i+1])
        if np.sum(mask) == 0: 
            if len(ridge_points) > 0: ridge_points.append(ridge_points[-1])
            else: ridge_points.append(macadam_jz[0])
            continue
        slice_pts = macadam_jz[mask]
        slice_chroma = chroma[mask]
        idx_max = np.argmax(slice_chroma)
        ridge_points.append(slice_pts[idx_max])
        
    raw_ridge = np.array(ridge_points)
    # Smoothing
    def smooth_channel(data, sigma=2):
        padded = np.concatenate([data[-5:], data, data[:5]])
        kernel = np.exp(-np.linspace(-2,2,5)**2 / (2*sigma**2))
        kernel /= kernel.sum()
        smoothed = np.convolve(padded, kernel, mode='same')
        return smoothed[5:-5]
    J_smooth = smooth_channel(raw_ridge[:,0])
    a_smooth = smooth_channel(raw_ridge[:,1])
    b_smooth = smooth_channel(raw_ridge[:,2])
    return np.column_stack([J_smooth, a_smooth, b_smooth])

# ==========================================
# 3. Optimization & Projection
# ==========================================

def get_gamut_surface(res=40):
    t = np.linspace(0, 1, res)
    g = np.meshgrid(t, t)
    faces = [np.stack([g[0], g[1], np.zeros_like(g[0])], -1), np.stack([g[0], g[1], np.ones_like(g[0])], -1),
             np.stack([g[0], np.zeros_like(g[0]), g[1]], -1), np.stack([g[0], np.ones_like(g[0]), g[1]], -1),
             np.stack([np.zeros_like(g[0]), g[0], g[1]], -1), np.stack([np.ones_like(g[0]), g[0], g[1]], -1)]
    pts = np.vstack([f.reshape(-1, 3) for f in faces])
    corners = np.array([[0,0,0], [1,0,0], [0,1,0], [0,0,1], [1,1,0], [1,0,1], [0,1,1], [1,1,1]])
    return np.vstack([pts, corners])

def fit_ellipsoid_anchored_solver(points, anchor_point):
    hull = ConvexHull(points)
    A = hull.equations[:, :3]; b_const = -hull.equations[:, 3]
    scale = 100.0; A_scaled = A / scale; anchor_scaled = anchor_point * scale
    c0 = np.mean(points, axis=0) * scale
    c0 = 0.5 * c0 + 0.5 * anchor_scaled 
    x0 = np.concatenate([c0, np.eye(3).flatten() * 0.05])
    def objective(x):
        M = x[3:].reshape(3, 3); sign, val = np.linalg.slogdet(M)
        return -val if sign > 0 else 1e9
    def constraints(x):
        d, M = x[:3], x[3:].reshape(3, 3)
        norm_AM = np.linalg.norm(A_scaled @ M, axis=1)
        hull_con = b_const - (A_scaled @ d + norm_AM)
        try: u = np.linalg.solve(M, anchor_scaled - d); anchor_con = 1.0 - np.linalg.norm(u)
        except: anchor_con = -1.0
        return np.concatenate([hull_con, [anchor_con]])
    res = opt.minimize(objective, x0, method='SLSQP', constraints={'type': 'ineq', 'fun': constraints}, options={'maxiter': 1000})
    return res.x[:3]/scale, res.x[3:].reshape(3, 3)/scale

def calculate_shadow_boundary(center, M, view_vector, res=72):
    w = np.linalg.solve(M, view_vector); w = w / np.linalg.norm(w)
    tmp = np.array([0.0, 1.0, 0.0]) if np.abs(w[1]) < 0.9 else np.array([0.0, 0.0, 1.0])
    s1 = np.cross(w, tmp); s1 /= np.linalg.norm(s1); s2 = np.cross(w, s1)
    theta = np.linspace(0, 2*np.pi, res)
    return center + (np.outer(np.cos(theta), s1) + np.outer(np.sin(theta), s2)) @ M.T

def inflate_to_max_gamut(center, M_init):
    scale = 1.0; step = 0.05; best_M = M_init.copy()
    for _ in range(50):
        test_M = M_init * (scale + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        scale += step; best_M = test_M
    step = 0.005; current_M = best_M
    for _ in range(20):
        test_M = current_M * (1.0 + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        current_M = test_M
    return current_M

def project_ridge_onto_ellipsoid(ridge_jz, center, M):
    vecs = ridge_jz - center
    M_inv = np.linalg.inv(M)
    u_vecs = vecs @ M_inv.T
    norms = np.linalg.norm(u_vecs, axis=1, keepdims=True)
    u_surface = u_vecs / norms
    return (u_surface @ M.T) + center

def shrink_ridge_into_gamut(ridge_jz, center_point):
    scale = 1.0; step = 0.01
    for _ in range(100):
        test_ridge = center_point + scale * (ridge_jz - center_point)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(test_ridge))
        if np.min(rgb) >= -0.005 and np.max(rgb) <= 1.005: # tolerance for noise
            return test_ridge
        scale -= step
    return ridge_jz * 0.1

# ==========================================
# 4. Omnipose Helpers
# ==========================================
def align_ring_red_to_green(ring_lin):
    n = len(ring_lin)
    red_ref = np.array([1.0, 0.0, 0.0])
    idx_r = np.argmin(np.linalg.norm(ring_lin - red_ref, axis=1))
    ring_aligned = np.roll(ring_lin, -idx_r, axis=0)
    green_ref = np.array([0.0, 1.0, 0.0])
    if np.linalg.norm(ring_aligned[(2*n)//3]-green_ref) < np.linalg.norm(ring_aligned[n//3]-green_ref):
        ring_aligned = ring_aligned[::-1]
    return ring_aligned

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    def build_disk(ring, center):
        y, x = np.mgrid[-1:1:256j, -1:1:256j]
        mag = np.sqrt(x*x + y*y)
        angle = np.mod(np.arctan2(y, x), 2*np.pi)
        n = ring.shape[0]
        hue_f = angle/(2*np.pi)*n
        idx0 = np.floor(hue_f).astype(int) % n
        idx1 = (idx0 + 1) % n
        t = hue_f - np.floor(hue_f)
        interp = (1-t[...,None])*ring[idx0] + t[...,None]*ring[idx1]
        rgb = (1-mag[...,None])*center + mag[...,None]*interp
        return np.clip(linear_to_srgb_np(rgb),0,1)

    disk = build_disk(rgb_ring_lin, center_lin)
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi)
    mag = np.clip(np.sqrt(u*u + v*v), 0, 1)
    n = rgb_ring_lin.shape[0]
    hue_f = angle/(2*np.pi)*n
    idx0 = np.floor(hue_f).astype(int) % n
    idx1 = (idx0 + 1) % n
    t = hue_f - np.floor(hue_f)
    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    r = mag[..., None]
    rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow),0,1)
    alpha = mag[...,None]
    white = np.ones_like(flow_rgb); black = np.zeros_like(flow_rgb)
    flow_white = alpha*flow_rgb + (1-alpha)*white
    flow_black = alpha*flow_rgb + (1-alpha)*black
    return disk, flow_rgb, flow_white, flow_black

# ==========================================
# 5. Main Execution
# ==========================================

def main():
    device_str = "cpu"
    dev = torch.device(device_str)
    
    print("1. Calculating Geometries...")
    surf_pts_opt = get_gamut_surface(40) 
    jz_gamut_opt = xyz_to_jzazbz(srgb_to_xyz(surf_pts_opt))
    centroid = np.mean(jz_gamut_opt, axis=0)
    green_vertex = xyz_to_jzazbz(srgb_to_xyz(np.array([[0.0, 1.0, 0.0]])))[0]
    anchor = centroid + 0.60 * (green_vertex - centroid)
    
    print("2. Fitting Maximal Ellipsoid...")
    center, M_solver = fit_ellipsoid_anchored_solver(jz_gamut_opt, anchor)
    M_max = inflate_to_max_gamut(center, M_solver)
    hoop = calculate_shadow_boundary(center, M_max, np.array([1.0, 0.0, 0.0]), res=72)
    rgb_hoop_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(hoop)),0,1)))
    
    # Sinebow
    t = np.linspace(0, 1, 72, endpoint=False)
    sr = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0/3))
    sg = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1/3))
    sb = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2/3))
    rgb_sine_std = np.stack([sr, sg, sb], axis=1) 
    rgb_sine_lin = align_ring_red_to_green(srgb_to_linear_np(rgb_sine_std))

    # 3. MacAdam Ridge
    print("3. Computing MacAdam Ridge...")
    macadam_xyz = compute_macadam_solid(res=90)
    macadam_jz = xyz_to_jzazbz(macadam_xyz)
    ridge_jz = extract_macadam_ridge_smoothed(macadam_jz, res=72)
    
    # Project Ridge onto Ellipsoid
    proj_ridge_jz = project_ridge_onto_ellipsoid(ridge_jz, center, M_max)
    rgb_proj_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(proj_ridge_jz)), 0, 1)))
    
    # Shrink Ridge into Gamut
    shrunk_ridge_jz = shrink_ridge_into_gamut(ridge_jz, centroid)
    rgb_shrunk_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(shrunk_ridge_jz)), 0, 1)))

    print("4. Loading Omnipose Flows...")
    omnidir = Path(omnirefactor.__file__).resolve().parent.parent.parent
    basedir = omnidir / "docs" / "_static"
    name = "ecoli"
    ext = ".tif"

    try:
        masks = io.imread(str(basedir / f"{name}_labels{ext}"))
        slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows = omnirefactor.core.masks_to_flows(masks, device=device_str)[4].to(dev)
        center_lin = srgb_to_linear_np(np.array([0.5, 0.5, 0.5]))

        def gen_set(ring, rotation_steps=0):
            r = np.roll(ring, rotation_steps, axis=0)
            return make_flow_images_for_ring(r, center_lin, flows)

        # 2D Rows
        rows = []
        d, f, w, b = gen_set(rgb_sine_lin, 0); d9, f9, w9, b9 = gen_set(rgb_sine_lin, 18)
        rows.append(("Sinebow", rgb_sine_lin, d, b, w, b9, w9))
        
        d, f, w, b = gen_set(rgb_hoop_lin, 0); d9, f9, w9, b9 = gen_set(rgb_hoop_lin, 18)
        rows.append(("Max Ellipse", rgb_hoop_lin, d, b, w, b9, w9))
        
        d, f, w, b = gen_set(rgb_proj_lin, 0); d9, f9, w9, b9 = gen_set(rgb_proj_lin, 18)
        rows.append(("Projected Ridge", rgb_proj_lin, d, b, w, b9, w9))

        d, f, w, b = gen_set(rgb_shrunk_lin, 0); d9, f9, w9, b9 = gen_set(rgb_shrunk_lin, 18)
        rows.append(("Shrunk Ridge", rgb_shrunk_lin, d, b, w, b9, w9))

        print("5. Plotting 2D...")
        plt.style.use('dark_background')
        fig, axes = plt.subplots(4, 5, figsize=(20, 16))
        titles = ["Gradient", "Hue Disk", "Black (0°)", "White (0°)", "Black (90°)"]

        for i, (name, ring, disk, b0, w0, b90, w90) in enumerate(rows):
            ax_row = axes[i]
            disp_gradient = linear_to_srgb_np(ring)[None, ...]
            ax_row[0].imshow(disp_gradient, aspect='auto')
            ax_row[0].set_ylabel(name, fontsize=14, color='white', rotation=90, labelpad=20)
            ax_row[1].imshow(disk); ax_row[2].imshow(b0); ax_row[3].imshow(w0); ax_row[4].imshow(b90)
            for j, ax in enumerate(ax_row):
                if i==0: ax.set_title(titles[j], fontsize=12, color='#dddddd')
                ax.set_xticks([]); ax.set_yticks([])
        plt.tight_layout()
        plt.show()
        
        print("6. Generating 3D...")
        fig3d = go.Figure()
        max_jz = np.max(jz_gamut_opt[:,0]); sc = np.array([1.0/max_jz, 1.0, 1.0]); 
        def mc(a): return a[:,1], a[:,2], a[:,0]

        # MacAdam Hull
        hull_macadam = ConvexHull(macadam_jz)
        mx, my, mz = mc(macadam_jz * sc)
        fig3d.add_trace(go.Mesh3d(x=mx, y=my, z=mz, i=hull_macadam.simplices[:,0], j=hull_macadam.simplices[:,1], k=hull_macadam.simplices[:,2],
                                  color='white', opacity=0.05, name='MacAdam Solid'))

        # sRGB Wireframe
        corners = [[0,0,0],[1,0,0],[1,1,0],[0,1,0],[0,0,1],[1,0,1],[1,1,1],[0,1,1]]
        edges_idx = [[0,1],[1,2],[2,3],[3,0],[4,5],[5,6],[6,7],[7,4],[0,4],[1,5],[2,6],[3,7]]
        for (s,e) in edges_idx:
            l = xyz_to_jzazbz(srgb_to_xyz(np.linspace(corners[s],corners[e],25))) * sc
            x,y,z = mc(l)
            fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', line=dict(color='gray', width=2), showlegend=False))

        # Ellipsoid
        u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
        sph = np.stack([np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v)], axis=-1).reshape(-1, 3)
        ell = (sph @ M_max.T + center) * sc
        ex, ey, ez = mc(ell)
        fig3d.add_trace(go.Surface(x=ex.reshape(u.shape), y=ey.reshape(u.shape), z=ez.reshape(u.shape),
                                colorscale=[(0,'#88aa88'),(1,'#88aa88')], opacity=0.3, showscale=False, name='Ellipsoid'))

        def plot_dots(pts, col, name):
             x,y,z = mc(pts * sc)
             # Use clipped sRGB for colors
             rgb = linear_to_srgb_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(pts)), 0, 1))
             cols = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in rgb]
             fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='markers', marker=dict(color=cols, size=4), name=name))

        plot_dots(hoop, 'orange', 'Ellipsoid Loop')
        plot_dots(xyz_to_jzazbz(srgb_to_xyz(rgb_sine_std)), 'green', 'Sinebow Loop')
        plot_dots(proj_ridge_jz, 'cyan', 'Projected MacAdam Ridge')
        plot_dots(shrunk_ridge_jz, 'magenta', 'Shrunk MacAdam Ridge')

        # Ridge Ref
        rrx, rry, rrz = mc(ridge_jz * sc)
        fig3d.add_trace(go.Scatter3d(x=rrx, y=rry, z=rrz, mode='lines', line=dict(color='white', width=3), name='MacAdam Ridge (Ref)'))

        fig3d.update_layout(template="plotly_dark", title="3D Jzazbz: MacAdam Ridge Projection", 
                            scene=dict(xaxis_title='az', yaxis_title='bz', zaxis_title='Jz', aspectmode='manual', aspectratio=dict(x=1,y=1,z=1)))
        fig3d.show()

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import scipy.optimize as opt
from scipy.spatial import ConvexHull
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import omnirefactor.core
from skimage import io
import fastremap

# ==========================================
# 1. Color Math
# ==========================================
def srgb_to_xyz(rgb):
    mask = rgb > 0.04045
    linear = np.zeros_like(rgb)
    linear[mask] = ((rgb[mask] + 0.055) / 1.055) ** 2.4
    linear[~mask] = rgb[~mask] / 12.92
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    return linear @ M.T

def xyz_to_jzazbz(xyz):
    b, g, d, d0 = 1.15, 0.66, -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    lms = xyz @ M1.T
    lms_norm = (lms * 200) / 10000.0 
    lms_prime = np.sign(lms_norm) * (((c1 + c2 * (np.abs(lms_norm) ** n)) / (1 + c3 * (np.abs(lms_norm) ** n))) ** p)
    izazbz = lms_prime @ M2.T
    Jz = ((1 + d) * izazbz[:, 0]) / (1 + d * izazbz[:, 0]) - d0
    return np.column_stack((Jz, izazbz[:, 1], izazbz[:, 2]))

def jzazbz_to_xyz(jzazbz):
    Jz, az, bz = jzazbz[:,0], jzazbz[:,1], jzazbz[:,2]
    d, d0 = -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    Jp = Jz + d0; Iz = Jp / (1 + d - d * Jp)
    izazbz_vec = np.stack([Iz, az, bz], axis=1)
    lms_prime = izazbz_vec @ np.linalg.inv(M2).T
    sign_lms = np.sign(lms_prime)
    Y = np.abs(lms_prime) ** (1/p)
    num = Y - c1; den = c2 - Y * c3
    An = np.clip(num / den, 0, None) 
    lms_norm = sign_lms * (An ** (1/n))
    lms = (lms_norm * 10000.0) / 200.0
    return lms @ np.linalg.inv(M1).T

def xyz_to_srgb_raw(xyz):
    M_inv = np.linalg.inv(np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]]))
    linear = xyz @ M_inv.T
    srgb = np.zeros_like(linear)
    mask = linear > 0.0031308
    srgb[mask] = 1.055 * (linear[mask] ** (1/2.4)) - 0.055
    srgb[~mask] = 12.92 * linear[~mask]
    return srgb

def xyz_to_srgb(xyz): return np.clip(xyz_to_srgb_raw(xyz), 0, 1)
def srgb_to_linear_np(srgb): return np.where(srgb <= 0.04045, srgb / 12.92, ((srgb + 0.055) / 1.055) ** 2.4)
def linear_to_srgb_np(lin): 
    s = np.zeros_like(lin)
    mask = lin > 0.0031308
    s[mask] = 1.055 * (lin[mask]**(1/2.4)) - 0.055
    s[~mask] = 12.92 * lin[~mask]
    return np.clip(s, 0, 1)

# ==========================================
# 2. MacAdam Solid & Ridge (High Res)
# ==========================================

def get_cmf_high_res(res=360):
    base_cmf = np.array([
        [0.0014,0.0000,0.0065], [0.0042,0.0001,0.0201], [0.0143,0.0004,0.0679], [0.0435,0.0012,0.2074],
        [0.1344,0.0040,0.6456], [0.2839,0.0116,1.3856], [0.3483,0.0230,1.7471], [0.3362,0.0380,1.7721],
        [0.2908,0.0600,1.6692], [0.1954,0.0910,1.2876], [0.0956,0.1390,0.8130], [0.0320,0.2080,0.4652],
        [0.0049,0.3230,0.2720], [0.0093,0.5030,0.1582], [0.0633,0.7100,0.0782], [0.1655,0.8620,0.0422],
        [0.2904,0.9540,0.0203], [0.4334,0.9950,0.0087], [0.5945,0.9950,0.0039], [0.7621,0.9520,0.0021],
        [0.9163,0.8700,0.0017], [1.0263,0.7570,0.0011], [1.0622,0.6310,0.0008], [1.0026,0.5030,0.0006],
        [0.8544,0.3810,0.0002], [0.6424,0.2650,0.0000], [0.4479,0.1750,0.0000], [0.2835,0.1070,0.0000],
        [0.1649,0.0610,0.0000], [0.0874,0.0320,0.0000], [0.0468,0.0170,0.0000], [0.0227,0.0082,0.0000],
        [0.0114,0.0041,0.0000], [0.0058,0.0021,0.0000], [0.0029,0.0010,0.0000], [0.0014,0.0005,0.0000]
    ])
    x = np.linspace(0, 1, len(base_cmf))
    xi = np.linspace(0, 1, res)
    interp_cmf = np.zeros((res, 3))
    for k in range(3):
        interp_cmf[:,k] = np.interp(xi, x, base_cmf[:,k])
    return interp_cmf

def compute_macadam_solid(res=200): # Increased to 200 for density
    cmf = get_cmf_high_res(400)
    n_lambda = len(cmf)
    cum_cmf = np.vstack([np.zeros(3), np.cumsum(cmf, axis=0)])
    white_xyz = cum_cmf[-1]
    
    optimal_colors = []
    indices = np.linspace(0, n_lambda, res).astype(int)
    
    for i in indices:
        for j in indices:
            if i == j: continue
            if j > i: xyz = cum_cmf[j] - cum_cmf[i]
            else:     xyz = cum_cmf[j] + (white_xyz - cum_cmf[i])
            optimal_colors.append(xyz)
    
    optimal_colors.append([0,0,0])
    optimal_colors.append(white_xyz)
    return np.array(optimal_colors) / white_xyz[1]

def extract_macadam_ridge_smoothed(macadam_jz, res=360): # Increased to 360 degrees
    az = macadam_jz[:, 1]
    bz = macadam_jz[:, 2]
    chroma = np.sqrt(az**2 + bz**2)
    angle = np.arctan2(bz, az)
    
    bins = np.linspace(-np.pi, np.pi, res+1)
    ridge_points = []
    
    for i in range(res):
        mask = (angle >= bins[i]) & (angle < bins[i+1])
        if np.sum(mask) == 0: 
            if len(ridge_points) > 0: ridge_points.append(ridge_points[-1])
            else: ridge_points.append(macadam_jz[0])
            continue
        slice_pts = macadam_jz[mask]
        slice_chroma = chroma[mask]
        idx_max = np.argmax(slice_chroma)
        ridge_points.append(slice_pts[idx_max])
        
    raw_ridge = np.array(ridge_points)
    
    # Circular Smoothing (Slightly stronger sigma for high res)
    def smooth_channel(data, sigma=3):
        padded = np.concatenate([data[-20:], data, data[:20]])
        kernel = np.exp(-np.linspace(-5,5,21)**2 / (2*sigma**2))
        kernel /= kernel.sum()
        smoothed = np.convolve(padded, kernel, mode='same')
        return smoothed[20:-20]

    J_smooth = smooth_channel(raw_ridge[:,0])
    a_smooth = smooth_channel(raw_ridge[:,1])
    b_smooth = smooth_channel(raw_ridge[:,2])
    
    return np.column_stack([J_smooth, a_smooth, b_smooth])

# ==========================================
# 3. Optimization & Projection
# ==========================================

def get_gamut_surface(res=40):
    t = np.linspace(0, 1, res)
    g = np.meshgrid(t, t)
    faces = [np.stack([g[0], g[1], np.zeros_like(g[0])], -1), np.stack([g[0], g[1], np.ones_like(g[0])], -1),
             np.stack([g[0], np.zeros_like(g[0]), g[1]], -1), np.stack([g[0], np.ones_like(g[0]), g[1]], -1),
             np.stack([np.zeros_like(g[0]), g[0], g[1]], -1), np.stack([np.ones_like(g[0]), g[0], g[1]], -1)]
    pts = np.vstack([f.reshape(-1, 3) for f in faces])
    corners = np.array([[0,0,0], [1,0,0], [0,1,0], [0,0,1], [1,1,0], [1,0,1], [0,1,1], [1,1,1]])
    return np.vstack([pts, corners])

def fit_ellipsoid_anchored_solver(points, anchor_point):
    hull = ConvexHull(points)
    A = hull.equations[:, :3]; b_const = -hull.equations[:, 3]
    scale = 100.0; A_scaled = A / scale; anchor_scaled = anchor_point * scale
    c0 = np.mean(points, axis=0) * scale
    c0 = 0.5 * c0 + 0.5 * anchor_scaled 
    x0 = np.concatenate([c0, np.eye(3).flatten() * 0.05])

    def objective(x):
        M = x[3:].reshape(3, 3); sign, val = np.linalg.slogdet(M)
        return -val if sign > 0 else 1e9

    def constraints(x):
        d, M = x[:3], x[3:].reshape(3, 3)
        norm_AM = np.linalg.norm(A_scaled @ M, axis=1)
        hull_con = b_const - (A_scaled @ d + norm_AM)
        try: u = np.linalg.solve(M, anchor_scaled - d); anchor_con = 1.0 - np.linalg.norm(u)
        except: anchor_con = -1.0
        return np.concatenate([hull_con, [anchor_con]])

    res = opt.minimize(objective, x0, method='SLSQP', constraints={'type': 'ineq', 'fun': constraints}, options={'maxiter': 1000})
    return res.x[:3]/scale, res.x[3:].reshape(3, 3)/scale

def calculate_shadow_boundary(center, M, view_vector, res=72):
    w = np.linalg.solve(M, view_vector); w = w / np.linalg.norm(w)
    tmp = np.array([0.0, 1.0, 0.0]) if np.abs(w[1]) < 0.9 else np.array([0.0, 0.0, 1.0])
    s1 = np.cross(w, tmp); s1 /= np.linalg.norm(s1); s2 = np.cross(w, s1)
    theta = np.linspace(0, 2*np.pi, res)
    return center + (np.outer(np.cos(theta), s1) + np.outer(np.sin(theta), s2)) @ M.T

def inflate_to_max_gamut(center, M_init):
    scale = 1.0; step = 0.05; best_M = M_init.copy()
    for _ in range(50):
        test_M = M_init * (scale + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        scale += step; best_M = test_M
    step = 0.005; current_M = best_M
    for _ in range(20):
        test_M = current_M * (1.0 + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        current_M = test_M
    return current_M

def project_ridge_onto_ellipsoid(ridge_jz, center, M):
    vecs = ridge_jz - center
    M_inv = np.linalg.inv(M)
    u_vecs = vecs @ M_inv.T
    norms = np.linalg.norm(u_vecs, axis=1, keepdims=True)
    u_surface = u_vecs / norms
    return (u_surface @ M.T) + center

def shrink_ridge_into_gamut(ridge_jz, center_point):
    scale = 1.0; step = 0.01
    for _ in range(100):
        test_ridge = center_point + scale * (ridge_jz - center_point)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(test_ridge))
        if np.min(rgb) >= -0.005 and np.max(rgb) <= 1.005:
             return test_ridge
        scale -= step
    return ridge_jz * 0.1

def project_points_to_srgb_surface_bsearch(points_jz, center_jz):
    projected = []
    for pt in points_jz:
        vec = pt - center_jz
        low = 0.0; high = 1.2; best_t = 0.0
        for _ in range(15):
            mid = (low + high) / 2.0
            test = center_jz + mid * vec
            rgb = xyz_to_srgb_raw(jzazbz_to_xyz(np.array([test])))
            if np.min(rgb) >= -0.001 and np.max(rgb) <= 1.001:
                best_t = mid; low = mid 
            else: high = mid 
        projected.append(center_jz + best_t * vec)
    return np.array(projected)

def reparameterize_curve(points, n_samples=256): # High samples for smoothness
    dists = np.linalg.norm(points[1:] - points[:-1], axis=1)
    cum_dist = np.insert(np.cumsum(dists), 0, 0.0)
    target_dists = np.linspace(0, cum_dist[-1], n_samples)
    new_points = np.zeros((n_samples, 3))
    for i in range(3):
        new_points[:, i] = np.interp(target_dists, cum_dist, points[:, i])
    return new_points

# ==========================================
# 4. Omnipose Helpers
# ==========================================
def align_ring_red_to_green(ring_lin):
    n = len(ring_lin)
    red_ref = np.array([1.0, 0.0, 0.0])
    idx_r = np.argmin(np.linalg.norm(ring_lin - red_ref, axis=1))
    ring_aligned = np.roll(ring_lin, -idx_r, axis=0)
    green_ref = np.array([0.0, 1.0, 0.0])
    if np.linalg.norm(ring_aligned[(2*n)//3]-green_ref) < np.linalg.norm(ring_aligned[n//3]-green_ref):
        ring_aligned = ring_aligned[::-1]
    return ring_aligned

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    def build_disk(ring, center):
        y, x = np.mgrid[-1:1:256j, -1:1:256j]
        mag = np.sqrt(x*x + y*y)
        angle = np.mod(np.arctan2(y, x), 2*np.pi)
        n = ring.shape[0]
        hue_f = angle/(2*np.pi)*n
        idx0 = np.floor(hue_f).astype(int) % n
        idx1 = (idx0 + 1) % n
        t = hue_f - np.floor(hue_f)
        interp = (1-t[...,None])*ring[idx0] + t[...,None]*ring[idx1]
        rgb = (1-mag[...,None])*center + mag[...,None]*interp
        return np.clip(linear_to_srgb_np(rgb),0,1)

    disk = build_disk(rgb_ring_lin, center_lin)
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi)
    mag = np.clip(np.sqrt(u*u + v*v), 0, 1)
    n = rgb_ring_lin.shape[0]
    hue_f = angle/(2*np.pi)*n
    idx0 = np.floor(hue_f).astype(int) % n
    idx1 = (idx0 + 1) % n
    t = hue_f - np.floor(hue_f)
    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    r = mag[..., None]
    rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow),0,1)
    alpha = mag[...,None]
    white = np.ones_like(flow_rgb); black = np.zeros_like(flow_rgb)
    flow_white = alpha*flow_rgb + (1-alpha)*white
    flow_black = alpha*flow_rgb + (1-alpha)*black
    return disk, flow_rgb, flow_white, flow_black

# ==========================================
# 5. Main Execution
# ==========================================

def main():
    device_str = "cpu"
    dev = torch.device(device_str)
    
    print("1. Calculating Geometries...")
    surf_pts_opt = get_gamut_surface(40) 
    jz_gamut_opt = xyz_to_jzazbz(srgb_to_xyz(surf_pts_opt))
    centroid = np.mean(jz_gamut_opt, axis=0)
    green_vertex = xyz_to_jzazbz(srgb_to_xyz(np.array([[0.0, 1.0, 0.0]])))[0]
    anchor = centroid + 0.60 * (green_vertex - centroid)
    
    print("2. Fitting Maximal Ellipsoid...")
    center, M_solver = fit_ellipsoid_anchored_solver(jz_gamut_opt, anchor)
    M_max = inflate_to_max_gamut(center, M_solver)
    
    # Loop A: Ellipsoid Loop
    hoop = calculate_shadow_boundary(center, M_max, np.array([1.0, 0.0, 0.0]), res=72)
    rgb_hoop_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(hoop)),0,1)))
    
    # Loop B: Sinebow
    t = np.linspace(0, 1, 256, endpoint=False) # Match res
    sr = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0/3)); sg = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1/3)); sb = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2/3))
    rgb_sine_std = np.stack([sr, sg, sb], axis=1) 
    rgb_sine_lin = align_ring_red_to_green(srgb_to_linear_np(rgb_sine_std))
    
    # Loop C: MacAdam Ridge
    print("3. Computing High-Res MacAdam Ridge...")
    macadam_xyz = compute_macadam_solid(res=200)
    macadam_jz = xyz_to_jzazbz(macadam_xyz)
    ridge_jz_highres = extract_macadam_ridge_smoothed(macadam_jz, res=360) # Very high res
    
    # Ridge -> Ellipsoid (Reparameterized)
    proj_ridge_jz_raw = project_ridge_onto_ellipsoid(ridge_jz_highres, center, M_max)
    proj_ridge_jz_uniform = reparameterize_curve(proj_ridge_jz_raw, n_samples=256)
    rgb_proj_ell_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(proj_ridge_jz_uniform)), 0, 1)))
    
    # Ridge -> sRGB (Reparameterized)
    proj_srgb_jz_raw = project_points_to_srgb_surface_bsearch(ridge_jz_highres, centroid)
    proj_srgb_jz_uniform = reparameterize_curve(proj_srgb_jz_raw, n_samples=256)
    rgb_proj_srgb_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(proj_srgb_jz_uniform)), 0, 1)))
    
    # Shrunk Ridge (Reparameterized)
    shrunk_ridge_jz_raw = shrink_ridge_into_gamut(ridge_jz_highres, centroid)
    shrunk_ridge_jz_uniform = reparameterize_curve(shrunk_ridge_jz_raw, n_samples=256)
    rgb_shrunk_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(shrunk_ridge_jz_uniform)), 0, 1)))

    print("4. Loading Omnipose Flows...")
    omnidir = Path(omnirefactor.__file__).resolve().parent.parent.parent
    basedir = omnidir / "docs" / "_static"
    name = "ecoli"
    ext = ".tif"

    try:
        masks = io.imread(str(basedir / f"{name}_labels{ext}"))
        slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows = omnirefactor.core.masks_to_flows(masks, device=device_str)[4].to(dev)
        center_lin = srgb_to_linear_np(np.array([0.5, 0.5, 0.5]))

        def gen_set(ring, rot_steps=0):
            # Adjust rotation based on size 256
            idx = int(rot_steps * len(ring) / 72.0)
            r = np.roll(ring, idx, axis=0)
            return make_flow_images_for_ring(r, center_lin, flows)

        rows = []
        d, f, w, b = gen_set(rgb_sine_lin, 0)
        rows.append(("Sinebow", rgb_sine_lin, d, b, w))
        
        d, f, w, b = gen_set(rgb_hoop_lin, 0)
        rows.append(("Max Ellipse", rgb_hoop_lin, d, b, w))
        
        d, f, w, b = gen_set(rgb_proj_ell_lin, 0)
        rows.append(("Ridge->Ellipsoid", rgb_proj_ell_lin, d, b, w))
        
        d, f, w, b = gen_set(rgb_proj_srgb_lin, 0)
        rows.append(("Ridge->sRGB", rgb_proj_srgb_lin, d, b, w))

        d, f, w, b = gen_set(rgb_shrunk_lin, 0)
        rows.append(("Shrunk Ridge", rgb_shrunk_lin, d, b, w))

        print("5. Plotting 2D...")
        plt.style.use('dark_background')
        fig, axes = plt.subplots(5, 4, figsize=(16, 20))
        titles = ["Gradient", "Hue Disk", "Black (0°)", "White (0°)"]

        for i, (name, ring, disk, b, w) in enumerate(rows):
            ax_row = axes[i]
            ax_row[0].imshow(linear_to_srgb_np(ring)[None, ...], aspect='auto')
            ax_row[0].set_ylabel(name, fontsize=14, color='white', rotation=90, labelpad=20)
            ax_row[1].imshow(disk); ax_row[2].imshow(b); ax_row[3].imshow(w)
            for j, ax in enumerate(ax_row):
                if i==0: ax.set_title(titles[j], fontsize=12, color='#dddddd')
                ax.set_xticks([]); ax.set_yticks([])
        plt.tight_layout()
        plt.show()
        
        print("6. Generating 3D...")
        fig3d = go.Figure()
        max_jz = np.max(jz_gamut_opt[:,0]); sc = np.array([1.0/max_jz, 1.0, 1.0]); 
        def mc(a): return a[:,1], a[:,2], a[:,0]

        # MacAdam Hull
        hull_macadam = ConvexHull(macadam_jz)
        mx, my, mz = mc(macadam_jz * sc)
        fig3d.add_trace(go.Mesh3d(x=mx, y=my, z=mz, i=hull_macadam.simplices[:,0], j=hull_macadam.simplices[:,1], k=hull_macadam.simplices[:,2],
                                  color='white', opacity=0.05, name='MacAdam Solid'))

        # sRGB Edges
        corners = [[0,0,0],[1,0,0],[1,1,0],[0,1,0],[0,0,1],[1,0,1],[1,1,1],[0,1,1]]
        edges_idx = [[0,1],[1,2],[2,3],[3,0],[4,5],[5,6],[6,7],[7,4],[0,4],[1,5],[2,6],[3,7]]
        for s,e in edges_idx:
            l = xyz_to_jzazbz(srgb_to_xyz(np.linspace(corners[s],corners[e],25))) * sc
            x,y,z = mc(l)
            fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', line=dict(color='gray', width=2), showlegend=False))

        # Ellipsoid
        u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
        sph = np.stack([np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v)], axis=-1).reshape(-1, 3)
        ell = (sph @ M_max.T + center) * sc
        ex, ey, ez = mc(ell)
        fig3d.add_trace(go.Surface(x=ex.reshape(u.shape), y=ey.reshape(u.shape), z=ez.reshape(u.shape),
                                colorscale=[(0,'#88aa88'),(1,'#88aa88')], opacity=0.3, showscale=False, name='Ellipsoid'))

        def plot_dots(pts, col, name):
             x,y,z = mc(pts * sc)
             rgb = linear_to_srgb_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(pts)), 0, 1))
             cols = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in rgb]
             fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='markers', marker=dict(color=cols, size=4), name=name))

        plot_dots(hoop, 'orange', 'Ellipsoid Loop')
        plot_dots(xyz_to_jzazbz(srgb_to_xyz(rgb_sine_std)), 'green', 'Sinebow Loop')
        plot_dots(proj_ridge_jz_uniform, 'cyan', 'Ridge->Ellipsoid')
        plot_dots(proj_srgb_jz_uniform, 'red', 'Ridge->sRGB')
        plot_dots(shrunk_ridge_jz_uniform, 'magenta', 'Shrunk Ridge')

        # Ridge Ref
        rrx, rry, rrz = mc(ridge_jz_highres * sc)
        fig3d.add_trace(go.Scatter3d(x=rrx, y=rry, z=rrz, mode='lines', line=dict(color='white', width=3), name='MacAdam Ridge (Ref)'))

        fig3d.update_layout(template="plotly_dark", title="3D Jzazbz: MacAdam Ridge Projection (Reparameterized)", 
                            scene=dict(xaxis_title='az', yaxis_title='bz', zaxis_title='Jz', aspectmode='manual', aspectratio=dict(x=1,y=1,z=1)))
        fig3d.show()

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import scipy.optimize as opt
from scipy.spatial import ConvexHull
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import omnirefactor.core
from skimage import io
import fastremap

# ==========================================
# 1. Color Math
# ==========================================
def srgb_to_xyz(rgb):
    mask = rgb > 0.04045
    linear = np.zeros_like(rgb)
    linear[mask] = ((rgb[mask] + 0.055) / 1.055) ** 2.4
    linear[~mask] = rgb[~mask] / 12.92
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    return linear @ M.T

def xyz_to_jzazbz(xyz):
    b, g, d, d0 = 1.15, 0.66, -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    lms = xyz @ M1.T
    lms_norm = (lms * 200) / 10000.0 
    lms_prime = np.sign(lms_norm) * (((c1 + c2 * (np.abs(lms_norm) ** n)) / (1 + c3 * (np.abs(lms_norm) ** n))) ** p)
    izazbz = lms_prime @ M2.T
    Jz = ((1 + d) * izazbz[:, 0]) / (1 + d * izazbz[:, 0]) - d0
    return np.column_stack((Jz, izazbz[:, 1], izazbz[:, 2]))

def jzazbz_to_xyz(jzazbz):
    Jz, az, bz = jzazbz[:,0], jzazbz[:,1], jzazbz[:,2]
    d, d0 = -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    Jp = Jz + d0; Iz = Jp / (1 + d - d * Jp)
    izazbz_vec = np.stack([Iz, az, bz], axis=1)
    lms_prime = izazbz_vec @ np.linalg.inv(M2).T
    sign_lms = np.sign(lms_prime)
    Y = np.abs(lms_prime) ** (1/p)
    num = Y - c1; den = c2 - Y * c3
    An = np.clip(num / den, 0, None) 
    lms_norm = sign_lms * (An ** (1/n))
    lms = (lms_norm * 10000.0) / 200.0
    return lms @ np.linalg.inv(M1).T

def xyz_to_srgb_raw(xyz):
    M_inv = np.linalg.inv(np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]]))
    linear = xyz @ M_inv.T
    srgb = np.zeros_like(linear)
    mask = linear > 0.0031308
    srgb[mask] = 1.055 * (linear[mask] ** (1/2.4)) - 0.055
    srgb[~mask] = 12.92 * linear[~mask]
    return srgb

def xyz_to_srgb(xyz): return np.clip(xyz_to_srgb_raw(xyz), 0, 1)
def srgb_to_linear_np(srgb): return np.where(srgb <= 0.04045, srgb / 12.92, ((srgb + 0.055) / 1.055) ** 2.4)
def linear_to_srgb_np(lin): 
    s = np.zeros_like(lin)
    mask = lin > 0.0031308
    s[mask] = 1.055 * (lin[mask]**(1/2.4)) - 0.055
    s[~mask] = 12.92 * lin[~mask]
    return np.clip(s, 0, 1)

# ==========================================
# 2. Geometry Generators
# ==========================================

def get_cmf_high_res(res=360):
    base_cmf = np.array([
        [0.0014,0.0000,0.0065], [0.0042,0.0001,0.0201], [0.0143,0.0004,0.0679], [0.0435,0.0012,0.2074],
        [0.1344,0.0040,0.6456], [0.2839,0.0116,1.3856], [0.3483,0.0230,1.7471], [0.3362,0.0380,1.7721],
        [0.2908,0.0600,1.6692], [0.1954,0.0910,1.2876], [0.0956,0.1390,0.8130], [0.0320,0.2080,0.4652],
        [0.0049,0.3230,0.2720], [0.0093,0.5030,0.1582], [0.0633,0.7100,0.0782], [0.1655,0.8620,0.0422],
        [0.2904,0.9540,0.0203], [0.4334,0.9950,0.0087], [0.5945,0.9950,0.0039], [0.7621,0.9520,0.0021],
        [0.9163,0.8700,0.0017], [1.0263,0.7570,0.0011], [1.0622,0.6310,0.0008], [1.0026,0.5030,0.0006],
        [0.8544,0.3810,0.0002], [0.6424,0.2650,0.0000], [0.4479,0.1750,0.0000], [0.2835,0.1070,0.0000],
        [0.1649,0.0610,0.0000], [0.0874,0.0320,0.0000], [0.0468,0.0170,0.0000], [0.0227,0.0082,0.0000],
        [0.0114,0.0041,0.0000], [0.0058,0.0021,0.0000], [0.0029,0.0010,0.0000], [0.0014,0.0005,0.0000]
    ])
    x = np.linspace(0, 1, len(base_cmf))
    xi = np.linspace(0, 1, res)
    interp_cmf = np.zeros((res, 3))
    for k in range(3):
        interp_cmf[:,k] = np.interp(xi, x, base_cmf[:,k])
    return interp_cmf

def compute_macadam_solid(res=90):
    cmf = get_cmf_high_res(360)
    n_lambda = len(cmf)
    cum_cmf = np.vstack([np.zeros(3), np.cumsum(cmf, axis=0)])
    white_xyz = cum_cmf[-1]
    optimal_colors = []
    indices = np.linspace(0, n_lambda, res).astype(int)
    for i in indices:
        for j in indices:
            if i == j: continue
            if j > i: xyz = cum_cmf[j] - cum_cmf[i]
            else:     xyz = cum_cmf[j] + (white_xyz - cum_cmf[i])
            optimal_colors.append(xyz)
    optimal_colors.append([0,0,0])
    optimal_colors.append(white_xyz)
    return np.array(optimal_colors) / white_xyz[1]

def extract_macadam_ridge_smoothed(macadam_jz, res=360):
    az = macadam_jz[:, 1]
    bz = macadam_jz[:, 2]
    chroma = np.sqrt(az**2 + bz**2)
    angle = np.arctan2(bz, az)
    bins = np.linspace(-np.pi, np.pi, res+1)
    ridge_points = []
    for i in range(res):
        mask = (angle >= bins[i]) & (angle < bins[i+1])
        if np.sum(mask) == 0: 
            if len(ridge_points) > 0: ridge_points.append(ridge_points[-1])
            else: ridge_points.append(macadam_jz[0])
            continue
        slice_pts = macadam_jz[mask]
        idx_max = np.argmax(chroma[mask])
        ridge_points.append(slice_pts[idx_max])
        
    raw_ridge = np.array(ridge_points)
    # Smooth
    def smooth_channel(data, sigma=3):
        padded = np.concatenate([data[-20:], data, data[:20]])
        kernel = np.exp(-np.linspace(-5,5,21)**2 / (2*sigma**2))
        kernel /= kernel.sum()
        return np.convolve(padded, kernel, mode='same')[20:-20]

    return np.column_stack([smooth_channel(raw_ridge[:,0]), smooth_channel(raw_ridge[:,1]), smooth_channel(raw_ridge[:,2])])

# ==========================================
# 3. Optimization & Projections
# ==========================================

def get_gamut_surface(res=40):
    t = np.linspace(0, 1, res)
    g = np.meshgrid(t, t)
    faces = [np.stack([g[0], g[1], np.zeros_like(g[0])], -1), np.stack([g[0], g[1], np.ones_like(g[0])], -1),
             np.stack([g[0], np.zeros_like(g[0]), g[1]], -1), np.stack([g[0], np.ones_like(g[0]), g[1]], -1),
             np.stack([np.zeros_like(g[0]), g[0], g[1]], -1), np.stack([np.ones_like(g[0]), g[0], g[1]], -1)]
    pts = np.vstack([f.reshape(-1, 3) for f in faces])
    corners = np.array([[0,0,0], [1,0,0], [0,1,0], [0,0,1], [1,1,0], [1,0,1], [0,1,1], [1,1,1]])
    return np.vstack([pts, corners])

def fit_ellipsoid_anchored_solver(points, anchor_point):
    hull = ConvexHull(points)
    A = hull.equations[:, :3]; b_const = -hull.equations[:, 3]
    scale = 100.0; A_scaled = A / scale; anchor_scaled = anchor_point * scale
    c0 = np.mean(points, axis=0) * scale
    x0 = np.concatenate([0.5*c0 + 0.5*anchor_scaled, np.eye(3).flatten() * 0.05])
    def objective(x):
        M = x[3:].reshape(3, 3); sign, val = np.linalg.slogdet(M)
        return -val if sign > 0 else 1e9
    def constraints(x):
        d, M = x[:3], x[3:].reshape(3, 3)
        norm_AM = np.linalg.norm(A_scaled @ M, axis=1)
        hull_con = b_const - (A_scaled @ d + norm_AM)
        try: u = np.linalg.solve(M, anchor_scaled - d); anchor_con = 1.0 - np.linalg.norm(u)
        except: anchor_con = -1.0
        return np.concatenate([hull_con, [anchor_con]])
    res = opt.minimize(objective, x0, method='SLSQP', constraints={'type': 'ineq', 'fun': constraints}, options={'maxiter': 1000})
    return res.x[:3]/scale, res.x[3:].reshape(3, 3)/scale

def calculate_shadow_boundary(center, M, view_vector, res=72):
    w = np.linalg.solve(M, view_vector); w = w / np.linalg.norm(w)
    tmp = np.array([0.0, 1.0, 0.0]) if np.abs(w[1]) < 0.9 else np.array([0.0, 0.0, 1.0])
    s1 = np.cross(w, tmp); s1 /= np.linalg.norm(s1); s2 = np.cross(w, s1)
    theta = np.linspace(0, 2*np.pi, res)
    return center + (np.outer(np.cos(theta), s1) + np.outer(np.sin(theta), s2)) @ M.T

def inflate_to_max_gamut(center, M_init):
    scale = 1.0; step = 0.05; best_M = M_init.copy()
    for _ in range(50):
        test_M = M_init * (scale + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        scale += step; best_M = test_M
    step = 0.005; current_M = best_M
    for _ in range(20):
        test_M = current_M * (1.0 + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        current_M = test_M
    return current_M

def project_ridge_onto_ellipsoid(ridge_jz, center, M):
    vecs = ridge_jz - center
    M_inv = np.linalg.inv(M)
    u_vecs = vecs @ M_inv.T
    norms = np.linalg.norm(u_vecs, axis=1, keepdims=True)
    u_surface = u_vecs / norms
    return (u_surface @ M.T) + center

def project_points_to_srgb_surface_bsearch(points_jz, center_jz):
    projected = []
    for pt in points_jz:
        vec = pt - center_jz
        low = 0.0; high = 1.2; best_t = 0.0
        for _ in range(15):
            mid = (low + high) / 2.0
            test = center_jz + mid * vec
            rgb = xyz_to_srgb_raw(jzazbz_to_xyz(np.array([test])))
            if np.min(rgb) >= -0.001 and np.max(rgb) <= 1.001:
                best_t = mid; low = mid 
            else: high = mid 
        projected.append(center_jz + best_t * vec)
    return np.array(projected)

def shrink_ridge_into_gamut(ridge_jz, center_point):
    scale = 1.0; step = 0.01
    for _ in range(100):
        test_ridge = center_point + scale * (ridge_jz - center_point)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(test_ridge))
        if np.mean(np.min(rgb, axis=1) >= -0.005) > 0.98: 
             if np.mean(np.max(rgb, axis=1) <= 1.005) > 0.98: return test_ridge
        scale -= step
    return ridge_jz * 0.1

def reparameterize_curve(points, n_samples=256):
    dists = np.linalg.norm(points[1:] - points[:-1], axis=1)
    cum_dist = np.insert(np.cumsum(dists), 0, 0.0)
    target_dists = np.linspace(0, cum_dist[-1], n_samples)
    new_points = np.zeros((n_samples, 3))
    for i in range(3):
        new_points[:, i] = np.interp(target_dists, cum_dist, points[:, i])
    return new_points

def fourier_approx_curve(points, harmonics=2):
    """
    DFT Approximation of a 3D closed curve.
    Retains DC + first N harmonics.
    """
    N = len(points)
    new_cols = []
    for i in range(3):
        coeffs = np.fft.rfft(points[:,i])
        # Filter: keep only low freq
        filt = np.zeros_like(coeffs)
        filt[:harmonics+1] = coeffs[:harmonics+1]
        new_cols.append(np.fft.irfft(filt, n=N))
    return np.column_stack(new_cols)

# ==========================================
# 4. Omnipose Helpers
# ==========================================
def align_ring_red_to_green(ring_lin):
    n = len(ring_lin)
    red_ref = np.array([1.0, 0.0, 0.0])
    idx_r = np.argmin(np.linalg.norm(ring_lin - red_ref, axis=1))
    ring_aligned = np.roll(ring_lin, -idx_r, axis=0)
    green_ref = np.array([0.0, 1.0, 0.0])
    if np.linalg.norm(ring_aligned[(2*n)//3]-green_ref) < np.linalg.norm(ring_aligned[n//3]-green_ref):
        ring_aligned = ring_aligned[::-1]
    return ring_aligned

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    def build_disk(ring, center):
        y, x = np.mgrid[-1:1:256j, -1:1:256j]
        mag = np.sqrt(x*x + y*y)
        angle = np.mod(np.arctan2(y, x), 2*np.pi)
        n = ring.shape[0]
        hue_f = angle/(2*np.pi)*n
        idx0 = np.floor(hue_f).astype(int) % n
        idx1 = (idx0 + 1) % n
        t = hue_f - np.floor(hue_f)
        interp = (1-t[...,None])*ring[idx0] + t[...,None]*ring[idx1]
        rgb = (1-mag[...,None])*center + mag[...,None]*interp
        return np.clip(linear_to_srgb_np(rgb),0,1)

    disk = build_disk(rgb_ring_lin, center_lin)
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi)
    mag = np.clip(np.sqrt(u*u + v*v), 0, 1)
    n = rgb_ring_lin.shape[0]
    hue_f = angle/(2*np.pi)*n
    idx0 = np.floor(hue_f).astype(int) % n
    idx1 = (idx0 + 1) % n
    t = hue_f - np.floor(hue_f)
    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    r = mag[..., None]
    rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow),0,1)
    alpha = mag[...,None]
    white = np.ones_like(flow_rgb); black = np.zeros_like(flow_rgb)
    flow_white = alpha*flow_rgb + (1-alpha)*white
    flow_black = alpha*flow_rgb + (1-alpha)*black
    return disk, flow_rgb, flow_white, flow_black

# ==========================================
# 5. Main Execution
# ==========================================

def main():
    device_str = "cpu"
    dev = torch.device(device_str)
    
    print("1. Calculating Geometries...")
    surf_pts_opt = get_gamut_surface(40) 
    jz_gamut_opt = xyz_to_jzazbz(srgb_to_xyz(surf_pts_opt))
    centroid = np.mean(jz_gamut_opt, axis=0)
    green_vertex = xyz_to_jzazbz(srgb_to_xyz(np.array([[0.0, 1.0, 0.0]])))[0]
    anchor = centroid + 0.60 * (green_vertex - centroid)
    
    print("2. Fitting Maximal Ellipsoid...")
    center, M_solver = fit_ellipsoid_anchored_solver(jz_gamut_opt, anchor)
    M_max = inflate_to_max_gamut(center, M_solver)
    
    # Loop A: Ellipsoid Loop
    hoop = calculate_shadow_boundary(center, M_max, np.array([1.0, 0.0, 0.0]), res=72)
    rgb_hoop_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(hoop)),0,1)))
    
    # Loop B: Sinebow
    t = np.linspace(0, 1, 256, endpoint=False)
    sr = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0/3)); sg = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1/3)); sb = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2/3))
    rgb_sine_std = np.stack([sr, sg, sb], axis=1) 
    rgb_sine_lin = align_ring_red_to_green(srgb_to_linear_np(rgb_sine_std))
    
    # MacAdam Ridge
    print("3. Computing MacAdam Ridge...")
    macadam_xyz = compute_macadam_solid(res=200)
    macadam_jz = xyz_to_jzazbz(macadam_xyz)
    ridge_jz_highres = extract_macadam_ridge_smoothed(macadam_jz, res=360)
    
    # Loop C: Ridge -> Ellipsoid (Reparameterized)
    proj_ridge_jz_raw = project_ridge_onto_ellipsoid(ridge_jz_highres, center, M_max)
    proj_ridge_jz_uniform = reparameterize_curve(proj_ridge_jz_raw, n_samples=256)
    rgb_proj_ell_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(proj_ridge_jz_uniform)), 0, 1)))
    
    # Loop D: Ridge -> sRGB (Reparameterized)
    proj_srgb_jz_raw = project_points_to_srgb_surface_bsearch(ridge_jz_highres, centroid)
    proj_srgb_jz_uniform = reparameterize_curve(proj_srgb_jz_raw, n_samples=256)
    rgb_proj_srgb_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(proj_srgb_jz_uniform)), 0, 1)))
    
    # Loop E: Fourier (N=2) of Ridge->sRGB
    # Input: Reparameterized Ridge->sRGB. Output: Smoothed Loop.
    fourier_jz = fourier_approx_curve(proj_srgb_jz_uniform, harmonics=2)
    rgb_fourier_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(fourier_jz)), 0, 1)))
    
    # Loop F: Shrunk Ridge
    shrunk_ridge_jz_raw = shrink_ridge_into_gamut(ridge_jz_highres, centroid)
    shrunk_ridge_jz_uniform = reparameterize_curve(shrunk_ridge_jz_raw, n_samples=256)
    rgb_shrunk_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(shrunk_ridge_jz_uniform)), 0, 1)))

    print("4. Loading Omnipose Flows...")
    omnidir = Path(omnirefactor.__file__).resolve().parent.parent.parent
    basedir = omnidir / "docs" / "_static"
    name = "ecoli"
    ext = ".tif"

    try:
        masks = io.imread(str(basedir / f"{name}_labels{ext}"))
        slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows = omnirefactor.core.masks_to_flows(masks, device=device_str)[4].to(dev)
        center_lin = srgb_to_linear_np(np.array([0.5, 0.5, 0.5]))

        def gen_set(ring, rot_steps=0):
            idx = int(rot_steps * len(ring) / 72.0)
            r = np.roll(ring, idx, axis=0)
            return make_flow_images_for_ring(r, center_lin, flows)

        rows = []
        # 1
        d, f, w, b = gen_set(rgb_sine_lin, 0)
        rows.append(("Sinebow", rgb_sine_lin, d, b, w))
        # 2
        d, f, w, b = gen_set(rgb_hoop_lin, 0)
        rows.append(("Ellipsoid Loop", rgb_hoop_lin, d, b, w))
        # 3
        d, f, w, b = gen_set(rgb_proj_ell_lin, 0)
        rows.append(("Ridge->Ellipsoid", rgb_proj_ell_lin, d, b, w))
        # 4
        d, f, w, b = gen_set(rgb_proj_srgb_lin, 0)
        rows.append(("Ridge->sRGB", rgb_proj_srgb_lin, d, b, w))
        # 5 (Fourier)
        d, f, w, b = gen_set(rgb_fourier_lin, 0)
        rows.append(("Fourier (N=2)", rgb_fourier_lin, d, b, w))
        # 6
        d, f, w, b = gen_set(rgb_shrunk_lin, 0)
        rows.append(("Shrunk Ridge", rgb_shrunk_lin, d, b, w))

        print("5. Plotting 2D...")
        plt.style.use('dark_background')
        fig, axes = plt.subplots(6, 4, figsize=(16, 24))
        titles = ["Gradient", "Hue Disk", "Black (0°)", "White (0°)"]

        for i, (name, ring, disk, b, w) in enumerate(rows):
            ax_row = axes[i]
            disp_gradient = linear_to_srgb_np(ring)[None, ...]
            ax_row[0].imshow(disp_gradient, aspect='auto')
            ax_row[0].set_ylabel(name, fontsize=14, color='white', rotation=90, labelpad=20)
            ax_row[1].imshow(disk); ax_row[2].imshow(b); ax_row[3].imshow(w)
            for j, ax in enumerate(ax_row):
                if i==0: ax.set_title(titles[j], fontsize=12, color='#dddddd')
                ax.set_xticks([]); ax.set_yticks([])
        plt.tight_layout()
        plt.show()
        
        print("6. Generating 3D...")
        fig3d = go.Figure()
        max_jz = np.max(jz_gamut_opt[:,0]); sc = np.array([1.0/max_jz, 1.0, 1.0]); 
        def mc(a): return a[:,1], a[:,2], a[:,0]

        # Hull, Wireframe, Ellipsoid
        hull_macadam = ConvexHull(macadam_jz)
        mx, my, mz = mc(macadam_jz * sc)
        fig3d.add_trace(go.Mesh3d(x=mx, y=my, z=mz, i=hull_macadam.simplices[:,0], j=hull_macadam.simplices[:,1], k=hull_macadam.simplices[:,2], color='white', opacity=0.05, name='Visible Gamut'))

        corners = [[0,0,0],[1,0,0],[1,1,0],[0,1,0],[0,0,1],[1,0,1],[1,1,1],[0,1,1]]
        edges_idx = [[0,1],[1,2],[2,3],[3,0],[4,5],[5,6],[6,7],[7,4],[0,4],[1,5],[2,6],[3,7]]
        for s,e in edges_idx:
            l = xyz_to_jzazbz(srgb_to_xyz(np.linspace(corners[s],corners[e],25))) * sc
            x,y,z = mc(l)
            fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', line=dict(color='gray', width=2), showlegend=False))

        u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
        sph = np.stack([np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v)], axis=-1).reshape(-1, 3)
        ell = (sph @ M_max.T + center) * sc
        ex, ey, ez = mc(ell)
        fig3d.add_trace(go.Surface(x=ex.reshape(u.shape), y=ey.reshape(u.shape), z=ez.reshape(u.shape), colorscale=[(0,'#88aa88'),(1,'#88aa88')], opacity=0.2, showscale=False, name='Ellipsoid'))

        def plot_dots(pts, col, name):
             x,y,z = mc(pts * sc)
             rgb = linear_to_srgb_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(pts)), 0, 1))
             cols = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in rgb]
             fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='markers', marker=dict(color=cols, size=4), name=name))

        plot_dots(hoop, 'orange', 'Ellipsoid Loop')
        plot_dots(xyz_to_jzazbz(srgb_to_xyz(rgb_sine_std)), 'green', 'Sinebow Loop')
        plot_dots(proj_ridge_jz_uniform, 'cyan', 'Ridge->Ellipsoid')
        plot_dots(proj_srgb_jz_uniform, 'red', 'Ridge->sRGB')
        plot_dots(fourier_jz, 'yellow', 'Fourier (N=2)')
        plot_dots(shrunk_ridge_jz_uniform, 'magenta', 'Shrunk Ridge')

        rrx, rry, rrz = mc(ridge_jz_highres * sc)
        fig3d.add_trace(go.Scatter3d(x=rrx, y=rry, z=rrz, mode='lines', line=dict(color='white', width=3), name='MacAdam Ridge (Ref)'))

        fig3d.update_layout(template="plotly_dark", title="3D Jzazbz: Fourier Approx & Comparisons", 
                            scene=dict(xaxis_title='az', yaxis_title='bz', zaxis_title='Jz', aspectmode='manual', aspectratio=dict(x=1,y=1,z=1)))
        fig3d.show()

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
see of projection from macadam drige to gray gives a nicer loop on srgb or ellipse

In [ ]:
import numpy as np
import scipy.optimize as opt
from scipy.spatial import ConvexHull
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import omnirefactor.core
from skimage import io
import fastremap

# ==========================================
# 1. Color Math
# ==========================================
def srgb_to_xyz(rgb):
    mask = rgb > 0.04045
    linear = np.zeros_like(rgb)
    linear[mask] = ((rgb[mask] + 0.055) / 1.055) ** 2.4
    linear[~mask] = rgb[~mask] / 12.92
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    return linear @ M.T

def xyz_to_jzazbz(xyz):
    b, g, d, d0 = 1.15, 0.66, -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    lms = xyz @ M1.T
    lms_norm = (lms * 200) / 10000.0 
    lms_prime = np.sign(lms_norm) * (((c1 + c2 * (np.abs(lms_norm) ** n)) / (1 + c3 * (np.abs(lms_norm) ** n))) ** p)
    izazbz = lms_prime @ M2.T
    Jz = ((1 + d) * izazbz[:, 0]) / (1 + d * izazbz[:, 0]) - d0
    return np.column_stack((Jz, izazbz[:, 1], izazbz[:, 2]))

def jzazbz_to_xyz(jzazbz):
    Jz, az, bz = jzazbz[:,0], jzazbz[:,1], jzazbz[:,2]
    d, d0 = -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    Jp = Jz + d0; Iz = Jp / (1 + d - d * Jp)
    izazbz_vec = np.stack([Iz, az, bz], axis=1)
    lms_prime = izazbz_vec @ np.linalg.inv(M2).T
    sign_lms = np.sign(lms_prime)
    Y = np.abs(lms_prime) ** (1/p)
    num = Y - c1; den = c2 - Y * c3
    An = np.clip(num / den, 0, None) 
    lms_norm = sign_lms * (An ** (1/n))
    lms = (lms_norm * 10000.0) / 200.0
    return lms @ np.linalg.inv(M1).T

def xyz_to_srgb_raw(xyz):
    M_inv = np.linalg.inv(np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]]))
    linear = xyz @ M_inv.T
    srgb = np.zeros_like(linear)
    mask = linear > 0.0031308
    srgb[mask] = 1.055 * (linear[mask] ** (1/2.4)) - 0.055
    srgb[~mask] = 12.92 * linear[~mask]
    return srgb

def xyz_to_srgb(xyz): return np.clip(xyz_to_srgb_raw(xyz), 0, 1)
def srgb_to_linear_np(srgb): return np.where(srgb <= 0.04045, srgb / 12.92, ((srgb + 0.055) / 1.055) ** 2.4)
def linear_to_srgb_np(lin): 
    s = np.zeros_like(lin)
    mask = lin > 0.0031308
    s[mask] = 1.055 * (lin[mask]**(1/2.4)) - 0.055
    s[~mask] = 12.92 * lin[~mask]
    return np.clip(s, 0, 1)

# ==========================================
# 2. MacAdam Solid & Ridge
# ==========================================

def get_cmf_high_res(res=360):
    base_cmf = np.array([
        [0.0014,0.0000,0.0065], [0.0042,0.0001,0.0201], [0.0143,0.0004,0.0679], [0.0435,0.0012,0.2074],
        [0.1344,0.0040,0.6456], [0.2839,0.0116,1.3856], [0.3483,0.0230,1.7471], [0.3362,0.0380,1.7721],
        [0.2908,0.0600,1.6692], [0.1954,0.0910,1.2876], [0.0956,0.1390,0.8130], [0.0320,0.2080,0.4652],
        [0.0049,0.3230,0.2720], [0.0093,0.5030,0.1582], [0.0633,0.7100,0.0782], [0.1655,0.8620,0.0422],
        [0.2904,0.9540,0.0203], [0.4334,0.9950,0.0087], [0.5945,0.9950,0.0039], [0.7621,0.9520,0.0021],
        [0.9163,0.8700,0.0017], [1.0263,0.7570,0.0011], [1.0622,0.6310,0.0008], [1.0026,0.5030,0.0006],
        [0.8544,0.3810,0.0002], [0.6424,0.2650,0.0000], [0.4479,0.1750,0.0000], [0.2835,0.1070,0.0000],
        [0.1649,0.0610,0.0000], [0.0874,0.0320,0.0000], [0.0468,0.0170,0.0000], [0.0227,0.0082,0.0000],
        [0.0114,0.0041,0.0000], [0.0058,0.0021,0.0000], [0.0029,0.0010,0.0000], [0.0014,0.0005,0.0000]
    ])
    x = np.linspace(0, 1, len(base_cmf))
    xi = np.linspace(0, 1, res)
    interp_cmf = np.zeros((res, 3))
    for k in range(3):
        interp_cmf[:,k] = np.interp(xi, x, base_cmf[:,k])
    return interp_cmf

def compute_macadam_solid(res=150):
    cmf = get_cmf_high_res(360)
    n_lambda = len(cmf)
    cum_cmf = np.vstack([np.zeros(3), np.cumsum(cmf, axis=0)])
    white_xyz = cum_cmf[-1]
    
    optimal_colors = []
    indices = np.linspace(0, n_lambda, res).astype(int)
    
    for i in indices:
        for j in indices:
            if i == j: continue
            if j > i: xyz = cum_cmf[j] - cum_cmf[i]
            else:     xyz = cum_cmf[j] + (white_xyz - cum_cmf[i])
            optimal_colors.append(xyz)
    
    optimal_colors.append([0,0,0])
    optimal_colors.append(white_xyz)
    return np.array(optimal_colors) / white_xyz[1]

def extract_macadam_ridge_smoothed(macadam_jz, res=360):
    az = macadam_jz[:, 1]
    bz = macadam_jz[:, 2]
    chroma = np.sqrt(az**2 + bz**2)
    angle = np.arctan2(bz, az)
    bins = np.linspace(-np.pi, np.pi, res+1)
    ridge_points = []
    
    for i in range(res):
        mask = (angle >= bins[i]) & (angle < bins[i+1])
        if np.sum(mask) == 0: 
            if len(ridge_points) > 0: ridge_points.append(ridge_points[-1])
            else: ridge_points.append(macadam_jz[0])
            continue
        slice_pts = macadam_jz[mask]
        idx_max = np.argmax(chroma[mask])
        ridge_points.append(slice_pts[idx_max])
        
    raw_ridge = np.array(ridge_points)
    
    # Gaussian Smoothing
    def smooth_channel(data, sigma=5):
        padded = np.concatenate([data[-30:], data, data[:30]])
        kernel = np.exp(-np.linspace(-10,10,21)**2 / (2*sigma**2))
        kernel /= kernel.sum()
        return np.convolve(padded, kernel, mode='same')[30:-30]

    return np.column_stack([smooth_channel(raw_ridge[:,0]), smooth_channel(raw_ridge[:,1]), smooth_channel(raw_ridge[:,2])])

# ==========================================
# 3. Optimization & Projections
# ==========================================

def get_gamut_surface(res=40):
    t = np.linspace(0, 1, res)
    g = np.meshgrid(t, t)
    faces = [np.stack([g[0], g[1], np.zeros_like(g[0])], -1), np.stack([g[0], g[1], np.ones_like(g[0])], -1),
             np.stack([g[0], np.zeros_like(g[0]), g[1]], -1), np.stack([g[0], np.ones_like(g[0]), g[1]], -1),
             np.stack([np.zeros_like(g[0]), g[0], g[1]], -1), np.stack([np.ones_like(g[0]), g[0], g[1]], -1)]
    pts = np.vstack([f.reshape(-1, 3) for f in faces])
    corners = np.array([[0,0,0], [1,0,0], [0,1,0], [0,0,1], [1,1,0], [1,0,1], [0,1,1], [1,1,1]])
    return np.vstack([pts, corners])

def fit_ellipsoid_anchored_solver(points, anchor_point):
    hull = ConvexHull(points)
    A = hull.equations[:, :3]; b_const = -hull.equations[:, 3]
    scale = 100.0; A_scaled = A / scale; anchor_scaled = anchor_point * scale
    c0 = np.mean(points, axis=0) * scale
    x0 = np.concatenate([0.5*c0 + 0.5*anchor_scaled, np.eye(3).flatten() * 0.05])
    def objective(x):
        M = x[3:].reshape(3, 3); sign, val = np.linalg.slogdet(M)
        return -val if sign > 0 else 1e9
    def constraints(x):
        d, M = x[:3], x[3:].reshape(3, 3)
        norm_AM = np.linalg.norm(A_scaled @ M, axis=1)
        hull_con = b_const - (A_scaled @ d + norm_AM)
        try: u = np.linalg.solve(M, anchor_scaled - d); anchor_con = 1.0 - np.linalg.norm(u)
        except: anchor_con = -1.0
        return np.concatenate([hull_con, [anchor_con]])
    res = opt.minimize(objective, x0, method='SLSQP', constraints={'type': 'ineq', 'fun': constraints}, options={'maxiter': 1000})
    return res.x[:3]/scale, res.x[3:].reshape(3, 3)/scale

def calculate_shadow_boundary(center, M, view_vector, res=72):
    w = np.linalg.solve(M, view_vector); w = w / np.linalg.norm(w)
    tmp = np.array([0.0, 1.0, 0.0]) if np.abs(w[1]) < 0.9 else np.array([0.0, 0.0, 1.0])
    s1 = np.cross(w, tmp); s1 /= np.linalg.norm(s1); s2 = np.cross(w, s1)
    theta = np.linspace(0, 2*np.pi, res)
    return center + (np.outer(np.cos(theta), s1) + np.outer(np.sin(theta), s2)) @ M.T

def inflate_to_max_gamut(center, M_init):
    scale = 1.0; step = 0.05; best_M = M_init.copy()
    for _ in range(50):
        test_M = M_init * (scale + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        scale += step; best_M = test_M
    step = 0.005; current_M = best_M
    for _ in range(20):
        test_M = current_M * (1.0 + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        current_M = test_M
    return current_M

def project_points_to_srgb_surface_bsearch(points_jz, center_jz):
    projected = []
    for pt in points_jz:
        vec = pt - center_jz
        low = 0.0; high = 1.5; best_t = 0.0
        for _ in range(15):
            mid = (low + high) / 2.0
            test = center_jz + mid * vec
            rgb = xyz_to_srgb_raw(jzazbz_to_xyz(np.array([test])))
            if np.min(rgb) >= -0.001 and np.max(rgb) <= 1.001:
                best_t = mid; low = mid 
            else: high = mid 
        projected.append(center_jz + best_t * vec)
    return np.array(projected)

def reparameterize_curve(points, n_samples=256):
    dists = np.linalg.norm(points[1:] - points[:-1], axis=1)
    cum_dist = np.insert(np.cumsum(dists), 0, 0.0)
    target_dists = np.linspace(0, cum_dist[-1], n_samples)
    new_points = np.zeros((n_samples, 3))
    for i in range(3):
        new_points[:, i] = np.interp(target_dists, cum_dist, points[:, i])
    return new_points

def fourier_approx_curve(points, harmonics=3):
    N = len(points)
    new_cols = []
    for i in range(3):
        sig = points[:,i]
        coeffs = np.fft.rfft(sig)
        filt = np.zeros_like(coeffs)
        filt[:harmonics+1] = coeffs[:harmonics+1]
        new_cols.append(np.fft.irfft(filt, n=N))
    return np.column_stack(new_cols)

# ==========================================
# 4. Omnipose Helpers
# ==========================================
def align_ring_red_to_green(ring_lin):
    n = len(ring_lin)
    red_ref = np.array([1.0, 0.0, 0.0])
    idx_r = np.argmin(np.linalg.norm(ring_lin - red_ref, axis=1))
    ring_aligned = np.roll(ring_lin, -idx_r, axis=0)
    green_ref = np.array([0.0, 1.0, 0.0])
    if np.linalg.norm(ring_aligned[(2*n)//3]-green_ref) < np.linalg.norm(ring_aligned[n//3]-green_ref):
        ring_aligned = ring_aligned[::-1]
    return ring_aligned

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    def build_disk(ring, center):
        y, x = np.mgrid[-1:1:256j, -1:1:256j]
        mag = np.sqrt(x*x + y*y)
        angle = np.mod(np.arctan2(y, x), 2*np.pi)
        n = ring.shape[0]
        hue_f = angle/(2*np.pi)*n
        idx0 = np.floor(hue_f).astype(int) % n
        idx1 = (idx0 + 1) % n
        t = hue_f - np.floor(hue_f)
        interp = (1-t[...,None])*ring[idx0] + t[...,None]*ring[idx1]
        rgb = (1-mag[...,None])*center + mag[...,None]*interp
        return np.clip(linear_to_srgb_np(rgb),0,1)

    disk = build_disk(rgb_ring_lin, center_lin)
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi)
    mag = np.clip(np.sqrt(u*u + v*v), 0, 1)
    n = rgb_ring_lin.shape[0]
    hue_f = angle/(2*np.pi)*n
    idx0 = np.floor(hue_f).astype(int) % n
    idx1 = (idx0 + 1) % n
    t = hue_f - np.floor(hue_f)
    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    r = mag[..., None]
    rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow),0,1)
    alpha = mag[...,None]
    white = np.ones_like(flow_rgb); black = np.zeros_like(flow_rgb)
    flow_white = alpha*flow_rgb + (1-alpha)*white
    flow_black = alpha*flow_rgb + (1-alpha)*black
    return disk, flow_rgb, flow_white, flow_black

# ==========================================
# 5. Main Execution
# ==========================================

def main():
    N_HARMONICS = 3 
    
    # TOGGLE: Use center of Visible Gamut? Or Manual?
    USE_MANUAL_CENTER = False
    MANUAL_CENTER_RGB = np.array([[0.5, 0.5, 0.5]]) # "Middle Grey"

    device_str = "cpu"
    dev = torch.device(device_str)
    
    print("1. Calculating Geometries...")
    surf_pts_opt = get_gamut_surface(40) 
    jz_gamut_opt = xyz_to_jzazbz(srgb_to_xyz(surf_pts_opt))
    centroid_srgb = np.mean(jz_gamut_opt, axis=0)
    green_vertex = xyz_to_jzazbz(srgb_to_xyz(np.array([[0.0, 1.0, 0.0]])))[0]
    anchor = centroid_srgb + 0.60 * (green_vertex - centroid_srgb)
    
    # 2. Ellipsoid
    print("2. Fitting Maximal Ellipsoid...")
    center, M_solver = fit_ellipsoid_anchored_solver(jz_gamut_opt, anchor)
    M_max = inflate_to_max_gamut(center, M_solver)
    hoop = calculate_shadow_boundary(center, M_max, np.array([1.0, 0.0, 0.0]), res=72)
    rgb_hoop_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(hoop)),0,1)))
    
    # 3. Sinebow
    t = np.linspace(0, 1, 256, endpoint=False)
    sr = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0/3)); sg = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1/3)); sb = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2/3))
    rgb_sine_std = np.stack([sr, sg, sb], axis=1)
    rgb_sine_lin = align_ring_red_to_green(srgb_to_linear_np(rgb_sine_std))
    
    # 4. MacAdam Ridge & Visible Center
    print("3. Computing MacAdam Ridge & Centroid...")
    macadam_xyz = compute_macadam_solid(res=200)
    macadam_jz = xyz_to_jzazbz(macadam_xyz)
    
    # Center Calculation
    centroid_macadam = np.mean(macadam_jz, axis=0)
    
    if USE_MANUAL_CENTER:
        print("   Using Manual Center for Projection.")
        projection_center = xyz_to_jzazbz(srgb_to_xyz(MANUAL_CENTER_RGB))[0]
    else:
        print("   Using MacAdam Centroid for Projection.")
        projection_center = centroid_macadam
    
    ridge_jz_highres = extract_macadam_ridge_smoothed(macadam_jz, res=360)
    
    # Ridge -> sRGB (Conical Projection towards chosen Center)
    proj_srgb_jz_raw = project_points_to_srgb_surface_bsearch(ridge_jz_highres, projection_center)
    proj_srgb_jz_uniform = reparameterize_curve(proj_srgb_jz_raw, n_samples=256)
    # Fourier Smooth
    fourier_jz = fourier_approx_curve(proj_srgb_jz_uniform, harmonics=N_HARMONICS)
    rgb_fourier_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(fourier_jz)), 0, 1)))

    print("4. Loading Omnipose Flows...")
    omnidir = Path(omnirefactor.__file__).resolve().parent.parent.parent
    basedir = omnidir / "docs" / "_static"
    name = "ecoli"
    ext = ".tif"

    try:
        masks = io.imread(str(basedir / f"{name}_labels{ext}"))
        slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows = omnirefactor.core.masks_to_flows(masks, device=device_str)[4].to(dev)
        center_lin = srgb_to_linear_np(np.array([0.5, 0.5, 0.5]))

        def gen_set(ring, rot_steps=0):
            idx = int(rot_steps * len(ring) / 72.0)
            r = np.roll(ring, idx, axis=0)
            return make_flow_images_for_ring(r, center_lin, flows)

        rows = []
        d, f, w, b = gen_set(rgb_sine_lin, 0)
        rows.append(("Sinebow", rgb_sine_lin, d, b, w))
        
        d, f, w, b = gen_set(rgb_hoop_lin, 0)
        rows.append(("Max Ellipse", rgb_hoop_lin, d, b, w))
        
        d, f, w, b = gen_set(rgb_fourier_lin, 0)
        rows.append((f"Projected (N={N_HARMONICS})", rgb_fourier_lin, d, b, w))

        print("5. Plotting 2D...")
        plt.style.use('dark_background')
        fig, axes = plt.subplots(3, 4, figsize=(16, 15))
        titles = ["Gradient", "Hue Disk", "Black (0°)", "White (0°)"]

        for i, (name, ring, disk, b, w) in enumerate(rows):
            ax_row = axes[i]
            ax_row[0].imshow(linear_to_srgb_np(ring)[None, ...], aspect='auto')
            ax_row[0].set_ylabel(name, fontsize=14, color='white', rotation=90, labelpad=20)
            ax_row[1].imshow(disk); ax_row[2].imshow(b); ax_row[3].imshow(w)
            for j, ax in enumerate(ax_row):
                if i==0: ax.set_title(titles[j], fontsize=12, color='#dddddd')
                ax.set_xticks([]); ax.set_yticks([])
        plt.tight_layout()
        plt.show()
        
        print("6. Generating 3D...")
        fig3d = go.Figure()
        max_jz = np.max(jz_gamut_opt[:,0]); sc = np.array([1.0/max_jz, 1.0, 1.0]); 
        def mc(a): return a[:,1], a[:,2], a[:,0]

        # MacAdam Hull
        hull_macadam = ConvexHull(macadam_jz)
        mx, my, mz = mc(macadam_jz * sc)
        
        fig3d.add_trace(go.Mesh3d(x=mx, y=my, z=mz, i=hull_macadam.simplices[:,0], j=hull_macadam.simplices[:,1], k=hull_macadam.simplices[:,2], color='white', opacity=0.05, name='Visible Gamut'))

        # sRGB Edges
        corners = [[0,0,0],[1,0,0],[1,1,0],[0,1,0],[0,0,1],[1,0,1],[1,1,1],[0,1,1]]
        edges_idx = [[0,1],[1,2],[2,3],[3,0],[4,5],[5,6],[6,7],[7,4],[0,4],[1,5],[2,6],[3,7]]
        for s,e in edges_idx:
            l = xyz_to_jzazbz(srgb_to_xyz(np.linspace(corners[s],corners[e],25))) * sc
            x,y,z = mc(l)
            fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', line=dict(color='gray', width=2), showlegend=False))

        # Ellipsoid
        u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
        sph = np.stack([np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v)], axis=-1).reshape(-1, 3)
        ell = (sph @ M_max.T + center) * sc
        ex, ey, ez = mc(ell)
        fig3d.add_trace(go.Surface(x=ex.reshape(u.shape), y=ey.reshape(u.shape), z=ez.reshape(u.shape), colorscale=[(0,'#88aa88'),(1,'#88aa88')], opacity=0.2, showscale=False, name='Ellipsoid'))

        def plot_dots(pts, col, name):
             x,y,z = mc(pts * sc)
             rgb = linear_to_srgb_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(pts)), 0, 1))
             cols = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in rgb]
             fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='markers', marker=dict(color=cols, size=4), name=name))

        plot_dots(hoop, 'orange', 'Ellipsoid Loop')
        plot_dots(xyz_to_jzazbz(srgb_to_xyz(rgb_sine_std)), 'green', 'Sinebow Loop')
        plot_dots(fourier_jz, 'yellow', f'Projected (N={N_HARMONICS})')

        # Centers
        cx, cy, cz = mc(np.array([centroid_srgb, projection_center]) * sc)
        fig3d.add_trace(go.Scatter3d(x=cx, y=cy, z=cz, mode='markers+text', text=["sRGB Center", "Proj Center"], 
                                     marker=dict(color=['white', 'red'], size=5)))

        rrx, rry, rrz = mc(ridge_jz_highres * sc)
        fig3d.add_trace(go.Scatter3d(x=rrx, y=rry, z=rrz, mode='lines', line=dict(color='white', width=3), name='MacAdam Ridge (Ref)'))

        fig3d.update_layout(template="plotly_dark", title=f"3D Jzazbz: Conical Projection (Center: {'Manual' if USE_MANUAL_CENTER else 'MacAdam'})", 
                            scene=dict(xaxis_title='az', yaxis_title='bz', zaxis_title='Jz', aspectmode='manual', aspectratio=dict(x=1,y=1,z=1)))
        fig3d.show()

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import scipy.optimize as opt
from scipy.spatial import ConvexHull
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import omnirefactor.core
from skimage import io
import fastremap

# ==========================================
# 1. Color Math
# ==========================================
def srgb_to_xyz(rgb):
    mask = rgb > 0.04045
    linear = np.zeros_like(rgb)
    linear[mask] = ((rgb[mask] + 0.055) / 1.055) ** 2.4
    linear[~mask] = rgb[~mask] / 12.92
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    return linear @ M.T

def xyz_to_jzazbz(xyz):
    b, g, d, d0 = 1.15, 0.66, -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    lms = xyz @ M1.T
    lms_norm = (lms * 200) / 10000.0 
    lms_prime = np.sign(lms_norm) * (((c1 + c2 * (np.abs(lms_norm) ** n)) / (1 + c3 * (np.abs(lms_norm) ** n))) ** p)
    izazbz = lms_prime @ M2.T
    Jz = ((1 + d) * izazbz[:, 0]) / (1 + d * izazbz[:, 0]) - d0
    return np.column_stack((Jz, izazbz[:, 1], izazbz[:, 2]))

def jzazbz_to_xyz(jzazbz):
    Jz, az, bz = jzazbz[:,0], jzazbz[:,1], jzazbz[:,2]
    d, d0 = -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    Jp = Jz + d0; Iz = Jp / (1 + d - d * Jp)
    izazbz_vec = np.stack([Iz, az, bz], axis=1)
    lms_prime = izazbz_vec @ np.linalg.inv(M2).T
    sign_lms = np.sign(lms_prime)
    Y = np.abs(lms_prime) ** (1/p)
    num = Y - c1; den = c2 - Y * c3
    An = np.clip(num / den, 0, None) 
    lms_norm = sign_lms * (An ** (1/n))
    lms = (lms_norm * 10000.0) / 200.0
    return lms @ np.linalg.inv(M1).T

def xyz_to_srgb_raw(xyz):
    M_inv = np.linalg.inv(np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]]))
    linear = xyz @ M_inv.T
    srgb = np.zeros_like(linear)
    mask = linear > 0.0031308
    srgb[mask] = 1.055 * (linear[mask] ** (1/2.4)) - 0.055
    srgb[~mask] = 12.92 * linear[~mask]
    return srgb

def xyz_to_srgb(xyz): return np.clip(xyz_to_srgb_raw(xyz), 0, 1)
def srgb_to_linear_np(srgb): return np.where(srgb <= 0.04045, srgb / 12.92, ((srgb + 0.055) / 1.055) ** 2.4)
def linear_to_srgb_np(lin): 
    s = np.zeros_like(lin)
    mask = lin > 0.0031308
    s[mask] = 1.055 * (lin[mask]**(1/2.4)) - 0.055
    s[~mask] = 12.92 * lin[~mask]
    return np.clip(s, 0, 1)

# ==========================================
# 2. High Res Geometry
# ==========================================

def get_cmf_high_res(res=360):
    base_cmf = np.array([
        [0.0014,0.0000,0.0065], [0.0042,0.0001,0.0201], [0.0143,0.0004,0.0679], [0.0435,0.0012,0.2074],
        [0.1344,0.0040,0.6456], [0.2839,0.0116,1.3856], [0.3483,0.0230,1.7471], [0.3362,0.0380,1.7721],
        [0.2908,0.0600,1.6692], [0.1954,0.0910,1.2876], [0.0956,0.1390,0.8130], [0.0320,0.2080,0.4652],
        [0.0049,0.3230,0.2720], [0.0093,0.5030,0.1582], [0.0633,0.7100,0.0782], [0.1655,0.8620,0.0422],
        [0.2904,0.9540,0.0203], [0.4334,0.9950,0.0087], [0.5945,0.9950,0.0039], [0.7621,0.9520,0.0021],
        [0.9163,0.8700,0.0017], [1.0263,0.7570,0.0011], [1.0622,0.6310,0.0008], [1.0026,0.5030,0.0006],
        [0.8544,0.3810,0.0002], [0.6424,0.2650,0.0000], [0.4479,0.1750,0.0000], [0.2835,0.1070,0.0000],
        [0.1649,0.0610,0.0000], [0.0874,0.0320,0.0000], [0.0468,0.0170,0.0000], [0.0227,0.0082,0.0000],
        [0.0114,0.0041,0.0000], [0.0058,0.0021,0.0000], [0.0029,0.0010,0.0000], [0.0014,0.0005,0.0000]
    ])
    x = np.linspace(0, 1, len(base_cmf))
    xi = np.linspace(0, 1, res)
    interp_cmf = np.zeros((res, 3))
    for k in range(3):
        interp_cmf[:,k] = np.interp(xi, x, base_cmf[:,k])
    return interp_cmf

def compute_macadam_solid(res=400): # DOUBLED RESOLUTION
    cmf = get_cmf_high_res(500)
    n_lambda = len(cmf)
    cum_cmf = np.vstack([np.zeros(3), np.cumsum(cmf, axis=0)])
    white_xyz = cum_cmf[-1]
    optimal_colors = []
    indices = np.linspace(0, n_lambda, res).astype(int)
    for i in indices:
        for j in indices:
            if i == j: continue
            if j > i: xyz = cum_cmf[j] - cum_cmf[i]
            else:     xyz = cum_cmf[j] + (white_xyz - cum_cmf[i])
            optimal_colors.append(xyz)
    optimal_colors.append([0,0,0])
    optimal_colors.append(white_xyz)
    return np.array(optimal_colors) / white_xyz[1]

def extract_macadam_ridge_smoothed(macadam_jz, res=720): # DOUBLED RESOLUTION
    az = macadam_jz[:, 1]
    bz = macadam_jz[:, 2]
    chroma = np.sqrt(az**2 + bz**2)
    angle = np.arctan2(bz, az)
    
    bins = np.linspace(-np.pi, np.pi, res+1)
    ridge_points = []
    
    for i in range(res):
        mask = (angle >= bins[i]) & (angle < bins[i+1])
        if np.sum(mask) == 0: 
            if len(ridge_points) > 0: ridge_points.append(ridge_points[-1])
            else: ridge_points.append(macadam_jz[0])
            continue
        slice_pts = macadam_jz[mask]
        idx_max = np.argmax(chroma[mask])
        ridge_points.append(slice_pts[idx_max])
        
    raw_ridge = np.array(ridge_points)
    return smooth_curve_circular(raw_ridge, sigma=5)

def smooth_curve_circular(points, sigma=5):
    # Gaussian smoothing for closed loops
    # Points: (N, 3)
    N = len(points)
    pad = int(3*sigma)
    padded = np.vstack([points[-pad:], points, points[:pad]])
    
    # Kernel
    x = np.arange(-pad, pad+1)
    kernel = np.exp(-x**2 / (2*sigma**2))
    kernel /= kernel.sum()
    
    smoothed = np.zeros_like(points)
    for i in range(3):
        smoothed[:, i] = np.convolve(padded[:, i], kernel, mode='same')[pad:-pad]
    return smoothed

# ==========================================
# 3. Optimization & Projections
# ==========================================

def get_gamut_surface(res=60): # Higher res for intersection checks
    t = np.linspace(0, 1, res)
    g = np.meshgrid(t, t)
    faces = [np.stack([g[0], g[1], np.zeros_like(g[0])], -1), np.stack([g[0], g[1], np.ones_like(g[0])], -1),
             np.stack([g[0], np.zeros_like(g[0]), g[1]], -1), np.stack([g[0], np.ones_like(g[0]), g[1]], -1),
             np.stack([np.zeros_like(g[0]), g[0], g[1]], -1), np.stack([np.ones_like(g[0]), g[0], g[1]], -1)]
    pts = np.vstack([f.reshape(-1, 3) for f in faces])
    corners = np.array([[0,0,0], [1,0,0], [0,1,0], [0,0,1], [1,1,0], [1,0,1], [0,1,1], [1,1,1]])
    return np.vstack([pts, corners])

def fit_ellipsoid_anchored_solver(points, anchor_point):
    hull = ConvexHull(points)
    A = hull.equations[:, :3]; b_const = -hull.equations[:, 3]
    scale = 100.0; A_scaled = A / scale; anchor_scaled = anchor_point * scale
    c0 = np.mean(points, axis=0) * scale
    x0 = np.concatenate([0.5*c0 + 0.5*anchor_scaled, np.eye(3).flatten() * 0.05])
    def objective(x):
        M = x[3:].reshape(3, 3); sign, val = np.linalg.slogdet(M)
        return -val if sign > 0 else 1e9
    def constraints(x):
        d, M = x[:3], x[3:].reshape(3, 3)
        norm_AM = np.linalg.norm(A_scaled @ M, axis=1)
        hull_con = b_const - (A_scaled @ d + norm_AM)
        try: u = np.linalg.solve(M, anchor_scaled - d); anchor_con = 1.0 - np.linalg.norm(u)
        except: anchor_con = -1.0
        return np.concatenate([hull_con, [anchor_con]])
    res = opt.minimize(objective, x0, method='SLSQP', constraints={'type': 'ineq', 'fun': constraints}, options={'maxiter': 1000})
    return res.x[:3]/scale, res.x[3:].reshape(3, 3)/scale, hull

def calculate_shadow_boundary(center, M, view_vector, res=72):
    w = np.linalg.solve(M, view_vector); w = w / np.linalg.norm(w)
    tmp = np.array([0.0, 1.0, 0.0]) if np.abs(w[1]) < 0.9 else np.array([0.0, 0.0, 1.0])
    s1 = np.cross(w, tmp); s1 /= np.linalg.norm(s1); s2 = np.cross(w, s1)
    theta = np.linspace(0, 2*np.pi, res)
    return center + (np.outer(np.cos(theta), s1) + np.outer(np.sin(theta), s2)) @ M.T

def inflate_to_max_gamut(center, M_init):
    scale = 1.0; step = 0.05; best_M = M_init.copy()
    for _ in range(50):
        test_M = M_init * (scale + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        scale += step; best_M = test_M
    step = 0.005; current_M = best_M
    for _ in range(20):
        test_M = current_M * (1.0 + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        current_M = test_M
    return current_M

def project_points_to_srgb_surface_conical(points_jz, center_jz):
    projected = []
    for pt in points_jz:
        vec = pt - center_jz
        low = 0.0; high = 1.5; best_t = 0.0
        for _ in range(15):
            mid = (low + high) / 2.0
            test = center_jz + mid * vec
            rgb = xyz_to_srgb_raw(jzazbz_to_xyz(np.array([test])))
            if np.min(rgb) >= -0.001 and np.max(rgb) <= 1.001:
                best_t = mid; low = mid 
            else: high = mid 
        projected.append(center_jz + best_t * vec)
    return np.array(projected)

def reparameterize_curve(points, n_samples=256):
    dists = np.linalg.norm(points[1:] - points[:-1], axis=1)
    cum_dist = np.insert(np.cumsum(dists), 0, 0.0)
    target_dists = np.linspace(0, cum_dist[-1], n_samples)
    new_points = np.zeros((n_samples, 3))
    for i in range(3):
        new_points[:, i] = np.interp(target_dists, cum_dist, points[:, i])
    return new_points

def fourier_approx_curve(points, harmonics=3):
    N = len(points)
    new_cols = []
    for i in range(3):
        sig = points[:,i]
        coeffs = np.fft.rfft(sig)
        filt = np.zeros_like(coeffs)
        filt[:harmonics+1] = coeffs[:harmonics+1]
        new_cols.append(np.fft.irfft(filt, n=N))
    return np.column_stack(new_cols)

# ==========================================
# 4. Omnipose Helpers
# ==========================================
def align_ring_red_to_green(ring_lin):
    n = len(ring_lin)
    red_ref = np.array([1.0, 0.0, 0.0])
    idx_r = np.argmin(np.linalg.norm(ring_lin - red_ref, axis=1))
    ring_aligned = np.roll(ring_lin, -idx_r, axis=0)
    green_ref = np.array([0.0, 1.0, 0.0])
    if np.linalg.norm(ring_aligned[(2*n)//3]-green_ref) < np.linalg.norm(ring_aligned[n//3]-green_ref):
        ring_aligned = ring_aligned[::-1]
    return ring_aligned

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    def build_disk(ring, center):
        y, x = np.mgrid[-1:1:256j, -1:1:256j]
        mag = np.sqrt(x*x + y*y)
        angle = np.mod(np.arctan2(y, x), 2*np.pi)
        n = ring.shape[0]
        hue_f = angle/(2*np.pi)*n
        idx0 = np.floor(hue_f).astype(int) % n
        idx1 = (idx0 + 1) % n
        t = hue_f - np.floor(hue_f)
        interp = (1-t[...,None])*ring[idx0] + t[...,None]*ring[idx1]
        rgb = (1-mag[...,None])*center + mag[...,None]*interp
        return np.clip(linear_to_srgb_np(rgb),0,1)

    disk = build_disk(rgb_ring_lin, center_lin)
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi)
    mag = np.clip(np.sqrt(u*u + v*v), 0, 1)
    n = rgb_ring_lin.shape[0]
    hue_f = angle/(2*np.pi)*n
    idx0 = np.floor(hue_f).astype(int) % n
    idx1 = (idx0 + 1) % n
    t = hue_f - np.floor(hue_f)
    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    r = mag[..., None]
    rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow),0,1)
    alpha = mag[...,None]
    white = np.ones_like(flow_rgb); black = np.zeros_like(flow_rgb)
    flow_white = alpha*flow_rgb + (1-alpha)*white
    flow_black = alpha*flow_rgb + (1-alpha)*black
    return disk, flow_rgb, flow_white, flow_black

# ==========================================
# 5. Main Execution
# ==========================================

def main():
    N_HARMONICS = 3 
    device_str = "cpu"
    dev = torch.device(device_str)
    
    print("1. Calculating Geometries (High Res)...")
    surf_pts_opt = get_gamut_surface(40) 
    jz_gamut_opt = xyz_to_jzazbz(srgb_to_xyz(surf_pts_opt))
    centroid = np.mean(jz_gamut_opt, axis=0)
    green_vertex = xyz_to_jzazbz(srgb_to_xyz(np.array([[0.0, 1.0, 0.0]])))[0]
    anchor = centroid + 0.60 * (green_vertex - centroid)
    
    print("2. Fitting Ellipsoid...")
    center, M_solver, hull = fit_ellipsoid_anchored_solver(jz_gamut_opt, anchor)
    M_max = inflate_to_max_gamut(center, M_solver)
    hoop = calculate_shadow_boundary(center, M_max, np.array([1.0, 0.0, 0.0]), res=72)
    rgb_hoop_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(hoop)),0,1)))
    
    t = np.linspace(0, 1, 256, endpoint=False)
    sr = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0/3)); sg = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1/3)); sb = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2/3))
    rgb_sine_std = np.stack([sr, sg, sb], axis=1)
    rgb_sine_lin = align_ring_red_to_green(srgb_to_linear_np(rgb_sine_std))
    
    print("3. Computing Ridge & Projections...")
    # Double Resolution
    macadam_xyz = compute_macadam_solid(res=400) 
    macadam_jz = xyz_to_jzazbz(macadam_xyz)
    ridge_jz_highres = extract_macadam_ridge_smoothed(macadam_jz, res=720) # Double Res
    
    # Ridge -> sRGB
    proj_srgb_jz_raw = project_points_to_srgb_surface_conical(ridge_jz_highres, centroid)
    
    # Apply Smoothing directly to projected curve (Round off the blue corner)
    proj_srgb_jz_smooth = smooth_curve_circular(proj_srgb_jz_raw, sigma=5)
    
    # Reparameterize
    proj_srgb_jz_uniform = reparameterize_curve(proj_srgb_jz_smooth, n_samples=256)
    rgb_proj_srgb_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(proj_srgb_jz_uniform)), 0, 1)))
    
    # Fourier Version
    fourier_jz = fourier_approx_curve(proj_srgb_jz_uniform, harmonics=N_HARMONICS)
    rgb_fourier_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(fourier_jz)), 0, 1)))

    print("4. Loading Omnipose Flows...")
    omnidir = Path(omnirefactor.__file__).resolve().parent.parent.parent
    basedir = omnidir / "docs" / "_static"
    name = "ecoli"
    ext = ".tif"

    try:
        masks = io.imread(str(basedir / f"{name}_labels{ext}"))
        slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows = omnirefactor.core.masks_to_flows(masks, device=device_str)[4].to(dev)
        center_lin = srgb_to_linear_np(np.array([0.5, 0.5, 0.5]))

        def gen_set(ring, rot_steps=0):
            idx = int(rot_steps * len(ring) / 72.0)
            r = np.roll(ring, idx, axis=0)
            return make_flow_images_for_ring(r, center_lin, flows)

        rows = []
        d, f, w, b = gen_set(rgb_sine_lin, 0)
        rows.append(("Sinebow", rgb_sine_lin, d, b, w))
        
        d, f, w, b = gen_set(rgb_hoop_lin, 0)
        rows.append(("Max Ellipse", rgb_hoop_lin, d, b, w))
        
        d, f, w, b = gen_set(rgb_proj_srgb_lin, 0)
        rows.append(("Ridge->sRGB", rgb_proj_srgb_lin, d, b, w))
        
        d, f, w, b = gen_set(rgb_fourier_lin, 0)
        rows.append((f"Fourier (N={N_HARMONICS})", rgb_fourier_lin, d, b, w))

        print("5. Plotting 2D...")
        plt.style.use('dark_background')
        fig, axes = plt.subplots(4, 4, figsize=(16, 20))
        for i, (name, ring, disk, b, w) in enumerate(rows):
            ax_row = axes[i]
            ax_row[0].imshow(linear_to_srgb_np(ring)[None, ...], aspect='auto')
            ax_row[0].set_ylabel(name, fontsize=14, color='white', rotation=90, labelpad=20)
            ax_row[1].imshow(disk); ax_row[2].imshow(b); ax_row[3].imshow(w)
            for j, ax in enumerate(ax_row): ax.set_xticks([]); ax.set_yticks([])
        plt.tight_layout()
        plt.show()
        
        print("6. Generating 3D...")
        fig3d = go.Figure()
        max_jz = np.max(jz_gamut_opt[:,0]); sc = np.array([1.0/max_jz, 1.0, 1.0]); 
        def mc(a): return a[:,1], a[:,2], a[:,0]

        hull_macadam = ConvexHull(macadam_jz)
        mx, my, mz = mc(macadam_jz * sc)
        fig3d.add_trace(go.Mesh3d(x=mx, y=my, z=mz, i=hull_macadam.simplices[:,0], j=hull_macadam.simplices[:,1], k=hull_macadam.simplices[:,2], color='white', opacity=0.05, name='Visible Gamut'))

        corners = [[0,0,0],[1,0,0],[1,1,0],[0,1,0],[0,0,1],[1,0,1],[1,1,1],[0,1,1]]
        edges_idx = [[0,1],[1,2],[2,3],[3,0],[4,5],[5,6],[6,7],[7,4],[0,4],[1,5],[2,6],[3,7]]
        for s,e in edges_idx:
            l = xyz_to_jzazbz(srgb_to_xyz(np.linspace(corners[s],corners[e],25))) * sc
            x,y,z = mc(l)
            fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', line=dict(color='gray', width=2), showlegend=False))

        def plot_dots(pts, col, name):
             x,y,z = mc(pts * sc)
             rgb = linear_to_srgb_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(pts)), 0, 1))
             cols = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in rgb]
             fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='markers', marker=dict(color=cols, size=4), name=name))

        plot_dots(hoop, 'orange', 'Ellipsoid Loop')
        plot_dots(xyz_to_jzazbz(srgb_to_xyz(rgb_sine_std)), 'green', 'Sinebow Loop')
        plot_dots(proj_srgb_jz_uniform, 'red', 'Ridge->sRGB')
        plot_dots(fourier_jz, 'yellow', f'Fourier (N={N_HARMONICS})')

        fig3d.update_layout(template="plotly_dark", title="3D Jzazbz: Ridge vs Fourier Comparison", 
                            scene=dict(xaxis_title='az', yaxis_title='bz', zaxis_title='Jz', aspectmode='manual', aspectratio=dict(x=1,y=1,z=1)))
        fig3d.show()

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
chrink macadam until each part of the srgb gamut has been touched

In [ ]:
import numpy as np
import scipy.optimize as opt
from scipy.spatial import ConvexHull
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import omnirefactor.core
from skimage import io
import fastremap

# ==========================================
# 1. Color Math
# ==========================================
def srgb_to_xyz(rgb):
    mask = rgb > 0.04045
    linear = np.zeros_like(rgb)
    linear[mask] = ((rgb[mask] + 0.055) / 1.055) ** 2.4
    linear[~mask] = rgb[~mask] / 12.92
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    return linear @ M.T

def xyz_to_jzazbz(xyz):
    b, g, d, d0 = 1.15, 0.66, -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    lms = xyz @ M1.T
    # sRGB white (1.0) maps to 200 nits
    lms_norm = (lms * 200) / 10000.0 
    lms_prime = np.sign(lms_norm) * (((c1 + c2 * (np.abs(lms_norm) ** n)) / (1 + c3 * (np.abs(lms_norm) ** n))) ** p)
    izazbz = lms_prime @ M2.T
    Jz = ((1 + d) * izazbz[:, 0]) / (1 + d * izazbz[:, 0]) - d0
    return np.column_stack((Jz, izazbz[:, 1], izazbz[:, 2]))

def jzazbz_to_xyz(jzazbz):
    Jz, az, bz = jzazbz[:,0], jzazbz[:,1], jzazbz[:,2]
    d, d0 = -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    Jp = Jz + d0; Iz = Jp / (1 + d - d * Jp)
    izazbz_vec = np.stack([Iz, az, bz], axis=1)
    lms_prime = izazbz_vec @ np.linalg.inv(M2).T
    sign_lms = np.sign(lms_prime)
    Y = np.abs(lms_prime) ** (1/p)
    num = Y - c1; den = c2 - Y * c3
    An = np.clip(num / den, 0, None) 
    lms_norm = sign_lms * (An ** (1/n))
    lms = (lms_norm * 10000.0) / 200.0
    return lms @ np.linalg.inv(M1).T

def xyz_to_srgb_raw(xyz):
    M_inv = np.linalg.inv(np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]]))
    linear = xyz @ M_inv.T
    srgb = np.zeros_like(linear)
    mask = linear > 0.0031308
    srgb[mask] = 1.055 * (linear[mask] ** (1/2.4)) - 0.055
    srgb[~mask] = 12.92 * linear[~mask]
    return srgb

def xyz_to_srgb(xyz): return np.clip(xyz_to_srgb_raw(xyz), 0, 1)
def srgb_to_linear_np(srgb): return np.where(srgb <= 0.04045, srgb / 12.92, ((srgb + 0.055) / 1.055) ** 2.4)
def linear_to_srgb_np(lin): 
    s = np.zeros_like(lin)
    mask = lin > 0.0031308
    s[mask] = 1.055 * (lin[mask]**(1/2.4)) - 0.055
    s[~mask] = 12.92 * lin[~mask]
    return np.clip(s, 0, 1)

# ==========================================
# 2. MacAdam Solid & Ridge
# ==========================================

def get_cmf_high_res(res=360):
    base_cmf = np.array([
        [0.0014,0.0000,0.0065], [0.0042,0.0001,0.0201], [0.0143,0.0004,0.0679], [0.0435,0.0012,0.2074],
        [0.1344,0.0040,0.6456], [0.2839,0.0116,1.3856], [0.3483,0.0230,1.7471], [0.3362,0.0380,1.7721],
        [0.2908,0.0600,1.6692], [0.1954,0.0910,1.2876], [0.0956,0.1390,0.8130], [0.0320,0.2080,0.4652],
        [0.0049,0.3230,0.2720], [0.0093,0.5030,0.1582], [0.0633,0.7100,0.0782], [0.1655,0.8620,0.0422],
        [0.2904,0.9540,0.0203], [0.4334,0.9950,0.0087], [0.5945,0.9950,0.0039], [0.7621,0.9520,0.0021],
        [0.9163,0.8700,0.0017], [1.0263,0.7570,0.0011], [1.0622,0.6310,0.0008], [1.0026,0.5030,0.0006],
        [0.8544,0.3810,0.0002], [0.6424,0.2650,0.0000], [0.4479,0.1750,0.0000], [0.2835,0.1070,0.0000],
        [0.1649,0.0610,0.0000], [0.0874,0.0320,0.0000], [0.0468,0.0170,0.0000], [0.0227,0.0082,0.0000],
        [0.0114,0.0041,0.0000], [0.0058,0.0021,0.0000], [0.0029,0.0010,0.0000], [0.0014,0.0005,0.0000]
    ])
    x = np.linspace(0, 1, len(base_cmf))
    xi = np.linspace(0, 1, res)
    interp_cmf = np.zeros((res, 3))
    for k in range(3):
        interp_cmf[:,k] = np.interp(xi, x, base_cmf[:,k])
    return interp_cmf

def compute_macadam_solid(res=200): 
    # High res CMF for integration
    cmf = get_cmf_high_res(400)
    n_lambda = len(cmf)
    cum_cmf = np.vstack([np.zeros(3), np.cumsum(cmf, axis=0)])
    white_xyz = cum_cmf[-1]
    
    optimal_colors = []
    indices = np.linspace(0, n_lambda, res).astype(int)
    
    for i in indices:
        for j in indices:
            if i == j: continue
            if j > i: xyz = cum_cmf[j] - cum_cmf[i]
            else:     xyz = cum_cmf[j] + (white_xyz - cum_cmf[i])
            optimal_colors.append(xyz)
    
    optimal_colors.append([0,0,0])
    optimal_colors.append(white_xyz)
    return np.array(optimal_colors) / white_xyz[1] # Normalize Ymax=1

def extract_macadam_ridge_smoothed(macadam_jz, res=360):
    # 1. Polar
    az = macadam_jz[:, 1]
    bz = macadam_jz[:, 2]
    chroma = np.sqrt(az**2 + bz**2)
    angle = np.arctan2(bz, az)
    
    # 2. Bin Max Chroma
    bins = np.linspace(-np.pi, np.pi, res+1)
    ridge_points = []
    
    for i in range(res):
        mask = (angle >= bins[i]) & (angle < bins[i+1])
        if np.sum(mask) == 0: 
            if len(ridge_points) > 0: ridge_points.append(ridge_points[-1])
            else: ridge_points.append(macadam_jz[0])
            continue
        slice_pts = macadam_jz[mask]
        idx_max = np.argmax(chroma[mask])
        ridge_points.append(slice_pts[idx_max])
        
    raw_ridge = np.array(ridge_points)
    
    # 3. Smooth
    def smooth_channel(data, sigma=5):
        padded = np.concatenate([data[-30:], data, data[:30]])
        kernel = np.exp(-np.linspace(-10,10,21)**2 / (2*sigma**2))
        kernel /= kernel.sum()
        return np.convolve(padded, kernel, mode='same')[30:-30]

    J_smooth = smooth_channel(raw_ridge[:,0])
    a_smooth = smooth_channel(raw_ridge[:,1])
    b_smooth = smooth_channel(raw_ridge[:,2])
    
    return np.column_stack([J_smooth, a_smooth, b_smooth])

# ==========================================
# 3. Optimization & Projections
# ==========================================

def get_gamut_surface(res=40):
    t = np.linspace(0, 1, res)
    g = np.meshgrid(t, t)
    faces = [np.stack([g[0], g[1], np.zeros_like(g[0])], -1), np.stack([g[0], g[1], np.ones_like(g[0])], -1),
             np.stack([g[0], np.zeros_like(g[0]), g[1]], -1), np.stack([g[0], np.ones_like(g[0]), g[1]], -1),
             np.stack([np.zeros_like(g[0]), g[0], g[1]], -1), np.stack([np.ones_like(g[0]), g[0], g[1]], -1)]
    pts = np.vstack([f.reshape(-1, 3) for f in faces])
    corners = np.array([[0,0,0], [1,0,0], [0,1,0], [0,0,1], [1,1,0], [1,0,1], [0,1,1], [1,1,1]])
    return np.vstack([pts, corners])

def fit_ellipsoid_anchored_solver(points, anchor_point):
    hull = ConvexHull(points)
    A = hull.equations[:, :3]; b_const = -hull.equations[:, 3]
    scale = 100.0; A_scaled = A / scale; anchor_scaled = anchor_point * scale
    c0 = np.mean(points, axis=0) * scale
    x0 = np.concatenate([0.5*c0 + 0.5*anchor_scaled, np.eye(3).flatten() * 0.05])
    def objective(x):
        M = x[3:].reshape(3, 3); sign, val = np.linalg.slogdet(M)
        return -val if sign > 0 else 1e9
    def constraints(x):
        d, M = x[:3], x[3:].reshape(3, 3)
        norm_AM = np.linalg.norm(A_scaled @ M, axis=1)
        hull_con = b_const - (A_scaled @ d + norm_AM)
        try: u = np.linalg.solve(M, anchor_scaled - d); anchor_con = 1.0 - np.linalg.norm(u)
        except: anchor_con = -1.0
        return np.concatenate([hull_con, [anchor_con]])
    res = opt.minimize(objective, x0, method='SLSQP', constraints={'type': 'ineq', 'fun': constraints}, options={'maxiter': 1000})
    return res.x[:3]/scale, res.x[3:].reshape(3, 3)/scale, hull

def calculate_shadow_boundary(center, M, view_vector, res=72):
    w = np.linalg.solve(M, view_vector); w = w / np.linalg.norm(w)
    tmp = np.array([0.0, 1.0, 0.0]) if np.abs(w[1]) < 0.9 else np.array([0.0, 0.0, 1.0])
    s1 = np.cross(w, tmp); s1 /= np.linalg.norm(s1); s2 = np.cross(w, s1)
    theta = np.linspace(0, 2*np.pi, res)
    return center + (np.outer(np.cos(theta), s1) + np.outer(np.sin(theta), s2)) @ M.T

def inflate_to_max_gamut(center, M_init):
    scale = 1.0; step = 0.05; best_M = M_init.copy()
    for _ in range(50):
        test_M = M_init * (scale + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        scale += step; best_M = test_M
    step = 0.005; current_M = best_M
    for _ in range(20):
        test_M = current_M * (1.0 + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        current_M = test_M
    return current_M

def project_ridge_onto_ellipsoid(ridge_jz, center, M):
    vecs = ridge_jz - center
    M_inv = np.linalg.inv(M)
    u_vecs = vecs @ M_inv.T
    norms = np.linalg.norm(u_vecs, axis=1, keepdims=True)
    u_surface = u_vecs / norms
    return (u_surface @ M.T) + center

def project_points_to_srgb_surface_conical(points_jz, center_jz):
    projected = []
    for pt in points_jz:
        vec = pt - center_jz
        low = 0.0; high = 1.5; best_t = 0.0
        for _ in range(15):
            mid = (low + high) / 2.0
            test = center_jz + mid * vec
            rgb = xyz_to_srgb_raw(jzazbz_to_xyz(np.array([test])))
            if np.min(rgb) >= -0.001 and np.max(rgb) <= 1.001:
                best_t = mid; low = mid 
            else: high = mid 
        projected.append(center_jz + best_t * vec)
    return np.array(projected)

def reparameterize_curve(points, n_samples=256):
    dists = np.linalg.norm(points[1:] - points[:-1], axis=1)
    cum_dist = np.insert(np.cumsum(dists), 0, 0.0)
    target_dists = np.linspace(0, cum_dist[-1], n_samples)
    new_points = np.zeros((n_samples, 3))
    for i in range(3):
        new_points[:, i] = np.interp(target_dists, cum_dist, points[:, i])
    return new_points

def fourier_approx_curve(points, harmonics=3):
    N = len(points)
    new_cols = []
    for i in range(3):
        sig = points[:,i]
        coeffs = np.fft.rfft(sig)
        filt = np.zeros_like(coeffs)
        filt[:harmonics+1] = coeffs[:harmonics+1]
        new_cols.append(np.fft.irfft(filt, n=N))
    return np.column_stack(new_cols)

# ==========================================
# 4. Omnipose Helpers
# ==========================================
def align_ring_red_to_green(ring_lin):
    n = len(ring_lin)
    red_ref = np.array([1.0, 0.0, 0.0])
    idx_r = np.argmin(np.linalg.norm(ring_lin - red_ref, axis=1))
    ring_aligned = np.roll(ring_lin, -idx_r, axis=0)
    green_ref = np.array([0.0, 1.0, 0.0])
    if np.linalg.norm(ring_aligned[(2*n)//3]-green_ref) < np.linalg.norm(ring_aligned[n//3]-green_ref):
        ring_aligned = ring_aligned[::-1]
    return ring_aligned

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    def build_disk(ring, center):
        y, x = np.mgrid[-1:1:256j, -1:1:256j]
        mag = np.sqrt(x*x + y*y)
        angle = np.mod(np.arctan2(y, x), 2*np.pi)
        n = ring.shape[0]
        hue_f = angle/(2*np.pi)*n
        idx0 = np.floor(hue_f).astype(int) % n
        idx1 = (idx0 + 1) % n
        t = hue_f - np.floor(hue_f)
        interp = (1-t[...,None])*ring[idx0] + t[...,None]*ring[idx1]
        rgb = (1-mag[...,None])*center + mag[...,None]*interp
        return np.clip(linear_to_srgb_np(rgb),0,1)

    disk = build_disk(rgb_ring_lin, center_lin)
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi)
    mag = np.clip(np.sqrt(u*u + v*v), 0, 1)
    n = rgb_ring_lin.shape[0]
    hue_f = angle/(2*np.pi)*n
    idx0 = np.floor(hue_f).astype(int) % n
    idx1 = (idx0 + 1) % n
    t = hue_f - np.floor(hue_f)
    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    r = mag[..., None]
    rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow),0,1)
    alpha = mag[...,None]
    white = np.ones_like(flow_rgb); black = np.zeros_like(flow_rgb)
    flow_white = alpha*flow_rgb + (1-alpha)*white
    flow_black = alpha*flow_rgb + (1-alpha)*black
    return disk, flow_rgb, flow_white, flow_black

# ==========================================
# 5. Main Execution
# ==========================================

def main():
    N_HARMONICS = 3 
    device_str = "cpu"
    dev = torch.device(device_str)
    
    print("1. Calculating Geometries...")
    surf_pts_opt = get_gamut_surface(40) 
    jz_gamut_opt = xyz_to_jzazbz(srgb_to_xyz(surf_pts_opt))
    centroid = np.mean(jz_gamut_opt, axis=0)
    green_vertex = xyz_to_jzazbz(srgb_to_xyz(np.array([[0.0, 1.0, 0.0]])))[0]
    anchor = centroid + 0.60 * (green_vertex - centroid)
    
    # 2. Ellipsoid
    print("2. Fitting Maximal Ellipsoid...")
    center, M_solver, hull = fit_ellipsoid_anchored_solver(jz_gamut_opt, anchor)
    M_max = inflate_to_max_gamut(center, M_solver)
    
    # Loop A: Ellipsoid Loop
    hoop = calculate_shadow_boundary(center, M_max, np.array([1.0, 0.0, 0.0]), res=72)
    rgb_hoop_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(hoop)),0,1)))
    
    # Loop B: Sinebow
    t = np.linspace(0, 1, 256, endpoint=False)
    sr = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0/3)); sg = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1/3)); sb = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2/3))
    rgb_sine_std = np.stack([sr, sg, sb], axis=1)
    rgb_sine_lin = align_ring_red_to_green(srgb_to_linear_np(rgb_sine_std))
    
    # Loop C: MacAdam Ridge
    print("3. Computing MacAdam Ridge...")
    macadam_xyz = compute_macadam_solid(res=200)
    macadam_jz = xyz_to_jzazbz(macadam_xyz)
    ridge_jz_highres = extract_macadam_ridge_smoothed(macadam_jz, res=360) # High Res + Smoothed
    
    # Ridge -> Ellipsoid
    proj_ridge_jz_raw = project_ridge_onto_ellipsoid(ridge_jz_highres, center, M_max)
    proj_ridge_jz_uniform = reparameterize_curve(proj_ridge_jz_raw, n_samples=256)
    rgb_proj_ell_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(proj_ridge_jz_uniform)), 0, 1)))
    
    # Ridge -> sRGB (Conical)
    proj_srgb_jz_raw = project_points_to_srgb_surface_conical(ridge_jz_highres, centroid) 
    proj_srgb_jz_uniform = reparameterize_curve(proj_srgb_jz_raw, n_samples=256) # Reparameterized
    rgb_proj_srgb_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(proj_srgb_jz_uniform)), 0, 1)))
    
    # Fourier Smoothed
    fourier_jz = fourier_approx_curve(proj_srgb_jz_uniform, harmonics=N_HARMONICS)
    fourier_jz_uniform = reparameterize_curve(fourier_jz, n_samples=256)
    rgb_fourier_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(fourier_jz_uniform)), 0, 1)))
    
    print("4. Loading Omnipose Flows...")
    omnidir = Path(omnirefactor.__file__).resolve().parent.parent.parent
    basedir = omnidir / "docs" / "_static"
    name = "ecoli"
    ext = ".tif"

    try:
        masks = io.imread(str(basedir / f"{name}_labels{ext}"))
        slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows = omnirefactor.core.masks_to_flows(masks, device=device_str)[4].to(dev)
        center_lin = srgb_to_linear_np(np.array([0.5, 0.5, 0.5]))

        def gen_set(ring, rot_steps=0):
            idx = int(rot_steps * len(ring) / 72.0)
            r = np.roll(ring, idx, axis=0)
            return make_flow_images_for_ring(r, center_lin, flows)

        rows = []
        d, f, w, b = gen_set(rgb_sine_lin, 0)
        rows.append(("Sinebow", rgb_sine_lin, d, b, w))
        
        d, f, w, b = gen_set(rgb_hoop_lin, 0)
        rows.append(("Max Ellipse", rgb_hoop_lin, d, b, w))
        
        d, f, w, b = gen_set(rgb_proj_ell_lin, 0)
        rows.append(("Ridge->Ellipsoid", rgb_proj_ell_lin, d, b, w))
        
        d, f, w, b = gen_set(rgb_proj_srgb_lin, 0)
        rows.append(("Ridge->sRGB", rgb_proj_srgb_lin, d, b, w))
        
        d, f, w, b = gen_set(rgb_fourier_lin, 0)
        rows.append((f"Fourier (N={N_HARMONICS})", rgb_fourier_lin, d, b, w))

        print("5. Plotting 2D...")
        plt.style.use('dark_background')
        fig, axes = plt.subplots(5, 4, figsize=(16, 24))
        titles = ["Gradient", "Hue Disk", "Black (0°)", "White (0°)"]

        for i, (name, ring, disk, b, w) in enumerate(rows):
            ax_row = axes[i]
            ax_row[0].imshow(linear_to_srgb_np(ring)[None, ...], aspect='auto')
            ax_row[0].set_ylabel(name, fontsize=14, color='white', rotation=90, labelpad=20)
            ax_row[1].imshow(disk); ax_row[2].imshow(b); ax_row[3].imshow(w)
            for j, ax in enumerate(ax_row):
                if i==0: ax.set_title(titles[j], fontsize=12, color='#dddddd')
                ax.set_xticks([]); ax.set_yticks([])
        plt.tight_layout()
        plt.show()
        
        print("6. Generating 3D...")
        fig3d = go.Figure()
        max_jz = np.max(jz_gamut_opt[:,0]); sc = np.array([1.0/max_jz, 1.0, 1.0]); 
        def mc(a): return a[:,1], a[:,2], a[:,0]

        hull_macadam = ConvexHull(macadam_jz)
        mx, my, mz = mc(macadam_jz * sc)
        fig3d.add_trace(go.Mesh3d(x=mx, y=my, z=mz, i=hull_macadam.simplices[:,0], j=hull_macadam.simplices[:,1], k=hull_macadam.simplices[:,2], color='white', opacity=0.05, name='Visible Gamut'))

        corners = [[0,0,0],[1,0,0],[1,1,0],[0,1,0],[0,0,1],[1,0,1],[1,1,1],[0,1,1]]
        edges_idx = [[0,1],[1,2],[2,3],[3,0],[4,5],[5,6],[6,7],[7,4],[0,4],[1,5],[2,6],[3,7]]
        for s,e in edges_idx:
            l = xyz_to_jzazbz(srgb_to_xyz(np.linspace(corners[s],corners[e],25))) * sc
            x,y,z = mc(l)
            fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', line=dict(color='gray', width=2), showlegend=False))

        u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
        sph = np.stack([np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v)], axis=-1).reshape(-1, 3)
        ell = (sph @ M_max.T + center) * sc
        ex, ey, ez = mc(ell)
        fig3d.add_trace(go.Surface(x=ex.reshape(u.shape), y=ey.reshape(u.shape), z=ez.reshape(u.shape), colorscale=[(0,'#88aa88'),(1,'#88aa88')], opacity=0.2, showscale=False, name='Ellipsoid'))

        def plot_dots(pts, col, name):
             x,y,z = mc(pts * sc)
             rgb = linear_to_srgb_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(pts)), 0, 1))
             cols = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in rgb]
             fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='markers', marker=dict(color=cols, size=4), name=name))

        plot_dots(hoop, 'orange', 'Ellipsoid Loop')
        plot_dots(xyz_to_jzazbz(srgb_to_xyz(rgb_sine_std)), 'green', 'Sinebow Loop')
        plot_dots(proj_srgb_jz_uniform, 'red', 'Ridge->sRGB')
        plot_dots(fourier_jz_uniform, 'yellow', f'Fourier (N={N_HARMONICS})')

        rrx, rry, rrz = mc(ridge_jz_highres * sc)
        fig3d.add_trace(go.Scatter3d(x=rrx, y=rry, z=rrz, mode='lines', line=dict(color='white', width=3), name='MacAdam Ridge (Ref)'))

        fig3d.update_layout(template="plotly_dark", title=f"3D Jzazbz: Fourier (N={N_HARMONICS}) Comparison", 
                            scene=dict(xaxis_title='az', yaxis_title='bz', zaxis_title='Jz', aspectmode='manual', aspectratio=dict(x=1,y=1,z=1)))
        fig3d.show()

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import scipy.optimize as opt
from scipy.spatial import ConvexHull
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import omnirefactor.core
from skimage import io
import fastremap

# ==========================================
# 1. Color Math
# ==========================================
def srgb_to_xyz(rgb):
    mask = rgb > 0.04045
    linear = np.zeros_like(rgb)
    linear[mask] = ((rgb[mask] + 0.055) / 1.055) ** 2.4
    linear[~mask] = rgb[~mask] / 12.92
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    return linear @ M.T

def xyz_to_jzazbz(xyz):
    b, g, d, d0 = 1.15, 0.66, -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    lms = xyz @ M1.T
    lms_norm = (lms * 200) / 10000.0 
    lms_prime = np.sign(lms_norm) * (((c1 + c2 * (np.abs(lms_norm) ** n)) / (1 + c3 * (np.abs(lms_norm) ** n))) ** p)
    izazbz = lms_prime @ M2.T
    Jz = ((1 + d) * izazbz[:, 0]) / (1 + d * izazbz[:, 0]) - d0
    return np.column_stack((Jz, izazbz[:, 1], izazbz[:, 2]))

def jzazbz_to_xyz(jzazbz):
    Jz, az, bz = jzazbz[:,0], jzazbz[:,1], jzazbz[:,2]
    d, d0 = -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    Jp = Jz + d0; Iz = Jp / (1 + d - d * Jp)
    izazbz_vec = np.stack([Iz, az, bz], axis=1)
    lms_prime = izazbz_vec @ np.linalg.inv(M2).T
    sign_lms = np.sign(lms_prime)
    Y = np.abs(lms_prime) ** (1/p)
    num = Y - c1; den = c2 - Y * c3
    An = np.clip(num / den, 0, None) 
    lms_norm = sign_lms * (An ** (1/n))
    lms = (lms_norm * 10000.0) / 200.0
    return lms @ np.linalg.inv(M1).T

def xyz_to_srgb_raw(xyz):
    M_inv = np.linalg.inv(np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]]))
    linear = xyz @ M_inv.T
    srgb = np.zeros_like(linear)
    mask = linear > 0.0031308
    srgb[mask] = 1.055 * (linear[mask] ** (1/2.4)) - 0.055
    srgb[~mask] = 12.92 * linear[~mask]
    return srgb

def xyz_to_srgb(xyz): return np.clip(xyz_to_srgb_raw(xyz), 0, 1)
def srgb_to_linear_np(srgb): return np.where(srgb <= 0.04045, srgb / 12.92, ((srgb + 0.055) / 1.055) ** 2.4)
def linear_to_srgb_np(lin): 
    s = np.zeros_like(lin)
    mask = lin > 0.0031308
    s[mask] = 1.055 * (lin[mask]**(1/2.4)) - 0.055
    s[~mask] = 12.92 * lin[~mask]
    return np.clip(s, 0, 1)

# ==========================================
# 2. MacAdam Solid & Ridge
# ==========================================

def get_cmf_high_res(res=360):
    # 2-deg 380-780nm
    base_cmf = np.array([
        [0.0014,0.0000,0.0065], [0.0042,0.0001,0.0201], [0.0143,0.0004,0.0679], [0.0435,0.0012,0.2074],
        [0.1344,0.0040,0.6456], [0.2839,0.0116,1.3856], [0.3483,0.0230,1.7471], [0.3362,0.0380,1.7721],
        [0.2908,0.0600,1.6692], [0.1954,0.0910,1.2876], [0.0956,0.1390,0.8130], [0.0320,0.2080,0.4652],
        [0.0049,0.3230,0.2720], [0.0093,0.5030,0.1582], [0.0633,0.7100,0.0782], [0.1655,0.8620,0.0422],
        [0.2904,0.9540,0.0203], [0.4334,0.9950,0.0087], [0.5945,0.9950,0.0039], [0.7621,0.9520,0.0021],
        [0.9163,0.8700,0.0017], [1.0263,0.7570,0.0011], [1.0622,0.6310,0.0008], [1.0026,0.5030,0.0006],
        [0.8544,0.3810,0.0002], [0.6424,0.2650,0.0000], [0.4479,0.1750,0.0000], [0.2835,0.1070,0.0000],
        [0.1649,0.0610,0.0000], [0.0874,0.0320,0.0000], [0.0468,0.0170,0.0000], [0.0227,0.0082,0.0000],
        [0.0114,0.0041,0.0000], [0.0058,0.0021,0.0000], [0.0029,0.0010,0.0000], [0.0014,0.0005,0.0000]
    ])
    x = np.linspace(0, 1, len(base_cmf))
    xi = np.linspace(0, 1, res)
    interp_cmf = np.zeros((res, 3))
    for k in range(3):
        interp_cmf[:,k] = np.interp(xi, x, base_cmf[:,k])
    return interp_cmf

def compute_macadam_solid(res=250): 
    cmf = get_cmf_high_res(400)
    n_lambda = len(cmf)
    cum_cmf = np.vstack([np.zeros(3), np.cumsum(cmf, axis=0)])
    white_xyz = cum_cmf[-1]
    optimal_colors = []
    indices = np.linspace(0, n_lambda, res).astype(int)
    for i in indices:
        for j in indices:
            if i == j: continue
            if j > i: xyz = cum_cmf[j] - cum_cmf[i]
            else:     xyz = cum_cmf[j] + (white_xyz - cum_cmf[i])
            optimal_colors.append(xyz)
    optimal_colors.append([0,0,0])
    optimal_colors.append(white_xyz)
    return np.array(optimal_colors) / white_xyz[1]

def fourier_smooth_points(points, harmonics=12):
    """Smoothing via Fourier Low-Pass to kill noise but keep shape."""
    N = len(points)
    new_cols = []
    for i in range(3):
        sig = points[:,i]
        coeffs = np.fft.rfft(sig)
        # Hard cut
        filt = np.zeros_like(coeffs)
        filt[:harmonics+1] = coeffs[:harmonics+1]
        new_cols.append(np.fft.irfft(filt, n=N))
    return np.column_stack(new_cols)

def extract_macadam_ridge_clean(macadam_jz, res=720):
    # Polar binning
    az = macadam_jz[:, 1]; bz = macadam_jz[:, 2]
    chroma = np.sqrt(az**2 + bz**2); angle = np.arctan2(bz, az)
    bins = np.linspace(-np.pi, np.pi, res+1)
    ridge_points = []
    
    # Optimized extraction
    sort_idx = np.argsort(angle)
    angle_s = angle[sort_idx]; chroma_s = chroma[sort_idx]; pts_s = macadam_jz[sort_idx]
    bin_idx = np.searchsorted(angle_s, bins)
    
    for i in range(res):
        start, end = bin_idx[i], bin_idx[i+1]
        if start == end:
            if len(ridge_points)>0: ridge_points.append(ridge_points[-1])
            else: ridge_points.append(macadam_jz[0])
            continue
        idx_max = np.argmax(chroma_s[start:end])
        ridge_points.append(pts_s[start + idx_max])
        
    raw = np.array(ridge_points)
    
    # Fourier Smooth immediately to clean up the ridge
    return fourier_smooth_points(raw, harmonics=12)

# ==========================================
# 3. Optimization & Projections
# ==========================================

def get_gamut_surface(res=40):
    t = np.linspace(0, 1, res)
    g = np.meshgrid(t, t)
    faces = [np.stack([g[0], g[1], np.zeros_like(g[0])], -1), np.stack([g[0], g[1], np.ones_like(g[0])], -1),
             np.stack([g[0], np.zeros_like(g[0]), g[1]], -1), np.stack([g[0], np.ones_like(g[0]), g[1]], -1),
             np.stack([np.zeros_like(g[0]), g[0], g[1]], -1), np.stack([np.ones_like(g[0]), g[0], g[1]], -1)]
    pts = np.vstack([f.reshape(-1, 3) for f in faces])
    corners = np.array([[0,0,0], [1,0,0], [0,1,0], [0,0,1], [1,1,0], [1,0,1], [0,1,1], [1,1,1]])
    return np.vstack([pts, corners])

def fit_ellipsoid_anchored_solver(points, anchor_point):
    hull = ConvexHull(points)
    A = hull.equations[:, :3]; b_const = -hull.equations[:, 3]
    scale = 100.0; A_scaled = A / scale; anchor_scaled = anchor_point * scale
    c0 = np.mean(points, axis=0) * scale
    x0 = np.concatenate([0.5*c0 + 0.5*anchor_scaled, np.eye(3).flatten() * 0.05])
    def objective(x):
        M = x[3:].reshape(3, 3); sign, val = np.linalg.slogdet(M)
        return -val if sign > 0 else 1e9
    def constraints(x):
        d, M = x[:3], x[3:].reshape(3, 3)
        norm_AM = np.linalg.norm(A_scaled @ M, axis=1)
        hull_con = b_const - (A_scaled @ d + norm_AM)
        try: u = np.linalg.solve(M, anchor_scaled - d); anchor_con = 1.0 - np.linalg.norm(u)
        except: anchor_con = -1.0
        return np.concatenate([hull_con, [anchor_con]])
    res = opt.minimize(objective, x0, method='SLSQP', constraints={'type': 'ineq', 'fun': constraints}, options={'maxiter': 1000})
    return res.x[:3]/scale, res.x[3:].reshape(3, 3)/scale, hull

def calculate_shadow_boundary(center, M, view_vector, res=72):
    w = np.linalg.solve(M, view_vector); w = w / np.linalg.norm(w)
    tmp = np.array([0.0, 1.0, 0.0]) if np.abs(w[1]) < 0.9 else np.array([0.0, 0.0, 1.0])
    s1 = np.cross(w, tmp); s1 /= np.linalg.norm(s1); s2 = np.cross(w, s1)
    theta = np.linspace(0, 2*np.pi, res)
    return center + (np.outer(np.cos(theta), s1) + np.outer(np.sin(theta), s2)) @ M.T

def inflate_to_max_gamut(center, M_init):
    scale = 1.0; step = 0.05; best_M = M_init.copy()
    for _ in range(50):
        test_M = M_init * (scale + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        scale += step; best_M = test_M
    step = 0.005; current_M = best_M
    for _ in range(20):
        test_M = current_M * (1.0 + step)
        hoop = calculate_shadow_boundary(center, test_M, np.array([1.0, 0.0, 0.0]), res=60)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        if np.min(rgb) < -0.001 or np.max(rgb) > 1.001: break 
        current_M = test_M
    return current_M

def shrink_ridge_uniform(ridge_jz, center_point):
    """
    Uniformly scales the ridge towards center_point until it fits.
    Preserves Shape.
    """
    print("   Shrinking Ridge Uniformly...")
    scale = 1.0
    step = 0.005
    
    # Pre-center
    vectors = ridge_jz - center_point
    
    for _ in range(200):
        test_ridge = center_point + scale * vectors
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(test_ridge))
        
        # Check gamut compliance (Strict: 99% points must pass)
        is_valid = (rgb >= -0.002) & (rgb <= 1.002)
        valid_pct = np.mean(np.all(is_valid, axis=1))
        
        if valid_pct > 0.99:
            print(f"      Fit found at scale {scale:.3f}")
            return test_ridge
            
        scale -= step
        
    return ridge_jz * 0.1 # Fail safe

def reparameterize_by_hue_angle(points, n_samples=256):
    az = points[:, 1]; bz = points[:, 2]
    angles = np.arctan2(bz, az)
    angles = np.unwrap(angles)
    target_angles = np.linspace(angles[0], angles[0] + 2*np.pi, n_samples, endpoint=False)
    angles_pad = np.concatenate([angles - 2*np.pi, angles, angles + 2*np.pi])
    J_pad = np.tile(points[:, 0], 3)
    a_pad = np.tile(points[:, 1], 3)
    b_pad = np.tile(points[:, 2], 3)
    return np.column_stack([
        np.interp(target_angles, angles_pad, J_pad),
        np.interp(target_angles, angles_pad, a_pad),
        np.interp(target_angles, angles_pad, b_pad)
    ])

# ==========================================
# 4. Omnipose Helpers
# ==========================================
def align_ring_red_to_green(ring_lin):
    n = len(ring_lin)
    red_ref = np.array([1.0, 0.0, 0.0])
    idx_r = np.argmin(np.linalg.norm(ring_lin - red_ref, axis=1))
    ring_aligned = np.roll(ring_lin, -idx_r, axis=0)
    green_ref = np.array([0.0, 1.0, 0.0])
    if np.linalg.norm(ring_aligned[(2*n)//3]-green_ref) < np.linalg.norm(ring_aligned[n//3]-green_ref):
        ring_aligned = ring_aligned[::-1]
    return ring_aligned

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    def build_disk(ring, center):
        y, x = np.mgrid[-1:1:256j, -1:1:256j]
        mag = np.sqrt(x*x + y*y)
        angle = np.mod(np.arctan2(y, x), 2*np.pi)
        n = ring.shape[0]
        hue_f = angle/(2*np.pi)*n
        idx0 = np.floor(hue_f).astype(int) % n
        idx1 = (idx0 + 1) % n
        t = hue_f - np.floor(hue_f)
        interp = (1-t[...,None])*ring[idx0] + t[...,None]*ring[idx1]
        rgb = (1-mag[...,None])*center + mag[...,None]*interp
        return np.clip(linear_to_srgb_np(rgb),0,1)

    disk = build_disk(rgb_ring_lin, center_lin)
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi)
    mag = np.clip(np.sqrt(u*u + v*v), 0, 1)
    n = rgb_ring_lin.shape[0]
    hue_f = angle/(2*np.pi)*n
    idx0 = np.floor(hue_f).astype(int) % n
    idx1 = (idx0 + 1) % n
    t = hue_f - np.floor(hue_f)
    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    r = mag[..., None]
    rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow),0,1)
    alpha = mag[...,None]
    white = np.ones_like(flow_rgb); black = np.zeros_like(flow_rgb)
    flow_white = alpha*flow_rgb + (1-alpha)*white
    flow_black = alpha*flow_rgb + (1-alpha)*black
    return disk, flow_rgb, flow_white, flow_black

# ==========================================
# 5. Main Execution
# ==========================================

def main():
    device_str = "cpu"
    dev = torch.device(device_str)
    
    print("1. Calculating Geometries...")
    surf_pts_opt = get_gamut_surface(40) 
    jz_gamut_opt = xyz_to_jzazbz(srgb_to_xyz(surf_pts_opt))
    centroid = np.mean(jz_gamut_opt, axis=0)
    green_vertex = xyz_to_jzazbz(srgb_to_xyz(np.array([[0.0, 1.0, 0.0]])))[0]
    anchor = centroid + 0.60 * (green_vertex - centroid)
    
    print("2. Fitting Maximal Ellipsoid...")
    center, M_solver, hull = fit_ellipsoid_anchored_solver(jz_gamut_opt, anchor)
    M_max = inflate_to_max_gamut(center, M_solver)
    hoop = calculate_shadow_boundary(center, M_max, np.array([1.0, 0.0, 0.0]), res=72)
    hoop_ang = reparameterize_by_hue_angle(hoop, n_samples=256)
    rgb_hoop_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(hoop_ang)),0,1)))
    
    # Sinebow
    t = np.linspace(0, 1, 256, endpoint=False)
    sr = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0/3)); sg = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1/3)); sb = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2/3))
    rgb_sine_std = np.stack([sr, sg, sb], axis=1)
    rgb_sine_lin = align_ring_red_to_green(srgb_to_linear_np(rgb_sine_std))
    
    # MacAdam Ridge
    print("3. Computing MacAdam Ridge...")
    macadam_xyz = compute_macadam_solid(res=250)
    macadam_jz = xyz_to_jzazbz(macadam_xyz)
    
    # Extract Clean Ridge (Fourier Smoothed Source)
    ridge_jz_clean = extract_macadam_ridge_clean(macadam_jz, res=720)
    
    # Uniform Shrink
    shrunk_jz_raw = shrink_ridge_uniform(ridge_jz_clean, centroid)
    shrunk_jz_ang = reparameterize_by_hue_angle(shrunk_jz_raw, n_samples=256)
    rgb_shrunk_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(shrunk_jz_ang)), 0, 1)))

    print("4. Loading Omnipose Flows...")
    omnidir = Path(omnirefactor.__file__).resolve().parent.parent.parent
    basedir = omnidir / "docs" / "_static"
    name = "ecoli"
    ext = ".tif"

    try:
        masks = io.imread(str(basedir / f"{name}_labels{ext}"))
        slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows = omnirefactor.core.masks_to_flows(masks, device=device_str)[4].to(dev)
        center_lin = srgb_to_linear_np(np.array([0.5, 0.5, 0.5]))

        def gen_set(ring, rot_steps=0):
            idx = int(rot_steps * len(ring) / 72.0)
            r = np.roll(ring, idx, axis=0)
            return make_flow_images_for_ring(r, center_lin, flows)

        rows = []
        d, f, w, b = gen_set(rgb_sine_lin, 0)
        rows.append(("Sinebow", rgb_sine_lin, d, b, w))
        d, f, w, b = gen_set(rgb_hoop_lin, 0)
        rows.append(("Max Ellipse", rgb_hoop_lin, d, b, w))
        d, f, w, b = gen_set(rgb_shrunk_lin, 0)
        rows.append(("Uniform Shrink", rgb_shrunk_lin, d, b, w))

        print("5. Plotting 2D...")
        plt.style.use('dark_background')
        fig, axes = plt.subplots(3, 4, figsize=(16, 12))
        titles = ["Gradient", "Hue Disk", "Black (0°)", "White (0°)"]

        for i, (name, ring, disk, b, w) in enumerate(rows):
            ax_row = axes[i]
            ax_row[0].imshow(linear_to_srgb_np(ring)[None, ...], aspect='auto')
            ax_row[0].set_ylabel(name, fontsize=14, color='white', rotation=90, labelpad=20)
            ax_row[1].imshow(disk); ax_row[2].imshow(b); ax_row[3].imshow(w)
            for j, ax in enumerate(ax_row):
                if i==0: ax.set_title(titles[j], fontsize=12, color='#dddddd')
                ax.set_xticks([]); ax.set_yticks([])
        plt.tight_layout()
        plt.show()
        
        print("6. Generating 3D...")
        fig3d = go.Figure()
        max_jz = np.max(jz_gamut_opt[:,0]); sc = np.array([1.0/max_jz, 1.0, 1.0]); 
        def mc(a): return a[:,1], a[:,2], a[:,0]

        hull_macadam = ConvexHull(macadam_jz)
        mx, my, mz = mc(macadam_jz * sc)
        fig3d.add_trace(go.Mesh3d(x=mx, y=my, z=mz, i=hull_macadam.simplices[:,0], j=hull_macadam.simplices[:,1], k=hull_macadam.simplices[:,2], color='white', opacity=0.05, name='Visible Gamut'))
        
        corners = [[0,0,0],[1,0,0],[1,1,0],[0,1,0],[0,0,1],[1,0,1],[1,1,1],[0,1,1]]
        edges_idx = [[0,1],[1,2],[2,3],[3,0],[4,5],[5,6],[6,7],[7,4],[0,4],[1,5],[2,6],[3,7]]
        for s,e in edges_idx:
            l = xyz_to_jzazbz(srgb_to_xyz(np.linspace(corners[s],corners[e],25))) * sc
            x,y,z = mc(l)
            fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', line=dict(color='gray', width=2), showlegend=False))

        u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
        sph = np.stack([np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v)], axis=-1).reshape(-1, 3)
        ell = (sph @ M_max.T + center) * sc
        ex, ey, ez = mc(ell)
        fig3d.add_trace(go.Surface(x=ex.reshape(u.shape), y=ey.reshape(u.shape), z=ez.reshape(u.shape), colorscale=[(0,'#88aa88'),(1,'#88aa88')], opacity=0.2, showscale=False, name='Ellipsoid'))

        def plot_dots(pts, col, name):
             x,y,z = mc(pts * sc)
             rgb = linear_to_srgb_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(pts)), 0, 1))
             cols = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in rgb]
             fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='markers', marker=dict(color=cols, size=4), name=name))

        plot_dots(hoop_ang, 'orange', 'Ellipsoid Loop')
        plot_dots(xyz_to_jzazbz(srgb_to_xyz(rgb_sine_std)), 'green', 'Sinebow Loop')
        plot_dots(shrunk_jz_ang, 'cyan', 'Uniformly Shrunk Ridge')

        rrx, rry, rrz = mc(ridge_jz_clean[::10] * sc)
        fig3d.add_trace(go.Scatter3d(x=rrx, y=rry, z=rrz, mode='lines', line=dict(color='white', width=3), name='MacAdam Ridge (Ref)'))

        fig3d.update_layout(template="plotly_dark", title="3D Jzazbz: Uniform Shrink Fit", 
                            scene=dict(xaxis_title='az', yaxis_title='bz', zaxis_title='Jz', aspectmode='manual', aspectratio=dict(x=1,y=1,z=1)))
        fig3d.show()

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import scipy.optimize as opt
import scipy.ndimage
from scipy.spatial import ConvexHull
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import omnirefactor.core
from skimage import io
import fastremap

# ==========================================
# 1. Color Math
# ==========================================
def srgb_to_xyz(rgb):
    mask = rgb > 0.04045
    linear = np.zeros_like(rgb)
    linear[mask] = ((rgb[mask] + 0.055) / 1.055) ** 2.4
    linear[~mask] = rgb[~mask] / 12.92
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    return linear @ M.T

def xyz_to_jzazbz(xyz):
    b, g, d, d0 = 1.15, 0.66, -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    lms = xyz @ M1.T
    lms_norm = (lms * 200) / 10000.0 
    lms_prime = np.sign(lms_norm) * (((c1 + c2 * (np.abs(lms_norm) ** n)) / (1 + c3 * (np.abs(lms_norm) ** n))) ** p)
    izazbz = lms_prime @ M2.T
    Jz = ((1 + d) * izazbz[:, 0]) / (1 + d * izazbz[:, 0]) - d0
    return np.column_stack((Jz, izazbz[:, 1], izazbz[:, 2]))

def jzazbz_to_xyz(jzazbz):
    Jz, az, bz = jzazbz[:,0], jzazbz[:,1], jzazbz[:,2]
    d, d0 = -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    Jp = Jz + d0; Iz = Jp / (1 + d - d * Jp)
    izazbz_vec = np.stack([Iz, az, bz], axis=1)
    lms_prime = izazbz_vec @ np.linalg.inv(M2).T
    sign_lms = np.sign(lms_prime)
    Y = np.abs(lms_prime) ** (1/p)
    num = Y - c1; den = c2 - Y * c3
    An = np.clip(num / den, 0, None) 
    lms_norm = sign_lms * (An ** (1/n))
    lms = (lms_norm * 10000.0) / 200.0
    return lms @ np.linalg.inv(M1).T

def xyz_to_srgb_raw(xyz):
    M_inv = np.linalg.inv(np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]]))
    linear = xyz @ M_inv.T
    srgb = np.zeros_like(linear)
    mask = linear > 0.0031308
    srgb[mask] = 1.055 * (linear[mask] ** (1/2.4)) - 0.055
    srgb[~mask] = 12.92 * linear[~mask]
    return srgb

def xyz_to_srgb(xyz): return np.clip(xyz_to_srgb_raw(xyz), 0, 1)
def srgb_to_linear_np(srgb): return np.where(srgb <= 0.04045, srgb / 12.92, ((srgb + 0.055) / 1.055) ** 2.4)
def linear_to_srgb_np(lin): 
    s = np.zeros_like(lin)
    mask = lin > 0.0031308
    s[mask] = 1.055 * (lin[mask]**(1/2.4)) - 0.055
    s[~mask] = 12.92 * lin[~mask]
    return np.clip(s, 0, 1)

# ==========================================
# 2. MacAdam Solid & Ridge (High Fidelity)
# ==========================================

def get_cmf_high_res(res=360):
    # Standard CIE 1931 2-deg (380-780nm)
    base_cmf = np.array([
        [0.0014,0.0000,0.0065], [0.0042,0.0001,0.0201], [0.0143,0.0004,0.0679], [0.0435,0.0012,0.2074],
        [0.1344,0.0040,0.6456], [0.2839,0.0116,1.3856], [0.3483,0.0230,1.7471], [0.3362,0.0380,1.7721],
        [0.2908,0.0600,1.6692], [0.1954,0.0910,1.2876], [0.0956,0.1390,0.8130], [0.0320,0.2080,0.4652],
        [0.0049,0.3230,0.2720], [0.0093,0.5030,0.1582], [0.0633,0.7100,0.0782], [0.1655,0.8620,0.0422],
        [0.2904,0.9540,0.0203], [0.4334,0.9950,0.0087], [0.5945,0.9950,0.0039], [0.7621,0.9520,0.0021],
        [0.9163,0.8700,0.0017], [1.0263,0.7570,0.0011], [1.0622,0.6310,0.0008], [1.0026,0.5030,0.0006],
        [0.8544,0.3810,0.0002], [0.6424,0.2650,0.0000], [0.4479,0.1750,0.0000], [0.2835,0.1070,0.0000],
        [0.1649,0.0610,0.0000], [0.0874,0.0320,0.0000], [0.0468,0.0170,0.0000], [0.0227,0.0082,0.0000],
        [0.0114,0.0041,0.0000], [0.0058,0.0021,0.0000], [0.0029,0.0010,0.0000], [0.0014,0.0005,0.0000]
    ])
    x = np.linspace(0, 1, len(base_cmf))
    xi = np.linspace(0, 1, res)
    interp_cmf = np.zeros((res, 3))
    for k in range(3):
        interp_cmf[:,k] = np.interp(xi, x, base_cmf[:,k])
    return interp_cmf

def compute_macadam_solid(res=250): 
    cmf = get_cmf_high_res(500)
    n_lambda = len(cmf)
    cum_cmf = np.vstack([np.zeros(3), np.cumsum(cmf, axis=0)])
    white_xyz = cum_cmf[-1]
    
    optimal_colors = []
    indices = np.linspace(0, n_lambda, res).astype(int)
    for i in indices:
        for j in indices:
            if i == j: continue
            if j > i: xyz = cum_cmf[j] - cum_cmf[i]
            else:     xyz = cum_cmf[j] + (white_xyz - cum_cmf[i])
            optimal_colors.append(xyz)
    optimal_colors.append([0,0,0]); optimal_colors.append(white_xyz)
    return np.array(optimal_colors) / white_xyz[1]

def extract_macadam_ridge_dense(macadam_jz, res=3600): 
    az = macadam_jz[:, 1]; bz = macadam_jz[:, 2]
    chroma = np.sqrt(az**2 + bz**2); angle = np.arctan2(bz, az)
    bins = np.linspace(-np.pi, np.pi, res+1)
    ridge_points = []
    
    sort_idx = np.argsort(angle)
    angle_s = angle[sort_idx]; chroma_s = chroma[sort_idx]; pts_s = macadam_jz[sort_idx]
    bin_idx = np.searchsorted(angle_s, bins)
    
    for i in range(res):
        start, end = bin_idx[i], bin_idx[i+1]
        if start == end:
            if len(ridge_points)>0: ridge_points.append(ridge_points[-1])
            else: ridge_points.append(macadam_jz[0])
            continue
        idx_max = np.argmax(chroma_s[start:end])
        ridge_points.append(pts_s[start + idx_max])
        
    # Apply Clean Fourier Smoothing to Source Ridge
    raw = np.array(ridge_points)
    N = len(raw)
    new_cols = []
    for i in range(3):
        sig = raw[:,i]
        coeffs = np.fft.rfft(sig)
        filt = np.zeros_like(coeffs)
        filt[:16] = coeffs[:16] # Keep N=15 harmonics for structure
        new_cols.append(np.fft.irfft(filt, n=N))
    return np.column_stack(new_cols)

# ==========================================
# 3. Optimization & Constraints
# ==========================================

def get_gamut_surface(res=40):
    t = np.linspace(0, 1, res); g = np.meshgrid(t, t)
    faces = [np.stack([g[0], g[1], np.zeros_like(g[0])], -1), np.stack([g[0], g[1], np.ones_like(g[0])], -1),
             np.stack([g[0], np.zeros_like(g[0]), g[1]], -1), np.stack([g[0], np.ones_like(g[0]), g[1]], -1),
             np.stack([np.zeros_like(g[0]), g[0], g[1]], -1), np.stack([np.ones_like(g[0]), g[0], g[1]], -1)]
    pts = np.vstack([f.reshape(-1, 3) for f in faces])
    corners = np.array([[0,0,0], [1,0,0], [0,1,0], [0,0,1], [1,1,0], [1,0,1], [0,1,1], [1,1,1]])
    return np.vstack([pts, corners])

def fit_ellipsoid_anchored_solver(points, anchor_point):
    hull = ConvexHull(points)
    A = hull.equations[:, :3]; b_const = -hull.equations[:, 3]
    scale = 100.0; A_scaled = A / scale; anchor_scaled = anchor_point * scale
    c0 = np.mean(points, axis=0) * scale
    x0 = np.concatenate([0.5*c0 + 0.5*anchor_scaled, np.eye(3).flatten() * 0.05])
    def objective(x):
        M = x[3:].reshape(3, 3); sign, val = np.linalg.slogdet(M)
        return -val if sign > 0 else 1e9
    def constraints(x):
        d, M = x[:3], x[3:].reshape(3, 3)
        norm_AM = np.linalg.norm(A_scaled @ M, axis=1)
        hull_con = b_const - (A_scaled @ d + norm_AM)
        try: u = np.linalg.solve(M, anchor_scaled - d); anchor_con = 1.0 - np.linalg.norm(u)
        except: anchor_con = -1.0
        return np.concatenate([hull_con, [anchor_con]])
    res = opt.minimize(objective, x0, method='SLSQP', constraints={'type': 'ineq', 'fun': constraints}, options={'maxiter': 1000})
    return res.x[:3]/scale, res.x[3:].reshape(3, 3)/scale, hull

def calculate_shadow_boundary(center, M, view_vector, res=72):
    w = np.linalg.solve(M, view_vector); w = w / np.linalg.norm(w)
    tmp = np.array([0.0, 1.0, 0.0]) if np.abs(w[1]) < 0.9 else np.array([0.0, 0.0, 1.0])
    s1 = np.cross(w, tmp); s1 /= np.linalg.norm(s1); s2 = np.cross(w, s1)
    theta = np.linspace(0, 2*np.pi, res)
    return center + (np.outer(np.cos(theta), s1) + np.outer(np.sin(theta), s2)) @ M.T

def ensure_curve_in_gamut(points, center_point):
    """
    Strictly shrinks a curve towards center_point until ALL points obey sRGB limits.
    Handles the "Concave Gamut" issue where hull solvers overestimate.
    """
    current_points = points.copy()
    scale = 1.0
    step = 0.002
    
    for i in range(200):
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(current_points))
        min_v, max_v = np.min(rgb), np.max(rgb)
        
        # Check bounds with tiny epsilon
        if min_v >= -0.002 and max_v <= 1.002:
            return current_points
        
        # Shrink
        scale -= step
        current_points = center_point + scale * (points - center_point)
        
    return current_points

def project_points_to_srgb_surface_conical(points_jz, center_jz):
    projected = []
    for pt in points_jz:
        vec = pt - center_jz
        low = 0.0; high = 1.5; best_t = 0.0
        for _ in range(15):
            mid = (low + high) / 2.0
            test = center_jz + mid * vec
            rgb = xyz_to_srgb_raw(jzazbz_to_xyz(np.array([test])))
            if np.min(rgb) >= -0.001 and np.max(rgb) <= 1.001:
                best_t = mid; low = mid 
            else: high = mid 
        projected.append(center_jz + best_t * vec)
    return np.array(projected)

def fit_fourier_constrained(points, center_point, harmonics=3, iterations=50):
    """
    Fits a Fourier curve that is SMOOTH but constrained inside sRGB.
    """
    print(f"   Fitting Constrained Fourier (N={harmonics})...")
    current_points = points.copy()
    N = len(points)
    
    for i in range(iterations):
        # 1. Fourier Smooth
        new_cols = []
        for d in range(3):
            sig = current_points[:,d]
            coeffs = np.fft.rfft(sig)
            filt = np.zeros_like(coeffs)
            filt[:harmonics+1] = coeffs[:harmonics+1]
            new_cols.append(np.fft.irfft(filt, n=N))
        smooth_points = np.column_stack(new_cols)
        
        # 2. Check Constraint
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(smooth_points))
        is_out = (rgb < -0.002) | (rgb > 1.002)
        out_idx = np.any(is_out, axis=1)
        
        if not np.any(out_idx):
            return smooth_points
            
        # 3. Push Invalid Points (Constraint Projection)
        # Move out-of-gamut points towards center
        vecs = center_point - smooth_points
        smooth_points[out_idx] += vecs[out_idx] * 0.15 # Strong nudge
        
        current_points = smooth_points
        
    return ensure_curve_in_gamut(current_points, center_point) # Final safety

def reparameterize_by_hue_angle(points, n_samples=256):
    az = points[:, 1]; bz = points[:, 2]
    angles = np.arctan2(bz, az)
    angles = np.unwrap(angles)
    target_angles = np.linspace(angles[0], angles[0] + 2*np.pi, n_samples, endpoint=False)
    angles_pad = np.concatenate([angles - 2*np.pi, angles, angles + 2*np.pi])
    J_pad = np.tile(points[:, 0], 3)
    a_pad = np.tile(points[:, 1], 3)
    b_pad = np.tile(points[:, 2], 3)
    return np.column_stack([
        np.interp(target_angles, angles_pad, J_pad),
        np.interp(target_angles, angles_pad, a_pad),
        np.interp(target_angles, angles_pad, b_pad)
    ])

# ==========================================
# 4. Omnipose Helpers
# ==========================================
def align_ring_red_to_green(ring_lin):
    n = len(ring_lin)
    red_ref = np.array([1.0, 0.0, 0.0])
    idx_r = np.argmin(np.linalg.norm(ring_lin - red_ref, axis=1))
    ring_aligned = np.roll(ring_lin, -idx_r, axis=0)
    green_ref = np.array([0.0, 1.0, 0.0])
    if np.linalg.norm(ring_aligned[(2*n)//3]-green_ref) < np.linalg.norm(ring_aligned[n//3]-green_ref):
        ring_aligned = ring_aligned[::-1]
    return ring_aligned

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    def build_disk(ring, center):
        y, x = np.mgrid[-1:1:256j, -1:1:256j]
        mag = np.sqrt(x*x + y*y); angle = np.mod(np.arctan2(y, x), 2*np.pi)
        n = ring.shape[0]; hue_f = angle/(2*np.pi)*n
        idx0 = np.floor(hue_f).astype(int) % n; idx1 = (idx0 + 1) % n; t = hue_f - np.floor(hue_f)
        interp = (1-t[...,None])*ring[idx0] + t[...,None]*ring[idx1]
        rgb = (1-mag[...,None])*center + mag[...,None]*interp
        return np.clip(linear_to_srgb_np(rgb),0,1)

    disk = build_disk(rgb_ring_lin, center_lin)
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi)
    mag = np.clip(np.sqrt(u*u + v*v), 0, 1)
    n = rgb_ring_lin.shape[0]; hue_f = angle/(2*np.pi)*n
    idx0 = np.floor(hue_f).astype(int) % n; idx1 = (idx0 + 1) % n; t = hue_f - np.floor(hue_f)
    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    r = mag[..., None]; rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow),0,1)
    alpha = mag[...,None]; white = np.ones_like(flow_rgb); black = np.zeros_like(flow_rgb)
    flow_white = alpha*flow_rgb + (1-alpha)*white; flow_black = alpha*flow_rgb + (1-alpha)*black
    return disk, flow_rgb, flow_white, flow_black

# ==========================================
# 5. Main Execution
# ==========================================

def main():
    N_HARMONICS = 2
    device_str = "cpu"
    dev = torch.device(device_str)
    
    print("1. Calculating Geometries...")
    surf_pts_opt = get_gamut_surface(40) 
    jz_gamut_opt = xyz_to_jzazbz(srgb_to_xyz(surf_pts_opt))
    centroid = np.mean(jz_gamut_opt, axis=0)
    green_vertex = xyz_to_jzazbz(srgb_to_xyz(np.array([[0.0, 1.0, 0.0]])))[0]
    anchor = centroid + 0.60 * (green_vertex - centroid)
    
    # 2. Ellipsoid Fitting
    print("2. Fitting Maximal Ellipsoid...")
    center, M_solver, hull = fit_ellipsoid_anchored_solver(jz_gamut_opt, anchor)
    
    # Hoop Calculation & Strict Shrink
    hoop_raw = calculate_shadow_boundary(center, M_solver, np.array([1.0, 0.0, 0.0]), res=256)
    hoop_valid = ensure_curve_in_gamut(hoop_raw, centroid) # New strict shrink
    hoop_ang = reparameterize_by_hue_angle(hoop_valid, n_samples=256)
    rgb_hoop_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(hoop_ang)),0,1)))
    
    # 3. Sinebow
    t = np.linspace(0, 1, 256, endpoint=False)
    sr = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0/3)); sg = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1/3)); sb = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2/3))
    rgb_sine_std = np.stack([sr, sg, sb], axis=1)
    rgb_sine_lin = align_ring_red_to_green(srgb_to_linear_np(rgb_sine_std))
    
    # 4. MacAdam Ridge
    print("3. Computing MacAdam Ridge...")
    macadam_xyz = compute_macadam_solid(res=200)
    macadam_jz = xyz_to_jzazbz(macadam_xyz)
    ridge_jz_highres = extract_macadam_ridge_dense(macadam_jz, res=3600)
    
    # Ridge -> sRGB (Conical)
    proj_srgb_jz_raw = project_points_to_srgb_surface_conical(ridge_jz_highres, centroid) 
    proj_srgb_jz_ang = reparameterize_by_hue_angle(proj_srgb_jz_raw, n_samples=256)
    rgb_proj_srgb_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(proj_srgb_jz_ang)), 0, 1)))
    
    # Fourier (Constrained)
    fourier_jz = fit_fourier_constrained(proj_srgb_jz_ang, centroid, harmonics=N_HARMONICS)
    fourier_jz_ang = reparameterize_by_hue_angle(fourier_jz, n_samples=256)
    rgb_fourier_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(fourier_jz_ang)), 0, 1)))
    
    print("4. Loading Omnipose Flows...")
    omnidir = Path(omnirefactor.__file__).resolve().parent.parent.parent
    basedir = omnidir / "docs" / "_static"
    name = "ecoli"
    ext = ".tif"

    try:
        masks = io.imread(str(basedir / f"{name}_labels{ext}"))
        slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows = omnirefactor.core.masks_to_flows(masks, device=device_str)[4].to(dev)
        center_lin = srgb_to_linear_np(np.array([0.5, 0.5, 0.5]))

        def gen_set(ring, rot_steps=0):
            idx = int(rot_steps * len(ring) / 72.0)
            r = np.roll(ring, idx, axis=0)
            return make_flow_images_for_ring(r, center_lin, flows)

        rows = []
        d, f, w, b = gen_set(rgb_sine_lin, 0)
        rows.append(("Sinebow", rgb_sine_lin, d, b, w))
        d, f, w, b = gen_set(rgb_hoop_lin, 0)
        rows.append(("Max Ellipse", rgb_hoop_lin, d, b, w))
        d, f, w, b = gen_set(rgb_proj_srgb_lin, 0)
        rows.append(("Ridge->sRGB", rgb_proj_srgb_lin, d, b, w))
        d, f, w, b = gen_set(rgb_fourier_lin, 0)
        rows.append((f"Fourier (N={N_HARMONICS})", rgb_fourier_lin, d, b, w))

        print("5. Plotting 2D...")
        plt.style.use('dark_background')
        fig, axes = plt.subplots(4, 4, figsize=(16, 16))
        for i, (name, ring, disk, b, w) in enumerate(rows):
            ax_row = axes[i]
            ax_row[0].imshow(linear_to_srgb_np(ring)[None, ...], aspect='auto')
            ax_row[0].set_ylabel(name, fontsize=14, color='white', rotation=90, labelpad=20)
            ax_row[1].imshow(disk); ax_row[2].imshow(b); ax_row[3].imshow(w)
            for j, ax in enumerate(ax_row): ax.set_xticks([]); ax.set_yticks([])
        plt.tight_layout()
        plt.show()
        
        print("6. Generating 3D...")
        fig3d = go.Figure()
        max_jz = np.max(jz_gamut_opt[:,0]); sc = np.array([1.0/max_jz, 1.0, 1.0]); 
        def mc(a): return a[:,1], a[:,2], a[:,0]

        hull_macadam = ConvexHull(macadam_jz)
        mx, my, mz = mc(macadam_jz * sc)
        fig3d.add_trace(go.Mesh3d(x=mx, y=my, z=mz, i=hull_macadam.simplices[:,0], j=hull_macadam.simplices[:,1], k=hull_macadam.simplices[:,2], color='white', opacity=0.05, name='Visible Gamut'))

        corners = [[0,0,0],[1,0,0],[1,1,0],[0,1,0],[0,0,1],[1,0,1],[1,1,1],[0,1,1]]
        edges_idx = [[0,1],[1,2],[2,3],[3,0],[4,5],[5,6],[6,7],[7,4],[0,4],[1,5],[2,6],[3,7]]
        for s,e in edges_idx:
            l = xyz_to_jzazbz(srgb_to_xyz(np.linspace(corners[s],corners[e],25))) * sc
            x,y,z = mc(l)
            fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', line=dict(color='gray', width=2), showlegend=False))

        u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
        sph = np.stack([np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v)], axis=-1).reshape(-1, 3)
        ell = (sph @ M_solver.T + center) * sc
        ex, ey, ez = mc(ell)
        fig3d.add_trace(go.Surface(x=ex.reshape(u.shape), y=ey.reshape(u.shape), z=ez.reshape(u.shape), colorscale=[(0,'#88aa88'),(1,'#88aa88')], opacity=0.2, showscale=False, name='Ellipsoid'))

        def plot_dots(pts, col, name):
             x,y,z = mc(pts * sc)
             rgb = linear_to_srgb_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(pts)), 0, 1))
             cols = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in rgb]
             fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='markers', marker=dict(color=cols, size=4), name=name))

        plot_dots(hoop_ang, 'orange', 'Ellipsoid Loop')
        plot_dots(xyz_to_jzazbz(srgb_to_xyz(rgb_sine_std)), 'green', 'Sinebow Loop')
        plot_dots(proj_srgb_jz_ang, 'red', 'Ridge->sRGB')
        plot_dots(fourier_jz_ang, 'cyan', 'Constrained Fourier')

        rrx, rry, rrz = mc(ridge_jz_highres[::10] * sc)
        fig3d.add_trace(go.Scatter3d(x=rrx, y=rry, z=rrz, mode='lines', line=dict(color='white', width=3), name='MacAdam Ridge (Ref)'))

        fig3d.update_layout(template="plotly_dark", title=f"3D Jzazbz: Constrained Fourier vs Ellipse", 
                            scene=dict(xaxis_title='az', yaxis_title='bz', zaxis_title='Jz', aspectmode='manual', aspectratio=dict(x=1,y=1,z=1)))
        fig3d.show()

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import scipy.optimize as opt
from scipy.spatial import ConvexHull
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import omnirefactor.core
from skimage import io
import fastremap
import scipy.ndimage

# ==========================================
# 1. Color Math
# ==========================================
def srgb_to_xyz(rgb):
    mask = rgb > 0.04045
    linear = np.zeros_like(rgb)
    linear[mask] = ((rgb[mask] + 0.055) / 1.055) ** 2.4
    linear[~mask] = rgb[~mask] / 12.92
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    return linear @ M.T

def xyz_to_jzazbz(xyz):
    b, g, d, d0 = 1.15, 0.66, -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    lms = xyz @ M1.T
    lms_norm = (lms * 200) / 10000.0 
    lms_prime = np.sign(lms_norm) * (((c1 + c2 * (np.abs(lms_norm) ** n)) / (1 + c3 * (np.abs(lms_norm) ** n))) ** p)
    izazbz = lms_prime @ M2.T
    Jz = ((1 + d) * izazbz[:, 0]) / (1 + d * izazbz[:, 0]) - d0
    return np.column_stack((Jz, izazbz[:, 1], izazbz[:, 2]))

def jzazbz_to_xyz(jzazbz):
    Jz, az, bz = jzazbz[:,0], jzazbz[:,1], jzazbz[:,2]
    d, d0 = -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    Jp = Jz + d0; Iz = Jp / (1 + d - d * Jp)
    izazbz_vec = np.stack([Iz, az, bz], axis=1)
    lms_prime = izazbz_vec @ np.linalg.inv(M2).T
    sign_lms = np.sign(lms_prime)
    Y = np.abs(lms_prime) ** (1/p)
    num = Y - c1; den = c2 - Y * c3
    An = np.clip(num / den, 0, None) 
    lms_norm = sign_lms * (An ** (1/n))
    lms = (lms_norm * 10000.0) / 200.0
    return lms @ np.linalg.inv(M1).T

def xyz_to_srgb_raw(xyz):
    M_inv = np.linalg.inv(np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]]))
    linear = xyz @ M_inv.T
    srgb = np.zeros_like(linear)
    mask = linear > 0.0031308
    srgb[mask] = 1.055 * (np.sign(linear[mask]) * np.abs(linear[mask]) ** (1/2.4)) - 0.055
    srgb[~mask] = 12.92 * linear[~mask]
    return srgb

def xyz_to_srgb(xyz): return np.clip(xyz_to_srgb_raw(xyz), 0, 1)
def srgb_to_linear_np(srgb): return np.where(srgb <= 0.04045, srgb / 12.92, ((srgb + 0.055) / 1.055) ** 2.4)
def linear_to_srgb_np(lin): 
    s = np.zeros_like(lin)
    mask = lin > 0.0031308
    s[mask] = 1.055 * (lin[mask]**(1/2.4)) - 0.055
    s[~mask] = 12.92 * lin[~mask]
    return np.clip(s, 0, 1)

# ==========================================
# 2. MacAdam Solid & Ridge
# ==========================================

def get_cmf_high_res(res=360):
    base_cmf = np.array([
        [0.0014,0.0000,0.0065], [0.0042,0.0001,0.0201], [0.0143,0.0004,0.0679], [0.0435,0.0012,0.2074],
        [0.1344,0.0040,0.6456], [0.2839,0.0116,1.3856], [0.3483,0.0230,1.7471], [0.3362,0.0380,1.7721],
        [0.2908,0.0600,1.6692], [0.1954,0.0910,1.2876], [0.0956,0.1390,0.8130], [0.0320,0.2080,0.4652],
        [0.0049,0.3230,0.2720], [0.0093,0.5030,0.1582], [0.0633,0.7100,0.0782], [0.1655,0.8620,0.0422],
        [0.2904,0.9540,0.0203], [0.4334,0.9950,0.0087], [0.5945,0.9950,0.0039], [0.7621,0.9520,0.0021],
        [0.9163,0.8700,0.0017], [1.0263,0.7570,0.0011], [1.0622,0.6310,0.0008], [1.0026,0.5030,0.0006],
        [0.8544,0.3810,0.0002], [0.6424,0.2650,0.0000], [0.4479,0.1750,0.0000], [0.2835,0.1070,0.0000],
        [0.1649,0.0610,0.0000], [0.0874,0.0320,0.0000], [0.0468,0.0170,0.0000], [0.0227,0.0082,0.0000],
        [0.0114,0.0041,0.0000], [0.0058,0.0021,0.0000], [0.0029,0.0010,0.0000], [0.0014,0.0005,0.0000]
    ])
    x = np.linspace(0, 1, len(base_cmf))
    xi = np.linspace(0, 1, res)
    interp_cmf = np.zeros((res, 3))
    for k in range(3):
        interp_cmf[:,k] = np.interp(xi, x, base_cmf[:,k])
    return interp_cmf

def compute_macadam_solid(res=400): 
    cmf = get_cmf_high_res(600)
    n_lambda = len(cmf)
    cum_cmf = np.vstack([np.zeros(3), np.cumsum(cmf, axis=0)])
    white_xyz = cum_cmf[-1]
    optimal_colors = []
    indices = np.linspace(0, n_lambda, res).astype(int)
    for i in indices:
        for j in indices:
            if i == j: continue
            if j > i: xyz = cum_cmf[j] - cum_cmf[i]
            else:     xyz = cum_cmf[j] + (white_xyz - cum_cmf[i])
            optimal_colors.append(xyz)
    optimal_colors.append([0,0,0])
    optimal_colors.append(white_xyz)
    return np.array(optimal_colors) / white_xyz[1]

def extract_macadam_ridge_dense(macadam_jz, res=3600): 
    az = macadam_jz[:, 1]; bz = macadam_jz[:, 2]
    chroma = np.sqrt(az**2 + bz**2); angle = np.arctan2(bz, az)
    bins = np.linspace(-np.pi, np.pi, res+1)
    ridge_points = []
    
    sort_idx = np.argsort(angle)
    angle_s = angle[sort_idx]; chroma_s = chroma[sort_idx]; pts_s = macadam_jz[sort_idx]
    bin_idx = np.searchsorted(angle_s, bins)
    
    for i in range(res):
        start, end = bin_idx[i], bin_idx[i+1]
        if start == end:
            if len(ridge_points)>0: ridge_points.append(ridge_points[-1])
            else: ridge_points.append(macadam_jz[0])
            continue
        idx_max = np.argmax(chroma_s[start:end])
        ridge_points.append(pts_s[start + idx_max])
        
    raw = np.array(ridge_points)
    return fourier_smooth_curve(raw, harmonics=20)

def fourier_smooth_curve(points, harmonics=15):
    N = len(points)
    new_cols = []
    for i in range(3):
        sig = points[:,i]
        coeffs = np.fft.rfft(sig)
        filt = np.zeros_like(coeffs)
        filt[:harmonics+1] = coeffs[:harmonics+1]
        new_cols.append(np.fft.irfft(filt, n=N))
    return np.column_stack(new_cols)

# ==========================================
# 3. Optimization & Projections
# ==========================================

def get_gamut_surface(res=40):
    t = np.linspace(0, 1, res)
    g = np.meshgrid(t, t)
    faces = [np.stack([g[0], g[1], np.zeros_like(g[0])], -1), np.stack([g[0], g[1], np.ones_like(g[0])], -1),
             np.stack([g[0], np.zeros_like(g[0]), g[1]], -1), np.stack([g[0], np.ones_like(g[0]), g[1]], -1),
             np.stack([np.zeros_like(g[0]), g[0], g[1]], -1), np.stack([np.ones_like(g[0]), g[0], g[1]], -1)]
    pts = np.vstack([f.reshape(-1, 3) for f in faces])
    corners = np.array([[0,0,0], [1,0,0], [0,1,0], [0,0,1], [1,1,0], [1,0,1], [0,1,1], [1,1,1]])
    return np.vstack([pts, corners])

def fit_ellipsoid_anchored_solver(points, anchor_point):
    hull = ConvexHull(points)
    A = hull.equations[:, :3]; b_const = -hull.equations[:, 3]
    scale = 100.0; A_scaled = A / scale; anchor_scaled = anchor_point * scale
    c0 = np.mean(points, axis=0) * scale
    x0 = np.concatenate([0.5*c0 + 0.5*anchor_scaled, np.eye(3).flatten() * 0.05])
    def objective(x):
        M = x[3:].reshape(3, 3); sign, val = np.linalg.slogdet(M)
        return -val if sign > 0 else 1e9
    def constraints(x):
        d, M = x[:3], x[3:].reshape(3, 3)
        norm_AM = np.linalg.norm(A_scaled @ M, axis=1)
        hull_con = b_const - (A_scaled @ d + norm_AM)
        try: u = np.linalg.solve(M, anchor_scaled - d); anchor_con = 1.0 - np.linalg.norm(u)
        except: anchor_con = -1.0
        return np.concatenate([hull_con, [anchor_con]])
    res = opt.minimize(objective, x0, method='SLSQP', constraints={'type': 'ineq', 'fun': constraints}, options={'maxiter': 1000})
    return res.x[:3]/scale, res.x[3:].reshape(3, 3)/scale, hull

def calculate_shadow_boundary(center, M, view_vector, res=72):
    w = np.linalg.solve(M, view_vector); w = w / np.linalg.norm(w)
    tmp = np.array([0.0, 1.0, 0.0]) if np.abs(w[1]) < 0.9 else np.array([0.0, 0.0, 1.0])
    s1 = np.cross(w, tmp); s1 /= np.linalg.norm(s1); s2 = np.cross(w, s1)
    theta = np.linspace(0, 2*np.pi, res)
    return center + (np.outer(np.cos(theta), s1) + np.outer(np.sin(theta), s2)) @ M.T

def optimize_ellipsoid_scale(center, M_shape):
    """
    Scales M_shape (from 0.0) until it hits the sRGB wall.
    Ignores hull inaccuracies by testing against true sRGB math.
    """
    print("   Optimizing Ellipsoid Scale...")
    
    # Binary Search for max valid scale
    low = 0.0
    high = 3.0
    best_s = 0.0
    
    for _ in range(25):
        mid = (low + high) / 2.0
        M_test = M_shape * mid
        hoop = calculate_shadow_boundary(center, M_test, np.array([1.0, 0.0, 0.0]), res=72)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
        
        # Allow small epsilon
        if np.min(rgb) >= -0.001 and np.max(rgb) <= 1.001:
            best_s = mid
            low = mid
        else:
            high = mid
            
    return M_shape * best_s

def project_points_to_srgb_surface_conical(points_jz, center_jz):
    projected = []
    for pt in points_jz:
        vec = pt - center_jz
        low = 0.0; high = 1.5; best_t = 0.0
        for _ in range(15):
            mid = (low + high) / 2.0
            test = center_jz + mid * vec
            rgb = xyz_to_srgb_raw(jzazbz_to_xyz(np.array([test])))
            if np.min(rgb) >= -0.001 and np.max(rgb) <= 1.001:
                best_t = mid; low = mid 
            else: high = mid 
        projected.append(center_jz + best_t * vec)
    return np.array(projected)

def project_ridge_onto_ellipsoid(ridge_jz, center, M):
    vecs = ridge_jz - center
    M_inv = np.linalg.inv(M)
    u_vecs = vecs @ M_inv.T
    norms = np.linalg.norm(u_vecs, axis=1, keepdims=True)
    u_surface = u_vecs / norms
    return (u_surface @ M.T) + center

def shrink_ridge_uniform(ridge_jz, center_point):
    scale = 1.0; step = 0.002
    for _ in range(500):
        test_ridge = center_point + scale * (ridge_jz - center_point)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(test_ridge))
        if np.mean(np.min(rgb, axis=1) >= -0.005) > 0.98: 
             if np.mean(np.max(rgb, axis=1) <= 1.005) > 0.98: return test_ridge
        scale -= step
    return ridge_jz * 0.1

def reparameterize_by_hue_angle(points, n_samples=256):
    az = points[:, 1]; bz = points[:, 2]
    angles = np.arctan2(bz, az)
    angles = np.unwrap(angles)
    target_angles = np.linspace(angles[0], angles[0] + 2*np.pi, n_samples, endpoint=False)
    angles_pad = np.concatenate([angles - 2*np.pi, angles, angles + 2*np.pi])
    J_pad = np.tile(points[:, 0], 3)
    a_pad = np.tile(points[:, 1], 3)
    b_pad = np.tile(points[:, 2], 3)
    return np.column_stack([
        np.interp(target_angles, angles_pad, J_pad),
        np.interp(target_angles, angles_pad, a_pad),
        np.interp(target_angles, angles_pad, b_pad)
    ])

def fourier_approx_curve(points, harmonics=2):
    N = len(points)
    new_cols = []
    for i in range(3):
        sig = points[:,i]
        coeffs = np.fft.rfft(sig)
        filt = np.zeros_like(coeffs)
        filt[:harmonics+1] = coeffs[:harmonics+1]
        new_cols.append(np.fft.irfft(filt, n=N))
    return np.column_stack(new_cols)

def fit_fourier_constrained(points, center_point, harmonics=3, iterations=50):
    print(f"   Fitting Constrained Fourier (N={harmonics})...")
    current_points = points.copy()
    N = len(points)
    for i in range(iterations):
        new_cols = []
        for d in range(3):
            sig = current_points[:,d]
            coeffs = np.fft.rfft(sig)
            filt = np.zeros_like(coeffs)
            filt[:harmonics+1] = coeffs[:harmonics+1]
            new_cols.append(np.fft.irfft(filt, n=N))
        smooth_points = np.column_stack(new_cols)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(smooth_points))
        is_out = (rgb < -0.002) | (rgb > 1.002)
        out_idx = np.any(is_out, axis=1)
        if not np.any(out_idx): return smooth_points
        vecs = center_point - smooth_points
        smooth_points[out_idx] += vecs[out_idx] * 0.2 
        current_points = smooth_points
    
    # Final Safety Shrink
    scale = 1.0
    for _ in range(100):
        test_curve = center_point + scale * (current_points - center_point)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(test_curve))
        if np.min(rgb) >= -0.005 and np.max(rgb) <= 1.005: return test_curve
        scale -= 0.005
    return current_points

def check_gamut_compliance(rgb_loop, name="Loop"):
    min_val = np.min(rgb_loop); max_val = np.max(rgb_loop)
    tol = 2e-3
    if min_val < -tol or max_val > 1.0 + tol:
        print(f"   >>> {name} FAIL: [{min_val:.4f}, {max_val:.4f}]")
    else:
        print(f"   >>> {name} PASS: Inside sRGB")

# ==========================================
# 4. Omnipose Helpers
# ==========================================
def align_ring_red_to_green(ring_lin):
    n = len(ring_lin)
    red_ref = np.array([1.0, 0.0, 0.0])
    idx_r = np.argmin(np.linalg.norm(ring_lin - red_ref, axis=1))
    ring_aligned = np.roll(ring_lin, -idx_r, axis=0)
    green_ref = np.array([0.0, 1.0, 0.0])
    if np.linalg.norm(ring_aligned[(2*n)//3]-green_ref) < np.linalg.norm(ring_aligned[n//3]-green_ref):
        ring_aligned = ring_aligned[::-1]
    return ring_aligned

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    def build_disk(ring, center):
        y, x = np.mgrid[-1:1:256j, -1:1:256j]
        mag = np.sqrt(x*x + y*y); angle = np.mod(np.arctan2(y, x), 2*np.pi)
        n = ring.shape[0]; hue_f = angle/(2*np.pi)*n
        idx0 = np.floor(hue_f).astype(int) % n; idx1 = (idx0 + 1) % n; t = hue_f - np.floor(hue_f)
        interp = (1-t[...,None])*ring[idx0] + t[...,None]*ring[idx1]
        rgb = (1-mag[...,None])*center + mag[...,None]*interp
        return np.clip(linear_to_srgb_np(rgb),0,1)
    disk = build_disk(rgb_ring_lin, center_lin)
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi); mag = np.clip(np.sqrt(u*u + v*v), 0, 1)
    n = rgb_ring_lin.shape[0]; hue_f = angle/(2*np.pi)*n
    idx0 = np.floor(hue_f).astype(int) % n; idx1 = (idx0 + 1) % n; t = hue_f - np.floor(hue_f)
    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    r = mag[..., None]; rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow),0,1)
    alpha = mag[...,None]; white = np.ones_like(flow_rgb); black = np.zeros_like(flow_rgb)
    flow_white = alpha*flow_rgb + (1-alpha)*white; flow_black = alpha*flow_rgb + (1-alpha)*black
    return disk, flow_rgb, flow_white, flow_black

# ==========================================
# 5. Main Execution
# ==========================================

def main():
    N_HARMONICS = 2 # Set to 2 per request
    device_str = "cpu"
    dev = torch.device(device_str)
    
    print("1. Calculating Geometries...")
    surf_pts_opt = get_gamut_surface(40) 
    jz_gamut_opt = xyz_to_jzazbz(srgb_to_xyz(surf_pts_opt))
    centroid = np.mean(jz_gamut_opt, axis=0)
    green_vertex = xyz_to_jzazbz(srgb_to_xyz(np.array([[0.0, 1.0, 0.0]])))[0]
    anchor = centroid + 0.60 * (green_vertex - centroid)
    
    # 2. Ellipsoid Fitting
    print("2. Fitting Maximal Ellipsoid...")
    center, M_solver, hull = fit_ellipsoid_anchored_solver(jz_gamut_opt, anchor)
    
    # Use robust binary search to find maximal fit
    M_max = optimize_ellipsoid_scale(center, M_solver)
    
    hoop = calculate_shadow_boundary(center, M_max, np.array([1.0, 0.0, 0.0]), res=72)
    rgb_hoop_raw = xyz_to_srgb_raw(jzazbz_to_xyz(hoop))
    check_gamut_compliance(rgb_hoop_raw, "Ellipsoid Loop")
    rgb_hoop_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(rgb_hoop_raw,0,1)))
    
    # 3. Sinebow
    t = np.linspace(0, 1, 256, endpoint=False)
    sr = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0/3)); sg = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1/3)); sb = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2/3))
    rgb_sine_std = np.stack([sr, sg, sb], axis=1)
    rgb_sine_lin = align_ring_red_to_green(srgb_to_linear_np(rgb_sine_std))
    
    # 4. MacAdam Ridge
    print("3. Computing MacAdam Ridge...")
    macadam_xyz = compute_macadam_solid(res=200)
    macadam_jz = xyz_to_jzazbz(macadam_xyz)
    ridge_jz_highres = extract_macadam_ridge_dense(macadam_jz, res=3600)
    
    # Ridge -> Ellipsoid
    proj_ridge_jz_raw = project_ridge_onto_ellipsoid(ridge_jz_highres, center, M_max)
    proj_ridge_jz_ang = reparameterize_by_hue_angle(proj_ridge_jz_raw, n_samples=256)
    rgb_proj_ell_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(proj_ridge_jz_ang)), 0, 1)))
    
    # Ridge -> sRGB (Conical)
    proj_srgb_jz_raw = project_points_to_srgb_surface_conical(ridge_jz_highres, centroid) 
    proj_srgb_jz_ang = reparameterize_by_hue_angle(proj_srgb_jz_raw, n_samples=256)
    rgb_proj_srgb_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(proj_srgb_jz_ang)), 0, 1)))
    
    # Fourier Constrained
    fourier_jz = fit_fourier_constrained(proj_srgb_jz_ang, centroid, harmonics=N_HARMONICS)
    fourier_jz_ang = reparameterize_by_hue_angle(fourier_jz, n_samples=256)
    rgb_fourier_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(fourier_jz_ang)), 0, 1)))
    
    # Uniform Shrink
    shrunk_ridge_jz_raw = shrink_ridge_uniform(ridge_jz_highres, centroid)
    shrunk_ridge_jz_ang = reparameterize_by_hue_angle(shrunk_ridge_jz_raw, n_samples=256)
    rgb_shrunk_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(shrunk_ridge_jz_ang)), 0, 1)))

    print("4. Loading Omnipose Flows...")
    omnidir = Path(omnirefactor.__file__).resolve().parent.parent.parent
    basedir = omnidir / "docs" / "_static"
    name = "ecoli"
    ext = ".tif"

    try:
        masks = io.imread(str(basedir / f"{name}_labels{ext}"))
        slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows = omnirefactor.core.masks_to_flows(masks, device=device_str)[4].to(dev)
        center_lin = srgb_to_linear_np(np.array([0.5, 0.5, 0.5]))

        def gen_set(ring, rot_steps=0):
            idx = int(rot_steps * len(ring) / 72.0)
            r = np.roll(ring, idx, axis=0)
            return make_flow_images_for_ring(r, center_lin, flows)

        rows = []
        d, f, w, b = gen_set(rgb_sine_lin, 0)
        rows.append(("Sinebow", rgb_sine_lin, d, b, w))
        d, f, w, b = gen_set(rgb_hoop_lin, 0)
        rows.append(("Max Ellipse", rgb_hoop_lin, d, b, w))
        d, f, w, b = gen_set(rgb_proj_ell_lin, 0)
        rows.append(("Ridge->Ellipsoid", rgb_proj_ell_lin, d, b, w))
        d, f, w, b = gen_set(rgb_proj_srgb_lin, 0)
        rows.append(("Ridge->sRGB", rgb_proj_srgb_lin, d, b, w))
        d, f, w, b = gen_set(rgb_fourier_lin, 0)
        rows.append((f"Fourier (N={N_HARMONICS})", rgb_fourier_lin, d, b, w))
        d, f, w, b = gen_set(rgb_shrunk_lin, 0)
        rows.append(("Uniform Shrink", rgb_shrunk_lin, d, b, w))

        print("5. Plotting 2D...")
        plt.style.use('dark_background')
        fig, axes = plt.subplots(6, 4, figsize=(16, 24))
        titles = ["Gradient", "Hue Disk", "Black (0°)", "White (0°)"]
        for i, (name, ring, disk, b, w) in enumerate(rows):
            ax_row = axes[i]
            ax_row[0].imshow(linear_to_srgb_np(ring)[None, ...], aspect='auto')
            ax_row[0].set_ylabel(name, fontsize=14, color='white', rotation=90, labelpad=20)
            ax_row[1].imshow(disk); ax_row[2].imshow(b); ax_row[3].imshow(w)
            for j, ax in enumerate(ax_row): ax.set_xticks([]); ax.set_yticks([])
        plt.tight_layout()
        plt.show()
        
        print("6. Generating 3D...")
        fig3d = go.Figure()
        max_jz = np.max(jz_gamut_opt[:,0]); sc = np.array([1.0/max_jz, 1.0, 1.0]); 
        def mc(a): return a[:,1], a[:,2], a[:,0]

        hull_macadam = ConvexHull(macadam_jz)
        mx, my, mz = mc(macadam_jz * sc)
        fig3d.add_trace(go.Mesh3d(x=mx, y=my, z=mz, i=hull_macadam.simplices[:,0], j=hull_macadam.simplices[:,1], k=hull_macadam.simplices[:,2], color='white', opacity=0.05, name='Visible Gamut'))

        corners = [[0,0,0],[1,0,0],[1,1,0],[0,1,0],[0,0,1],[1,0,1],[1,1,1],[0,1,1]]
        edges_idx = [[0,1],[1,2],[2,3],[3,0],[4,5],[5,6],[6,7],[7,4],[0,4],[1,5],[2,6],[3,7]]
        for s,e in edges_idx:
            l = xyz_to_jzazbz(srgb_to_xyz(np.linspace(corners[s],corners[e],25))) * sc
            x,y,z = mc(l)
            fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', line=dict(color='gray', width=2), showlegend=False))

        u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
        sph = np.stack([np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v)], axis=-1).reshape(-1, 3)
        ell = (sph @ M_max.T + center) * sc
        ex, ey, ez = mc(ell)
        fig3d.add_trace(go.Surface(x=ex.reshape(u.shape), y=ey.reshape(u.shape), z=ez.reshape(u.shape), colorscale=[(0,'#88aa88'),(1,'#88aa88')], opacity=0.2, showscale=False, name='Ellipsoid'))

        def plot_dots(pts, col, name):
             x,y,z = mc(pts * sc)
             rgb = linear_to_srgb_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(pts)), 0, 1))
             cols = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in rgb]
             fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='markers', marker=dict(color=cols, size=4), name=name))

        plot_dots(hoop, 'orange', 'Ellipsoid Loop')
        plot_dots(xyz_to_jzazbz(srgb_to_xyz(rgb_sine_std)), 'green', 'Sinebow Loop')
        plot_dots(proj_srgb_jz_ang, 'red', 'Ridge->sRGB')
        plot_dots(fourier_jz_ang, 'cyan', 'Constrained Fourier')
        plot_dots(shrunk_ridge_jz_ang, 'magenta', 'Uniform Shrink')

        rrx, rry, rrz = mc(ridge_jz_highres[::10] * sc)
        fig3d.add_trace(go.Scatter3d(x=rrx, y=rry, z=rrz, mode='lines', line=dict(color='white', width=3), name='MacAdam Ridge (Ref)'))

        fig3d.update_layout(template="plotly_dark", title=f"3D Jzazbz: Final Comparison", 
                            scene=dict(xaxis_title='az', yaxis_title='bz', zaxis_title='Jz', aspectmode='manual', aspectratio=dict(x=1,y=1,z=1)))
        fig3d.show()

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import scipy.optimize as opt
import scipy.ndimage
from scipy.spatial import ConvexHull
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import omnirefactor.core
from skimage import io
import fastremap

# ==========================================
# 1. Color Math
# ==========================================
def srgb_to_xyz(rgb):
    mask = rgb > 0.04045
    linear = np.zeros_like(rgb)
    linear[mask] = ((rgb[mask] + 0.055) / 1.055) ** 2.4
    linear[~mask] = rgb[~mask] / 12.92
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    return linear @ M.T

def xyz_to_jzazbz(xyz):
    b, g, d, d0 = 1.15, 0.66, -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    lms = xyz @ M1.T
    lms_norm = (lms * 200) / 10000.0 
    lms_prime = np.sign(lms_norm) * (((c1 + c2 * (np.abs(lms_norm) ** n)) / (1 + c3 * (np.abs(lms_norm) ** n))) ** p)
    izazbz = lms_prime @ M2.T
    Jz = ((1 + d) * izazbz[:, 0]) / (1 + d * izazbz[:, 0]) - d0
    return np.column_stack((Jz, izazbz[:, 1], izazbz[:, 2]))

def jzazbz_to_xyz(jzazbz):
    Jz, az, bz = jzazbz[:,0], jzazbz[:,1], jzazbz[:,2]
    d, d0 = -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    Jp = Jz + d0; Iz = Jp / (1 + d - d * Jp)
    izazbz_vec = np.stack([Iz, az, bz], axis=1)
    lms_prime = izazbz_vec @ np.linalg.inv(M2).T
    sign_lms = np.sign(lms_prime)
    Y = np.abs(lms_prime) ** (1/p)
    num = Y - c1; den = c2 - Y * c3
    An = np.clip(num / den, 0, None) 
    lms_norm = sign_lms * (An ** (1/n))
    lms = (lms_norm * 10000.0) / 200.0
    return lms @ np.linalg.inv(M1).T

def xyz_to_srgb_raw(xyz):
    M_inv = np.linalg.inv(np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]]))
    linear = xyz @ M_inv.T
    srgb = np.zeros_like(linear)
    mask = linear > 0.0031308
    srgb[mask] = 1.055 * (np.sign(linear[mask]) * np.abs(linear[mask]) ** (1/2.4)) - 0.055
    srgb[~mask] = 12.92 * linear[~mask]
    return srgb

def xyz_to_srgb(xyz): return np.clip(xyz_to_srgb_raw(xyz), 0, 1)
def srgb_to_linear_np(srgb): return np.where(srgb <= 0.04045, srgb / 12.92, ((srgb + 0.055) / 1.055) ** 2.4)
def linear_to_srgb_np(lin): 
    s = np.zeros_like(lin)
    mask = lin > 0.0031308
    s[mask] = 1.055 * (lin[mask]**(1/2.4)) - 0.055
    s[~mask] = 12.92 * lin[~mask]
    return np.clip(s, 0, 1)

# ==========================================
# 2. MacAdam Solid & Ridge
# ==========================================

def get_cmf_high_res(res=360):
    base_cmf = np.array([
        [0.0014,0.0000,0.0065], [0.0042,0.0001,0.0201], [0.0143,0.0004,0.0679], [0.0435,0.0012,0.2074],
        [0.1344,0.0040,0.6456], [0.2839,0.0116,1.3856], [0.3483,0.0230,1.7471], [0.3362,0.0380,1.7721],
        [0.2908,0.0600,1.6692], [0.1954,0.0910,1.2876], [0.0956,0.1390,0.8130], [0.0320,0.2080,0.4652],
        [0.0049,0.3230,0.2720], [0.0093,0.5030,0.1582], [0.0633,0.7100,0.0782], [0.1655,0.8620,0.0422],
        [0.2904,0.9540,0.0203], [0.4334,0.9950,0.0087], [0.5945,0.9950,0.0039], [0.7621,0.9520,0.0021],
        [0.9163,0.8700,0.0017], [1.0263,0.7570,0.0011], [1.0622,0.6310,0.0008], [1.0026,0.5030,0.0006],
        [0.8544,0.3810,0.0002], [0.6424,0.2650,0.0000], [0.4479,0.1750,0.0000], [0.2835,0.1070,0.0000],
        [0.1649,0.0610,0.0000], [0.0874,0.0320,0.0000], [0.0468,0.0170,0.0000], [0.0227,0.0082,0.0000],
        [0.0114,0.0041,0.0000], [0.0058,0.0021,0.0000], [0.0029,0.0010,0.0000], [0.0014,0.0005,0.0000]
    ])
    x = np.linspace(0, 1, len(base_cmf))
    xi = np.linspace(0, 1, res)
    interp_cmf = np.zeros((res, 3))
    for k in range(3):
        interp_cmf[:,k] = np.interp(xi, x, base_cmf[:,k])
    return interp_cmf

def compute_macadam_solid(res=300):
    cmf = get_cmf_high_res(600)
    n_lambda = len(cmf)
    cum_cmf = np.vstack([np.zeros(3), np.cumsum(cmf, axis=0)])
    white_xyz = cum_cmf[-1]
    optimal_colors = []
    indices = np.linspace(0, n_lambda, res).astype(int)
    for i in indices:
        for j in indices:
            if i == j: continue
            if j > i: xyz = cum_cmf[j] - cum_cmf[i]
            else:     xyz = cum_cmf[j] + (white_xyz - cum_cmf[i])
            optimal_colors.append(xyz)
    optimal_colors.append([0,0,0]); optimal_colors.append(white_xyz)
    return np.array(optimal_colors) / white_xyz[1]

def fourier_smooth_curve(points, harmonics=15):
    N = len(points)
    new_cols = []
    for i in range(3):
        sig = points[:,i]
        coeffs = np.fft.rfft(sig)
        filt = np.zeros_like(coeffs)
        filt[:harmonics+1] = coeffs[:harmonics+1]
        new_cols.append(np.fft.irfft(filt, n=N))
    return np.column_stack(new_cols)

def extract_macadam_ridge_dense(macadam_jz, res=3600): 
    az = macadam_jz[:, 1]; bz = macadam_jz[:, 2]
    chroma = np.sqrt(az**2 + bz**2); angle = np.arctan2(bz, az)
    bins = np.linspace(-np.pi, np.pi, res+1)
    sort_idx = np.argsort(angle)
    angle_s = angle[sort_idx]; chroma_s = chroma[sort_idx]; pts_s = macadam_jz[sort_idx]
    bin_idx = np.searchsorted(angle_s, bins)
    ridge_points = []
    for i in range(res):
        start, end = bin_idx[i], bin_idx[i+1]
        if start == end:
            if len(ridge_points)>0: ridge_points.append(ridge_points[-1])
            else: ridge_points.append(macadam_jz[0])
            continue
        idx_max = np.argmax(chroma_s[start:end])
        ridge_points.append(pts_s[start + idx_max])
    raw = np.array(ridge_points)
    # Pre-smooth with Fourier to kill quantization noise
    return fourier_smooth_curve(raw, harmonics=15)

# ==========================================
# 3. Optimization & Fitting
# ==========================================

def get_gamut_surface(res=40):
    t = np.linspace(0, 1, res); g = np.meshgrid(t, t)
    faces = [np.stack([g[0], g[1], np.zeros_like(g[0])], -1), np.stack([g[0], g[1], np.ones_like(g[0])], -1),
             np.stack([g[0], np.zeros_like(g[0]), g[1]], -1), np.stack([g[0], np.ones_like(g[0]), g[1]], -1),
             np.stack([np.zeros_like(g[0]), g[0], g[1]], -1), np.stack([np.ones_like(g[0]), g[0], g[1]], -1)]
    pts = np.vstack([f.reshape(-1, 3) for f in faces])
    corners = np.array([[0,0,0], [1,0,0], [0,1,0], [0,0,1], [1,1,0], [1,0,1], [0,1,1], [1,1,1]])
    return np.vstack([pts, corners])

def fit_ellipsoid_anchored_solver(points, anchor_point):
    hull = ConvexHull(points)
    A = hull.equations[:, :3]; b_const = -hull.equations[:, 3]
    scale = 100.0; A_scaled = A / scale; anchor_scaled = anchor_point * scale
    c0 = np.mean(points, axis=0) * scale
    x0 = np.concatenate([0.5*c0 + 0.5*anchor_scaled, np.eye(3).flatten() * 0.05])
    def objective(x):
        M = x[3:].reshape(3, 3); sign, val = np.linalg.slogdet(M)
        return -val if sign > 0 else 1e9
    def constraints(x):
        d, M = x[:3], x[3:].reshape(3, 3)
        norm_AM = np.linalg.norm(A_scaled @ M, axis=1)
        hull_con = b_const - (A_scaled @ d + norm_AM)
        try: u = np.linalg.solve(M, anchor_scaled - d); anchor_con = 1.0 - np.linalg.norm(u)
        except: anchor_con = -1.0
        return np.concatenate([hull_con, [anchor_con]])
    res = opt.minimize(objective, x0, method='SLSQP', constraints={'type': 'ineq', 'fun': constraints}, options={'maxiter': 1000})
    return res.x[:3]/scale, res.x[3:].reshape(3, 3)/scale, hull

def optimize_maximal_loop(center, M_basis):
    """
    Optimizes a 2D ellipse (U, V) to maximally fill the gamut.
    Stretches width/height independently.
    """
    print("   Optimizing 2D Loop Dimensions...")
    # 1. Get basis vectors of the ellipsoid plane
    # Plane is orthogonal to [1,0,0] in M-space
    view_vec = np.array([1.0, 0.0, 0.0])
    w = np.linalg.solve(M_basis, view_vec); w = w / np.linalg.norm(w)
    tmp = np.array([0.0, 1.0, 0.0]) if np.abs(w[1]) < 0.9 else np.array([0.0, 0.0, 1.0])
    s1 = np.cross(w, tmp); s1 /= np.linalg.norm(s1)
    s2 = np.cross(w, s1)
    
    U_world = s1 @ M_basis.T
    V_world = s2 @ M_basis.T
    
    # 2. Grid Search for Max Scale
    # We check scale factors for U and V
    best_su, best_sv = 0.0, 0.0
    theta = np.linspace(0, 2*np.pi, 128)
    cos_t = np.cos(theta); sin_t = np.sin(theta)
    
    # Coarse search
    for su in np.linspace(0.5, 3.0, 30):
        for sv in np.linspace(0.5, 3.0, 30):
            loop = center + np.outer(cos_t, U_world * su) + np.outer(sin_t, V_world * sv)
            rgb = xyz_to_srgb_raw(jzazbz_to_xyz(loop))
            if np.min(rgb) >= -0.001 and np.max(rgb) <= 1.001:
                if (su * sv) > (best_su * best_sv):
                    best_su, best_sv = su, sv
                    
    # Fine tune
    print(f"      Refining from {best_su:.2f}, {best_sv:.2f}...")
    current_su, current_sv = best_su, best_sv
    for _ in range(20):
        # Try growing
        test_su = current_su + 0.02
        loop = center + np.outer(cos_t, U_world * test_su) + np.outer(sin_t, V_world * current_sv)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(loop))
        if np.min(rgb) >= -0.001 and np.max(rgb) <= 1.001:
            current_su = test_su
            
        test_sv = current_sv + 0.02
        loop = center + np.outer(cos_t, U_world * current_su) + np.outer(sin_t, V_world * test_sv)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(loop))
        if np.min(rgb) >= -0.001 and np.max(rgb) <= 1.001:
            current_sv = test_sv
            
    # Final high res loop
    theta_HR = np.linspace(0, 2*np.pi, 256)
    final_loop = center + np.outer(np.cos(theta_HR), U_world * current_su) + np.outer(np.sin(theta_HR), V_world * current_sv)
    return final_loop

def project_points_to_srgb_surface_conical(points_jz, center_jz):
    projected = []
    for pt in points_jz:
        vec = pt - center_jz
        low = 0.0; high = 1.5; best_t = 0.0
        for _ in range(15):
            mid = (low + high) / 2.0
            test = center_jz + mid * vec
            rgb = xyz_to_srgb_raw(jzazbz_to_xyz(np.array([test])))
            if np.min(rgb) >= -0.001 and np.max(rgb) <= 1.001:
                best_t = mid; low = mid 
            else: high = mid 
        projected.append(center_jz + best_t * vec)
    return np.array(projected)

def project_ridge_onto_ellipsoid(ridge_jz, center, M):
    vecs = ridge_jz - center
    M_inv = np.linalg.inv(M)
    u_vecs = vecs @ M_inv.T
    norms = np.linalg.norm(u_vecs, axis=1, keepdims=True)
    u_surface = u_vecs / norms
    return (u_surface @ M.T) + center

def reparameterize_by_hue_angle(points, n_samples=256):
    az = points[:, 1]; bz = points[:, 2]
    angles = np.arctan2(bz, az)
    angles = np.unwrap(angles)
    target_angles = np.linspace(angles[0], angles[0] + 2*np.pi, n_samples, endpoint=False)
    angles_pad = np.concatenate([angles - 2*np.pi, angles, angles + 2*np.pi])
    J_pad = np.tile(points[:, 0], 3)
    a_pad = np.tile(points[:, 1], 3)
    b_pad = np.tile(points[:, 2], 3)
    return np.column_stack([
        np.interp(target_angles, angles_pad, J_pad),
        np.interp(target_angles, angles_pad, a_pad),
        np.interp(target_angles, angles_pad, b_pad)
    ])

def fit_fourier_constrained(points, center_point, harmonics=2, iterations=50):
    # Constrained Fourier Smoothing
    print(f"   Fitting Constrained Fourier (N={harmonics})...")
    current_points = points.copy()
    N = len(points)
    for i in range(iterations):
        # Smooth
        new_cols = []
        for d in range(3):
            sig = current_points[:,d]
            coeffs = np.fft.rfft(sig)
            filt = np.zeros_like(coeffs)
            filt[:harmonics+1] = coeffs[:harmonics+1]
            new_cols.append(np.fft.irfft(filt, n=N))
        smooth_points = np.column_stack(new_cols)
        
        # Check constraints
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(smooth_points))
        is_out = (rgb < -0.002) | (rgb > 1.002)
        out_idx = np.any(is_out, axis=1)
        if not np.any(out_idx): return smooth_points
        
        # Push violators IN
        vecs = center_point - smooth_points
        smooth_points[out_idx] += vecs[out_idx] * 0.2
        current_points = smooth_points

    # Final safety shrink
    scale = 1.0
    for _ in range(100):
        test = center_point + scale * (current_points - center_point)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(test))
        if np.min(rgb) >= -0.001 and np.max(rgb) <= 1.001: return test
        scale -= 0.005
    return current_points

def check_gamut_compliance(rgb_loop, name="Loop"):
    min_val = np.min(rgb_loop); max_val = np.max(rgb_loop)
    tol = 2e-3
    if min_val < -tol or max_val > 1.0 + tol:
        print(f"   >>> {name} FAIL: [{min_val:.4f}, {max_val:.4f}]")
    else:
        print(f"   >>> {name} PASS: Inside sRGB")

# ==========================================
# 4. Omnipose Helpers
# ==========================================
def align_ring_red_to_green(ring_lin):
    n = len(ring_lin)
    red_ref = np.array([1.0, 0.0, 0.0])
    idx_r = np.argmin(np.linalg.norm(ring_lin - red_ref, axis=1))
    ring_aligned = np.roll(ring_lin, -idx_r, axis=0)
    green_ref = np.array([0.0, 1.0, 0.0])
    if np.linalg.norm(ring_aligned[(2*n)//3]-green_ref) < np.linalg.norm(ring_aligned[n//3]-green_ref):
        ring_aligned = ring_aligned[::-1]
    return ring_aligned

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    def build_disk(ring, center):
        y, x = np.mgrid[-1:1:256j, -1:1:256j]
        mag = np.sqrt(x*x + y*y); angle = np.mod(np.arctan2(y, x), 2*np.pi)
        n = ring.shape[0]; hue_f = angle/(2*np.pi)*n
        idx0 = np.floor(hue_f).astype(int) % n; idx1 = (idx0 + 1) % n; t = hue_f - np.floor(hue_f)
        interp = (1-t[...,None])*ring[idx0] + t[...,None]*ring[idx1]
        rgb = (1-mag[...,None])*center + mag[...,None]*interp
        return np.clip(linear_to_srgb_np(rgb),0,1)
    disk = build_disk(rgb_ring_lin, center_lin)
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi); mag = np.clip(np.sqrt(u*u + v*v), 0, 1)
    n = rgb_ring_lin.shape[0]; hue_f = angle/(2*np.pi)*n
    idx0 = np.floor(hue_f).astype(int) % n; idx1 = (idx0 + 1) % n; t = hue_f - np.floor(hue_f)
    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    r = mag[..., None]; rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow),0,1)
    alpha = mag[...,None]; white = np.ones_like(flow_rgb); black = np.zeros_like(flow_rgb)
    flow_white = alpha*flow_rgb + (1-alpha)*white; flow_black = alpha*flow_rgb + (1-alpha)*black
    return disk, flow_rgb, flow_white, flow_black

# ==========================================
# 5. Main Execution
# ==========================================

def main():
    N_HARMONICS = 2
    device_str = "cpu"
    dev = torch.device(device_str)
    
    print("1. Calculating Geometries...")
    surf_pts_opt = get_gamut_surface(40) 
    jz_gamut_opt = xyz_to_jzazbz(srgb_to_xyz(surf_pts_opt))
    centroid = np.mean(jz_gamut_opt, axis=0)
    green_vertex = xyz_to_jzazbz(srgb_to_xyz(np.array([[0.0, 1.0, 0.0]])))[0]
    anchor = centroid + 0.60 * (green_vertex - centroid)
    
    # 2. Ellipsoid Optimization
    print("2. Maximizing Ellipsoid Loop...")
    center, M_solver, hull = fit_ellipsoid_anchored_solver(jz_gamut_opt, anchor)
    # STRETCH 2D Loop to Limits
    max_loop_jz = optimize_loop_dimensions(center, M_solver)
    # Reparameterize angularly
    max_loop_ang = reparameterize_by_hue_angle(max_loop_jz, n_samples=256)
    
    rgb_hoop_raw = xyz_to_srgb_raw(jzazbz_to_xyz(max_loop_ang))
    check_gamut_compliance(rgb_hoop_raw, "Maximal Loop")
    rgb_hoop_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(rgb_hoop_raw,0,1)))
    
    # 3. Sinebow
    t = np.linspace(0, 1, 256, endpoint=False)
    sr = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0/3)); sg = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1/3)); sb = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2/3))
    rgb_sine_std = np.stack([sr, sg, sb], axis=1)
    rgb_sine_lin = align_ring_red_to_green(srgb_to_linear_np(rgb_sine_std))
    
    # 4. MacAdam Ridge
    print("3. Computing MacAdam Ridge...")
    macadam_xyz = compute_macadam_solid(res=200)
    macadam_jz = xyz_to_jzazbz(macadam_xyz)
    # Pre-smooth clean source
    ridge_jz_clean = extract_macadam_ridge_dense(macadam_jz, res=3600)
    
    # Ridge -> Ellipsoid (Project onto stretched M_solver shape for orientation)
    proj_ridge_jz_raw = project_ridge_onto_ellipsoid(ridge_jz_clean, center, M_solver)
    proj_ridge_jz_ang = reparameterize_by_hue_angle(proj_ridge_jz_raw, n_samples=256)
    rgb_proj_ell_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(proj_ridge_jz_ang)), 0, 1)))
    
    # Ridge -> sRGB (Conical)
    proj_srgb_jz_raw = project_points_to_srgb_surface_conical(ridge_jz_clean, centroid) 
    proj_srgb_jz_ang = reparameterize_by_hue_angle(proj_srgb_jz_raw, n_samples=256)
    rgb_proj_srgb_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(proj_srgb_jz_ang)), 0, 1)))
    
    # Fourier Constrained
    fourier_jz = fit_fourier_constrained(proj_srgb_jz_ang, centroid, harmonics=N_HARMONICS)
    fourier_jz_ang = reparameterize_by_hue_angle(fourier_jz, n_samples=256)
    rgb_fourier_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(fourier_jz_ang)), 0, 1)))
    
    print("4. Loading Omnipose Flows...")
    omnidir = Path(omnirefactor.__file__).resolve().parent.parent.parent
    basedir = omnidir / "docs" / "_static"
    name = "ecoli"
    ext = ".tif"

    try:
        masks = io.imread(str(basedir / f"{name}_labels{ext}"))
        slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows = omnirefactor.core.masks_to_flows(masks, device=device_str)[4].to(dev)
        center_lin = srgb_to_linear_np(np.array([0.5, 0.5, 0.5]))

        def gen_set(ring, rot_steps=0):
            idx = int(rot_steps * len(ring) / 72.0)
            r = np.roll(ring, idx, axis=0)
            return make_flow_images_for_ring(r, center_lin, flows)

        rows = []
        d, f, w, b = gen_set(rgb_sine_lin, 0)
        rows.append(("Sinebow", rgb_sine_lin, d, b, w))
        d, f, w, b = gen_set(rgb_hoop_lin, 0)
        rows.append(("Max Ellipse", rgb_hoop_lin, d, b, w))
        d, f, w, b = gen_set(rgb_proj_ell_lin, 0)
        rows.append(("Ridge->Ellipsoid", rgb_proj_ell_lin, d, b, w))
        d, f, w, b = gen_set(rgb_proj_srgb_lin, 0)
        rows.append(("Ridge->sRGB", rgb_proj_srgb_lin, d, b, w))
        d, f, w, b = gen_set(rgb_fourier_lin, 0)
        rows.append((f"Fourier (N={N_HARMONICS})", rgb_fourier_lin, d, b, w))

        print("5. Plotting 2D...")
        plt.style.use('dark_background')
        fig, axes = plt.subplots(5, 4, figsize=(16, 20))
        titles = ["Gradient", "Hue Disk", "Black (0°)", "White (0°)"]

        for i, (name, ring, disk, b, w) in enumerate(rows):
            ax_row = axes[i]
            ax_row[0].imshow(linear_to_srgb_np(ring)[None, ...], aspect='auto')
            ax_row[0].set_ylabel(name, fontsize=14, color='white', rotation=90, labelpad=20)
            ax_row[1].imshow(disk); ax_row[2].imshow(b); ax_row[3].imshow(w)
            for j, ax in enumerate(ax_row): 
                if i==0: ax.set_title(titles[j], fontsize=12, color='#dddddd')
                ax.set_xticks([]); ax.set_yticks([])
        plt.tight_layout()
        plt.show()
        
        print("6. Generating 3D...")
        fig3d = go.Figure()
        max_jz = np.max(jz_gamut_opt[:,0]); sc = np.array([1.0/max_jz, 1.0, 1.0]); 
        def mc(a): return a[:,1], a[:,2], a[:,0]

        hull_macadam = ConvexHull(macadam_jz)
        mx, my, mz = mc(macadam_jz * sc)
        fig3d.add_trace(go.Mesh3d(x=mx, y=my, z=mz, i=hull_macadam.simplices[:,0], j=hull_macadam.simplices[:,1], k=hull_macadam.simplices[:,2], color='white', opacity=0.05, name='Visible Gamut'))

        corners = [[0,0,0],[1,0,0],[1,1,0],[0,1,0],[0,0,1],[1,0,1],[1,1,1],[0,1,1]]
        edges_idx = [[0,1],[1,2],[2,3],[3,0],[4,5],[5,6],[6,7],[7,4],[0,4],[1,5],[2,6],[3,7]]
        for s,e in edges_idx:
            l = xyz_to_jzazbz(srgb_to_xyz(np.linspace(corners[s],corners[e],25))) * sc
            x,y,z = mc(l)
            fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', line=dict(color='gray', width=2), showlegend=False))

        # Plot just the orientation ellipsoid for ref, not the stretched loop
        u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
        sph = np.stack([np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v)], axis=-1).reshape(-1, 3)
        ell = (sph @ M_solver.T + center) * sc
        ex, ey, ez = mc(ell)
        fig3d.add_trace(go.Surface(x=ex.reshape(u.shape), y=ey.reshape(u.shape), z=ez.reshape(u.shape), colorscale=[(0,'#88aa88'),(1,'#88aa88')], opacity=0.2, showscale=False, name='Orientation Ellipsoid'))

        def plot_dots(pts, col, name):
             x,y,z = mc(pts * sc)
             rgb = linear_to_srgb_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(pts)), 0, 1))
             cols = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in rgb]
             fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='markers', marker=dict(color=cols, size=4), name=name))

        plot_dots(max_loop_ang, 'orange', 'Maximal Loop')
        plot_dots(xyz_to_jzazbz(srgb_to_xyz(rgb_sine_std)), 'green', 'Sinebow Loop')
        plot_dots(proj_srgb_jz_ang, 'red', 'Ridge->sRGB')
        plot_dots(fourier_jz_ang, 'cyan', f'Constrained Fourier')

        rrx, rry, rrz = mc(ridge_jz_clean[::10] * sc)
        fig3d.add_trace(go.Scatter3d(x=rrx, y=rry, z=rrz, mode='lines', line=dict(color='white', width=3), name='MacAdam Ridge (Ref)'))

        fig3d.update_layout(template="plotly_dark", title=f"3D Jzazbz: Strict Constrained Fourier (N={N_HARMONICS})", 
                            scene=dict(xaxis_title='az', yaxis_title='bz', zaxis_title='Jz', aspectmode='manual', aspectratio=dict(x=1,y=1,z=1)))
        fig3d.show()

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import scipy.optimize as opt
from scipy.spatial import ConvexHull
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import omnirefactor.core
from skimage import io
import fastremap
import scipy.ndimage

# ==========================================
# 1. Color Math
# ==========================================
def srgb_to_xyz(rgb):
    mask = rgb > 0.04045
    linear = np.zeros_like(rgb)
    linear[mask] = ((rgb[mask] + 0.055) / 1.055) ** 2.4
    linear[~mask] = rgb[~mask] / 12.92
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    return linear @ M.T

def xyz_to_jzazbz(xyz):
    b, g, d, d0 = 1.15, 0.66, -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    lms = xyz @ M1.T
    lms_norm = (lms * 200) / 10000.0 
    lms_prime = np.sign(lms_norm) * (((c1 + c2 * (np.abs(lms_norm) ** n)) / (1 + c3 * (np.abs(lms_norm) ** n))) ** p)
    izazbz = lms_prime @ M2.T
    Jz = ((1 + d) * izazbz[:, 0]) / (1 + d * izazbz[:, 0]) - d0
    return np.column_stack((Jz, izazbz[:, 1], izazbz[:, 2]))

def jzazbz_to_xyz(jzazbz):
    Jz, az, bz = jzazbz[:,0], jzazbz[:,1], jzazbz[:,2]
    d, d0 = -0.56, 1.6295499532821566e-11
    c1, c2, c3 = 3424/4096, 2413/128, 2392/128
    n, p = 2610/16384, 1.7 * 2523/32
    M1 = np.array([[0.674207838, 0.382799340, -0.047570458], [0.149284160, 0.739628340, 0.083327300], [0.070941080, 0.174768000, 0.670970020]])
    M2 = np.array([[0.5, 0.5, 0], [3.524000, -4.066708, 0.542708], [0.199076, 1.096799, -1.295875]])
    Jp = Jz + d0; Iz = Jp / (1 + d - d * Jp)
    izazbz_vec = np.stack([Iz, az, bz], axis=1)
    lms_prime = izazbz_vec @ np.linalg.inv(M2).T
    sign_lms = np.sign(lms_prime)
    Y = np.abs(lms_prime) ** (1/p)
    num = Y - c1; den = c2 - Y * c3
    An = np.clip(num / den, 0, None) 
    lms_norm = sign_lms * (An ** (1/n))
    lms = (lms_norm * 10000.0) / 200.0
    return lms @ np.linalg.inv(M1).T

def xyz_to_srgb_raw(xyz):
    M_inv = np.linalg.inv(np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]]))
    linear = xyz @ M_inv.T
    # Gamma encode
    srgb = np.zeros_like(linear)
    mask = np.abs(linear) > 0.0031308
    srgb[mask] = 1.055 * (np.sign(linear[mask]) * np.abs(linear[mask]) ** (1/2.4)) - 0.055
    srgb[~mask] = 12.92 * linear[~mask]
    return srgb

def srgb_to_linear_np(srgb): return np.where(srgb <= 0.04045, srgb / 12.92, ((srgb + 0.055) / 1.055) ** 2.4)
def linear_to_srgb_np(lin): 
    s = np.zeros_like(lin)
    mask = lin > 0.0031308
    s[mask] = 1.055 * (lin[mask]**(1/2.4)) - 0.055
    s[~mask] = 12.92 * lin[~mask]
    return np.clip(s, 0, 1)

# ==========================================
# 2. MacAdam Ridge (High Res + Fourier Clean)
# ==========================================

def get_cmf_high_res(res=360):
    base_cmf = np.array([
        [0.0014,0.0000,0.0065], [0.0042,0.0001,0.0201], [0.0143,0.0004,0.0679], [0.0435,0.0012,0.2074],
        [0.1344,0.0040,0.6456], [0.2839,0.0116,1.3856], [0.3483,0.0230,1.7471], [0.3362,0.0380,1.7721],
        [0.2908,0.0600,1.6692], [0.1954,0.0910,1.2876], [0.0956,0.1390,0.8130], [0.0320,0.2080,0.4652],
        [0.0049,0.3230,0.2720], [0.0093,0.5030,0.1582], [0.0633,0.7100,0.0782], [0.1655,0.8620,0.0422],
        [0.2904,0.9540,0.0203], [0.4334,0.9950,0.0087], [0.5945,0.9950,0.0039], [0.7621,0.9520,0.0021],
        [0.9163,0.8700,0.0017], [1.0263,0.7570,0.0011], [1.0622,0.6310,0.0008], [1.0026,0.5030,0.0006],
        [0.8544,0.3810,0.0002], [0.6424,0.2650,0.0000], [0.4479,0.1750,0.0000], [0.2835,0.1070,0.0000],
        [0.1649,0.0610,0.0000], [0.0874,0.0320,0.0000], [0.0468,0.0170,0.0000], [0.0227,0.0082,0.0000],
        [0.0114,0.0041,0.0000], [0.0058,0.0021,0.0000], [0.0029,0.0010,0.0000], [0.0014,0.0005,0.0000]
    ])
    x = np.linspace(0, 1, len(base_cmf))
    xi = np.linspace(0, 1, res)
    interp_cmf = np.zeros((res, 3))
    for k in range(3):
        interp_cmf[:,k] = np.interp(xi, x, base_cmf[:,k])
    return interp_cmf

def compute_macadam_solid(res=400): 
    cmf = get_cmf_high_res(600)
    n_lambda = len(cmf)
    cum_cmf = np.vstack([np.zeros(3), np.cumsum(cmf, axis=0)])
    white_xyz = cum_cmf[-1]
    
    optimal_colors = []
    indices = np.linspace(0, n_lambda, res).astype(int)
    for i in indices:
        for j in indices:
            if i == j: continue
            if j > i: xyz = cum_cmf[j] - cum_cmf[i]
            else:     xyz = cum_cmf[j] + (white_xyz - cum_cmf[i])
            optimal_colors.append(xyz)
    optimal_colors.append([0,0,0])
    optimal_colors.append(white_xyz)
    return np.array(optimal_colors) / white_xyz[1]

def fourier_smooth_curve(points, harmonics=15):
    N = len(points)
    new_cols = []
    for i in range(3):
        sig = points[:,i]
        coeffs = np.fft.rfft(sig)
        filt = np.zeros_like(coeffs)
        filt[:harmonics+1] = coeffs[:harmonics+1]
        new_cols.append(np.fft.irfft(filt, n=N))
    return np.column_stack(new_cols)

def extract_macadam_ridge_dense(macadam_jz, res=3600): 
    az = macadam_jz[:, 1]; bz = macadam_jz[:, 2]
    chroma = np.sqrt(az**2 + bz**2); angle = np.arctan2(bz, az)
    bins = np.linspace(-np.pi, np.pi, res+1)
    
    sort_idx = np.argsort(angle)
    angle_s = angle[sort_idx]; chroma_s = chroma[sort_idx]; pts_s = macadam_jz[sort_idx]
    bin_idx = np.searchsorted(angle_s, bins)
    
    ridge_points = []
    for i in range(res):
        start, end = bin_idx[i], bin_idx[i+1]
        if start == end:
            if len(ridge_points)>0: ridge_points.append(ridge_points[-1])
            else: ridge_points.append(macadam_jz[0])
            continue
        idx_max = np.argmax(chroma_s[start:end])
        ridge_points.append(pts_s[start + idx_max])
        
    raw = np.array(ridge_points)
    # N=15 Fourier Smooth to kill quantization noise
    return fourier_smooth_curve(raw, harmonics=15)

# ==========================================
# 3. Optimization & Projections
# ==========================================

def get_gamut_surface(res=40):
    t = np.linspace(0, 1, res); g = np.meshgrid(t, t)
    faces = [np.stack([g[0], g[1], np.zeros_like(g[0])], -1), np.stack([g[0], g[1], np.ones_like(g[0])], -1),
             np.stack([g[0], np.zeros_like(g[0]), g[1]], -1), np.stack([g[0], np.ones_like(g[0]), g[1]], -1),
             np.stack([np.zeros_like(g[0]), g[0], g[1]], -1), np.stack([np.ones_like(g[0]), g[0], g[1]], -1)]
    pts = np.vstack([f.reshape(-1, 3) for f in faces])
    return np.vstack([pts, [[0,0,0], [1,0,0], [0,1,0], [0,0,1], [1,1,0], [1,0,1], [0,1,1], [1,1,1]]])

def fit_ellipsoid_anchored_solver(points, anchor_point):
    hull = ConvexHull(points)
    A = hull.equations[:, :3]; b_const = -hull.equations[:, 3]
    scale = 100.0; A_scaled = A / scale; anchor_scaled = anchor_point * scale
    c0 = np.mean(points, axis=0) * scale
    x0 = np.concatenate([0.5*c0 + 0.5*anchor_scaled, np.eye(3).flatten() * 0.05])
    def objective(x):
        M = x[3:].reshape(3, 3); sign, val = np.linalg.slogdet(M)
        return -val if sign > 0 else 1e9
    def constraints(x):
        d, M = x[:3], x[3:].reshape(3, 3)
        norm_AM = np.linalg.norm(A_scaled @ M, axis=1)
        hull_con = b_const - (A_scaled @ d + norm_AM)
        try: u = np.linalg.solve(M, anchor_scaled - d); anchor_con = 1.0 - np.linalg.norm(u)
        except: anchor_con = -1.0
        return np.concatenate([hull_con, [anchor_con]])
    res = opt.minimize(objective, x0, method='SLSQP', constraints={'type': 'ineq', 'fun': constraints}, options={'maxiter': 1000})
    return res.x[:3]/scale, res.x[3:].reshape(3, 3)/scale, hull

def optimize_maximal_2d_loop(center, M_solver):
    print("   Maximizing 2D Loop Dimensions...")
    view_vec = np.array([1.0, 0.0, 0.0])
    w = np.linalg.solve(M_solver, view_vec); w = w / np.linalg.norm(w)
    tmp = np.array([0.0, 1.0, 0.0]) if np.abs(w[1]) < 0.9 else np.array([0.0, 0.0, 1.0])
    s1 = np.cross(w, tmp); s1 /= np.linalg.norm(s1); s2 = np.cross(w, s1)
    
    U_base = s1 @ M_solver.T
    V_base = s2 @ M_solver.T
    
    # Coarse Grid + Fine Tune
    best_su, best_sv = 1.0, 1.0
    
    theta = np.linspace(0, 2*np.pi, 128)
    cos_t = np.cos(theta); sin_t = np.sin(theta)
    
    # 1. Find Approximate Bounds
    for su in np.linspace(1.0, 3.0, 30):
        for sv in np.linspace(1.0, 3.0, 30):
            loop = center + np.outer(cos_t, U_base * su) + np.outer(sin_t, V_base * sv)
            rgb = xyz_to_srgb_raw(jzazbz_to_xyz(loop))
            if np.min(rgb) >= -0.001 and np.max(rgb) <= 1.001:
                if (su * sv) > (best_su * best_sv):
                    best_su, best_sv = su, sv
                    
    # 2. Safety Shrink (Guarantee valid)
    # If optimizer went slightly over (due to 128 pts), shrink until valid
    for _ in range(50):
        theta_hr = np.linspace(0, 2*np.pi, 360)
        loop = center + np.outer(np.cos(theta_hr), U_base * best_su) + np.outer(np.sin(theta_hr), V_base * best_sv)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(loop))
        if np.min(rgb) >= -0.001 and np.max(rgb) <= 1.001:
            break
        best_su *= 0.99
        best_sv *= 0.99
        
    print(f"      Final Scale: U={best_su:.2f}, V={best_sv:.2f}")
    
    # Reconstruct High Res
    theta_hr = np.linspace(0, 2*np.pi, 256)
    return center + np.outer(np.cos(theta_hr), U_base * best_su) + np.outer(np.sin(theta_hr), V_base * best_sv)

def project_points_to_srgb_surface_conical(points_jz, center_jz):
    projected = []
    for pt in points_jz:
        vec = pt - center_jz
        low = 0.0; high = 1.5; best_t = 0.0
        for _ in range(15):
            mid = (low + high) / 2.0
            test = center_jz + mid * vec
            rgb = xyz_to_srgb_raw(jzazbz_to_xyz(np.array([test])))
            if np.min(rgb) >= -0.001 and np.max(rgb) <= 1.001:
                best_t = mid; low = mid 
            else: high = mid 
        projected.append(center_jz + best_t * vec)
    return np.array(projected)

def reparameterize_by_hue_angle(points, n_samples=256):
    az = points[:, 1]; bz = points[:, 2]
    angles = np.arctan2(bz, az)
    angles = np.unwrap(angles)
    target_angles = np.linspace(angles[0], angles[0] + 2*np.pi, n_samples, endpoint=False)
    angles_pad = np.concatenate([angles - 2*np.pi, angles, angles + 2*np.pi])
    J_pad = np.tile(points[:, 0], 3)
    a_pad = np.tile(points[:, 1], 3)
    b_pad = np.tile(points[:, 2], 3)
    return np.column_stack([
        np.interp(target_angles, angles_pad, J_pad),
        np.interp(target_angles, angles_pad, a_pad),
        np.interp(target_angles, angles_pad, b_pad)
    ])

def fit_fourier_constrained(points, center_point, harmonics=2, iterations=50):
    print(f"   Fitting Constrained Fourier (N={harmonics})...")
    current_points = points.copy()
    N = len(points)
    
    for i in range(iterations):
        new_cols = []
        for d in range(3):
            sig = current_points[:,d]
            coeffs = np.fft.rfft(sig)
            filt = np.zeros_like(coeffs)
            filt[:harmonics+1] = coeffs[:harmonics+1]
            new_cols.append(np.fft.irfft(filt, n=N))
        smooth_points = np.column_stack(new_cols)
        
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(smooth_points))
        is_out = (rgb < -0.001) | (rgb > 1.001)
        out_idx = np.any(is_out, axis=1)
        
        if not np.any(out_idx): return smooth_points
        
        vecs = center_point - smooth_points
        smooth_points[out_idx] += vecs[out_idx] * 0.2
        current_points = smooth_points
        
    # Safety Shrink
    scale = 1.0
    for _ in range(100):
        test = center_point + scale * (current_points - center_point)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(test))
        if np.min(rgb) >= -0.002 and np.max(rgb) <= 1.002: return test
        scale -= 0.005
    return current_points

def shrink_ridge_uniform(ridge_jz, center_point):
    scale = 1.0; step = 0.005
    for _ in range(500):
        test_ridge = center_point + scale * (ridge_jz - center_point)
        rgb = xyz_to_srgb_raw(jzazbz_to_xyz(test_ridge))
        if np.mean(np.min(rgb, axis=1) >= -0.001) > 0.99: 
             if np.mean(np.max(rgb, axis=1) <= 1.001) > 0.99: return test_ridge
        scale -= step
    return ridge_jz * 0.1

def check_gamut_compliance(rgb_loop, name="Loop"):
    min_val = np.min(rgb_loop); max_val = np.max(rgb_loop)
    if min_val < -0.002 or max_val > 1.002:
        print(f"   >>> {name} FAIL: [{min_val:.4f}, {max_val:.4f}]")
    else:
        print(f"   >>> {name} PASS: Inside sRGB")

# ==========================================
# 4. Omnipose Helpers
# ==========================================
def align_ring_red_to_green(ring_lin):
    n = len(ring_lin)
    red_ref = np.array([1.0, 0.0, 0.0])
    idx_r = np.argmin(np.linalg.norm(ring_lin - red_ref, axis=1))
    ring_aligned = np.roll(ring_lin, -idx_r, axis=0)
    green_ref = np.array([0.0, 1.0, 0.0])
    if np.linalg.norm(ring_aligned[(2*n)//3]-green_ref) < np.linalg.norm(ring_aligned[n//3]-green_ref):
        ring_aligned = ring_aligned[::-1]
    return ring_aligned

def make_flow_images_for_ring(rgb_ring_lin, center_lin, flows):
    def build_disk(ring, center):
        y, x = np.mgrid[-1:1:256j, -1:1:256j]
        mag = np.sqrt(x*x + y*y); angle = np.mod(np.arctan2(y, x), 2*np.pi)
        n = ring.shape[0]; hue_f = angle/(2*np.pi)*n
        idx0 = np.floor(hue_f).astype(int) % n; idx1 = (idx0 + 1) % n; t = hue_f - np.floor(hue_f)
        interp = (1-t[...,None])*ring[idx0] + t[...,None]*ring[idx1]
        rgb = (1-mag[...,None])*center + mag[...,None]*interp
        return np.clip(linear_to_srgb_np(rgb),0,1)
    disk = build_disk(rgb_ring_lin, center_lin)
    u = flows[0].cpu().numpy(); v = flows[1].cpu().numpy()
    angle = np.mod(np.arctan2(v, u), 2*np.pi); mag = np.clip(np.sqrt(u*u + v*v), 0, 1)
    n = rgb_ring_lin.shape[0]; hue_f = angle/(2*np.pi)*n
    idx0 = np.floor(hue_f).astype(int) % n; idx1 = (idx0 + 1) % n; t = hue_f - np.floor(hue_f)
    ring_interp = (1 - t[..., None]) * rgb_ring_lin[idx0] + t[..., None] * rgb_ring_lin[idx1]
    r = mag[..., None]; rgb_lin_flow = (1 - r) * center_lin + r * ring_interp
    flow_rgb = np.clip(linear_to_srgb_np(rgb_lin_flow),0,1)
    alpha = mag[...,None]; white = np.ones_like(flow_rgb); black = np.zeros_like(flow_rgb)
    flow_white = alpha*flow_rgb + (1-alpha)*white; flow_black = alpha*flow_rgb + (1-alpha)*black
    return disk, flow_rgb, flow_white, flow_black

# ==========================================
# 5. Main Execution
# ==========================================

def main():
    N_HARMONICS = 2
    device_str = "cpu"
    dev = torch.device(device_str)
    
    print("1. Calculating Geometries...")
    surf_pts_opt = get_gamut_surface(40) 
    jz_gamut_opt = xyz_to_jzazbz(srgb_to_xyz(surf_pts_opt))
    centroid = np.mean(jz_gamut_opt, axis=0)
    green_vertex = xyz_to_jzazbz(srgb_to_xyz(np.array([[0.0, 1.0, 0.0]])))[0]
    anchor = centroid + 0.60 * (green_vertex - centroid)
    
    # 2. Ellipsoid Fitting (Maximal 2D Stretch)
    print("2. Fitting Maximal Ellipsoid...")
    center, M_solver, hull = fit_ellipsoid_anchored_solver(jz_gamut_opt, anchor)
    max_loop_jz = optimize_maximal_2d_loop(center, M_solver) # Stretcher
    max_loop_ang = reparameterize_by_hue_angle(max_loop_jz, n_samples=256)
    rgb_hoop_raw = xyz_to_srgb_raw(jzazbz_to_xyz(max_loop_ang))
    check_gamut_compliance(rgb_hoop_raw, "Maximal Loop")
    rgb_hoop_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(rgb_hoop_raw,0,1)))
    
    # 3. Sinebow
    t = np.linspace(0, 1, 256, endpoint=False)
    sr = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 0/3)); sg = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 1/3)); sb = 0.5 + 0.5 * np.sin(2 * np.pi * (t + 2/3))
    rgb_sine_std = np.stack([sr, sg, sb], axis=1)
    rgb_sine_lin = align_ring_red_to_green(srgb_to_linear_np(rgb_sine_std))
    
    # 4. MacAdam Ridge
    print("3. Computing MacAdam Ridge...")
    macadam_xyz = compute_macadam_solid(res=200)
    macadam_jz = xyz_to_jzazbz(macadam_xyz)
    ridge_jz_clean = extract_macadam_ridge_dense(macadam_jz, res=3600)
    
    # Ridge -> sRGB (Conical)
    proj_srgb_jz_raw = project_points_to_srgb_surface_conical(ridge_jz_clean, centroid) 
    proj_srgb_jz_ang = reparameterize_by_hue_angle(proj_srgb_jz_raw, n_samples=256)
    rgb_proj_srgb_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(proj_srgb_jz_ang)), 0, 1)))
    
    # Fourier Constrained (N=2)
    fourier_jz = fit_fourier_constrained(proj_srgb_jz_ang, centroid, harmonics=N_HARMONICS)
    fourier_jz_ang = reparameterize_by_hue_angle(fourier_jz, n_samples=256)
    rgb_fourier_raw = xyz_to_srgb_raw(jzazbz_to_xyz(fourier_jz_ang))
    check_gamut_compliance(rgb_fourier_raw, "Fourier")
    rgb_fourier_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(rgb_fourier_raw, 0, 1)))
    
    # Uniform Shrink (Rigid Fit)
    shrunk_ridge_jz_raw = shrink_ridge_uniform(ridge_jz_clean, centroid)
    shrunk_ridge_jz_ang = reparameterize_by_hue_angle(shrunk_ridge_jz_raw, n_samples=256)
    rgb_shrunk_lin = align_ring_red_to_green(srgb_to_linear_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(shrunk_ridge_jz_ang)), 0, 1)))

    print("4. Loading Omnipose Flows...")
    omnidir = Path(omnirefactor.__file__).resolve().parent.parent.parent
    basedir = omnidir / "docs" / "_static"
    name = "ecoli"
    ext = ".tif"

    try:
        masks = io.imread(str(basedir / f"{name}_labels{ext}"))
        slc = omnirefactor.measure.crop_bbox(masks > 0, pad=0)[0]
        masks = fastremap.renumber(masks[slc])[0]
        flows = omnirefactor.core.masks_to_flows(masks, device=device_str)[4].to(dev)
        center_lin = srgb_to_linear_np(np.array([0.5, 0.5, 0.5]))

        def gen_set(ring, rot_steps=0):
            idx = int(rot_steps * len(ring) / 72.0)
            r = np.roll(ring, idx, axis=0)
            return make_flow_images_for_ring(r, center_lin, flows)

        rows = []
        d, f, w, b = gen_set(rgb_sine_lin, 0)
        rows.append(("Sinebow", rgb_sine_lin, d, b, w))
        d, f, w, b = gen_set(rgb_hoop_lin, 0)
        rows.append(("Maximal Loop", rgb_hoop_lin, d, b, w))
        d, f, w, b = gen_set(rgb_proj_srgb_lin, 0)
        rows.append(("Ridge->sRGB", rgb_proj_srgb_lin, d, b, w))
        d, f, w, b = gen_set(rgb_fourier_lin, 0)
        rows.append((f"Fourier (N={N_HARMONICS})", rgb_fourier_lin, d, b, w))
        d, f, w, b = gen_set(rgb_shrunk_lin, 0)
        rows.append(("Rigid Fit", rgb_shrunk_lin, d, b, w))

        print("5. Plotting 2D...")
        plt.style.use('dark_background')
        fig, axes = plt.subplots(5, 4, figsize=(16, 20))
        for i, (name, ring, disk, b, w) in enumerate(rows):
            ax_row = axes[i]
            ax_row[0].imshow(linear_to_srgb_np(ring)[None, ...], aspect='auto')
            ax_row[0].set_ylabel(name, fontsize=14, color='white', rotation=90, labelpad=20)
            ax_row[1].imshow(disk); ax_row[2].imshow(b); ax_row[3].imshow(w)
            for j, ax in enumerate(ax_row): ax.set_xticks([]); ax.set_yticks([])
        plt.tight_layout()
        plt.show()
        
        print("6. Generating 3D...")
        fig3d = go.Figure()
        max_jz = np.max(jz_gamut_opt[:,0]); sc = np.array([1.0/max_jz, 1.0, 1.0]); 
        def mc(a): return a[:,1], a[:,2], a[:,0]

        hull_macadam = ConvexHull(macadam_jz)
        mx, my, mz = mc(macadam_jz * sc)
        fig3d.add_trace(go.Mesh3d(x=mx, y=my, z=mz, i=hull_macadam.simplices[:,0], j=hull_macadam.simplices[:,1], k=hull_macadam.simplices[:,2], color='white', opacity=0.05, name='Visible Gamut'))
        
        corners = [[0,0,0],[1,0,0],[1,1,0],[0,1,0],[0,0,1],[1,0,1],[1,1,1],[0,1,1]]
        edges_idx = [[0,1],[1,2],[2,3],[3,0],[4,5],[5,6],[6,7],[7,4],[0,4],[1,5],[2,6],[3,7]]
        for s,e in edges_idx:
            l = xyz_to_jzazbz(srgb_to_xyz(np.linspace(corners[s],corners[e],25))) * sc
            x,y,z = mc(l)
            fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', line=dict(color='gray', width=2), showlegend=False))

        u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
        sph = np.stack([np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v)], axis=-1).reshape(-1, 3)
        ell = (sph @ M_solver.T + center) * sc
        ex, ey, ez = mc(ell)
        fig3d.add_trace(go.Surface(x=ex.reshape(u.shape), y=ey.reshape(u.shape), z=ez.reshape(u.shape), colorscale=[(0,'#88aa88'),(1,'#88aa88')], opacity=0.2, showscale=False, name='Ellipsoid'))

        def plot_dots(pts, col, name):
             x,y,z = mc(pts * sc)
             rgb = linear_to_srgb_np(np.clip(xyz_to_srgb_raw(jzazbz_to_xyz(pts)), 0, 1))
             cols = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r,g,b in rgb]
             fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='markers', marker=dict(color=cols, size=4), name=name))

        plot_dots(max_loop_ang, 'orange', 'Maximal Loop')
        plot_dots(xyz_to_jzazbz(srgb_to_xyz(rgb_sine_std)), 'green', 'Sinebow Loop')
        plot_dots(proj_srgb_jz_ang, 'red', 'Ridge->sRGB')
        plot_dots(fourier_jz_ang, 'cyan', f'Constrained Fourier')
        plot_dots(shrunk_ridge_jz_ang, 'magenta', 'Rigid Fit')

        rrx, rry, rrz = mc(ridge_jz_clean[::10] * sc)
        fig3d.add_trace(go.Scatter3d(x=rrx, y=rry, z=rrz, mode='lines', line=dict(color='white', width=3), name='MacAdam Ridge (Ref)'))

        fig3d.update_layout(template="plotly_dark", title=f"3D Jzazbz: Final Solution", 
                            scene=dict(xaxis_title='az', yaxis_title='bz', zaxis_title='Jz', aspectmode='manual', aspectratio=dict(x=1,y=1,z=1)))
        fig3d.show()

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
try smoothing the rgb gamut to knock off the convex corners. 